Private free CPU joint optimization of all24 route coefficients, current route warmstart,25% per-coefficient trust bounds and10% perturbations. Fixed11-member route/v1/neural rival pool, shared four-seat training80m, fresh81m selection and82m final. No model or main promotion; all inputs own/public only.

In [ ]:
import pathlib,subprocess,json,os
work=pathlib.Path("/kaggle/temp/triad-uno-joint-route")
work.mkdir(parents=True,exist_ok=True)
sources={'uno_policy.py': '"""Linear action-ranking policy; inference uses filtered observations only."""\nfrom collections import Counter\nimport json\nfrom pathlib import Path\n\nFEATURE_NAMES = [\n    "play", "number", "skip", "reverse", "draw_two", "wild", "wild_draw_four",\n    "remaining_colour_count", "remaining_colour_fraction", "remaining_match_count",\n    "attack_next_threat", "reverse_escape_threat", "wild_when_small_hand",\n    "finish", "keep", "choose_colour", "next_can_finish", "own_hand_size",\n]\nINITIAL_WEIGHTS = [8., .5, -.2, -.2, -.2, -2., -3., 2., 1., 1., 4., 2., 2., 12., -4., 0., 0., 0.]\n\n\ndef action_features(observation, action):\n    hand = observation["hand"]\n    by_ref = {card["card_ref"]: card for card in hand}\n    card = action.get("card") or by_ref.get(action.get("card_ref")) or {}\n    kind = card.get("kind")\n    play = action["type"] == "play"\n    colour = action.get("color") or card.get("color") or observation.get("active_color")\n    remaining = [c for c in hand if not (play and c["card_ref"] == action.get("card_ref"))]\n    colours = Counter(c.get("color") for c in remaining if c.get("color"))\n    matching = sum(c.get("color") == colour or c.get("kind") in ("wild", "wild_draw_four")\n                   or (kind == "number" and c.get("kind") == "number" and c.get("value") == card.get("value"))\n                   or (kind not in (None, "number", "wild", "wild_draw_four") and c.get("kind") == kind)\n                   for c in remaining)\n    sizes = observation.get("hand_sizes", [len(hand)] * 4)\n    seat, direction = observation["seat"], observation.get("direction", 1)\n    active = [s for s in range(len(sizes)) if s not in observation.get("finished_order", [])]\n    def neighbour(d):\n        for distance in range(1, len(sizes)):\n            other = (seat + distance * d) % len(sizes)\n            if other in active:\n                return max(1, sizes[other])\n        return max(1, len(hand))\n    next_size, previous_size = neighbour(direction), neighbour(-direction)\n    threat = 1. / next_size\n    attack = play and kind in ("skip", "draw_two", "wild_draw_four")\n    return [float(play), float(kind == "number"), float(kind == "skip"),\n            float(kind == "reverse"), float(kind == "draw_two"), float(kind == "wild"),\n            float(kind == "wild_draw_four"), min(colours[colour], 12) / 4.,\n            colours[colour] / max(1, len(remaining)), min(matching, 12) / 4.,\n            float(attack) * threat,\n            float(play and kind == "reverse") * (threat - 1. / previous_size),\n            float(play and kind in ("wild", "wild_draw_four")) / max(1, len(hand)),\n            float(play and len(hand) == 1), float(action["type"] == "keep"),\n            float(action["type"] == "choose_color"), float(play) * float(next_size <= 2),\n            float(play) * min(len(hand), 24) / 12.]\n\n\nclass Policy:\n    def __init__(self, weights):\n        if len(weights) != len(FEATURE_NAMES):\n            raise ValueError("wrong policy feature count")\n        self.weights = [float(w) for w in weights]\n\n    def select(self, observation):\n        legal = observation["legal_actions"]\n        return max(legal, key=lambda action: sum(w * x for w, x in\n                   zip(self.weights, action_features(observation, action))))["action_id"]\n\n    def save(self, path, metadata):\n        Path(path).parent.mkdir(parents=True, exist_ok=True)\n        Path(path).write_text(json.dumps({"format": "triad-uno-linear-v1",\n            "features": FEATURE_NAMES, "weights": self.weights,\n            "training": metadata}, indent=2) + "\\n")\n', 'uno_policy_v2.py': '"""History-aware non-language UNO policy with public-information beliefs."""\nfrom collections import Counter, defaultdict\nimport json\nimport math\nfrom pathlib import Path\n\nfrom uno_policy import FEATURE_NAMES as V1_FEATURES, action_features as v1_features\n\nEXTRA_FEATURES = [\n    "successor_colour_absence", "successor_symbol_absence", "successor_colour_preference",\n    "successor_finish_risk", "remaining_colour_components", "largest_component_fraction",\n    "remaining_bridge_symbols", "remaining_duplicates", "remaining_colour_entropy",\n    "remaining_wilds", "chosen_colour_exhausted", "stack_penalty_play",\n    "successor_one_card_colour_absence", "successor_one_card_symbol_absence",\n]\nFEATURE_NAMES = V1_FEATURES + EXTRA_FEATURES\n\n\ndef symbol(card):\n    kind = card.get("kind")\n    if kind == "number":\n        return (kind, card.get("value"))\n    return (kind, None) if kind in ("skip", "reverse", "draw_two") else None\n\n\ndef public_beliefs(observation):\n    """Fold public events; a normal forced draw reveals pre-draw absences.\n\n    Unknown draws soften older evidence. Auto-kept or played drawn cards allow\n    exact post-draw absence deductions. Penalty draws do not imply any absence.\n    No simulator state, private opponent cards or seed is accessed here.\n    """\n    lack_colour, lack_symbol, preference, pending_draw = {}, {}, {}, {}\n    active_colour, top_symbol, pending_penalty = None, None, 0\n    for event in observation.get("history", []):\n        kind, seat = event.get("type"), event.get("seat")\n        if kind in ("game_start", "initial_discard"):\n            card = event.get("card", event.get("initial_discard", {})) or {}\n            active_colour = card.get("color", active_colour)\n            top_symbol = symbol(card)\n        elif kind == "initial_color_choice":\n            active_colour = event["color"]\n            preference[seat] = active_colour\n        elif kind in ("draw", "penalty_draw"):\n            count = event.get("drawn", 0) if kind == "draw" else event.get("count", 0)\n            for key in list(lack_colour):\n                if key[0] == seat:\n                    lack_colour[key] *= .75 ** count\n            for key in list(lack_symbol):\n                if key[0] == seat:\n                    lack_symbol[key] *= .92 ** count\n            if count:\n                preference.pop(seat, None)\n            if kind == "draw":\n                if active_colour:\n                    lack_colour[(seat, active_colour)] = .75 ** count\n                if top_symbol:\n                    lack_symbol[(seat, top_symbol)] = .92 ** count\n                pending_draw[seat] = (event.get("turn_number"), active_colour, top_symbol)\n            else:\n                pending_penalty = 0\n        elif kind in ("keep_auto", "play"):\n            card = event.get("card") or {}\n            if kind == "play":\n                colour, played_symbol = card.get("color"), symbol(card)\n                if colour:\n                    lack_colour[(seat, colour)] = 0.\n                if played_symbol:\n                    lack_symbol[(seat, played_symbol)] = 0.\n            drawn = pending_draw.get(seat)\n            if drawn and drawn[0] == event.get("turn_number") and (kind == "keep_auto" or event.get("from_draw")):\n                if drawn[1]:\n                    lack_colour[(seat, drawn[1])] = 1.\n                if drawn[2]:\n                    lack_symbol[(seat, drawn[2])] = 1.\n                pending_draw.pop(seat, None)\n            if kind == "play":\n                active_colour = event.get("active_color", card.get("color"))\n                top_symbol = symbol(card)\n                if card.get("kind") in ("wild", "wild_draw_four"):\n                    preference[seat] = active_colour\n                if "pending_penalty" in event:\n                    pending_penalty = event["pending_penalty"]\n    return {"colours": lack_colour, "symbols": lack_symbol, "preference": preference, "penalty": pending_penalty}\n\n\ndef structure(hand):\n    counts = Counter(c.get("color") for c in hand if c.get("color"))\n    groups = defaultdict(set)\n    for card in hand:\n        key = symbol(card)\n        if key and card.get("color"):\n            groups[key].add(card["color"])\n    adjacency = {colour: {colour} for colour in counts}\n    for colours in groups.values():\n        for colour in colours:\n            adjacency[colour].update(colours)\n    components, visited = [], set()\n    for colour in counts:\n        if colour in visited:\n            continue\n        frontier, connected = [colour], set()\n        while frontier:\n            current = frontier.pop()\n            if current not in connected:\n                connected.add(current)\n                frontier.extend(adjacency[current] - connected)\n        visited.update(connected)\n        components.append(sum(counts[c] for c in connected))\n    wilds = sum(c.get("kind") in ("wild", "wild_draw_four") for c in hand)\n    if wilds and components:\n        components = [sum(components)]\n    coloured = sum(counts.values())\n    entropy = -sum((n / coloured) * math.log(n / coloured) for n in counts.values()) if coloured else 0.\n    identical = Counter((c.get("color"), c.get("kind"), c.get("value")) for c in hand)\n    return [len(components) / 4., max(components, default=0) / max(1, len(hand)),\n            sum(len(colours) > 1 for colours in groups.values()) / 8.,\n            sum(n - 1 for n in identical.values()) / 4., entropy / 2., wilds / 4.]\n\n\ndef action_features(observation, action, beliefs=None):\n    beliefs = beliefs or public_beliefs(observation)\n    hand = observation["hand"]\n    play = action["type"] == "play"\n    card = action.get("card") or next((c for c in hand if c["card_ref"] == action.get("card_ref")), {})\n    kind = card.get("kind")\n    colour = action.get("color") or card.get("color") or observation.get("active_color")\n    remaining = [c for c in hand if not (play and c["card_ref"] == action.get("card_ref"))]\n    sizes, seat = observation["hand_sizes"], observation["seat"]\n    direction = observation.get("direction", 1)\n    active = [s for s in range(len(sizes)) if s not in observation.get("finished_order", [])]\n    if play and kind == "reverse":\n        direction = -direction\n    steps = 2 if play and kind in ("skip", "draw_two", "wild_draw_four") else 1\n    successor = seat\n    for _ in range(steps):\n        for distance in range(1, len(sizes) + 1):\n            other = (successor + distance * direction) % len(sizes)\n            if other in active:\n                successor = other\n                break\n    absence = beliefs["colours"].get((successor, colour), 0.)\n    absent_symbol = beliefs["symbols"].get((successor, symbol(card)), 0.)\n    threat = 1. / max(1, sizes[successor])\n    one_card = float(sizes[successor] == 1)\n    extras = [absence, absent_symbol, float(beliefs["preference"].get(successor) == colour),\n              threat * (1. - absence), *structure(remaining),\n              float(not any(c.get("color") == colour for c in remaining)),\n              float(play) * min(beliefs["penalty"], 24) / 12.,\n              one_card * absence, one_card * absent_symbol]\n    return v1_features(observation, action) + extras\n\n\nclass Policy:\n    def __init__(self, weights):\n        if len(weights) != len(FEATURE_NAMES) or not all(math.isfinite(float(w)) for w in weights):\n            raise ValueError("invalid v2 policy weights")\n        self.weights = list(weights)\n\n    def select(self, observation):\n        legal = observation["legal_actions"]\n        if len(legal) == 1:\n            return legal[0]["action_id"]\n        beliefs = public_beliefs(observation)\n        return max(legal, key=lambda action: sum(w * x for w, x in\n                   zip(self.weights, action_features(observation, action, beliefs))))["action_id"]\n\n    def save(self, path, metadata):\n        Path(path).parent.mkdir(parents=True, exist_ok=True)\n        Path(path).write_text(json.dumps({"format": "triad-uno-linear-v2", "features": FEATURE_NAMES,\n            "weights": self.weights, "training": metadata}, indent=2) + "\\n")\n', 'uno_route_policy.py': '"""Own-hand sequence features for a frozen learned UNO base policy.\n\nThe bounded search imagines subsequent own plays with no intervening top-card\nchanges. It is a hand-structure feature, not a claim about future opponents.\nOnly filtered observations are read. No simulator or hidden state is imported.\n"""\nfrom collections import Counter\nfrom functools import lru_cache\nimport math\nfrom uno_policy import Policy as Base, action_features as base_features, FEATURE_NAMES as BASE_NAMES\nfrom uno_policy_v2 import public_beliefs, symbol\n\nEXTRA_NAMES = ["sequence_fraction", "sequence_stranded", "ending_wild",\n               "next_colour_absence", "next_symbol_absence", "next_colour_preference"]\nFEATURE_NAMES = BASE_NAMES + EXTRA_NAMES\n\n\ndef route_length(hand, colour, top, max_states=128, max_depth=8):\n    """Longest discovered normal-play path, with a fixed deterministic budget."""\n    cards = tuple(sorted(Counter((c.get(\'color\') or \'\', c[\'kind\'], c.get(\'value\') if c.get(\'value\') is not None else -1)\n                                 for c in hand).items()))\n    types = tuple(c for c, _ in cards)\n    counts = tuple(n for _, n in cards)\n    visits = 0\n\n    @lru_cache(maxsize=256)\n    def walk(counts, active, last_kind, last_value, depth):\n        nonlocal visits\n        if not depth or visits >= max_states:\n            return 0\n        visits += 1\n        best = 0\n        active_count = sum(n for n, (c, _, _) in zip(counts, types) if c == active)\n        for i, ((c, kind, value), n) in enumerate(zip(types, counts)):\n            if not n:\n                continue\n            wild = kind in (\'wild\', \'wild_draw_four\')\n            legal = (kind == \'wild\' or (kind == \'wild_draw_four\' and not active_count)\n                     or c == active or (kind == \'number\' and last_kind == \'number\' and value == last_value)\n                     or kind in (\'skip\', \'reverse\', \'draw_two\') and kind == last_kind)\n            if not legal:\n                continue\n            reduced = counts[:i] + (n - 1,) + counts[i + 1:]\n            options = sorted({other[0] for other, number in zip(types, reduced) if number and other[0]}) if wild else [c]\n            if not options:\n                options = [active]\n            for chosen in options:\n                length = 1 + walk(reduced, chosen, kind, value, depth - 1)\n                best = max(best, length)\n                if best >= min(depth, sum(counts)):\n                    return best\n        return best\n    return walk(counts, colour or \'\', top.get(\'kind\'), top.get(\'value\', -1), min(max_depth, len(hand)))\n\n\ndef action_features(observation, action, beliefs=None, cache=None):\n    beliefs = beliefs or public_beliefs(observation)\n    play = action[\'type\'] == \'play\'\n    card = action.get(\'card\') or next((c for c in observation[\'hand\'] if c[\'card_ref\'] == action.get(\'card_ref\')), {})\n    colour = action.get(\'color\') or card.get(\'color\') or observation.get(\'active_color\')\n    remaining = [c for c in observation[\'hand\'] if not(play and c[\'card_ref\'] == action.get(\'card_ref\'))]\n    top = card if play else observation.get(\'top_discard\', {})\n    key = (tuple(sorted((c.get(\'color\') or \'\', c[\'kind\'], c.get(\'value\', -1) if c.get(\'value\') is not None else -1)\n                        for c in remaining)), colour, top.get(\'kind\'), top.get(\'value\'))\n    if cache is not None and key in cache:\n        length = cache[key]\n    else:\n        length = route_length(remaining, colour, top)\n        if cache is not None:\n            cache[key] = length\n    sizes = observation[\'hand_sizes\']\n    direction = observation.get(\'direction\', 1)\n    if play and card.get(\'kind\') == \'reverse\':\n        direction = -direction\n    active = [s for s in range(len(sizes)) if s not in observation.get(\'finished_order\', [])]\n    successor = observation[\'seat\']\n    steps = 2 if play and card.get(\'kind\') in (\'skip\', \'draw_two\', \'wild_draw_four\') else 1\n    for _ in range(steps):\n        successor = next((successor + distance * direction) % len(sizes) for distance in range(1, len(sizes) + 1)\n                         if (successor + distance * direction) % len(sizes) in active)\n    denom = min(8, len(remaining))\n    extras = [length / max(1, denom), (denom - length) / 8.,\n              float(len(remaining) == 1 and remaining[0][\'kind\'] in (\'wild\', \'wild_draw_four\')),\n              beliefs[\'colours\'].get((successor, colour), 0.),\n              beliefs[\'symbols\'].get((successor, symbol(top)), 0.),\n              float(beliefs[\'preference\'].get(successor) == colour)]\n    return base_features(observation, action) + extras\n\n\nclass Policy:\n    def __init__(self, weights):\n        if len(weights) != len(FEATURE_NAMES) or not all(math.isfinite(w) for w in weights):\n            raise ValueError(\'invalid route policy weights\')\n        self.weights = list(weights)\n\n    def select(self, observation):\n        legal = observation[\'legal_actions\']\n        if len(legal) == 1:\n            return legal[0][\'action_id\']\n        beliefs, cache = public_beliefs(observation), {}\n        return max(legal, key=lambda a: sum(w * x for w, x in zip(self.weights, action_features(observation, a, beliefs, cache))))[\'action_id\']\n', 'uno_policy_features_v3.py': '"""Richer numeric own-hand encoding, canonicalized under colour permutations."""\nfrom collections import Counter\nimport math\nfrom uno_policy_v2 import FEATURE_NAMES as V2_FEATURES, action_features as v2_features, public_beliefs, symbol\n\nKINDS=[(\'number\',n) for n in range(10)]+[(\'skip\',None),(\'reverse\',None),(\'draw_two\',None)]\nFEATURE_NAMES=V2_FEATURES+[\n f\'remaining_colour_slot_{slot}_{kind}_{value}\' for slot in range(4) for kind,value in KINDS]+[\n \'remaining_wild_count\',\'remaining_wild_draw_four_count\']+[\n f\'action_number_{number}\' for number in range(10)]+[\n f\'opponent_slot_{slot}_{name}\' for slot in range(3) for name in (\'present\',\'hand_size\',\'one_card\',\'colour_absence\',\'symbol_absence\',\'colour_preference\')]+[\n \'selected_is_active_colour\',\'pending_penalty_size\',\'turn_progress\',\'active_opponents\',\'closest_opponent_risk\']\n\n\ndef action_features(observation,action,beliefs=None):\n beliefs=beliefs or public_beliefs(observation)\n hand=observation[\'hand\'];play=action[\'type\']==\'play\'\n card=action.get(\'card\') or next((c for c in hand if c[\'card_ref\']==action.get(\'card_ref\')), {})\n selected=action.get(\'color\') or card.get(\'color\') or observation.get(\'active_color\')\n remaining=[c for c in hand if not(play and c[\'card_ref\']==action.get(\'card_ref\'))]\n counts=Counter((c.get(\'color\'),c.get(\'kind\'),c.get(\'value\') if c.get(\'kind\')==\'number\' else None) for c in remaining)\n colours=[\'red\',\'yellow\',\'green\',\'blue\']\n order=[]\n for colour in (selected,observation.get(\'active_color\')):\n  if colour in colours and colour not in order:order.append(colour)\n def key(colour):\n  vector=tuple(counts[(colour,kind,value)] for kind,value in KINDS)\n  return sum(vector),vector\n order+=sorted((c for c in colours if c not in order),key=key,reverse=True)\n vector=[counts[(colour,kind,value)]/2. for colour in order for kind,value in KINDS]\n vector.extend(sum(c.get(\'kind\')==kind for c in remaining)/4. for kind in (\'wild\',\'wild_draw_four\'))\n vector.extend(float(card.get(\'kind\')==\'number\' and card.get(\'value\')==n) for n in range(10))\n seat=observation[\'seat\'];sizes=observation[\'hand_sizes\'];direction=observation.get(\'direction\',1)\n if play and card.get(\'kind\')==\'reverse\':direction=-direction\n finished=set(observation.get(\'finished_order\',[]))\n opponents=[(seat+i*direction)%len(sizes) for i in range(1,len(sizes)) if (seat+i*direction)%len(sizes) not in finished]\n for slot in range(3):\n  if slot>=len(opponents):vector.extend([0.]*6);continue\n  other=opponents[slot]\n  vector.extend([1.,sizes[other]/20.,float(sizes[other]==1),\n   beliefs[\'colours\'].get((other,selected),0.),beliefs[\'symbols\'].get((other,symbol(card)),0.),\n   float(beliefs[\'preference\'].get(other)==selected)])\n vector.extend([float(selected==observation.get(\'active_color\')),min(48,beliefs[\'penalty\'])/12.,\n  min(1000,observation.get(\'turn_number\',0))/1000.,len(opponents)/3.,\n  1./max(1,sizes[opponents[0]]) if opponents else 0.])\n return v2_features(observation,action,beliefs)+vector\n', 'uno_policy_features_v4.py': '"""Full own/public encoding plus exact retained hand-route feature values."""\nfrom uno_policy_features_v3 import FEATURE_NAMES as RICH_NAMES, action_features as rich_features\nfrom uno_route_policy import EXTRA_NAMES, action_features as route_features\n\nFEATURE_NAMES=RICH_NAMES+[\'route_\'+name for name in EXTRA_NAMES]\n\n\ndef action_features(observation,action,beliefs=None):\n    return rich_features(observation,action,beliefs)+route_features(observation,action,beliefs)[-len(EXTRA_NAMES):]\n', 'uno_mlp_policy.py': '"""Small non-language action scorer with a learned residual over the incumbent."""\nimport json\nimport math\nfrom pathlib import Path\nfrom uno_policy_v2 import FEATURE_NAMES, action_features, public_beliefs\n\n\nclass Policy:\n    def __init__(self, model):\n        self.feature_names = FEATURE_NAMES\n        self.action_features = action_features\n        if model.get(\'format\') == \'triad-uno-residual-mlp-v3\':\n            from uno_policy_features_v4 import FEATURE_NAMES as rich_names, action_features as rich_features\n            self.feature_names = rich_names\n            self.action_features = rich_features\n        elif model.get(\'format\') == \'triad-uno-residual-mlp-v2\':\n            from uno_policy_features_v3 import FEATURE_NAMES as rich_names, action_features as rich_features\n            self.feature_names = rich_names\n            self.action_features = rich_features\n        elif model.get(\'format\') != \'triad-uno-residual-mlp-v1\':\n            raise ValueError(\'incompatible model format\')\n        if model.get(\'features\') != self.feature_names:\n            raise ValueError(\'incompatible model format/features\')\n        self.model = model\n        self.base = model[\'base_weights\']\n        self.w1, self.b1 = model[\'hidden_weight\'], model[\'hidden_bias\']\n        self.w2, self.b2 = model[\'output_weight\'], model[\'output_bias\']\n        width = len(self.b1)\n        if len(self.base) != len(self.feature_names) or len(self.w1) != width or len(self.w2) != width or not 1 <= width <= 128:\n            raise ValueError(\'invalid model dimensions\')\n        if any(len(row) != len(self.feature_names) for row in self.w1):\n            raise ValueError(\'invalid hidden dimensions\')\n        values = self.base + self.b1 + self.w2 + [self.b2] + [x for row in self.w1 for x in row]\n        if not all(math.isfinite(float(x)) for x in values):\n            raise ValueError(\'nonfinite model parameter\')\n\n    def score_features(self, vector):\n        hidden = [max(0., sum(w*x for w,x in zip(row,vector)) + bias) for row,bias in zip(self.w1,self.b1)]\n        return sum(w*x for w,x in zip(self.base,vector)) + sum(w*x for w,x in zip(self.w2,hidden)) + self.b2\n\n    def select(self, observation):\n        legal = observation[\'legal_actions\']\n        if len(legal) == 1:\n            return legal[0][\'action_id\']\n        beliefs = public_beliefs(observation)\n        return max(legal,key=lambda action:self.score_features(self.action_features(observation,action,beliefs)))[\'action_id\']\n\n    def save(self, path):\n        Path(path).write_text(json.dumps(self.model,indent=2)+\'\\n\')\n\n\nclass FastPolicy(Policy):\n    """NumPy batch scoring for training evaluation only; never shipped."""\n    def __init__(self, model):\n        super().__init__(model)\n        import numpy as np\n        self.np = np\n        self.fast_base = np.asarray(self.base, dtype=float)\n        self.fast_w1 = np.asarray(self.w1, dtype=float)\n        self.fast_b1 = np.asarray(self.b1, dtype=float)\n        self.fast_w2 = np.asarray(self.w2, dtype=float)\n\n    def select(self, observation):\n        legal = observation[\'legal_actions\']\n        if len(legal) == 1:\n            return legal[0][\'action_id\']\n        beliefs = public_beliefs(observation)\n        x = self.np.asarray([self.action_features(observation,a,beliefs) for a in legal], dtype=float)\n        hidden = self.np.maximum(0., x @ self.fast_w1.T + self.fast_b1)\n        scores = x @ self.fast_base + hidden @ self.fast_w2 + self.b2\n        return legal[int(self.np.argmax(scores))][\'action_id\']\n', 'train_uno_league.py': '"""PSRO-inspired archive league with paired seat blocks and separate test seeds.\n\nThis is a practical population training method, not an exact equilibrium solver.\nAll policies act on filtered observations; simulator state only computes rewards.\n"""\nimport argparse\nimport concurrent.futures\nimport json\nimport multiprocessing\nimport os\nfrom pathlib import Path\nimport random\nimport sys\nimport time\n\nimport numpy as np\n\nROOT = Path(__file__).resolve().parent\nSDK = Path(os.environ.get("UNO_SDK", ROOT.parent / "unoSDK"))\nsys.path.insert(0, str(SDK / "local_evaluator" / "src"))\nfrom triad_uno import engine\nfrom triad_uno.baselines import HeuristicAgent, RandomAgent\nfrom uno_policy import Policy as V1, INITIAL_WEIGHTS\nfrom uno_policy_v2 import Policy as V2, FEATURE_NAMES\n\n\nclass Starter:\n    def select(self, observation):\n        legal = observation["legal_actions"]\n        return next((a["action_id"] for a in legal if a["type"] == "play"), legal[0]["action_id"])\n\n\ndef make_policy(spec, seed):\n    if spec["kind"] == "mlp_python":\n        from uno_mlp_policy import Policy\n        return Policy(spec["model"])\n    if spec["kind"] == "route":\n        from uno_route_policy import Policy as RoutePolicy\n        return RoutePolicy(spec["weights"])\n    if spec["kind"] == "mlp":\n        from uno_mlp_policy import FastPolicy\n        return FastPolicy(spec["model"])\n    if spec["kind"] == "v1":\n        return V1(spec["weights"])\n    if spec["kind"] == "v2":\n        return V2(spec["weights"])\n    if spec["kind"] == "random":\n        return RandomAgent(seed)\n    if spec["kind"] == "starter":\n        return Starter()\n    return HeuristicAgent(seed)\n\n\ndef score_game(seed, seat, candidate, rivals):\n    actors = [None] * 4\n    actors[seat] = make_policy(candidate, seed * 10 + 9)\n    for offset, spec in enumerate(rivals, 1):\n        actors[(seat + offset) % 4] = make_policy(spec, seed * 10 + offset)\n    state = engine.new_game(engine.GameConfig(max_turns=1000), seed)\n    while state.phase != engine.FINISHED and seat not in state.finished_set:\n        current = state.current_seat\n        observation = engine.observe(state, current)\n        selected = actors[current].select(observation)\n        action = engine.engine_action_from_id(state, current, selected)\n        if action is None:\n            raise RuntimeError("policy selected an illegal action")\n        if not engine.step(state, action).ok:\n            raise RuntimeError("policy transition failed")\n    if seat in state.finished_order:\n        placement = state.finished_order.index(seat)\n        return state.config.position_points[placement], float(placement == 0)\n    outcome = state.outcome\n    remaining = state.config.position_points[len(outcome.placements):]\n    return sum(remaining) / len(outcome.unfinished), 0.\n\n\ndef evaluate_job(job):\n    candidate, seeds, pool, mode = job\n    blocks, win_blocks = [], []\n    for seed in seeds:\n        rng = random.Random(seed ^ 0x4C4541475545)\n        if mode == "league":\n            rivals = [rng.choice(pool) for _ in range(3)]\n        else:\n            rivals = [{"kind": mode}] * 3\n        outcomes = [score_game(seed, seat, candidate, rivals) for seat in range(4)]\n        blocks.append(sum(row[0] for row in outcomes) / 4.)\n        win_blocks.append(sum(row[1] for row in outcomes) / 4.)\n    return {"points": float(np.mean(blocks)), "win_rate": float(np.mean(win_blocks)),\n            "games": 4 * len(seeds), "block_points": blocks, "block_wins": win_blocks}\n\n\ndef paired_interval(candidate, incumbent, seed=20261014):\n    differences = np.asarray(candidate["block_points"]) - np.asarray(incumbent["block_points"])\n    rng = np.random.default_rng(seed)\n    if len(differences) < 2:\n        return {"mean_gain": float(np.mean(differences)), "ci95": None}\n    means = differences[rng.integers(0, len(differences), (5000, len(differences)))].mean(axis=1)\n    return {"mean_gain": float(differences.mean()),\n            "ci95": [float(v) for v in np.quantile(means, [.025, .975])],\n            "unit": "points per game", "sampling_unit": "four-seat seed block"}\n\n\ndef summarize(result):\n    return {k: v for k, v in result.items() if not k.startswith("block_")}\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--incumbent", type=Path, default=SDK / "models" / "policy.json")\n    parser.add_argument("--iterations", type=int, default=40)\n    parser.add_argument("--population", type=int, default=48)\n    parser.add_argument("--train-blocks", type=int, default=16)\n    parser.add_argument("--validation-blocks", type=int, default=128)\n    parser.add_argument("--test-blocks", type=int, default=256)\n    parser.add_argument("--workers", type=int, default=min(4, os.cpu_count() or 1))\n    parser.add_argument("--seed", type=int, default=20261015)\n    parser.add_argument("--output", type=Path, default=ROOT / "uno-league-output")\n    args = parser.parse_args()\n    if min(args.iterations, args.population, args.train_blocks, args.validation_blocks, args.test_blocks, args.workers) < 1:\n        parser.error("counts must be positive")\n    args.output.mkdir(parents=True, exist_ok=True)\n    document = json.loads(args.incumbent.read_text())\n    incumbent = {"kind": "v1", "weights": document["weights"]}\n    mean = np.asarray(document["weights"] + [0.] * (len(FEATURE_NAMES) - len(document["weights"])))\n    best = mean.copy()\n    sigma = np.full(len(mean), 1.5)\n    rng = np.random.default_rng(args.seed)\n    attacking = list(INITIAL_WEIGHTS)\n    for i in (2, 4, 6, 10):\n        attacking[i] += 2.\n    preserve = list(INITIAL_WEIGHTS)\n    preserve[5] -= 2.\n    preserve[6] -= 2.\n    reference = [{"kind": "starter"}, {"kind": "heuristic"}, {"kind": "random"},\n                 incumbent, {"kind": "v1", "weights": INITIAL_WEIGHTS},\n                 {"kind": "v1", "weights": attacking}, {"kind": "v1", "weights": preserve}]\n    archive = list(reference)\n    selection_seeds = list(range(6_000_000, 6_000_000 + args.validation_blocks))\n    history, started = [], time.time()\n    context = multiprocessing.get_context("spawn")\n    with concurrent.futures.ProcessPoolExecutor(max_workers=args.workers, mp_context=context) as pool:\n        baseline = list(pool.map(evaluate_job, [(incumbent, selection_seeds, reference, "league")]))[0]\n        selected_score = baseline["points"]\n        for generation in range(args.iterations):\n            if generation and generation % 10 == 0:\n                mean, sigma = best.copy(), np.full(len(mean), 1.5)\n            population = rng.normal(mean, sigma, (args.population, len(mean)))\n            # Broad probes keep the population from collapsing into one local basin.\n            probes = max(1, args.population // 5)\n            population[-probes:] = rng.normal(best, 3., (probes, len(mean)))\n            population = np.clip(population, -20., 20.)\n            population[0] = mean\n            if args.population > 1:\n                population[1] = best\n            seeds = list(range(30_000 + generation * args.train_blocks, 30_000 + (generation + 1) * args.train_blocks))\n            scores = list(pool.map(evaluate_job,\n                [({"kind": "v2", "weights": w.tolist()}, seeds, archive, "league") for w in population]))\n            ranked = sorted(range(len(scores)), key=lambda i: (scores[i]["points"], scores[i]["win_rate"]), reverse=True)\n            elite = population[ranked[:max(2, args.population // 5)]]\n            mean = .35 * mean + .65 * elite.mean(axis=0)\n            sigma = np.maximum(.3, .35 * sigma + .65 * elite.std(axis=0))\n            candidate = population[ranked[0]]\n            spec = {"kind": "v2", "weights": candidate.tolist()}\n            validation = list(pool.map(evaluate_job, [(spec, selection_seeds, reference, "league")]))[0]\n            if validation["points"] > selected_score:\n                best, selected_score = candidate.copy(), validation["points"]\n            if generation % 4 == 3 and not any(s.get("kind") == "v2" and np.linalg.norm(np.asarray(s["weights"]) - candidate) < .2 for s in archive):\n                archive.append(spec)\n                archive = reference + archive[len(reference):][-12:]\n            record = {"generation": generation + 1, "train": summarize(scores[ranked[0]]),\n                      "validation": summarize(validation), "paired_validation_gain": paired_interval(validation, baseline),\n                      "selected_points": selected_score, "archive_size": len(archive),\n                      "elapsed_seconds": round(time.time() - started, 1)}\n            history.append(record)\n            print(json.dumps(record), flush=True)\n            V2(best).save(args.output / "uno-policy-v2.json", {"method": "archive-league cross-entropy policy optimization",\n                "seed": args.seed, "generations": generation + 1, "population": args.population,\n                "train_blocks": args.train_blocks, "validation_blocks": args.validation_blocks,\n                "selected_validation_points": selected_score, "incumbent_validation_points": baseline["points"],\n                "sdk_commit": "0cda3acc1bb5c05fa1803a8fbd7761770dc6d5fe"})\n            (args.output / "training-progress.json").write_text(json.dumps(history, indent=2) + "\\n")\n            (args.output / "archive.json").write_text(json.dumps(archive, indent=2) + "\\n")\n        test_seeds = list(range(7_000_000, 7_000_000 + args.test_blocks))\n        chosen = {"kind": "v2", "weights": best.tolist()}\n        modes = ("league", "starter", "heuristic", "random")\n        jobs = [(spec, test_seeds, reference, mode) for mode in modes for spec in (chosen, incumbent)]\n        results = list(pool.map(evaluate_job, jobs))\n        report = {mode: {"candidate": summarize(results[2*i]), "incumbent": summarize(results[2*i+1]),\n                        "paired_gain": paired_interval(results[2*i], results[2*i+1])}\n                  for i, mode in enumerate(modes)}\n        (args.output / "evaluation.json").write_text(json.dumps(report, indent=2) + "\\n")\n        print(json.dumps({"held_out_evaluation": report}), flush=True)\n\n\nif __name__ == "__main__":\n    main()\n', 'train_uno_joint_route.py': '"""Warm-start trust-bounded joint optimization of all24 own/public route weights.\n\nUNO\'s conventional model has no frozen-base requirement. Shared four-seat blocks\nreduce seat/deal variance; this practical search is not a proof of optimality.\n"""\nimport argparse\nimport concurrent.futures\nimport json\nimport multiprocessing\nfrom pathlib import Path\nimport time\nimport numpy as np\nfrom uno_policy import INITIAL_WEIGHTS\nfrom uno_route_policy import FEATURE_NAMES\nfrom uno_mlp_policy import Policy as NeuralPolicy\nfrom train_uno_league import evaluate_job,summarize,paired_interval\n\n\ndef main():\n    p=argparse.ArgumentParser()\n    p.add_argument(\'--incumbent\',type=Path,required=True)\n    p.add_argument(\'--v1\',type=Path,required=True)\n    p.add_argument(\'--neural-rival\',type=Path,action=\'append\',default=[])\n    p.add_argument(\'--iterations\',type=int,default=24)\n    p.add_argument(\'--population\',type=int,default=24)\n    p.add_argument(\'--train-blocks\',type=int,default=32)\n    p.add_argument(\'--selection-blocks\',type=int,default=256)\n    p.add_argument(\'--test-blocks\',type=int,default=512)\n    p.add_argument(\'--train-seed-start\',type=int,default=80_000_000)\n    p.add_argument(\'--selection-seed-start\',type=int,default=81_000_000)\n    p.add_argument(\'--test-seed-start\',type=int,default=82_000_000)\n    p.add_argument(\'--seed\',type=int,default=20261023)\n    p.add_argument(\'--workers\',type=int,default=4)\n    p.add_argument(\'--trust-fraction\',type=float,default=.25)\n    p.add_argument(\'--initial-sigma-fraction\',type=float,default=.10)\n    p.add_argument(\'--output\',type=Path,required=True)\n    a=p.parse_args()\n    if min(a.iterations,a.population,a.train_blocks,a.selection_blocks,a.test_blocks,a.workers)<1 or a.population<4:p.error(\'positive counts/population>=4 required\')\n    if not 0<a.initial_sigma_fraction<=a.trust_fraction<=.5:p.error(\'bounded positive trust configuration required\')\n    if not a.train_seed_start+a.iterations*a.train_blocks<=a.selection_seed_start or not a.selection_seed_start+a.selection_blocks<=a.test_seed_start:p.error(\'seed families overlap\')\n    a.output.mkdir(parents=True,exist_ok=True)\n    original=json.loads(a.incumbent.read_text())\n    if original[\'format\']!=\'triad-uno-route-linear-v1\' or original[\'features\']!=FEATURE_NAMES:raise ValueError(\'retained route schema required\')\n    center=np.asarray(original[\'weights\'],dtype=float)\n    if center.shape!=(24,) or not np.isfinite(center).all():raise ValueError(\'invalid incumbent\')\n    incumbent={\'kind\':\'route\',\'weights\':center.tolist()}\n    v1=json.loads(a.v1.read_text())\n    if v1[\'format\']!=\'triad-uno-linear-v1\' or len(v1[\'weights\'])!=18:raise ValueError(\'learnedv1 rival required\')\n    attack=list(INITIAL_WEIGHTS)\n    for i in (2,4,6,10):attack[i]+=2.\n    preserve=list(INITIAL_WEIGHTS);preserve[5]-=2.;preserve[6]-=2.\n    reference=[{\'kind\':\'starter\'},{\'kind\':\'heuristic\'},{\'kind\':\'random\'},incumbent,\n        {\'kind\':\'v1\',\'weights\':v1[\'weights\']},{\'kind\':\'v1\',\'weights\':INITIAL_WEIGHTS},\n        {\'kind\':\'v1\',\'weights\':attack},{\'kind\':\'v1\',\'weights\':preserve}]\n    rival_documents=[]\n    for path in a.neural_rival:\n        document=json.loads(path.read_text());NeuralPolicy(document)\n        rival_documents.append(document);reference.append({\'kind\':\'mlp\',\'model\':document})\n    (a.output/\'rival-pool.json\').write_text(json.dumps(reference,indent=2)+\'\\n\')\n    scale=np.maximum(1.,np.abs(center));radius=a.trust_fraction*scale\n    lower,upper=center-radius,center+radius\n    mean=center.copy();sigma=a.initial_sigma_fraction*scale\n    rng=np.random.default_rng(a.seed);finalists=[center.tolist()];history=[];started=time.time()\n    with concurrent.futures.ProcessPoolExecutor(max_workers=a.workers,mp_context=multiprocessing.get_context(\'spawn\')) as pool:\n        for generation in range(a.iterations):\n            population=np.clip(rng.normal(mean,sigma,(a.population,24)),lower,upper)\n            population[0]=mean;population[1]=center\n            specs=[{\'kind\':\'route\',\'weights\':row.tolist()} for row in population]\n            seeds=list(range(a.train_seed_start+generation*a.train_blocks,a.train_seed_start+(generation+1)*a.train_blocks))\n            results=list(pool.map(evaluate_job,[(s,seeds,reference,\'league\') for s in specs]))\n            order=sorted(range(len(results)),key=lambda i:results[i][\'points\'],reverse=True)\n            elite=population[order[:max(2,a.population//5)]]\n            mean=np.clip(.3*mean+.7*elite.mean(axis=0),lower,upper)\n            sigma=np.minimum(radius/2,np.maximum(.03*scale,.3*sigma+.7*elite.std(axis=0)))\n            best=population[order[0]]\n            if generation%3==2 and not any(np.linalg.norm((np.asarray(w)-best)/scale)<.12 for w in finalists):finalists.append(best.tolist())\n            row={\'generation\':generation+1,\'best_points\':summarize(results[order[0]]),\'incumbent_points\':summarize(results[1]),\n                 \'weights\':best.tolist(),\'normalized_distance\':float(np.linalg.norm((best-center)/scale)),\n                 \'seconds\':round(time.time()-started,1)}\n            history.append(row);print(json.dumps(row),flush=True)\n            (a.output/\'training-progress.json\').write_text(json.dumps(history,indent=2)+\'\\n\')\n        if not any(np.linalg.norm((np.asarray(w)-mean)/scale)<.12 for w in finalists):finalists.append(mean.tolist())\n        specs=[{\'kind\':\'route\',\'weights\':w} for w in finalists]\n        seeds=list(range(a.selection_seed_start,a.selection_seed_start+a.selection_blocks))\n        results=list(pool.map(evaluate_job,[(s,seeds,reference,\'league\') for s in specs]))\n        chosen=max(range(len(results)),key=lambda i:results[i][\'points\'])\n        selection={\'selected\':chosen,\'seed_start\':a.selection_seed_start,\'blocks\':a.selection_blocks,\'specs\':specs,\n            \'scores\':[summarize(r) for r in results],\'paired_gains\':[paired_interval(r,results[0]) for r in results[1:]]}\n        (a.output/\'selection.json\').write_text(json.dumps(selection,indent=2)+\'\\n\');print(json.dumps({\'selection\':selection}),flush=True)\n        document={\'format\':\'triad-uno-route-linear-v1\',\'features\':FEATURE_NAMES,\'weights\':specs[chosen][\'weights\'],\n            \'training\':{\'method\':\'warm-start joint24-coefficient trust-bounded CEM\',\'frozen_base\':False,\'seed\':a.seed,\n            \'iterations\':a.iterations,\'population\':a.population,\'train_blocks_per_generation\':a.train_blocks,\n            \'trust_fraction\':a.trust_fraction,\'initial_sigma_fraction\':a.initial_sigma_fraction,\n            \'lower_bounds\':lower.tolist(),\'upper_bounds\':upper.tolist(),\'train_seed_start\':a.train_seed_start,\n            \'selection_seed_start\':a.selection_seed_start,\'test_seed_start\':a.test_seed_start,\n            \'rival_pool_size\':len(reference),\'neural_rival_formats\':[m[\'format\'] for m in rival_documents],\n            \'base_checkpoint\':original,\'sdk_commit\':\'0cda3acc1bb5c05fa1803a8fbd7761770dc6d5fe\'}}\n        (a.output/\'selected-policy.json\').write_text(json.dumps(document,indent=2)+\'\\n\')\n        seeds=list(range(a.test_seed_start,a.test_seed_start+a.test_blocks));modes=(\'league\',\'starter\',\'heuristic\',\'random\')\n        results=list(pool.map(evaluate_job,[(s,seeds,reference,m) for m in modes for s in (specs[chosen],incumbent)]))\n        report={m:{\'candidate\':summarize(results[2*i]),\'incumbent\':summarize(results[2*i+1]),\'paired_gain\':paired_interval(results[2*i],results[2*i+1])} for i,m in enumerate(modes)}\n        report[\'provenance\']={\'fixed_selected_candidate\':chosen,\'train_seed_start\':a.train_seed_start,\n            \'selection_seed_start\':a.selection_seed_start,\'test_seed_start\':a.test_seed_start,\n            \'test_blocks\':a.test_blocks,\'rival_pool_size\':len(reference),\'joint_dimensions\':24,\n            \'trust_fraction\':a.trust_fraction,\'strength_warning\':\'New richer rival pool; compare paired incumbent within this gate only.\'}\n        (a.output/\'evaluation.json\').write_text(json.dumps(report,indent=2)+\'\\n\');print(json.dumps({\'final\':report}),flush=True)\n\n\nif __name__==\'__main__\':main()\n'}
for name,source in sources.items():(work/name).write_text(source)
models={'incumbent.json': '{\n  "format": "triad-uno-route-linear-v1",\n  "features": [\n    "play",\n    "number",\n    "skip",\n    "reverse",\n    "draw_two",\n    "wild",\n    "wild_draw_four",\n    "remaining_colour_count",\n    "remaining_colour_fraction",\n    "remaining_match_count",\n    "attack_next_threat",\n    "reverse_escape_threat",\n    "wild_when_small_hand",\n    "finish",\n    "keep",\n    "choose_colour",\n    "next_can_finish",\n    "own_hand_size",\n    "sequence_fraction",\n    "sequence_stranded",\n    "ending_wild",\n    "next_colour_absence",\n    "next_symbol_absence",\n    "next_colour_preference"\n  ],\n  "weights": [\n    8.022466349142684,\n    0.44153069585311683,\n    -0.06834035646341297,\n    1.4643534535685507,\n    -2.338619866848944,\n    -3.4540620544700498,\n    -4.276279368110987,\n    1.6602353386949849,\n    1.0165100023007814,\n    0.901341302040876,\n    3.853185081203389,\n    2.4436903583872347,\n    1.7223969671761579,\n    11.872419584625783,\n    -4.598739912392293,\n    3.347870880522802,\n    0.32778952616834095,\n    -0.24548152213113314,\n    0.054801310646275025,\n    -1.1349214815798592,\n    -0.04225874297028254,\n    1.1191018681573717,\n    0.21891410472226475,\n    -0.08122782541248305\n  ],\n  "training": {\n    "method": "frozen-base residual CEM, own-hand bounded sequence features",\n    "seed": 20261019,\n    "iterations": 24,\n    "sdk_commit": "0cda3acc1bb5c05fa1803a8fbd7761770dc6d5fe",\n    "selection_seed_start": 21000000,\n    "test_seed_start": 22000000,\n    "base": {\n      "format": "triad-uno-linear-v1",\n      "features": [\n        "play",\n        "number",\n        "skip",\n        "reverse",\n        "draw_two",\n        "wild",\n        "wild_draw_four",\n        "remaining_colour_count",\n        "remaining_colour_fraction",\n        "remaining_match_count",\n        "attack_next_threat",\n        "reverse_escape_threat",\n        "wild_when_small_hand",\n        "finish",\n        "keep",\n        "choose_colour",\n        "next_can_finish",\n        "own_hand_size"\n      ],\n      "weights": [\n        8.022466349142684,\n        0.44153069585311683,\n        -0.06834035646341297,\n        1.4643534535685507,\n        -2.338619866848944,\n        -3.4540620544700498,\n        -4.276279368110987,\n        1.6602353386949849,\n        1.0165100023007814,\n        0.901341302040876,\n        3.853185081203389,\n        2.4436903583872347,\n        1.7223969671761579,\n        11.872419584625783,\n        -4.598739912392293,\n        3.347870880522802,\n        0.32778952616834095,\n        -0.24548152213113314\n      ],\n      "training": {\n        "method": "cross-entropy policy optimization",\n        "seed": 20261014,\n        "generations": 30,\n        "population": 32,\n        "games_per_candidate": 64,\n        "validation": 3.005859375,\n        "sdk_commit": "0cda3acc1bb5c05fa1803a8fbd7761770dc6d5fe",\n        "feature_count": 18\n      }\n    }\n  }\n}\n', 'v1.json': '{\n  "format": "triad-uno-linear-v1",\n  "features": [\n    "play",\n    "number",\n    "skip",\n    "reverse",\n    "draw_two",\n    "wild",\n    "wild_draw_four",\n    "remaining_colour_count",\n    "remaining_colour_fraction",\n    "remaining_match_count",\n    "attack_next_threat",\n    "reverse_escape_threat",\n    "wild_when_small_hand",\n    "finish",\n    "keep",\n    "choose_colour",\n    "next_can_finish",\n    "own_hand_size"\n  ],\n  "weights": [\n    8.022466349142684,\n    0.44153069585311683,\n    -0.06834035646341297,\n    1.4643534535685507,\n    -2.338619866848944,\n    -3.4540620544700498,\n    -4.276279368110987,\n    1.6602353386949849,\n    1.0165100023007814,\n    0.901341302040876,\n    3.853185081203389,\n    2.4436903583872347,\n    1.7223969671761579,\n    11.872419584625783,\n    -4.598739912392293,\n    3.347870880522802,\n    0.32778952616834095,\n    -0.24548152213113314\n  ],\n  "training": {\n    "method": "cross-entropy policy optimization",\n    "seed": 20261014,\n    "generations": 30,\n    "population": 32,\n    "games_per_candidate": 64,\n    "validation": 3.005859375,\n    "sdk_commit": "0cda3acc1bb5c05fa1803a8fbd7761770dc6d5fe",\n    "feature_count": 18\n  }\n}\n', 'neural-route.json': '{\n  "format": "triad-uno-residual-mlp-v3",\n  "features": [\n    "play",\n    "number",\n    "skip",\n    "reverse",\n    "draw_two",\n    "wild",\n    "wild_draw_four",\n    "remaining_colour_count",\n    "remaining_colour_fraction",\n    "remaining_match_count",\n    "attack_next_threat",\n    "reverse_escape_threat",\n    "wild_when_small_hand",\n    "finish",\n    "keep",\n    "choose_colour",\n    "next_can_finish",\n    "own_hand_size",\n    "successor_colour_absence",\n    "successor_symbol_absence",\n    "successor_colour_preference",\n    "successor_finish_risk",\n    "remaining_colour_components",\n    "largest_component_fraction",\n    "remaining_bridge_symbols",\n    "remaining_duplicates",\n    "remaining_colour_entropy",\n    "remaining_wilds",\n    "chosen_colour_exhausted",\n    "stack_penalty_play",\n    "successor_one_card_colour_absence",\n    "successor_one_card_symbol_absence",\n    "remaining_colour_slot_0_number_0",\n    "remaining_colour_slot_0_number_1",\n    "remaining_colour_slot_0_number_2",\n    "remaining_colour_slot_0_number_3",\n    "remaining_colour_slot_0_number_4",\n    "remaining_colour_slot_0_number_5",\n    "remaining_colour_slot_0_number_6",\n    "remaining_colour_slot_0_number_7",\n    "remaining_colour_slot_0_number_8",\n    "remaining_colour_slot_0_number_9",\n    "remaining_colour_slot_0_skip_None",\n    "remaining_colour_slot_0_reverse_None",\n    "remaining_colour_slot_0_draw_two_None",\n    "remaining_colour_slot_1_number_0",\n    "remaining_colour_slot_1_number_1",\n    "remaining_colour_slot_1_number_2",\n    "remaining_colour_slot_1_number_3",\n    "remaining_colour_slot_1_number_4",\n    "remaining_colour_slot_1_number_5",\n    "remaining_colour_slot_1_number_6",\n    "remaining_colour_slot_1_number_7",\n    "remaining_colour_slot_1_number_8",\n    "remaining_colour_slot_1_number_9",\n    "remaining_colour_slot_1_skip_None",\n    "remaining_colour_slot_1_reverse_None",\n    "remaining_colour_slot_1_draw_two_None",\n    "remaining_colour_slot_2_number_0",\n    "remaining_colour_slot_2_number_1",\n    "remaining_colour_slot_2_number_2",\n    "remaining_colour_slot_2_number_3",\n    "remaining_colour_slot_2_number_4",\n    "remaining_colour_slot_2_number_5",\n    "remaining_colour_slot_2_number_6",\n    "remaining_colour_slot_2_number_7",\n    "remaining_colour_slot_2_number_8",\n    "remaining_colour_slot_2_number_9",\n    "remaining_colour_slot_2_skip_None",\n    "remaining_colour_slot_2_reverse_None",\n    "remaining_colour_slot_2_draw_two_None",\n    "remaining_colour_slot_3_number_0",\n    "remaining_colour_slot_3_number_1",\n    "remaining_colour_slot_3_number_2",\n    "remaining_colour_slot_3_number_3",\n    "remaining_colour_slot_3_number_4",\n    "remaining_colour_slot_3_number_5",\n    "remaining_colour_slot_3_number_6",\n    "remaining_colour_slot_3_number_7",\n    "remaining_colour_slot_3_number_8",\n    "remaining_colour_slot_3_number_9",\n    "remaining_colour_slot_3_skip_None",\n    "remaining_colour_slot_3_reverse_None",\n    "remaining_colour_slot_3_draw_two_None",\n    "remaining_wild_count",\n    "remaining_wild_draw_four_count",\n    "action_number_0",\n    "action_number_1",\n    "action_number_2",\n    "action_number_3",\n    "action_number_4",\n    "action_number_5",\n    "action_number_6",\n    "action_number_7",\n    "action_number_8",\n    "action_number_9",\n    "opponent_slot_0_present",\n    "opponent_slot_0_hand_size",\n    "opponent_slot_0_one_card",\n    "opponent_slot_0_colour_absence",\n    "opponent_slot_0_symbol_absence",\n    "opponent_slot_0_colour_preference",\n    "opponent_slot_1_present",\n    "opponent_slot_1_hand_size",\n    "opponent_slot_1_one_card",\n    "opponent_slot_1_colour_absence",\n    "opponent_slot_1_symbol_absence",\n    "opponent_slot_1_colour_preference",\n    "opponent_slot_2_present",\n    "opponent_slot_2_hand_size",\n    "opponent_slot_2_one_card",\n    "opponent_slot_2_colour_absence",\n    "opponent_slot_2_symbol_absence",\n    "opponent_slot_2_colour_preference",\n    "selected_is_active_colour",\n    "pending_penalty_size",\n    "turn_progress",\n    "active_opponents",\n    "closest_opponent_risk",\n    "route_sequence_fraction",\n    "route_sequence_stranded",\n    "route_ending_wild",\n    "route_next_colour_absence",\n    "route_next_symbol_absence",\n    "route_next_colour_preference"\n  ],\n  "base_weights": [\n    8.022466349142684,\n    0.44153069585311683,\n    -0.06834035646341297,\n    1.4643534535685507,\n    -2.338619866848944,\n    -3.4540620544700498,\n    -4.276279368110987,\n    1.6602353386949849,\n    1.0165100023007814,\n    0.901341302040876,\n    3.853185081203389,\n    2.4436903583872347,\n    1.7223969671761579,\n    11.872419584625783,\n    -4.598739912392293,\n    3.347870880522802,\n    0.32778952616834095,\n    -0.24548152213113314,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.054801310646275025,\n    -1.1349214815798592,\n    -0.04225874297028254,\n    1.1191018681573717,\n    0.21891410472226475,\n    -0.08122782541248305\n  ],\n  "hidden_weight": [\n    [\n      0.16806447505950928,\n      -0.0002771278377622366,\n      0.08349359035491943,\n      0.15573252737522125,\n      -0.02468879148364067,\n      0.024173129349946976,\n      0.07976287603378296,\n      0.04715687036514282,\n      -0.11508692800998688,\n      0.05999409034848213,\n      0.20493565499782562,\n      0.09545838832855225,\n      0.19186721742153168,\n      0.15676245093345642,\n      -0.32176804542541504,\n      0.36060652136802673,\n      0.0034882929176092148,\n      0.11662553250789642,\n      -0.007184733636677265,\n      -0.02520468272268772,\n      -0.01281052827835083,\n      0.1752278059720993,\n      -0.10889803618192673,\n      0.011094792746007442,\n      -0.15968294441699982,\n      -0.04106386378407478,\n      -0.012675321660935879,\n      0.18580622971057892,\n      0.19039160013198853,\n      -0.132821723818779,\n      0.10469574481248856,\n      0.11950840801000595,\n      -0.0762999877333641,\n      -0.03367253765463829,\n      0.028450222685933113,\n      -0.10635103285312653,\n      0.043166887015104294,\n      0.050144679844379425,\n      0.04359225928783417,\n      -0.010541150346398354,\n      0.022409293800592422,\n      0.0774681344628334,\n      -0.024106400087475777,\n      0.03410472348332405,\n      -0.08195032924413681,\n      -0.01956184022128582,\n      -0.14677651226520538,\n      -0.18651337921619415,\n      -0.008121087215840816,\n      -0.23100905120372772,\n      -0.1305890679359436,\n      -0.017291391268372536,\n      -0.08193543553352356,\n      -0.11030738800764084,\n      -0.08446598052978516,\n      -0.18361453711986542,\n      -0.11037103831768036,\n      -0.11969347298145294,\n      0.06133247911930084,\n      -0.0788983404636383,\n      -0.036432404071092606,\n      0.11686797440052032,\n      0.0796964168548584,\n      0.1469763070344925,\n      -0.09442929178476334,\n      0.03205254301428795,\n      -0.06644396483898163,\n      0.02279249206185341,\n      -0.09151890128850937,\n      -0.02665693871676922,\n      -0.019109776243567467,\n      0.025787802413105965,\n      -0.07098421454429626,\n      0.037629179656505585,\n      -0.024929748848080635,\n      -0.09418162703514099,\n      0.037495896220207214,\n      -0.0543118491768837,\n      0.002850709017366171,\n      0.12927360832691193,\n      -0.026499656960368156,\n      -0.05007689446210861,\n      -0.14416147768497467,\n      -0.08191069215536118,\n      0.07560717314481735,\n      0.06362698972225189,\n      0.02097816951572895,\n      -0.05725175142288208,\n      0.13808360695838928,\n      0.0728636309504509,\n      -0.037613674998283386,\n      -0.017880752682685852,\n      0.09065865725278854,\n      0.0931483581662178,\n      0.013504952192306519,\n      0.09619833528995514,\n      -0.04159603640437126,\n      0.08786627650260925,\n      0.011216887272894382,\n      -0.0009419134585186839,\n      -0.056174445897340775,\n      0.02517516352236271,\n      -0.018019359558820724,\n      0.0002186961064580828,\n      0.21498526632785797,\n      0.0059059965424239635,\n      0.04886431246995926,\n      0.03341732546687126,\n      0.10120555013418198,\n      -0.018292786553502083,\n      0.026323340833187103,\n      -0.024043407291173935,\n      0.05194380506873131,\n      -0.0014121680287644267,\n      -0.12795564532279968,\n      0.4518200159072876,\n      0.2908458113670349,\n      0.11023566871881485,\n      -0.04955320805311203,\n      0.027559693902730942,\n      -0.33870503306388855,\n      0.03504551947116852,\n      0.0761953815817833,\n      -0.20549224317073822,\n      -0.00410149060189724\n    ],\n    [\n      0.23095369338989258,\n      0.07172947376966476,\n      0.1545996218919754,\n      0.1272384375333786,\n      0.010807262733578682,\n      -0.028018129989504814,\n      0.027627641335129738,\n      -0.10124500840902328,\n      -0.06048431992530823,\n      0.16299894452095032,\n      0.11481120437383652,\n      0.04301382228732109,\n      0.259024441242218,\n      0.09524507075548172,\n      -0.3127494752407074,\n      -0.05581894889473915,\n      0.1229950487613678,\n      0.041428692638874054,\n      0.06846983730792999,\n      0.04583768546581268,\n      0.005055899266153574,\n      -0.07051275670528412,\n      -0.03146987035870552,\n      -0.016137361526489258,\n      -0.01184130646288395,\n      -0.09227550029754639,\n      -0.09209304302930832,\n      0.24539224803447723,\n      0.16845089197158813,\n      -0.2215597927570343,\n      0.04773968830704689,\n      0.03316421061754227,\n      -0.08350434899330139,\n      0.03094298020005226,\n      -0.07473354786634445,\n      0.006117207929491997,\n      -0.10047946870326996,\n      0.06501679867506027,\n      -0.0716545581817627,\n      -0.1109180897474289,\n      0.034311581403017044,\n      0.045247845351696014,\n      0.07746715098619461,\n      -0.07839033752679825,\n      0.09484845399856567,\n      -0.054063454270362854,\n      -0.14672468602657318,\n      -0.196406751871109,\n      0.04245283454656601,\n      -0.1426144242286682,\n      -0.15251541137695312,\n      0.029061980545520782,\n      -0.1314118355512619,\n      0.02804255485534668,\n      -0.18585743010044098,\n      -0.02819748967885971,\n      -0.026987560093402863,\n      -0.020824600011110306,\n      0.0001937700144480914,\n      -0.20288535952568054,\n      0.005458659026771784,\n      0.026079993695020676,\n      0.043251100927591324,\n      0.1170722246170044,\n      -0.07313553243875504,\n      0.017525356262922287,\n      -0.0023409679997712374,\n      0.05129198357462883,\n      -0.2719946503639221,\n      0.02736811153590679,\n      -0.09610714763402939,\n      0.10263010114431381,\n      -0.053112760186195374,\n      0.13478019833564758,\n      -0.05090675130486488,\n      -0.3347545266151428,\n      0.014948274940252304,\n      -0.2054525464773178,\n      0.03768335282802582,\n      0.20107389986515045,\n      -0.19650663435459137,\n      -0.011849800124764442,\n      -0.1540733426809311,\n      -0.2872813940048218,\n      0.08456302434206009,\n      0.12421347200870514,\n      0.0164715014398098,\n      0.015267028473317623,\n      0.1590701788663864,\n      0.05466979742050171,\n      0.12517483532428741,\n      0.15756848454475403,\n      -0.09480464458465576,\n      0.07466119527816772,\n      0.032378021627664566,\n      0.049812667071819305,\n      0.01653563417494297,\n      0.08962184935808182,\n      -0.018894901499152184,\n      0.04089931398630142,\n      0.08060532063245773,\n      0.016761908307671547,\n      0.008565045893192291,\n      0.0038606387097388506,\n      0.24337774515151978,\n      0.01657477580010891,\n      -0.05326678231358528,\n      0.02525980770587921,\n      0.04005898907780647,\n      -0.0031409733928740025,\n      0.16046841442584991,\n      -0.014713367447257042,\n      -0.05851360037922859,\n      -0.09716583788394928,\n      -0.14093871414661407,\n      0.2697015404701233,\n      0.20574256777763367,\n      -0.0632922351360321,\n      -0.028538186103105545,\n      0.06724363565444946,\n      -0.4129357933998108,\n      -0.023521116003394127,\n      -0.03019081801176071,\n      -0.10329774767160416,\n      -0.03680303320288658\n    ],\n    [\n      0.20588459074497223,\n      0.07874073088169098,\n      0.04795093461871147,\n      0.07026420533657074,\n      -0.07668771594762802,\n      -0.03354228287935257,\n      -0.01894475892186165,\n      0.08905625343322754,\n      0.10942845791578293,\n      0.009144138544797897,\n      0.09814393520355225,\n      0.20119166374206543,\n      0.17458385229110718,\n      0.03959403559565544,\n      -0.24473443627357483,\n      0.34077659249305725,\n      -0.015320458449423313,\n      0.12396533787250519,\n      0.08810121566057205,\n      0.14249305427074432,\n      6.446008774219081e-05,\n      0.05862252786755562,\n      0.08123425394296646,\n      0.02400204725563526,\n      0.027367211878299713,\n      0.07982464879751205,\n      0.1645439714193344,\n      -0.037469565868377686,\n      -0.045790523290634155,\n      -0.10442637652158737,\n      0.004369326401501894,\n      -0.10929664969444275,\n      -0.055283110588788986,\n      -0.010729353874921799,\n      0.005484352819621563,\n      0.04764334484934807,\n      0.04486459493637085,\n      0.02169143408536911,\n      0.0655311793088913,\n      0.056092314422130585,\n      -0.04532378539443016,\n      -0.07756871730089188,\n      -0.01294899731874466,\n      -0.059696588665246964,\n      0.0531875379383564,\n      -0.11613766103982925,\n      -0.019779520109295845,\n      0.10747198015451431,\n      -0.09756072610616684,\n      -0.1289493590593338,\n      -0.18622742593288422,\n      -0.08164746314287186,\n      -0.10771242529153824,\n      -0.11616217344999313,\n      -0.09626142680644989,\n      0.019461916759610176,\n      -0.037096329033374786,\n      0.06666546314954758,\n      0.12277819216251373,\n      -0.04222575202584267,\n      0.03580794483423233,\n      0.06361174583435059,\n      -0.0030440688133239746,\n      -0.03444105386734009,\n      -0.08000601828098297,\n      0.10087166726589203,\n      0.04312564432621002,\n      0.07806265354156494,\n      -0.0621589757502079,\n      -0.007259376812726259,\n      -0.021833598613739014,\n      0.0454515665769577,\n      -0.1031520739197731,\n      -0.043037060648202896,\n      0.07412321120500565,\n      -0.08135230839252472,\n      0.08866995573043823,\n      0.08709166198968887,\n      -0.024295708164572716,\n      0.011409539729356766,\n      -0.08427120745182037,\n      0.0776761993765831,\n      0.0811217650771141,\n      -0.002187897451221943,\n      0.02896806038916111,\n      -0.0613505020737648,\n      0.004477130249142647,\n      0.12378042936325073,\n      0.08498971909284592,\n      0.01307396125048399,\n      0.03882760554552078,\n      0.13654057681560516,\n      0.05895337089896202,\n      0.014168218709528446,\n      -0.010448755696415901,\n      0.05848434939980507,\n      0.0895053893327713,\n      0.03908933326601982,\n      0.07461201399564743,\n      -0.10218246281147003,\n      -0.05630696564912796,\n      0.030931226909160614,\n      0.0054880124516785145,\n      0.0696459412574768,\n      0.10298324376344681,\n      0.061759136617183685,\n      -0.09172069281339645,\n      -0.06320623308420181,\n      0.15413641929626465,\n      0.07204849272966385,\n      -0.018850009888410568,\n      -0.08478105813264847,\n      0.06890639662742615,\n      0.10588974505662918,\n      0.0621877945959568,\n      0.33474966883659363,\n      0.1526823490858078,\n      0.07873483747243881,\n      0.06633134931325912,\n      -0.05138604715466499,\n      -0.21673814952373505,\n      -0.0893917977809906,\n      -0.047633178532123566,\n      -0.20162524282932281,\n      -0.022997861728072166\n    ],\n    [\n      -0.0453643836081028,\n      -0.036422912031412125,\n      0.07136604189872742,\n      -0.031604815274477005,\n      0.0035612829960882664,\n      -0.04045634716749191,\n      0.0459413155913353,\n      -0.075450100004673,\n      0.0018178692553192377,\n      -0.029917148873209953,\n      -0.011579531244933605,\n      0.006277991458773613,\n      0.019201261922717094,\n      -0.04907497391104698,\n      -0.02398233860731125,\n      -0.0257291067391634,\n      -0.0516623854637146,\n      -0.06842562556266785,\n      -0.04202678054571152,\n      0.07605800032615662,\n      0.02709301933646202,\n      0.03847150877118111,\n      -0.030128734186291695,\n      -0.02349691092967987,\n      0.08323696255683899,\n      -0.06865354627370834,\n      0.029146715998649597,\n      -0.037193067371845245,\n      -0.0773448571562767,\n      0.03633375093340874,\n      0.08540022373199463,\n      -0.08001690357923508,\n      -0.006264897994697094,\n      0.009294887073338032,\n      0.045297037810087204,\n      0.00951372180134058,\n      -0.06852680444717407,\n      0.08568299561738968,\n      0.015253198333084583,\n      0.08239065855741501,\n      0.07928846031427383,\n      0.0605439655482769,\n      0.08877666294574738,\n      0.04698758199810982,\n      0.004216143861413002,\n      -0.06802906095981598,\n      0.030082950368523598,\n      0.02867390215396881,\n      0.05742057412862778,\n      -0.07767263054847717,\n      -0.0467410683631897,\n      -0.05315901339054108,\n      -0.030345724895596504,\n      -0.01397542655467987,\n      -0.04843725636601448,\n      -0.07551240175962448,\n      0.031290359795093536,\n      0.019451573491096497,\n      -0.00781787745654583,\n      -0.03131524845957756,\n      0.08935702592134476,\n      0.05769597366452217,\n      0.004293030593544245,\n      -0.0453270860016346,\n      0.03846842795610428,\n      -0.016413087025284767,\n      0.011129139922559261,\n      -0.08408375829458237,\n      -0.016917172819375992,\n      -0.008190528489649296,\n      -0.060856662690639496,\n      -0.06358729302883148,\n      0.016385694965720177,\n      -0.05031691864132881,\n      -0.03354337811470032,\n      0.04917058348655701,\n      0.019390542060136795,\n      0.023132266476750374,\n      -0.026975413784384727,\n      0.08397626131772995,\n      0.08508685976266861,\n      -0.052482880651950836,\n      -0.07366655766963959,\n      -0.08456338942050934,\n      -0.027076449245214462,\n      -0.006021358072757721,\n      -0.052228592336177826,\n      0.027168167755007744,\n      -0.01728055812418461,\n      -0.08807927370071411,\n      0.06708846241235733,\n      0.018537143245339394,\n      -0.0664244294166565,\n      0.024258088320493698,\n      -0.05039745196700096,\n      0.07621296495199203,\n      -0.05274263769388199,\n      -0.08096292614936829,\n      -0.050542887300252914,\n      -0.021540071815252304,\n      0.004872777499258518,\n      -0.010744365863502026,\n      -0.058872878551483154,\n      -0.0423729382455349,\n      -0.032423026859760284,\n      -0.03259672969579697,\n      -0.03958003968000412,\n      0.04507562145590782,\n      0.07117113471031189,\n      -0.004619641229510307,\n      0.07390619814395905,\n      -0.082786925137043,\n      0.08615970611572266,\n      0.059571802616119385,\n      -0.016443634405732155,\n      0.0564359650015831,\n      0.04751014709472656,\n      -0.07650446891784668,\n      0.08130958676338196,\n      -0.06049169972538948,\n      0.06888194382190704,\n      -0.03662838786840439,\n      -0.06038639694452286,\n      -0.03698574751615524,\n      0.020572906360030174\n    ],\n    [\n      -0.2118096947669983,\n      -0.11898039281368256,\n      -0.04180820286273956,\n      -0.23789511620998383,\n      0.009770151227712631,\n      0.043650127947330475,\n      -0.0419042631983757,\n      -0.11341378837823868,\n      -0.03360797464847565,\n      -0.038830921053886414,\n      -0.07911647111177444,\n      -0.017130916938185692,\n      -0.20259326696395874,\n      -0.06404983997344971,\n      0.33444613218307495,\n      -0.36099764704704285,\n      0.034302279353141785,\n      0.008008083328604698,\n      -0.09661217778921127,\n      -0.08192330598831177,\n      -0.016527026891708374,\n      0.012369560077786446,\n      0.0356278121471405,\n      0.06392855197191238,\n      0.02107328362762928,\n      -0.028859030455350876,\n      -0.08613520860671997,\n      0.18997983634471893,\n      0.18265067040920258,\n      0.13717104494571686,\n      -0.03262726217508316,\n      0.03988867625594139,\n      -0.05133364722132683,\n      0.039510179311037064,\n      0.056095510721206665,\n      0.008476157672703266,\n      -0.10118506103754044,\n      -0.011769931763410568,\n      0.05433836206793785,\n      -0.060585152357816696,\n      0.05504661053419113,\n      -0.02768535166978836,\n      0.03275277838110924,\n      0.014204015955328941,\n      -0.035254817456007004,\n      0.1247590184211731,\n      -0.032416652888059616,\n      -0.04528983682394028,\n      0.13277912139892578,\n      -0.08309874683618546,\n      -0.04516661912202835,\n      0.09343479573726654,\n      -0.07014918327331543,\n      0.07771176844835281,\n      0.005278690252453089,\n      0.07102042436599731,\n      0.032969482243061066,\n      -0.0755646675825119,\n      -0.03569350764155388,\n      -0.04638245701789856,\n      -0.05249342694878578,\n      0.07888799905776978,\n      0.011667120270431042,\n      0.10100338608026505,\n      -0.048373714089393616,\n      -0.006148753222078085,\n      0.05035857856273651,\n      -0.0371951088309288,\n      -0.04797147214412689,\n      0.04568098112940788,\n      -0.04784085601568222,\n      0.01640460267663002,\n      0.022784685716032982,\n      0.15059587359428406,\n      -0.04660338908433914,\n      -0.03735485300421715,\n      -0.03412024676799774,\n      0.0787864550948143,\n      0.15610149502754211,\n      0.09770815074443817,\n      -0.04406755045056343,\n      -0.06408318877220154,\n      -0.08325811475515366,\n      0.049939900636672974,\n      -0.03366609662771225,\n      0.3541172444820404,\n      -0.004805170930922031,\n      -0.03467010334134102,\n      -0.10615591704845428,\n      0.017861686646938324,\n      -0.06347683072090149,\n      0.024651402607560158,\n      0.015553505159914494,\n      -0.030000176280736923,\n      -0.01192252803593874,\n      -0.09515442699193954,\n      0.18993553519248962,\n      0.19256241619586945,\n      -0.07371947914361954,\n      0.055274538695812225,\n      0.06461300700902939,\n      0.025557557120919228,\n      0.05999118834733963,\n      0.005364180542528629,\n      0.23916688561439514,\n      0.09310460835695267,\n      -0.05661623924970627,\n      0.149330735206604,\n      0.13452646136283875,\n      0.008152933791279793,\n      0.049862198531627655,\n      0.1440461128950119,\n      0.06947927922010422,\n      -0.06954764574766159,\n      0.14836299419403076,\n      -0.29227471351623535,\n      0.008920739404857159,\n      0.18469500541687012,\n      0.17632773518562317,\n      0.23288850486278534,\n      0.06818057596683502,\n      0.08875074982643127,\n      -0.07303869724273682,\n      0.04018630459904671,\n      0.018172496929764748\n    ],\n    [\n      0.07758940011262894,\n      0.10570928454399109,\n      0.12413282692432404,\n      0.22595755755901337,\n      -0.07000448554754257,\n      0.06395799666643143,\n      -0.08411362022161484,\n      -0.0646037757396698,\n      -0.017526401206851006,\n      0.14552615582942963,\n      0.16671514511108398,\n      0.0334382988512516,\n      0.18694497644901276,\n      0.12424115836620331,\n      -0.3882192075252533,\n      0.193893164396286,\n      0.1136469841003418,\n      0.14009720087051392,\n      0.07121003419160843,\n      0.13498616218566895,\n      0.02319362759590149,\n      0.0029663885943591595,\n      -0.16250570118427277,\n      0.0182794202119112,\n      -0.20494796335697174,\n      -0.05161348357796669,\n      -0.040099676698446274,\n      0.4079068899154663,\n      0.033476341515779495,\n      -0.3189343810081482,\n      0.03847215324640274,\n      -0.07318001985549927,\n      -0.0323953777551651,\n      0.03872912749648094,\n      -0.10093691945075989,\n      -0.08646039664745331,\n      0.04438454285264015,\n      0.10212240368127823,\n      -0.15581168234348297,\n      -0.005325739737600088,\n      0.03510112687945366,\n      0.004456263035535812,\n      0.03339331969618797,\n      -0.04201129451394081,\n      -0.024497590959072113,\n      -0.04982299357652664,\n      -0.30370691418647766,\n      -0.02474367991089821,\n      -0.11637740582227707,\n      -0.06836138665676117,\n      -0.056696418672800064,\n      -0.058091625571250916,\n      -0.1291409581899643,\n      -0.09628312289714813,\n      -0.07342513650655746,\n      -0.11323054879903793,\n      -0.07717838883399963,\n      -0.026165073737502098,\n      0.04618016257882118,\n      -0.26298341155052185,\n      -0.0030397691298276186,\n      -0.0311539676040411,\n      0.03762955963611603,\n      0.0649736300110817,\n      0.06504122167825699,\n      -0.08923202008008957,\n      0.018455807119607925,\n      0.032426927238702774,\n      -0.13681207597255707,\n      0.10694008320569992,\n      -0.0909004881978035,\n      0.04463588446378708,\n      -0.019680824130773544,\n      0.04933658987283707,\n      -0.03371753171086311,\n      -0.3149917423725128,\n      0.03189076855778694,\n      -0.15349647402763367,\n      -0.0477902777493,\n      0.2069394737482071,\n      -0.09581951797008514,\n      -0.05841982737183571,\n      0.02746685966849327,\n      -0.041302938014268875,\n      0.22841842472553253,\n      0.25285604596138,\n      0.0015111141838133335,\n      0.04868912696838379,\n      0.11652113497257233,\n      0.12297762930393219,\n      0.07662297040224075,\n      -0.05980686470866203,\n      0.023665539920330048,\n      -0.004268856253474951,\n      0.0810413658618927,\n      0.048735205084085464,\n      -0.012582316994667053,\n      -0.01010280754417181,\n      0.04047729820013046,\n      0.04642641544342041,\n      0.04798118397593498,\n      -0.10793808847665787,\n      0.04773254320025444,\n      -0.10850746184587479,\n      0.1073773056268692,\n      -0.08123287558555603,\n      0.04495697095990181,\n      0.07413008064031601,\n      0.06927210092544556,\n      0.1257106065750122,\n      0.08390866965055466,\n      0.021361270919442177,\n      0.07194901257753372,\n      -0.12635554373264313,\n      0.006251429673284292,\n      0.229326069355011,\n      0.15083211660385132,\n      0.0240608062595129,\n      -0.05062231421470642,\n      0.033266831189394,\n      -0.35667330026626587,\n      0.007960187271237373,\n      -0.04511474445462227,\n      -0.08522477746009827,\n      0.05305023491382599\n    ],\n    [\n      -0.14449438452720642,\n      -0.09842535853385925,\n      -0.04956786334514618,\n      -0.15491357445716858,\n      -0.015356821939349174,\n      -0.1002039909362793,\n      -0.045302774757146835,\n      0.014326413162052631,\n      0.0650455579161644,\n      -0.05105754733085632,\n      -0.16675007343292236,\n      -0.11250077933073044,\n      -0.22963377833366394,\n      -0.08620582520961761,\n      0.3791557848453522,\n      -0.41126370429992676,\n      0.028207426890730858,\n      0.018356407061219215,\n      0.025339631363749504,\n      -0.00970238633453846,\n      -0.07254710048437119,\n      -0.029208436608314514,\n      0.01991472765803337,\n      0.049358855932950974,\n      -0.05389919504523277,\n      0.07365044951438904,\n      -0.09131071716547012,\n      0.3576439619064331,\n      0.16144736111164093,\n      0.10474101454019547,\n      0.016980230808258057,\n      0.1245647668838501,\n      -0.05448555946350098,\n      0.04833628609776497,\n      0.045181289315223694,\n      -0.04413413628935814,\n      0.04502491280436516,\n      -0.09977280348539352,\n      -0.03188829869031906,\n      0.03508268669247627,\n      0.034084346145391464,\n      0.05788607895374298,\n      -0.05654362216591835,\n      0.00810312945395708,\n      -0.09107540547847748,\n      0.09338901937007904,\n      -0.006663666572421789,\n      0.038480162620544434,\n      0.10867923498153687,\n      -0.039267390966415405,\n      0.10764313489198685,\n      0.06473302841186523,\n      0.058012839406728745,\n      0.12074050307273865,\n      0.1168024018406868,\n      0.013990206643939018,\n      -0.014833438210189342,\n      -0.04365631937980652,\n      0.04487686976790428,\n      -0.013114591129124165,\n      0.059698887169361115,\n      0.1447288542985916,\n      0.08088790625333786,\n      0.12218373268842697,\n      -0.05294128879904747,\n      -0.07886074483394623,\n      0.07127600163221359,\n      -0.0014038588851690292,\n      0.014009260572493076,\n      -0.052469998598098755,\n      -0.02158760465681553,\n      0.059913162142038345,\n      -0.05467288941144943,\n      0.04615109786391258,\n      -0.07548435032367706,\n      0.004851463250815868,\n      0.022015433758497238,\n      -0.056996751576662064,\n      0.06642472743988037,\n      0.002088526263833046,\n      -0.09896430373191833,\n      -0.0066069429740309715,\n      -0.044244565069675446,\n      0.07881367206573486,\n      -0.061100270599126816,\n      0.2747340202331543,\n      -0.07042208313941956,\n      0.01680084504187107,\n      -0.015872489660978317,\n      -0.09003050625324249,\n      0.04344533383846283,\n      -0.04375632107257843,\n      -0.015519344247877598,\n      0.06258504837751389,\n      -0.0626225546002388,\n      0.010601755231618881,\n      0.08630964159965515,\n      0.11982801556587219,\n      0.06119956448674202,\n      0.019480101764202118,\n      0.03710651397705078,\n      -0.03325382247567177,\n      0.1394294649362564,\n      -0.021810995414853096,\n      0.07219116389751434,\n      0.03612259402871132,\n      0.003242599545046687,\n      0.06828977912664413,\n      0.08379346132278442,\n      0.12779802083969116,\n      0.1924094706773758,\n      0.1406477838754654,\n      0.025555085390806198,\n      0.03454877808690071,\n      0.12264475226402283,\n      -0.23455138504505157,\n      -0.09435269981622696,\n      0.03372979536652565,\n      0.1567399948835373,\n      0.17877422273159027,\n      0.04935380816459656,\n      0.1216956377029419,\n      -0.04116398096084595,\n      0.06500619649887085,\n      0.08222317695617676\n    ],\n    [\n      -0.1152712032198906,\n      0.017152588814496994,\n      -0.02635842189192772,\n      -0.15174321830272675,\n      -0.02295863628387451,\n      -0.03654574230313301,\n      -0.0635739341378212,\n      0.007511505391448736,\n      -0.058059196919202805,\n      -0.05591374263167381,\n      -0.04356133192777634,\n      -0.18537050485610962,\n      -0.22522474825382233,\n      -0.11800696700811386,\n      0.21687068045139313,\n      -0.37607261538505554,\n      0.0733715072274208,\n      0.05319443717598915,\n      -0.10463449358940125,\n      -0.08023883402347565,\n      0.04358513280749321,\n      -0.04197191819548607,\n      0.05052868276834488,\n      -0.028571588918566704,\n      -0.07275430113077164,\n      -0.004740400705486536,\n      -0.1092660054564476,\n      0.16391830146312714,\n      0.1286945939064026,\n      0.16517110168933868,\n      -0.0564441941678524,\n      -0.005296587012708187,\n      -0.03863015025854111,\n      -0.04198125749826431,\n      0.05817414075136185,\n      -0.0057235294952988625,\n      -0.03596623241901398,\n      -0.09095021337270737,\n      0.009088801220059395,\n      0.025408446788787842,\n      -0.005307609215378761,\n      0.09130661934614182,\n      -0.03323208540678024,\n      0.04316867142915726,\n      0.04111215099692345,\n      0.06073019281029701,\n      0.0808132067322731,\n      -0.04206978529691696,\n      0.12481565773487091,\n      0.09650415927171707,\n      0.042012736201286316,\n      0.12077094614505768,\n      0.08830676972866058,\n      0.026800716295838356,\n      0.13836117088794708,\n      0.09714391082525253,\n      0.023025745525956154,\n      0.08018100261688232,\n      0.06913053244352341,\n      0.05007104203104973,\n      0.008582495152950287,\n      0.06476345658302307,\n      -0.003006452927365899,\n      0.014653367921710014,\n      0.08090472221374512,\n      -0.010089405812323093,\n      0.08141279965639114,\n      -0.07544152438640594,\n      0.0313410647213459,\n      0.08828415721654892,\n      -0.12342902272939682,\n      0.09914308786392212,\n      -0.03748674690723419,\n      0.08604445308446884,\n      0.08364903181791306,\n      -0.050287824124097824,\n      -0.03477067872881889,\n      -0.0757921114563942,\n      0.05639544501900673,\n      0.035588547587394714,\n      -0.047674454748630524,\n      -0.0038342317566275597,\n      -0.06331194192171097,\n      0.03245439752936363,\n      -0.07771427929401398,\n      0.3513181507587433,\n      -0.02549310401082039,\n      -0.025300268083810806,\n      -0.07177362591028214,\n      0.004983398597687483,\n      -0.06415945291519165,\n      -0.006154044531285763,\n      -0.08250179141759872,\n      -0.07762625068426132,\n      -0.044107820838689804,\n      -0.06382513791322708,\n      0.16485822200775146,\n      0.08477447926998138,\n      -0.10698701441287994,\n      -0.05055486038327217,\n      -0.047632355242967606,\n      0.1214657798409462,\n      0.0029934023041278124,\n      0.06184053048491478,\n      0.2064771056175232,\n      0.06964165717363358,\n      0.031938306987285614,\n      0.05151398852467537,\n      0.17137950658798218,\n      0.0778937116265297,\n      0.133637934923172,\n      0.06613946706056595,\n      -0.004906190559267998,\n      0.03479618951678276,\n      0.04239484667778015,\n      -0.2830977737903595,\n      -0.08195511251688004,\n      0.07759396731853485,\n      0.10850217193365097,\n      0.11011677235364914,\n      0.11808602511882782,\n      0.017596011981368065,\n      -0.05054861307144165,\n      0.02314738556742668,\n      0.06544984132051468\n    ],\n    [\n      0.0848957970738411,\n      0.004416593350470066,\n      0.08347271382808685,\n      0.22083942592144012,\n      0.01770828291773796,\n      0.044914186000823975,\n      0.09323084354400635,\n      0.013339202851057053,\n      0.0594891719520092,\n      0.1426173597574234,\n      0.13479840755462646,\n      0.1320989429950714,\n      0.2964393198490143,\n      0.014732565730810165,\n      -0.24051998555660248,\n      0.370012104511261,\n      0.036557771265506744,\n      0.11827652901411057,\n      0.032710663974285126,\n      0.14064911007881165,\n      0.07807989418506622,\n      0.12966331839561462,\n      0.2020092010498047,\n      0.1952914148569107,\n      0.050204187631607056,\n      -0.12269002944231033,\n      0.08609209954738617,\n      -0.02837318740785122,\n      -0.12192341685295105,\n      -0.1393381953239441,\n      0.04122396185994148,\n      0.013441548682749271,\n      0.08630581945180893,\n      -0.026859432458877563,\n      0.010454772971570492,\n      -0.01737028732895851,\n      0.037473682314157486,\n      -0.0005868764710612595,\n      -0.03836795315146446,\n      0.08964495360851288,\n      0.008178852498531342,\n      0.0322459377348423,\n      0.1034836545586586,\n      -0.03671276196837425,\n      -0.006099549587816,\n      0.006455754395574331,\n      -0.07563701272010803,\n      0.08354049921035767,\n      -0.01843586564064026,\n      -0.09902295470237732,\n      -0.1182638481259346,\n      0.013728971593081951,\n      0.051160458475351334,\n      -0.011851362884044647,\n      -0.08522383123636246,\n      -0.20480941236019135,\n      0.034300610423088074,\n      -0.010461792349815369,\n      0.08793047070503235,\n      0.026800084859132767,\n      0.06283298134803772,\n      -0.04245954751968384,\n      -0.039001911878585815,\n      0.0596592016518116,\n      0.03374024108052254,\n      0.10804122686386108,\n      -0.03209185600280762,\n      0.15315288305282593,\n      0.10726649314165115,\n      -0.06565199047327042,\n      0.05642225965857506,\n      -0.07023956626653671,\n      -0.029159175232052803,\n      -0.04940764605998993,\n      0.06723880767822266,\n      0.004275929182767868,\n      0.1365550458431244,\n      -0.003956692758947611,\n      0.022172603756189346,\n      -0.10020903497934341,\n      0.005029628984630108,\n      0.017590023577213287,\n      0.11150564253330231,\n      -0.033768802881240845,\n      0.1582721769809723,\n      -0.19207808375358582,\n      0.052120376378297806,\n      0.03244110941886902,\n      0.064390167593956,\n      0.11341793835163116,\n      -0.07461871951818466,\n      0.034982021898031235,\n      0.07863986492156982,\n      0.08205488324165344,\n      -0.015208568423986435,\n      0.061232808977365494,\n      -0.009525398723781109,\n      0.05278662592172623,\n      0.11816123127937317,\n      0.04269043356180191,\n      -0.06550316512584686,\n      0.026850849390029907,\n      0.03806448355317116,\n      0.03455683961510658,\n      0.06503114104270935,\n      0.0753503143787384,\n      -0.0759466215968132,\n      -0.04470326378941536,\n      0.11857941001653671,\n      0.110346719622612,\n      -0.08866927027702332,\n      -0.07527032494544983,\n      0.0490679107606411,\n      0.0016977336490526795,\n      0.09163442999124527,\n      0.35314372181892395,\n      0.2527070641517639,\n      0.06411971151828766,\n      -0.11698993295431137,\n      -0.05051979050040245,\n      -0.033181242644786835,\n      0.05494542047381401,\n      -0.034249886870384216,\n      -0.1348055899143219,\n      0.02695976197719574\n    ],\n    [\n      -0.08521121740341187,\n      -0.05876652151346207,\n      -0.04833236336708069,\n      -0.09817450493574142,\n      -0.05408339574933052,\n      0.03241851180791855,\n      0.023723054677248,\n      -0.03783949464559555,\n      -0.06512226909399033,\n      0.03277392312884331,\n      -0.10935693979263306,\n      -0.15840336680412292,\n      -0.11723561584949493,\n      -0.029696868732571602,\n      0.273043155670166,\n      -0.2855108380317688,\n      0.07485821098089218,\n      0.04342566803097725,\n      -0.07629410922527313,\n      -0.10566381365060806,\n      -0.007371614687144756,\n      0.012157396413385868,\n      -0.03945694491267204,\n      0.017750434577465057,\n      -0.06877602636814117,\n      0.04747911915183067,\n      -0.08824633061885834,\n      0.2385474145412445,\n      0.017526086419820786,\n      0.21453039348125458,\n      0.009715487249195576,\n      0.07028552889823914,\n      0.04354781657457352,\n      0.008509811013936996,\n      0.03422179073095322,\n      -0.018648555502295494,\n      0.0020785436499863863,\n      -0.10983271896839142,\n      -0.02102217637002468,\n      -0.022123266011476517,\n      -0.09507286548614502,\n      0.08646246045827866,\n      -0.008292884565889835,\n      -0.037311214953660965,\n      0.001597633701749146,\n      0.018453339114785194,\n      -0.08629225194454193,\n      -0.0073284790851175785,\n      0.11317988485097885,\n      0.020890289917588234,\n      -0.024542879313230515,\n      0.1597643941640854,\n      0.091791532933712,\n      0.11048673838376999,\n      0.0032742423936724663,\n      0.07889091223478317,\n      -0.05131635442376137,\n      -0.029765548184514046,\n      -0.02130466140806675,\n      -0.03462757170200348,\n      0.005575686227530241,\n      0.13369165360927582,\n      0.009379422292113304,\n      0.10005029290914536,\n      -0.052845075726509094,\n      -0.08300086110830307,\n      -0.014265557751059532,\n      -0.1079818606376648,\n      -0.025901494547724724,\n      0.044181324541568756,\n      -0.010045099072158337,\n      0.0525260791182518,\n      -0.03384425491094589,\n      0.18067798018455505,\n      0.006716839969158173,\n      0.09342748671770096,\n      0.042388685047626495,\n      -0.0018222321523353457,\n      0.12868867814540863,\n      -0.02311924286186695,\n      -0.07842841744422913,\n      0.01798935793340206,\n      0.049260664731264114,\n      -0.07842458784580231,\n      -0.09690216183662415,\n      0.27141377329826355,\n      0.03544953092932701,\n      -0.02736547216773033,\n      0.029954025521874428,\n      -0.08655255287885666,\n      0.00297355093061924,\n      -0.06771760433912277,\n      -0.07112613320350647,\n      -0.05788068845868111,\n      -0.06211934983730316,\n      -0.04689253121614456,\n      0.07582976669073105,\n      0.07581718266010284,\n      -0.029012177139520645,\n      0.07868332415819168,\n      0.07483802735805511,\n      -0.01643393188714981,\n      0.15465722978115082,\n      0.03957149758934975,\n      0.0390804298222065,\n      0.06887181103229523,\n      0.08755666762590408,\n      0.12870988249778748,\n      0.1265939176082611,\n      0.1334735006093979,\n      0.1134379506111145,\n      0.0502791665494442,\n      0.01042885985225439,\n      0.0141314547508955,\n      0.02289443463087082,\n      -0.34795138239860535,\n      -0.06357059627771378,\n      0.09671089053153992,\n      0.12897267937660217,\n      0.12187241017818451,\n      -0.013553382828831673,\n      0.10600849986076355,\n      -0.008266030810773373,\n      0.001950594480149448,\n      0.055752936750650406\n    ],\n    [\n      0.18746018409729004,\n      0.06590479612350464,\n      0.12431766092777252,\n      0.07845568656921387,\n      -0.07770060002803802,\n      0.10391847789287567,\n      0.0319356694817543,\n      0.03794334456324577,\n      0.001725616049952805,\n      0.13775627315044403,\n      0.21249984204769135,\n      0.10757943242788315,\n      0.2961170971393585,\n      0.004192117601633072,\n      -0.4550824761390686,\n      0.32110849022865295,\n      -0.027322035282850266,\n      0.11175324022769928,\n      -0.031127609312534332,\n      0.12386071681976318,\n      -0.01014936063438654,\n      0.05426269769668579,\n      -0.1449885368347168,\n      -0.023105140775442123,\n      -0.10171344876289368,\n      -0.10532619059085846,\n      -0.09175263345241547,\n      0.2021602839231491,\n      0.09600968658924103,\n      -0.11384925991296768,\n      0.05703752860426903,\n      -0.1265021115541458,\n      -0.09122151136398315,\n      0.007316929288208485,\n      -0.09983508288860321,\n      0.033781107515096664,\n      0.06416422873735428,\n      0.10863844305276871,\n      -0.04100522771477699,\n      0.08962251245975494,\n      0.06288255006074905,\n      -0.07439160346984863,\n      -0.03720537945628166,\n      -0.10169506818056107,\n      -0.027830401435494423,\n      -0.06859580427408218,\n      -0.23801098763942719,\n      -0.18635492026805878,\n      0.01153513789176941,\n      -0.1283293515443802,\n      -0.11444524675607681,\n      -0.0492250956594944,\n      -0.15613636374473572,\n      -0.13467071950435638,\n      -0.031951405107975006,\n      -0.015772439539432526,\n      -0.19204971194267273,\n      -0.18224568665027618,\n      -0.0390661284327507,\n      0.0015947256470099092,\n      -0.12415359914302826,\n      0.04152446985244751,\n      -0.031816404312849045,\n      0.09151976555585861,\n      -0.07306891679763794,\n      -0.17504294216632843,\n      -0.03822418674826622,\n      -0.05309354513883591,\n      -0.1373273879289627,\n      -0.06469342112541199,\n      0.0008094630320556462,\n      -0.037988074123859406,\n      -0.04314284026622772,\n      0.10062868893146515,\n      0.012821922078728676,\n      -0.03852171450853348,\n      -0.11828587204217911,\n      0.05029803887009621,\n      -0.05840877816081047,\n      0.05572926253080368,\n      -0.1527383029460907,\n      0.03380544111132622,\n      -0.026936333626508713,\n      -0.01072321180254221,\n      0.16504906117916107,\n      0.11506729573011398,\n      0.07669907063245773,\n      0.10471396893262863,\n      -0.004677253309637308,\n      0.05011307820677757,\n      0.12294924259185791,\n      0.12384975701570511,\n      -0.0005914235371164978,\n      0.02503257617354393,\n      0.03458870202302933,\n      0.10417919605970383,\n      0.04399644583463669,\n      0.07085372507572174,\n      0.0468076728284359,\n      0.06424015015363693,\n      0.011073319241404533,\n      -0.016658267006278038,\n      0.10053690522909164,\n      -0.009793616831302643,\n      0.1766285002231598,\n      -0.006922543980181217,\n      0.04232839494943619,\n      0.07526160776615143,\n      0.020627936348319054,\n      -0.027084941044449806,\n      0.06403560936450958,\n      -0.001595530309714377,\n      0.07994315028190613,\n      -0.05088544264435768,\n      0.03999943658709526,\n      0.3973560929298401,\n      0.2030484825372696,\n      0.014647901058197021,\n      -0.04490380734205246,\n      0.003081247676163912,\n      -0.2991620898246765,\n      0.023124128580093384,\n      -0.04632479324936867,\n      -0.09659691154956818,\n      0.009052110835909843\n    ],\n    [\n      -0.16143378615379333,\n      -0.07920555025339127,\n      -0.11204809695482254,\n      -0.12042909860610962,\n      -0.08071939647197723,\n      -0.050935667008161545,\n      -0.08279586583375931,\n      -0.08107850700616837,\n      -0.06927766650915146,\n      -0.04298330470919609,\n      -0.1268628090620041,\n      -0.15004225075244904,\n      -0.23429425060749054,\n      -0.14185094833374023,\n      0.2590298354625702,\n      -0.38415369391441345,\n      -0.02129644714295864,\n      -0.08808595687150955,\n      0.06080356985330582,\n      -0.11163605004549026,\n      0.05555100739002228,\n      -0.07463487982749939,\n      -0.09703778475522995,\n      0.020145706832408905,\n      0.04912174120545387,\n      0.10595708340406418,\n      -0.15850499272346497,\n      0.27883562445640564,\n      0.18832027912139893,\n      0.16765986382961273,\n      0.11020941287279129,\n      0.0047891526482999325,\n      -0.04447786509990692,\n      0.09444022923707962,\n      0.0684664323925972,\n      -0.07684920728206635,\n      0.050721779465675354,\n      -0.009605564177036285,\n      -0.06420169025659561,\n      0.012414256110787392,\n      -0.008712004870176315,\n      0.06195076182484627,\n      0.047487299889326096,\n      0.040030285716056824,\n      0.0037390298675745726,\n      0.008653712458908558,\n      -0.02444298192858696,\n      -0.06087536737322807,\n      0.13604731857776642,\n      0.040972914546728134,\n      -0.008636570535600185,\n      0.16823318600654602,\n      0.03176570311188698,\n      0.01894916221499443,\n      0.10014218091964722,\n      0.09424682706594467,\n      0.05311494320631027,\n      -0.01747928000986576,\n      0.03159651905298233,\n      -0.019265087321400642,\n      -0.06107785925269127,\n      0.07127951830625534,\n      0.12173667550086975,\n      0.040466081351041794,\n      -0.03671715781092644,\n      0.04036495462059975,\n      -0.0209918525069952,\n      -0.10967455059289932,\n      0.018367767333984375,\n      0.1024920716881752,\n      0.0038403423968702555,\n      -0.05889773368835449,\n      0.022433724254369736,\n      0.14768296480178833,\n      0.05405993387103081,\n      0.0982171967625618,\n      -0.019926732406020164,\n      0.051308974623680115,\n      0.08310022205114365,\n      0.01349932886660099,\n      0.02166306972503662,\n      -0.061886947602033615,\n      -0.03829655423760414,\n      0.07956764101982117,\n      0.05649006366729736,\n      0.31159502267837524,\n      -0.059868451207876205,\n      -0.018874147906899452,\n      -0.0026784709189087152,\n      -0.07337235659360886,\n      -0.0855504721403122,\n      -0.06929485499858856,\n      0.004758162889629602,\n      -0.019955962896347046,\n      -0.010417956858873367,\n      -0.03732965886592865,\n      0.12135057896375656,\n      0.07741986215114594,\n      -0.05161462724208832,\n      0.015002429485321045,\n      -0.07745279371738434,\n      0.09021763503551483,\n      0.1304408758878708,\n      0.08676688373088837,\n      0.22829511761665344,\n      0.02831573598086834,\n      0.03224604204297066,\n      0.07440853863954544,\n      0.16008378565311432,\n      0.10456285625696182,\n      0.12944984436035156,\n      0.08404719084501266,\n      -0.013556019403040409,\n      0.06745276600122452,\n      0.0786629319190979,\n      -0.35339245200157166,\n      0.022064095363020897,\n      0.2006484419107437,\n      0.09017632901668549,\n      0.30588433146476746,\n      0.11280103027820587,\n      -0.0453038364648819,\n      0.0348389595746994,\n      0.10050039738416672,\n      -0.054530780762434006\n    ],\n    [\n      0.22143574059009552,\n      -0.0012250093277543783,\n      0.15980832278728485,\n      0.1768733710050583,\n      0.06756476312875748,\n      0.0015632420545443892,\n      0.0569000169634819,\n      -0.07477828115224838,\n      -0.06630302965641022,\n      0.03655489906668663,\n      0.06188857927918434,\n      0.169242724776268,\n      0.2198476344347,\n      0.020978551357984543,\n      -0.30599313974380493,\n      0.2850716710090637,\n      0.028203554451465607,\n      0.06271099299192429,\n      0.0794137716293335,\n      0.08720958977937698,\n      -0.00456949882209301,\n      0.032871063798666,\n      -0.028378674760460854,\n      0.1494242399930954,\n      -0.09582112729549408,\n      -0.10392006486654282,\n      -0.07866097241640091,\n      0.2427644580602646,\n      0.04071992263197899,\n      -0.10131898522377014,\n      0.02877449244260788,\n      0.021698366850614548,\n      -0.0483817420899868,\n      -0.019750483334064484,\n      -0.12528784573078156,\n      -0.07946210354566574,\n      -0.10797618329524994,\n      0.02642093226313591,\n      -0.016649460420012474,\n      -0.0824432522058487,\n      -0.06742101162672043,\n      0.07357838749885559,\n      0.028220070526003838,\n      -0.044961296021938324,\n      0.024007484316825867,\n      -0.07700753957033157,\n      -0.08820580691099167,\n      -0.13648903369903564,\n      0.017070122063159943,\n      -0.09831305593252182,\n      -0.046233467757701874,\n      -0.025080999359488487,\n      -0.07272347807884216,\n      -0.06328587979078293,\n      0.02051282674074173,\n      -0.08507328480482101,\n      -0.22057537734508514,\n      -0.06883012503385544,\n      0.017086682841181755,\n      0.012556501664221287,\n      -0.07374036312103271,\n      0.04125891998410225,\n      0.009753396734595299,\n      0.1074499860405922,\n      -0.06253622472286224,\n      -0.10901068896055222,\n      0.050750430673360825,\n      0.006160434801131487,\n      -0.15175685286521912,\n      -0.04919111728668213,\n      -0.08453045785427094,\n      -0.07608111202716827,\n      -0.006900161504745483,\n      0.017643185332417488,\n      0.04286589473485947,\n      -0.011709365993738174,\n      0.05130980536341667,\n      -0.001034729415550828,\n      0.026260269805788994,\n      -0.008383085019886494,\n      -0.05642935261130333,\n      -0.005222793202847242,\n      -0.11334999650716782,\n      -0.006865506526082754,\n      0.1806076318025589,\n      0.01770792156457901,\n      0.057186227291822433,\n      0.03949866071343422,\n      0.07531529664993286,\n      -0.026667047291994095,\n      0.05357242003083229,\n      0.12112070620059967,\n      -0.007203006185591221,\n      -0.05481961742043495,\n      0.0950784757733345,\n      0.0971342921257019,\n      -0.009852513670921326,\n      0.02586500719189644,\n      -0.0006301607354544103,\n      -0.04943695291876793,\n      0.07019254565238953,\n      0.030689196661114693,\n      0.031854163855314255,\n      0.0767940804362297,\n      0.14657729864120483,\n      0.015110527165234089,\n      -0.10150271654129028,\n      0.07868745923042297,\n      0.10064689069986343,\n      0.033955011516809464,\n      0.055690374225378036,\n      -0.011354784481227398,\n      0.04896225035190582,\n      0.032361775636672974,\n      -0.08594193309545517,\n      0.47581538558006287,\n      0.21028640866279602,\n      0.002187088830396533,\n      -0.07545357942581177,\n      -0.08007233589887619,\n      -0.24468305706977844,\n      -0.10312769562005997,\n      0.0484013557434082,\n      -0.06887747347354889,\n      0.047233060002326965\n    ],\n    [\n      0.1768103688955307,\n      0.1017075777053833,\n      0.15654610097408295,\n      0.24128812551498413,\n      -0.018380170688033104,\n      0.01694655604660511,\n      -0.051752109080553055,\n      0.07198695093393326,\n      -0.032368920743465424,\n      0.09882823377847672,\n      0.22493691742420197,\n      0.12854315340518951,\n      0.21640311181545258,\n      0.03192530572414398,\n      -0.39533352851867676,\n      0.3111604154109955,\n      -0.010607614181935787,\n      0.11101865023374557,\n      0.09065897017717361,\n      0.09283652156591415,\n      -0.022295724600553513,\n      0.07957406342029572,\n      -0.0423249788582325,\n      0.03395986557006836,\n      -0.10753072798252106,\n      -0.0430479422211647,\n      0.051934536546468735,\n      0.175761416554451,\n      -0.01635129190981388,\n      -0.05427226796746254,\n      -0.010686391033232212,\n      -0.1016765832901001,\n      0.05436741188168526,\n      0.02752053365111351,\n      -0.09094571322202682,\n      -0.029701881110668182,\n      -0.01849491335451603,\n      -0.03762222081422806,\n      0.045352283865213394,\n      -0.019636793062090874,\n      0.0690091997385025,\n      -0.08513551950454712,\n      -0.011066537350416183,\n      -0.03823227435350418,\n      0.06922304630279541,\n      0.040465135127305984,\n      -0.048161763697862625,\n      -0.1331108957529068,\n      0.00042690764530561864,\n      -0.097845159471035,\n      -0.16537350416183472,\n      -0.04082176089286804,\n      -0.18065442144870758,\n      -0.09396319836378098,\n      -0.06304880231618881,\n      -0.1733420491218567,\n      -0.08950189501047134,\n      -0.12583068013191223,\n      0.09750702977180481,\n      -0.04657910391688347,\n      -0.06906281411647797,\n      0.0668499618768692,\n      0.03239387646317482,\n      0.03439769893884659,\n      0.0014793543377891183,\n      -0.11946761608123779,\n      -0.09629672020673752,\n      -0.051449161022901535,\n      -0.006660817191004753,\n      -0.12698890268802643,\n      -0.07141435891389847,\n      -0.020131031051278114,\n      -0.051998816430568695,\n      0.032728008925914764,\n      -0.04096169024705887,\n      -0.09396585077047348,\n      0.02044103294610977,\n      -0.03745748847723007,\n      -0.06755705177783966,\n      0.14152522385120392,\n      -0.044131115078926086,\n      -0.11051902174949646,\n      0.020446447655558586,\n      -0.12138994038105011,\n      0.07757923007011414,\n      -0.058854591101408005,\n      -0.04167371615767479,\n      0.06139924377202988,\n      0.0526447631418705,\n      -0.017099209129810333,\n      0.1197710931301117,\n      0.02284763753414154,\n      0.011218249797821045,\n      0.08279208093881607,\n      0.09601898491382599,\n      0.021818241104483604,\n      0.029697701334953308,\n      0.043528832495212555,\n      0.06044600158929825,\n      -0.0014688154915347695,\n      0.07087114453315735,\n      0.001107257092371583,\n      0.07172335684299469,\n      0.039513688534498215,\n      0.09119445830583572,\n      0.03303324431180954,\n      0.0060025108978152275,\n      0.04902038350701332,\n      0.11149882525205612,\n      -0.011193628422915936,\n      -0.008925638161599636,\n      -0.024204831570386887,\n      -0.03059201128780842,\n      -0.05791132152080536,\n      0.041212815791368484,\n      0.39765164256095886,\n      0.11735162138938904,\n      0.044786542654037476,\n      -0.018252361565828323,\n      0.04513368010520935,\n      -0.2451598346233368,\n      -0.046569496393203735,\n      -0.05337105691432953,\n      -0.06255890429019928,\n      0.0619284026324749\n    ],\n    [\n      -0.07839637249708176,\n      -0.7096160054206848,\n      -0.4180433750152588,\n      -0.42910778522491455,\n      -0.5760513544082642,\n      0.41835105419158936,\n      0.30970555543899536,\n      -0.5402432680130005,\n      -0.49785542488098145,\n      -0.26932600140571594,\n      0.05024373531341553,\n      0.4543835520744324,\n      0.20386101305484772,\n      0.009219911880791187,\n      0.3839534819126129,\n      -0.37174537777900696,\n      0.0810026004910469,\n      0.02217561937868595,\n      -0.27139556407928467,\n      0.035797540098428726,\n      -0.047788407653570175,\n      0.06763448566198349,\n      -0.1776876300573349,\n      0.05298061668872833,\n      0.06040274724364281,\n      -0.2580634653568268,\n      0.03372443839907646,\n      0.5760690569877625,\n      0.3712124228477478,\n      0.5671447515487671,\n      0.21328827738761902,\n      -0.15888477861881256,\n      -0.09878012537956238,\n      -0.25976964831352234,\n      -0.005787086207419634,\n      -0.48916345834732056,\n      -0.23833529651165009,\n      -0.03826913610100746,\n      -0.2182004302740097,\n      -0.09647994488477707,\n      -0.5000563263893127,\n      -0.17268048226833344,\n      -0.4724736511707306,\n      -0.08271879702806473,\n      -0.24949294328689575,\n      -0.15047724545001984,\n      -0.3436239957809448,\n      0.17572495341300964,\n      -0.08171103149652481,\n      -0.0005109104095026851,\n      0.07825927436351776,\n      0.22826582193374634,\n      -0.32484170794487,\n      0.1942983716726303,\n      0.2025476098060608,\n      -0.05687639117240906,\n      -0.13085415959358215,\n      -0.13844376802444458,\n      0.36529985070228577,\n      0.0860835462808609,\n      -0.01744109019637108,\n      0.2030772715806961,\n      -0.00894368626177311,\n      0.11475256830453873,\n      -0.08001353591680527,\n      -0.004573220387101173,\n      0.09895166754722595,\n      0.1592080444097519,\n      0.14124086499214172,\n      0.14487388730049133,\n      0.013604036532342434,\n      0.41453272104263306,\n      0.3950599730014801,\n      0.14358651638031006,\n      0.15125250816345215,\n      -0.2942279875278473,\n      -0.05649271979928017,\n      0.43659141659736633,\n      0.3314729928970337,\n      0.30122077465057373,\n      0.02201463095843792,\n      -0.13829904794692993,\n      0.19434714317321777,\n      -0.14639133214950562,\n      0.6138895750045776,\n      0.17670251429080963,\n      -0.33546414971351624,\n      -0.2919114828109741,\n      -0.20612896978855133,\n      -0.18142880499362946,\n      -0.4156620502471924,\n      -0.3945072293281555,\n      -0.23159345984458923,\n      -0.598149836063385,\n      -0.41944485902786255,\n      -0.4479048550128937,\n      0.2595687210559845,\n      -0.00039690747507847846,\n      0.181723490357399,\n      -0.12589439749717712,\n      0.0413183718919754,\n      0.03939501568675041,\n      0.02451479248702526,\n      -0.2333696335554123,\n      0.4341403841972351,\n      -0.1289370208978653,\n      -0.000915766810066998,\n      0.17930975556373596,\n      -0.15236002206802368,\n      0.28042852878570557,\n      -0.12130334228277206,\n      -0.134823739528656,\n      -0.3520033061504364,\n      -0.3168429434299469,\n      -0.5052125453948975,\n      0.010387022979557514,\n      -0.20103231072425842,\n      0.07315821945667267,\n      0.2924658954143524,\n      0.29530298709869385,\n      0.17772330343723297,\n      -0.5246614217758179,\n      -0.25555339455604553,\n      0.13380633294582367,\n      0.04873591661453247\n    ],\n    [\n      0.23328690230846405,\n      0.1230340227484703,\n      0.06135432794690132,\n      0.2216751128435135,\n      -0.06385605037212372,\n      0.12041213363409042,\n      -0.07872752100229263,\n      0.006694605108350515,\n      -0.10091475397348404,\n      0.06658364087343216,\n      0.13047662377357483,\n      0.09477424621582031,\n      0.2503191828727722,\n      0.06544195115566254,\n      -0.3716225326061249,\n      0.24575629830360413,\n      -0.055231526494026184,\n      0.20375952124595642,\n      -0.02340717613697052,\n      0.0953146442770958,\n      -0.04509430006146431,\n      0.0036284998059272766,\n      -0.0818910077214241,\n      0.016287432983517647,\n      -0.08295618742704391,\n      -0.028372719883918762,\n      0.0019337909761816263,\n      0.2838445007801056,\n      0.1660914421081543,\n      -0.09542129188776016,\n      0.051429759711027145,\n      -0.08956114947795868,\n      -0.11441074311733246,\n      -0.07787380367517471,\n      -0.03645262494683266,\n      -0.012425058521330357,\n      0.017143165692687035,\n      0.0161597840487957,\n      0.05151014029979706,\n      -0.10305006802082062,\n      -0.10198351740837097,\n      0.0743277445435524,\n      -0.07303882390260696,\n      -0.13035550713539124,\n      0.10153691470623016,\n      0.015469134785234928,\n      -0.15185992419719696,\n      -0.10614936798810959,\n      0.028546422719955444,\n      -0.14219841361045837,\n      -0.02376837655901909,\n      -0.009477316401898861,\n      -0.09563016891479492,\n      -0.101248599588871,\n      0.017269212752580643,\n      -0.010799696668982506,\n      -0.08051790297031403,\n      -0.1060706377029419,\n      0.05652397871017456,\n      -0.05286094546318054,\n      -0.022020254284143448,\n      -0.0034875033888965845,\n      -0.011543097905814648,\n      0.019776897504925728,\n      -0.011561618186533451,\n      -0.07344568520784378,\n      -0.01661616563796997,\n      -0.06884085386991501,\n      -0.15852537751197815,\n      -0.13318054378032684,\n      -0.15441055595874786,\n      -0.031020259484648705,\n      -0.08466967195272446,\n      0.034776583313941956,\n      -0.03516165539622307,\n      -0.05368158593773842,\n      -0.0960133969783783,\n      -0.0815555527806282,\n      0.02850937843322754,\n      0.034179478883743286,\n      -0.17660996317863464,\n      -0.09733228385448456,\n      0.007442833390086889,\n      -0.10203130543231964,\n      0.22266162931919098,\n      0.2340669482946396,\n      0.06349120289087296,\n      -0.034030403941869736,\n      0.004450574982911348,\n      -0.0011981092393398285,\n      0.02748517133295536,\n      0.10841209441423416,\n      -0.059627581387758255,\n      0.007975094951689243,\n      0.11284470558166504,\n      0.008041656576097012,\n      -0.009816235862672329,\n      -0.0056684878654778,\n      0.04749976098537445,\n      0.003627980127930641,\n      0.11640588194131851,\n      0.043790433555841446,\n      -0.018689600750803947,\n      -0.08769608289003372,\n      0.15636686980724335,\n      -0.05561775714159012,\n      -0.03709649294614792,\n      0.07536134868860245,\n      0.03767957538366318,\n      -0.04962208867073059,\n      0.07611965388059616,\n      0.09786131232976913,\n      -0.07599812000989914,\n      -0.029921885579824448,\n      0.03831026330590248,\n      0.39781996607780457,\n      0.03059530444443226,\n      0.06155083328485489,\n      -0.17145396769046783,\n      0.014107045717537403,\n      -0.24365636706352234,\n      -0.041671186685562134,\n      0.05901755020022392,\n      -0.16973988711833954,\n      -0.05911645293235779\n    ],\n    [\n      -0.09318094700574875,\n      -0.11792536824941635,\n      -0.09600183367729187,\n      -0.20198485255241394,\n      -0.03527823090553284,\n      0.04171732813119888,\n      0.040313638746738434,\n      -0.013934655115008354,\n      -0.07909741252660751,\n      0.0038922526873648167,\n      -0.17964386940002441,\n      -0.19052371382713318,\n      -0.2101619988679886,\n      -0.14785335958003998,\n      0.3810519874095917,\n      -0.4341103434562683,\n      0.025591330602765083,\n      -0.01344421785324812,\n      -0.036734532564878464,\n      0.031079165637493134,\n      -0.05904335156083107,\n      0.07266603410243988,\n      0.04890357702970505,\n      -0.016321104019880295,\n      -0.07889322191476822,\n      0.13461226224899292,\n      -0.12010164558887482,\n      0.259464830160141,\n      0.13042353093624115,\n      0.12488474696874619,\n      0.004520115442574024,\n      0.09473799914121628,\n      -0.069667749106884,\n      -0.05770033970475197,\n      -0.032581355422735214,\n      0.01524445042014122,\n      -0.1007879227399826,\n      -0.1349942982196808,\n      -0.05644379183650017,\n      -0.12624794244766235,\n      0.049595851451158524,\n      -0.047047972679138184,\n      -0.12767180800437927,\n      0.07397487014532089,\n      -0.09440279752016068,\n      -0.007127772551029921,\n      -0.014286900870501995,\n      0.04323025047779083,\n      0.19103500247001648,\n      -0.05907038226723671,\n      0.015077348798513412,\n      0.16859011352062225,\n      -0.00932697206735611,\n      -0.03518253564834595,\n      0.1307182013988495,\n      0.09114403277635574,\n      0.06086927279829979,\n      -0.020481234416365623,\n      -0.004814448766410351,\n      0.04167325049638748,\n      -0.040841761976480484,\n      -0.013770844787359238,\n      0.04637778922915459,\n      0.1358984112739563,\n      0.04161820188164711,\n      -0.0878661498427391,\n      0.09197752177715302,\n      -0.03842451050877571,\n      0.08231236785650253,\n      -0.007121466100215912,\n      0.00642992602661252,\n      -0.029033079743385315,\n      -0.10654962062835693,\n      0.030183302238583565,\n      -0.012401833198964596,\n      -0.04652241989970207,\n      0.0029597384855151176,\n      -0.04150180518627167,\n      0.03166830912232399,\n      0.1666465848684311,\n      0.013841867446899414,\n      0.08175758272409439,\n      0.016925012692809105,\n      -0.04487459734082222,\n      -0.05500708520412445,\n      0.27561601996421814,\n      0.01883518323302269,\n      -0.0003034829569514841,\n      -0.13293364644050598,\n      -0.03667924925684929,\n      -0.029194125905632973,\n      -0.01935993693768978,\n      -0.059256382286548615,\n      -0.06933705508708954,\n      -0.10588563978672028,\n      -0.04317522048950195,\n      0.17329028248786926,\n      0.06271795183420181,\n      -0.04955316334962845,\n      -0.039079077541828156,\n      -0.0047000618651509285,\n      0.04379468038678169,\n      0.1411771923303604,\n      0.0999545156955719,\n      0.0835670754313469,\n      0.07960870862007141,\n      -0.017549267038702965,\n      0.04567236453294754,\n      0.07853424549102783,\n      -0.006811174564063549,\n      0.054629113525152206,\n      0.15299193561077118,\n      0.02589631825685501,\n      0.0727931410074234,\n      0.049439266324043274,\n      -0.25537335872650146,\n      0.03253914788365364,\n      0.15605835616588593,\n      0.10245229303836823,\n      0.2427484691143036,\n      0.10674425959587097,\n      0.07661059498786926,\n      -0.03769050911068916,\n      -0.001994114136323333,\n      -0.054858773946762085\n    ],\n    [\n      -0.12599512934684753,\n      -0.09104866534471512,\n      -0.0470680296421051,\n      -0.07370291650295258,\n      -0.038725677877664566,\n      -0.010422389954328537,\n      -0.020953651517629623,\n      0.06425031274557114,\n      -0.01831260323524475,\n      -0.08228748291730881,\n      -0.1389569789171219,\n      -0.09312725067138672,\n      -0.12656138837337494,\n      -0.10333399474620819,\n      0.3492935001850128,\n      -0.40836283564567566,\n      0.00112472556065768,\n      0.10797536373138428,\n      0.03362344205379486,\n      -0.014126935973763466,\n      -0.03766218572854996,\n      -0.10188567638397217,\n      -0.07134827971458435,\n      -0.014065939001739025,\n      -0.014585547149181366,\n      0.1390884816646576,\n      -0.17867475748062134,\n      0.21542847156524658,\n      0.17202267050743103,\n      0.27201953530311584,\n      0.0011092799250036478,\n      0.02850501239299774,\n      -0.052225612103939056,\n      0.02129419706761837,\n      -0.06956880539655685,\n      -0.02035020850598812,\n      0.08142972737550735,\n      0.029364503920078278,\n      0.06279563903808594,\n      -0.0063866726122796535,\n      -0.02925756759941578,\n      0.08401390165090561,\n      0.02544230967760086,\n      0.027731338515877724,\n      0.04022304341197014,\n      -0.033970002084970474,\n      0.07738132029771805,\n      -0.11333854496479034,\n      0.20148849487304688,\n      0.05081120878458023,\n      0.0990167111158371,\n      0.10121006518602371,\n      -0.05072300136089325,\n      -0.04518121853470802,\n      0.028180036693811417,\n      0.030715756118297577,\n      0.060694627463817596,\n      -0.10377634316682816,\n      0.011179725639522076,\n      -0.0471835695207119,\n      0.04735114052891731,\n      0.059698063880205154,\n      0.07313136756420135,\n      0.029403764754533768,\n      0.08452276140451431,\n      -0.04550617188215256,\n      -0.02855510078370571,\n      0.027467086911201477,\n      -0.03530784696340561,\n      -0.012215468101203442,\n      0.009714281186461449,\n      -0.01891561597585678,\n      -0.014886853285133839,\n      0.048551540821790695,\n      -0.031649474054574966,\n      0.05526440218091011,\n      0.05883665755391121,\n      -0.03360197693109512,\n      0.12716680765151978,\n      0.10728377848863602,\n      -0.03596540912985802,\n      -0.09943870455026627,\n      -0.10821928083896637,\n      -0.03287947177886963,\n      0.10915359109640121,\n      0.28071504831314087,\n      -0.016041703522205353,\n      0.02130633220076561,\n      -0.07003870606422424,\n      0.05135186389088631,\n      -0.08377186954021454,\n      -0.09391272813081741,\n      -0.053498364984989166,\n      0.04484618827700615,\n      0.06776789575815201,\n      0.026015227660536766,\n      0.08395865559577942,\n      0.15469105541706085,\n      -0.08988512307405472,\n      -0.04529913514852524,\n      -0.061436645686626434,\n      -0.04208097234368324,\n      -0.004328124225139618,\n      0.07002740353345871,\n      0.05756733566522598,\n      -0.05285113304853439,\n      0.12076923996210098,\n      0.12185589224100113,\n      0.057805389165878296,\n      0.0663837268948555,\n      -0.021299446001648903,\n      0.04016183689236641,\n      -0.022262685000896454,\n      0.06011323630809784,\n      -0.014625008217990398,\n      -0.3923712968826294,\n      -0.10045088082551956,\n      0.11491071432828903,\n      0.22201679646968842,\n      0.1853424310684204,\n      0.014785443432629108,\n      0.06157525256276131,\n      0.04013190418481827,\n      0.14520502090454102,\n      -0.0818699449300766\n    ],\n    [\n      -0.040934234857559204,\n      -0.10534128546714783,\n      -0.07747188955545425,\n      -0.22594048082828522,\n      0.017538152635097504,\n      -0.08263048529624939,\n      0.014510652981698513,\n      0.07083823531866074,\n      -0.06812664866447449,\n      0.1500309705734253,\n      -0.15024138987064362,\n      -0.16703158617019653,\n      -0.09615010768175125,\n      0.013460210524499416,\n      0.3464507460594177,\n      -0.37411192059516907,\n      0.13340188562870026,\n      -0.03736164793372154,\n      -0.045125752687454224,\n      -0.11051862686872482,\n      0.04417869821190834,\n      -0.03820454701781273,\n      -0.09569904953241348,\n      -0.00481893215328455,\n      -0.11356388032436371,\n      0.0539877824485302,\n      -0.16525255143642426,\n      0.32769694924354553,\n      0.19406285881996155,\n      0.23733632266521454,\n      0.08306536823511124,\n      0.0303487591445446,\n      -0.029803767800331116,\n      0.003235533135011792,\n      0.06158006191253662,\n      0.05068683996796608,\n      -0.009457683190703392,\n      0.018021902069449425,\n      0.0062403203919529915,\n      0.01637248694896698,\n      0.014146358706057072,\n      0.06701604276895523,\n      0.08563052117824554,\n      0.08147469907999039,\n      0.02756652981042862,\n      0.020025867968797684,\n      -0.10637732595205307,\n      0.003151077078655362,\n      0.17601695656776428,\n      0.003986310679465532,\n      0.04149758070707321,\n      0.10129512846469879,\n      0.07996677607297897,\n      0.03632083162665367,\n      0.14768502116203308,\n      0.030696190893650055,\n      -0.03166574239730835,\n      0.07084017992019653,\n      -0.024772025644779205,\n      0.046735577285289764,\n      -0.08746784925460815,\n      0.06235901638865471,\n      0.1402968019247055,\n      0.06479959189891815,\n      -0.04169116169214249,\n      -0.009660803712904453,\n      0.06856372952461243,\n      -0.01113627664744854,\n      -0.09535631537437439,\n      0.029524657875299454,\n      -0.06304215639829636,\n      0.017704658210277557,\n      -0.10534399747848511,\n      0.17330017685890198,\n      -0.07611312717199326,\n      0.01210437435656786,\n      -0.05095558241009712,\n      0.0602450855076313,\n      0.10614507645368576,\n      0.003983400762081146,\n      -0.10606830567121506,\n      -0.0816468745470047,\n      0.013119752518832684,\n      -0.014868411235511303,\n      0.016667228192090988,\n      0.2971973419189453,\n      -0.08685001730918884,\n      -0.0874047726392746,\n      -0.013302342034876347,\n      0.03482017293572426,\n      0.006885201204568148,\n      -0.05091531574726105,\n      0.020986337214708328,\n      -0.07952297478914261,\n      -0.049061860889196396,\n      -0.08653101325035095,\n      0.13564595580101013,\n      0.05015904828906059,\n      -0.034869126975536346,\n      0.08793783187866211,\n      -0.028285151347517967,\n      0.09281701594591141,\n      0.00662242854014039,\n      0.10336703062057495,\n      0.2050163596868515,\n      -0.022548215463757515,\n      -0.021921362727880478,\n      -0.0039899623952806,\n      -0.018174221739172935,\n      0.024965662509202957,\n      0.04208444803953171,\n      -0.011034328490495682,\n      0.05374183878302574,\n      0.018499992787837982,\n      0.05828671529889107,\n      -0.41995036602020264,\n      0.04591332748532295,\n      0.04449240118265152,\n      0.13620178401470184,\n      0.16332395374774933,\n      0.10555198043584824,\n      -0.0034052629489451647,\n      0.12215616554021835,\n      0.01643737219274044,\n      0.0932932049036026\n    ],\n    [\n      0.17175240814685822,\n      0.08585312962532043,\n      0.14577732980251312,\n      0.15999934077262878,\n      -0.0695820078253746,\n      0.05798714607954025,\n      -0.061999060213565826,\n      0.016062451526522636,\n      -0.13519243896007538,\n      0.11787440627813339,\n      0.11529053747653961,\n      0.10408797115087509,\n      0.2111651450395584,\n      0.01612056791782379,\n      -0.4786147475242615,\n      0.20432168245315552,\n      -0.03199271485209465,\n      0.12675438821315765,\n      0.07309844344854355,\n      0.13207441568374634,\n      -0.00026442669332027435,\n      0.10104820877313614,\n      -0.0461902990937233,\n      0.019344987347722054,\n      0.02442646026611328,\n      -0.1204688623547554,\n      -0.07903839647769928,\n      0.1898127794265747,\n      0.2041134536266327,\n      -0.18195950984954834,\n      0.043905358761548996,\n      -0.04687049612402916,\n      -0.04209805279970169,\n      -0.12295661121606827,\n      0.0036759248469024897,\n      -0.0034698634408414364,\n      -0.10463957488536835,\n      -0.01529549341648817,\n      -0.05647580325603485,\n      -0.08864226937294006,\n      -0.09352686256170273,\n      -0.03846043720841408,\n      0.03636704385280609,\n      -0.014972069300711155,\n      0.04566527530550957,\n      0.038868844509124756,\n      -0.09578480571508408,\n      -0.06625876575708389,\n      0.029835980385541916,\n      -0.12734684348106384,\n      -0.008556054905056953,\n      0.03828263655304909,\n      -0.09373483806848526,\n      0.0005561970174312592,\n      0.012584840878844261,\n      -0.13427776098251343,\n      -0.16998428106307983,\n      -0.03930051997303963,\n      0.0333263985812664,\n      0.06848491728305817,\n      0.016444656997919083,\n      0.004641372710466385,\n      0.05181174352765083,\n      0.09835638850927353,\n      -0.012068777345120907,\n      -0.0729612186551094,\n      0.04139627143740654,\n      -0.08296596258878708,\n      0.04169049486517906,\n      -0.021982891485095024,\n      -0.027162304148077965,\n      -0.056516923010349274,\n      0.10789039731025696,\n      0.02400992624461651,\n      -0.1018250361084938,\n      0.03227527067065239,\n      0.005508039612323046,\n      -0.03932185843586922,\n      0.03622640669345856,\n      0.08246581256389618,\n      -0.11631577461957932,\n      0.037515971809625626,\n      -0.052226364612579346,\n      -0.07882820814847946,\n      0.151856929063797,\n      0.23806536197662354,\n      0.11504832655191422,\n      -0.02294222079217434,\n      0.13832956552505493,\n      0.1123109981417656,\n      -0.025942429900169373,\n      0.11239306628704071,\n      0.07747289538383484,\n      0.04889355227351189,\n      -0.0031880547758191824,\n      0.034164220094680786,\n      -0.02166224829852581,\n      0.052152663469314575,\n      0.01911104843020439,\n      -0.09248799830675125,\n      0.06788109987974167,\n      0.02831091545522213,\n      -0.00041694045648910105,\n      0.01384025439620018,\n      0.12224361300468445,\n      -0.05940850451588631,\n      -0.032462961971759796,\n      0.007921759970486164,\n      0.11208726465702057,\n      -0.018194124102592468,\n      0.02427692525088787,\n      0.021756989881396294,\n      -0.09511751681566238,\n      0.07825786620378494,\n      0.06334191560745239,\n      0.24838139116764069,\n      0.17681069672107697,\n      0.012471640482544899,\n      -0.13897380232810974,\n      -0.044395945966243744,\n      -0.24145783483982086,\n      0.02419835887849331,\n      0.06846805661916733,\n      -0.13386300206184387,\n      0.006285548210144043\n    ],\n    [\n      0.1455187052488327,\n      0.1034967377781868,\n      0.1253027468919754,\n      0.16493214666843414,\n      -0.09858862310647964,\n      -0.006660412065684795,\n      0.04621628299355507,\n      -0.010323175229132175,\n      -0.031023453921079636,\n      0.17358292639255524,\n      0.22848089039325714,\n      0.11562693119049072,\n      0.18590471148490906,\n      0.07309609651565552,\n      -0.4165308177471161,\n      0.33501219749450684,\n      0.09257303178310394,\n      0.18980206549167633,\n      0.017997093498706818,\n      0.06159196421504021,\n      0.014626462012529373,\n      -0.03823207691311836,\n      -0.00640766229480505,\n      0.0010762613965198398,\n      -0.11053172498941422,\n      0.009305248968303204,\n      0.04755168408155441,\n      0.3475547730922699,\n      0.07347530126571655,\n      -0.19733455777168274,\n      0.052128322422504425,\n      -0.09622714668512344,\n      0.06607350707054138,\n      -0.09133240580558777,\n      -0.03878737986087799,\n      -0.06731251627206802,\n      -0.040382664650678635,\n      -0.03130189701914787,\n      0.07845408469438553,\n      0.06516283750534058,\n      -0.06614647060632706,\n      0.0037197116762399673,\n      0.05351823568344116,\n      -0.028851231560111046,\n      -0.02517862617969513,\n      -0.08607836812734604,\n      -0.2631843686103821,\n      -0.09840959310531616,\n      0.03877924382686615,\n      -0.18827956914901733,\n      -0.0751447007060051,\n      0.0067700594663619995,\n      -0.1327221840620041,\n      -0.06295633316040039,\n      -0.00014197503332979977,\n      0.010639631189405918,\n      -0.18800482153892517,\n      -0.12229003012180328,\n      0.007754655089229345,\n      -0.07375095784664154,\n      -0.1632920205593109,\n      0.1000216081738472,\n      0.026247479021549225,\n      0.1073690801858902,\n      -0.06651656329631805,\n      -0.011034972965717316,\n      -0.01546233706176281,\n      0.06174122914671898,\n      -0.16834846138954163,\n      -0.06751614809036255,\n      -0.010278414003551006,\n      0.020628463476896286,\n      0.020464861765503883,\n      0.04463602975010872,\n      -0.0392519049346447,\n      -0.22424064576625824,\n      -0.10179878771305084,\n      -0.02439647726714611,\n      -0.026369864121079445,\n      -0.011679478920996189,\n      -0.1534849852323532,\n      -0.10182595252990723,\n      -0.1399187445640564,\n      -0.01515152957290411,\n      0.226799875497818,\n      0.1682286262512207,\n      0.07969320565462112,\n      0.02144390530884266,\n      0.018900468945503235,\n      0.07360079884529114,\n      0.1284593641757965,\n      -0.009614949114620686,\n      0.030024658888578415,\n      0.07387164980173111,\n      -0.01312728226184845,\n      0.054442550987005234,\n      -0.035710182040929794,\n      0.04629482328891754,\n      -0.0015513936523348093,\n      -0.04387059807777405,\n      -0.00674662459641695,\n      0.036264702677726746,\n      0.028677428141236305,\n      -0.049544595181941986,\n      0.16162580251693726,\n      -0.07012939453125,\n      0.019047824665904045,\n      0.11640448123216629,\n      0.10373613983392715,\n      -0.000489467813167721,\n      0.04490980878472328,\n      0.019880352541804314,\n      0.02130069211125374,\n      -0.07342980802059174,\n      -0.027455469593405724,\n      0.3454991281032562,\n      0.16780492663383484,\n      0.08394382894039154,\n      -0.15675169229507446,\n      0.07921242713928223,\n      -0.280223548412323,\n      0.044277217239141464,\n      0.09063107520341873,\n      -0.19358645379543304,\n      0.010789251886308193\n    ],\n    [\n      0.11455023288726807,\n      0.07460060715675354,\n      0.09456195682287216,\n      0.1455419361591339,\n      0.01225616130977869,\n      0.07854007184505463,\n      0.029341982677578926,\n      -0.12182138115167618,\n      0.011431717313826084,\n      0.08855301886796951,\n      0.18710270524024963,\n      0.03444909304380417,\n      0.2033046931028366,\n      0.16075575351715088,\n      -0.46490299701690674,\n      0.32960277795791626,\n      0.029649993404746056,\n      0.05947904661297798,\n      0.09973693639039993,\n      0.12608705461025238,\n      -0.08924886584281921,\n      0.00017712537373881787,\n      -0.020211653783917427,\n      0.11741627752780914,\n      -0.1219073086977005,\n      -0.038170430809259415,\n      -0.13719944655895233,\n      0.35853761434555054,\n      0.1638684719800949,\n      -0.25210604071617126,\n      0.034055858850479126,\n      -0.030787473544478416,\n      -0.1176203116774559,\n      -0.022380122914910316,\n      -0.014713935554027557,\n      0.05806467682123184,\n      -0.11872166395187378,\n      0.09089427441358566,\n      -0.08457794785499573,\n      -0.03564954176545143,\n      -0.016243139281868935,\n      0.058756276965141296,\n      0.06935382634401321,\n      0.05053842440247536,\n      0.10022205114364624,\n      -0.031684476882219315,\n      -0.12694475054740906,\n      -0.147348552942276,\n      0.012226554565131664,\n      -0.17751038074493408,\n      -0.16684910655021667,\n      0.0518030971288681,\n      -0.077850840985775,\n      -0.027524277567863464,\n      -0.11995993554592133,\n      -0.0391678512096405,\n      -0.19971023499965668,\n      -0.12338314205408096,\n      0.08947166800498962,\n      -0.10790599137544632,\n      -0.042606692761182785,\n      0.09646639227867126,\n      0.07174592465162277,\n      0.06853442639112473,\n      -0.09197864681482315,\n      -0.1665349006652832,\n      -0.11714480072259903,\n      -0.0467839278280735,\n      0.003132983110845089,\n      -0.039011988788843155,\n      -0.14685875177383423,\n      -0.01763780042529106,\n      0.04008277505636215,\n      0.011222386732697487,\n      -0.10479963570833206,\n      -0.24184028804302216,\n      -0.12816056609153748,\n      -0.024074116721749306,\n      0.06505880504846573,\n      0.09799633920192719,\n      -0.11065038293600082,\n      -0.0914328545331955,\n      -0.10621881484985352,\n      -0.09079577028751373,\n      0.08900820463895798,\n      0.19995781779289246,\n      0.0799848660826683,\n      0.10225824266672134,\n      0.042959075421094894,\n      0.15645474195480347,\n      0.08986356109380722,\n      0.042806901037693024,\n      0.09628033638000488,\n      0.11020984500646591,\n      0.03336288779973984,\n      0.13188719749450684,\n      -0.04990999400615692,\n      0.06908021867275238,\n      0.05784223973751068,\n      -0.049911368638277054,\n      0.07048621028661728,\n      0.01117426436394453,\n      0.0757673978805542,\n      -0.051441729068756104,\n      0.20502203702926636,\n      -0.014696936123073101,\n      0.06220126152038574,\n      0.05176251009106636,\n      0.01708895154297352,\n      0.04829060658812523,\n      0.10493732243776321,\n      0.03951103240251541,\n      -0.04525986313819885,\n      0.04434465616941452,\n      -0.01220259815454483,\n      0.2776741087436676,\n      0.10261934250593185,\n      -0.030325712636113167,\n      -0.13415618240833282,\n      -0.056756600737571716,\n      -0.34995919466018677,\n      0.017423532903194427,\n      0.07319120317697525,\n      -0.14434412121772766,\n      0.06969714909791946\n    ],\n    [\n      -0.11895236372947693,\n      -0.10702327638864517,\n      -0.07381594181060791,\n      -0.08252933621406555,\n      -0.09257661551237106,\n      -0.04912763833999634,\n      -0.07995116710662842,\n      0.022715721279382706,\n      -0.02507036365568638,\n      0.01118601206690073,\n      -0.17965823411941528,\n      -0.11112619191408157,\n      -0.12831023335456848,\n      -0.024047428742051125,\n      0.2279115915298462,\n      -0.3349292278289795,\n      -0.007636073976755142,\n      0.005075543187558651,\n      -0.019465049728751183,\n      -0.10763926804065704,\n      0.032617706805467606,\n      0.0791606530547142,\n      -0.09972035884857178,\n      0.09359518438577652,\n      -0.04972871020436287,\n      -0.006349754054099321,\n      -0.09732142090797424,\n      0.3313215374946594,\n      0.1906973272562027,\n      0.14743423461914062,\n      0.027187250554561615,\n      0.03942090645432472,\n      -0.008360587991774082,\n      -0.053047582507133484,\n      0.08329750597476959,\n      -0.03496840223670006,\n      -0.06786731630563736,\n      -0.045739445835351944,\n      0.03209458664059639,\n      -0.06584063917398453,\n      0.030686186626553535,\n      0.02213897742331028,\n      -0.02047237567603588,\n      -0.04206313192844391,\n      -0.07564189285039902,\n      0.1485469490289688,\n      0.05630266293883324,\n      0.027127565816044807,\n      0.13991107046604156,\n      0.042340777814388275,\n      -0.023627914488315582,\n      0.04982460290193558,\n      -0.06749644130468369,\n      0.112924724817276,\n      0.1714704930782318,\n      0.009481463581323624,\n      0.05809808522462845,\n      0.010872124694287777,\n      0.036284949630498886,\n      -0.04997028037905693,\n      -0.1039809137582779,\n      0.07138475030660629,\n      0.04926608130335808,\n      0.13652636110782623,\n      -0.060878824442625046,\n      -0.048054058104753494,\n      0.05756067857146263,\n      -0.05641390010714531,\n      0.004472943022847176,\n      -0.06536645442247391,\n      0.0009589297696948051,\n      0.03528149425983429,\n      -0.026249907910823822,\n      0.18463163077831268,\n      -0.01967203989624977,\n      0.05486173555254936,\n      0.026751454919576645,\n      0.020809106528759003,\n      0.057748131453990936,\n      0.06579572707414627,\n      -0.03149795159697533,\n      -0.05613516643643379,\n      -0.08974788337945938,\n      -0.02871987223625183,\n      -0.04316062107682228,\n      0.3222200572490692,\n      0.045099057257175446,\n      0.023303870111703873,\n      -0.13018767535686493,\n      -0.11852315068244934,\n      -0.0820978581905365,\n      -0.04961857199668884,\n      -0.06727739423513412,\n      -0.10707747936248779,\n      -0.08165857195854187,\n      -0.08225499838590622,\n      0.10143920034170151,\n      0.22791796922683716,\n      -0.007628417108207941,\n      0.022314971312880516,\n      -0.06891225278377533,\n      0.05480027198791504,\n      0.00039291460416279733,\n      -0.028324026614427567,\n      0.15304376184940338,\n      -0.020980965346097946,\n      -0.06836090981960297,\n      -0.03889355808496475,\n      0.017054658383131027,\n      -0.005334046669304371,\n      0.03752019256353378,\n      0.10340316593647003,\n      0.03934323415160179,\n      -0.04037807509303093,\n      0.08175778388977051,\n      -0.3366239368915558,\n      -0.10569406300783157,\n      0.14954248070716858,\n      0.12710750102996826,\n      0.15369601547718048,\n      0.15043500065803528,\n      -0.04779237136244774,\n      0.022887462750077248,\n      0.0512085035443306,\n      0.04714814946055412\n    ],\n    [\n      0.10980822890996933,\n      0.12429119646549225,\n      0.12009827792644501,\n      0.24768996238708496,\n      0.056052904576063156,\n      0.08174198865890503,\n      0.04269532486796379,\n      0.008619368076324463,\n      -0.009882405400276184,\n      0.0023933022748678923,\n      0.06689104437828064,\n      0.16277021169662476,\n      0.3224753737449646,\n      0.03396526724100113,\n      -0.35769888758659363,\n      0.4467429220676422,\n      0.1914815604686737,\n      0.1382761150598526,\n      0.10069122910499573,\n      0.16598492860794067,\n      -0.06100194528698921,\n      -0.06065729260444641,\n      -0.2087853103876114,\n      0.08974289894104004,\n      -0.17050600051879883,\n      -0.09303542971611023,\n      -0.10007180273532867,\n      0.2653948664665222,\n      0.16838395595550537,\n      -0.26136600971221924,\n      -0.04732615873217583,\n      -0.00488240784034133,\n      -0.11449373513460159,\n      0.1638641506433487,\n      -0.11868727207183838,\n      -0.06685416400432587,\n      -0.04881361871957779,\n      0.13436278700828552,\n      -0.03255404904484749,\n      0.04526106268167496,\n      0.06963244825601578,\n      0.036897629499435425,\n      -0.07326561212539673,\n      -0.09287489950656891,\n      -0.015190837904810905,\n      -0.0035322392359375954,\n      -0.2260492593050003,\n      0.023730464279651642,\n      0.026205210015177727,\n      -0.2750939726829529,\n      -0.2002740353345871,\n      0.07658500969409943,\n      -0.1778561770915985,\n      -0.05897596850991249,\n      -0.04746447876095772,\n      -0.09408747404813766,\n      -0.14767348766326904,\n      -0.07189177721738815,\n      -0.05359377712011337,\n      -0.10285964608192444,\n      -0.10444431006908417,\n      -0.023846859112381935,\n      0.05690695345401764,\n      0.09852531552314758,\n      -0.004151353146880865,\n      0.02616414986550808,\n      -0.17627497017383575,\n      -0.0025825249031186104,\n      -0.22829224169254303,\n      0.05687789246439934,\n      -0.18793199956417084,\n      -0.01678606867790222,\n      0.11485022306442261,\n      -0.00024240916536655277,\n      -0.07806410640478134,\n      -0.16619783639907837,\n      -0.10194040834903717,\n      0.014269744046032429,\n      0.005079955793917179,\n      0.25909140706062317,\n      -0.06437073647975922,\n      0.009034700691699982,\n      -0.03713971748948097,\n      -0.32571184635162354,\n      0.16943027079105377,\n      0.12354620546102524,\n      -0.1104661151766777,\n      0.1098736971616745,\n      0.12227360904216766,\n      0.04229603335261345,\n      0.04869254678487778,\n      0.004768448416143656,\n      0.08937150985002518,\n      0.00782798696309328,\n      0.011421802453696728,\n      0.11077698320150375,\n      -0.03670499473810196,\n      0.051200155168771744,\n      0.05466949939727783,\n      0.008119409903883934,\n      0.006967503577470779,\n      0.006425259169191122,\n      0.005465990398079157,\n      -0.032010048627853394,\n      0.16589313745498657,\n      -0.07554633170366287,\n      0.049413975328207016,\n      0.10508456081151962,\n      0.07039469480514526,\n      0.05025452375411987,\n      0.04873990640044212,\n      0.043729156255722046,\n      -0.10287397354841232,\n      -0.06475237756967545,\n      -0.0009879650315269828,\n      0.19513599574565887,\n      -0.010195563547313213,\n      -0.025910377502441406,\n      -0.08587607741355896,\n      0.013339651748538017,\n      -0.33793243765830994,\n      -0.0935848131775856,\n      -0.001432441407814622,\n      -0.13729502260684967,\n      -0.02448517642915249\n    ],\n    [\n      -0.12187422811985016,\n      -0.12558695673942566,\n      -0.0931447446346283,\n      -0.17805258929729462,\n      -0.0672769770026207,\n      -0.035224560648202896,\n      0.006181576754897833,\n      -0.051198817789554596,\n      -0.08556584268808365,\n      0.05319270119071007,\n      -0.1985367238521576,\n      -0.19230510294437408,\n      -0.31050875782966614,\n      -0.08320805430412292,\n      0.3715342879295349,\n      -0.35640373826026917,\n      -0.019336888566613197,\n      -0.11358941346406937,\n      -0.07869836688041687,\n      0.0030975902918726206,\n      0.09980244189500809,\n      -0.07972624152898788,\n      -0.1166069507598877,\n      -0.013307963497936726,\n      0.0022990966681391,\n      -0.04692342132329941,\n      -0.12096969038248062,\n      0.2319461703300476,\n      0.056401804089546204,\n      0.23014317452907562,\n      0.0017288841772824526,\n      -0.02185201086103916,\n      -0.05080876871943474,\n      0.031261175870895386,\n      0.035767484456300735,\n      -0.02108229137957096,\n      -0.08272688090801239,\n      0.0074964589439332485,\n      -0.03287779539823532,\n      -0.013557233847677708,\n      -0.08638492971658707,\n      -0.07378639280796051,\n      -0.13274171948432922,\n      0.05837646499276161,\n      0.018110699951648712,\n      0.057861462235450745,\n      0.03798028454184532,\n      0.0704508125782013,\n      0.1529218852519989,\n      -0.03346822038292885,\n      -0.021984320133924484,\n      0.2192540466785431,\n      -0.053976792842149734,\n      0.013357145711779594,\n      0.042915455996990204,\n      0.00909130834043026,\n      -0.042577650398015976,\n      0.05080326646566391,\n      -0.08880899101495743,\n      0.08026614785194397,\n      0.037186212837696075,\n      0.03560778126120567,\n      -0.024451332166790962,\n      0.08232799917459488,\n      -0.029728379100561142,\n      -0.132456436753273,\n      0.08477619290351868,\n      0.016734929755330086,\n      0.016882991418242455,\n      -0.0701102763414383,\n      0.042026448994874954,\n      0.014500260353088379,\n      0.0019243296701461077,\n      0.03435342013835907,\n      0.007817923091351986,\n      -0.06719354540109634,\n      0.06670786440372467,\n      -0.007850117050111294,\n      -0.017210837453603745,\n      0.08186347037553787,\n      -0.03588779643177986,\n      -0.08821991831064224,\n      0.03104238025844097,\n      0.0032483262475579977,\n      -0.043102946132421494,\n      0.3103752136230469,\n      -0.01666107214987278,\n      -0.11471303552389145,\n      -0.016910577192902565,\n      -0.09426285326480865,\n      0.02238129824399948,\n      -0.07267794013023376,\n      -0.042889200150966644,\n      -0.0942077562212944,\n      -0.09899956732988358,\n      -0.05336207523941994,\n      0.1727786511182785,\n      0.1542707234621048,\n      0.010179883800446987,\n      0.06185876950621605,\n      -0.07620523124933243,\n      -0.02114184759557247,\n      0.1793716698884964,\n      0.12380830943584442,\n      0.20666421949863434,\n      -0.00476005207747221,\n      0.010040461085736752,\n      0.014935115352272987,\n      0.15971505641937256,\n      0.03998660668730736,\n      0.05983437970280647,\n      0.08181779831647873,\n      -0.057492904365062714,\n      0.022471196949481964,\n      0.11591192334890366,\n      -0.25816574692726135,\n      -0.09434740990400314,\n      0.1725049614906311,\n      0.16138078272342682,\n      0.14798474311828613,\n      0.150874063372612,\n      0.07340797781944275,\n      0.017015080899000168,\n      0.016980459913611412,\n      0.08136306703090668\n    ],\n    [\n      0.08453833311796188,\n      0.1544504314661026,\n      0.24643948674201965,\n      0.3463439643383026,\n      0.3195427358150482,\n      -0.47009578347206116,\n      -0.05845551937818527,\n      -0.38798072934150696,\n      -0.5551103949546814,\n      -0.15963348746299744,\n      0.29756176471710205,\n      -0.26262471079826355,\n      -0.03814534842967987,\n      -0.07589803636074066,\n      -0.540597140789032,\n      -0.034478895366191864,\n      0.16012059152126312,\n      0.03377411141991615,\n      0.10033225268125534,\n      0.1329968124628067,\n      0.02471361681818962,\n      0.11546671390533447,\n      -0.23402144014835358,\n      0.13450023531913757,\n      0.31529685854911804,\n      0.24459700286388397,\n      -0.02428971417248249,\n      0.13333506882190704,\n      0.29583367705345154,\n      0.03916545212268829,\n      -0.02273176610469818,\n      0.07580076903104782,\n      -0.13615262508392334,\n      0.01198079064488411,\n      -0.27807557582855225,\n      -0.7113025188446045,\n      -0.08295787125825882,\n      -0.1721818745136261,\n      -0.041173771023750305,\n      -0.41478660702705383,\n      -0.31009918451309204,\n      -0.12545673549175262,\n      -0.38037779927253723,\n      0.009895398281514645,\n      0.224930077791214,\n      -0.09220775216817856,\n      0.2565101981163025,\n      0.175585076212883,\n      0.02946578525006771,\n      0.07156345993280411,\n      0.11476358026266098,\n      0.22444948554039001,\n      0.3742377460002899,\n      0.34154170751571655,\n      0.2768927812576294,\n      0.002737955655902624,\n      0.21915562450885773,\n      -0.011783919297158718,\n      -0.006725671701133251,\n      0.5776863694190979,\n      0.0821150466799736,\n      -0.09025510400533676,\n      0.19051627814769745,\n      -0.16558033227920532,\n      0.22145797312259674,\n      -0.04907530918717384,\n      0.2087891399860382,\n      -0.13434094190597534,\n      0.3989402949810028,\n      0.15483501553535461,\n      -0.048528704792261124,\n      0.06023373827338219,\n      0.20408792793750763,\n      0.04953331872820854,\n      -0.21948058903217316,\n      0.4785184860229492,\n      0.18225499987602234,\n      0.2698448598384857,\n      0.4502260386943817,\n      0.17453591525554657,\n      -0.0539400652050972,\n      -0.058700427412986755,\n      0.22174398601055145,\n      0.2608671486377716,\n      0.4238828718662262,\n      -0.3001536726951599,\n      0.18775705993175507,\n      0.11041462421417236,\n      -0.2034474015235901,\n      -0.07806464284658432,\n      0.1493656486272812,\n      0.038200121372938156,\n      0.07141493260860443,\n      0.2898358404636383,\n      0.1659456342458725,\n      0.311211496591568,\n      -0.08324148505926132,\n      0.13933736085891724,\n      0.2061452865600586,\n      0.12184689939022064,\n      0.003938419744372368,\n      0.09105338156223297,\n      0.09402414411306381,\n      0.10126137733459473,\n      -0.22006282210350037,\n      0.14871227741241455,\n      0.16736583411693573,\n      0.0946660041809082,\n      -0.05946106091141701,\n      -0.047226402908563614,\n      0.14482833445072174,\n      -0.002274337224662304,\n      0.06847215443849564,\n      0.13092489540576935,\n      0.2830195426940918,\n      0.5854626297950745,\n      0.40142637491226196,\n      -0.06935395300388336,\n      -0.026039065793156624,\n      -0.14101429283618927,\n      -0.11921678483486176,\n      0.12460020184516907,\n      -0.0220272745937109,\n      0.28213241696357727,\n      -0.06559044867753983\n    ],\n    [\n      -0.18851497769355774,\n      -0.02117704413831234,\n      -0.1489788293838501,\n      -0.08591543138027191,\n      -0.10861878842115402,\n      0.04621293395757675,\n      0.015186811797320843,\n      0.02542073279619217,\n      -0.13003288209438324,\n      0.026519782841205597,\n      -0.07011208683252335,\n      -0.10865887254476547,\n      -0.14982520043849945,\n      -0.04188273847103119,\n      0.38395386934280396,\n      -0.3176786005496979,\n      0.03461553528904915,\n      -0.02317710407078266,\n      -0.019763639196753502,\n      -0.06817933171987534,\n      -0.06411988288164139,\n      0.08020595461130142,\n      -0.058055248111486435,\n      0.05026255548000336,\n      0.026010680943727493,\n      0.11168672889471054,\n      -0.15267862379550934,\n      0.21904544532299042,\n      0.1910223811864853,\n      0.24094133079051971,\n      0.0014985192101448774,\n      -0.017161987721920013,\n      0.009343041107058525,\n      0.003280035685747862,\n      -0.07108630985021591,\n      -0.11804454028606415,\n      0.05246277153491974,\n      -0.03413674607872963,\n      -0.06598863005638123,\n      0.036845047026872635,\n      -0.04225833714008331,\n      -0.09749368578195572,\n      -0.008540305308997631,\n      -0.016947319731116295,\n      -0.023797141388058662,\n      0.132743701338768,\n      -0.043049369007349014,\n      -0.10490081459283829,\n      0.09381714463233948,\n      -0.06858053803443909,\n      0.04918775334954262,\n      0.13427090644836426,\n      0.04632863029837608,\n      0.007876544259488583,\n      0.0027383402921259403,\n      0.07609762996435165,\n      -0.0829816460609436,\n      -0.07819616794586182,\n      -0.10292045772075653,\n      0.07089032232761383,\n      0.04893563687801361,\n      0.06942597031593323,\n      0.09303413331508636,\n      0.051789287477731705,\n      -0.015397829934954643,\n      -0.12137842923402786,\n      0.0261837150901556,\n      0.04649151861667633,\n      -0.02380109392106533,\n      0.06917796283960342,\n      0.049030181020498276,\n      0.09952014684677124,\n      -0.09031957387924194,\n      0.13639044761657715,\n      -0.07744747400283813,\n      0.07737041264772415,\n      -0.019085053354501724,\n      -0.07144159823656082,\n      0.02657368779182434,\n      0.08272174000740051,\n      -0.12123268097639084,\n      0.052898332476615906,\n      -0.07660239934921265,\n      0.0014106952585279942,\n      -0.021789170801639557,\n      0.24995753169059753,\n      -0.08305871486663818,\n      -0.019017163664102554,\n      -0.10684693604707718,\n      -0.03534366190433502,\n      -0.055314622819423676,\n      -0.11141269654035568,\n      0.04560760408639908,\n      -0.10943084210157394,\n      -0.1176881268620491,\n      -0.043016981333494186,\n      0.20929330587387085,\n      0.10860178619623184,\n      0.057367954403162,\n      -0.060710955411195755,\n      -0.047807469964027405,\n      -0.00453504454344511,\n      0.09032908082008362,\n      0.028118658810853958,\n      0.2281198799610138,\n      0.055385153740644455,\n      0.01738281548023224,\n      0.12899866700172424,\n      0.05762557312846184,\n      0.008152347058057785,\n      0.12391918897628784,\n      0.1178901344537735,\n      0.02334301918745041,\n      0.007004237733781338,\n      0.04800083860754967,\n      -0.292233943939209,\n      0.06308244913816452,\n      0.09688499569892883,\n      0.16288882493972778,\n      0.21611663699150085,\n      0.1389733999967575,\n      0.09285833686590195,\n      -0.02837415225803852,\n      0.13304048776626587,\n      -0.04216840863227844\n    ],\n    [\n      -0.1798558235168457,\n      -0.12635599076747894,\n      -0.10380913317203522,\n      -0.19924061000347137,\n      -0.06193048506975174,\n      0.04269838705658913,\n      -0.005837441422045231,\n      -0.036365218460559845,\n      -0.07261906564235687,\n      -0.033355943858623505,\n      -0.14464031159877777,\n      -0.10872645676136017,\n      -0.16044163703918457,\n      -0.02587861381471157,\n      0.3549976944923401,\n      -0.23573645949363708,\n      -0.005145801696926355,\n      -0.017430739477276802,\n      -0.0030609136447310448,\n      -0.036777984350919724,\n      -0.05068959668278694,\n      0.06706473976373672,\n      -0.03325946629047394,\n      0.009890833869576454,\n      -0.035901617258787155,\n      0.10509033501148224,\n      -0.09972788393497467,\n      0.30061453580856323,\n      0.12577109038829803,\n      0.12382007390260696,\n      0.06862638890743256,\n      -0.00958780013024807,\n      0.010325976647436619,\n      0.027244698256254196,\n      0.012822337448596954,\n      -0.0014300212496891618,\n      -0.09995932132005692,\n      -0.113584503531456,\n      -0.03364754468202591,\n      0.012452797032892704,\n      -0.08920254558324814,\n      0.03199413791298866,\n      -0.017022669315338135,\n      0.014547108672559261,\n      -0.11973092705011368,\n      0.14013902842998505,\n      0.07669302821159363,\n      0.01690082438290119,\n      0.07388348132371902,\n      -0.060746610164642334,\n      -0.0017790381098166108,\n      0.21215568482875824,\n      0.07518924027681351,\n      0.0459679439663887,\n      0.10279037058353424,\n      0.07353145629167557,\n      -0.04417598620057106,\n      0.039377398788928986,\n      -0.05661287158727646,\n      -0.027112463489174843,\n      -0.10654793679714203,\n      0.12001421302556992,\n      0.0908646285533905,\n      -0.010847486555576324,\n      0.04335368424654007,\n      0.005715509876608849,\n      0.04208426550030708,\n      -0.07087337225675583,\n      0.07347691804170609,\n      0.06619969010353088,\n      -0.03401538357138634,\n      -0.06462112814188004,\n      -0.07385852932929993,\n      0.03598777949810028,\n      0.033941783010959625,\n      0.020388245582580566,\n      0.048442672938108444,\n      -0.0510852187871933,\n      0.10735835134983063,\n      0.03284927085042,\n      0.024740854278206825,\n      0.05377392843365669,\n      -0.08125414699316025,\n      -0.0020552196074277163,\n      -0.09695949405431747,\n      0.22780543565750122,\n      0.022713419049978256,\n      -0.04690172150731087,\n      -0.05262676626443863,\n      0.011232585646212101,\n      0.005708308890461922,\n      -0.08557981997728348,\n      0.014874715358018875,\n      -0.012737487442791462,\n      -0.03268827870488167,\n      -0.10072977095842361,\n      0.14102217555046082,\n      0.19854068756103516,\n      0.011244113557040691,\n      0.005784689914435148,\n      0.009904413484036922,\n      -0.036684781312942505,\n      0.03901740908622742,\n      0.05029765143990517,\n      0.13057319819927216,\n      0.13253305852413177,\n      0.08673151582479477,\n      0.06279681622982025,\n      0.03379559516906738,\n      0.11559431254863739,\n      0.17874713242053986,\n      0.13250483572483063,\n      0.0462305024266243,\n      0.021795855835080147,\n      0.09655352681875229,\n      -0.3230435252189636,\n      0.03319815918803215,\n      0.17541250586509705,\n      0.13289666175842285,\n      0.27960723638534546,\n      0.011251035146415234,\n      -0.04967522248625755,\n      -0.02464030124247074,\n      0.018988100811839104,\n      0.02358098514378071\n    ],\n    [\n      0.09931743144989014,\n      0.11378093808889389,\n      0.14227089285850525,\n      0.11735909432172775,\n      -0.052366890013217926,\n      0.17350974678993225,\n      0.01775088906288147,\n      -0.004888875875622034,\n      -0.005571085028350353,\n      0.05345511436462402,\n      0.053198348730802536,\n      0.17080071568489075,\n      0.28925999999046326,\n      0.11358366906642914,\n      -0.3537723422050476,\n      0.4101046919822693,\n      -0.08238368481397629,\n      0.15181700885295868,\n      0.11455857753753662,\n      0.1003083661198616,\n      0.06323616206645966,\n      0.09756765514612198,\n      -0.04717838391661644,\n      0.009487028233706951,\n      -0.0818416029214859,\n      -0.10625838488340378,\n      0.005010819528251886,\n      0.2159350961446762,\n      0.03211934119462967,\n      -0.11974336206912994,\n      0.05251609534025192,\n      -0.0638502985239029,\n      0.014221319928765297,\n      -0.03710605576634407,\n      -0.11200778931379318,\n      0.00944665540009737,\n      -0.002397798700258136,\n      -0.07369344681501389,\n      -0.03703593835234642,\n      0.016774771735072136,\n      -0.08518831431865692,\n      -0.029664216563105583,\n      -0.10752584785223007,\n      -0.08494605123996735,\n      0.026381712406873703,\n      -0.10245750099420547,\n      -0.09361341595649719,\n      -0.13497863709926605,\n      0.031706683337688446,\n      -0.14812630414962769,\n      -0.09895730763673782,\n      0.027044199407100677,\n      -0.038085322827100754,\n      -0.038885027170181274,\n      -0.07107074558734894,\n      -0.04885638877749443,\n      -0.0674712061882019,\n      -0.09748277813196182,\n      -0.02574899233877659,\n      0.0870027095079422,\n      -0.055072300136089325,\n      0.09267759323120117,\n      0.0699758529663086,\n      0.09532590955495834,\n      -0.11781901866197586,\n      -0.07638794183731079,\n      0.08744610100984573,\n      -0.0540330596268177,\n      0.04889753460884094,\n      0.045165568590164185,\n      0.10339205712080002,\n      0.023173252120614052,\n      -0.004189337603747845,\n      0.04383279010653496,\n      -0.019099952653050423,\n      -0.011579756624996662,\n      -0.03563150018453598,\n      0.0817636251449585,\n      0.06312281638383865,\n      0.04077206924557686,\n      -0.05322042107582092,\n      0.03574034571647644,\n      -0.06550655514001846,\n      0.012459811754524708,\n      0.13766011595726013,\n      0.08549631386995316,\n      0.012292793951928616,\n      -0.009232995100319386,\n      -0.020930539816617966,\n      0.10179099440574646,\n      -0.041105642914772034,\n      0.06081889942288399,\n      0.09669248014688492,\n      -0.058868151158094406,\n      0.0362972728908062,\n      -0.043640363961458206,\n      -0.05335237458348274,\n      0.08124502748250961,\n      -0.03427297994494438,\n      -0.06254906207323074,\n      -0.0392347052693367,\n      -0.07067599892616272,\n      0.016249652951955795,\n      -0.0008263186900876462,\n      0.13952018320560455,\n      -0.030158616602420807,\n      0.07157407701015472,\n      0.025713404640555382,\n      0.04301893711090088,\n      0.13381913304328918,\n      0.03879809007048607,\n      0.01940239779651165,\n      -0.043135616928339005,\n      0.020849229767918587,\n      0.10805141180753708,\n      0.29162389039993286,\n      0.20581620931625366,\n      -0.0010120703373104334,\n      -0.10405616462230682,\n      -0.0012100572930648923,\n      -0.22158224880695343,\n      0.018640587106347084,\n      -0.0029796839226037264,\n      -0.1019289568066597,\n      0.058307990431785583\n    ],\n    [\n      0.21742507815361023,\n      0.12464186549186707,\n      0.09626809507608414,\n      0.1576991230249405,\n      -0.024860195815563202,\n      -0.015697455033659935,\n      -0.013584091328084469,\n      0.07235033065080643,\n      0.12066813558340073,\n      -0.007299528922885656,\n      0.06684941798448563,\n      0.08998467028141022,\n      0.1840212196111679,\n      0.12231549620628357,\n      -0.3581007421016693,\n      0.3030429184436798,\n      0.026464883238077164,\n      0.16893664002418518,\n      0.070992611348629,\n      -0.027097320184111595,\n      0.12690143287181854,\n      0.04855741932988167,\n      0.09631111472845078,\n      0.14202165603637695,\n      0.06738627701997757,\n      -0.04948103055357933,\n      0.09962194412946701,\n      0.00757484370842576,\n      -0.057563748210668564,\n      -0.16811445355415344,\n      -0.016832338646054268,\n      -0.031998150050640106,\n      0.04052968695759773,\n      0.07922355830669403,\n      0.08499900251626968,\n      0.0032944970298558474,\n      0.025777339935302734,\n      0.03777559474110603,\n      -0.011951368302106857,\n      -0.05649799481034279,\n      0.10978665202856064,\n      0.023022184148430824,\n      0.046043068170547485,\n      0.05625415965914726,\n      0.036097269505262375,\n      -0.1089986264705658,\n      -0.10525837540626526,\n      0.026031438261270523,\n      -0.03013908676803112,\n      -0.04243764653801918,\n      -0.128693625330925,\n      0.0790812149643898,\n      -0.16447491943836212,\n      -0.07141703367233276,\n      0.009452779777348042,\n      0.0056789712980389595,\n      -0.02944561094045639,\n      0.030126450583338737,\n      0.11870669573545456,\n      -0.05448387563228607,\n      0.020571475848555565,\n      -0.017290404066443443,\n      0.013142740353941917,\n      0.016157202422618866,\n      0.027111448347568512,\n      0.036567676812410355,\n      -0.003152508521452546,\n      0.07396100461483002,\n      -0.11018183827400208,\n      -0.07143032550811768,\n      0.13701623678207397,\n      -0.04161432385444641,\n      0.10131041705608368,\n      -0.08212655037641525,\n      -0.010007914155721664,\n      0.02909133955836296,\n      -0.02004285529255867,\n      0.04502051696181297,\n      -0.1458352655172348,\n      -0.059364840388298035,\n      -0.08206836879253387,\n      -0.05645463988184929,\n      -0.027602070942521095,\n      0.06565845757722855,\n      -0.011016285978257656,\n      -0.08822141587734222,\n      -0.023270083591341972,\n      0.12340132147073746,\n      0.06296077370643616,\n      0.02900378592312336,\n      0.00229810387827456,\n      0.055984411388635635,\n      0.09100247919559479,\n      0.046526409685611725,\n      -0.023246467113494873,\n      0.040291182696819305,\n      -0.009300914593040943,\n      0.1159186065196991,\n      0.008937625214457512,\n      -0.08643636852502823,\n      -0.09967697411775589,\n      -0.035649001598358154,\n      0.03310161828994751,\n      0.09889276325702667,\n      0.06175920367240906,\n      -0.01817416027188301,\n      -0.05110543221235275,\n      0.048153795301914215,\n      0.05147028714418411,\n      0.14661848545074463,\n      -0.02171579748392105,\n      0.05298647657036781,\n      -0.047229867428541183,\n      0.034253910183906555,\n      0.15409891307353973,\n      0.26798880100250244,\n      0.22236858308315277,\n      0.10516952723264694,\n      -0.0003187355468980968,\n      -0.14308764040470123,\n      -0.16927587985992432,\n      0.016476688906550407,\n      -0.00421871617436409,\n      -0.19374293088912964,\n      -0.01858392357826233\n    ],\n    [\n      -0.16326874494552612,\n      -0.12725119292736053,\n      -0.15450511872768402,\n      -0.07402726262807846,\n      0.0963786318898201,\n      -0.04227510839700699,\n      -0.008685215376317501,\n      -0.12163061648607254,\n      0.0014571346109732985,\n      -0.09564648568630219,\n      -0.152487114071846,\n      -0.0666794702410698,\n      -0.21958597004413605,\n      -0.1141592487692833,\n      0.23149661719799042,\n      -0.3538784086704254,\n      -0.07173387706279755,\n      -0.11858855187892914,\n      -0.09121482819318771,\n      -0.08214086294174194,\n      -0.015862833708524704,\n      0.01137463841587305,\n      0.04234572872519493,\n      0.07374531030654907,\n      0.05462133511900902,\n      0.09039600938558578,\n      -0.06605605781078339,\n      0.32508164644241333,\n      0.18055389821529388,\n      0.20829059183597565,\n      -0.024038754403591156,\n      0.11361752450466156,\n      -0.045604344457387924,\n      0.012082422152161598,\n      -0.07333244383335114,\n      -0.025546161457896233,\n      -0.008430239744484425,\n      -0.1403466910123825,\n      0.0656796544790268,\n      -0.107267826795578,\n      -0.05243046581745148,\n      0.018428344279527664,\n      0.02008027955889702,\n      0.09569799154996872,\n      -0.000787111755926162,\n      -0.011036441661417484,\n      0.004292335361242294,\n      -0.059393513947725296,\n      0.15862594544887543,\n      -0.037398528307676315,\n      -0.02175056003034115,\n      0.04617678374052048,\n      0.07187211513519287,\n      0.048047635704278946,\n      0.07768692076206207,\n      0.10728271305561066,\n      -0.03878642991185188,\n      -0.0254514217376709,\n      -0.10875541716814041,\n      0.07632561028003693,\n      -0.10749559849500656,\n      0.14544343948364258,\n      0.09989358484745026,\n      0.06237354129552841,\n      0.07816681265830994,\n      -0.03254937008023262,\n      -0.035724155604839325,\n      -0.021279992535710335,\n      0.08474048227071762,\n      0.036239851266145706,\n      -0.07730575650930405,\n      0.021282369270920753,\n      -0.06376541405916214,\n      0.0941111147403717,\n      -0.03587687760591507,\n      0.007624862715601921,\n      -0.05228176712989807,\n      0.07195112854242325,\n      0.07676966488361359,\n      0.08990169316530228,\n      0.030148014426231384,\n      -0.07529579848051071,\n      0.0029728731606155634,\n      -0.040888700634241104,\n      -0.11470459401607513,\n      0.308228462934494,\n      -0.04999617487192154,\n      -0.0766158476471901,\n      -0.14235471189022064,\n      -0.05809228867292404,\n      0.005271436646580696,\n      -0.09230697900056839,\n      -0.023997284471988678,\n      -0.11021111160516739,\n      -0.11490817368030548,\n      0.010294110514223576,\n      0.13070376217365265,\n      0.23536987602710724,\n      0.05711529031395912,\n      -0.04593769088387489,\n      0.05474104359745979,\n      -0.03719114512205124,\n      0.08238359540700912,\n      0.0023812095168977976,\n      0.22946563363075256,\n      0.037906989455223083,\n      -0.05881774052977562,\n      0.14835447072982788,\n      0.12800152599811554,\n      0.017945220693945885,\n      0.10224758088588715,\n      0.07803943008184433,\n      -0.044812001287937164,\n      -0.00913101527839899,\n      0.015443569049239159,\n      -0.29378703236579895,\n      0.03451129049062729,\n      0.17927244305610657,\n      0.10020111501216888,\n      0.232412189245224,\n      0.13040791451931,\n      0.02983591891825199,\n      0.029872383922338486,\n      0.14920863509178162,\n      0.03131695091724396\n    ],\n    [\n      -0.08636423945426941,\n      -0.019873816519975662,\n      -0.1585734337568283,\n      -0.21431311964988708,\n      -0.057700853794813156,\n      -0.06154843047261238,\n      0.006400353740900755,\n      0.04263194650411606,\n      -0.10030870884656906,\n      -0.06825806945562363,\n      -0.11647586524486542,\n      -0.14614897966384888,\n      -0.11747979372739792,\n      -0.05923724174499512,\n      0.3611362874507904,\n      -0.331387460231781,\n      0.0541253499686718,\n      0.04289156571030617,\n      -0.06203824281692505,\n      -0.01808580756187439,\n      0.01804427057504654,\n      -0.1077205240726471,\n      -0.10452494770288467,\n      0.09782537072896957,\n      -0.014598778448998928,\n      0.07440625131130219,\n      -0.002892065327614546,\n      0.19138136506080627,\n      0.20641057193279266,\n      0.19913671910762787,\n      0.018554843962192535,\n      0.053434375673532486,\n      -0.07390274107456207,\n      0.0014778987970203161,\n      -0.04901427775621414,\n      -0.06202520802617073,\n      0.04894140735268593,\n      -0.04008171334862709,\n      0.08248383551836014,\n      -0.022753136232495308,\n      0.033033568412065506,\n      -0.035837095230817795,\n      -0.03109009936451912,\n      -0.0337473601102829,\n      -0.011290390975773335,\n      0.03540556877851486,\n      0.05245589464902878,\n      -0.049404386430978775,\n      0.14282168447971344,\n      -0.004939169157296419,\n      -0.052742280066013336,\n      0.058368176221847534,\n      -0.031250376254320145,\n      0.0984015092253685,\n      0.1461304873228073,\n      0.059959277510643005,\n      0.06710459291934967,\n      -0.08761254698038101,\n      -0.018753528594970703,\n      -0.0618097297847271,\n      0.051303815096616745,\n      0.01972026936709881,\n      0.06093289330601692,\n      0.022693699225783348,\n      -0.04664253070950508,\n      0.03624196723103523,\n      0.01261476892977953,\n      -0.09050806611776352,\n      -0.05466286838054657,\n      0.05263540521264076,\n      -0.06079268082976341,\n      0.05600659176707268,\n      -0.03994046524167061,\n      0.015793990343809128,\n      -0.05787111818790436,\n      0.02669016271829605,\n      -0.04168233647942543,\n      0.01492500863969326,\n      -0.012366685084998608,\n      0.11718229949474335,\n      -0.0060272617265582085,\n      0.06220109015703201,\n      0.010173526592552662,\n      -0.05830497667193413,\n      -0.10657649487257004,\n      0.34151721000671387,\n      -0.03531801328063011,\n      -0.10597394406795502,\n      -0.02406727336347103,\n      -0.11115379631519318,\n      -0.01317504234611988,\n      -0.1211770698428154,\n      -0.06493951380252838,\n      -0.032478559762239456,\n      -0.04097656160593033,\n      -0.0131577979773283,\n      0.19613775610923767,\n      0.20729920268058777,\n      -0.05807226523756981,\n      0.01509025227278471,\n      -0.05745746195316315,\n      0.08838488906621933,\n      0.09881561249494553,\n      0.01638931594789028,\n      0.06380123645067215,\n      -0.014597555622458458,\n      -0.043030302971601486,\n      -0.00025789797655306756,\n      0.0053491718135774136,\n      0.10248240828514099,\n      -0.006490836385637522,\n      -0.00881132110953331,\n      -0.008460835553705692,\n      -0.025141067802906036,\n      -0.01883671246469021,\n      -0.4166962504386902,\n      -0.10781819373369217,\n      0.07315780967473984,\n      0.1205994114279747,\n      0.2175675928592682,\n      0.12957771122455597,\n      0.023658424615859985,\n      -0.05966728553175926,\n      0.13180239498615265,\n      0.03795250132679939\n    ],\n    [\n      0.228984534740448,\n      -0.03043973259627819,\n      0.06042126193642616,\n      0.1330128312110901,\n      -0.0698961466550827,\n      0.04427053779363632,\n      0.013987744227051735,\n      0.06369119137525558,\n      0.006486616097390652,\n      0.14369304478168488,\n      0.08368749171495438,\n      0.051908645778894424,\n      0.17110204696655273,\n      0.12044311314821243,\n      -0.28704777359962463,\n      0.23157361149787903,\n      -0.05637974664568901,\n      0.18124757707118988,\n      -0.019162321463227272,\n      0.09435372054576874,\n      0.05533389374613762,\n      0.02848992682993412,\n      -0.059957001358270645,\n      0.13491109013557434,\n      -0.07142376154661179,\n      -0.0397978276014328,\n      0.05010982230305672,\n      0.12843656539916992,\n      0.011398505419492722,\n      -0.06923823803663254,\n      -0.04844864830374718,\n      -0.1429504156112671,\n      -0.08771199733018875,\n      -0.0017844222020357847,\n      0.05918029323220253,\n      0.10018340498209,\n      -0.06557724624872208,\n      0.04697718471288681,\n      0.07817626744508743,\n      0.058608751744031906,\n      -0.02430148795247078,\n      0.03788873180747032,\n      0.010580652393400669,\n      -0.013017641380429268,\n      0.07119083404541016,\n      -0.07118172198534012,\n      -0.044229649007320404,\n      -0.2155962884426117,\n      -0.01884319633245468,\n      -0.02099205181002617,\n      -0.09674489498138428,\n      -0.13576410710811615,\n      -0.14529481530189514,\n      -0.13520777225494385,\n      -0.10001909732818604,\n      0.01038765162229538,\n      -0.16057993471622467,\n      -0.025845874100923538,\n      0.09902165830135345,\n      -0.09768113493919373,\n      -0.055108264088630676,\n      0.03528889641165733,\n      -0.044988129287958145,\n      -0.034809574484825134,\n      0.022289246320724487,\n      -0.11500491201877594,\n      -0.0540238581597805,\n      -0.08160638064146042,\n      -0.15303745865821838,\n      0.025277744978666306,\n      -0.13460074365139008,\n      -0.043893467634916306,\n      -0.10000129789113998,\n      0.00257837469689548,\n      -0.06781870126724243,\n      -0.06083938479423523,\n      0.02526092901825905,\n      -0.0324539989233017,\n      -0.02417704649269581,\n      0.018944839015603065,\n      -0.01683570258319378,\n      -0.04157431423664093,\n      -0.020312542095780373,\n      -0.15226192772388458,\n      0.15664006769657135,\n      -0.06017162650823593,\n      0.05397813394665718,\n      0.09157674759626389,\n      0.048449642956256866,\n      0.004812442231923342,\n      0.02473324164748192,\n      0.09440640360116959,\n      0.08914966136217117,\n      -0.014284869655966759,\n      0.04715477302670479,\n      0.10882239788770676,\n      0.020395083352923393,\n      0.016904957592487335,\n      0.054798513650894165,\n      -0.04845331236720085,\n      -0.00406267773360014,\n      0.008025976829230785,\n      0.0809616819024086,\n      -0.022403113543987274,\n      0.09720565378665924,\n      0.05995718389749527,\n      0.07373487204313278,\n      0.08380965143442154,\n      0.06089964136481285,\n      0.05945561081171036,\n      0.033796023577451706,\n      -0.034510623663663864,\n      -0.06010096147656441,\n      -0.008002092130482197,\n      0.024293402209877968,\n      0.3361038863658905,\n      0.19348503649234772,\n      0.10091701149940491,\n      -0.04673001542687416,\n      -0.05056144669651985,\n      -0.3083210289478302,\n      -0.05989915877580643,\n      0.1315045952796936,\n      0.020529359579086304,\n      -0.029238350689411163\n    ],\n    [\n      0.23944707214832306,\n      0.03710811212658882,\n      0.11500830203294754,\n      0.1749134510755539,\n      -0.06098267808556557,\n      0.0186664666980505,\n      0.08169328421354294,\n      -0.01348246168345213,\n      -0.012337563559412956,\n      0.04280661791563034,\n      0.2359641045331955,\n      0.16845931112766266,\n      0.1822613775730133,\n      0.1060904711484909,\n      -0.25077179074287415,\n      0.18122562766075134,\n      0.05970191955566406,\n      0.11371017247438431,\n      0.053194671869277954,\n      0.023747848346829414,\n      0.06847439706325531,\n      0.09634724259376526,\n      -0.0775294080376625,\n      0.052934516221284866,\n      -0.13079307973384857,\n      -0.05668632313609123,\n      -0.09885869920253754,\n      0.24847464263439178,\n      0.04312344640493393,\n      -0.13409239053726196,\n      -0.011706031858921051,\n      0.05135643854737282,\n      0.010706746019423008,\n      -0.06422031670808792,\n      0.014543242752552032,\n      0.0604814775288105,\n      0.011096533387899399,\n      -0.05711064860224724,\n      -0.04630475863814354,\n      -0.036870263516902924,\n      -0.008064529858529568,\n      0.03112095780670643,\n      -0.042966216802597046,\n      0.026129482313990593,\n      0.030639145523309708,\n      -0.10358073562383652,\n      -0.22731468081474304,\n      -0.1530204564332962,\n      0.0031928198877722025,\n      -0.04897086322307587,\n      -0.12152475118637085,\n      -0.03399845212697983,\n      -0.14132045209407806,\n      -0.09113381803035736,\n      -0.005960328038781881,\n      -0.07139663398265839,\n      -0.16807334125041962,\n      -0.10655663162469864,\n      -0.023840410634875298,\n      -0.02371411770582199,\n      -0.18841052055358887,\n      0.11462763696908951,\n      0.10263873636722565,\n      0.15477114915847778,\n      -0.03822214901447296,\n      -0.08915498852729797,\n      -0.13094764947891235,\n      -0.001225110376253724,\n      -0.14107108116149902,\n      -0.015043413266539574,\n      -0.03210445120930672,\n      0.06973620504140854,\n      0.06223069503903389,\n      -0.022811030969023705,\n      -0.07442063838243484,\n      -0.21648207306861877,\n      -0.08682209253311157,\n      -0.07455899566411972,\n      0.0004600370302796364,\n      0.03586099296808243,\n      -0.02846118062734604,\n      -0.07727617025375366,\n      -0.011932924389839172,\n      -0.09819651395082474,\n      0.1280691772699356,\n      0.08531458675861359,\n      0.058815594762563705,\n      0.039460599422454834,\n      0.1070842444896698,\n      -0.0016218858072534204,\n      0.049370862543582916,\n      0.11214891821146011,\n      0.09207814186811447,\n      -0.07199844717979431,\n      -0.06656472384929657,\n      0.03388912230730057,\n      -0.08196476101875305,\n      0.07723291963338852,\n      0.08704934269189835,\n      0.022556278854608536,\n      0.005344738252460957,\n      0.0405525304377079,\n      0.07449999451637268,\n      -0.02828892506659031,\n      0.1304301619529724,\n      -0.06420540064573288,\n      0.07378992438316345,\n      0.09659390151500702,\n      0.07290102541446686,\n      0.0543384850025177,\n      0.08456996083259583,\n      0.03224657103419304,\n      0.05393175035715103,\n      -0.06500086188316345,\n      -0.06655402481555939,\n      0.3870537281036377,\n      0.1636718213558197,\n      0.0559878833591938,\n      -0.1494351625442505,\n      0.05020001158118248,\n      -0.17845922708511353,\n      0.047422248870134354,\n      0.022597864270210266,\n      -0.14990735054016113,\n      -0.06768123805522919\n    ],\n    [\n      0.1389540135860443,\n      0.031581662595272064,\n      0.11677136272192001,\n      0.24468140304088593,\n      -0.0753939300775528,\n      0.13099835813045502,\n      0.10369676351547241,\n      0.009396479465067387,\n      0.03495307266712189,\n      0.10216399282217026,\n      0.13294921815395355,\n      0.20253057777881622,\n      0.13940908014774323,\n      -0.03368968516588211,\n      -0.27865999937057495,\n      0.23523245751857758,\n      0.10428951680660248,\n      0.06409605592489243,\n      0.10867957025766373,\n      0.07013572752475739,\n      0.013241386972367764,\n      0.08972058445215225,\n      -0.11883477866649628,\n      0.07508257776498795,\n      -0.12347837537527084,\n      -0.010435006581246853,\n      0.065738245844841,\n      0.16796408593654633,\n      -0.03881244733929634,\n      -0.09775347262620926,\n      0.02689557895064354,\n      -0.11003883928060532,\n      0.01795020140707493,\n      0.09084312617778778,\n      0.007631675805896521,\n      0.08553778380155563,\n      -0.07689127326011658,\n      0.03843549266457558,\n      0.016899706795811653,\n      -0.026741182431578636,\n      0.056915462017059326,\n      0.06784643232822418,\n      -0.05651004984974861,\n      0.03171559050679207,\n      0.0551878847181797,\n      -0.04963274300098419,\n      -0.20096658170223236,\n      -0.011507874354720116,\n      -0.027911502867937088,\n      -0.1559857428073883,\n      -0.12635457515716553,\n      0.02582038752734661,\n      -0.03277004882693291,\n      -0.040504150092601776,\n      0.06638435274362564,\n      -0.09356162697076797,\n      -0.2537875473499298,\n      0.0011975534725934267,\n      0.009898087941110134,\n      -0.08763715624809265,\n      -0.09578955918550491,\n      0.032317884266376495,\n      0.05258652940392494,\n      0.0034298410173505545,\n      0.059479255229234695,\n      0.08148136734962463,\n      -0.0804200991988182,\n      -0.0002336409961571917,\n      -0.025242457166314125,\n      -0.008663048967719078,\n      0.07881379872560501,\n      0.021934697404503822,\n      -0.16692869365215302,\n      0.03790503740310669,\n      0.03694712743163109,\n      -0.14959904551506042,\n      0.07044634968042374,\n      -0.13514071702957153,\n      -0.05232509225606918,\n      0.06214002147316933,\n      0.017737824469804764,\n      -0.0944090262055397,\n      0.0012382654240354896,\n      -0.011327598243951797,\n      0.19414307177066803,\n      -0.06453757733106613,\n      0.031239449977874756,\n      0.09162117540836334,\n      0.14977960288524628,\n      0.05546104162931442,\n      0.08810824155807495,\n      -0.020023122429847717,\n      0.034749094396829605,\n      -0.006236691493541002,\n      0.09014865010976791,\n      0.0708797425031662,\n      0.03199233114719391,\n      0.030940677970647812,\n      0.08445993065834045,\n      -0.07679224014282227,\n      0.0225549153983593,\n      -0.0030016375239938498,\n      0.07457149773836136,\n      0.06332961469888687,\n      0.06470523029565811,\n      0.0018782175611704588,\n      -0.0471828319132328,\n      -0.00871354155242443,\n      0.1545914262533188,\n      0.1248025968670845,\n      -0.06490419059991837,\n      -0.05070652440190315,\n      -0.03685314953327179,\n      -0.04731433838605881,\n      0.037534985691308975,\n      0.3749311566352844,\n      0.12899842858314514,\n      -0.03624144569039345,\n      -0.14242441952228546,\n      0.03694158047437668,\n      -0.2722545266151428,\n      0.06687337160110474,\n      -0.004948676098138094,\n      -0.08180779963731766,\n      0.02157912775874138\n    ],\n    [\n      0.246724933385849,\n      0.0520944818854332,\n      0.020768718793988228,\n      0.20906873047351837,\n      0.004626308102160692,\n      0.008935653604567051,\n      0.07625054568052292,\n      0.03090824745595455,\n      -0.03320734202861786,\n      -0.011210475116968155,\n      0.1865389496088028,\n      0.061281152069568634,\n      0.19065327942371368,\n      0.11409936100244522,\n      -0.4071202576160431,\n      0.3599106967449188,\n      0.06374555826187134,\n      0.11339500546455383,\n      0.018878426402807236,\n      -0.003892191918566823,\n      -0.01807764731347561,\n      0.04820128530263901,\n      -0.08622027188539505,\n      0.05903508514165878,\n      -0.12767891585826874,\n      -0.015883207321166992,\n      -0.05462636798620224,\n      0.13718445599079132,\n      0.17466571927070618,\n      -0.17290110886096954,\n      0.055116329342126846,\n      -0.0249024610966444,\n      0.040405746549367905,\n      -0.09651774168014526,\n      -0.13208487629890442,\n      -0.05597778037190437,\n      0.04057793319225311,\n      -0.0360860675573349,\n      -0.023010583594441414,\n      0.0549568235874176,\n      -0.1019265204668045,\n      0.05547741428017616,\n      0.022597957402467728,\n      -0.030071545392274857,\n      -0.03814408555626869,\n      0.03318895399570465,\n      -0.10081032663583755,\n      -0.1310962736606598,\n      0.0044191754423081875,\n      -0.06070047244429588,\n      -0.06860872358083725,\n      0.01036241464316845,\n      -0.02730935998260975,\n      -0.06755760312080383,\n      -0.09935419261455536,\n      -0.14596576988697052,\n      -0.17659920454025269,\n      -0.06233031675219536,\n      -0.029609892517328262,\n      -0.003246721113100648,\n      -0.07174195349216461,\n      0.0606117807328701,\n      0.025286901742219925,\n      0.09565004706382751,\n      -0.12097765505313873,\n      -0.08640225231647491,\n      0.023716092109680176,\n      0.020862070843577385,\n      0.006862318143248558,\n      -0.09590327739715576,\n      -0.08203154802322388,\n      0.01796656660735607,\n      0.039435841143131256,\n      -0.04914369061589241,\n      -0.06682281941175461,\n      -0.0903797298669815,\n      -0.03669043630361557,\n      0.031459685415029526,\n      0.011627747677266598,\n      -0.008498094975948334,\n      -0.020397165790200233,\n      -0.0029274853877723217,\n      -0.02391195297241211,\n      -0.05453500896692276,\n      0.12065470963716507,\n      0.11018669605255127,\n      -0.01791493035852909,\n      0.0436612106859684,\n      0.12218981981277466,\n      0.006336679682135582,\n      0.12475737929344177,\n      0.0046186246909201145,\n      0.040960945188999176,\n      -0.06920620799064636,\n      0.039700910449028015,\n      -0.04094095528125763,\n      -0.03827909380197525,\n      0.11996486037969589,\n      -0.013069238513708115,\n      -0.020134620368480682,\n      0.10272184014320374,\n      -0.030562201514840126,\n      0.09156686812639236,\n      0.08206096291542053,\n      0.16483594477176666,\n      -0.02966373786330223,\n      -0.00017023077816702425,\n      0.04935615137219429,\n      0.03541696444153786,\n      0.09350511431694031,\n      0.09290271997451782,\n      0.06540991365909576,\n      0.010955780744552612,\n      -0.032396260648965836,\n      -0.02371278777718544,\n      0.4560350775718689,\n      0.11440509557723999,\n      0.08002343028783798,\n      -0.07016320526599884,\n      -0.09288457781076431,\n      -0.2362179309129715,\n      -0.009871273301541805,\n      0.07343927770853043,\n      -0.10050024837255478,\n      0.05744285508990288\n    ],\n    [\n      0.14386646449565887,\n      0.10411574691534042,\n      0.08676622062921524,\n      0.047735415399074554,\n      -0.043565381318330765,\n      0.046277325600385666,\n      0.11539744585752487,\n      -0.1183081641793251,\n      0.036202747374773026,\n      0.11034391075372696,\n      0.18036799132823944,\n      0.10473519563674927,\n      0.18600136041641235,\n      -0.009770001284778118,\n      -0.4576490521430969,\n      0.19429408013820648,\n      0.04317745566368103,\n      0.10244382172822952,\n      0.07660084962844849,\n      -0.04833758994936943,\n      -0.0214735958725214,\n      0.01419293787330389,\n      -0.11918241530656815,\n      0.10369757562875748,\n      -0.028939342126250267,\n      -0.0990915521979332,\n      -0.06057339161634445,\n      0.18059806525707245,\n      0.14882440865039825,\n      -0.22378431260585785,\n      0.12501348555088043,\n      0.013887518085539341,\n      -0.024156862869858742,\n      -0.047360509634017944,\n      -0.04874103143811226,\n      0.05721883848309517,\n      -0.039124857634305954,\n      -0.018627382814884186,\n      0.02341381087899208,\n      0.08975575119256973,\n      -0.06294117867946625,\n      -0.0130885886028409,\n      -0.027512749657034874,\n      -0.011773008853197098,\n      0.018423086032271385,\n      -0.0819711685180664,\n      -0.14832481741905212,\n      -0.19082090258598328,\n      -0.004134242422878742,\n      -0.15764279663562775,\n      -0.05768812075257301,\n      0.02506559155881405,\n      -0.1277504712343216,\n      -0.10046864300966263,\n      -0.07295510917901993,\n      -0.1708117425441742,\n      -0.24154461920261383,\n      -0.19222067296504974,\n      -0.0538051575422287,\n      -0.014691689051687717,\n      -0.09748056530952454,\n      0.10766889899969101,\n      0.06610220670700073,\n      0.027140045538544655,\n      -0.10396154969930649,\n      -0.042714059352874756,\n      0.028408903628587723,\n      -0.03297405317425728,\n      -0.09126447886228561,\n      -0.06600655615329742,\n      -0.07005707174539566,\n      -0.04393965005874634,\n      -0.013899191282689571,\n      0.026285750791430473,\n      -0.01793498359620571,\n      -0.1557527482509613,\n      -0.006643781904131174,\n      0.02899641916155815,\n      0.009174082428216934,\n      0.12164619565010071,\n      -0.10027208179235458,\n      -0.09265471249818802,\n      -0.11107391864061356,\n      -0.08951721340417862,\n      0.17746824026107788,\n      0.13999861478805542,\n      0.0504453144967556,\n      0.0781266838312149,\n      0.025822777301073074,\n      0.1423438936471939,\n      0.06996835023164749,\n      0.005289588589221239,\n      0.0511540062725544,\n      0.06570281088352203,\n      0.11981812864542007,\n      0.10245545953512192,\n      -0.00523789320141077,\n      -0.014197854325175285,\n      0.013861236162483692,\n      -0.05036090314388275,\n      -0.0537402369081974,\n      -0.048572294414043427,\n      0.04274225980043411,\n      0.03391219303011894,\n      0.1972798854112625,\n      -0.016945855692029,\n      -0.022834209725260735,\n      0.03600478172302246,\n      0.032805781811475754,\n      -0.042202673852443695,\n      0.05800831317901611,\n      0.013828062452375889,\n      -0.03197481855750084,\n      -0.07208310812711716,\n      -0.04900406301021576,\n      0.3098613917827606,\n      0.17801909148693085,\n      0.09920715540647507,\n      -0.15861260890960693,\n      0.00015794845239724964,\n      -0.2933254837989807,\n      -0.006386913359165192,\n      -0.025190263986587524,\n      -0.17088034749031067,\n      0.02419259399175644\n    ],\n    [\n      -0.10682114213705063,\n      -0.06676556915044785,\n      -0.10860744118690491,\n      -0.17082661390304565,\n      0.060950785875320435,\n      -0.07224663347005844,\n      -0.1074686124920845,\n      -0.02194262109696865,\n      -0.0892506018280983,\n      -0.02195132151246071,\n      -0.1726607382297516,\n      -0.13450457155704498,\n      -0.15154820680618286,\n      -0.09303230792284012,\n      0.3653162717819214,\n      -0.24388758838176727,\n      -0.054968852549791336,\n      -0.15099693834781647,\n      -0.015017418190836906,\n      -0.11232317239046097,\n      -0.06354447454214096,\n      -0.0579499788582325,\n      0.007754788268357515,\n      0.043555036187171936,\n      0.051395367830991745,\n      0.060887496918439865,\n      -0.009153520688414574,\n      0.15599653124809265,\n      0.13270898163318634,\n      0.16940951347351074,\n      -0.0014778918121010065,\n      0.0657639130949974,\n      -0.07633720338344574,\n      -0.03261639550328255,\n      -0.01301099918782711,\n      0.020101359114050865,\n      -0.07984691858291626,\n      -0.012234443798661232,\n      -0.04287733510136604,\n      -0.03666366636753082,\n      -0.10489250719547272,\n      0.045357584953308105,\n      -0.06460735201835632,\n      -0.04768995940685272,\n      -0.0281835924834013,\n      0.12567001581192017,\n      0.03988686576485634,\n      -0.028772149235010147,\n      0.1707865297794342,\n      0.04830113798379898,\n      0.11896225064992905,\n      0.16648396849632263,\n      -0.03882865980267525,\n      0.07530958205461502,\n      0.10413394123315811,\n      0.08638788014650345,\n      0.09512016922235489,\n      -0.014252379536628723,\n      -0.005072268657386303,\n      -0.03326993063092232,\n      -0.11125411093235016,\n      0.05086936429142952,\n      0.01856340654194355,\n      0.12529009580612183,\n      0.06389621645212173,\n      -0.05825243890285492,\n      0.08153748512268066,\n      -0.10739601403474808,\n      0.05976809561252594,\n      0.09238005429506302,\n      -0.11033104360103607,\n      0.00800302904099226,\n      0.06667537987232208,\n      0.08141142129898071,\n      -0.011842228472232819,\n      0.05487179011106491,\n      0.0536310076713562,\n      -0.06962854415178299,\n      0.1128469705581665,\n      0.11778632551431656,\n      -0.10931040346622467,\n      -0.03443685546517372,\n      -0.0196479894220829,\n      -0.022577201947569847,\n      -0.0635029673576355,\n      0.24018341302871704,\n      -0.006975051946938038,\n      -0.060171738266944885,\n      0.010057041421532631,\n      -0.050745200365781784,\n      -0.11873604357242584,\n      -0.020173143595457077,\n      -0.09327148646116257,\n      -0.004172771703451872,\n      -0.1300811767578125,\n      -0.09999451786279678,\n      0.16104446351528168,\n      0.13356232643127441,\n      0.04723120480775833,\n      0.0144193759188056,\n      -0.052049607038497925,\n      0.003205859800800681,\n      0.17194251716136932,\n      0.009132148697972298,\n      0.2101345658302307,\n      0.015611493028700352,\n      0.015451905317604542,\n      0.005963488947600126,\n      0.18852286040782928,\n      0.026152128353714943,\n      0.0742110088467598,\n      0.09444204717874527,\n      -0.07555415481328964,\n      0.005528077017515898,\n      0.1294989138841629,\n      -0.1931217461824417,\n      -0.08145268261432648,\n      0.20110251009464264,\n      0.08511421084403992,\n      0.20854686200618744,\n      0.13486620783805847,\n      -0.06554706394672394,\n      -0.09647770971059799,\n      0.08616632223129272,\n      -0.05031086504459381\n    ],\n    [\n      0.15446524322032928,\n      0.10569237917661667,\n      0.14586995542049408,\n      0.18309636414051056,\n      -0.05096958950161934,\n      -0.000602386484388262,\n      0.024916863068938255,\n      -0.06247211992740631,\n      -0.09048409014940262,\n      0.11333248764276505,\n      0.1823674589395523,\n      0.041466716676950455,\n      0.1707185059785843,\n      0.07876964658498764,\n      -0.41506439447402954,\n      0.3959105312824249,\n      0.11539217084646225,\n      0.04809826985001564,\n      0.03189663961529732,\n      0.14770670235157013,\n      0.0715104341506958,\n      -0.0016667735762894154,\n      -0.054656848311424255,\n      0.0020525024738162756,\n      -0.07350704073905945,\n      -0.13288471102714539,\n      0.018283378332853317,\n      0.27374666929244995,\n      0.19083243608474731,\n      -0.20861662924289703,\n      0.0877080038189888,\n      -0.12167912721633911,\n      -0.038734883069992065,\n      -0.10069947689771652,\n      -0.008152664639055729,\n      0.022397615015506744,\n      0.058637168258428574,\n      0.05120453983545303,\n      0.08836567401885986,\n      0.006814566906541586,\n      -0.10141527652740479,\n      -0.03482693061232567,\n      0.06452276557683945,\n      0.041538503021001816,\n      -0.015851546078920364,\n      -0.044712867587804794,\n      -0.23176345229148865,\n      -0.16755880415439606,\n      0.047384653240442276,\n      -0.2612302303314209,\n      -0.07377898693084717,\n      0.010024218820035458,\n      -0.08724287152290344,\n      -0.09298613667488098,\n      -0.06227051839232445,\n      -0.04864178225398064,\n      -0.2325555682182312,\n      -0.18277914822101593,\n      -0.030148813501000404,\n      -0.024266405031085014,\n      -0.20979508757591248,\n      0.0639595314860344,\n      0.03153283894062042,\n      0.04924226924777031,\n      -0.12532642483711243,\n      -0.12536554038524628,\n      -0.09434453397989273,\n      0.022855691611766815,\n      -0.09799782931804657,\n      0.04095671325922012,\n      -0.02258642576634884,\n      0.013322697952389717,\n      0.050713784992694855,\n      0.06686308234930038,\n      0.009335312992334366,\n      -0.19306328892707825,\n      -0.0810508280992508,\n      -0.006139201577752829,\n      -0.05573801323771477,\n      0.13153252005577087,\n      -0.0665871798992157,\n      -0.031424131244421005,\n      0.013300362974405289,\n      -0.13212831318378448,\n      0.20857088267803192,\n      0.0819692462682724,\n      0.07535377144813538,\n      0.06210412085056305,\n      0.059861842542886734,\n      -0.004452084191143513,\n      0.13167528808116913,\n      0.013952439650893211,\n      0.11024905741214752,\n      -0.008354006335139275,\n      0.07034356147050858,\n      0.11708887666463852,\n      0.06044555455446243,\n      0.05241016298532486,\n      0.006299173925071955,\n      -0.055272556841373444,\n      0.03658902272582054,\n      -0.05234810337424278,\n      0.016251321882009506,\n      0.0343080535531044,\n      0.1715749055147171,\n      -0.022126834839582443,\n      -0.08304231613874435,\n      0.06781508028507233,\n      0.02699187956750393,\n      0.08202409744262695,\n      0.06404630839824677,\n      0.04987000674009323,\n      -0.04291023313999176,\n      0.032722458243370056,\n      -0.04553169012069702,\n      0.37636181712150574,\n      0.17859353125095367,\n      0.012070093303918839,\n      -0.11451973766088486,\n      -0.03145325556397438,\n      -0.21439185738563538,\n      0.07023901492357254,\n      0.03653501346707344,\n      -0.12445783615112305,\n      -0.037765149027109146\n    ],\n    [\n      -0.17690593004226685,\n      -0.09500260651111603,\n      -0.10327127575874329,\n      -0.1413431465625763,\n      0.08156973123550415,\n      -0.0166462492197752,\n      -0.027969449758529663,\n      -0.06638248264789581,\n      -0.13999579846858978,\n      -0.10651828348636627,\n      -0.15174855291843414,\n      -0.0765429437160492,\n      -0.24478328227996826,\n      -0.0837203860282898,\n      0.3069396913051605,\n      -0.36330661177635193,\n      -0.07365070283412933,\n      -0.10908236354589462,\n      -0.020941171795129776,\n      -0.12082263082265854,\n      -0.060102421790361404,\n      -0.010435743257403374,\n      -0.08702290058135986,\n      0.14058679342269897,\n      0.026501284912228584,\n      -0.023462552577257156,\n      0.008657447062432766,\n      0.2709980607032776,\n      0.035988617688417435,\n      0.1931547075510025,\n      0.008347833529114723,\n      0.11306256800889969,\n      -0.012145514599978924,\n      -0.03759632632136345,\n      -0.03975324705243111,\n      0.03453530743718147,\n      -0.04717954248189926,\n      -0.08571761846542358,\n      0.04543696343898773,\n      -0.0045494697988033295,\n      -1.1399029062886257e-05,\n      0.06612768769264221,\n      -0.016627894714474678,\n      0.0990501195192337,\n      -0.031527601182460785,\n      0.15311500430107117,\n      0.04128909111022949,\n      0.03636141121387482,\n      0.10105472803115845,\n      0.07675786316394806,\n      0.05202387645840645,\n      0.21514399349689484,\n      0.03390946239233017,\n      0.050770353525877,\n      0.05231600999832153,\n      0.11286046355962753,\n      -0.061252743005752563,\n      0.07876426726579666,\n      -0.0784507691860199,\n      0.08733873069286346,\n      -0.0984300971031189,\n      0.04172476381063461,\n      -0.016909193247556686,\n      0.040099628269672394,\n      -0.04528702050447464,\n      -0.09365995228290558,\n      -0.04865468665957451,\n      0.06293964385986328,\n      -0.017773495987057686,\n      -0.05189911648631096,\n      -0.04459676146507263,\n      0.10924439877271652,\n      -0.010509088635444641,\n      0.06775148212909698,\n      0.002928146393969655,\n      -0.05558372661471367,\n      -0.03400994464755058,\n      0.08019843697547913,\n      0.03571015223860741,\n      0.0015286506386473775,\n      0.03196452558040619,\n      -0.0017868517898023129,\n      -0.04347340017557144,\n      -0.08645334839820862,\n      -0.11191393435001373,\n      0.22191312909126282,\n      -0.09489210695028305,\n      -0.11705968528985977,\n      -0.03774271532893181,\n      -0.053352195769548416,\n      -0.032835252583026886,\n      -0.06270559132099152,\n      -0.07980797439813614,\n      0.03196962550282478,\n      -0.03266272321343422,\n      0.03494652733206749,\n      0.24681155383586884,\n      0.16941003501415253,\n      0.05731755867600441,\n      0.04120340198278427,\n      -0.06996434181928635,\n      0.04886838048696518,\n      0.1627730280160904,\n      0.014691587537527084,\n      0.23379158973693848,\n      0.030295109376311302,\n      -0.018281331285834312,\n      0.11608424782752991,\n      0.1286119967699051,\n      0.014302251860499382,\n      0.09659427404403687,\n      -0.004791191313415766,\n      0.046731963753700256,\n      -0.07079095393419266,\n      0.08631011098623276,\n      -0.21433214843273163,\n      -0.024053113535046577,\n      0.08179788291454315,\n      0.22929348051548004,\n      0.17632663249969482,\n      0.113583043217659,\n      0.007523260544985533,\n      0.021180691197514534,\n      0.07463021576404572,\n      0.012088554911315441\n    ],\n    [\n      0.23696747422218323,\n      0.07465962320566177,\n      0.10441553592681885,\n      0.2329828441143036,\n      0.02159794420003891,\n      -0.020112821832299232,\n      -0.027505217120051384,\n      -0.055268462747335434,\n      -0.0969277173280716,\n      0.0783691257238388,\n      0.10750062018632889,\n      0.10697893798351288,\n      0.26564711332321167,\n      0.03217262402176857,\n      -0.3313097357749939,\n      0.3924195468425751,\n      -0.027467893436551094,\n      0.11331437528133392,\n      0.006756044924259186,\n      0.1312204748392105,\n      0.06903347373008728,\n      0.01793093793094158,\n      -0.12334491312503815,\n      0.057909999042749405,\n      -0.018598083406686783,\n      -0.05672798305749893,\n      -0.08619891107082367,\n      0.16873757541179657,\n      0.05827990546822548,\n      -0.08239223808050156,\n      0.02473173849284649,\n      -0.11101804673671722,\n      -0.05880861356854439,\n      0.04312901943922043,\n      0.003009855980053544,\n      0.04482370987534523,\n      0.006673149764537811,\n      0.03707252815365791,\n      0.0016769004287198186,\n      -0.010961491614580154,\n      -0.10712078213691711,\n      -0.03875276818871498,\n      0.08205519616603851,\n      -0.0744447261095047,\n      -0.007697250694036484,\n      0.02365666814148426,\n      -0.21553684771060944,\n      -0.12853357195854187,\n      0.021701060235500336,\n      -0.1311010867357254,\n      -0.10541870445013046,\n      -0.0029096996877342463,\n      -0.18157972395420074,\n      -0.020420128479599953,\n      -0.0013038022443652153,\n      -0.057712435722351074,\n      -0.16280575096607208,\n      -0.09502073377370834,\n      0.07415524125099182,\n      -0.07182063162326813,\n      -0.13454782962799072,\n      0.04642250016331673,\n      -0.00036951852962374687,\n      0.022586941719055176,\n      -0.09258563071489334,\n      -0.08426324278116226,\n      -0.00991367269307375,\n      -0.052464284002780914,\n      -0.09900158643722534,\n      -0.016163814812898636,\n      -0.10465935617685318,\n      -0.018841462209820747,\n      -0.09598565846681595,\n      0.016454080119729042,\n      -0.046500083059072495,\n      -0.03359752893447876,\n      -0.07100311666727066,\n      -0.08543319255113602,\n      -0.060460928827524185,\n      0.13366341590881348,\n      -0.05327780917286873,\n      -0.05292140692472458,\n      -0.08494187146425247,\n      -0.13995975255966187,\n      0.15048067271709442,\n      0.048424333333969116,\n      0.06486320495605469,\n      0.019449638202786446,\n      0.05182584002614021,\n      0.09336718916893005,\n      0.05456710234284401,\n      0.027263546362519264,\n      0.045005276799201965,\n      -0.003475189208984375,\n      0.02480221353471279,\n      0.03669772297143936,\n      -0.08994291722774506,\n      -0.0059852199628949165,\n      0.010004570707678795,\n      -0.04146968945860863,\n      0.03357689455151558,\n      0.00545071717351675,\n      0.06460583955049515,\n      0.016609322279691696,\n      0.14722590148448944,\n      -0.024151692166924477,\n      0.042567379772663116,\n      0.029930179938673973,\n      0.0927664041519165,\n      0.043299999088048935,\n      0.01542019285261631,\n      -0.021942399442195892,\n      0.06214292719960213,\n      0.014108551666140556,\n      -0.07158942520618439,\n      0.4497312605381012,\n      0.21659140288829803,\n      0.05007762089371681,\n      -0.025539718568325043,\n      0.062169477343559265,\n      -0.17800892889499664,\n      0.07946053147315979,\n      0.0008546533645130694,\n      -0.09257936477661133,\n      -0.048029690980911255\n    ],\n    [\n      0.09203194826841354,\n      0.06333278119564056,\n      0.16478502750396729,\n      0.16728021204471588,\n      -0.11611168086528778,\n      -0.0046068583615124226,\n      0.11357943713665009,\n      0.10863181948661804,\n      0.02806330844759941,\n      0.16396483778953552,\n      0.06236458942294121,\n      0.14239071309566498,\n      0.14679954946041107,\n      0.14738352596759796,\n      -0.3848274052143097,\n      0.33193567395210266,\n      -0.02123940922319889,\n      0.14460362493991852,\n      0.042719148099422455,\n      0.025268876925110817,\n      -0.041794415563344955,\n      0.1556926965713501,\n      0.13396187126636505,\n      0.07775592058897018,\n      -0.12198823690414429,\n      -0.03433791548013687,\n      0.07583796232938766,\n      -0.008067804388701916,\n      -0.09435304999351501,\n      -0.13931067287921906,\n      -0.04742024466395378,\n      -0.09348990768194199,\n      0.013126731850206852,\n      0.024628840386867523,\n      0.11169443279504776,\n      0.023964472115039825,\n      -0.02802928164601326,\n      0.04395192489027977,\n      0.057715099304914474,\n      -0.03869546577334404,\n      0.03667692840099335,\n      0.00037364009767770767,\n      0.0029450540896505117,\n      0.03599121794104576,\n      -0.0016067849937826395,\n      -0.0654359832406044,\n      -0.06679454445838928,\n      -0.02052920125424862,\n      -0.03423226624727249,\n      -0.008069401606917381,\n      -0.07526945322751999,\n      -0.05344893038272858,\n      -0.1959042102098465,\n      -0.04373863339424133,\n      -0.10712847858667374,\n      -0.12552469968795776,\n      -0.16904133558273315,\n      -0.03066583164036274,\n      0.09101571142673492,\n      0.009524325840175152,\n      0.007267094682902098,\n      0.05406075716018677,\n      0.030712073668837547,\n      0.06330481171607971,\n      0.06202022358775139,\n      -0.07150325179100037,\n      0.019887683913111687,\n      -0.05576975271105766,\n      -0.05684658885002136,\n      -0.0016373719554394484,\n      -0.010287979617714882,\n      -0.045921359211206436,\n      -0.0889773741364479,\n      -0.049751125276088715,\n      0.07396168261766434,\n      -0.021183403208851814,\n      0.07295475900173187,\n      0.027319947257637978,\n      -0.10655627399682999,\n      0.035979922860860825,\n      -0.10216767340898514,\n      -0.06946524232625961,\n      -0.004723413847386837,\n      0.005536967422813177,\n      0.01976468414068222,\n      -0.154906764626503,\n      -0.028339089825749397,\n      0.08848405629396439,\n      0.09743642061948776,\n      0.10212137550115585,\n      -0.01749234087765217,\n      0.07025853544473648,\n      0.09280269593000412,\n      0.12066267430782318,\n      0.008451041765511036,\n      -0.0019742774311453104,\n      0.10049881786108017,\n      0.11276926845312119,\n      0.08195491880178452,\n      -0.06862981617450714,\n      -0.06056766211986542,\n      0.06769992411136627,\n      0.15599380433559418,\n      0.09275117516517639,\n      0.11378434300422668,\n      -0.026954101398587227,\n      0.018882067874073982,\n      0.06967614591121674,\n      0.08287940919399261,\n      0.016989868134260178,\n      -0.01637277752161026,\n      0.04816148430109024,\n      0.06191294267773628,\n      -0.08706742525100708,\n      0.08349835872650146,\n      0.27080971002578735,\n      0.10345257073640823,\n      -0.007903842255473137,\n      -0.06219799816608429,\n      -0.015563135035336018,\n      -0.0877390056848526,\n      -0.011394912376999855,\n      0.041459403932094574,\n      -0.10383209586143494,\n      0.02625102549791336\n    ],\n    [\n      0.25924932956695557,\n      -0.022748012095689774,\n      0.15744727849960327,\n      0.09984918683767319,\n      -0.057783447206020355,\n      0.026841307058930397,\n      0.00021927656780462712,\n      0.03600182756781578,\n      -0.0879562497138977,\n      0.05636162683367729,\n      0.16249436140060425,\n      0.02420605719089508,\n      0.26540228724479675,\n      0.1522453874349594,\n      -0.47427302598953247,\n      0.41614800691604614,\n      -0.024497179314494133,\n      0.03299970179796219,\n      0.012985986657440662,\n      0.0881262794137001,\n      0.01341413427144289,\n      0.06628690659999847,\n      -0.027254020795226097,\n      0.012060868553817272,\n      -0.13972826302051544,\n      0.04714176803827286,\n      0.0306711383163929,\n      0.29178574681282043,\n      0.13651366531848907,\n      -0.22042328119277954,\n      0.049873873591423035,\n      -0.036065343767404556,\n      -0.05837312340736389,\n      0.04614776745438576,\n      0.032127220183610916,\n      -0.12010258436203003,\n      -0.10517343878746033,\n      -0.012200547382235527,\n      -0.014656728133559227,\n      -0.015601550228893757,\n      -0.036677759140729904,\n      0.000752217136323452,\n      0.01774185337126255,\n      0.02852797508239746,\n      -0.01176401786506176,\n      -0.06352685391902924,\n      -0.08132569491863251,\n      -0.16135205328464508,\n      0.013983442448079586,\n      -0.16851206123828888,\n      -0.06832984834909439,\n      0.011991197243332863,\n      -0.15068885684013367,\n      -0.06875675171613693,\n      0.005746690556406975,\n      -0.10332793742418289,\n      -0.15192104876041412,\n      -0.010208413004875183,\n      -0.010494240559637547,\n      0.023998340591788292,\n      -0.11353535205125809,\n      0.12420427054166794,\n      0.018013061955571175,\n      0.0777100995182991,\n      -0.01670989766716957,\n      -0.008436264470219612,\n      -0.008869004435837269,\n      -0.01634048856794834,\n      -0.10743095725774765,\n      0.014797722920775414,\n      -0.009089180268347263,\n      0.07850173860788345,\n      -0.033012986183166504,\n      -0.018184572458267212,\n      -0.0190341267734766,\n      -0.03606635704636574,\n      -0.041206102818250656,\n      0.04035138711333275,\n      -0.02884470671415329,\n      -0.008340567350387573,\n      -0.045169200748205185,\n      -0.11886017769575119,\n      -0.0003102496266365051,\n      -0.024289945140480995,\n      0.16683368384838104,\n      0.06062996760010719,\n      -0.028269121423363686,\n      0.0913572683930397,\n      0.01539697591215372,\n      -0.007034356240183115,\n      0.0659569501876831,\n      0.12004093080759048,\n      0.07277055084705353,\n      -0.016684647649526596,\n      0.13322336971759796,\n      0.11730585992336273,\n      -0.016156012192368507,\n      0.015905722975730896,\n      0.08772298693656921,\n      -0.06604229658842087,\n      0.0054864720441401005,\n      0.04168182983994484,\n      0.03036101721227169,\n      0.07476072013378143,\n      0.12109202146530151,\n      -0.06662669032812119,\n      -0.0992099866271019,\n      0.07829701900482178,\n      0.10579019039869308,\n      0.10615658015012741,\n      0.05153459683060646,\n      0.018400033935904503,\n      -0.05647359788417816,\n      -0.02384733222424984,\n      0.04775967821478844,\n      0.2895370423793793,\n      0.09651260077953339,\n      0.0822312980890274,\n      -0.12501733005046844,\n      -0.052925124764442444,\n      -0.23010385036468506,\n      0.029940251260995865,\n      0.04771461710333824,\n      -0.06084898114204407,\n      0.050896596163511276\n    ],\n    [\n      0.12645217776298523,\n      0.0027959535364061594,\n      0.011091155000030994,\n      0.15034647285938263,\n      -0.13690197467803955,\n      0.12010928988456726,\n      0.08456526696681976,\n      0.026242036372423172,\n      -0.05808871611952782,\n      0.12990793585777283,\n      -0.047724638134241104,\n      0.19433680176734924,\n      0.26268038153648376,\n      0.07023940235376358,\n      -0.29193413257598877,\n      0.3529791235923767,\n      -0.14544978737831116,\n      -0.008769199252128601,\n      0.06339703500270844,\n      0.035828329622745514,\n      -0.09612511843442917,\n      0.07359284162521362,\n      0.011385100893676281,\n      0.12189237028360367,\n      -0.23949746787548065,\n      -0.02824157476425171,\n      -0.0833742544054985,\n      0.27074381709098816,\n      0.03157266229391098,\n      -0.38284021615982056,\n      0.0422925241291523,\n      -0.0764111652970314,\n      -0.0940665602684021,\n      0.02542499452829361,\n      -0.04163547605276108,\n      -0.016996772959828377,\n      -0.14298930764198303,\n      0.024329472333192825,\n      0.0024740505032241344,\n      -0.1182384192943573,\n      -0.025063736364245415,\n      0.057289063930511475,\n      0.026516390964388847,\n      -0.046731483191251755,\n      0.08271932601928711,\n      -0.0011905906721949577,\n      -0.12100639939308167,\n      -0.06187868490815163,\n      -0.07224489003419876,\n      -0.16032974421977997,\n      -0.15145893394947052,\n      -0.06799361854791641,\n      -0.19062112271785736,\n      -0.11364459246397018,\n      -0.17572130262851715,\n      -0.10367448627948761,\n      -0.1791456937789917,\n      -0.1373053938150406,\n      0.06876609474420547,\n      -0.130240336060524,\n      -0.0018812966300174594,\n      -0.09617296606302261,\n      0.08497191220521927,\n      0.034553855657577515,\n      -0.10387817025184631,\n      -0.15453208982944489,\n      -0.16636763513088226,\n      -0.08117037266492844,\n      0.0053865499794483185,\n      -0.07165801525115967,\n      -0.24810680747032166,\n      -0.00494221318513155,\n      -0.049749620258808136,\n      0.02373206801712513,\n      -0.20519720017910004,\n      -0.09482613950967789,\n      -0.1254650056362152,\n      -0.13743561506271362,\n      -0.18704968690872192,\n      0.14391593635082245,\n      -0.008312955498695374,\n      -0.11160621047019958,\n      0.015091490931808949,\n      -0.13757899403572083,\n      0.13124683499336243,\n      0.10392402112483978,\n      -0.02975834533572197,\n      -0.1294584572315216,\n      0.06572100520133972,\n      0.09539361298084259,\n      -0.0940503403544426,\n      -0.10658802092075348,\n      0.09532097727060318,\n      0.12340898811817169,\n      0.06401870399713516,\n      0.06900055706501007,\n      0.024821091443300247,\n      -0.011869941838085651,\n      0.13110366463661194,\n      0.022186601534485817,\n      -0.12988245487213135,\n      -0.12847302854061127,\n      0.07704997062683105,\n      -0.03564511612057686,\n      0.2148486226797104,\n      0.00757690379396081,\n      0.08787672221660614,\n      0.03761201351881027,\n      0.04270128160715103,\n      -0.04425302520394325,\n      0.1426716446876526,\n      0.037373438477516174,\n      -0.08400554209947586,\n      0.03792523965239525,\n      -0.042938780039548874,\n      0.15159666538238525,\n      0.22092905640602112,\n      0.022994844242930412,\n      -0.19470587372779846,\n      -0.03136902675032616,\n      -0.2480032742023468,\n      -0.10208150744438171,\n      0.012428293004631996,\n      -0.24412569403648376,\n      -0.06864319741725922\n    ],\n    [\n      -0.20875538885593414,\n      -0.12109220027923584,\n      -0.13000603020191193,\n      -0.10797574371099472,\n      -0.06720168888568878,\n      -0.13693861663341522,\n      0.05621441826224327,\n      -0.02088846266269684,\n      -0.09669440984725952,\n      0.09889446943998337,\n      -0.10497305542230606,\n      -0.14158548414707184,\n      -0.22646549344062805,\n      -0.12566739320755005,\n      0.2864845395088196,\n      -0.34642156958580017,\n      -0.004671880044043064,\n      -0.03805074840784073,\n      0.02037375420331955,\n      -0.06704599410295486,\n      0.027508506551384926,\n      -0.010346002876758575,\n      0.03728120028972626,\n      0.02735086902976036,\n      0.042847659438848495,\n      -0.05833607167005539,\n      -0.08935368061065674,\n      0.3203830420970917,\n      0.18849211931228638,\n      0.15789179503917694,\n      0.03584269434213638,\n      -0.031180402263998985,\n      0.058503154665231705,\n      -0.045779939740896225,\n      -0.036030564457178116,\n      0.07274793088436127,\n      -0.0369282066822052,\n      -0.11366671323776245,\n      -0.061714690178632736,\n      -0.02318750135600567,\n      0.03080541081726551,\n      -0.0010196429211646318,\n      -0.00015591902774758637,\n      0.028549471870064735,\n      -0.03315826132893562,\n      0.09749331325292587,\n      0.01676161400973797,\n      -0.02992762252688408,\n      0.0944736897945404,\n      -0.02977066859602928,\n      0.07907731831073761,\n      0.052654147148132324,\n      -0.0245624128729105,\n      0.09277316927909851,\n      0.1097157746553421,\n      -0.01952626369893551,\n      0.03547978028655052,\n      -0.06883249431848526,\n      -0.10704787820577621,\n      -0.03923780843615532,\n      -0.141328364610672,\n      0.03764958679676056,\n      0.011584574356675148,\n      -0.00781845860183239,\n      -0.08765080571174622,\n      0.01926576904952526,\n      0.06469030678272247,\n      -0.048106707632541656,\n      -0.062954843044281,\n      -0.005414844956248999,\n      0.016666190698742867,\n      0.029552320018410683,\n      0.003587098093703389,\n      0.13072529435157776,\n      0.03880718722939491,\n      -0.0564877986907959,\n      0.023155150935053825,\n      0.022646620869636536,\n      0.11961201578378677,\n      0.13435518741607666,\n      0.003333760192617774,\n      -0.09514851868152618,\n      0.037344541400671005,\n      -0.07698747515678406,\n      0.0770469605922699,\n      0.37808841466903687,\n      -0.010183347389101982,\n      -0.04519560560584068,\n      -0.11864836513996124,\n      -0.022661464288830757,\n      -0.03737436980009079,\n      0.0025491532869637012,\n      -0.11471207439899445,\n      -0.10109119117259979,\n      -0.0871337428689003,\n      0.01486257091164589,\n      0.07632981240749359,\n      0.11606348305940628,\n      0.04502607509493828,\n      0.05393363535404205,\n      0.0044639743864536285,\n      -0.05066172778606415,\n      0.13043275475502014,\n      0.0002495921216905117,\n      0.21727333962917328,\n      0.07323914766311646,\n      0.04177241027355194,\n      -0.009150343015789986,\n      0.06510623544454575,\n      0.052558716386556625,\n      0.20454499125480652,\n      0.09098879247903824,\n      0.047933146357536316,\n      -0.024915656074881554,\n      0.07521788030862808,\n      -0.2148088663816452,\n      0.022834181785583496,\n      0.021973324939608574,\n      0.20239746570587158,\n      0.28304150700569153,\n      0.07693447917699814,\n      -0.06537734717130661,\n      -0.014778164215385914,\n      0.05625053867697716,\n      -0.08415418863296509\n    ],\n    [\n      0.1289297193288803,\n      -0.0510265938937664,\n      0.14155124127864838,\n      0.21619828045368195,\n      0.05104195699095726,\n      0.021092213690280914,\n      -0.054429035633802414,\n      -0.018577195703983307,\n      0.10317151993513107,\n      0.018848635256290436,\n      0.0637442097067833,\n      0.19154922664165497,\n      0.2733992338180542,\n      0.14793004095554352,\n      -0.33813634514808655,\n      0.38181689381599426,\n      0.025252746418118477,\n      0.07132963091135025,\n      0.0010701504070311785,\n      0.09023907780647278,\n      0.04180654138326645,\n      0.08883386850357056,\n      -0.04159676656126976,\n      0.01630222797393799,\n      -0.02339898608624935,\n      -0.002796028507873416,\n      0.0129169886931777,\n      0.020168522372841835,\n      0.056960973888635635,\n      -0.21695534884929657,\n      0.011285500600934029,\n      0.003441635752096772,\n      -0.004015868529677391,\n      0.034731071442365646,\n      0.04686439409852028,\n      0.07924994081258774,\n      0.0661841332912445,\n      -0.009234397672116756,\n      -0.0005080794217064977,\n      0.10227346420288086,\n      0.047229550778865814,\n      -0.04277860373258591,\n      -0.0277111679315567,\n      0.03131967782974243,\n      0.08806343376636505,\n      -0.048152998089790344,\n      -0.020083507522940636,\n      -0.07657334953546524,\n      -0.04351944848895073,\n      -0.09409543871879578,\n      -0.15031342208385468,\n      -0.06286738067865372,\n      -0.17930638790130615,\n      -0.1408490240573883,\n      -0.0805475041270256,\n      -0.08739183098077774,\n      -0.25064796209335327,\n      -0.12028887122869492,\n      0.11516401916742325,\n      -0.06484083086252213,\n      -0.09643018245697021,\n      0.054809220135211945,\n      -0.0516410693526268,\n      0.013938942924141884,\n      -0.023090383037924767,\n      0.05364808067679405,\n      -0.0314524807035923,\n      -0.07441136986017227,\n      -0.019915420562028885,\n      0.04142436012625694,\n      0.020038209855556488,\n      0.015647592023015022,\n      -0.0509299598634243,\n      -0.014618958346545696,\n      0.002902178792282939,\n      -0.05403715372085571,\n      -0.016986850649118423,\n      0.004416691605001688,\n      -0.09959153831005096,\n      0.1126742884516716,\n      -0.09913984686136246,\n      0.002741614356637001,\n      -0.020378688350319862,\n      0.0045771729201078415,\n      0.015630412846803665,\n      -0.02664131484925747,\n      0.036538656800985336,\n      -0.010894284583628178,\n      0.06856796890497208,\n      0.06820028275251389,\n      0.029371298849582672,\n      0.06932032853364944,\n      -0.040594473481178284,\n      -0.11728140711784363,\n      0.04020596668124199,\n      0.024875478819012642,\n      0.10164153575897217,\n      0.12980982661247253,\n      0.038931138813495636,\n      -0.09290005266666412,\n      -0.08831574022769928,\n      0.02601100318133831,\n      0.15509440004825592,\n      0.0483856126666069,\n      0.13841992616653442,\n      -0.040239959955215454,\n      0.0753767266869545,\n      -0.019247982650995255,\n      0.09135298430919647,\n      0.16808196902275085,\n      -0.0009920946322381496,\n      0.028672082349658012,\n      0.050208885222673416,\n      -0.046074554324150085,\n      -0.0633450374007225,\n      0.3747306168079376,\n      0.10271862894296646,\n      0.1173640713095665,\n      -0.008726698346436024,\n      -0.14569804072380066,\n      -0.28959646821022034,\n      0.049706216901540756,\n      0.06923413276672363,\n      -0.11102648824453354,\n      -0.06346719712018967\n    ],\n    [\n      -0.2050354778766632,\n      -0.12180712074041367,\n      -0.10028007626533508,\n      -0.11040954291820526,\n      0.06875398010015488,\n      0.030420826748013496,\n      -0.00462341820821166,\n      -0.040673233568668365,\n      0.025486186146736145,\n      -0.09621506929397583,\n      -0.20208066701889038,\n      -0.11423381417989731,\n      -0.14437370002269745,\n      -0.13061383366584778,\n      0.2450808584690094,\n      -0.26408907771110535,\n      0.07247984409332275,\n      0.04647531360387802,\n      -0.09166800230741501,\n      -0.11052684485912323,\n      0.0754702240228653,\n      0.06739436835050583,\n      -0.06246069446206093,\n      0.12391025573015213,\n      -0.07663892209529877,\n      0.04009656608104706,\n      -0.13785874843597412,\n      0.16262586414813995,\n      0.11131042242050171,\n      0.13789379596710205,\n      -0.01423368789255619,\n      0.11538495868444443,\n      -0.025339724496006966,\n      0.02803613618016243,\n      0.013691995292901993,\n      -0.020357931032776833,\n      0.019281018525362015,\n      -0.008191028609871864,\n      -0.061706069856882095,\n      -0.0765763521194458,\n      -0.03343875706195831,\n      -0.0895870253443718,\n      0.016299936920404434,\n      0.04244007170200348,\n      -0.03971895948052406,\n      0.12508107721805573,\n      -0.036845169961452484,\n      0.0026196762919425964,\n      0.1892620474100113,\n      -0.04801582545042038,\n      0.0020209166686981916,\n      0.1448460966348648,\n      0.057393744587898254,\n      0.01904227025806904,\n      0.15678627789020538,\n      0.021835077553987503,\n      -0.0032925812993198633,\n      -0.05524245649576187,\n      -0.015492066740989685,\n      0.031425412744283676,\n      -0.04381692782044411,\n      -0.0040620360523462296,\n      0.1182037964463234,\n      0.06806594133377075,\n      0.06203843280673027,\n      -0.026199769228696823,\n      -0.02805618941783905,\n      0.007394066080451012,\n      0.040740277618169785,\n      0.07695972919464111,\n      -0.05553438514471054,\n      0.08220745623111725,\n      -0.10209840536117554,\n      0.020315220579504967,\n      -0.0581759512424469,\n      -0.04085434973239899,\n      0.06710337847471237,\n      0.04792495444417,\n      -0.006973738316446543,\n      0.0746968612074852,\n      -0.10749166458845139,\n      -0.0006506647914648056,\n      0.044779181480407715,\n      0.08012255281209946,\n      -0.0062096817418932915,\n      0.2349897176027298,\n      -0.0060500227846205235,\n      0.00759811932221055,\n      -0.09075666964054108,\n      -0.04410673305392265,\n      -0.04635477066040039,\n      -0.10946926474571228,\n      -0.09198347479104996,\n      -0.09165745228528976,\n      -0.09401609003543854,\n      -0.010446438565850258,\n      0.08100877702236176,\n      0.11976468563079834,\n      -0.026539336889982224,\n      -0.06142245605587959,\n      -0.03072015568614006,\n      0.11957032978534698,\n      0.13398228585720062,\n      0.009092497639358044,\n      0.1698962301015854,\n      0.012159867212176323,\n      0.0130171999335289,\n      0.04278552159667015,\n      0.15355125069618225,\n      0.12131521850824356,\n      0.0851612240076065,\n      0.007289071101695299,\n      -0.005576500203460455,\n      -0.029234007000923157,\n      0.1214267760515213,\n      -0.34891384840011597,\n      0.011676295660436153,\n      0.16163772344589233,\n      0.22045934200286865,\n      0.23793365061283112,\n      0.15881335735321045,\n      -0.03452672064304352,\n      -0.013529878109693527,\n      0.03840915113687515,\n      -0.051669951528310776\n    ],\n    [\n      -0.21156787872314453,\n      -0.07695869356393814,\n      -0.16271108388900757,\n      -0.12764161825180054,\n      0.03454315662384033,\n      0.03055010922253132,\n      0.006888644769787788,\n      -0.07951998710632324,\n      -0.09295162558555603,\n      0.011889088898897171,\n      -0.10232905298471451,\n      -0.15015660226345062,\n      -0.098964624106884,\n      -0.03538433834910393,\n      0.3983634114265442,\n      -0.3646888732910156,\n      0.03580893203616142,\n      -0.1070321798324585,\n      0.019220875576138496,\n      -0.12001793831586838,\n      0.0003122898342553526,\n      0.024948224425315857,\n      -0.08011643588542938,\n      0.018840795382857323,\n      0.07014883309602737,\n      0.10520444065332413,\n      -0.0638217106461525,\n      0.3421522080898285,\n      0.045522406697273254,\n      0.15524564683437347,\n      -0.008574018254876137,\n      0.06393091380596161,\n      0.041706327348947525,\n      -0.0363486185669899,\n      0.06858580559492111,\n      -0.046883367002010345,\n      -0.03410027548670769,\n      -0.03469426929950714,\n      -0.0037429463118314743,\n      0.00016073888400569558,\n      -0.028061196208000183,\n      0.05667778477072716,\n      -0.08698052167892456,\n      -0.006497474387288094,\n      0.05124855414032936,\n      0.12216753512620926,\n      0.011708950623869896,\n      -0.023199357092380524,\n      0.12277663499116898,\n      0.024807924404740334,\n      0.10153627395629883,\n      0.10280130058526993,\n      0.09171005338430405,\n      0.025224601849913597,\n      0.06727451831102371,\n      0.11033323407173157,\n      0.034728944301605225,\n      0.061933938413858414,\n      0.0118962237611413,\n      0.06455480307340622,\n      -0.0013666938757523894,\n      0.1155562698841095,\n      0.12410836666822433,\n      0.011740394867956638,\n      -0.008435809053480625,\n      0.020834360271692276,\n      0.04642406105995178,\n      -0.011672524735331535,\n      -0.0009383375872857869,\n      0.048478495329618454,\n      0.0018170035909861326,\n      0.02678733319044113,\n      0.0340021550655365,\n      0.14849525690078735,\n      -0.0502055361866951,\n      -0.00032568792812526226,\n      0.05452403053641319,\n      0.08461261540651321,\n      0.12453413009643555,\n      0.06800936907529831,\n      -0.04337825998663902,\n      -0.03355615213513374,\n      -0.03870785981416702,\n      -0.053036123514175415,\n      -0.039264023303985596,\n      0.32957884669303894,\n      0.0195023063570261,\n      0.029755879193544388,\n      -0.13754740357398987,\n      -0.01382820401340723,\n      -0.10086221247911453,\n      0.0287189744412899,\n      0.038878198713064194,\n      0.0045391470193862915,\n      0.02995944209396839,\n      -0.017134057357907295,\n      0.19784067571163177,\n      0.17549827694892883,\n      0.06731542944908142,\n      0.10809952765703201,\n      -0.04915760084986687,\n      -0.04396619647741318,\n      0.17518684267997742,\n      0.08745959401130676,\n      0.16341891884803772,\n      0.01763891614973545,\n      0.005923872347921133,\n      0.0376981757581234,\n      0.05242268741130829,\n      0.06519215553998947,\n      0.10668627917766571,\n      0.06674394011497498,\n      0.015629027038812637,\n      0.049846697598695755,\n      0.08280231803655624,\n      -0.30123406648635864,\n      0.03853544592857361,\n      0.05507868528366089,\n      0.20792439579963684,\n      0.3095318675041199,\n      0.09455614537000656,\n      -0.02471209317445755,\n      0.005077165551483631,\n      0.13382601737976074,\n      -0.004596223589032888\n    ],\n    [\n      -0.2341916859149933,\n      -0.0340234637260437,\n      -0.039826832711696625,\n      -0.2433365136384964,\n      -0.002294820500537753,\n      0.03647927567362785,\n      0.02578771859407425,\n      -0.07039306312799454,\n      -0.08762471377849579,\n      0.056287769228219986,\n      -0.13328640162944794,\n      -0.03818551450967789,\n      -0.15470130741596222,\n      -0.15879155695438385,\n      0.242263525724411,\n      -0.3147934079170227,\n      0.05404764786362648,\n      -0.09508893638849258,\n      -0.0471593402326107,\n      -0.10153898596763611,\n      -0.01961263082921505,\n      0.03490135446190834,\n      0.006996071897447109,\n      -0.001982647692784667,\n      0.0668041780591011,\n      -0.03212440386414528,\n      -0.10427433997392654,\n      0.1476086527109146,\n      0.17888647317886353,\n      0.21737323701381683,\n      0.04345580190420151,\n      0.0505833700299263,\n      -0.025285091251134872,\n      -0.05499797314405441,\n      0.04219435900449753,\n      -0.0658792108297348,\n      0.04438783600926399,\n      0.01461539976298809,\n      -0.057667285203933716,\n      0.027133261784911156,\n      -0.03276960179209709,\n      -0.04641091078519821,\n      0.015015209093689919,\n      -0.04240366816520691,\n      0.03940744325518608,\n      0.029293492436408997,\n      -0.06986235827207565,\n      -0.07775039225816727,\n      0.09218995273113251,\n      0.09438710659742355,\n      -0.023352568969130516,\n      0.21942196786403656,\n      0.022105583921074867,\n      0.08370282500982285,\n      0.12383104860782623,\n      0.07257109880447388,\n      -0.006477836985141039,\n      0.004673508461564779,\n      -0.09032614529132843,\n      0.006132453680038452,\n      0.022956810891628265,\n      0.03999609127640724,\n      0.017794989049434662,\n      0.044834017753601074,\n      0.058520905673503876,\n      -0.07447134703397751,\n      0.0661160871386528,\n      -0.09436871856451035,\n      -0.040491197258234024,\n      0.02494169771671295,\n      0.0223835501819849,\n      -0.002324905013665557,\n      0.06050096079707146,\n      0.04485810920596123,\n      0.0038830467965453863,\n      0.07120313495397568,\n      0.07259620726108551,\n      0.04711125046014786,\n      0.03340008854866028,\n      0.11236559599637985,\n      0.034514009952545166,\n      0.02572350949048996,\n      0.04959365352988243,\n      0.002887136535719037,\n      -0.0804467424750328,\n      0.20143775641918182,\n      -0.04634430631995201,\n      -0.019872348755598068,\n      -0.01745951920747757,\n      0.004164759535342455,\n      -0.04155835881829262,\n      -0.1262562870979309,\n      -0.040098920464515686,\n      -0.07915595918893814,\n      0.03421202674508095,\n      -0.0493970550596714,\n      0.17811435461044312,\n      0.08639736473560333,\n      0.05320754647254944,\n      -0.035383038222789764,\n      0.03694487735629082,\n      0.09907638281583786,\n      0.05273526906967163,\n      0.046555910259485245,\n      0.13291902840137482,\n      0.1114087626338005,\n      -0.044576164335012436,\n      0.0408715158700943,\n      0.19052943587303162,\n      0.10726548731327057,\n      0.06128283217549324,\n      0.10310401767492294,\n      0.02924942970275879,\n      -0.08961126208305359,\n      0.14241313934326172,\n      -0.37825024127960205,\n      -0.051722824573516846,\n      0.12135695666074753,\n      0.09457121789455414,\n      0.1951424479484558,\n      0.1440712809562683,\n      0.04387380927801132,\n      -0.07564596831798553,\n      0.13770192861557007,\n      0.0755331739783287\n    ],\n    [\n      0.2614760994911194,\n      0.1297285407781601,\n      0.1689443439245224,\n      0.19437871873378754,\n      -0.056726038455963135,\n      0.09934567660093307,\n      0.05515877157449722,\n      -0.008106482215225697,\n      0.02250894531607628,\n      0.13130521774291992,\n      0.16751569509506226,\n      0.037613645195961,\n      0.16680413484573364,\n      0.08091343194246292,\n      -0.44920220971107483,\n      0.3489859402179718,\n      -0.004838582128286362,\n      0.17003601789474487,\n      0.06895330548286438,\n      0.15190090239048004,\n      0.02334986813366413,\n      0.005596322938799858,\n      -0.17194080352783203,\n      0.05292774364352226,\n      -0.1421491503715515,\n      -0.03235861286520958,\n      -0.05822219327092171,\n      0.35053765773773193,\n      0.1577174961566925,\n      -0.15669670701026917,\n      0.034069888293743134,\n      -0.028024110943078995,\n      -0.05311007797718048,\n      0.05467289686203003,\n      -0.03183774650096893,\n      -0.05766444280743599,\n      0.0069342320784926414,\n      -0.015910467132925987,\n      -0.03040585294365883,\n      -0.027908770367503166,\n      0.030122358351945877,\n      -0.03437185287475586,\n      0.003162721637636423,\n      -0.011229772120714188,\n      0.02102087251842022,\n      0.03601010888814926,\n      -0.24135395884513855,\n      -0.20184943079948425,\n      0.0026069211307913065,\n      -0.1552571952342987,\n      -0.14628058671951294,\n      -0.07216044515371323,\n      -0.20310918986797333,\n      -0.00912421103566885,\n      -0.07357916980981827,\n      -0.04631202667951584,\n      -0.14826591312885284,\n      -0.12884855270385742,\n      -0.028705229982733727,\n      -0.06195717677474022,\n      -0.07440068572759628,\n      0.065968357026577,\n      0.06862835586071014,\n      0.024569042026996613,\n      -0.030785903334617615,\n      -0.0663985013961792,\n      0.035569071769714355,\n      0.013971848413348198,\n      -0.0815880224108696,\n      -0.05303511396050453,\n      -0.10370658338069916,\n      0.04720320552587509,\n      -0.0765257328748703,\n      -0.044805847108364105,\n      -0.017797276377677917,\n      -0.045294057577848434,\n      -0.07880599051713943,\n      -0.09452703595161438,\n      -0.08801917731761932,\n      0.04108789935708046,\n      -0.13059215247631073,\n      -0.021102827042341232,\n      -0.07235892117023468,\n      -0.11713650822639465,\n      0.17400410771369934,\n      0.17424322664737701,\n      -0.01065730769187212,\n      0.050870392471551895,\n      0.014780960977077484,\n      0.08865967392921448,\n      0.04146236926317215,\n      -0.02963009849190712,\n      0.06416042149066925,\n      -0.03742799907922745,\n      0.05204726755619049,\n      -0.001466493122279644,\n      0.00925470981746912,\n      0.0563664510846138,\n      0.003336468245834112,\n      -0.055580709129571915,\n      -0.014554242603480816,\n      0.0023214847315102816,\n      0.10656193643808365,\n      0.04698988050222397,\n      0.1497536152601242,\n      -0.0412948876619339,\n      -0.04309265315532684,\n      0.0897759422659874,\n      0.10739701986312866,\n      -0.04763248562812805,\n      0.0758000984787941,\n      0.010527263395488262,\n      -0.032913751900196075,\n      0.029492033645510674,\n      -0.030350668355822563,\n      0.40658074617385864,\n      0.21662625670433044,\n      0.03636734560132027,\n      -0.06996145099401474,\n      -0.038054440170526505,\n      -0.2747170627117157,\n      0.02277611568570137,\n      0.009747529402375221,\n      -0.0643782988190651,\n      0.03652676194906235\n    ],\n    [\n      -0.10599538683891296,\n      -0.09300968796014786,\n      -0.007424050010740757,\n      -0.21971309185028076,\n      -0.02147332951426506,\n      -0.05969829484820366,\n      -0.09486492723226547,\n      -0.02095312811434269,\n      -0.1153278574347496,\n      0.016261719167232513,\n      -0.12303843349218369,\n      -0.1575227677822113,\n      -0.1595529168844223,\n      -0.028314460068941116,\n      0.3401956856250763,\n      -0.3559987246990204,\n      -0.04649168625473976,\n      -0.08279436081647873,\n      -0.11051766574382782,\n      -0.08138616383075714,\n      -0.05610707774758339,\n      -0.007930277846753597,\n      0.03668224439024925,\n      -0.05412246286869049,\n      -0.02090628631412983,\n      -0.05049172043800354,\n      -0.04745122417807579,\n      0.2441534847021103,\n      0.15806685388088226,\n      0.11166955530643463,\n      0.08702922612428665,\n      -0.03650481998920441,\n      0.025943519547581673,\n      0.050599489361047745,\n      -0.0373336598277092,\n      0.0591161884367466,\n      -0.01412320788949728,\n      -0.08749374747276306,\n      0.06365790218114853,\n      -0.08061303198337555,\n      0.04881281778216362,\n      0.026368334889411926,\n      -0.008965575136244297,\n      0.013505271635949612,\n      -0.004120383877307177,\n      0.1242225170135498,\n      -0.03204284980893135,\n      -0.04776391014456749,\n      0.04351271688938141,\n      -0.0736481174826622,\n      0.09282761812210083,\n      0.16560427844524384,\n      2.5565383111825213e-05,\n      0.06397269666194916,\n      0.1385646015405655,\n      0.07332105934619904,\n      -0.013023979030549526,\n      0.036992352455854416,\n      0.03266916796565056,\n      0.03472139313817024,\n      0.054803889244794846,\n      0.08580265194177628,\n      0.010221731849014759,\n      0.1315574198961258,\n      -0.05680125579237938,\n      -0.03757992014288902,\n      -0.05105679854750633,\n      -0.08367075026035309,\n      -0.04635826498270035,\n      0.02975906804203987,\n      -0.07507021725177765,\n      0.027902403846383095,\n      -0.09483268111944199,\n      0.15738332271575928,\n      0.07662428170442581,\n      0.09623098373413086,\n      -0.01636902429163456,\n      -0.042716287076473236,\n      0.022836215794086456,\n      -0.00390459131449461,\n      -0.050148360431194305,\n      -0.027695802971720695,\n      0.018715646117925644,\n      0.04655832052230835,\n      -1.248790340468986e-05,\n      0.37273916602134705,\n      0.011112868785858154,\n      0.023397110402584076,\n      -0.0359383262693882,\n      -0.11871425807476044,\n      -0.05553115904331207,\n      -0.06729929894208908,\n      -0.09017446637153625,\n      -0.024854231625795364,\n      -0.06801898032426834,\n      -0.09189056605100632,\n      0.08990335464477539,\n      0.17964693903923035,\n      -0.08550681173801422,\n      0.03368796408176422,\n      -0.06662645936012268,\n      0.015208873897790909,\n      0.13074149191379547,\n      0.07839745283126831,\n      0.19552157819271088,\n      0.061145707964897156,\n      0.07938870787620544,\n      0.07551431655883789,\n      0.18174946308135986,\n      0.04993315041065216,\n      0.08895397186279297,\n      0.04962807893753052,\n      0.029035992920398712,\n      -0.09399925172328949,\n      0.07751036435365677,\n      -0.22982636094093323,\n      -0.03357481583952904,\n      0.11529666930437088,\n      0.06903548538684845,\n      0.2243763506412506,\n      0.013948386535048485,\n      0.08112764358520508,\n      -0.003319387324154377,\n      0.02109460160136223,\n      0.07071664929389954\n    ],\n    [\n      0.1873229742050171,\n      0.10204515606164932,\n      0.16229797899723053,\n      0.16232839226722717,\n      0.0137338787317276,\n      0.10745646059513092,\n      -0.026689628139138222,\n      -0.061280690133571625,\n      0.038517456501722336,\n      0.05731554701924324,\n      0.11353589594364166,\n      0.11375994235277176,\n      0.12177106738090515,\n      0.15234147012233734,\n      -0.4647682011127472,\n      0.2818487286567688,\n      -0.011234637349843979,\n      0.046926211565732956,\n      0.004534624516963959,\n      0.15171872079372406,\n      0.07577839493751526,\n      0.03142374008893967,\n      -0.03051658160984516,\n      0.09928886592388153,\n      -0.004808718338608742,\n      0.002079468686133623,\n      0.019272126257419586,\n      0.2259751409292221,\n      0.05223345011472702,\n      -0.1491665095090866,\n      0.049787867814302444,\n      -0.007555176969617605,\n      -0.017965303733944893,\n      -0.07070573419332504,\n      -0.0012832781067118049,\n      -0.04530254378914833,\n      -0.07991687953472137,\n      0.05277766287326813,\n      -0.024244481697678566,\n      0.04585767909884453,\n      -0.08814086019992828,\n      0.055904749780893326,\n      0.036825746297836304,\n      -0.04624457657337189,\n      0.011822439730167389,\n      -0.012318041175603867,\n      -0.10271582007408142,\n      -0.10631117969751358,\n      0.015378246083855629,\n      -0.11744861304759979,\n      -0.11198224872350693,\n      0.03168780729174614,\n      -0.11385234445333481,\n      -0.12467043101787567,\n      0.02267470583319664,\n      -0.13016042113304138,\n      -0.1927882730960846,\n      -0.12544994056224823,\n      -0.01863928884267807,\n      -0.08006074279546738,\n      -0.02111969329416752,\n      0.017277272418141365,\n      0.05111817643046379,\n      0.042481523007154465,\n      -0.11683211475610733,\n      0.013452008366584778,\n      -0.07595871388912201,\n      -0.0049496302381157875,\n      -0.09062787145376205,\n      -0.01935836672782898,\n      -0.0879654586315155,\n      0.0008264405187219381,\n      -0.04347909986972809,\n      0.051401931792497635,\n      0.019068213179707527,\n      -0.047402214258909225,\n      0.05212656408548355,\n      -0.09436137974262238,\n      -0.01947508379817009,\n      0.05180712789297104,\n      -0.06855903565883636,\n      0.04969560354948044,\n      -0.10202573984861374,\n      -0.05708453059196472,\n      0.20299270749092102,\n      0.014963534660637379,\n      -0.009648419916629791,\n      0.06458760797977448,\n      0.046877097338438034,\n      -0.005271510221064091,\n      0.0644335001707077,\n      0.06795399636030197,\n      -0.04657945781946182,\n      0.045810114592313766,\n      0.07527225464582443,\n      0.0282757505774498,\n      -0.0742483139038086,\n      0.035374585539102554,\n      0.07599956542253494,\n      -0.033072780817747116,\n      0.0397966206073761,\n      0.003181206062436104,\n      0.046799901872873306,\n      0.05407138913869858,\n      0.15717490017414093,\n      -0.04711363837122917,\n      0.020332753658294678,\n      0.03312384709715843,\n      0.09947369247674942,\n      0.12018686532974243,\n      0.010551007464528084,\n      0.012382394634187222,\n      -0.026644734665751457,\n      0.0304515790194273,\n      0.05325597897171974,\n      0.23409922420978546,\n      0.18640156090259552,\n      0.03840908035635948,\n      -0.16906696557998657,\n      0.007412927690893412,\n      -0.341986745595932,\n      -0.05873829126358032,\n      0.05663585290312767,\n      -0.14867529273033142,\n      -0.020387353375554085\n    ],\n    [\n      -0.2231987565755844,\n      0.02914164960384369,\n      -0.045410219579935074,\n      -0.13436663150787354,\n      -0.11622074991464615,\n      -0.2712883949279785,\n      0.07496451586484909,\n      0.027561048045754433,\n      0.042419157922267914,\n      0.06513362377882004,\n      -0.016341879963874817,\n      -0.06296760588884354,\n      -0.31367507576942444,\n      -0.013163400813937187,\n      0.4201727509498596,\n      -0.4105202555656433,\n      -0.024998348206281662,\n      -0.183166041970253,\n      0.012447135522961617,\n      -0.058630671352148056,\n      -0.026458466425538063,\n      -0.07407277077436447,\n      -0.1695106327533722,\n      -0.019221043214201927,\n      -0.055607739835977554,\n      -0.017709141597151756,\n      -0.131136953830719,\n      0.2068883627653122,\n      0.11806530505418777,\n      0.34228360652923584,\n      -0.07344956696033478,\n      0.10278300940990448,\n      0.07532887905836105,\n      -0.039315130561590195,\n      0.13304847478866577,\n      0.03741829842329025,\n      0.08891059458255768,\n      0.0939401239156723,\n      0.06896786391735077,\n      0.057045020163059235,\n      -0.07288490980863571,\n      -0.0019010056275874376,\n      0.05571151152253151,\n      0.04781360924243927,\n      0.19456207752227783,\n      -0.003974077291786671,\n      0.032180625945329666,\n      -0.17150060832500458,\n      0.1146329790353775,\n      -0.09348484128713608,\n      -0.0539303757250309,\n      -0.010279607027769089,\n      -0.08989612013101578,\n      0.019699962809681892,\n      0.019793463870882988,\n      -0.07694952189922333,\n      -0.29181456565856934,\n      -0.06764596700668335,\n      -0.06136298552155495,\n      -0.0054425825364887714,\n      -0.1900472342967987,\n      -0.008596712723374367,\n      0.011191286146640778,\n      0.13679002225399017,\n      -0.1371888816356659,\n      -0.304411381483078,\n      0.05953050032258034,\n      0.04363522306084633,\n      -0.00342385470867157,\n      -0.01454127486795187,\n      -0.015806760638952255,\n      -0.07014241814613342,\n      -0.06903696805238724,\n      0.09917913377285004,\n      0.048814233392477036,\n      0.03098907507956028,\n      -0.014333700761198997,\n      -0.011769854463636875,\n      0.10095512866973877,\n      0.0011199668515473604,\n      -0.061522696167230606,\n      -0.08251280337572098,\n      -0.10401704907417297,\n      -0.1166655421257019,\n      -0.0009137055021710694,\n      0.3846399486064911,\n      0.025532551109790802,\n      -0.05326062813401222,\n      0.04241248592734337,\n      0.02156204730272293,\n      0.061842236667871475,\n      0.04686672240495682,\n      0.08106092363595963,\n      0.049869827926158905,\n      0.060962338000535965,\n      0.05514874681830406,\n      0.05347492918372154,\n      0.08018478006124496,\n      -0.10268507152795792,\n      -0.0203857459127903,\n      0.0353611595928669,\n      0.012073414400219917,\n      0.043037936091423035,\n      0.002011776901781559,\n      0.2061413675546646,\n      -0.07130143791437149,\n      8.754115697229281e-05,\n      0.02687695622444153,\n      0.17126719653606415,\n      0.1806269735097885,\n      0.06645770370960236,\n      0.047374177724123,\n      0.0004019019834231585,\n      0.017117666080594063,\n      0.03296158090233803,\n      -0.24854815006256104,\n      -0.08869173377752304,\n      0.042962826788425446,\n      -0.027382103726267815,\n      0.2229967564344406,\n      0.003965018782764673,\n      -0.07213293761014938,\n      0.054182808846235275,\n      0.08271780610084534,\n      0.04349950700998306\n    ],\n    [\n      0.21714667975902557,\n      0.12699581682682037,\n      0.09069404751062393,\n      0.14930133521556854,\n      0.0029653592500835657,\n      0.047222841531038284,\n      0.021647997200489044,\n      0.0475902333855629,\n      -0.11073272675275803,\n      0.09538421779870987,\n      0.18959805369377136,\n      0.13222499191761017,\n      0.25717589259147644,\n      0.15122050046920776,\n      -0.4603077471256256,\n      0.40188705921173096,\n      0.013004500418901443,\n      0.037737276405096054,\n      -0.015466847456991673,\n      0.115670345723629,\n      -0.10318386554718018,\n      0.045329272747039795,\n      0.0046132695861160755,\n      0.031000817194581032,\n      -0.1274438053369522,\n      0.008046865463256836,\n      -0.06952910125255585,\n      0.24607248604297638,\n      0.17120133340358734,\n      -0.14901067316532135,\n      0.04523273557424545,\n      -0.006286303512752056,\n      0.047006987035274506,\n      -0.04834471270442009,\n      -0.02554926462471485,\n      -0.08346523344516754,\n      -0.08007292449474335,\n      -0.032553885132074356,\n      0.017370326444506645,\n      0.021824007853865623,\n      -0.10637430846691132,\n      -0.012253636494278908,\n      0.05259908735752106,\n      -0.09112556278705597,\n      0.034446921199560165,\n      -0.06274449080228806,\n      -0.12076865136623383,\n      -0.17412328720092773,\n      -0.0019787473138421774,\n      -0.08859707415103912,\n      -0.15886187553405762,\n      0.00661467993631959,\n      -0.16448751091957092,\n      -0.10739065706729889,\n      -0.09956328570842743,\n      -0.09132612496614456,\n      -0.1363685429096222,\n      -0.11695373803377151,\n      -0.023107247427105904,\n      -0.10927436500787735,\n      -0.15495070815086365,\n      0.11871647834777832,\n      0.036975063383579254,\n      0.11400246620178223,\n      -0.07122236490249634,\n      -0.11346946656703949,\n      -0.05315697193145752,\n      -0.06798127293586731,\n      -0.05974654108285904,\n      0.0009666718542575836,\n      0.01149566750973463,\n      0.0007483414956368506,\n      0.021873338147997856,\n      0.08018898218870163,\n      -0.03168650344014168,\n      -0.1769976168870926,\n      -0.015293077565729618,\n      -0.03559063374996185,\n      0.006476720795035362,\n      0.15240193903446198,\n      -0.1341172605752945,\n      -0.10365402698516846,\n      -0.12373148649930954,\n      -0.08597176522016525,\n      0.2133222222328186,\n      0.09058284014463425,\n      -0.0644315630197525,\n      0.09937310963869095,\n      0.022291678935289383,\n      0.03167257457971573,\n      -0.028387615457177162,\n      0.051725488156080246,\n      0.12215375900268555,\n      0.03354315087199211,\n      0.07025574892759323,\n      -0.007771336939185858,\n      0.018784884363412857,\n      -0.0018164593493565917,\n      0.06400207430124283,\n      -0.00013421922631096095,\n      0.020185181871056557,\n      0.03705652058124542,\n      -0.005728019401431084,\n      0.03806411847472191,\n      0.17171607911586761,\n      -0.02175317518413067,\n      -0.06184320151805878,\n      0.02677014470100403,\n      0.0752481147646904,\n      0.08659595251083374,\n      0.03746158257126808,\n      0.043702349066734314,\n      -0.05058525875210762,\n      0.06897932291030884,\n      -0.027267802506685257,\n      0.320514053106308,\n      0.2635219693183899,\n      0.11443137377500534,\n      -0.16954123973846436,\n      -0.0005784564418718219,\n      -0.35564184188842773,\n      -0.015236863866448402,\n      0.03091122955083847,\n      -0.1461520493030548,\n      0.04156467691063881\n    ],\n    [\n      0.10412200540304184,\n      -0.03784840181469917,\n      0.15701811015605927,\n      0.19276337325572968,\n      -0.0383056215941906,\n      0.14056909084320068,\n      0.10967361927032471,\n      -0.046594519168138504,\n      0.01518018078058958,\n      0.14182975888252258,\n      0.12103810906410217,\n      0.12450501322746277,\n      0.2251875251531601,\n      0.10008364915847778,\n      -0.371159166097641,\n      0.40078309178352356,\n      -0.0009333207854069769,\n      0.14165948331356049,\n      0.05725378170609474,\n      -0.013535532169044018,\n      0.006786522455513477,\n      -0.004924354143440723,\n      -0.08956718444824219,\n      0.10246835649013519,\n      -0.022272083908319473,\n      -0.02540440857410431,\n      -0.04459822550415993,\n      0.30127841234207153,\n      0.03732111677527428,\n      -0.15481871366500854,\n      0.06877835839986801,\n      -0.023482926189899445,\n      0.08507858216762543,\n      -0.02198215201497078,\n      -0.07452937215566635,\n      0.0644378736615181,\n      -0.09403136372566223,\n      -0.05908988416194916,\n      0.06842359155416489,\n      0.05327548459172249,\n      0.0331868901848793,\n      0.03664318844676018,\n      0.08627676218748093,\n      -0.0594412200152874,\n      0.04213514178991318,\n      0.010503481142222881,\n      -0.2144966721534729,\n      -0.1488954871892929,\n      -0.006719434633851051,\n      -0.05211738497018814,\n      -0.1030728816986084,\n      -0.08404439687728882,\n      -0.1364060640335083,\n      -0.12479350715875626,\n      0.022166533395648003,\n      -0.10345325618982315,\n      -0.0726764053106308,\n      -0.08786401152610779,\n      0.02874554507434368,\n      -0.15988793969154358,\n      -0.07362500578165054,\n      0.06215403974056244,\n      0.038710616528987885,\n      0.07784458249807358,\n      -0.05327220633625984,\n      -0.08297142386436462,\n      0.02075982838869095,\n      -0.030743155628442764,\n      -0.10798496752977371,\n      -0.029998773708939552,\n      -0.08787429332733154,\n      -0.0652410089969635,\n      -0.023387398570775986,\n      0.08025084435939789,\n      -0.13830190896987915,\n      0.015627732500433922,\n      -0.016269849613308907,\n      -0.05097474530339241,\n      -0.0737057775259018,\n      0.04007229581475258,\n      -0.03162788972258568,\n      -0.10479959845542908,\n      -0.004690210334956646,\n      -0.015338622964918613,\n      0.21410022675991058,\n      0.0284133143723011,\n      0.05177658051252365,\n      0.09271795302629471,\n      0.07527956366539001,\n      0.07301975786685944,\n      -0.008065743371844292,\n      -0.08964114636182785,\n      0.06220047175884247,\n      0.09106875956058502,\n      -0.025225231423974037,\n      0.08095087110996246,\n      0.051590025424957275,\n      0.05245894566178322,\n      0.008498537354171276,\n      0.03988773375749588,\n      0.07779093831777573,\n      0.05540568381547928,\n      0.10332898050546646,\n      -0.013705573044717312,\n      0.16085423529148102,\n      0.00956426840275526,\n      -0.02989657036960125,\n      0.12170613557100296,\n      0.030729565769433975,\n      0.10049162060022354,\n      0.09605025500059128,\n      0.052022770047187805,\n      0.06388972699642181,\n      -0.05110198259353638,\n      -0.05163091421127319,\n      0.396929532289505,\n      0.19977717101573944,\n      0.06330176442861557,\n      -0.16117410361766815,\n      -0.04688681662082672,\n      -0.2591717839241028,\n      -0.02023179456591606,\n      -0.0347585454583168,\n      -0.0840546041727066,\n      -0.05261611565947533\n    ],\n    [\n      0.12466885894536972,\n      0.07717709243297577,\n      0.15830601751804352,\n      0.11350087821483612,\n      -0.0435016043484211,\n      0.06882551312446594,\n      0.014891529455780983,\n      -0.06809139996767044,\n      0.010996554046869278,\n      0.07969929277896881,\n      0.060577958822250366,\n      0.07851836085319519,\n      0.1602686494588852,\n      0.15226811170578003,\n      -0.34955894947052,\n      -0.048017896711826324,\n      0.09557610750198364,\n      0.11317676305770874,\n      0.05895150825381279,\n      0.019206063821911812,\n      -0.04501666501164436,\n      -0.013968521729111671,\n      -0.11899175494909286,\n      0.12183918058872223,\n      -0.12009172141551971,\n      0.02225717529654503,\n      -0.06327689439058304,\n      0.27486249804496765,\n      0.026309289038181305,\n      -0.13584020733833313,\n      0.014685790054500103,\n      0.0005430139717645943,\n      -0.05022916942834854,\n      -0.11039803177118301,\n      0.042834192514419556,\n      -0.08141700178384781,\n      -0.09915774315595627,\n      0.07252384722232819,\n      -0.078497514128685,\n      0.06273478269577026,\n      0.0718693658709526,\n      -0.003372170962393284,\n      -0.032106805592775345,\n      -0.09714541584253311,\n      0.11425857990980148,\n      -0.08917219191789627,\n      -0.20221737027168274,\n      -0.11930598318576813,\n      -0.07757710665464401,\n      -0.0425436832010746,\n      -0.1856985241174698,\n      -0.0749254897236824,\n      -0.029541905969381332,\n      -0.08371797949075699,\n      -0.005421772599220276,\n      -0.1430480033159256,\n      -0.17634357511997223,\n      0.007685478776693344,\n      0.0746854692697525,\n      -0.12539315223693848,\n      -0.0945071130990982,\n      0.059468384832143784,\n      -0.0077369785867631435,\n      0.12543514370918274,\n      0.09878598153591156,\n      -0.11838594079017639,\n      -0.015323984436690807,\n      0.0920238271355629,\n      -0.15824291110038757,\n      -0.019354844465851784,\n      -0.09113362431526184,\n      -0.002046473091468215,\n      -0.06278058141469955,\n      0.10876072198152542,\n      -0.068773053586483,\n      -0.2933620810508728,\n      0.03752169385552406,\n      -0.007412120699882507,\n      -0.02889075316488743,\n      0.18727147579193115,\n      -0.15139615535736084,\n      0.04499688744544983,\n      -0.0017887081485241652,\n      -0.058540645986795425,\n      0.23451362550258636,\n      0.06234130263328552,\n      -0.09921426326036453,\n      0.016213836148381233,\n      -0.012705071829259396,\n      0.0440925695002079,\n      0.03838720917701721,\n      0.010944470763206482,\n      0.03152693063020706,\n      0.014449580572545528,\n      0.10308604687452316,\n      0.01617600955069065,\n      -0.053801875561475754,\n      0.12750642001628876,\n      -0.01805470883846283,\n      -0.047862518578767776,\n      -0.019199397414922714,\n      -0.031004875898361206,\n      0.08011738955974579,\n      0.026286648586392403,\n      0.07005512714385986,\n      0.029300380498170853,\n      -0.05248883739113808,\n      0.07617124915122986,\n      -0.03332424536347389,\n      0.1186622604727745,\n      0.07684486359357834,\n      0.01285715401172638,\n      0.04262014850974083,\n      0.010356717742979527,\n      0.02826937660574913,\n      0.1401940882205963,\n      0.25313857197761536,\n      0.10135239362716675,\n      -0.023736530914902687,\n      0.05239146947860718,\n      -0.29575541615486145,\n      -0.06338430196046829,\n      0.07621052116155624,\n      0.04994584619998932,\n      -0.06549592316150665\n    ],\n    [\n      -0.1159573644399643,\n      -0.027381623163819313,\n      -0.06959396600723267,\n      -0.1824685037136078,\n      0.0299211498349905,\n      0.03114527277648449,\n      -0.08238919079303741,\n      -0.008928859606385231,\n      -0.060978326946496964,\n      -0.04299125075340271,\n      -0.17236797511577606,\n      -0.17838826775550842,\n      -0.24473020434379578,\n      -0.24351246654987335,\n      0.3821450173854828,\n      -0.2699543535709381,\n      0.09669763594865799,\n      0.022293779999017715,\n      0.04149564728140831,\n      -0.10653305798768997,\n      0.060259800404310226,\n      -0.05394618213176727,\n      -0.01766817644238472,\n      0.06587997823953629,\n      -0.03754176199436188,\n      0.06983216851949692,\n      0.0029049452859908342,\n      0.20182478427886963,\n      0.1648522913455963,\n      0.05491775646805763,\n      0.10131175071001053,\n      -0.028777331113815308,\n      0.05677291378378868,\n      -0.0299507025629282,\n      -0.02637028880417347,\n      0.02594456449151039,\n      0.04164119437336922,\n      -0.03262205049395561,\n      0.10697656869888306,\n      0.01098897960036993,\n      -0.05095607787370682,\n      -0.012591456063091755,\n      -0.08128923177719116,\n      0.015930382534861565,\n      0.066159188747406,\n      0.03095339983701706,\n      -0.020264796912670135,\n      -0.022943682968616486,\n      0.1764613687992096,\n      0.06245565786957741,\n      -0.013092156499624252,\n      0.0834544450044632,\n      -0.0077279796823859215,\n      0.029594661667943,\n      0.1555222123861313,\n      0.00326473917812109,\n      0.0940937027335167,\n      0.010770906694233418,\n      0.008025907911360264,\n      0.0819351077079773,\n      -0.046528130769729614,\n      0.14441817998886108,\n      -0.006054195575416088,\n      0.00565972039476037,\n      -0.04565122723579407,\n      0.024798788130283356,\n      0.045533496886491776,\n      -0.01776612550020218,\n      -0.02978735789656639,\n      -0.03271080181002617,\n      -0.03874664753675461,\n      -0.011363741010427475,\n      -0.07625027745962143,\n      0.023028986528515816,\n      -0.03670540452003479,\n      0.09339975565671921,\n      0.020191218703985214,\n      0.05304889753460884,\n      0.09492265433073044,\n      0.12893381714820862,\n      -0.09459824860095978,\n      0.07485111802816391,\n      -0.07274279743432999,\n      -0.09245168417692184,\n      0.016005104407668114,\n      0.21924391388893127,\n      -0.09765239804983139,\n      -0.07803132385015488,\n      0.019880658015608788,\n      0.03377804532647133,\n      -0.023690320551395416,\n      0.013152031227946281,\n      0.06931104511022568,\n      -0.08281785994768143,\n      -0.018050476908683777,\n      -0.003428215626627207,\n      0.09552225470542908,\n      0.18292930722236633,\n      0.009348804131150246,\n      0.06283479183912277,\n      -0.0449618399143219,\n      0.09807810932397842,\n      0.08788488805294037,\n      0.04347238317131996,\n      0.23355436325073242,\n      0.1506316065788269,\n      0.09831935167312622,\n      0.12062854319810867,\n      0.052693188190460205,\n      0.041081901639699936,\n      0.18638794124126434,\n      0.09599412977695465,\n      0.035338692367076874,\n      0.03148422762751579,\n      -0.010751872323453426,\n      -0.45092931389808655,\n      0.04971069097518921,\n      0.1350003033876419,\n      0.1272849142551422,\n      0.21018846333026886,\n      0.07516416162252426,\n      0.05436284467577934,\n      -0.03966118022799492,\n      0.10684378445148468,\n      0.05946563556790352\n    ],\n    [\n      0.15749280154705048,\n      0.04309873655438423,\n      0.04722357913851738,\n      0.14088207483291626,\n      -0.11083588749170303,\n      0.009668964892625809,\n      0.02589445188641548,\n      0.01937851496040821,\n      -0.07835570722818375,\n      0.017802122980356216,\n      0.10829667001962662,\n      0.1641446352005005,\n      0.29744571447372437,\n      0.01808166690170765,\n      -0.28383269906044006,\n      0.15986402332782745,\n      0.13667069375514984,\n      0.17192848026752472,\n      0.08219470828771591,\n      -0.012521151453256607,\n      -0.04882960766553879,\n      0.027757719159126282,\n      -0.14604628086090088,\n      0.07650642096996307,\n      -0.17555131018161774,\n      -0.14094454050064087,\n      -0.13355892896652222,\n      0.3007982075214386,\n      0.045314326882362366,\n      -0.22743965685367584,\n      2.6333447749493644e-05,\n      -0.09430458396673203,\n      -0.15655744075775146,\n      0.05117033049464226,\n      0.026631733402609825,\n      -0.12045899033546448,\n      -0.02440350130200386,\n      -0.0020784090738743544,\n      0.025374989956617355,\n      0.03190732002258301,\n      -0.04720289260149002,\n      -0.03927327319979668,\n      -0.060950059443712234,\n      0.0013942981604486704,\n      -0.0246761254966259,\n      -0.05225544422864914,\n      -0.18761096894741058,\n      -0.04849940538406372,\n      0.011434188112616539,\n      -0.2948298752307892,\n      -0.13042104244232178,\n      0.017349330708384514,\n      -0.16263124346733093,\n      -0.08976870030164719,\n      -0.14989814162254333,\n      0.019669631496071815,\n      -0.21916800737380981,\n      -0.049070216715335846,\n      0.09713327139616013,\n      -0.09535224735736847,\n      -0.08418109267950058,\n      0.04215029627084732,\n      0.028499817475676537,\n      0.08894971013069153,\n      0.014999503269791603,\n      -0.03796931356191635,\n      -0.13488292694091797,\n      0.03673716261982918,\n      -0.08046969026327133,\n      -0.0226287804543972,\n      -0.06842579692602158,\n      -0.0013779617147520185,\n      0.0670464038848877,\n      0.004673422314226627,\n      -0.1597728133201599,\n      -0.2344931811094284,\n      -0.020694106817245483,\n      -0.14205220341682434,\n      -0.0011214070254936814,\n      0.18405671417713165,\n      -0.043590739369392395,\n      -0.031397078186273575,\n      -0.10459544509649277,\n      -0.09993842989206314,\n      0.19888155162334442,\n      0.046030767261981964,\n      0.10328586399555206,\n      -0.04823741316795349,\n      0.0335489958524704,\n      0.03589204326272011,\n      0.06509055942296982,\n      0.01063655223697424,\n      0.015721699222922325,\n      0.10353914648294449,\n      0.0480157807469368,\n      0.10110744833946228,\n      0.01106087677180767,\n      0.06130162253975868,\n      0.012064828537404537,\n      0.004712164402008057,\n      0.0448620542883873,\n      0.04075182229280472,\n      -0.02783852443099022,\n      -0.006181992124766111,\n      0.23995985090732574,\n      -0.01382561307400465,\n      0.03654174506664276,\n      0.06458679586648941,\n      0.03220483288168907,\n      0.11331592500209808,\n      -0.0013233923818916082,\n      0.010898550972342491,\n      0.0934041365981102,\n      0.03312431275844574,\n      -0.021375058218836784,\n      0.1421283334493637,\n      0.0990658849477768,\n      -0.008134921081364155,\n      -0.05351397022604942,\n      0.06320254504680634,\n      -0.27895262837409973,\n      -0.0806790217757225,\n      0.013195320032536983,\n      -0.22846069931983948,\n      0.004514545202255249\n    ],\n    [\n      0.20948989689350128,\n      0.1339220553636551,\n      0.0031785874161869287,\n      0.198565274477005,\n      0.052471768110990524,\n      0.07185797393321991,\n      -0.037328850477933884,\n      0.011439312249422073,\n      0.03924285247921944,\n      0.12695157527923584,\n      0.15570944547653198,\n      0.047949012368917465,\n      0.2994772791862488,\n      0.13891293108463287,\n      -0.37240803241729736,\n      0.24948640167713165,\n      -0.015793681144714355,\n      0.03125087544322014,\n      0.11202291399240494,\n      0.032624371349811554,\n      0.08017859607934952,\n      0.12512409687042236,\n      0.00756079750135541,\n      0.14166337251663208,\n      -0.08588123321533203,\n      0.04265914857387543,\n      0.07935494184494019,\n      0.0004485097888391465,\n      -0.08977195620536804,\n      -0.059678398072719574,\n      -0.02992766536772251,\n      -0.07863739877939224,\n      0.08118266612291336,\n      -0.0003174117882736027,\n      0.016780894249677658,\n      -0.023584973067045212,\n      0.098610058426857,\n      0.10215412825345993,\n      0.05594588443636894,\n      -0.015217417851090431,\n      0.08510122448205948,\n      0.09636189043521881,\n      0.02511722408235073,\n      -0.0845174789428711,\n      -0.007754357997328043,\n      -0.05164278298616409,\n      0.0033931091893464327,\n      -0.14876709878444672,\n      0.02231517620384693,\n      -0.0256295558065176,\n      -0.07187727093696594,\n      -0.02438693679869175,\n      -0.12364085018634796,\n      0.0017154604429379106,\n      0.04388481006026268,\n      -0.12233120203018188,\n      -0.20493537187576294,\n      -0.10160187631845474,\n      -0.022798145189881325,\n      -0.002861388958990574,\n      -0.06135144457221031,\n      -0.08261636644601822,\n      0.019573591649532318,\n      -0.019576899707317352,\n      -0.00884210504591465,\n      -0.07285012304782867,\n      -0.14029787480831146,\n      0.0022657597437500954,\n      -0.017881646752357483,\n      0.004248884040862322,\n      0.050023891031742096,\n      -0.025382807478308678,\n      -0.04705144092440605,\n      -0.04229393228888512,\n      -0.024499040096998215,\n      0.005456999875605106,\n      0.0036200974136590958,\n      -0.05030326545238495,\n      -0.032462719827890396,\n      0.034000106155872345,\n      0.0144504988566041,\n      0.07264502346515656,\n      0.07854091376066208,\n      0.051494017243385315,\n      0.10808958858251572,\n      -0.10113658756017685,\n      0.09784280508756638,\n      -0.01405447069555521,\n      0.14962241053581238,\n      0.035967156291007996,\n      0.05291600152850151,\n      -0.004033900331705809,\n      0.09699805080890656,\n      -0.04211908206343651,\n      0.03449342027306557,\n      0.04306986182928085,\n      0.07234100997447968,\n      0.1484554409980774,\n      -0.012154575437307358,\n      -0.07830363512039185,\n      -0.011460921727120876,\n      -0.00613781251013279,\n      0.09613493829965591,\n      0.037440989166498184,\n      0.1275501251220703,\n      -0.05109814926981926,\n      -0.09337884932756424,\n      0.061271049082279205,\n      0.06175897270441055,\n      0.008236581459641457,\n      -0.00933125987648964,\n      0.06128676235675812,\n      0.04411574453115463,\n      -0.0415271632373333,\n      0.09803779423236847,\n      0.37524402141571045,\n      0.1750802993774414,\n      0.14923396706581116,\n      -0.09402558207511902,\n      -0.10824695229530334,\n      -0.1250593215227127,\n      0.045771170407533646,\n      0.005042430944740772,\n      -0.12098005414009094,\n      0.10967849940061569\n    ],\n    [\n      -0.1949865221977234,\n      -0.008363673463463783,\n      -0.07443735748529434,\n      -0.1757245808839798,\n      -0.029973801225423813,\n      -0.11402371525764465,\n      -0.03652546554803848,\n      -0.10296840220689774,\n      -0.06066029518842697,\n      -0.026345519348978996,\n      -0.09424899518489838,\n      -0.15895909070968628,\n      -0.18946348130702972,\n      -0.059862565249204636,\n      0.33438992500305176,\n      -0.3141880929470062,\n      -0.023918302729725838,\n      -0.02769462950527668,\n      -0.10952484607696533,\n      -0.02661159448325634,\n      -0.003605600679293275,\n      0.00487872026860714,\n      -0.06383798271417618,\n      0.07639636099338531,\n      0.01956041343510151,\n      0.029746614396572113,\n      -0.07142315059900284,\n      0.1528717577457428,\n      0.15660905838012695,\n      0.22464321553707123,\n      0.10531821101903915,\n      0.14661580324172974,\n      0.05117267742753029,\n      0.0151996910572052,\n      0.025297492742538452,\n      -0.05717320740222931,\n      0.0003825599851552397,\n      -0.06298414617776871,\n      0.03803205117583275,\n      -0.03794949874281883,\n      0.04418620839715004,\n      -0.012060477398335934,\n      -0.01720413565635681,\n      -0.03593730181455612,\n      -0.036809906363487244,\n      0.10621779412031174,\n      0.06985047459602356,\n      -0.06315013766288757,\n      0.1435316801071167,\n      0.023603476583957672,\n      0.12310617417097092,\n      0.1441418081521988,\n      0.08710961043834686,\n      -0.01665448397397995,\n      0.019661953672766685,\n      -0.016666142269968987,\n      -0.037422046065330505,\n      0.03482137992978096,\n      -0.02943236380815506,\n      0.0009693531319499016,\n      0.060064077377319336,\n      0.03235512971878052,\n      0.052962832152843475,\n      0.14513902366161346,\n      0.08111635595560074,\n      0.0002737218455877155,\n      0.015460425987839699,\n      0.0389048270881176,\n      -0.005642583128064871,\n      0.004785372409969568,\n      -0.1061854436993599,\n      0.026798566803336143,\n      -0.01409156247973442,\n      0.13352897763252258,\n      0.03309836983680725,\n      0.09476032853126526,\n      0.08220995217561722,\n      0.08293139189481735,\n      0.12263111025094986,\n      0.0927303284406662,\n      -0.00018780403479468077,\n      -0.08964645862579346,\n      -0.006074515637010336,\n      -0.05942913889884949,\n      0.017244886606931686,\n      0.3353384733200073,\n      -0.12104073911905289,\n      0.024317029863595963,\n      -0.0714593157172203,\n      -0.08764137327671051,\n      -0.01491357758641243,\n      -0.11170915514230728,\n      -0.12239646911621094,\n      -0.021622683852910995,\n      0.013220543973147869,\n      -0.020150937139987946,\n      0.11805570125579834,\n      0.08738578855991364,\n      0.004755866248160601,\n      0.07617518305778503,\n      -0.0038166320882737637,\n      -0.02995317615568638,\n      0.09311152994632721,\n      0.02750239335000515,\n      0.050773948431015015,\n      0.14677980542182922,\n      0.0038941726088523865,\n      -0.0012650779681280255,\n      0.06772058457136154,\n      -0.007368083111941814,\n      0.0693068727850914,\n      0.11136658489704132,\n      -0.03659525886178017,\n      0.0034214744810014963,\n      0.0031559232156723738,\n      -0.36319705843925476,\n      -0.062173280864953995,\n      0.18386194109916687,\n      0.1477828025817871,\n      0.27120181918144226,\n      0.13636508584022522,\n      0.03915460780262947,\n      -0.058179523795843124,\n      0.06268416345119476,\n      -0.06537730246782303\n    ],\n    [\n      0.2189948409795761,\n      0.03150156885385513,\n      0.030959216877818108,\n      0.184613898396492,\n      0.017900967970490456,\n      0.04201244190335274,\n      0.07879021018743515,\n      0.017707910388708115,\n      0.015492185950279236,\n      0.08715789020061493,\n      0.22814948856830597,\n      0.15578731894493103,\n      0.21487590670585632,\n      0.010180961340665817,\n      -0.32310786843299866,\n      0.2847576141357422,\n      0.029059309512376785,\n      0.17413629591464996,\n      0.003291955217719078,\n      -0.03791090101003647,\n      0.09269312769174576,\n      0.12169972062110901,\n      -0.05991093069314957,\n      0.08975780755281448,\n      -0.19743819534778595,\n      0.038360126316547394,\n      0.07661200314760208,\n      0.10572262853384018,\n      -0.0554063655436039,\n      -0.08348020166158676,\n      -0.03483486548066139,\n      -0.14104150235652924,\n      -0.005085159093141556,\n      -0.025737743824720383,\n      0.046352628618478775,\n      -0.035769663751125336,\n      0.0708608478307724,\n      0.04681819677352905,\n      -0.07252591103315353,\n      0.030921826139092445,\n      0.0964227169752121,\n      0.05858168378472328,\n      0.07142072916030884,\n      0.029118278995156288,\n      0.10767832398414612,\n      -0.04189642146229744,\n      -0.09569883346557617,\n      -0.17810511589050293,\n      -0.021679924800992012,\n      -0.0727587416768074,\n      -0.09450580179691315,\n      -0.0685150995850563,\n      -0.24947112798690796,\n      -0.1012706458568573,\n      -0.06398534029722214,\n      -0.11841414123773575,\n      -0.16221770644187927,\n      0.018711429089307785,\n      -0.002187049947679043,\n      -0.12344322353601456,\n      -0.03007148765027523,\n      0.015593118034303188,\n      0.0015470089856535196,\n      0.021357813850045204,\n      0.053725577890872955,\n      -0.12481370568275452,\n      -0.02282027155160904,\n      -0.10748621076345444,\n      0.014669601805508137,\n      -0.09815351665019989,\n      -0.06657696515321732,\n      -0.06895807385444641,\n      -0.06623820215463638,\n      -0.16988533735275269,\n      -0.11361869424581528,\n      -0.09839391708374023,\n      0.02247067727148533,\n      -0.014652740210294724,\n      -0.15302079916000366,\n      -0.048851724714040756,\n      -0.08104094117879868,\n      -0.07362433522939682,\n      0.05241119861602783,\n      -0.08906050771474838,\n      0.1123761534690857,\n      -0.053771380335092545,\n      0.11669967323541641,\n      -0.03651032596826553,\n      0.07180356234312057,\n      0.1525076925754547,\n      -0.03553924337029457,\n      -0.04590200260281563,\n      0.043893635272979736,\n      0.037111591547727585,\n      0.011937536299228668,\n      0.05336036533117294,\n      0.03719876706600189,\n      0.09472763538360596,\n      0.032934799790382385,\n      -0.03270842880010605,\n      -0.045128993690013885,\n      0.08212445676326752,\n      0.05635606124997139,\n      0.047445110976696014,\n      0.15571661293506622,\n      -0.007700216956436634,\n      -0.038646113127470016,\n      0.1062714159488678,\n      0.16310174763202667,\n      0.1091328039765358,\n      0.02638402208685875,\n      0.035372719168663025,\n      0.03507283329963684,\n      -0.026554156094789505,\n      0.02370176836848259,\n      0.4910346567630768,\n      0.157451793551445,\n      0.06711874157190323,\n      -0.04987984150648117,\n      -0.10343781113624573,\n      -0.12289940565824509,\n      -0.10231738537549973,\n      0.08086779713630676,\n      -0.13846541941165924,\n      -0.03732216730713844\n    ],\n    [\n      0.12705831229686737,\n      0.13092784583568573,\n      -0.014145292341709137,\n      0.06515710055828094,\n      0.062240906059741974,\n      0.02410936914384365,\n      -0.045756589621305466,\n      0.07329829782247543,\n      0.0633842721581459,\n      0.09103618562221527,\n      0.10754166543483734,\n      0.12965118885040283,\n      0.30556243658065796,\n      0.10121164470911026,\n      -0.35415077209472656,\n      0.3944259583950043,\n      0.04195345938205719,\n      0.02832329273223877,\n      -0.032808758318424225,\n      0.00031087137176655233,\n      0.031416572630405426,\n      0.09954036772251129,\n      0.08934076875448227,\n      0.1189410611987114,\n      -0.07489985227584839,\n      -0.08709022402763367,\n      0.08615735918283463,\n      0.13133670389652252,\n      -0.10080193728208542,\n      -0.07704326510429382,\n      -0.07204536348581314,\n      0.02304842136800289,\n      0.07068625837564468,\n      0.055794354528188705,\n      0.026025792583823204,\n      -0.05757828801870346,\n      0.014688023366034031,\n      -0.0294949933886528,\n      0.09655014425516129,\n      0.03435685858130455,\n      0.07229332625865936,\n      -0.04937908425927162,\n      0.08060640841722488,\n      -0.05026496574282646,\n      0.06373012065887451,\n      0.029379991814494133,\n      -0.0235745906829834,\n      -0.1377260982990265,\n      -0.03895346447825432,\n      0.005578550044447184,\n      -0.1397237926721573,\n      -0.12084777653217316,\n      -0.09137043356895447,\n      -0.020519986748695374,\n      -0.10797063261270523,\n      -0.07903191447257996,\n      -0.18514589965343475,\n      -0.023904452100396156,\n      -0.007562700193375349,\n      0.03092363476753235,\n      0.06337487697601318,\n      -0.052826449275016785,\n      -0.06782151013612747,\n      0.011755254119634628,\n      -0.08888228237628937,\n      0.1236148253083229,\n      -0.06935827434062958,\n      0.07733852416276932,\n      0.07097706198692322,\n      0.0569336973130703,\n      -0.03859381005167961,\n      0.0013488581171259284,\n      -0.12584470212459564,\n      -0.04698380455374718,\n      0.005953884683549404,\n      0.05672375112771988,\n      -0.02992725372314453,\n      -0.03552813455462456,\n      -0.1517661064863205,\n      0.015440283343195915,\n      -0.011592407710850239,\n      0.0712510421872139,\n      -0.04285934194922447,\n      -0.0621122308075428,\n      0.08816458284854889,\n      -0.03268962353467941,\n      0.04667510464787483,\n      0.09249356389045715,\n      0.030153600499033928,\n      0.04217665269970894,\n      -0.017909688875079155,\n      0.09222012758255005,\n      -0.02565961889922619,\n      0.11821802705526352,\n      -0.02805268205702305,\n      -0.02142397128045559,\n      -0.03395047038793564,\n      0.009073002263903618,\n      0.11769955605268478,\n      0.03162392973899841,\n      -0.015209962613880634,\n      0.022647833451628685,\n      0.14840689301490784,\n      0.05038387328386307,\n      0.08908438682556152,\n      0.05139704421162605,\n      0.006649679504334927,\n      0.039100293070077896,\n      0.11597106605768204,\n      0.11157221347093582,\n      0.02750629559159279,\n      -0.021691862493753433,\n      0.06886351853609085,\n      -0.0809638723731041,\n      0.02943430095911026,\n      0.4270786643028259,\n      0.32810908555984497,\n      0.13074176013469696,\n      -0.02404167503118515,\n      -0.13626985251903534,\n      -0.0829707533121109,\n      0.06889078766107559,\n      0.09368015825748444,\n      -0.04880223423242569,\n      -0.004063749685883522\n    ],\n    [\n      0.2041318416595459,\n      0.08765263855457306,\n      0.1216077208518982,\n      0.12330946326255798,\n      0.03982587531208992,\n      0.06509526818990707,\n      -0.008540340699255466,\n      0.07831516861915588,\n      0.02484498918056488,\n      0.03591590374708176,\n      0.07839855551719666,\n      0.09679409861564636,\n      0.23509661853313446,\n      0.006494126748293638,\n      -0.360757440328598,\n      0.3461838662624359,\n      -0.08014769852161407,\n      0.10547541826963425,\n      0.11571671813726425,\n      0.0848110243678093,\n      0.057594530284404755,\n      0.15064531564712524,\n      0.19114898145198822,\n      0.07114484906196594,\n      0.02349281497299671,\n      0.10232406109571457,\n      0.2013712376356125,\n      -0.04224320501089096,\n      -0.05589652806520462,\n      -0.18206746876239777,\n      -0.07486578077077866,\n      -0.14313726127147675,\n      0.03702951967716217,\n      0.009755946695804596,\n      0.004316919948905706,\n      -0.07270006090402603,\n      -0.0009513023542240262,\n      0.018644209951162338,\n      -0.09795740246772766,\n      0.08105183392763138,\n      0.10848415642976761,\n      -0.009018761105835438,\n      0.02954171597957611,\n      0.013591788709163666,\n      -0.030213454738259315,\n      -0.02487827278673649,\n      0.04776447266340256,\n      -0.06552871316671371,\n      -0.05309249460697174,\n      -0.008304871618747711,\n      -0.16564272344112396,\n      -0.09566804766654968,\n      0.02963709831237793,\n      0.03721262887120247,\n      -0.026767995208501816,\n      -0.08356256783008575,\n      -0.04679994657635689,\n      0.10570079833269119,\n      0.04559659585356712,\n      0.023272082209587097,\n      0.031264569610357285,\n      -0.02241915836930275,\n      -0.02124848961830139,\n      -0.04859567806124687,\n      0.009455736726522446,\n      0.12406361848115921,\n      0.05043920874595642,\n      0.03687583655118942,\n      -0.04472045600414276,\n      0.061137083917856216,\n      -0.009739232249557972,\n      -0.0428059846162796,\n      0.09927906841039658,\n      -0.0925896167755127,\n      0.04901793226599693,\n      -0.08795490860939026,\n      0.091715008020401,\n      -0.039222270250320435,\n      -0.03404815122485161,\n      -0.09801219403743744,\n      -0.0198904350399971,\n      0.11121173948049545,\n      -0.01898583211004734,\n      0.12418501079082489,\n      -0.007442523259669542,\n      -0.007172151934355497,\n      -0.025854123756289482,\n      0.05849194526672363,\n      0.13805294036865234,\n      0.04356708377599716,\n      -0.04443356394767761,\n      0.09362105280160904,\n      0.07846643030643463,\n      0.09738825261592865,\n      0.1283743679523468,\n      -0.019804801791906357,\n      -0.013410135172307491,\n      0.014600556343793869,\n      0.039361242204904556,\n      -0.06081139296293259,\n      -0.018201518803834915,\n      0.009751546196639538,\n      0.12145578861236572,\n      0.20513486862182617,\n      0.04842638596892357,\n      -0.03152118995785713,\n      -0.06204438582062721,\n      0.011561308056116104,\n      0.1401057243347168,\n      0.19362884759902954,\n      0.02110973559319973,\n      0.033778078854084015,\n      0.05405988171696663,\n      0.012829885818064213,\n      0.07340481132268906,\n      0.43243712186813354,\n      0.2642299234867096,\n      0.007458480540663004,\n      0.050956983119249344,\n      -0.10151463747024536,\n      -0.09145573526620865,\n      -0.09389383345842361,\n      0.08538578450679779,\n      0.03537726774811745,\n      0.07217693328857422\n    ],\n    [\n      0.2041379064321518,\n      0.034962281584739685,\n      0.10934015363454819,\n      0.1924244463443756,\n      -0.030537661164999008,\n      -0.04196394234895706,\n      0.10048594325780869,\n      0.07361004501581192,\n      0.09659435600042343,\n      -0.007747471798211336,\n      0.0632680207490921,\n      0.14311014115810394,\n      0.21035851538181305,\n      0.09850937128067017,\n      -0.38974976539611816,\n      0.27413469552993774,\n      -0.012059548869729042,\n      0.148288294672966,\n      0.009894994087517262,\n      0.0838686004281044,\n      -0.012042570859193802,\n      0.11142680048942566,\n      0.01639346405863762,\n      0.026497220620512962,\n      -0.10975003242492676,\n      -0.02814510650932789,\n      0.12638168036937714,\n      -0.004192285239696503,\n      -0.07536064088344574,\n      -0.1446569412946701,\n      0.046857524663209915,\n      -0.06100057438015938,\n      -0.01108519546687603,\n      0.033191997557878494,\n      -0.042014483362436295,\n      0.023960640653967857,\n      -0.026986131444573402,\n      -0.0342225506901741,\n      -0.04858268052339554,\n      -0.04169023782014847,\n      -0.018557822331786156,\n      -0.04555230215191841,\n      0.05278554931282997,\n      -0.07619721442461014,\n      -0.05544494837522507,\n      0.03234458714723587,\n      -0.10927484929561615,\n      -0.1276896595954895,\n      0.04521329700946808,\n      -0.04108966886997223,\n      -0.12902024388313293,\n      -0.12737388908863068,\n      0.04608333855867386,\n      -0.0949205681681633,\n      -0.10100243240594864,\n      -0.1613752692937851,\n      -0.13196617364883423,\n      -0.022257046774029732,\n      0.10946308821439743,\n      -0.10095767676830292,\n      -0.07492989301681519,\n      -0.05214900150895119,\n      0.05923905968666077,\n      -0.006596412975341082,\n      -0.13770565390586853,\n      -0.016643846407532692,\n      -0.10472042113542557,\n      0.018201079219579697,\n      0.09695667773485184,\n      0.08132722973823547,\n      -0.012288297526538372,\n      0.011647497303783894,\n      -0.006038390100002289,\n      -0.05953162536025047,\n      -0.02996949478983879,\n      -0.10821452736854553,\n      0.07930788397789001,\n      0.04665719345211983,\n      -0.07881501317024231,\n      0.028918467462062836,\n      -0.03878683224320412,\n      -0.04174438491463661,\n      0.04792303591966629,\n      -0.08240633457899094,\n      0.10533419251441956,\n      -0.16030173003673553,\n      0.10912534594535828,\n      0.06878642737865448,\n      0.13932296633720398,\n      0.009799382649362087,\n      0.08927125483751297,\n      0.06348152458667755,\n      -0.0125749995931983,\n      0.06003931164741516,\n      0.07693960517644882,\n      -0.008536451496183872,\n      0.06937231123447418,\n      0.0714561715722084,\n      0.06706144660711288,\n      0.05875370278954506,\n      -0.05463311821222305,\n      -0.017017154023051262,\n      0.1348949372768402,\n      0.0006826665485277772,\n      0.08455720543861389,\n      0.049587611109018326,\n      -0.013453060761094093,\n      0.04086603224277496,\n      0.1289636641740799,\n      0.030711177736520767,\n      0.02954794466495514,\n      0.004580617882311344,\n      0.05700932815670967,\n      0.047467779368162155,\n      0.038585953414440155,\n      0.380965918302536,\n      0.1434822380542755,\n      0.09435157477855682,\n      0.009723564609885216,\n      -0.051269326359033585,\n      -0.08257004618644714,\n      0.011450226418673992,\n      0.03293497860431671,\n      0.004985630977898836,\n      0.05133064463734627\n    ]\n  ],\n  "hidden_bias": [\n    -0.05621889606118202,\n    -0.019115613773465157,\n    -0.034422170370817184,\n    -0.05893378332257271,\n    0.12219759076833725,\n    0.016282206401228905,\n    0.16965745389461517,\n    0.23360852897167206,\n    -0.05097753927111626,\n    0.07897807657718658,\n    0.0029049937147647142,\n    0.0856524109840393,\n    0.06322012096643448,\n    -0.0415206104516983,\n    0.26465508341789246,\n    -0.03580057621002197,\n    0.1157040148973465,\n    0.17678402364253998,\n    0.06413913518190384,\n    -0.0065202913247048855,\n    -0.06348580121994019,\n    0.04923239350318909,\n    0.24278776347637177,\n    0.023842865601181984,\n    0.13564586639404297,\n    -0.08616997301578522,\n    0.15125975012779236,\n    0.22613205015659332,\n    0.03907884284853935,\n    0.059309523552656174,\n    0.23390400409698486,\n    0.12049560248851776,\n    -0.0757390558719635,\n    -0.048587068915367126,\n    -0.06801921874284744,\n    -0.021018167957663536,\n    0.03770667314529419,\n    0.09541088342666626,\n    0.0013614082708954811,\n    0.1662861406803131,\n    0.05561617389321327,\n    -0.06874682754278183,\n    -0.05922737717628479,\n    -0.09176001697778702,\n    0.18415987491607666,\n    -0.0046700648963451385,\n    0.12471731007099152,\n    0.07760708779096603,\n    0.23359078168869019,\n    -0.07944095879793167,\n    0.2265137881040573,\n    0.05842474848031998,\n    0.18367010354995728,\n    -0.04620521888136864,\n    -0.012569636106491089,\n    -0.09429153054952621,\n    0.06431460380554199,\n    -0.007195712067186832,\n    -0.0729711726307869,\n    0.23842181265354156,\n    -0.012287686578929424,\n    0.008105174638330936,\n    -0.01617196947336197,\n    -0.0040941396728158\n  ],\n  "output_weight": [\n    -0.18037454783916473,\n    -0.21211254596710205,\n    -0.10587097704410553,\n    0.0,\n    0.14978133141994476,\n    -0.16929154098033905,\n    0.17064155638217926,\n    0.11502841860055923,\n    -0.1284772902727127,\n    0.1026948094367981,\n    -0.17519542574882507,\n    0.16846208274364471,\n    -0.16227029263973236,\n    -0.1352003812789917,\n    0.6271225214004517,\n    -0.20685741305351257,\n    0.14921079576015472,\n    0.1991942822933197,\n    0.20539908111095428,\n    -0.23723261058330536,\n    -0.16070608794689178,\n    -0.21920329332351685,\n    0.13242486119270325,\n    -0.22753967344760895,\n    0.16346587240695953,\n    -0.7902892827987671,\n    0.16956284642219543,\n    0.1597963571548462,\n    -0.1789867877960205,\n    -0.15584002435207367,\n    0.13500654697418213,\n    0.17357833683490753,\n    -0.13280129432678223,\n    -0.1382886916399002,\n    -0.12276452034711838,\n    -0.18893349170684814,\n    -0.18767301738262177,\n    0.13466447591781616,\n    -0.20172612369060516,\n    0.13719883561134338,\n    -0.15825384855270386,\n    -0.12388750165700912,\n    -0.22110256552696228,\n    -0.14928315579891205,\n    0.1884916126728058,\n    -0.1635376363992691,\n    0.13104499876499176,\n    0.1642802506685257,\n    0.13330936431884766,\n    -0.18845392763614655,\n    0.15305529534816742,\n    -0.18959495425224304,\n    0.262775182723999,\n    -0.2180405855178833,\n    -0.16421689093112946,\n    -0.14148974418640137,\n    0.15590746700763702,\n    -0.1487697958946228,\n    -0.16371606290340424,\n    0.17154084146022797,\n    -0.14915665984153748,\n    -0.1468460112810135,\n    -0.19439435005187988,\n    -0.1377745419740677\n  ],\n  "output_bias": 0.15052388608455658,\n  "training": {\n    "method": "paired counterfactual policy distillation with residual scorer",\n    "seed": 20261019,\n    "regularization": 0.1,\n    "best_epoch": 26,\n    "validation_rank_loss": 0.771247386932373,\n    "baseline_validation_rank_loss": 1.188873529434204,\n    "training_games": 2400,\n    "validation_games": 599,\n    "source_dataset": {\n      "method": "paired one-step policy-improvement rollouts",\n      "input_features": [\n        "play",\n        "number",\n        "skip",\n        "reverse",\n        "draw_two",\n        "wild",\n        "wild_draw_four",\n        "remaining_colour_count",\n        "remaining_colour_fraction",\n        "remaining_match_count",\n        "attack_next_threat",\n        "reverse_escape_threat",\n        "wild_when_small_hand",\n        "finish",\n        "keep",\n        "choose_colour",\n        "next_can_finish",\n        "own_hand_size",\n        "successor_colour_absence",\n        "successor_symbol_absence",\n        "successor_colour_preference",\n        "successor_finish_risk",\n        "remaining_colour_components",\n        "largest_component_fraction",\n        "remaining_bridge_symbols",\n        "remaining_duplicates",\n        "remaining_colour_entropy",\n        "remaining_wilds",\n        "chosen_colour_exhausted",\n        "stack_penalty_play",\n        "successor_one_card_colour_absence",\n        "successor_one_card_symbol_absence",\n        "remaining_colour_slot_0_number_0",\n        "remaining_colour_slot_0_number_1",\n        "remaining_colour_slot_0_number_2",\n        "remaining_colour_slot_0_number_3",\n        "remaining_colour_slot_0_number_4",\n        "remaining_colour_slot_0_number_5",\n        "remaining_colour_slot_0_number_6",\n        "remaining_colour_slot_0_number_7",\n        "remaining_colour_slot_0_number_8",\n        "remaining_colour_slot_0_number_9",\n        "remaining_colour_slot_0_skip_None",\n        "remaining_colour_slot_0_reverse_None",\n        "remaining_colour_slot_0_draw_two_None",\n        "remaining_colour_slot_1_number_0",\n        "remaining_colour_slot_1_number_1",\n        "remaining_colour_slot_1_number_2",\n        "remaining_colour_slot_1_number_3",\n        "remaining_colour_slot_1_number_4",\n        "remaining_colour_slot_1_number_5",\n        "remaining_colour_slot_1_number_6",\n        "remaining_colour_slot_1_number_7",\n        "remaining_colour_slot_1_number_8",\n        "remaining_colour_slot_1_number_9",\n        "remaining_colour_slot_1_skip_None",\n        "remaining_colour_slot_1_reverse_None",\n        "remaining_colour_slot_1_draw_two_None",\n        "remaining_colour_slot_2_number_0",\n        "remaining_colour_slot_2_number_1",\n        "remaining_colour_slot_2_number_2",\n        "remaining_colour_slot_2_number_3",\n        "remaining_colour_slot_2_number_4",\n        "remaining_colour_slot_2_number_5",\n        "remaining_colour_slot_2_number_6",\n        "remaining_colour_slot_2_number_7",\n        "remaining_colour_slot_2_number_8",\n        "remaining_colour_slot_2_number_9",\n        "remaining_colour_slot_2_skip_None",\n        "remaining_colour_slot_2_reverse_None",\n        "remaining_colour_slot_2_draw_two_None",\n        "remaining_colour_slot_3_number_0",\n        "remaining_colour_slot_3_number_1",\n        "remaining_colour_slot_3_number_2",\n        "remaining_colour_slot_3_number_3",\n        "remaining_colour_slot_3_number_4",\n        "remaining_colour_slot_3_number_5",\n        "remaining_colour_slot_3_number_6",\n        "remaining_colour_slot_3_number_7",\n        "remaining_colour_slot_3_number_8",\n        "remaining_colour_slot_3_number_9",\n        "remaining_colour_slot_3_skip_None",\n        "remaining_colour_slot_3_reverse_None",\n        "remaining_colour_slot_3_draw_two_None",\n        "remaining_wild_count",\n        "remaining_wild_draw_four_count",\n        "action_number_0",\n        "action_number_1",\n        "action_number_2",\n        "action_number_3",\n        "action_number_4",\n        "action_number_5",\n        "action_number_6",\n        "action_number_7",\n        "action_number_8",\n        "action_number_9",\n        "opponent_slot_0_present",\n        "opponent_slot_0_hand_size",\n        "opponent_slot_0_one_card",\n        "opponent_slot_0_colour_absence",\n        "opponent_slot_0_symbol_absence",\n        "opponent_slot_0_colour_preference",\n        "opponent_slot_1_present",\n        "opponent_slot_1_hand_size",\n        "opponent_slot_1_one_card",\n        "opponent_slot_1_colour_absence",\n        "opponent_slot_1_symbol_absence",\n        "opponent_slot_1_colour_preference",\n        "opponent_slot_2_present",\n        "opponent_slot_2_hand_size",\n        "opponent_slot_2_one_card",\n        "opponent_slot_2_colour_absence",\n        "opponent_slot_2_symbol_absence",\n        "opponent_slot_2_colour_preference",\n        "selected_is_active_colour",\n        "pending_penalty_size",\n        "turn_progress",\n        "active_opponents",\n        "closest_opponent_risk",\n        "route_sequence_fraction",\n        "route_sequence_stranded",\n        "route_ending_wild",\n        "route_next_colour_absence",\n        "route_next_symbol_absence",\n        "route_next_colour_preference"\n      ],\n      "feature_version": 4,\n      "games": 3000,\n      "states": 11872,\n      "actions": 38902,\n      "trials": 8,\n      "future_shuffle": true,\n      "reward_conditioning": "present hidden simulator hands held fixed; future draw-pile permutations averaged",\n      "max_actions": 8,\n      "seed_start": 49000000,\n      "sdk_commit": "0cda3acc1bb5c05fa1803a8fbd7761770dc6d5fe",\n      "incumbent": {\n        "format": "triad-uno-route-linear-v1",\n        "features": [\n          "play",\n          "number",\n          "skip",\n          "reverse",\n          "draw_two",\n          "wild",\n          "wild_draw_four",\n          "remaining_colour_count",\n          "remaining_colour_fraction",\n          "remaining_match_count",\n          "attack_next_threat",\n          "reverse_escape_threat",\n          "wild_when_small_hand",\n          "finish",\n          "keep",\n          "choose_colour",\n          "next_can_finish",\n          "own_hand_size",\n          "sequence_fraction",\n          "sequence_stranded",\n          "ending_wild",\n          "next_colour_absence",\n          "next_symbol_absence",\n          "next_colour_preference"\n        ],\n        "weights": [\n          8.022466349142684,\n          0.44153069585311683,\n          -0.06834035646341297,\n          1.4643534535685507,\n          -2.338619866848944,\n          -3.4540620544700498,\n          -4.276279368110987,\n          1.6602353386949849,\n          1.0165100023007814,\n          0.901341302040876,\n          3.853185081203389,\n          2.4436903583872347,\n          1.7223969671761579,\n          11.872419584625783,\n          -4.598739912392293,\n          3.347870880522802,\n          0.32778952616834095,\n          -0.24548152213113314,\n          0.054801310646275025,\n          -1.1349214815798592,\n          -0.04225874297028254,\n          1.1191018681573717,\n          0.21891410472226475,\n          -0.08122782541248305\n        ],\n        "training": {\n          "method": "frozen-base residual CEM, own-hand bounded sequence features",\n          "seed": 20261019,\n          "iterations": 24,\n          "sdk_commit": "0cda3acc1bb5c05fa1803a8fbd7761770dc6d5fe",\n          "selection_seed_start": 21000000,\n          "test_seed_start": 22000000,\n          "base": {\n            "format": "triad-uno-linear-v1",\n            "features": [\n              "play",\n              "number",\n              "skip",\n              "reverse",\n              "draw_two",\n              "wild",\n              "wild_draw_four",\n              "remaining_colour_count",\n              "remaining_colour_fraction",\n              "remaining_match_count",\n              "attack_next_threat",\n              "reverse_escape_threat",\n              "wild_when_small_hand",\n              "finish",\n              "keep",\n              "choose_colour",\n              "next_can_finish",\n              "own_hand_size"\n            ],\n            "weights": [\n              8.022466349142684,\n              0.44153069585311683,\n              -0.06834035646341297,\n              1.4643534535685507,\n              -2.338619866848944,\n              -3.4540620544700498,\n              -4.276279368110987,\n              1.6602353386949849,\n              1.0165100023007814,\n              0.901341302040876,\n              3.853185081203389,\n              2.4436903583872347,\n              1.7223969671761579,\n              11.872419584625783,\n              -4.598739912392293,\n              3.347870880522802,\n              0.32778952616834095,\n              -0.24548152213113314\n            ],\n            "training": {\n              "method": "cross-entropy policy optimization",\n              "seed": 20261014,\n              "generations": 30,\n              "population": 32,\n              "games_per_candidate": 64,\n              "validation": 3.005859375,\n              "sdk_commit": "0cda3acc1bb5c05fa1803a8fbd7761770dc6d5fe",\n              "feature_count": 18\n            }\n          }\n        }\n      },\n      "seconds": 1638.4,\n      "reservoir_sample": true,\n      "own_hand_size_histogram": {\n        "1": 106,\n        "2": 1146,\n        "3": 1762,\n        "4": 1882,\n        "5": 1900,\n        "6": 1847,\n        "7": 1744,\n        "8": 566,\n        "9": 361,\n        "10": 225,\n        "11": 139,\n        "12": 74,\n        "13": 52,\n        "14": 27,\n        "15": 21,\n        "16": 9,\n        "17": 5,\n        "18": 4,\n        "19": 1,\n        "22": 1\n      },\n      "phase_histogram": {\n        "TURN": 9755,\n        "AFTER_DRAW": 2104,\n        "CHOOSE_INITIAL_COLOR": 13\n      }\n    },\n    "sdk_commit": "0cda3acc1bb5c05fa1803a8fbd7761770dc6d5fe",\n    "residual_scale": 1.0\n  }\n}\n', 'neural-ranking.json': '{\n  "format": "triad-uno-residual-mlp-v3",\n  "features": [\n    "play",\n    "number",\n    "skip",\n    "reverse",\n    "draw_two",\n    "wild",\n    "wild_draw_four",\n    "remaining_colour_count",\n    "remaining_colour_fraction",\n    "remaining_match_count",\n    "attack_next_threat",\n    "reverse_escape_threat",\n    "wild_when_small_hand",\n    "finish",\n    "keep",\n    "choose_colour",\n    "next_can_finish",\n    "own_hand_size",\n    "successor_colour_absence",\n    "successor_symbol_absence",\n    "successor_colour_preference",\n    "successor_finish_risk",\n    "remaining_colour_components",\n    "largest_component_fraction",\n    "remaining_bridge_symbols",\n    "remaining_duplicates",\n    "remaining_colour_entropy",\n    "remaining_wilds",\n    "chosen_colour_exhausted",\n    "stack_penalty_play",\n    "successor_one_card_colour_absence",\n    "successor_one_card_symbol_absence",\n    "remaining_colour_slot_0_number_0",\n    "remaining_colour_slot_0_number_1",\n    "remaining_colour_slot_0_number_2",\n    "remaining_colour_slot_0_number_3",\n    "remaining_colour_slot_0_number_4",\n    "remaining_colour_slot_0_number_5",\n    "remaining_colour_slot_0_number_6",\n    "remaining_colour_slot_0_number_7",\n    "remaining_colour_slot_0_number_8",\n    "remaining_colour_slot_0_number_9",\n    "remaining_colour_slot_0_skip_None",\n    "remaining_colour_slot_0_reverse_None",\n    "remaining_colour_slot_0_draw_two_None",\n    "remaining_colour_slot_1_number_0",\n    "remaining_colour_slot_1_number_1",\n    "remaining_colour_slot_1_number_2",\n    "remaining_colour_slot_1_number_3",\n    "remaining_colour_slot_1_number_4",\n    "remaining_colour_slot_1_number_5",\n    "remaining_colour_slot_1_number_6",\n    "remaining_colour_slot_1_number_7",\n    "remaining_colour_slot_1_number_8",\n    "remaining_colour_slot_1_number_9",\n    "remaining_colour_slot_1_skip_None",\n    "remaining_colour_slot_1_reverse_None",\n    "remaining_colour_slot_1_draw_two_None",\n    "remaining_colour_slot_2_number_0",\n    "remaining_colour_slot_2_number_1",\n    "remaining_colour_slot_2_number_2",\n    "remaining_colour_slot_2_number_3",\n    "remaining_colour_slot_2_number_4",\n    "remaining_colour_slot_2_number_5",\n    "remaining_colour_slot_2_number_6",\n    "remaining_colour_slot_2_number_7",\n    "remaining_colour_slot_2_number_8",\n    "remaining_colour_slot_2_number_9",\n    "remaining_colour_slot_2_skip_None",\n    "remaining_colour_slot_2_reverse_None",\n    "remaining_colour_slot_2_draw_two_None",\n    "remaining_colour_slot_3_number_0",\n    "remaining_colour_slot_3_number_1",\n    "remaining_colour_slot_3_number_2",\n    "remaining_colour_slot_3_number_3",\n    "remaining_colour_slot_3_number_4",\n    "remaining_colour_slot_3_number_5",\n    "remaining_colour_slot_3_number_6",\n    "remaining_colour_slot_3_number_7",\n    "remaining_colour_slot_3_number_8",\n    "remaining_colour_slot_3_number_9",\n    "remaining_colour_slot_3_skip_None",\n    "remaining_colour_slot_3_reverse_None",\n    "remaining_colour_slot_3_draw_two_None",\n    "remaining_wild_count",\n    "remaining_wild_draw_four_count",\n    "action_number_0",\n    "action_number_1",\n    "action_number_2",\n    "action_number_3",\n    "action_number_4",\n    "action_number_5",\n    "action_number_6",\n    "action_number_7",\n    "action_number_8",\n    "action_number_9",\n    "opponent_slot_0_present",\n    "opponent_slot_0_hand_size",\n    "opponent_slot_0_one_card",\n    "opponent_slot_0_colour_absence",\n    "opponent_slot_0_symbol_absence",\n    "opponent_slot_0_colour_preference",\n    "opponent_slot_1_present",\n    "opponent_slot_1_hand_size",\n    "opponent_slot_1_one_card",\n    "opponent_slot_1_colour_absence",\n    "opponent_slot_1_symbol_absence",\n    "opponent_slot_1_colour_preference",\n    "opponent_slot_2_present",\n    "opponent_slot_2_hand_size",\n    "opponent_slot_2_one_card",\n    "opponent_slot_2_colour_absence",\n    "opponent_slot_2_symbol_absence",\n    "opponent_slot_2_colour_preference",\n    "selected_is_active_colour",\n    "pending_penalty_size",\n    "turn_progress",\n    "active_opponents",\n    "closest_opponent_risk",\n    "route_sequence_fraction",\n    "route_sequence_stranded",\n    "route_ending_wild",\n    "route_next_colour_absence",\n    "route_next_symbol_absence",\n    "route_next_colour_preference"\n  ],\n  "base_weights": [\n    0.40112331745713425,\n    0.022076534792655842,\n    -0.0034170178231706485,\n    0.07321767267842753,\n    -0.11693099334244722,\n    -0.1727031027235025,\n    -0.21381396840554934,\n    0.08301176693474925,\n    0.05082550011503907,\n    0.04506706510204381,\n    0.19265925406016948,\n    0.12218451791936175,\n    0.0861198483588079,\n    0.5936209792312892,\n    -0.22993699561961467,\n    0.16739354402614012,\n    0.01638947630841705,\n    -0.012274076106556658,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0027400655323137515,\n    -0.05674607407899296,\n    -0.002112937148514127,\n    0.055955093407868584,\n    0.010945705236113238,\n    -0.0040613912706241526\n  ],\n  "hidden_weight": [\n    [\n      0.047738585621118546,\n      0.04530264437198639,\n      -0.026320375502109528,\n      -0.06481600552797318,\n      -0.03419226035475731,\n      -0.030444851145148277,\n      -0.12304177135229111,\n      -0.07977847009897232,\n      0.0821894034743309,\n      0.058192018419504166,\n      -0.09686897695064545,\n      -0.10345493257045746,\n      -0.049430470913648605,\n      -0.05512383207678795,\n      0.15448737144470215,\n      -0.05907967686653137,\n      -0.04552450031042099,\n      0.041501644998788834,\n      -0.031808555126190186,\n      0.02713623456656933,\n      0.04283159598708153,\n      -0.08502308279275894,\n      -0.07638102024793625,\n      0.05363272503018379,\n      0.08228049427270889,\n      -0.07563139498233795,\n      -0.05343231186270714,\n      0.058682817965745926,\n      0.07096688449382782,\n      -0.026655763387680054,\n      0.04945972189307213,\n      -0.05443689599633217,\n      -0.06604204326868057,\n      0.02155235968530178,\n      0.006816371344029903,\n      -0.07560954242944717,\n      0.038441624492406845,\n      -0.09210237860679626,\n      -0.08227933198213577,\n      -0.11022494733333588,\n      -0.1063658595085144,\n      -0.0992869958281517,\n      0.005905965808779001,\n      0.048307865858078,\n      -0.028698069974780083,\n      -0.05057316645979881,\n      0.08259140700101852,\n      0.03883549943566322,\n      -0.07854883372783661,\n      0.0013100493233650923,\n      -0.016112331300973892,\n      -0.03142315894365311,\n      0.001955755054950714,\n      0.08283934742212296,\n      0.042265623807907104,\n      -0.026503710076212883,\n      -0.1116599291563034,\n      0.05980841442942619,\n      -0.11285991966724396,\n      0.06944479793310165,\n      -0.06529023498296738,\n      0.04962320253252983,\n      -0.03516548126935959,\n      0.025238389149308205,\n      0.08647136390209198,\n      -0.04955286905169487,\n      0.08442313224077225,\n      0.02698330208659172,\n      0.006492777727544308,\n      0.03893376886844635,\n      0.07249397039413452,\n      0.00805602315813303,\n      0.02603483945131302,\n      0.012915773317217827,\n      0.08114428073167801,\n      -0.0403614416718483,\n      -0.028814923018217087,\n      -0.0525723360478878,\n      -0.016056347638368607,\n      0.01979980804026127,\n      0.005917528178542852,\n      0.027910463511943817,\n      -0.02681351639330387,\n      -3.0159979360178113e-05,\n      0.004122069105505943,\n      0.010772095993161201,\n      0.09243236482143402,\n      0.020241746678948402,\n      -0.04777370020747185,\n      -0.062362778931856155,\n      0.02113698609173298,\n      0.04589397832751274,\n      0.03971610590815544,\n      -0.06791450083255768,\n      -0.08962661772966385,\n      0.003928742371499538,\n      0.059095580130815506,\n      0.041326895356178284,\n      0.05634100362658501,\n      0.06877649575471878,\n      -0.05491505563259125,\n      -0.027139566838741302,\n      -0.04037068411707878,\n      0.11771722137928009,\n      0.03434263542294502,\n      0.010288005694746971,\n      0.060451626777648926,\n      -0.04209170863032341,\n      0.018731167539954185,\n      0.08030884712934494,\n      -0.03158421441912651,\n      -0.07176119834184647,\n      0.052817706018686295,\n      0.06781274825334549,\n      -0.0007142520626075566,\n      -0.18673691153526306,\n      0.055134665220975876,\n      -0.031001295894384384,\n      0.03764013573527336,\n      0.060473669320344925,\n      0.04709870368242264,\n      0.07622501254081726,\n      0.04636778309941292,\n      0.10234473645687103,\n      0.05999011918902397\n    ],\n    [\n      -0.02228645421564579,\n      -0.03808995708823204,\n      0.07333078980445862,\n      -0.00036325788823887706,\n      -0.023318396881222725,\n      0.0816844180226326,\n      -0.019215410575270653,\n      0.009710275568068027,\n      -0.08177532255649567,\n      -0.0729464739561081,\n      0.026545802131295204,\n      -0.02341197431087494,\n      0.05951903387904167,\n      -0.04999789968132973,\n      -0.07199382781982422,\n      -0.02147093415260315,\n      0.08994072675704956,\n      -0.10840170830488205,\n      0.01601979322731495,\n      0.06683018803596497,\n      -0.0369071364402771,\n      0.020175952464342117,\n      0.05310136079788208,\n      -0.031869374215602875,\n      -0.07632432132959366,\n      0.05851564556360245,\n      -0.06679441034793854,\n      -0.07735376060009003,\n      -0.019283335655927658,\n      -0.12427546828985214,\n      0.049108147621154785,\n      0.021748729050159454,\n      -0.059129904955625534,\n      0.03608942776918411,\n      -0.08310987800359726,\n      0.005248512141406536,\n      -0.09654765576124191,\n      -0.1631292849779129,\n      0.09912101179361343,\n      0.012877264991402626,\n      -0.11484351754188538,\n      -0.0017721550539135933,\n      0.04168080911040306,\n      -0.04130205512046814,\n      0.08812665939331055,\n      -0.07734452933073044,\n      0.10470343381166458,\n      -0.09274568408727646,\n      -0.10137621313333511,\n      -0.036295607686042786,\n      0.07175304740667343,\n      -0.0834583193063736,\n      -0.11707724630832672,\n      -0.009919453412294388,\n      -0.02899816818535328,\n      -0.028795763850212097,\n      0.06607798486948013,\n      0.04129472374916077,\n      -0.05075090378522873,\n      -0.05980079248547554,\n      0.08981551975011826,\n      -0.031181851401925087,\n      -0.05350785702466965,\n      0.16206452250480652,\n      -0.0868329331278801,\n      0.1048957034945488,\n      -0.026592938229441643,\n      -0.10185451805591583,\n      0.01900031790137291,\n      -0.00014398692292161286,\n      -0.04345191642642021,\n      -0.06366705894470215,\n      0.055861346423625946,\n      -0.11513396352529526,\n      -0.0193631611764431,\n      -0.005187303759157658,\n      0.05511026829481125,\n      -0.08126214146614075,\n      0.028684275224804878,\n      -0.047955069690942764,\n      0.0799245834350586,\n      0.029478121548891068,\n      -0.10215288400650024,\n      -0.06206067278981209,\n      -0.15079055726528168,\n      0.062476761639118195,\n      0.011727892793715,\n      0.070273756980896,\n      0.07065194845199585,\n      -0.040273331105709076,\n      -0.031604036688804626,\n      0.032861124724149704,\n      -0.06630592793226242,\n      -0.06591387838125229,\n      -0.09984106570482254,\n      0.009211503900587559,\n      -0.072581946849823,\n      0.0024301151279360056,\n      0.03331320360302925,\n      -0.005701635032892227,\n      -0.021352913230657578,\n      -0.03470514342188835,\n      -0.04400208964943886,\n      0.08839114010334015,\n      -0.018320152536034584,\n      -0.06198844686150551,\n      0.13086025416851044,\n      -0.013851356692612171,\n      -0.039179038256406784,\n      0.011407480575144291,\n      0.022818392142653465,\n      0.014680702239274979,\n      0.04701344668865204,\n      0.021441727876663208,\n      -0.03928621858358383,\n      -0.05444413051009178,\n      0.0835583359003067,\n      0.0506226010620594,\n      0.08726461976766586,\n      -0.052159279584884644,\n      0.019492819905281067,\n      0.02562870644032955,\n      -0.08451814949512482,\n      0.042497146874666214,\n      -0.03377889469265938\n    ],\n    [\n      -0.07080556452274323,\n      0.08681754767894745,\n      -0.017202306538820267,\n      0.012130343355238438,\n      -0.06905826181173325,\n      -0.0072688153013587,\n      -0.16803503036499023,\n      0.0013896360760554671,\n      0.08210405707359314,\n      0.04828304424881935,\n      -0.05288328602910042,\n      -0.05791717767715454,\n      -0.14075472950935364,\n      -0.008982043713331223,\n      0.1449107825756073,\n      -0.02077202871441841,\n      -0.09440851956605911,\n      -0.10520363599061966,\n      0.052648648619651794,\n      -0.09924038499593735,\n      -0.0331377349793911,\n      -0.02487393654882908,\n      0.038827866315841675,\n      0.02408275008201599,\n      0.09847383201122284,\n      -0.016067104414105415,\n      0.032632216811180115,\n      0.08190394937992096,\n      0.08236231654882431,\n      0.10184894502162933,\n      0.08561424911022186,\n      0.005068222992122173,\n      -0.019456472247838974,\n      -0.05392041429877281,\n      -0.03937513753771782,\n      -0.037318821996450424,\n      -0.07667900621891022,\n      0.05363183841109276,\n      -0.08975332230329514,\n      0.060678500682115555,\n      0.030160099267959595,\n      -0.003588881343603134,\n      -0.09733366966247559,\n      0.012892250902950764,\n      0.00722765875980258,\n      0.047118157148361206,\n      -0.004225754179060459,\n      0.04521437734365463,\n      -0.024340372532606125,\n      -0.05098627507686615,\n      0.017077669501304626,\n      0.05953528732061386,\n      -0.08598965406417847,\n      -0.03630896657705307,\n      0.08595851063728333,\n      0.03398044779896736,\n      -0.11402542144060135,\n      -0.0629686713218689,\n      -0.007939154282212257,\n      0.07726116478443146,\n      0.04972054064273834,\n      -0.03138855844736099,\n      -0.08020288497209549,\n      0.003680228954181075,\n      0.006638652179390192,\n      0.04327964782714844,\n      0.047692641615867615,\n      0.06545274704694748,\n      -0.08780302852392197,\n      -0.019618628546595573,\n      0.114493228495121,\n      -0.0935644805431366,\n      0.10182882845401764,\n      0.02560233883559704,\n      -0.025032641366124153,\n      -0.034980371594429016,\n      -0.0486622229218483,\n      0.007238175719976425,\n      0.03561169654130936,\n      -0.02435951679944992,\n      -0.015976620838046074,\n      -0.04366361349821091,\n      0.051616281270980835,\n      -0.005677057430148125,\n      -0.02699228748679161,\n      0.06428792327642441,\n      -0.03862493485212326,\n      0.05373683571815491,\n      0.0723569393157959,\n      0.038745176047086716,\n      0.0731341689825058,\n      -0.04704994708299637,\n      0.012248681858181953,\n      0.011648326180875301,\n      -0.07206045091152191,\n      0.05528585612773895,\n      0.07975253462791443,\n      -0.0384870320558548,\n      -0.06182728335261345,\n      0.08525111526250839,\n      -0.06846064329147339,\n      -0.11683151870965958,\n      0.028268955647945404,\n      0.10097561776638031,\n      -0.042028676718473434,\n      0.06358709186315536,\n      0.0030484094750136137,\n      -0.10105255246162415,\n      0.02849739044904709,\n      0.03320834040641785,\n      0.049378931522369385,\n      0.019081130623817444,\n      0.07623183727264404,\n      0.027418378740549088,\n      -0.0003680663648992777,\n      -0.20321089029312134,\n      0.018881581723690033,\n      0.006683526560664177,\n      0.06026441231369972,\n      0.06468027085065842,\n      0.1604427993297577,\n      0.04842478036880493,\n      0.020106706768274307,\n      0.03242427855730057,\n      -0.029879562556743622\n    ],\n    [\n      0.04350023716688156,\n      -0.06862401217222214,\n      -0.0593281053006649,\n      -0.07599495351314545,\n      0.0001640848204260692,\n      0.10042577236890793,\n      0.03843185678124428,\n      0.13790978491306305,\n      -0.002773661632090807,\n      0.07139089703559875,\n      0.0992383360862732,\n      -0.04784439504146576,\n      0.03472907096147537,\n      0.0027269700076431036,\n      -0.002967554610222578,\n      -0.010646043345332146,\n      0.060007136315107346,\n      0.02125142142176628,\n      -0.04520518705248833,\n      0.10214384645223618,\n      -0.06629222631454468,\n      0.06190263107419014,\n      -0.029220709577202797,\n      0.02116044983267784,\n      -0.08081132918596268,\n      -0.06533344835042953,\n      0.04898104444146156,\n      -0.10148542374372482,\n      0.09695793688297272,\n      -0.12717977166175842,\n      -0.13668474555015564,\n      0.03967823460698128,\n      -0.0227789506316185,\n      0.012372573837637901,\n      -0.014887251891195774,\n      0.050448931753635406,\n      -0.04985424503684044,\n      0.07449159771203995,\n      -0.057561859488487244,\n      0.10684950649738312,\n      0.09445708990097046,\n      -0.01976545713841915,\n      0.018880197778344154,\n      -0.08448504656553268,\n      0.07647614926099777,\n      0.02467709593474865,\n      -0.07290735095739365,\n      -0.034993723034858704,\n      -0.05385340377688408,\n      0.006172304041683674,\n      -0.05057835578918457,\n      -0.07547833770513535,\n      0.008080760948359966,\n      -0.08293446153402328,\n      -0.08460673689842224,\n      -0.0307150948792696,\n      0.01986280456185341,\n      -0.028015725314617157,\n      0.10640309005975723,\n      0.04832981899380684,\n      -0.015222043730318546,\n      -0.006632605567574501,\n      0.11403471976518631,\n      0.034995485097169876,\n      0.057301297783851624,\n      0.01634867489337921,\n      0.019071422517299652,\n      0.07475816458463669,\n      -0.03630269318819046,\n      0.005953290965408087,\n      -0.03892500326037407,\n      0.006093563046306372,\n      -0.04481792449951172,\n      -0.03875832259654999,\n      -0.033164188265800476,\n      0.0005660844035446644,\n      0.02571749873459339,\n      -0.05155565217137337,\n      -0.10759879648685455,\n      0.032551541924476624,\n      -0.07687050849199295,\n      0.07311543822288513,\n      -0.08210933208465576,\n      0.033452108502388,\n      -0.016436534002423286,\n      -0.06129077076911926,\n      0.035728175193071365,\n      0.010095350444316864,\n      -0.02669556252658367,\n      0.055211350321769714,\n      -0.034412775188684464,\n      0.08118234574794769,\n      0.04039853811264038,\n      0.05478880554437637,\n      -0.04855401813983917,\n      0.07386486977338791,\n      0.0614943690598011,\n      0.03302685543894768,\n      0.035179782658815384,\n      0.0736321434378624,\n      0.011151469312608242,\n      0.090388223528862,\n      -0.04599872976541519,\n      -0.030535966157913208,\n      -0.02301563136279583,\n      -0.09580010920763016,\n      0.057720933109521866,\n      0.09337203204631805,\n      0.04975887015461922,\n      -0.006984734442085028,\n      -0.03077240101993084,\n      0.0019145382102578878,\n      0.07796673476696014,\n      0.01344483345746994,\n      0.04135727882385254,\n      -0.0718839094042778,\n      -0.023647191002964973,\n      -0.01014042366296053,\n      0.037801504135131836,\n      -0.055986374616622925,\n      0.012989292852580547,\n      -0.057058803737163544,\n      0.038028404116630554,\n      0.011890711262822151,\n      0.08138334006071091\n    ],\n    [\n      -0.03116629645228386,\n      0.07127878814935684,\n      -0.06136632338166237,\n      0.03828037530183792,\n      0.0041648270562291145,\n      -0.001647814642637968,\n      -0.049769360572099686,\n      0.11232021450996399,\n      -0.053115036338567734,\n      0.02684471383690834,\n      -0.09385635703802109,\n      -0.012994170188903809,\n      0.06437179446220398,\n      0.0182111244648695,\n      -0.13599151372909546,\n      -0.035834383219480515,\n      -0.0633690282702446,\n      0.05572327971458435,\n      -0.002524257404729724,\n      0.08343074470758438,\n      0.04900464043021202,\n      0.04452045261859894,\n      0.0410354882478714,\n      0.14215271174907684,\n      0.09081637859344482,\n      -0.03281678259372711,\n      0.02169635519385338,\n      -0.015560050494968891,\n      -0.03550639748573303,\n      -0.09974951297044754,\n      0.026047002524137497,\n      -0.07893669605255127,\n      0.030387450009584427,\n      0.0968974381685257,\n      0.04588461294770241,\n      0.13220132887363434,\n      0.020487256348133087,\n      -0.020866425707936287,\n      -0.006257360801100731,\n      0.05663213133811951,\n      -0.0009676795452833176,\n      -0.07863062620162964,\n      -0.012216913513839245,\n      0.06253993511199951,\n      -0.031075920909643173,\n      0.09939336031675339,\n      -0.006200780626386404,\n      0.1437220275402069,\n      0.031181905418634415,\n      0.10355105996131897,\n      0.023040123283863068,\n      0.02497902140021324,\n      -0.0490502193570137,\n      0.12211234122514725,\n      -0.013617209158837795,\n      -0.04034322872757912,\n      0.009939284063875675,\n      -0.04864238575100899,\n      0.02291889488697052,\n      -0.08481185138225555,\n      -0.06803499162197113,\n      -0.034959644079208374,\n      0.11438185721635818,\n      -0.02687300369143486,\n      -0.021627184003591537,\n      0.02240157499909401,\n      0.02606283128261566,\n      0.014172214083373547,\n      -0.028376474976539612,\n      0.09751824289560318,\n      0.007987592369318008,\n      -0.06103547662496567,\n      -0.005904955789446831,\n      -0.03319164738059044,\n      -0.017295293509960175,\n      -0.05225905403494835,\n      -0.030460484325885773,\n      -0.020315490663051605,\n      0.07339377701282501,\n      0.06240595877170563,\n      0.021245691925287247,\n      0.06894552707672119,\n      0.04555058106780052,\n      -0.06245541572570801,\n      -0.009654292836785316,\n      -0.0249693114310503,\n      0.060277506709098816,\n      0.008175822906196117,\n      -0.019206754863262177,\n      -0.00026347438688389957,\n      0.10235624760389328,\n      -0.01897374726831913,\n      0.06933698058128357,\n      0.01675226353108883,\n      0.012783409096300602,\n      0.08346664905548096,\n      0.0036422177217900753,\n      0.04877013340592384,\n      0.07240533083677292,\n      0.07643114775419235,\n      0.008141978643834591,\n      0.02397869899868965,\n      -0.025475729256868362,\n      0.08919262886047363,\n      -0.05508946627378464,\n      -0.08739158511161804,\n      -0.03624549135565758,\n      0.13181065022945404,\n      -0.05134525150060654,\n      0.008041281253099442,\n      -0.060822125524282455,\n      0.018873997032642365,\n      -0.05036201328039169,\n      0.011138616129755974,\n      0.0645395964384079,\n      0.01824404112994671,\n      0.013962469063699245,\n      -0.008526302874088287,\n      0.051661863923072815,\n      0.08447287231683731,\n      -0.15530741214752197,\n      -0.09084067493677139,\n      0.09207586199045181,\n      0.07959631085395813,\n      -0.07337072491645813\n    ],\n    [\n      -0.07242389768362045,\n      -0.02305672876536846,\n      0.04166818782687187,\n      -0.0910407155752182,\n      -0.045460984110832214,\n      0.06806130707263947,\n      0.0214729905128479,\n      -0.010541185736656189,\n      0.040091730654239655,\n      0.044411610811948776,\n      -0.06728976219892502,\n      -0.08463599532842636,\n      -0.07767392694950104,\n      -0.18349771201610565,\n      0.021469589322805405,\n      -0.0031497622840106487,\n      0.09945493191480637,\n      0.027874061837792397,\n      0.12228601425886154,\n      -0.09135217219591141,\n      0.056330375373363495,\n      -0.08656202256679535,\n      0.04463307932019234,\n      -0.03029477410018444,\n      0.016285572201013565,\n      -0.06630558520555496,\n      -0.02868707850575447,\n      -0.1115921288728714,\n      -0.061264630407094955,\n      -0.07797146588563919,\n      0.08847156912088394,\n      -0.07684928923845291,\n      -0.001168232993222773,\n      0.021277688443660736,\n      -0.08319683372974396,\n      0.0248080026358366,\n      -0.050778768956661224,\n      0.008687189780175686,\n      -0.052181243896484375,\n      -0.03269185870885849,\n      0.040109362453222275,\n      -0.0032630430068820715,\n      0.07607730478048325,\n      -0.03378315642476082,\n      0.027813544496893883,\n      0.05842326581478119,\n      -0.10025113821029663,\n      -0.057107679545879364,\n      -0.06456020474433899,\n      0.04782576486468315,\n      -0.02629663795232773,\n      0.04624244570732117,\n      0.07432769238948822,\n      0.07981118559837341,\n      0.05082719773054123,\n      0.0975283831357956,\n      0.06991080194711685,\n      0.026100020855665207,\n      0.04106482118368149,\n      0.03216780349612236,\n      -0.0026397411711513996,\n      -0.04440958425402641,\n      -0.007284882478415966,\n      0.054573722183704376,\n      0.06412909179925919,\n      0.1282527893781662,\n      0.10284024477005005,\n      0.13035732507705688,\n      0.056927721947431564,\n      -0.05553198233246803,\n      0.021011024713516235,\n      0.03464658930897713,\n      -0.10059040784835815,\n      -0.01603543385863304,\n      -0.03990333899855614,\n      -0.051083698868751526,\n      0.050906889140605927,\n      -0.015516859479248524,\n      -0.04676184058189392,\n      0.06080302968621254,\n      0.002967587672173977,\n      -0.06859799474477768,\n      0.012254015542566776,\n      0.05860530585050583,\n      -0.11854080855846405,\n      -0.04758353903889656,\n      -0.01708359457552433,\n      0.07156646996736526,\n      -0.04285571724176407,\n      0.10644043982028961,\n      -0.01851535215973854,\n      -0.046184226870536804,\n      -0.037972159683704376,\n      0.0036114202812314034,\n      -0.013516053557395935,\n      -0.02001824416220188,\n      -0.022421959787607193,\n      0.06676957756280899,\n      0.14365990459918976,\n      0.1091172993183136,\n      -0.06207557022571564,\n      0.009599631652235985,\n      0.07031317055225372,\n      0.02140015736222267,\n      0.06771907955408096,\n      0.0070265308022499084,\n      -0.004522112663835287,\n      -0.06442725658416748,\n      -0.07613832503557205,\n      0.06309622526168823,\n      -0.06786011159420013,\n      -0.020132629200816154,\n      0.02939838171005249,\n      0.08830863982439041,\n      -0.031801071017980576,\n      -0.12543556094169617,\n      0.04184787720441818,\n      -0.00892546959221363,\n      0.007270362228155136,\n      -0.09732387214899063,\n      0.07281838357448578,\n      -0.07862372696399689,\n      0.05336592718958855,\n      0.011781525798141956,\n      -0.025552542880177498\n    ],\n    [\n      0.08717276901006699,\n      -0.08582322299480438,\n      -0.0715932548046112,\n      0.005282683297991753,\n      -0.10822644084692001,\n      -0.000633931253105402,\n      0.06555814296007156,\n      0.007445460185408592,\n      -0.12286113947629929,\n      0.01637551560997963,\n      0.08308613300323486,\n      0.03541797026991844,\n      0.08285904675722122,\n      0.0010968467686325312,\n      -0.05326712876558304,\n      -0.11814985424280167,\n      -0.07381923496723175,\n      0.02586120367050171,\n      0.07405996322631836,\n      0.11042749881744385,\n      0.014903586357831955,\n      0.03971828147768974,\n      0.07566241919994354,\n      0.09291592240333557,\n      -0.017846575006842613,\n      -0.01043247152119875,\n      0.010604296810925007,\n      -0.006155012175440788,\n      0.06708063930273056,\n      -0.14832369983196259,\n      -0.06893607974052429,\n      -0.06002559885382652,\n      0.00011600053403526545,\n      -0.016753194853663445,\n      0.051158592104911804,\n      0.09371411055326462,\n      -0.018797224387526512,\n      -0.02617119997739792,\n      -0.04826309159398079,\n      0.03809119388461113,\n      0.05250462889671326,\n      -0.07142286002635956,\n      -0.03590036556124687,\n      0.07937463372945786,\n      0.0866127610206604,\n      -0.0734536424279213,\n      -0.06423293799161911,\n      0.01319460291415453,\n      0.046713635325431824,\n      -0.09816429018974304,\n      -0.11974386125802994,\n      0.02018175646662712,\n      0.034000709652900696,\n      0.04301276430487633,\n      0.004672809969633818,\n      -0.08388657122850418,\n      0.07424010336399078,\n      -0.03318082541227341,\n      0.06885188817977905,\n      -0.06579697877168655,\n      0.02333010360598564,\n      -0.007994846440851688,\n      0.06548994034528732,\n      -0.09396348148584366,\n      0.032767776399850845,\n      0.07360856980085373,\n      -0.09715186059474945,\n      -0.05671629309654236,\n      0.07708180695772171,\n      -0.030735265463590622,\n      0.06655915081501007,\n      0.08664527535438538,\n      -0.11292892694473267,\n      0.03613870590925217,\n      0.05373154953122139,\n      -0.07872822880744934,\n      -0.0077574183233082294,\n      0.006584419868886471,\n      -0.13685736060142517,\n      0.05022869259119034,\n      -0.02237759158015251,\n      0.03873489797115326,\n      -0.0811312273144722,\n      0.006273975130170584,\n      -0.04428006708621979,\n      -0.054605644196271896,\n      -0.02015739679336548,\n      0.010528682731091976,\n      0.01791832223534584,\n      0.05139005929231644,\n      -0.06628908962011337,\n      -0.07200215756893158,\n      0.03263431787490845,\n      -0.04934372380375862,\n      0.09057989716529846,\n      -0.0005447635194286704,\n      -0.035734761506319046,\n      -0.12368553876876831,\n      0.04216707497835159,\n      -0.0459723100066185,\n      -0.021988719701766968,\n      0.007702947128564119,\n      0.004213304724544287,\n      0.00749873137101531,\n      -0.07161509990692139,\n      -0.005843827500939369,\n      -0.08050001412630081,\n      0.047045111656188965,\n      -0.0074531580321490765,\n      0.04789953678846359,\n      0.03300167992711067,\n      -0.04433402791619301,\n      -0.028184279799461365,\n      0.0045356135815382,\n      -0.01791565679013729,\n      -0.08591386675834656,\n      -0.022856108844280243,\n      0.014474689960479736,\n      0.0006420293939299881,\n      -0.027164364233613014,\n      -0.06254367530345917,\n      -0.11199925094842911,\n      0.011378326453268528,\n      0.05077851936221123,\n      0.04788101464509964\n    ],\n    [\n      0.10061518847942352,\n      0.04940604045987129,\n      0.14107462763786316,\n      0.07063549011945724,\n      0.05027444288134575,\n      -0.13802745938301086,\n      -0.029637297615408897,\n      -0.09525249153375626,\n      0.1590752750635147,\n      0.02212066575884819,\n      0.09263616055250168,\n      -0.013837362639605999,\n      0.08387580513954163,\n      0.11202551424503326,\n      -0.03800605982542038,\n      0.05068589746952057,\n      -0.09038975834846497,\n      -0.04922812432050705,\n      -0.08072277903556824,\n      0.05498248711228371,\n      0.0349210724234581,\n      -0.02748623862862587,\n      -0.098553366959095,\n      0.0811462551355362,\n      -0.008358928374946117,\n      0.021685481071472168,\n      -0.07366060465574265,\n      0.06559987366199493,\n      -0.1550823152065277,\n      0.0455445721745491,\n      0.058562830090522766,\n      -0.03194639831781387,\n      -0.10322638601064682,\n      -0.04330606013536453,\n      -0.01977594383060932,\n      -0.009016774594783783,\n      0.023005809634923935,\n      0.05192453786730766,\n      0.09879036247730255,\n      0.026764998212456703,\n      -0.12108907103538513,\n      -0.06409580260515213,\n      -0.06343840062618256,\n      -0.0342218279838562,\n      0.0051099867559969425,\n      -0.006784556433558464,\n      -0.06243666633963585,\n      -0.04737449437379837,\n      -0.03450634703040123,\n      -0.03465840220451355,\n      0.0049677020870149136,\n      -0.10786569863557816,\n      -0.14544135332107544,\n      -0.046166472136974335,\n      -0.0379723496735096,\n      -0.06530414521694183,\n      -0.11026740819215775,\n      -0.003523278282955289,\n      -0.10862144082784653,\n      -0.0555974543094635,\n      -0.0725642591714859,\n      -0.09372000396251678,\n      -0.06349944323301315,\n      0.02880578115582466,\n      -0.03806626424193382,\n      -0.011229501105844975,\n      -0.03433988243341446,\n      -0.06533566117286682,\n      0.033107172697782516,\n      -0.13123942911624908,\n      -0.12788444757461548,\n      -0.08977069705724716,\n      -0.02714865282177925,\n      0.05208035930991173,\n      -0.03080332651734352,\n      0.03142019733786583,\n      0.09771005064249039,\n      0.08423281461000443,\n      -0.005194915924221277,\n      -0.013087849132716656,\n      0.01823672093451023,\n      0.04602973908185959,\n      -0.0970597192645073,\n      -0.00884720403701067,\n      -0.023451585322618484,\n      0.054989032447338104,\n      0.10657624900341034,\n      0.05800230801105499,\n      -0.10224852710962296,\n      -0.0067805880680680275,\n      0.024124551564455032,\n      0.0482877716422081,\n      -0.03196490928530693,\n      -0.029735233634710312,\n      -0.007786205504089594,\n      -0.03309454023838043,\n      0.027680881321430206,\n      -0.03713155537843704,\n      0.07284291088581085,\n      -0.06607268005609512,\n      -0.0712386816740036,\n      -0.05083207041025162,\n      -0.015151962637901306,\n      0.03467213362455368,\n      -0.04948646202683449,\n      0.04167487099766731,\n      -0.07579493522644043,\n      0.006721334531903267,\n      0.08983799815177917,\n      -0.03376459330320358,\n      0.0594535693526268,\n      0.035227373242378235,\n      0.0009228350827470422,\n      0.009149760007858276,\n      -0.053080249577760696,\n      0.053381819278001785,\n      -0.12671679258346558,\n      -0.043604545295238495,\n      0.027639320120215416,\n      0.04181084409356117,\n      -0.12030643969774246,\n      0.10514648258686066,\n      -0.055875953286886215,\n      -0.03932512551546097,\n      0.0547432005405426\n    ],\n    [\n      -0.01826757751405239,\n      0.05057845264673233,\n      -0.014954062178730965,\n      0.031497031450271606,\n      0.12345704436302185,\n      -0.12058065086603165,\n      -0.039096664637327194,\n      -0.1160382628440857,\n      0.12295323610305786,\n      0.05956500396132469,\n      -0.047431353479623795,\n      0.029786959290504456,\n      -0.017987824976444244,\n      0.11468221247196198,\n      0.01531991921365261,\n      -0.12334255129098892,\n      0.02664029598236084,\n      -0.15338248014450073,\n      -0.10888620465993881,\n      -0.058815501630306244,\n      0.03060935251414776,\n      0.05457334592938423,\n      -0.022123809903860092,\n      0.024141570553183556,\n      0.016292786225676537,\n      -0.02839188650250435,\n      -0.07023544609546661,\n      0.11611443758010864,\n      0.01155926939100027,\n      0.16292333602905273,\n      0.03727682679891586,\n      0.06420145183801651,\n      -0.04601584002375603,\n      -0.06003085896372795,\n      0.07473918050527573,\n      0.07578911632299423,\n      0.06566820293664932,\n      -0.09937679767608643,\n      0.0364382341504097,\n      0.0009464290924370289,\n      -0.14645208418369293,\n      -0.013956795446574688,\n      -0.05057363957166672,\n      -0.07444583624601364,\n      0.0787595734000206,\n      -0.034776706248521805,\n      0.08092868328094482,\n      -0.04498995468020439,\n      -0.045915260910987854,\n      -0.04544640704989433,\n      -0.009203774854540825,\n      -0.0321984589099884,\n      0.024643847718834877,\n      -0.00971080083400011,\n      -0.08758064359426498,\n      -0.08643385022878647,\n      -0.08447907119989395,\n      -0.017742646858096123,\n      -0.04587278142571449,\n      -0.10384179651737213,\n      -0.06227492541074753,\n      0.02790921926498413,\n      -0.15551021695137024,\n      -0.0454302616417408,\n      -0.013834754005074501,\n      -0.018644418567419052,\n      -0.03178399056196213,\n      -0.032180119305849075,\n      -0.05112108588218689,\n      -0.02450377680361271,\n      -0.03929297626018524,\n      -0.10557490587234497,\n      0.06769607961177826,\n      0.07786376774311066,\n      0.02945149689912796,\n      0.034157611429691315,\n      0.058773238211870193,\n      0.03209344297647476,\n      0.1373879611492157,\n      -0.06210183724761009,\n      0.09191623330116272,\n      -0.008745729923248291,\n      0.024073755368590355,\n      0.01778162270784378,\n      0.11189065873622894,\n      0.04908543825149536,\n      0.006300922948867083,\n      0.0020347987301647663,\n      -0.017951801419258118,\n      -0.0306320209056139,\n      0.06030948460102081,\n      -0.023541012778878212,\n      -0.0011119606206193566,\n      0.0024261188227683306,\n      0.004264840856194496,\n      -0.004489935468882322,\n      0.046882085502147675,\n      -0.028853710740804672,\n      -0.07023558765649796,\n      -0.09229260683059692,\n      0.07317869365215302,\n      -0.0419800728559494,\n      0.042533084750175476,\n      0.030402885749936104,\n      -0.07965037226676941,\n      0.06341640651226044,\n      0.021486608311533928,\n      -0.04577811434864998,\n      0.07668585330247879,\n      -0.018435386940836906,\n      0.07248541712760925,\n      0.01358381099998951,\n      -0.025734836235642433,\n      -0.13053232431411743,\n      -0.010351800359785557,\n      0.038879845291376114,\n      -0.04798202961683273,\n      0.06677834689617157,\n      -0.11855225265026093,\n      0.11406658589839935,\n      -0.04577450826764107,\n      0.1286546140909195,\n      -0.07873483747243881,\n      0.11201909929513931,\n      -0.06814902275800705\n    ],\n    [\n      0.06673600524663925,\n      -0.018517840653657913,\n      -0.00939334835857153,\n      0.02284722402691841,\n      -0.02005782537162304,\n      0.06332626193761826,\n      0.040487244725227356,\n      0.1361832618713379,\n      -0.04354984685778618,\n      -0.0765228122472763,\n      -0.016895717009902,\n      0.04567735269665718,\n      0.07884690910577774,\n      -0.1612812876701355,\n      -0.15717017650604248,\n      0.09528848528862,\n      -0.011872073635458946,\n      0.10613411664962769,\n      0.10799798369407654,\n      0.07810042053461075,\n      0.09108489006757736,\n      -0.04340844228863716,\n      -0.005719092208892107,\n      0.05567184463143349,\n      -0.015085694380104542,\n      -0.04808145761489868,\n      0.12900744378566742,\n      -0.03919404372572899,\n      0.08249719440937042,\n      -0.2065621316432953,\n      0.042908526957035065,\n      -0.07537283003330231,\n      0.0053788130171597,\n      -0.036512624472379684,\n      0.027009638026356697,\n      0.0053768339566886425,\n      -0.03809227794408798,\n      -0.02175186388194561,\n      0.019862070679664612,\n      0.09469344466924667,\n      0.023675404489040375,\n      0.08457189053297043,\n      0.009323455393314362,\n      -0.05749260261654854,\n      0.11420349031686783,\n      0.05639180913567543,\n      -0.026215627789497375,\n      -0.06277492642402649,\n      0.012027548626065254,\n      -0.01106846984475851,\n      -0.00110269570723176,\n      -0.06904860585927963,\n      -0.012916062958538532,\n      0.08370967209339142,\n      0.025034381076693535,\n      0.014918262138962746,\n      -0.036559849977493286,\n      0.04709488898515701,\n      0.09131023287773132,\n      -0.08342218399047852,\n      0.0389692448079586,\n      -0.08868298679590225,\n      -0.007466927170753479,\n      0.018746716901659966,\n      0.022989556193351746,\n      0.046422477811574936,\n      -0.11346015334129333,\n      -0.007760578766465187,\n      -0.03604449704289436,\n      0.016201229766011238,\n      -0.007336549460887909,\n      0.03652827814221382,\n      -0.0469474159181118,\n      0.014564981684088707,\n      0.01825731433928013,\n      -0.08100157976150513,\n      0.004971485584974289,\n      0.0912926122546196,\n      -0.15097689628601074,\n      -0.03344864770770073,\n      0.005533108953386545,\n      0.03237813338637352,\n      -0.011311143636703491,\n      0.03166850656270981,\n      -0.032823383808135986,\n      0.0037884372286498547,\n      -0.10424245893955231,\n      -0.05703483149409294,\n      0.007814180105924606,\n      -0.04489346966147423,\n      0.04682822525501251,\n      -0.0694812461733818,\n      -0.022889362648129463,\n      -0.057567667216062546,\n      -0.0485156811773777,\n      -0.07179387658834457,\n      0.03954704478383064,\n      0.0383400060236454,\n      0.04016353562474251,\n      -0.045368146151304245,\n      -0.04496009647846222,\n      0.015814494341611862,\n      0.057004522532224655,\n      -0.06547625362873077,\n      0.01656571589410305,\n      -0.10182806104421616,\n      0.020110003650188446,\n      0.08991885930299759,\n      0.048712123185396194,\n      0.057038359344005585,\n      0.0036043478175997734,\n      -0.06488028913736343,\n      -0.029659155756235123,\n      0.04977099597454071,\n      0.07152114063501358,\n      0.04633275419473648,\n      0.04966587573289871,\n      0.07045264542102814,\n      -0.04688888415694237,\n      -0.1253833770751953,\n      -0.13601654767990112,\n      -0.05969904363155365,\n      -0.05550697073340416,\n      -0.08808884769678116,\n      -0.03084949590265751\n    ],\n    [\n      0.06645888835191727,\n      0.10603923350572586,\n      -0.10168534517288208,\n      -0.08686704188585281,\n      0.05625802278518677,\n      -0.03556991368532181,\n      -0.030160190537571907,\n      -0.007972786203026772,\n      0.05065795034170151,\n      -0.031311534345149994,\n      0.09655717015266418,\n      -0.024473343044519424,\n      -0.08898089826107025,\n      0.10391788929700851,\n      -0.004507122561335564,\n      -0.09646563977003098,\n      -0.005248408764600754,\n      -0.054254088550806046,\n      0.023826919496059418,\n      0.0919867753982544,\n      0.04922935739159584,\n      0.015073100104928017,\n      -0.12555758655071259,\n      -0.09590592980384827,\n      -0.08682192862033844,\n      -0.002278349595144391,\n      -0.08772382885217667,\n      0.08549974858760834,\n      0.03055749461054802,\n      0.11206543445587158,\n      -0.07847544550895691,\n      -0.014757798984646797,\n      0.05853335186839104,\n      0.017719946801662445,\n      0.053476203233003616,\n      -0.01602162979543209,\n      -0.007959660142660141,\n      -0.039848506450653076,\n      0.00992645975202322,\n      0.031325407326221466,\n      -0.005006111692637205,\n      -0.028775028884410858,\n      -0.003101199399679899,\n      -0.08690501004457474,\n      0.004926885012537241,\n      -0.08626242727041245,\n      0.06493319571018219,\n      -0.013691039755940437,\n      -0.04924857243895531,\n      -0.07712022215127945,\n      0.01155469287186861,\n      0.030866647139191628,\n      0.05600573122501373,\n      0.02731085941195488,\n      -0.05418006703257561,\n      -0.10385406762361526,\n      -0.15451982617378235,\n      -0.0724257230758667,\n      -0.07495341449975967,\n      0.05616412311792374,\n      -0.09172993898391724,\n      -0.0032961368560791016,\n      -0.10864994674921036,\n      0.04092254117131233,\n      -0.044676318764686584,\n      -0.05618423596024513,\n      0.02186966873705387,\n      0.10619165003299713,\n      0.11711926013231277,\n      0.10340540111064911,\n      0.0455918125808239,\n      0.041615016758441925,\n      -0.0289925467222929,\n      0.003629254177212715,\n      0.10332085937261581,\n      0.05481504648923874,\n      0.030445123091340065,\n      -0.08686601370573044,\n      -0.03279346600174904,\n      -0.02565087005496025,\n      0.0959680825471878,\n      -0.10975053906440735,\n      -0.04901372641324997,\n      0.03281501680612564,\n      -0.011807676404714584,\n      0.0926252156496048,\n      -0.012913217768073082,\n      -0.049658484756946564,\n      -0.005673312582075596,\n      -0.030716264620423317,\n      0.015163064934313297,\n      -0.037759847939014435,\n      0.03555777668952942,\n      0.07339267432689667,\n      0.027774203568696976,\n      0.09889663010835648,\n      -0.030024096369743347,\n      0.0053838700987398624,\n      -0.06481567025184631,\n      -0.03755206987261772,\n      -0.06124914065003395,\n      -0.11566198617219925,\n      0.09682903438806534,\n      0.05610700696706772,\n      -0.005275255534797907,\n      0.02894267812371254,\n      0.05269179120659828,\n      0.1023607924580574,\n      0.022996922954916954,\n      -0.004940795712172985,\n      0.09443919360637665,\n      0.06828926503658295,\n      0.0028408903162926435,\n      -0.02944100648164749,\n      -0.0839693620800972,\n      0.029787499457597733,\n      -0.08260764181613922,\n      -0.03863044083118439,\n      -0.06751595437526703,\n      0.11210888624191284,\n      0.12068795412778854,\n      0.14051572978496552,\n      0.018680429086089134,\n      0.07534367591142654,\n      0.004575223196297884\n    ],\n    [\n      0.09797416627407074,\n      0.04665815457701683,\n      0.020873868837952614,\n      -0.030124839395284653,\n      0.06230336055159569,\n      0.04424136504530907,\n      -0.01070459932088852,\n      0.027900537475943565,\n      -0.015387158840894699,\n      -0.005084076896309853,\n      0.09679553657770157,\n      -0.03500305116176605,\n      -0.01021965965628624,\n      -0.13166892528533936,\n      -0.16941048204898834,\n      0.04448683559894562,\n      0.04459453374147415,\n      -0.05413999408483505,\n      -0.005473931320011616,\n      0.10227392613887787,\n      -0.042419496923685074,\n      -0.04761723801493645,\n      -0.02405114844441414,\n      0.09791000187397003,\n      -0.0706903412938118,\n      -0.07637399435043335,\n      0.04008951038122177,\n      -0.06993904709815979,\n      0.11093614250421524,\n      -0.12096292525529861,\n      -0.05408196896314621,\n      -0.06408119201660156,\n      -0.07440567761659622,\n      0.008128801360726357,\n      0.020233478397130966,\n      0.04937175288796425,\n      -0.04554912820458412,\n      0.030435914173722267,\n      0.03247731178998947,\n      0.09515050053596497,\n      0.009084997698664665,\n      -0.021221768110990524,\n      0.04836403205990791,\n      -0.07498642802238464,\n      0.03048679791390896,\n      0.05464239418506622,\n      0.08753350377082825,\n      -0.033882975578308105,\n      0.07117455452680588,\n      0.06991156190633774,\n      0.06370172649621964,\n      -0.003844543593004346,\n      0.032350726425647736,\n      0.021304434165358543,\n      0.03757556900382042,\n      -0.09482454508543015,\n      0.07537473738193512,\n      -0.0029949850868433714,\n      -0.04771600291132927,\n      -0.04766051471233368,\n      0.023251637816429138,\n      0.0683709979057312,\n      -0.04042625427246094,\n      0.03632842376828194,\n      0.0814727395772934,\n      0.07124429941177368,\n      0.005671944003552198,\n      0.05929024890065193,\n      0.0614735446870327,\n      -0.12059243768453598,\n      0.0005573402158915997,\n      0.06891369074583054,\n      -0.01176016591489315,\n      -0.08572843670845032,\n      -0.0681203082203865,\n      0.043895334005355835,\n      -0.01819576695561409,\n      -0.055449534207582474,\n      -0.03662581741809845,\n      -0.02614629454910755,\n      -0.11484793573617935,\n      0.0552034005522728,\n      0.02897372841835022,\n      -0.07163923233747482,\n      -0.08871252834796906,\n      -0.05360617861151695,\n      0.0040495311841368675,\n      0.07130127400159836,\n      -0.013236699625849724,\n      -0.044545531272888184,\n      -0.049638181924819946,\n      -0.056510549038648605,\n      0.09021320194005966,\n      0.07092855125665665,\n      0.04261394962668419,\n      0.04292917624115944,\n      0.0330401211977005,\n      0.09638093411922455,\n      0.05370426923036575,\n      -0.04272862896323204,\n      -0.07076161354780197,\n      0.02015196532011032,\n      -0.034335315227508545,\n      0.054872382432222366,\n      0.055669158697128296,\n      0.07864870131015778,\n      0.048145148903131485,\n      0.012361702509224415,\n      -0.021716825664043427,\n      -0.0018008796032518148,\n      -0.0030988994985818863,\n      0.05187295004725456,\n      -0.022075433284044266,\n      0.07171863317489624,\n      0.07152806222438812,\n      -0.13566705584526062,\n      0.048976536840200424,\n      0.049913860857486725,\n      0.07367202639579773,\n      -0.0955968052148819,\n      -0.10386888682842255,\n      -0.14154766499996185,\n      0.1187741830945015,\n      -0.022495919838547707,\n      0.05757826194167137\n    ],\n    [\n      0.09265100955963135,\n      0.01952981948852539,\n      -0.03865186870098114,\n      -0.08155204355716705,\n      -0.07772018015384674,\n      -0.027210818603634834,\n      0.1276363581418991,\n      0.1401911824941635,\n      -0.13135689496994019,\n      -0.05871456116437912,\n      -0.0108296824619174,\n      -0.05367866903543472,\n      0.08796445280313492,\n      -0.1562899798154831,\n      -0.05063033476471901,\n      -0.07534020394086838,\n      -0.017257366329431534,\n      0.08161234110593796,\n      0.030895505100488663,\n      0.05445784702897072,\n      0.0054697380401194096,\n      -0.008358556777238846,\n      0.0012383363209664822,\n      0.002228648867458105,\n      -0.07919936627149582,\n      -0.06524895876646042,\n      0.11279035359621048,\n      -0.05128820985555649,\n      0.01680649258196354,\n      -0.17325355112552643,\n      0.022743068635463715,\n      -0.07419521361589432,\n      -0.035705652087926865,\n      0.025629382580518723,\n      -0.014438650570809841,\n      0.0677182599902153,\n      -0.006731668021529913,\n      0.10852662473917007,\n      -0.0012619562912732363,\n      -0.0147028136998415,\n      0.09799163043498993,\n      0.0783379003405571,\n      0.07701843976974487,\n      0.04476996511220932,\n      0.029750525951385498,\n      0.006038089282810688,\n      0.03835393488407135,\n      0.012612213380634785,\n      -0.019653698429465294,\n      -0.028865069150924683,\n      -0.09179476648569107,\n      0.05422715097665787,\n      -0.004636356141418219,\n      -0.010551241226494312,\n      -0.008310738950967789,\n      0.0638989806175232,\n      -0.04322103038430214,\n      -0.04939015209674835,\n      -0.03328308090567589,\n      -0.0022435630671679974,\n      0.05441676825284958,\n      -0.06180235370993614,\n      -0.03876042738556862,\n      -0.061187636107206345,\n      0.03624631464481354,\n      0.017680510878562927,\n      -0.08755716681480408,\n      -0.0351107195019722,\n      -0.03033769503235817,\n      -0.03298570215702057,\n      -0.03913765400648117,\n      -0.04439527913928032,\n      -0.13503773510456085,\n      -0.09073323011398315,\n      -0.06098688021302223,\n      -0.0488005094230175,\n      -0.08355340361595154,\n      0.04353782534599304,\n      -0.06002015247941017,\n      -0.03259041905403137,\n      -0.02686164714396,\n      -0.003831475740298629,\n      0.038779743015766144,\n      -0.07493855804204941,\n      0.06162648648023605,\n      -0.13511072099208832,\n      0.05935905873775482,\n      0.018631596118211746,\n      -0.03461382910609245,\n      -0.04065433517098427,\n      0.0026320461183786392,\n      -0.07974202930927277,\n      -0.05415289103984833,\n      0.04986434802412987,\n      -0.08422620594501495,\n      0.09035129100084305,\n      0.11456737667322159,\n      -0.04256761819124222,\n      0.02462090738117695,\n      -0.04225248843431473,\n      0.03659634664654732,\n      0.04044438898563385,\n      0.11093703657388687,\n      0.0962405875325203,\n      -0.031945522874593735,\n      -0.07709363102912903,\n      0.07478617876768112,\n      -0.013726888224482536,\n      0.10157406330108643,\n      0.12946705520153046,\n      -0.04097813740372658,\n      -0.09611398726701736,\n      0.06809523701667786,\n      0.042520154267549515,\n      0.07363806664943695,\n      0.024139365181326866,\n      -0.03361179679632187,\n      0.044987987726926804,\n      0.09449145197868347,\n      -0.12231042981147766,\n      0.004668430425226688,\n      0.01730683259665966,\n      0.0037226362619549036,\n      -0.06722273677587509,\n      0.06687699258327484\n    ],\n    [\n      0.06537169963121414,\n      -0.12114214897155762,\n      0.0349365659058094,\n      -0.011272815056145191,\n      -0.08027981221675873,\n      0.023079799488186836,\n      0.07101498544216156,\n      0.15612617135047913,\n      -0.07953125238418579,\n      -0.05027860775589943,\n      0.06700122356414795,\n      -0.060828011482954025,\n      0.0006727881846018136,\n      -0.031325049698352814,\n      -0.03807956352829933,\n      0.05886288732290268,\n      -0.06404153257608414,\n      0.06941787898540497,\n      -0.012800822034478188,\n      0.006947018671780825,\n      -0.05464034155011177,\n      -0.07097577303647995,\n      -0.0766477957367897,\n      0.06064336746931076,\n      -0.015862906351685524,\n      -0.05782777816057205,\n      -0.03321072459220886,\n      -0.008121284656226635,\n      -0.002735685557126999,\n      -0.19358223676681519,\n      -0.08284984529018402,\n      0.029744522646069527,\n      -0.06703919917345047,\n      0.03966251388192177,\n      -0.008388943038880825,\n      -0.04989173635840416,\n      0.034735601395368576,\n      -0.034032780677080154,\n      0.045675866305828094,\n      0.06535515934228897,\n      0.03716319054365158,\n      0.02593134343624115,\n      0.09250637888908386,\n      -0.014534975402057171,\n      0.10780266672372818,\n      -0.050756990909576416,\n      0.04564342275261879,\n      -0.009317818097770214,\n      0.0564543791115284,\n      -0.08093231171369553,\n      -0.061966657638549805,\n      -0.008890701457858086,\n      -0.02396339364349842,\n      0.02714444138109684,\n      -0.04475174844264984,\n      -0.058861903846263885,\n      -0.0707477554678917,\n      0.018659818917512894,\n      0.004848041106015444,\n      0.005489940755069256,\n      0.0013366458006203175,\n      -0.07402234524488449,\n      -0.07458167523145676,\n      -0.1361732929944992,\n      0.06139945238828659,\n      -0.033713970333337784,\n      -0.07384485006332397,\n      0.041031524538993835,\n      -0.1297343373298645,\n      0.009132545441389084,\n      -0.10756286978721619,\n      0.04085969552397728,\n      -0.1113220602273941,\n      -0.05491911247372627,\n      0.005665651056915522,\n      0.048625946044921875,\n      -0.0880357101559639,\n      0.04667260870337486,\n      -0.08976033329963684,\n      -0.05221367999911308,\n      -0.0065610771998763084,\n      -0.08259964734315872,\n      0.04014882072806358,\n      -0.04504865035414696,\n      -0.08098422735929489,\n      -0.08945024758577347,\n      -0.06932633370161057,\n      -0.1119430884718895,\n      0.021163756027817726,\n      -0.09824454039335251,\n      0.03037429042160511,\n      0.09660155326128006,\n      -0.048159003257751465,\n      -0.04318346455693245,\n      -0.07230166345834732,\n      0.04748581722378731,\n      -0.035029686987400055,\n      -0.010796491988003254,\n      -0.09572941064834595,\n      -0.05548940971493721,\n      0.08470720797777176,\n      -0.023968642577528954,\n      0.028077013790607452,\n      0.08572110533714294,\n      -0.00751498993486166,\n      0.036243390291929245,\n      -0.10430976748466492,\n      0.028977585956454277,\n      0.11133544892072678,\n      0.04206014424562454,\n      0.044387347996234894,\n      0.018443085253238678,\n      0.05508206784725189,\n      0.02555171772837639,\n      0.03822663798928261,\n      -0.1427762657403946,\n      -0.15802252292633057,\n      0.006845060735940933,\n      -0.03805530443787575,\n      -0.11575024574995041,\n      -0.0060384562239050865,\n      -0.08002108335494995,\n      0.06902924180030823,\n      -0.10828489810228348,\n      0.09770069271326065\n    ],\n    [\n      -0.07910303771495819,\n      -0.1353432983160019,\n      -0.07367278635501862,\n      -0.06422794610261917,\n      -0.02140536531805992,\n      0.09800954163074493,\n      0.12198194116353989,\n      -0.012009233236312866,\n      -0.03634272888302803,\n      0.011551043950021267,\n      -0.04309922829270363,\n      -0.011009085923433304,\n      0.03163256496191025,\n      -0.1676521599292755,\n      -0.16029395163059235,\n      -0.021999618038535118,\n      -0.10617686063051224,\n      0.06841044872999191,\n      0.022385312244296074,\n      0.04777928441762924,\n      0.015168621204793453,\n      -0.011994831264019012,\n      -0.022207284346222878,\n      0.05329454690217972,\n      -0.06414462625980377,\n      -0.0365411713719368,\n      -0.03500029072165489,\n      -0.10187514126300812,\n      0.025445779785513878,\n      -0.07213526219129562,\n      -0.01596488617360592,\n      0.02297532558441162,\n      -0.06956817954778671,\n      -0.04345269873738289,\n      0.06424585729837418,\n      -0.03514278680086136,\n      0.0780530497431755,\n      0.0494566336274147,\n      -0.013622509315609932,\n      0.03547117859125137,\n      0.03419783338904381,\n      0.05799021199345589,\n      -0.026159070432186127,\n      0.046562373638153076,\n      -0.08458054065704346,\n      0.04606979340314865,\n      -0.002120492747053504,\n      -0.06143733859062195,\n      0.07856468111276627,\n      0.03954020142555237,\n      0.015702715143561363,\n      0.06424415856599808,\n      -0.003220101585611701,\n      -0.09045058488845825,\n      -0.09576863050460815,\n      -0.02431376650929451,\n      -0.10348676890134811,\n      0.02807309478521347,\n      0.040964219719171524,\n      0.0334283784031868,\n      -0.013277985155582428,\n      -0.01526847667992115,\n      -0.07738202810287476,\n      0.0016933792503550649,\n      -0.07456623762845993,\n      0.06530392169952393,\n      0.031697385013103485,\n      -0.06217822805047035,\n      -0.04409658536314964,\n      0.0608210451900959,\n      0.008805747143924236,\n      -0.021487373858690262,\n      -0.06455722451210022,\n      -0.009770101867616177,\n      -0.08195624500513077,\n      0.05748726800084114,\n      0.07259071618318558,\n      -0.013257857412099838,\n      -0.13178083300590515,\n      0.06738556921482086,\n      -0.06539847701787949,\n      0.09660075604915619,\n      0.06426729261875153,\n      -0.09665527194738388,\n      -0.059059642255306244,\n      -0.12163423001766205,\n      0.0345776230096817,\n      -0.12065387517213821,\n      0.03016015700995922,\n      -0.030925029888749123,\n      -0.07618290930986404,\n      -0.07777033746242523,\n      0.040589094161987305,\n      -0.04664947837591171,\n      -0.024918043985962868,\n      -0.03270071744918823,\n      -0.04882073402404785,\n      0.05146582052111626,\n      -0.07847008109092712,\n      -0.05558178573846817,\n      -0.0428624153137207,\n      -0.0002881584223359823,\n      0.032026808708906174,\n      0.11826092004776001,\n      -0.09021017700433731,\n      0.03757673129439354,\n      -0.10153521597385406,\n      0.0983768105506897,\n      0.0958910584449768,\n      0.0266790259629488,\n      0.061673883348703384,\n      -0.02052299864590168,\n      0.04544622078537941,\n      0.09306969493627548,\n      -0.0096491239964962,\n      -0.03540054336190224,\n      -0.11688193678855896,\n      0.11369577795267105,\n      -0.12474287301301956,\n      -0.03257617726922035,\n      -0.14473484456539154,\n      0.014913152903318405,\n      0.016470761969685555,\n      -0.008064130321145058,\n      0.09159612655639648\n    ],\n    [\n      -0.07241777330636978,\n      0.025339879095554352,\n      0.010904262773692608,\n      -0.01542857475578785,\n      0.046540774405002594,\n      -0.06795913726091385,\n      -0.02589820884168148,\n      0.009380788542330265,\n      0.14340740442276,\n      0.024710291996598244,\n      0.02992640994489193,\n      -0.04686805605888367,\n      -0.14486336708068848,\n      0.020375153049826622,\n      0.02697397954761982,\n      0.02081354521214962,\n      -0.02958478406071663,\n      -0.11493898928165436,\n      -0.0005226880311965942,\n      -0.06812676787376404,\n      0.009837046265602112,\n      -0.0003159840125590563,\n      -0.08990836888551712,\n      -0.036749955266714096,\n      -0.07413915544748306,\n      0.024531373754143715,\n      -0.11878208816051483,\n      0.10191577672958374,\n      -0.02982112020254135,\n      0.06068752706050873,\n      0.05316098406910896,\n      -0.04275817051529884,\n      0.004921768791973591,\n      -0.010002527385950089,\n      -0.08683829754590988,\n      0.05761106684803963,\n      -0.03544790670275688,\n      -0.06008384749293327,\n      -0.0781809538602829,\n      0.014066094532608986,\n      -0.04288747161626816,\n      0.003403295995667577,\n      -0.07642217725515366,\n      0.02035536803305149,\n      -0.08667731285095215,\n      0.003509410424157977,\n      -0.00812682043761015,\n      0.07059456408023834,\n      0.034747056663036346,\n      0.04539380595088005,\n      -0.04628143087029457,\n      -0.02389310859143734,\n      -0.09614631533622742,\n      -0.002252861624583602,\n      -0.00010046962415799499,\n      -0.026046019047498703,\n      -0.09989301860332489,\n      0.0520758219063282,\n      -0.07670452445745468,\n      -0.026131564751267433,\n      0.04317145049571991,\n      0.059703949838876724,\n      -0.06491122394800186,\n      0.01357299741357565,\n      -0.10082592815160751,\n      -0.07817281037569046,\n      -0.008659281767904758,\n      -0.08324887603521347,\n      -0.06196426972746849,\n      0.02431161142885685,\n      -0.031276144087314606,\n      -0.15325108170509338,\n      0.05478392168879509,\n      -0.019052639603614807,\n      0.017670510336756706,\n      0.0075963581912219524,\n      -0.08410073816776276,\n      -0.06616028398275375,\n      0.07739679515361786,\n      -0.13139592111110687,\n      0.03678072616457939,\n      -0.05347694084048271,\n      -0.07765622437000275,\n      -0.03214282914996147,\n      0.017045127227902412,\n      0.12354282289743423,\n      -0.020572412759065628,\n      0.0787159875035286,\n      0.019058167934417725,\n      0.06770592927932739,\n      0.06430213898420334,\n      0.020653136074543,\n      -0.061357565224170685,\n      -0.0007187820156104863,\n      -0.018886134028434753,\n      -0.08206319808959961,\n      0.006509771104902029,\n      0.019944481551647186,\n      0.05146793648600578,\n      0.052178140729665756,\n      0.054632145911455154,\n      -0.10100840032100677,\n      0.10972467809915543,\n      0.1307259351015091,\n      0.01657853275537491,\n      0.09778767079114914,\n      0.006260277703404427,\n      -0.04177333042025566,\n      0.09752479195594788,\n      0.034810446202754974,\n      0.06349518150091171,\n      0.04664326086640358,\n      -0.03622116893529892,\n      -0.0909915417432785,\n      0.05814141407608986,\n      0.03777771070599556,\n      -0.056713469326496124,\n      -0.005692931357771158,\n      0.044609393924474716,\n      0.09354893118143082,\n      0.011188055388629436,\n      0.06234932690858841,\n      0.06305871903896332,\n      -0.02218308486044407,\n      0.02177581377327442\n    ],\n    [\n      0.03572790324687958,\n      0.11242030560970306,\n      -0.01790800876915455,\n      -0.02371474727988243,\n      0.02044879086315632,\n      -0.02368117868900299,\n      -0.040732964873313904,\n      -0.009917165152728558,\n      0.15326744318008423,\n      -0.09680664539337158,\n      0.0639028251171112,\n      -0.09001480042934418,\n      -0.06914864480495453,\n      0.16060565412044525,\n      0.18022704124450684,\n      -0.017144985496997833,\n      -0.0511787086725235,\n      -0.07414726167917252,\n      -0.04349561035633087,\n      -0.05330102518200874,\n      -0.07934261113405228,\n      -0.013278496451675892,\n      -0.1111149713397026,\n      -0.030640531331300735,\n      0.048125263303518295,\n      0.031536757946014404,\n      -0.06302353739738464,\n      0.0958038866519928,\n      -0.026859115809202194,\n      0.10095341503620148,\n      0.026904365047812462,\n      0.008788185194134712,\n      -0.0746675431728363,\n      0.012387515045702457,\n      0.04884198680520058,\n      -0.05658111721277237,\n      0.07820610702037811,\n      -0.051016915589571,\n      -0.03057244047522545,\n      0.018433567136526108,\n      -0.05983200669288635,\n      0.007247301749885082,\n      -0.008539199829101562,\n      0.05288679897785187,\n      -0.079591765999794,\n      -0.11445213109254837,\n      -0.07224617153406143,\n      -0.06998634338378906,\n      0.02865128591656685,\n      0.10107190161943436,\n      0.04505698010325432,\n      -0.011075833812355995,\n      -0.09047698229551315,\n      0.06083619222044945,\n      0.026990747079253197,\n      -0.02458777464926243,\n      -0.01944967731833458,\n      -0.01484655775129795,\n      -0.010377035476267338,\n      -0.09805034101009369,\n      -0.08068746328353882,\n      -0.03392862528562546,\n      -0.09964905679225922,\n      -0.04115570709109306,\n      -0.0011499931570142508,\n      0.06096820533275604,\n      0.07127396762371063,\n      0.07854626327753067,\n      0.03728461265563965,\n      -0.03589507192373276,\n      0.0038857338950037956,\n      -0.12379051744937897,\n      -0.03842635452747345,\n      0.014056333340704441,\n      0.004065662156790495,\n      -0.09007082134485245,\n      0.07552904635667801,\n      -0.013948360458016396,\n      0.11370769888162613,\n      0.05781196057796478,\n      0.12172508984804153,\n      -0.0320221409201622,\n      -0.02682003565132618,\n      -0.03269229456782341,\n      -0.03811601921916008,\n      0.1000320240855217,\n      0.000908376183360815,\n      -0.05451064929366112,\n      -0.05747479200363159,\n      0.049748681485652924,\n      0.04157722368836403,\n      0.033986933529376984,\n      0.019007857888936996,\n      0.04347599670290947,\n      0.0776006430387497,\n      0.02890954725444317,\n      0.0698603093624115,\n      0.035918258130550385,\n      -0.024615803733468056,\n      0.02300250343978405,\n      0.02029366046190262,\n      0.027627313509583473,\n      0.09905306249856949,\n      0.0665489211678505,\n      0.043369125574827194,\n      0.019311049953103065,\n      -0.06003155559301376,\n      0.014264126308262348,\n      0.045410383492708206,\n      0.0513654388487339,\n      -0.07533016055822372,\n      0.06227103993296623,\n      -0.012190018780529499,\n      -0.01618371531367302,\n      -0.08813893795013428,\n      0.04071859270334244,\n      -0.11886226385831833,\n      0.049247708171606064,\n      0.05855338275432587,\n      0.05792952701449394,\n      0.05118162930011749,\n      0.05799757316708565,\n      0.06131261959671974,\n      0.0677371695637703,\n      0.07324234396219254\n    ],\n    [\n      0.03948269411921501,\n      0.07397639751434326,\n      0.11997999995946884,\n      0.019465532153844833,\n      0.07475798577070236,\n      -0.09564919769763947,\n      0.00717688724398613,\n      -0.04291564226150513,\n      0.07971283048391342,\n      -0.07716543972492218,\n      -0.007337822578847408,\n      -0.04932122677564621,\n      -0.05869598686695099,\n      0.08342393487691879,\n      0.11177080869674683,\n      -0.12409380078315735,\n      0.07300181686878204,\n      0.0073999762535095215,\n      0.07309320569038391,\n      -0.08844781666994095,\n      -0.016598347574472427,\n      0.03339213877916336,\n      0.018638718873262405,\n      -0.1160985454916954,\n      -0.030418386682868004,\n      0.030096303671598434,\n      -0.05889115855097771,\n      0.06300891190767288,\n      -0.018982984125614166,\n      0.22076398134231567,\n      0.0044403825886547565,\n      0.06543641537427902,\n      0.07171663641929626,\n      -0.04597830772399902,\n      0.03329717367887497,\n      0.0485353097319603,\n      0.024365970864892006,\n      -0.09408469498157501,\n      -0.020529966801404953,\n      -0.006336966063827276,\n      0.030161235481500626,\n      -0.1225951686501503,\n      0.009563212282955647,\n      -0.06180490925908089,\n      -0.06561663001775742,\n      0.05807112902402878,\n      -0.028728876262903214,\n      0.04378771409392357,\n      0.04734072834253311,\n      -0.02387143485248089,\n      0.022737376391887665,\n      0.013255574740469456,\n      -0.06349708884954453,\n      -0.016219906508922577,\n      -0.04531780630350113,\n      0.10142944753170013,\n      -0.037318333983421326,\n      -0.014397842809557915,\n      -0.08514858037233353,\n      -0.019939906895160675,\n      0.038493361324071884,\n      -0.062350038439035416,\n      0.04569307342171669,\n      0.0028194908518344164,\n      0.035186901688575745,\n      -0.05129125341773033,\n      0.09181948006153107,\n      0.007548125460743904,\n      -0.06145523115992546,\n      -0.041994087398052216,\n      0.09330955892801285,\n      0.03912107273936272,\n      0.11328873783349991,\n      0.02705248072743416,\n      0.028418250381946564,\n      -0.06844338774681091,\n      -0.014909548684954643,\n      -0.05196414142847061,\n      -0.007675481028854847,\n      -0.010387315414845943,\n      0.04398927465081215,\n      -0.10262604802846909,\n      0.07963122427463531,\n      0.015529489144682884,\n      -0.016069704666733742,\n      0.09234704077243805,\n      0.10084062069654465,\n      0.06085439398884773,\n      0.08342325687408447,\n      -0.08267057687044144,\n      9.418826084583998e-05,\n      0.07473397254943848,\n      -0.017799995839595795,\n      0.06670922785997391,\n      0.05700758099555969,\n      -0.0848362073302269,\n      0.041148245334625244,\n      0.14753931760787964,\n      0.006769887637346983,\n      0.05608734115958214,\n      0.05145169049501419,\n      -0.04895088076591492,\n      0.032937001436948776,\n      0.06362982839345932,\n      0.03676488623023033,\n      -0.004378045909106731,\n      -0.019105780869722366,\n      0.03282463923096657,\n      0.07899794727563858,\n      0.052225250750780106,\n      0.0178462453186512,\n      -0.004251325037330389,\n      -0.07420004159212112,\n      0.049333784729242325,\n      0.04606487229466438,\n      -0.032492317259311676,\n      0.06792403012514114,\n      0.12928099930286407,\n      0.06242796406149864,\n      0.09372088313102722,\n      0.09908757358789444,\n      0.11255629360675812,\n      0.013611165806651115,\n      0.09741227328777313,\n      -0.00533669488504529\n    ],\n    [\n      0.06662937253713608,\n      0.03746498376131058,\n      -0.002968002576380968,\n      -0.050484973937273026,\n      0.03549104928970337,\n      0.05071432515978813,\n      0.08444365859031677,\n      0.10049021244049072,\n      -0.10829345881938934,\n      0.04525880888104439,\n      0.0005076571251265705,\n      -0.0857541486620903,\n      0.056175269186496735,\n      -0.11691921204328537,\n      -0.033453721553087234,\n      0.06521104276180267,\n      0.01830250211060047,\n      -0.04332195594906807,\n      -0.0028059447649866343,\n      0.1111677810549736,\n      0.010605796240270138,\n      0.09790101647377014,\n      -0.03018316999077797,\n      0.0780283659696579,\n      -0.08249752223491669,\n      -0.05033323913812637,\n      0.011315751820802689,\n      -0.04311599209904671,\n      0.06543386727571487,\n      -0.0910809338092804,\n      -0.01759217120707035,\n      -0.07330580800771713,\n      -0.01983431540429592,\n      -0.0588185116648674,\n      -0.03531217202544212,\n      0.06195330619812012,\n      -0.06664851307868958,\n      0.04006367176771164,\n      0.07086645811796188,\n      0.009374524466693401,\n      0.06113162636756897,\n      -0.04045697674155235,\n      0.09966815263032913,\n      -0.01130160503089428,\n      0.0573342889547348,\n      -0.09873490035533905,\n      -0.08288057893514633,\n      -0.10231922566890717,\n      -0.03269973769783974,\n      0.044461674988269806,\n      -0.022399451583623886,\n      0.002009723335504532,\n      -0.08514618873596191,\n      0.0599321685731411,\n      -0.029112758114933968,\n      -0.03276614472270012,\n      0.04842913895845413,\n      -0.029463469982147217,\n      0.08499914407730103,\n      -0.05130019411444664,\n      -0.029769545421004295,\n      -0.051936786621809006,\n      0.06508591026067734,\n      -0.018024154007434845,\n      -0.08934608846902847,\n      -0.10773085802793503,\n      0.04733185097575188,\n      -0.025480853393673897,\n      0.014936591498553753,\n      0.05279567465186119,\n      -0.06122824549674988,\n      -0.06090919300913811,\n      -0.0008650901727378368,\n      0.040067799389362335,\n      -0.05951301008462906,\n      0.003460588399320841,\n      0.01427425816655159,\n      -0.04869880899786949,\n      -0.0708186998963356,\n      -0.03858165815472603,\n      -0.15558764338493347,\n      0.0940273255109787,\n      0.058881305158138275,\n      0.04330579563975334,\n      0.0038484276738017797,\n      -0.01577186957001686,\n      -0.08524850755929947,\n      0.08915197849273682,\n      -0.0804746076464653,\n      -0.060465678572654724,\n      -0.013029642403125763,\n      0.08326126635074615,\n      -0.04396061599254608,\n      0.09339173138141632,\n      -0.05022595450282097,\n      -0.0793818011879921,\n      -0.0213924590498209,\n      -0.06295812875032425,\n      -0.05221502110362053,\n      0.03145083412528038,\n      0.021832913160324097,\n      0.05883949249982834,\n      -0.061361175030469894,\n      0.08044858276844025,\n      0.027113178744912148,\n      0.017854204401373863,\n      0.007023278158158064,\n      0.022637834772467613,\n      0.05517501384019852,\n      -0.03847387433052063,\n      0.058009568601846695,\n      0.021193623542785645,\n      0.052266478538513184,\n      0.02616841346025467,\n      0.018580591306090355,\n      -0.033843912184238434,\n      0.018033267930150032,\n      -0.01471535861492157,\n      -0.08614148199558258,\n      -0.135250061750412,\n      0.005772955249994993,\n      -0.05432065203785896,\n      0.011309077963232994,\n      -0.043781645596027374,\n      0.06412190943956375\n    ],\n    [\n      -0.051970697939395905,\n      0.013358263298869133,\n      0.022811397910118103,\n      -0.0031454917043447495,\n      -0.05099757760763168,\n      0.05435303971171379,\n      0.027872508391737938,\n      -0.11073563247919083,\n      0.16774128377437592,\n      0.06810065358877182,\n      0.06032654643058777,\n      -0.09684152156114578,\n      0.012834638357162476,\n      0.029832065105438232,\n      0.14854265749454498,\n      -0.10207170993089676,\n      -0.05894235894083977,\n      -0.09558916836977005,\n      0.01771559566259384,\n      -0.06174444407224655,\n      0.10200905799865723,\n      0.014389175921678543,\n      -0.00947989709675312,\n      0.04290853813290596,\n      -0.07511899620294571,\n      -0.10914681851863861,\n      -0.043480996042490005,\n      0.09080833941698074,\n      -0.04815206304192543,\n      0.17865051329135895,\n      -0.009465246461331844,\n      0.09953290969133377,\n      0.10852648317813873,\n      0.02102651074528694,\n      0.05074110999703407,\n      0.06197743862867355,\n      -0.02018899843096733,\n      0.028667638078331947,\n      -0.05024370551109314,\n      -0.0008923850255087018,\n      0.06886286288499832,\n      0.025045201182365417,\n      0.012670209631323814,\n      0.10048843920230865,\n      -0.17581771314144135,\n      0.06460127234458923,\n      0.021984731778502464,\n      -0.12621963024139404,\n      -0.061682458966970444,\n      0.07695398479700089,\n      0.0997835323214531,\n      -0.04993819817900658,\n      -0.08556418865919113,\n      0.07688846439123154,\n      0.026462579146027565,\n      -0.04931844770908356,\n      0.056707583367824554,\n      0.07640837877988815,\n      -0.10078702121973038,\n      -0.009019284509122372,\n      -0.11914542317390442,\n      -0.014371052384376526,\n      0.04268716275691986,\n      0.040272898972034454,\n      -0.057088736444711685,\n      -0.022780535742640495,\n      -0.10829830169677734,\n      0.03576432913541794,\n      0.014115112833678722,\n      -0.09122879058122635,\n      -0.03415763005614281,\n      -0.021553916856646538,\n      0.10602492839097977,\n      0.13269752264022827,\n      0.04628690332174301,\n      0.08747636526823044,\n      -0.020450107753276825,\n      0.05329197645187378,\n      0.09572552144527435,\n      -0.09428203105926514,\n      0.04614134505391121,\n      -0.07230813056230545,\n      -0.0041295127011835575,\n      0.04850682616233826,\n      0.00826086476445198,\n      0.05732075870037079,\n      0.09675538539886475,\n      0.03485557809472084,\n      0.046011973172426224,\n      -0.06574913114309311,\n      0.0542600117623806,\n      0.04618081822991371,\n      -0.14460135996341705,\n      -0.04477742686867714,\n      -0.11120022088289261,\n      -0.10724830627441406,\n      0.058000870048999786,\n      -0.09243720769882202,\n      0.03478969261050224,\n      0.07879688590765,\n      -0.029517773538827896,\n      -0.08899008482694626,\n      -0.07381114363670349,\n      -0.07472562789916992,\n      -0.024738766252994537,\n      0.021106241270899773,\n      -0.07036025077104568,\n      -0.012684622779488564,\n      -0.06526294350624084,\n      -0.07452454417943954,\n      0.048588551580905914,\n      0.09698806703090668,\n      -0.012020650319755077,\n      0.07215288281440735,\n      -0.03524652123451233,\n      -0.13238458335399628,\n      0.031679920852184296,\n      0.09671279788017273,\n      -0.016525428742170334,\n      0.03695989400148392,\n      0.056021012365818024,\n      -0.020638391375541687,\n      0.02521579898893833,\n      0.06366094946861267,\n      0.03397749736905098\n    ],\n    [\n      0.011191310361027718,\n      0.05477658286690712,\n      0.017675388604402542,\n      -0.060346364974975586,\n      -0.01740180142223835,\n      0.05038794130086899,\n      0.11827266961336136,\n      -0.03312282636761665,\n      -0.03744781017303467,\n      -0.07903002202510834,\n      0.010232562199234962,\n      0.0633554458618164,\n      0.12192758172750473,\n      -0.04335275664925575,\n      -0.17145778238773346,\n      0.007331227883696556,\n      0.028083398938179016,\n      -0.05624185875058174,\n      0.002187113044783473,\n      -0.06204093620181084,\n      -0.05205279216170311,\n      0.005397767759859562,\n      -0.023860210552811623,\n      0.10816923528909683,\n      -0.1494092494249344,\n      -0.0508657731115818,\n      0.06504514068365097,\n      0.034122273325920105,\n      0.09189669787883759,\n      -0.041141167283058167,\n      -0.02848915010690689,\n      0.10505086183547974,\n      0.09122001379728317,\n      0.05756104364991188,\n      -0.029835758730769157,\n      -0.06008331477642059,\n      -0.04797941818833351,\n      -0.08273475617170334,\n      0.04501097649335861,\n      0.01492979284375906,\n      -0.047370072454214096,\n      0.05840509757399559,\n      -0.050695911049842834,\n      -0.021262604743242264,\n      -0.07305405288934708,\n      -0.005765520501881838,\n      -0.05491257458925247,\n      0.05915636196732521,\n      0.06371618062257767,\n      0.020879318937659264,\n      -0.05330973491072655,\n      -0.07488681375980377,\n      0.11157867312431335,\n      0.0471351183950901,\n      -0.04520522058010101,\n      0.01818937249481678,\n      -8.167052874341607e-05,\n      0.04604263976216316,\n      0.028583023697137833,\n      0.05135636404156685,\n      0.03767276555299759,\n      -0.05446060374379158,\n      -0.005625668913125992,\n      -0.04831003025174141,\n      -0.03653473034501076,\n      -0.002794163068756461,\n      -0.018656035885214806,\n      0.018786825239658356,\n      0.00885843951255083,\n      -0.0915483757853508,\n      0.011073843576014042,\n      0.011824490502476692,\n      0.012556091882288456,\n      -0.15693654119968414,\n      0.06434507668018341,\n      0.021787846460938454,\n      0.03200376406311989,\n      -0.08007171005010605,\n      -0.16744257509708405,\n      0.022592119872570038,\n      -0.04583590477705002,\n      -0.04921780899167061,\n      0.03270736709237099,\n      0.08041737973690033,\n      -0.09697124361991882,\n      -0.08151058852672577,\n      -0.046173907816410065,\n      0.009826069697737694,\n      -0.0968458279967308,\n      0.09557242691516876,\n      -0.051665712147951126,\n      -0.04083944484591484,\n      -0.0642058327794075,\n      -0.037739742547273636,\n      0.058619190007448196,\n      -0.07402649521827698,\n      -0.040173280984163284,\n      -0.029505757614970207,\n      -0.033275049179792404,\n      0.04221785441040993,\n      -0.059000011533498764,\n      0.030256211757659912,\n      -0.03589645028114319,\n      0.009797423146665096,\n      0.11422666162252426,\n      0.0060502695851027966,\n      0.006618746556341648,\n      0.05343179777264595,\n      0.10351378470659256,\n      0.08777768909931183,\n      0.08560573309659958,\n      0.022999312728643417,\n      -0.09226028621196747,\n      0.05695004016160965,\n      0.021471446380019188,\n      -0.13753575086593628,\n      0.06336642801761627,\n      -0.03269774094223976,\n      0.07681304961442947,\n      -0.12473298609256744,\n      -0.055473100394010544,\n      0.01649116724729538,\n      0.03930680826306343,\n      0.027177125215530396,\n      -0.03466446325182915\n    ],\n    [\n      -0.037135303020477295,\n      0.049085937440395355,\n      0.06130785495042801,\n      -0.04533209279179573,\n      0.10094300657510757,\n      -0.05068209767341614,\n      -0.1159224659204483,\n      -0.12285653501749039,\n      0.1493484079837799,\n      0.05405711755156517,\n      0.08587785810232162,\n      -0.057790011167526245,\n      -0.07714506983757019,\n      -0.01777062751352787,\n      0.07468906790018082,\n      -0.0538356713950634,\n      -0.041931867599487305,\n      0.012009483762085438,\n      -0.036059364676475525,\n      -0.05258407071232796,\n      -0.010807457379996777,\n      -0.00542024802416563,\n      0.13252730667591095,\n      0.01982334442436695,\n      0.07800818979740143,\n      -0.0025394370313733816,\n      0.0009816631209105253,\n      0.1151118054986,\n      0.03437503054738045,\n      0.10744345188140869,\n      -0.01370819192379713,\n      -0.03814712166786194,\n      0.02010365016758442,\n      -0.07483713328838348,\n      -0.039372365921735764,\n      0.05839487910270691,\n      0.054612014442682266,\n      -0.09644563496112823,\n      0.042241647839546204,\n      -0.04871147498488426,\n      0.008320645429193974,\n      -0.04832644388079643,\n      -0.051648180931806564,\n      -0.05176457017660141,\n      0.05393291264772415,\n      0.10492789000272751,\n      0.038640204817056656,\n      0.06626256555318832,\n      -0.049476537853479385,\n      0.055604010820388794,\n      0.07288762927055359,\n      0.015222261659801006,\n      -0.019556231796741486,\n      0.10351534187793732,\n      0.061384283006191254,\n      -0.04455075040459633,\n      0.02462739869952202,\n      -0.01683076098561287,\n      -0.08303593099117279,\n      0.05688154324889183,\n      -0.024133557453751564,\n      -0.003972902894020081,\n      0.002472618129104376,\n      0.07189973443746567,\n      -0.02431630529463291,\n      -0.008757015690207481,\n      0.11121494323015213,\n      -0.04362686350941658,\n      -0.04713336005806923,\n      0.06255275756120682,\n      0.015240466222167015,\n      -0.0302427988499403,\n      0.02313268929719925,\n      0.0397426038980484,\n      0.068006232380867,\n      -0.012390286661684513,\n      -0.061346039175987244,\n      -0.04411143437027931,\n      0.14661231637001038,\n      -0.06389083713293076,\n      0.07428553700447083,\n      -0.011709485203027725,\n      -0.03492734208703041,\n      -0.04193967580795288,\n      0.10487165302038193,\n      0.04499159753322601,\n      0.09099680185317993,\n      -0.07182785123586655,\n      -0.01304328627884388,\n      0.007042891811579466,\n      0.03412611410021782,\n      -0.03579092398285866,\n      0.02055157907307148,\n      -0.023924510926008224,\n      0.06858827918767929,\n      -0.07935792207717896,\n      0.06432519853115082,\n      0.10025648027658463,\n      0.10871519893407822,\n      -0.005194011610001326,\n      -0.05777991935610771,\n      0.043695271015167236,\n      0.08680029213428497,\n      0.08726515620946884,\n      0.016027241945266724,\n      0.0681171640753746,\n      0.08434890955686569,\n      0.04235808551311493,\n      0.007144434843212366,\n      0.09229644387960434,\n      0.10309535264968872,\n      0.10994015634059906,\n      -0.07372751832008362,\n      -0.0018304139375686646,\n      -0.06472168862819672,\n      -0.058110348880290985,\n      -0.05512728914618492,\n      -0.005188324488699436,\n      0.12118831276893616,\n      0.07921742647886276,\n      0.051566604524850845,\n      0.039785683155059814,\n      0.03605740889906883,\n      0.07471739500761032,\n      0.057537224143743515\n    ],\n    [\n      0.04434860125184059,\n      0.03692254796624184,\n      0.018791362643241882,\n      0.010374314151704311,\n      -0.04571617767214775,\n      -0.026434145867824554,\n      0.12501832842826843,\n      0.12098617851734161,\n      -0.1320340931415558,\n      -0.0329594723880291,\n      -0.005263458471745253,\n      0.0017399442149326205,\n      0.07130273431539536,\n      -0.17010456323623657,\n      -0.1773666888475418,\n      0.0688152089715004,\n      -0.04618936404585838,\n      0.08081389218568802,\n      0.033728811889886856,\n      0.07108821719884872,\n      0.06570599973201752,\n      0.10820236802101135,\n      0.03442423418164253,\n      0.10046970844268799,\n      0.023784276098012924,\n      -0.06228674203157425,\n      0.05479935184121132,\n      -0.04053610563278198,\n      0.13952377438545227,\n      -0.12036079168319702,\n      -0.0857873484492302,\n      -0.07877328246831894,\n      -0.03196687251329422,\n      -0.02570323646068573,\n      -0.05034364014863968,\n      -0.02615051344037056,\n      -0.07476893812417984,\n      -0.013567549176514149,\n      -0.009868409484624863,\n      0.07091370224952698,\n      -0.000830085133202374,\n      0.03725169971585274,\n      0.03452740237116814,\n      -0.017625883221626282,\n      0.02252999134361744,\n      -0.02781224250793457,\n      0.07887309044599533,\n      -0.07846999913454056,\n      -0.03275984525680542,\n      0.03590747341513634,\n      -0.05616229400038719,\n      0.012165652588009834,\n      0.0759602040052414,\n      0.005512397736310959,\n      -0.02044917829334736,\n      0.016272608190774918,\n      -0.0563986636698246,\n      0.031150421127676964,\n      -0.039110217243433,\n      -0.06702224910259247,\n      0.11200772970914841,\n      -0.043952688574790955,\n      0.035065822303295135,\n      0.05990143492817879,\n      -0.028979286551475525,\n      0.0772152692079544,\n      -0.10923796892166138,\n      0.052442263811826706,\n      -0.029760830104351044,\n      -0.00032711209496483207,\n      0.08735649287700653,\n      0.09217119961977005,\n      -0.1033826470375061,\n      -0.018250230699777603,\n      -0.005857828073203564,\n      -0.08591491729021072,\n      -0.10017123073339462,\n      -0.03405379503965378,\n      -0.07998021692037582,\n      -0.000905260443687439,\n      -0.05244823917746544,\n      0.07904035598039627,\n      -0.0727524384856224,\n      0.07548476755619049,\n      0.017261194065213203,\n      -0.005117719527333975,\n      0.027916058897972107,\n      -0.0029378272593021393,\n      0.07195302844047546,\n      -0.01286356057971716,\n      0.03521222993731499,\n      0.024176744744181633,\n      -0.00902484729886055,\n      0.0803062841296196,\n      -0.075105682015419,\n      -0.030121078714728355,\n      -0.02583862468600273,\n      -0.028115101158618927,\n      -0.03603089228272438,\n      -0.021303564310073853,\n      0.0882374495267868,\n      0.09352170675992966,\n      -0.04476972669363022,\n      0.03911389783024788,\n      0.09073002636432648,\n      -0.005536661483347416,\n      -0.004909624345600605,\n      -0.01482024323195219,\n      -0.08957993984222412,\n      0.033491574227809906,\n      -0.048351362347602844,\n      -0.060220327228307724,\n      -0.0009113617707043886,\n      0.06591323763132095,\n      -0.03453389182686806,\n      -0.16972126066684723,\n      0.06029742211103439,\n      0.09616491198539734,\n      0.06341122090816498,\n      -0.031949736177921295,\n      -0.013301764614880085,\n      0.007500619161874056,\n      0.01767970807850361,\n      -0.015955833718180656,\n      0.0293879397213459\n    ],\n    [\n      -0.14594553411006927,\n      -0.07675346732139587,\n      -0.04689842090010643,\n      -0.052629254758358,\n      -0.0228643249720335,\n      -0.12228095531463623,\n      -0.03726469352841377,\n      -0.013589760288596153,\n      -0.0931481122970581,\n      -0.09035864472389221,\n      -0.08263623714447021,\n      -0.03246159106492996,\n      -0.054741501808166504,\n      0.039987724274396896,\n      0.1724870353937149,\n      -0.04947018623352051,\n      0.008617425337433815,\n      -0.06752423197031021,\n      -0.08299194276332855,\n      -0.017424093559384346,\n      0.019969729706645012,\n      -0.07977952063083649,\n      -0.0023441934026777744,\n      0.058704014867544174,\n      -0.03123648092150688,\n      -0.07197915017604828,\n      0.02338361367583275,\n      -0.07785383611917496,\n      0.021982496604323387,\n      0.10378336906433105,\n      0.011661620810627937,\n      0.06167707219719887,\n      -0.10900484025478363,\n      -0.0828932598233223,\n      -0.009197470732033253,\n      -0.1191691905260086,\n      -0.023797214031219482,\n      0.003422387642785907,\n      -0.011310899630188942,\n      -0.012603073380887508,\n      0.005623894277960062,\n      0.04723566025495529,\n      -0.03545838221907616,\n      -0.05375820770859718,\n      0.058671511709690094,\n      -0.01937592215836048,\n      0.0010882130591198802,\n      -0.037789441645145416,\n      0.0664948970079422,\n      0.0891946330666542,\n      0.06769832968711853,\n      -0.0003506132052280009,\n      -0.05940521880984306,\n      0.08041563630104065,\n      -0.07747086137533188,\n      0.10398624837398529,\n      0.10174062848091125,\n      0.06799746304750443,\n      0.017677180469036102,\n      -0.023129213601350784,\n      -0.010930826887488365,\n      0.07656697928905487,\n      0.08534363657236099,\n      0.08187524229288101,\n      -0.0643327534198761,\n      -0.004631212446838617,\n      0.0594986230134964,\n      -0.08928553760051727,\n      -0.10178433358669281,\n      -0.12112952023744583,\n      -0.05383570119738579,\n      0.0016923561925068498,\n      0.0650167390704155,\n      0.052388403564691544,\n      0.05387911573052406,\n      0.01420538779348135,\n      -0.09420300275087357,\n      -0.08688825368881226,\n      -0.024661216884851456,\n      0.009943241253495216,\n      -0.024169472977519035,\n      0.14385777711868286,\n      -0.05108145251870155,\n      -0.06632484495639801,\n      0.013308005407452583,\n      -0.014671378768980503,\n      -0.07512238621711731,\n      -0.10979171842336655,\n      -0.03506600484251976,\n      -0.05966009199619293,\n      -0.055866215378046036,\n      0.05076533928513527,\n      -0.007354130037128925,\n      -0.08914552628993988,\n      -0.05713434889912605,\n      -0.08783966302871704,\n      -0.009219609200954437,\n      0.031115081161260605,\n      -0.011371642351150513,\n      -0.028251998126506805,\n      -0.047323793172836304,\n      0.07186935842037201,\n      -0.022508544847369194,\n      -0.0001498183119110763,\n      0.047839634120464325,\n      -0.025734705850481987,\n      0.006162704899907112,\n      0.06262624263763428,\n      -0.0871986523270607,\n      0.057489775121212006,\n      0.03934309259057045,\n      0.08494541049003601,\n      -0.015747619792819023,\n      0.0648384541273117,\n      0.10010172426700592,\n      -0.145607590675354,\n      0.03439541906118393,\n      -0.025234533473849297,\n      -0.058615006506443024,\n      0.0042429156601428986,\n      0.000915737240575254,\n      -0.09394717216491699,\n      -0.0630679652094841,\n      0.0847921073436737,\n      0.05200532451272011\n    ],\n    [\n      -0.07531402260065079,\n      0.10971532016992569,\n      0.08065642416477203,\n      0.11565110087394714,\n      -0.06904224306344986,\n      -0.0019550740253180265,\n      0.005744106601923704,\n      -0.12761476635932922,\n      0.18281473219394684,\n      0.03523377701640129,\n      -0.011067100800573826,\n      0.04106205701828003,\n      -0.01592680998146534,\n      0.17682722210884094,\n      0.17048627138137817,\n      -0.013310139067471027,\n      -0.017031285911798477,\n      -0.11202198266983032,\n      -0.0329412966966629,\n      -0.027238067239522934,\n      -0.05639205500483513,\n      0.0011531332274898887,\n      -0.02859557792544365,\n      0.015386293642222881,\n      -0.18801701068878174,\n      -0.03101002424955368,\n      -0.08591694384813309,\n      0.04701092094182968,\n      -0.01876857317984104,\n      0.03889022395014763,\n      0.03104092925786972,\n      0.07903355360031128,\n      -0.01866445504128933,\n      -0.06777936965227127,\n      0.0954202339053154,\n      0.01606316678225994,\n      0.03181887045502663,\n      0.048613108694553375,\n      0.01440525520592928,\n      -0.0487038753926754,\n      -0.04288199171423912,\n      -0.10190961509943008,\n      0.00846516340970993,\n      -0.06463994085788727,\n      -0.11566873639822006,\n      -0.05837850645184517,\n      0.031104572117328644,\n      -0.07320601493120193,\n      -0.09960044175386429,\n      -0.053868651390075684,\n      -0.05537443980574608,\n      -0.0740421786904335,\n      -0.08871907740831375,\n      -0.11332923918962479,\n      -0.07070993632078171,\n      -0.07545573264360428,\n      0.019772637635469437,\n      0.045286618173122406,\n      -0.11639565229415894,\n      0.017968077212572098,\n      -0.04191916063427925,\n      -0.014319616369903088,\n      0.00042506540194153786,\n      -0.10493434965610504,\n      -0.1267511397600174,\n      -0.1223067194223404,\n      -0.017481794580817223,\n      -0.1289113163948059,\n      -0.00672284746542573,\n      0.020688088610768318,\n      -0.0309377983212471,\n      -0.03794146701693535,\n      0.09480254352092743,\n      -0.06035307049751282,\n      0.046820275485515594,\n      -0.050937894731760025,\n      0.06965909898281097,\n      -0.001794520765542984,\n      0.1009298637509346,\n      -0.05520755797624588,\n      0.05974511429667473,\n      0.0063457190990448,\n      0.027578605338931084,\n      -0.08743252605199814,\n      0.09492767602205276,\n      0.012883256189525127,\n      0.09559492766857147,\n      -0.0337788462638855,\n      -0.04267537221312523,\n      -0.0638439953327179,\n      0.002758834743872285,\n      -0.0034524984657764435,\n      -0.006464326288551092,\n      0.07968311756849289,\n      0.08145907521247864,\n      -0.06812270730733871,\n      0.022129174321889877,\n      0.022314446046948433,\n      -0.09905077517032623,\n      -0.06055288389325142,\n      0.059335965663194656,\n      0.06259186565876007,\n      0.06634806841611862,\n      0.004716462455689907,\n      -0.12594978511333466,\n      0.04747757688164711,\n      0.05417529493570328,\n      -0.02068263664841652,\n      -0.023479733616113663,\n      0.02628953382372856,\n      0.06350038200616837,\n      0.03412088379263878,\n      -0.006483381614089012,\n      -0.06206928566098213,\n      0.07229199260473251,\n      -0.17277736961841583,\n      -0.04806865006685257,\n      0.07589490711688995,\n      -0.11749113351106644,\n      0.07131566107273102,\n      -0.13524293899536133,\n      0.1127030998468399,\n      -0.08981078863143921,\n      0.11438057571649551,\n      0.06771911680698395\n    ],\n    [\n      -0.010423693805932999,\n      -0.08895226567983627,\n      -0.06749185919761658,\n      -0.026120474562048912,\n      -0.032073888927698135,\n      0.11529049277305603,\n      0.06502877920866013,\n      0.09220029413700104,\n      0.05541716516017914,\n      0.1309332251548767,\n      0.046389393508434296,\n      0.040796563029289246,\n      -0.10083691030740738,\n      -0.03190144896507263,\n      -0.023068338632583618,\n      0.10136343538761139,\n      -0.033761538565158844,\n      -0.015265030786395073,\n      -0.09848290681838989,\n      0.14183026552200317,\n      -0.03163577988743782,\n      -0.05343176797032356,\n      0.023239288479089737,\n      -0.0904608815908432,\n      -0.09085327386856079,\n      -0.06447672098875046,\n      0.09510403126478195,\n      -0.07192447781562805,\n      -0.04050649330019951,\n      -0.11347867548465729,\n      -0.10550945997238159,\n      -0.04486466571688652,\n      0.031408343464136124,\n      0.11213582009077072,\n      0.005098084453493357,\n      -0.00976201705634594,\n      -0.05302499607205391,\n      0.060133375227451324,\n      0.0406382717192173,\n      0.032408881932497025,\n      0.10811082273721695,\n      0.09745050966739655,\n      0.13051587343215942,\n      0.05502494052052498,\n      0.05393544211983681,\n      0.020887194201350212,\n      -0.0065489262342453,\n      -0.029072854667901993,\n      -0.045554306358098984,\n      -0.01093354169279337,\n      0.01779344491660595,\n      -0.11468103528022766,\n      -0.07245434075593948,\n      -0.025410916656255722,\n      -0.03041238710284233,\n      -0.027490952983498573,\n      -0.06581144034862518,\n      0.05581933632493019,\n      -0.015298103913664818,\n      0.04028110206127167,\n      0.10929238051176071,\n      0.09070321917533875,\n      0.013387816958129406,\n      -0.03218645974993706,\n      -0.06449652463197708,\n      -0.040414080023765564,\n      -0.024307088926434517,\n      0.02510802447795868,\n      0.0367688424885273,\n      -0.019060660153627396,\n      0.06832535564899445,\n      -0.11162503063678741,\n      0.018108781427145004,\n      -0.05834829807281494,\n      0.02400515414774418,\n      -0.010622423142194748,\n      0.06014939397573471,\n      -0.043226391077041626,\n      0.01944046840071678,\n      0.008972009643912315,\n      -0.0476120300590992,\n      0.07047650218009949,\n      0.08212992548942566,\n      -0.04927798733115196,\n      0.05086096376180649,\n      -0.0323760099709034,\n      -0.07020635157823563,\n      -0.10517147928476334,\n      -0.012244067154824734,\n      -0.007125048898160458,\n      -0.03298623487353325,\n      -0.029734643176198006,\n      0.011748573742806911,\n      -0.028239836916327477,\n      -0.001711330027319491,\n      -0.014852991327643394,\n      -0.07309029251337051,\n      -0.06706053763628006,\n      -0.048435889184474945,\n      0.031708043068647385,\n      -0.04071803390979767,\n      -0.035378824919462204,\n      0.03482664376497269,\n      -0.04874672368168831,\n      0.0503050871193409,\n      -0.14526060223579407,\n      -0.07945919781923294,\n      -0.01593431830406189,\n      0.0978398472070694,\n      0.12318585067987442,\n      0.00627901591360569,\n      -0.0912034884095192,\n      0.05011289566755295,\n      0.042033784091472626,\n      -0.005099005065858364,\n      -0.17789620161056519,\n      0.040504034608602524,\n      -0.06396966427564621,\n      -0.0807231217622757,\n      0.0413273349404335,\n      0.04340531677007675,\n      0.13084906339645386,\n      -0.062334705144166946,\n      -0.02258642017841339,\n      -0.013858197256922722\n    ],\n    [\n      -0.023633476346731186,\n      -0.022885210812091827,\n      -0.03529921546578407,\n      0.010829877108335495,\n      0.09159598499536514,\n      0.013588754460215569,\n      -0.11680269241333008,\n      -0.09859190136194229,\n      0.05497141554951668,\n      0.0684521421790123,\n      0.08675477653741837,\n      -0.039892565459012985,\n      -0.020008908584713936,\n      0.12985697388648987,\n      0.15406379103660583,\n      -0.0070863161236047745,\n      -0.0767047107219696,\n      -0.02010893076658249,\n      -0.05986923351883888,\n      -0.007717925123870373,\n      -0.03843826428055763,\n      0.015478451736271381,\n      0.03450467064976692,\n      -0.01091716717928648,\n      0.02814556658267975,\n      -0.04520292207598686,\n      0.0009259511716663837,\n      0.07473214715719223,\n      -0.03303033858537674,\n      0.21180200576782227,\n      0.13771973550319672,\n      -0.056722987443208694,\n      0.05823072791099548,\n      0.019995786249637604,\n      0.06274201720952988,\n      -0.019381200894713402,\n      -0.03019539825618267,\n      -0.07526015490293503,\n      -0.12161184102296829,\n      -0.030949503183364868,\n      -0.06351760774850845,\n      -0.11444290727376938,\n      -0.04126303642988205,\n      -0.05756671726703644,\n      -0.11568431556224823,\n      -0.025638941675424576,\n      -0.03387261927127838,\n      -0.0506155826151371,\n      0.09003132581710815,\n      0.004070276860147715,\n      0.10877842456102371,\n      0.04758533090353012,\n      0.013288957066833973,\n      0.058361150324344635,\n      0.04537978395819664,\n      0.021138695999979973,\n      0.04557216539978981,\n      -0.04963403195142746,\n      0.04457754269242287,\n      -0.05228676646947861,\n      -0.0026972575578838587,\n      -0.03182399645447731,\n      -0.07383306324481964,\n      0.0222763828933239,\n      0.06036771088838577,\n      0.026928171515464783,\n      0.10656574368476868,\n      0.06251046806573868,\n      0.08180931955575943,\n      -0.03706687316298485,\n      0.09118557721376419,\n      0.05226434767246246,\n      0.0972861722111702,\n      0.006500284653156996,\n      -0.030773047357797623,\n      -0.021157728508114815,\n      -0.0008391041192226112,\n      0.029702849686145782,\n      0.14319366216659546,\n      -0.04708234593272209,\n      0.09943149983882904,\n      -0.05231468006968498,\n      0.060596782714128494,\n      0.009176768362522125,\n      0.10018868744373322,\n      0.08072163164615631,\n      0.006187820341438055,\n      0.06948529183864594,\n      0.000622747465968132,\n      -0.07619133591651917,\n      0.021586285904049873,\n      -0.017050767317414284,\n      -0.004883734975010157,\n      0.07349954545497894,\n      -0.008052359335124493,\n      -0.06497468799352646,\n      0.08182039856910706,\n      0.12288771569728851,\n      -0.05737845227122307,\n      0.04079857096076012,\n      -0.029003191739320755,\n      0.06667296588420868,\n      0.016879552975296974,\n      0.11484302580356598,\n      0.046227335929870605,\n      -0.04798208922147751,\n      -0.04490722343325615,\n      -0.0920250192284584,\n      0.036761846393346786,\n      -0.022704388946294785,\n      0.07776805013418198,\n      0.02775026671588421,\n      -0.057753656059503555,\n      -0.10246303677558899,\n      0.052974652498960495,\n      0.07830510288476944,\n      0.0377764068543911,\n      0.06836600601673126,\n      0.09318549931049347,\n      0.01653783582150936,\n      0.13037022948265076,\n      0.12932951748371124,\n      -0.07016018033027649,\n      0.119663305580616,\n      0.04338260740041733\n    ],\n    [\n      -0.06159331277012825,\n      -0.018349289894104004,\n      0.09225183725357056,\n      -0.057645466178655624,\n      0.09691983461380005,\n      -0.0491606704890728,\n      -0.1266859471797943,\n      0.021331792697310448,\n      0.10022906959056854,\n      -0.09655895829200745,\n      -0.06052052974700928,\n      -0.08820770680904388,\n      -0.014874548651278019,\n      -0.09719692915678024,\n      0.15187175571918488,\n      -0.07996032387018204,\n      -0.0036314812023192644,\n      -0.11675740033388138,\n      -0.03650593012571335,\n      -0.0244480948895216,\n      -0.023857273161411285,\n      -0.07742344588041306,\n      -0.049227483570575714,\n      0.07572713494300842,\n      0.1006135642528534,\n      -0.0003172067226842046,\n      0.01030846405774355,\n      0.08625240623950958,\n      0.06966297328472137,\n      0.07117415964603424,\n      0.06701113283634186,\n      0.05687231197953224,\n      -0.08867369592189789,\n      -0.06443297117948532,\n      -0.13962751626968384,\n      0.009148325771093369,\n      0.03690436854958534,\n      -0.0064512877725064754,\n      0.006239671725779772,\n      -0.026617085561156273,\n      0.06407249718904495,\n      0.08192615211009979,\n      -0.015314622782170773,\n      -0.049962982535362244,\n      0.0032324716448783875,\n      0.06896483153104782,\n      -0.07642924785614014,\n      0.13180473446846008,\n      -0.06854281574487686,\n      0.09862063080072403,\n      0.04236648976802826,\n      -0.03765030950307846,\n      -0.004268907476216555,\n      0.03418976068496704,\n      0.03511680290102959,\n      0.1250322163105011,\n      -0.037724919617176056,\n      0.02031148038804531,\n      -0.014389834366738796,\n      0.09597204625606537,\n      0.04293306544423103,\n      0.11343914270401001,\n      -0.06079670786857605,\n      0.0040524983778595924,\n      -0.020916422829031944,\n      0.018488967791199684,\n      -0.05495217442512512,\n      -0.06958077847957611,\n      -0.012170800007879734,\n      -0.002000273671001196,\n      -0.07005997002124786,\n      -0.019007431343197823,\n      0.022592149674892426,\n      0.06936120986938477,\n      0.0857783779501915,\n      -0.060218244791030884,\n      -0.05088770389556885,\n      0.0939650684595108,\n      -0.020580651238560677,\n      -0.034376125782728195,\n      0.07689964771270752,\n      0.04290950298309326,\n      0.034528978168964386,\n      0.022115709260106087,\n      -0.04990636557340622,\n      0.050798140466213226,\n      0.12649165093898773,\n      0.0601053312420845,\n      0.007725914008915424,\n      0.04758404940366745,\n      -0.04988415539264679,\n      0.08279095590114594,\n      0.041034381836652756,\n      -0.031017471104860306,\n      -0.06310616433620453,\n      0.00228747446089983,\n      -0.01589742861688137,\n      -0.03508768603205681,\n      0.08795035630464554,\n      0.09875297546386719,\n      0.02241966873407364,\n      0.06652934849262238,\n      -0.05982719734311104,\n      0.049785200506448746,\n      -0.06999745219945908,\n      -0.021939223632216454,\n      0.025256242603063583,\n      0.03167058899998665,\n      0.049264900386333466,\n      0.06832879036664963,\n      -0.08528599888086319,\n      0.08309589326381683,\n      -0.024697227403521538,\n      -0.07077883183956146,\n      -0.01775497943162918,\n      -0.16745711863040924,\n      0.010163528844714165,\n      -0.006244956981390715,\n      0.016759462654590607,\n      -0.022991511970758438,\n      0.014259271323680878,\n      0.09938936680555344,\n      -0.003706991672515869,\n      0.12325111776590347,\n      0.054851606488227844\n    ],\n    [\n      -0.01132155116647482,\n      -0.07234860211610794,\n      -0.04461301490664482,\n      0.04090668261051178,\n      -0.01720084249973297,\n      -0.013671733438968658,\n      -0.09119770675897598,\n      0.09873747825622559,\n      -0.056366994976997375,\n      -0.005453062243759632,\n      -0.06546716392040253,\n      -0.06661175191402435,\n      -0.14438772201538086,\n      0.1296883225440979,\n      0.0438799224793911,\n      -0.002845535986125469,\n      0.08316624164581299,\n      0.07896117866039276,\n      -0.014917158521711826,\n      -0.01833519898355007,\n      0.04156952723860741,\n      -0.035105831921100616,\n      -0.021282054483890533,\n      0.043360721319913864,\n      0.012844715267419815,\n      0.06599313020706177,\n      -0.07312635332345963,\n      -0.07147011905908585,\n      0.062289971858263016,\n      0.02637907862663269,\n      -0.03997392952442169,\n      0.05017969757318497,\n      0.09225669503211975,\n      0.04393121972680092,\n      0.06655003130435944,\n      0.05531315132975578,\n      -0.01885613426566124,\n      -0.018438538536429405,\n      0.018572811037302017,\n      -0.0026701451279222965,\n      0.019014552235603333,\n      -0.0035851015709340572,\n      0.015625396743416786,\n      -0.09057930111885071,\n      0.13155634701251984,\n      0.09942203015089035,\n      0.004960320889949799,\n      -0.023308176547288895,\n      -0.021370796486735344,\n      0.009386617690324783,\n      -0.09682251513004303,\n      0.017230791971087456,\n      0.0617462582886219,\n      0.11198366433382034,\n      -0.038693081587553024,\n      -0.1175839751958847,\n      0.07654976099729538,\n      -0.0686238631606102,\n      -0.033680856227874756,\n      0.09874487668275833,\n      -0.08155439794063568,\n      0.0839131698012352,\n      -0.0067722611129283905,\n      -0.013465741649270058,\n      0.11116240173578262,\n      0.06432873755693436,\n      -0.045149169862270355,\n      0.10949496924877167,\n      0.038508735597133636,\n      0.058338310569524765,\n      -0.049513839185237885,\n      0.030930442735552788,\n      -0.06865589320659637,\n      0.1003182977437973,\n      0.09410311281681061,\n      0.0299795251339674,\n      0.10403097420930862,\n      -0.06060824915766716,\n      -0.0063692000694572926,\n      -0.019732918590307236,\n      0.07321006059646606,\n      -0.037886764854192734,\n      -0.04258255660533905,\n      0.0254354290664196,\n      -0.033197108656167984,\n      -0.025483740493655205,\n      0.028782997280359268,\n      0.050355419516563416,\n      0.043952908366918564,\n      0.053445085883140564,\n      -0.06010587140917778,\n      -0.014739555306732655,\n      -0.025560244917869568,\n      -0.01923791505396366,\n      0.04389059916138649,\n      0.08535821735858917,\n      -0.07300946861505508,\n      -0.09008371829986572,\n      -0.06901387870311737,\n      0.0764772817492485,\n      -0.027743441984057426,\n      -0.01432100310921669,\n      0.061208248138427734,\n      -0.03085433878004551,\n      -0.030484991148114204,\n      -0.023071881383657455,\n      0.07923354208469391,\n      0.03253456950187683,\n      0.013484809547662735,\n      -0.09352684020996094,\n      -0.019770177081227303,\n      0.044312067329883575,\n      -0.0097166383638978,\n      -0.038204435259103775,\n      -0.014593110419809818,\n      0.06395355612039566,\n      -0.03326844796538353,\n      0.05112465098500252,\n      0.052425727248191833,\n      -0.12507997453212738,\n      0.12486307322978973,\n      0.022814005613327026,\n      0.044378213584423065,\n      -0.08532587438821793,\n      -0.10569269210100174\n    ],\n    [\n      0.07875587046146393,\n      -0.060472242534160614,\n      0.04694562405347824,\n      0.060752324759960175,\n      -0.07611093670129776,\n      -0.019929394125938416,\n      -0.0024204247165471315,\n      0.12847551703453064,\n      -0.11415886133909225,\n      0.04126076027750969,\n      0.09830691665410995,\n      -0.06287514418363571,\n      0.07336319237947464,\n      -0.010169142857193947,\n      -0.157891646027565,\n      0.07069572061300278,\n      0.0470789410173893,\n      -0.017435021698474884,\n      0.040133990347385406,\n      0.11043862998485565,\n      0.05019931122660637,\n      0.07772774249315262,\n      0.02397724986076355,\n      0.06905904412269592,\n      -0.02614605613052845,\n      0.029188670217990875,\n      0.14737330377101898,\n      -0.011286827735602856,\n      0.023098468780517578,\n      -0.04877195134758949,\n      -0.04211083799600601,\n      -0.0560758002102375,\n      -0.01848510280251503,\n      0.08670833706855774,\n      -0.02362901344895363,\n      -0.05433959141373634,\n      0.09480413794517517,\n      0.06700184941291809,\n      -0.04185938835144043,\n      0.012298968620598316,\n      0.10888487845659256,\n      -0.03826029971241951,\n      0.018475504592061043,\n      -0.031666889786720276,\n      0.0819263681769371,\n      -0.05934792011976242,\n      -0.026902027428150177,\n      0.019844278693199158,\n      0.01925799250602722,\n      0.05005417391657829,\n      0.05178151652216911,\n      -0.03899702429771423,\n      -0.0017923328559845686,\n      0.0714387372136116,\n      0.019558575004339218,\n      0.012946732342243195,\n      -0.03883129358291626,\n      -0.00813969410955906,\n      0.09186498820781708,\n      -0.026524988934397697,\n      0.006441939622163773,\n      -0.026191171258687973,\n      0.06318224966526031,\n      0.07930532097816467,\n      0.014509622938930988,\n      -0.07668106257915497,\n      -0.026361428201198578,\n      -0.009170441888272762,\n      0.05727877840399742,\n      -0.07178648561239243,\n      -0.009273056872189045,\n      0.08456987887620926,\n      -0.035617850720882416,\n      0.0033444161526858807,\n      -0.0050113569013774395,\n      -0.0037094764411449432,\n      0.03650428354740143,\n      0.07345420122146606,\n      -0.06152883172035217,\n      -0.0182981975376606,\n      -0.10233547538518906,\n      0.05413072556257248,\n      0.059159811586141586,\n      -0.0062681022100150585,\n      0.06654482334852219,\n      -0.10126522183418274,\n      0.03917264938354492,\n      -0.08403607457876205,\n      -0.024437403306365013,\n      0.07134407013654709,\n      -0.03440110385417938,\n      -0.010144132189452648,\n      -0.0035490677691996098,\n      -0.08197204768657684,\n      0.027918094769120216,\n      -0.04676751047372818,\n      0.11738014966249466,\n      0.09043945372104645,\n      0.0652557909488678,\n      0.009701121598482132,\n      0.017398279160261154,\n      0.06998486816883087,\n      -0.0013797810534015298,\n      0.10142642259597778,\n      0.07075129449367523,\n      -0.048735227435827255,\n      -0.09692800045013428,\n      -0.018064534291625023,\n      0.08310212939977646,\n      0.010133534669876099,\n      0.07380533218383789,\n      -0.05212612822651863,\n      -0.023105314001441002,\n      0.10163817554712296,\n      -0.0008343416266143322,\n      -0.08112585544586182,\n      0.1311294585466385,\n      0.12870462238788605,\n      0.07338134944438934,\n      -0.11370594054460526,\n      -0.07933562248945236,\n      -0.05383099243044853,\n      -0.007640891708433628,\n      -0.028974922373890877,\n      0.051587846130132675\n    ],\n    [\n      0.09833633154630661,\n      -0.020694756880402565,\n      -0.0999651625752449,\n      0.05679456517100334,\n      -0.002543504349887371,\n      -0.011811231262981892,\n      -0.0030401237308979034,\n      0.14064405858516693,\n      -0.08957187831401825,\n      0.013746450655162334,\n      0.11082737892866135,\n      0.0062989164143800735,\n      0.008126827888190746,\n      -0.10344672948122025,\n      -0.05468978360295296,\n      0.04328421503305435,\n      0.03767893835902214,\n      0.09526684880256653,\n      0.06373923271894455,\n      -0.0010385442292317748,\n      0.07894031703472137,\n      0.05978124961256981,\n      -0.07818678766489029,\n      0.1303825080394745,\n      -0.053936686366796494,\n      0.007076880428940058,\n      0.10407904535531998,\n      -0.13839024305343628,\n      0.11019092053174973,\n      -0.10601586848497391,\n      -0.060390423983335495,\n      0.004050596617162228,\n      -0.07492750883102417,\n      0.009047244675457478,\n      -0.00952407531440258,\n      0.11039554327726364,\n      0.0647243857383728,\n      -0.027206044644117355,\n      0.01305632758885622,\n      0.030263442546129227,\n      0.11616034805774689,\n      0.025420231744647026,\n      0.06649643182754517,\n      0.05667775496840477,\n      -0.004428510088473558,\n      0.046491388231515884,\n      -0.06746277213096619,\n      0.04302771016955376,\n      0.04450160264968872,\n      -0.10528804361820221,\n      -0.010003876872360706,\n      0.06297329068183899,\n      0.01502017118036747,\n      -0.033770810812711716,\n      0.06814005970954895,\n      -0.08201771974563599,\n      0.052096568048000336,\n      0.09083735197782516,\n      0.032895948737859726,\n      -0.011590762995183468,\n      -0.05561050400137901,\n      -0.009777502156794071,\n      0.0840580016374588,\n      -0.010254286229610443,\n      0.032615289092063904,\n      -0.03788357228040695,\n      -0.03656747564673424,\n      0.06388546526432037,\n      0.014487496577203274,\n      0.05989902466535568,\n      -0.028726905584335327,\n      0.01592138037085533,\n      -0.0014878801302984357,\n      -0.1081344410777092,\n      0.0484502911567688,\n      -0.05178340524435043,\n      0.040591202676296234,\n      0.0013002394698560238,\n      -0.03280378878116608,\n      -0.028634728863835335,\n      -0.04218050837516785,\n      -0.0151961175724864,\n      0.05127740278840065,\n      0.03213087096810341,\n      0.04041235148906708,\n      0.022694973275065422,\n      0.05813005194067955,\n      -0.024550750851631165,\n      0.0755530297756195,\n      0.08027754724025726,\n      -0.06457486003637314,\n      0.007447652518749237,\n      0.030197791755199432,\n      0.044431496411561966,\n      -0.06961450725793839,\n      0.040445923805236816,\n      -0.03310175612568855,\n      -0.03656643256545067,\n      -0.09338624030351639,\n      -0.0768897607922554,\n      0.07157176733016968,\n      0.07970834523439407,\n      -0.07278653979301453,\n      -0.039952389895915985,\n      -0.028898615390062332,\n      -0.028594529256224632,\n      0.07699810713529587,\n      -0.016602350398898125,\n      0.026620440185070038,\n      0.034167803823947906,\n      0.05418930575251579,\n      -0.07241974771022797,\n      0.08211545646190643,\n      -0.07710198312997818,\n      -0.01517912931740284,\n      -0.13911877572536469,\n      0.0734846442937851,\n      0.04112424701452255,\n      0.058635104447603226,\n      -0.11003019660711288,\n      0.0267396979033947,\n      -0.05968112871050835,\n      0.027761263772845268,\n      -0.011501333676278591,\n      -0.05377313122153282\n    ],\n    [\n      -0.020916631445288658,\n      -0.06339948624372482,\n      -0.02326173521578312,\n      0.03168383613228798,\n      0.1206994280219078,\n      0.04339468106627464,\n      -0.006939878687262535,\n      -0.16248221695423126,\n      0.06140000373125076,\n      0.065436452627182,\n      0.052176013588905334,\n      0.04871811345219612,\n      -0.1686752289533615,\n      0.014897404238581657,\n      0.17152468860149384,\n      -0.03467424958944321,\n      -0.015065567567944527,\n      0.005060091149061918,\n      -0.07020805776119232,\n      -0.006776255555450916,\n      -0.026079891249537468,\n      0.0013691956410184503,\n      0.04587520658969879,\n      0.02096082642674446,\n      0.03224460780620575,\n      -0.0644562765955925,\n      -0.04071173816919327,\n      -0.037067342549562454,\n      -0.05382388085126877,\n      0.0745118111371994,\n      0.012914841994643211,\n      0.07674331963062286,\n      -0.053426820784807205,\n      0.04088413342833519,\n      0.04587096720933914,\n      0.0013529893476516008,\n      0.06852836161851883,\n      -0.1271432638168335,\n      -0.03808357194066048,\n      -0.009459328837692738,\n      -0.00670537818223238,\n      -0.1209295392036438,\n      -0.09978200495243073,\n      -0.0199802964925766,\n      -0.0033733523450791836,\n      -0.0718328058719635,\n      0.025691036134958267,\n      0.005881850142031908,\n      0.08853185176849365,\n      0.06880118697881699,\n      -0.018237829208374023,\n      0.08212869614362717,\n      0.03108941577374935,\n      -0.021473493427038193,\n      0.08609054237604141,\n      -0.06000886112451553,\n      -0.01760029047727585,\n      0.049480605870485306,\n      -0.09676921367645264,\n      -0.069358229637146,\n      -0.015846459195017815,\n      -0.007400066591799259,\n      -0.0402943380177021,\n      0.03813838213682175,\n      0.09098684787750244,\n      0.0016865157522261143,\n      0.05123692750930786,\n      0.05298878625035286,\n      -0.009216738864779472,\n      -0.04177127778530121,\n      0.02587893418967724,\n      -0.0023972652852535248,\n      0.027191167697310448,\n      0.1387064903974533,\n      0.04780399054288864,\n      -0.019197747111320496,\n      0.09189466387033463,\n      0.029792290180921555,\n      0.15538501739501953,\n      -0.07469567656517029,\n      0.02872454933822155,\n      -0.041063085198402405,\n      -0.037422191351652145,\n      -0.0264434851706028,\n      -0.01879817247390747,\n      0.038236308842897415,\n      -0.014061002060770988,\n      -0.06605454534292221,\n      0.0388202965259552,\n      -0.04090133681893349,\n      0.08021712303161621,\n      -0.06312239170074463,\n      0.059609998017549515,\n      0.06693762540817261,\n      0.08158687502145767,\n      -0.03318563848733902,\n      0.0788150504231453,\n      -0.03523055464029312,\n      0.005912304390221834,\n      0.007891277782619,\n      0.009983385913074017,\n      0.06783987581729889,\n      -0.01191752590239048,\n      -0.04383241385221481,\n      -0.09290651977062225,\n      0.044955797493457794,\n      0.09508758038282394,\n      -0.05974452570080757,\n      0.06208847463130951,\n      -0.07112859934568405,\n      0.06618458032608032,\n      0.04620307311415672,\n      0.061256058514118195,\n      -0.11648902297019958,\n      0.06016658991575241,\n      -0.0550716407597065,\n      -0.024350125342607498,\n      -0.026923034340143204,\n      0.09200651198625565,\n      0.12151499837636948,\n      0.1518694907426834,\n      -0.021119095385074615,\n      0.022009529173374176,\n      -0.049645934253931046,\n      -0.054889582097530365\n    ],\n    [\n      0.07674454152584076,\n      -0.05644164979457855,\n      -0.0895233079791069,\n      -0.07274036109447479,\n      -0.012150164693593979,\n      -0.039947643876075745,\n      0.11428685486316681,\n      0.02575916051864624,\n      0.0012554014101624489,\n      0.10459782928228378,\n      0.04980034381151199,\n      -0.04085038974881172,\n      0.1252967119216919,\n      -0.15276718139648438,\n      -0.11296174675226212,\n      0.00029787790845148265,\n      -0.05479401722550392,\n      0.10726463794708252,\n      0.0924326628446579,\n      0.07608222961425781,\n      0.05048027262091637,\n      0.06299605220556259,\n      0.04624621197581291,\n      -0.013954563066363335,\n      0.013566171750426292,\n      0.03988134488463402,\n      0.04574228450655937,\n      0.014298083260655403,\n      0.04567505046725273,\n      -0.1510341614484787,\n      -0.020958200097084045,\n      0.07006073743104935,\n      0.06437807530164719,\n      -0.04204772040247917,\n      0.08410679548978806,\n      0.10236486047506332,\n      0.00023793993750587106,\n      0.049309805035591125,\n      0.10999108850955963,\n      -0.01076802983880043,\n      0.07166804373264313,\n      0.08474414795637131,\n      0.020494548603892326,\n      0.01743990182876587,\n      -0.022852923721075058,\n      -0.07543013244867325,\n      0.0891861543059349,\n      0.007193780969828367,\n      -0.003173697041347623,\n      -0.001096954452805221,\n      -0.08380500972270966,\n      0.002741215517744422,\n      0.00887428317219019,\n      -0.02978334203362465,\n      0.041281748563051224,\n      -0.06423191726207733,\n      0.01837058737874031,\n      0.01917780004441738,\n      -0.01005302369594574,\n      0.018301505595445633,\n      0.028240572661161423,\n      -0.010591383092105389,\n      0.08831864595413208,\n      0.07233808934688568,\n      0.0104161212220788,\n      -0.07357548177242279,\n      -0.02341356687247753,\n      0.03844218701124191,\n      -0.05811292678117752,\n      0.06055533513426781,\n      -0.030443623661994934,\n      -0.07438280433416367,\n      -0.1468261182308197,\n      -0.10258055478334427,\n      -0.07192414999008179,\n      -0.0588761568069458,\n      0.004755688831210136,\n      0.08687058091163635,\n      -0.037880271673202515,\n      -0.037209201604127884,\n      0.03772410377860069,\n      -0.04951609671115875,\n      -0.07873940467834473,\n      -0.037389062345027924,\n      0.05610252916812897,\n      0.011244462803006172,\n      -0.05087892338633537,\n      -0.008600269444286823,\n      -0.062052007764577866,\n      0.04228729009628296,\n      -0.0714324489235878,\n      0.038732971996068954,\n      0.058884892612695694,\n      0.023974517360329628,\n      0.07992241531610489,\n      0.06211456283926964,\n      0.06235514208674431,\n      0.10996383428573608,\n      0.012690100818872452,\n      -0.06326980888843536,\n      0.08014708757400513,\n      -0.03317828103899956,\n      -0.0022604765836149454,\n      0.032449863851070404,\n      0.037752583622932434,\n      0.02340921014547348,\n      0.06381304562091827,\n      0.03129556402564049,\n      0.09074882417917252,\n      0.025644365698099136,\n      -0.06655000150203705,\n      -0.06645265221595764,\n      0.0674644187092781,\n      -0.02964492328464985,\n      0.10034752637147903,\n      -0.10161604732275009,\n      0.0387330986559391,\n      0.001163383130915463,\n      0.12440531700849533,\n      -0.06852702796459198,\n      0.0029210883658379316,\n      0.010974116623401642,\n      0.000715652946382761,\n      -0.03406764194369316,\n      0.08203421533107758\n    ],\n    [\n      -0.09433478116989136,\n      -0.09815090894699097,\n      -0.01674192026257515,\n      0.06289069354534149,\n      0.07350246608257294,\n      0.10389631986618042,\n      -0.04490945115685463,\n      -0.07375893741846085,\n      0.07999516278505325,\n      0.00751124694943428,\n      -0.03432212769985199,\n      0.02002425119280815,\n      0.03488575667142868,\n      0.07812439650297165,\n      0.15888172388076782,\n      0.03994588553905487,\n      0.1420600563287735,\n      -0.0506746768951416,\n      0.03321153670549393,\n      -0.06282991915941238,\n      0.10118915140628815,\n      -0.07300204038619995,\n      -0.022095736116170883,\n      -0.028452908620238304,\n      0.057425275444984436,\n      0.008347004652023315,\n      -0.0698515847325325,\n      -0.01676449179649353,\n      -0.007136212661862373,\n      -0.012006477452814579,\n      0.07583483308553696,\n      0.0911739245057106,\n      0.06930403411388397,\n      -0.12936843931674957,\n      0.004330669064074755,\n      0.008160673081874847,\n      0.018130969256162643,\n      -0.0018161662155762315,\n      -0.04997996240854263,\n      0.01010514423251152,\n      -0.04620124027132988,\n      -0.05468996614217758,\n      0.04036791995167732,\n      -0.04538461193442345,\n      0.0014751150738447905,\n      0.012586427852511406,\n      -0.0002261853078380227,\n      0.09721814841032028,\n      0.11684947460889816,\n      0.057392094284296036,\n      0.07747965306043625,\n      0.06354942172765732,\n      -0.049980200827121735,\n      0.10528160631656647,\n      0.03518461808562279,\n      0.06505780667066574,\n      0.03414497151970863,\n      0.01882411539554596,\n      0.004288888536393642,\n      0.1015915721654892,\n      0.04769769310951233,\n      -0.007935982197523117,\n      0.06661795824766159,\n      0.05699754133820534,\n      0.04527212306857109,\n      -0.03456290811300278,\n      -0.06312968581914902,\n      0.10130294412374496,\n      0.03701895847916603,\n      0.022316034883260727,\n      0.0007706302567385137,\n      0.024243615567684174,\n      0.057197920978069305,\n      0.14912913739681244,\n      0.10626190155744553,\n      -0.0722930058836937,\n      0.10888825356960297,\n      0.045437563210725784,\n      -0.061163417994976044,\n      0.043483488261699677,\n      -0.008271199651062489,\n      -0.01569489948451519,\n      0.01262977346777916,\n      -0.028555281460285187,\n      0.023700132966041565,\n      0.06436679512262344,\n      0.03596584126353264,\n      0.06605254113674164,\n      -0.05830561742186546,\n      -0.07013463228940964,\n      0.06804488599300385,\n      0.04460204392671585,\n      -0.020934341475367546,\n      0.019980650395154953,\n      -0.13792158663272858,\n      -0.005457587074488401,\n      -0.00968124158680439,\n      0.03463870659470558,\n      0.032939743250608444,\n      0.0771121010184288,\n      0.027318662032485008,\n      -0.06353694200515747,\n      -0.0452059768140316,\n      0.07717934250831604,\n      -0.028931904584169388,\n      0.03421650826931,\n      -0.005505719222128391,\n      -0.00822240486741066,\n      -0.08837491273880005,\n      -0.09071312844753265,\n      -0.050954997539520264,\n      -0.04794565960764885,\n      -0.05413465574383736,\n      -0.04304605349898338,\n      -0.06787901371717453,\n      -0.0765276849269867,\n      0.010576647706329823,\n      0.047927774488925934,\n      0.029150405898690224,\n      0.04011869430541992,\n      0.015667922794818878,\n      0.002702676458284259,\n      0.06434229761362076,\n      0.07609477639198303,\n      0.027539357542991638\n    ],\n    [\n      -0.015791382640600204,\n      0.07628249377012253,\n      -0.03913726285099983,\n      0.011247309856116772,\n      -0.024418246001005173,\n      -0.0585155263543129,\n      -0.11709576100111008,\n      -0.11354108899831772,\n      0.030040115118026733,\n      -0.08086036890745163,\n      -0.09810971468687057,\n      0.028339682146906853,\n      -0.12133427709341049,\n      0.17720377445220947,\n      0.05723828449845314,\n      -0.09515160322189331,\n      -0.04231477156281471,\n      -0.08550925552845001,\n      0.061816971749067307,\n      0.04397140070796013,\n      -0.05744611844420433,\n      -0.012690017931163311,\n      0.05429956316947937,\n      -0.017939167097210884,\n      -0.05072320997714996,\n      0.05797381326556206,\n      -0.04570424184203148,\n      0.013060685247182846,\n      0.016040846705436707,\n      0.1785951852798462,\n      -0.0070447372272610664,\n      -0.019941043108701706,\n      0.09131572395563126,\n      0.0222645103931427,\n      -0.022478261962532997,\n      -0.05742577463388443,\n      -0.056606393307447433,\n      -0.09291122853755951,\n      -0.11489031463861465,\n      -0.030255869030952454,\n      0.05362607538700104,\n      -0.08638846129179001,\n      -0.09234795719385147,\n      -0.05773932486772537,\n      -0.07028540223836899,\n      0.04371882230043411,\n      0.09284070879220963,\n      -0.043398112058639526,\n      0.01623225398361683,\n      -0.0008389297872781754,\n      0.05407857149839401,\n      -0.04201024770736694,\n      -0.008348722942173481,\n      0.0020827343687415123,\n      0.10391303896903992,\n      -0.013104163110256195,\n      -0.09069632738828659,\n      -0.025155197829008102,\n      0.01543494313955307,\n      -0.030630595982074738,\n      -0.0503956601023674,\n      0.003443589434027672,\n      -0.06643561273813248,\n      0.0930815041065216,\n      0.06559016555547714,\n      -0.09744209051132202,\n      -0.023499755188822746,\n      -0.06822868436574936,\n      0.07825559377670288,\n      0.013748250901699066,\n      0.004867703653872013,\n      -0.0028865993954241276,\n      0.03644365072250366,\n      0.05577138066291809,\n      0.03371443599462509,\n      0.028461627662181854,\n      -0.07586738467216492,\n      -0.023108312860131264,\n      0.14233753085136414,\n      -0.03963438421487808,\n      0.10427084565162659,\n      -0.01828862726688385,\n      -0.09292132407426834,\n      0.08236996829509735,\n      -0.05523029342293739,\n      0.06021210178732872,\n      0.08087771385908127,\n      0.059066805988550186,\n      -0.01778152585029602,\n      -0.0788305401802063,\n      -0.008275547064840794,\n      0.03448449447751045,\n      -0.061701349914073944,\n      0.0038772206753492355,\n      -0.04246796667575836,\n      -0.06672687083482742,\n      0.07860671728849411,\n      0.11729224771261215,\n      0.0596156120300293,\n      -0.0064509231597185135,\n      -0.09044177085161209,\n      0.07859836518764496,\n      0.020772049203515053,\n      0.06137742102146149,\n      0.0409478060901165,\n      0.07450490444898605,\n      0.0024868331383913755,\n      0.0045108250342309475,\n      0.05373534932732582,\n      -0.03612012416124344,\n      0.05270013585686684,\n      -0.020840279757976532,\n      -0.06349936127662659,\n      -0.05364740267395973,\n      -0.05560901015996933,\n      0.05921327322721481,\n      -0.05822811275720596,\n      0.023107895627617836,\n      0.005308696534484625,\n      0.11966395378112793,\n      0.036835554987192154,\n      0.07071315497159958,\n      0.0007097911438904703,\n      0.03410731256008148,\n      -0.08289863914251328\n    ],\n    [\n      -0.012023139744997025,\n      0.07558149099349976,\n      0.08129870146512985,\n      -0.06787526607513428,\n      -0.10662553459405899,\n      -0.08004771918058395,\n      0.1315850466489792,\n      0.06339750438928604,\n      -0.044937510043382645,\n      -0.023655468598008156,\n      -0.010386447422206402,\n      0.10288544744253159,\n      0.16660305857658386,\n      -0.06594403833150864,\n      -0.017049802467226982,\n      0.09916260093450546,\n      0.054803863167762756,\n      0.10365459322929382,\n      -0.06166921183466911,\n      0.08521497994661331,\n      -0.015717949718236923,\n      -0.03346796706318855,\n      0.04028391093015671,\n      0.11405378580093384,\n      -0.07618027925491333,\n      0.06854991614818573,\n      0.009750845842063427,\n      0.02248258888721466,\n      0.12049086391925812,\n      -0.11113366484642029,\n      0.10314889997243881,\n      0.02490631304681301,\n      0.057000234723091125,\n      0.08503884822130203,\n      -0.05484373867511749,\n      -0.11593478173017502,\n      0.0484611913561821,\n      -0.05811058729887009,\n      -0.0815063938498497,\n      0.03271898999810219,\n      -0.011961085721850395,\n      -0.007154708728194237,\n      0.08692189306020737,\n      0.008946720510721207,\n      0.04781654477119446,\n      0.03783480450510979,\n      -0.007416955195367336,\n      0.011952734552323818,\n      -0.003529119770973921,\n      -0.079676553606987,\n      -0.10313780605792999,\n      -0.04978049919009209,\n      -0.03530611842870712,\n      -0.02383250929415226,\n      -0.053088050335645676,\n      0.0317542590200901,\n      -0.15669351816177368,\n      -0.043813131749629974,\n      0.08904433250427246,\n      -0.008501912467181683,\n      0.07005090266466141,\n      -0.048516515642404556,\n      0.09632712602615356,\n      -0.06769495457410812,\n      -0.08477313071489334,\n      0.010531839914619923,\n      0.014656458981335163,\n      0.0282796248793602,\n      -0.017341764643788338,\n      -0.016259463503956795,\n      -0.07989427447319031,\n      -0.015624357387423515,\n      0.07052763551473618,\n      -0.15374265611171722,\n      0.0019650848116725683,\n      0.061757221817970276,\n      0.02499999850988388,\n      0.04383938014507294,\n      0.02176041156053543,\n      0.07181249558925629,\n      -0.1177157610654831,\n      -0.07565655559301376,\n      -0.011776544153690338,\n      -0.030757203698158264,\n      -0.048165563493967056,\n      -0.13183970749378204,\n      -0.05536063015460968,\n      -0.03964032232761383,\n      0.07427630573511124,\n      0.006005609408020973,\n      0.08699570596218109,\n      0.0890326201915741,\n      0.02534000761806965,\n      0.0843690037727356,\n      -0.046008363366127014,\n      -0.12214230746030807,\n      -0.06343095004558563,\n      0.030407622456550598,\n      -0.0851808711886406,\n      0.049700237810611725,\n      -0.05092284455895424,\n      -0.0945204421877861,\n      0.05661462992429733,\n      -0.06443148106336594,\n      -0.06855148077011108,\n      -0.023538775742053986,\n      0.06874876469373703,\n      -0.017250018194317818,\n      -0.06496809422969818,\n      -0.06697623431682587,\n      0.028970368206501007,\n      0.048787619918584824,\n      -0.014314874075353146,\n      -0.09312528371810913,\n      -0.07971116900444031,\n      -0.172895610332489,\n      -0.0412127710878849,\n      0.064110666513443,\n      0.003729958552867174,\n      -0.1515190750360489,\n      -0.1231093779206276,\n      0.06090943142771721,\n      0.04745778068900108,\n      -0.07933340221643448,\n      -0.02517157979309559\n    ],\n    [\n      -0.009859774261713028,\n      -0.0813298150897026,\n      -0.10750620812177658,\n      0.031112754717469215,\n      0.037413205951452255,\n      0.08255715668201447,\n      0.004067253787070513,\n      0.017466897144913673,\n      -0.07180909067392349,\n      -0.04793738201260567,\n      0.11047135293483734,\n      0.01692751981317997,\n      0.058933697640895844,\n      -0.010237058624625206,\n      -0.1408654749393463,\n      0.0617593489587307,\n      0.03881382569670677,\n      0.0992627665400505,\n      -0.06645463407039642,\n      -0.051458366215229034,\n      0.01975608617067337,\n      0.07775472104549408,\n      0.05055004358291626,\n      0.0827903151512146,\n      -0.007846194319427013,\n      -0.11388769000768661,\n      0.03464316949248314,\n      0.03815823793411255,\n      0.09709703922271729,\n      -0.053709834814071655,\n      0.03633704409003258,\n      -0.016662219539284706,\n      0.07300955802202225,\n      0.06908517330884933,\n      -0.03601856902241707,\n      0.07299918681383133,\n      0.05600212886929512,\n      0.09378806501626968,\n      0.051665741950273514,\n      0.043486643582582474,\n      -0.05349206179380417,\n      -0.009350883774459362,\n      0.11485201120376587,\n      0.06558866053819656,\n      0.09974662214517593,\n      0.0460418201982975,\n      -0.007622562348842621,\n      0.021017983555793762,\n      0.05782213807106018,\n      0.0450136698782444,\n      -0.07887768745422363,\n      -0.06824776530265808,\n      0.04984968155622482,\n      0.06183348223567009,\n      -0.02516544796526432,\n      -0.04124138876795769,\n      -0.0386541523039341,\n      -0.05016587674617767,\n      -0.029368743300437927,\n      0.0444691926240921,\n      -0.03218967840075493,\n      0.056540995836257935,\n      0.02261561155319214,\n      -0.007507732138037682,\n      -0.006870513316243887,\n      0.015131430700421333,\n      0.02330573834478855,\n      -0.014897845685482025,\n      -0.037538859993219376,\n      -0.08507755398750305,\n      0.022805185988545418,\n      0.06448496878147125,\n      -0.14171373844146729,\n      0.0037437223363667727,\n      -0.10754195600748062,\n      -0.008623622357845306,\n      -0.0914766788482666,\n      -0.008676830679178238,\n      -0.07379057258367538,\n      0.06834663450717926,\n      0.01600548066198826,\n      0.07189340889453888,\n      0.054736699908971786,\n      -0.027407627552747726,\n      -0.013795407488942146,\n      -0.053477756679058075,\n      0.038692645728588104,\n      -0.01564161106944084,\n      -0.03396416828036308,\n      -0.05669789761304855,\n      0.0100248446688056,\n      0.0699189230799675,\n      -0.0846654623746872,\n      -0.0752989649772644,\n      0.07364930957555771,\n      0.06523498892784119,\n      0.011673226952552795,\n      -0.057648852467536926,\n      0.005095985718071461,\n      -0.0018816872034221888,\n      -0.061104726046323776,\n      -0.04752456024289131,\n      -0.08136497437953949,\n      -0.003308533690869808,\n      -0.005595022812485695,\n      -0.07624610513448715,\n      0.06150781363248825,\n      0.10811375826597214,\n      0.08847364783287048,\n      0.054573092609643936,\n      -0.03178131952881813,\n      -0.09644393622875214,\n      0.02651841752231121,\n      0.07940170913934708,\n      0.0060479892417788506,\n      -0.08144332468509674,\n      0.05097029730677605,\n      0.0020022348035126925,\n      -0.04880453273653984,\n      -0.001032200176268816,\n      0.034273140132427216,\n      -0.023829564452171326,\n      0.007001862395554781,\n      0.029457202181220055,\n      -0.049770113080739975\n    ],\n    [\n      0.08925234526395798,\n      -0.039284106343984604,\n      -0.11801772564649582,\n      -0.059131380170583725,\n      0.04993044584989548,\n      0.10223743319511414,\n      0.03564254194498062,\n      0.04024491831660271,\n      -0.10021477192640305,\n      0.11085934191942215,\n      0.08285688608884811,\n      0.0667332112789154,\n      0.13231177628040314,\n      -0.10138192772865295,\n      -0.13682101666927338,\n      0.07323667407035828,\n      -0.005284950137138367,\n      0.10026530176401138,\n      -0.001503201201558113,\n      0.08031509816646576,\n      -0.01843338832259178,\n      0.06080453097820282,\n      0.06832198053598404,\n      0.05753997340798378,\n      -0.028227508068084717,\n      -0.016598084941506386,\n      0.01598135009407997,\n      -0.041192200034856796,\n      0.02203027904033661,\n      -0.17986296117305756,\n      0.024098290130496025,\n      -0.08149979263544083,\n      -0.005547379143536091,\n      0.011402049101889133,\n      0.05755056068301201,\n      -0.05482962727546692,\n      0.05136562138795853,\n      0.08934911340475082,\n      -0.0041679455898702145,\n      0.09920211136341095,\n      0.06749147921800613,\n      0.0142385084182024,\n      0.014045043848454952,\n      0.011051751673221588,\n      -0.021169325336813927,\n      0.0034691805485635996,\n      0.07449036836624146,\n      0.02656366303563118,\n      0.05939841642975807,\n      -0.035335611552000046,\n      -0.08433185517787933,\n      -0.024781469255685806,\n      0.06542007625102997,\n      0.08484811335802078,\n      0.050929903984069824,\n      0.007210045121610165,\n      0.054432716220617294,\n      0.010136947967112064,\n      -0.011752292513847351,\n      0.06686995923519135,\n      -0.038178637623786926,\n      0.005079696886241436,\n      -0.005047450307756662,\n      -0.024540603160858154,\n      0.07339566946029663,\n      -0.0023735035210847855,\n      0.03186360001564026,\n      0.01654493249952793,\n      -0.07367806881666183,\n      0.02640674076974392,\n      -0.004589897580444813,\n      0.01734117977321148,\n      -0.13770897686481476,\n      -0.06542249768972397,\n      0.03797358646988869,\n      0.00033747043926268816,\n      0.07219351828098297,\n      0.07575419545173645,\n      -0.07310735434293747,\n      0.043990954756736755,\n      0.001962448935955763,\n      0.06635238230228424,\n      0.08563099801540375,\n      -0.013011418282985687,\n      -0.03422030806541443,\n      0.03727266564965248,\n      0.005811589770019054,\n      0.05677231401205063,\n      -0.059348657727241516,\n      0.019439447671175003,\n      0.05328476056456566,\n      0.02504160813987255,\n      0.021653639152646065,\n      0.07546662539243698,\n      -0.014820408076047897,\n      -0.030748454853892326,\n      0.0035875188186764717,\n      0.08148296177387238,\n      0.012170720845460892,\n      -0.06473326683044434,\n      0.10271628946065903,\n      -0.0041630808264017105,\n      0.06795433908700943,\n      0.035042326897382736,\n      0.07955048978328705,\n      -0.0415799617767334,\n      -0.02447919175028801,\n      0.1181764230132103,\n      0.052528638392686844,\n      0.0385674349963665,\n      -0.0628165528178215,\n      0.02198130078613758,\n      -0.03762764483690262,\n      0.10147807002067566,\n      0.018559353426098824,\n      -0.10855557024478912,\n      0.07430673390626907,\n      0.016658231616020203,\n      0.061826325953006744,\n      0.011256968602538109,\n      -0.14357510209083557,\n      -0.019870787858963013,\n      -0.036645472049713135,\n      0.005233533214777708,\n      -0.06494437903165817\n    ],\n    [\n      0.023547865450382233,\n      -0.07560428231954575,\n      -0.04099985212087631,\n      -0.07327508181333542,\n      -0.02023734338581562,\n      0.08380765467882156,\n      -0.031954679638147354,\n      -0.002812642138451338,\n      -0.03827830031514168,\n      0.08049513399600983,\n      -0.07371611893177032,\n      0.09884017705917358,\n      -0.040890373289585114,\n      -0.11556311696767807,\n      -0.0883670225739479,\n      0.056670933961868286,\n      0.0192446056753397,\n      0.033107299357652664,\n      -0.10985202342271805,\n      0.06277190148830414,\n      0.0030952435918152332,\n      0.05674918368458748,\n      -0.04403013736009598,\n      0.022010818123817444,\n      0.0027407356537878513,\n      -0.023728402331471443,\n      0.02040858194231987,\n      -0.06137964129447937,\n      -0.016563814133405685,\n      -0.18761393427848816,\n      0.023283135145902634,\n      -0.03432964161038399,\n      0.04550567641854286,\n      0.08929575979709625,\n      -0.06796246767044067,\n      -0.003052104962989688,\n      -0.07101566344499588,\n      0.11548179388046265,\n      -0.0030459596309810877,\n      0.002435593167319894,\n      0.05450139939785004,\n      0.11226703226566315,\n      0.06338131427764893,\n      -0.007371079176664352,\n      0.03112909570336342,\n      -0.010058557614684105,\n      0.0126413032412529,\n      0.02015797421336174,\n      0.062478888779878616,\n      0.04446267709136009,\n      0.016335489228367805,\n      0.052513089030981064,\n      -0.08800038695335388,\n      -0.0309855155646801,\n      -0.08481543511152267,\n      -0.09377656131982803,\n      0.05182015523314476,\n      0.06229701638221741,\n      0.11014046519994736,\n      0.07200831174850464,\n      0.06294288486242294,\n      -0.03972097486257553,\n      0.08639603853225708,\n      0.009317203424870968,\n      0.025388289242982864,\n      -0.06442347913980484,\n      -0.0945904552936554,\n      -0.041858378797769547,\n      0.04150025174021721,\n      -0.011350095272064209,\n      0.07038751244544983,\n      -0.056072015315294266,\n      -0.014182545244693756,\n      0.01080712303519249,\n      -0.0480545237660408,\n      -0.0416027307510376,\n      0.050432607531547546,\n      -0.04407867044210434,\n      -0.12072223424911499,\n      0.05279979482293129,\n      0.060056619346141815,\n      0.11074510216712952,\n      -0.04365837201476097,\n      -0.06622489541769028,\n      0.002363756066188216,\n      -0.06966308504343033,\n      0.008984697982668877,\n      -0.09871125221252441,\n      0.013516112230718136,\n      -0.013825087808072567,\n      -0.052589163184165955,\n      0.05294451862573624,\n      -0.020172497257590294,\n      0.0075798132456839085,\n      -0.036015626043081284,\n      -0.008347363211214542,\n      -0.01934860460460186,\n      -0.0430104024708271,\n      -0.08295989781618118,\n      -0.09109176695346832,\n      0.003888912033289671,\n      0.08479639142751694,\n      0.10064411163330078,\n      0.017335070297122,\n      0.029000060632824898,\n      -0.10550173372030258,\n      -0.005723407492041588,\n      -0.01139017939567566,\n      -0.03601931035518646,\n      0.10308077931404114,\n      0.043145034462213516,\n      -0.09008872509002686,\n      0.10158538818359375,\n      -0.04037724807858467,\n      -0.08893633633852005,\n      -0.12098575383424759,\n      -0.0595511868596077,\n      0.10338636487722397,\n      0.034110672771930695,\n      -0.0019111442379653454,\n      -0.04649931564927101,\n      -0.014535780064761639,\n      -0.13295449316501617,\n      -0.037760574370622635,\n      0.07892992347478867\n    ],\n    [\n      -0.011179041117429733,\n      0.043405160307884216,\n      -0.055928293615579605,\n      0.04725820571184158,\n      0.06495697796344757,\n      -0.07582001388072968,\n      -0.04343108832836151,\n      -0.06720227003097534,\n      -0.05640558525919914,\n      -0.019608698785305023,\n      -0.04927898198366165,\n      -2.021552063524723e-06,\n      -0.0464978851377964,\n      0.1547229290008545,\n      -0.002278880914673209,\n      -0.07198167592287064,\n      0.02528732642531395,\n      0.005309061612933874,\n      -0.09007482975721359,\n      -0.027098296210169792,\n      -0.04439724236726761,\n      0.013762364163994789,\n      -0.016284598037600517,\n      -0.04505918174982071,\n      0.03625493124127388,\n      -0.04403654485940933,\n      0.10153823345899582,\n      0.05494048446416855,\n      -0.035611651837825775,\n      0.13465628027915955,\n      -0.03652643784880638,\n      0.02373647503554821,\n      0.021173017099499702,\n      0.11841154098510742,\n      0.04101189970970154,\n      -0.017918072640895844,\n      0.1077190712094307,\n      -0.08001184463500977,\n      -0.07992423325777054,\n      0.004247950855642557,\n      -0.0064442516304552555,\n      -0.09269219636917114,\n      -0.013911346904933453,\n      0.06371066719293594,\n      -0.04553263261914253,\n      0.04532018303871155,\n      0.08291639387607574,\n      -0.01046746876090765,\n      0.11379321664571762,\n      -0.01662943884730339,\n      0.08198697119951248,\n      0.03780373930931091,\n      0.038013655692338943,\n      0.11736647039651871,\n      0.022925984114408493,\n      -0.046251095831394196,\n      -0.06297482550144196,\n      -0.03226510435342789,\n      -0.02979128807783127,\n      -0.04657686501741409,\n      -0.09156268835067749,\n      -0.003305424004793167,\n      0.013477803207933903,\n      0.12990282475948334,\n      0.03192198649048805,\n      -0.04202974587678909,\n      -0.009150199592113495,\n      -0.031170416623353958,\n      0.03446922451257706,\n      0.08603458106517792,\n      -0.07066245377063751,\n      -0.12342791259288788,\n      0.043253201991319656,\n      0.03240066021680832,\n      0.09615524113178253,\n      0.0016570098232477903,\n      0.0032990865875035524,\n      -0.004189382307231426,\n      -0.0500071719288826,\n      -0.05790664255619049,\n      0.04209800437092781,\n      -0.013289240188896656,\n      -0.044726647436618805,\n      0.0868731141090393,\n      0.015424154698848724,\n      0.017072444781661034,\n      0.059932053089141846,\n      0.04532861337065697,\n      0.02005367912352085,\n      0.025570476427674294,\n      -0.07912766188383102,\n      -0.039573121815919876,\n      0.0675613060593605,\n      0.08048366010189056,\n      0.034981533885002136,\n      -0.03575117141008377,\n      0.026288310065865517,\n      -0.006540975067764521,\n      -0.07750681042671204,\n      -0.0811396986246109,\n      -0.04818510636687279,\n      -0.0625821053981781,\n      0.02294926717877388,\n      -0.014914052560925484,\n      0.023940565064549446,\n      0.006230369675904512,\n      0.035661108791828156,\n      -0.14484895765781403,\n      0.11864693462848663,\n      0.029185788705945015,\n      0.0782228633761406,\n      0.032804276794195175,\n      -0.07849560678005219,\n      -0.08438730239868164,\n      0.05323130637407303,\n      0.04725256562232971,\n      0.02486496977508068,\n      -0.025528229773044586,\n      0.0039352113381028175,\n      -0.10509887337684631,\n      0.15309709310531616,\n      0.08237234503030777,\n      0.030547132715582848,\n      0.09254773706197739,\n      -0.13305148482322693\n    ],\n    [\n      -0.04763738811016083,\n      -0.006934185978025198,\n      0.03632790222764015,\n      -0.06633373349905014,\n      0.0464344285428524,\n      0.03933423385024071,\n      -0.10376033186912537,\n      -0.06141351908445358,\n      0.11285975575447083,\n      -0.03498286008834839,\n      0.05851487070322037,\n      -0.10995888710021973,\n      -0.06537330895662308,\n      0.09357476979494095,\n      0.12815099954605103,\n      0.01793290115892887,\n      -0.010086415335536003,\n      0.004581596702337265,\n      0.05441783741116524,\n      -0.05332829803228378,\n      -0.010455500334501266,\n      0.049606431275606155,\n      -0.006525980308651924,\n      0.04290226101875305,\n      -0.011175278574228287,\n      -0.0444871261715889,\n      0.013081185519695282,\n      0.04989597946405411,\n      0.06806696951389313,\n      0.10307344794273376,\n      -0.03510666266083717,\n      0.03807989880442619,\n      0.030844073742628098,\n      -0.0819634273648262,\n      -0.07694904506206512,\n      0.01111474446952343,\n      -0.0438479520380497,\n      -0.11924925446510315,\n      -0.00912136398255825,\n      -0.04707195982336998,\n      0.02156141586601734,\n      -0.010728592053055763,\n      -0.0923701822757721,\n      0.04627733677625656,\n      0.030048387125134468,\n      -0.05872131511569023,\n      0.08814060688018799,\n      0.0021166803780943155,\n      -0.027817830443382263,\n      -0.056754037737846375,\n      0.09394173324108124,\n      0.011778884567320347,\n      0.05287771299481392,\n      0.05456368997693062,\n      -0.013956873677670956,\n      0.11871478706598282,\n      0.02159583382308483,\n      -0.02981494925916195,\n      0.05904892459511757,\n      0.04253800958395004,\n      -0.01074301078915596,\n      -0.03644489869475365,\n      -0.05092360079288483,\n      0.04532817006111145,\n      -0.025627782568335533,\n      0.022230984643101692,\n      0.008421959355473518,\n      0.0618412010371685,\n      -0.043565116822719574,\n      -0.010882190428674221,\n      0.010346720926463604,\n      0.06491706520318985,\n      0.06653785705566406,\n      -0.01805606484413147,\n      0.048294004052877426,\n      0.09002707898616791,\n      -0.009969990700483322,\n      -0.005543042905628681,\n      0.0981140211224556,\n      -0.014668897725641727,\n      -0.016144348308444023,\n      0.015904609113931656,\n      0.03303281217813492,\n      -0.017001386731863022,\n      0.046729981899261475,\n      0.13683921098709106,\n      -0.04733670502901077,\n      0.05294152721762657,\n      -0.061868175864219666,\n      0.0051172589883208275,\n      0.014449412934482098,\n      -0.05204504355788231,\n      0.06697265803813934,\n      -0.05694449692964554,\n      0.07342823594808578,\n      0.03204475715756416,\n      -0.010881703346967697,\n      0.06283866614103317,\n      0.04723995923995972,\n      0.09271951019763947,\n      0.06240714341402054,\n      0.037033386528491974,\n      0.040641460567712784,\n      0.025614753365516663,\n      0.05620693042874336,\n      -0.026579922065138817,\n      -0.02285033091902733,\n      -0.07569114863872528,\n      0.08279209583997726,\n      0.08569131791591644,\n      -0.03141886740922928,\n      0.021075144410133362,\n      -0.04645642265677452,\n      -0.1146380603313446,\n      0.004858003929257393,\n      -0.07283255457878113,\n      -0.036077599972486496,\n      -0.024269310757517815,\n      0.10441936552524567,\n      0.05363382771611214,\n      0.016630005091428757,\n      0.10926127433776855,\n      -0.04636530205607414,\n      0.08311684429645538,\n      0.004893837962299585\n    ],\n    [\n      -0.04793526232242584,\n      0.06299958378076553,\n      0.11508987843990326,\n      -0.08383429795503616,\n      0.11623585224151611,\n      -0.03399437665939331,\n      -0.07240819185972214,\n      -0.0926598459482193,\n      0.08300399035215378,\n      0.01122023444622755,\n      -0.024570882320404053,\n      0.012373452074825764,\n      -0.017151150852441788,\n      -0.029307009652256966,\n      0.13406188786029816,\n      -0.025938766077160835,\n      0.0598614327609539,\n      -0.025402311235666275,\n      -0.005940371658653021,\n      -0.08442432433366776,\n      0.008207762613892555,\n      0.0040646884590387344,\n      0.05051995441317558,\n      -0.009440995752811432,\n      0.07194201648235321,\n      0.006116636097431183,\n      0.09139110893011093,\n      0.02483329549431801,\n      0.019772684201598167,\n      0.05411979556083679,\n      0.07364878058433533,\n      -0.08078189194202423,\n      0.03916117921471596,\n      -0.009446440264582634,\n      -0.07741165906190872,\n      -0.06318401545286179,\n      0.035639792680740356,\n      -0.06123451516032219,\n      6.902980385348201e-05,\n      -0.06894249469041824,\n      0.004901484120637178,\n      -0.08604838699102402,\n      -0.054509665817022324,\n      -0.004727235529571772,\n      0.043925926089286804,\n      0.09577914327383041,\n      0.009775827638804913,\n      -0.006281133741140366,\n      0.007189047522842884,\n      0.0644596666097641,\n      -0.03639432415366173,\n      0.04780316352844238,\n      0.05052931606769562,\n      0.020069029182195663,\n      0.10386452078819275,\n      0.10993476212024689,\n      -0.08560997247695923,\n      -0.06413908302783966,\n      0.019268862903118134,\n      -0.036377936601638794,\n      0.014562392607331276,\n      0.040575120598077774,\n      -0.0808802917599678,\n      0.029445750638842583,\n      0.0684271976351738,\n      -0.018497131764888763,\n      0.09581019729375839,\n      0.051678240299224854,\n      0.019961580634117126,\n      0.07327906042337418,\n      0.026143446564674377,\n      -0.04042067378759384,\n      0.02375870570540428,\n      0.07860469073057175,\n      0.11751090735197067,\n      0.0264006108045578,\n      0.022309768944978714,\n      -0.07389997690916061,\n      0.07195311784744263,\n      0.0313245914876461,\n      0.053935375064611435,\n      0.06393145024776459,\n      0.10215288400650024,\n      0.0401727557182312,\n      -0.016427434980869293,\n      0.01721237786114216,\n      -0.020417021587491035,\n      -0.043095801025629044,\n      0.03697938472032547,\n      0.042792197316884995,\n      0.07193878293037415,\n      -0.07677017897367477,\n      -0.03914752975106239,\n      -0.009362127631902695,\n      0.07674326747655869,\n      0.06440931558609009,\n      0.1242092102766037,\n      0.07428371906280518,\n      0.1070496067404747,\n      0.01170689519494772,\n      -0.005429546814411879,\n      0.07931928336620331,\n      -0.03382851928472519,\n      0.049112468957901,\n      -0.019370682537555695,\n      0.08839529007673264,\n      0.01177164539694786,\n      0.032875385135412216,\n      0.10039137303829193,\n      0.1070275753736496,\n      0.008364646695554256,\n      0.018710579723119736,\n      0.07705117017030716,\n      -0.018353257328271866,\n      -0.026345374062657356,\n      -0.11258513480424881,\n      0.10398735105991364,\n      0.0634712502360344,\n      -0.035463057458400726,\n      0.05518368259072304,\n      0.1427101045846939,\n      -0.0020644795149564743,\n      0.0875735655426979,\n      0.006359520833939314,\n      0.040393680334091187\n    ],\n    [\n      -0.02313879318535328,\n      -0.07446780800819397,\n      0.05543309077620506,\n      -0.05383452773094177,\n      0.016592474654316902,\n      0.02695763111114502,\n      -0.03391365334391594,\n      -0.021078698337078094,\n      0.0092287827283144,\n      -0.07206094264984131,\n      -0.002167574129998684,\n      0.06898307055234909,\n      -0.08562131971120834,\n      0.1311994343996048,\n      0.11456496268510818,\n      0.05080879107117653,\n      0.00028693326748907566,\n      -0.03881790488958359,\n      0.0021235530730336905,\n      -0.036609042435884476,\n      0.01458751130849123,\n      0.07692243903875351,\n      -0.022742893546819687,\n      -0.012814097106456757,\n      -0.04356001690030098,\n      0.07979937642812729,\n      0.0008341483771800995,\n      0.0745307207107544,\n      0.06678803265094757,\n      0.07321325689554214,\n      0.0647512897849083,\n      -0.020754942670464516,\n      0.07081849128007889,\n      0.03190671652555466,\n      -0.14833267033100128,\n      -0.05098149925470352,\n      0.03389444202184677,\n      -0.11248322576284409,\n      -0.07667934894561768,\n      -0.05756649374961853,\n      -0.085941843688488,\n      0.07884309440851212,\n      -0.015054815448820591,\n      0.031029298901557922,\n      -0.076962411403656,\n      0.0032646844629198313,\n      0.07957848906517029,\n      0.10144051164388657,\n      -0.05383653566241264,\n      0.02813796140253544,\n      0.04523163288831711,\n      0.1042051687836647,\n      -0.04777400568127632,\n      0.08979407697916031,\n      -0.04010652005672455,\n      0.08112777769565582,\n      0.06029049679636955,\n      -0.06085766851902008,\n      -0.07745743542909622,\n      -0.04025288298726082,\n      0.10009083151817322,\n      -0.013091114349663258,\n      0.01244929526001215,\n      -0.06204438954591751,\n      -0.056787069886922836,\n      -0.01498170755803585,\n      0.07130621373653412,\n      0.017697153612971306,\n      -0.024302326142787933,\n      0.05614376440644264,\n      0.09496371448040009,\n      -0.0762363001704216,\n      0.0680057480931282,\n      0.08850660920143127,\n      0.0005094930529594421,\n      -0.02727854810655117,\n      0.09364427626132965,\n      0.05420359969139099,\n      0.08166585862636566,\n      0.06248713284730911,\n      0.017896484583616257,\n      0.00808218214660883,\n      0.0680403783917427,\n      -0.05989832431077957,\n      0.003593502100557089,\n      0.10751792788505554,\n      0.10416080802679062,\n      -0.014340346679091454,\n      -0.04667744040489197,\n      -0.0827680230140686,\n      -0.03655669465661049,\n      -0.018074313178658485,\n      -0.12666276097297668,\n      0.07497527450323105,\n      0.07661205530166626,\n      0.0531432069838047,\n      -0.015841683372855186,\n      -0.030002953484654427,\n      0.08730385452508926,\n      0.061037905514240265,\n      0.08486829698085785,\n      0.04841098561882973,\n      0.06534619629383087,\n      0.055428843945264816,\n      -0.007796387188136578,\n      -0.006515772081911564,\n      -0.013266308233141899,\n      -0.00013053149450570345,\n      0.06717748194932938,\n      -0.07385711371898651,\n      -0.04828775301575661,\n      -0.003999681677669287,\n      -0.036561399698257446,\n      -0.07421364635229111,\n      0.08487528562545776,\n      -0.1514032483100891,\n      0.036626074463129044,\n      -0.03653443977236748,\n      -0.07040165364742279,\n      0.11954279243946075,\n      0.07027629017829895,\n      0.09862147271633148,\n      -0.057149939239025116,\n      0.07166647166013718,\n      -0.059034693986177444\n    ],\n    [\n      -0.03850354999303818,\n      0.02303958125412464,\n      0.07773992419242859,\n      0.0528852641582489,\n      0.0321524403989315,\n      -0.09329739958047867,\n      -0.11507778614759445,\n      -0.14523334801197052,\n      0.17533184587955475,\n      -0.049659375101327896,\n      -0.026102891191840172,\n      -0.007516246754676104,\n      -0.12095224112272263,\n      0.17774733901023865,\n      0.1147027388215065,\n      -0.09564752131700516,\n      0.07104319334030151,\n      -0.07510337233543396,\n      -0.03445129841566086,\n      -0.0482192225754261,\n      0.05602652579545975,\n      0.008280489593744278,\n      0.08642604202032089,\n      -0.09911925345659256,\n      -0.01840754970908165,\n      -0.020165473222732544,\n      -0.10087776184082031,\n      -0.0029221652075648308,\n      -0.026500383391976357,\n      0.20828093588352203,\n      0.042680226266384125,\n      -0.04107561334967613,\n      0.08132901042699814,\n      0.03164077550172806,\n      -0.008919057436287403,\n      -0.060981009155511856,\n      -0.021939272060990334,\n      -0.028498411178588867,\n      -0.013265249319374561,\n      0.03653969615697861,\n      -0.017393022775650024,\n      0.050894662737846375,\n      -0.045341167598962784,\n      0.08755142986774445,\n      0.051203131675720215,\n      0.016898492351174355,\n      0.03941978141665459,\n      0.10418447852134705,\n      -0.007243948522955179,\n      -0.03779952973127365,\n      -0.02679779753088951,\n      0.010655016638338566,\n      -0.007713619153946638,\n      0.07945122569799423,\n      0.07859037816524506,\n      -0.04846520349383354,\n      -0.08493934571743011,\n      -0.0031139589846134186,\n      -0.07436789572238922,\n      -0.023934274911880493,\n      0.006596608553081751,\n      0.03409622982144356,\n      0.029571011662483215,\n      0.03466901183128357,\n      0.0850071832537651,\n      0.042165808379650116,\n      0.004510813858360052,\n      -0.006987804546952248,\n      0.08341115713119507,\n      -0.005008120089769363,\n      0.023605691269040108,\n      -0.029416222125291824,\n      0.008151324465870857,\n      0.017213517799973488,\n      0.021944886073470116,\n      -0.0072725312784314156,\n      0.0041479673236608505,\n      -0.10774283856153488,\n      0.1415514051914215,\n      -0.06750781834125519,\n      0.00798716675490141,\n      -0.045158978551626205,\n      -0.09389374405145645,\n      0.04619988426566124,\n      0.06646987050771713,\n      0.12075965851545334,\n      0.040286656469106674,\n      0.02869373932480812,\n      -0.049984581768512726,\n      -0.06393483281135559,\n      0.04450143873691559,\n      -0.029803475365042686,\n      0.06645934283733368,\n      -0.041480861604213715,\n      -0.06907045096158981,\n      -0.026907630264759064,\n      0.0878126323223114,\n      -0.002692083129659295,\n      0.029843950644135475,\n      0.07444605231285095,\n      -0.01754780299961567,\n      0.019124384969472885,\n      0.07835368812084198,\n      0.08121329545974731,\n      0.0002485396689735353,\n      0.033849846571683884,\n      0.03943207114934921,\n      0.05345817282795906,\n      -0.012570547871291637,\n      -0.02970103733241558,\n      -0.04862060397863388,\n      -0.032216500490903854,\n      0.010425470769405365,\n      -0.026657002046704292,\n      0.06654202193021774,\n      -0.006261554080992937,\n      0.08311422914266586,\n      0.038786645978689194,\n      0.08129142224788666,\n      0.04928343743085861,\n      0.08917835354804993,\n      -0.013923588208854198,\n      0.0035151741467416286,\n      0.11937980353832245,\n      -0.06479301303625107\n    ],\n    [\n      -0.02550831437110901,\n      0.00851403921842575,\n      0.028174418956041336,\n      -0.038429658859968185,\n      -0.014883816242218018,\n      0.098988838493824,\n      -0.0035228508058935404,\n      0.0769147127866745,\n      -0.0018499491270631552,\n      0.04905063658952713,\n      -0.03362783044576645,\n      -0.006385950371623039,\n      0.01265906635671854,\n      -0.020292652770876884,\n      0.0051845065318048,\n      0.08698859065771103,\n      0.08247994631528854,\n      0.03403688222169876,\n      -0.05910759046673775,\n      -0.08847995847463608,\n      -0.05198686942458153,\n      -0.09696441888809204,\n      -0.09863921254873276,\n      0.007012435235083103,\n      0.0015040794387459755,\n      -0.003182448912411928,\n      -0.11573746800422668,\n      -0.11354485154151917,\n      0.06037863716483116,\n      -0.02607540227472782,\n      0.06373455375432968,\n      0.008515676483511925,\n      -0.03562339022755623,\n      0.1507105827331543,\n      -0.009171160869300365,\n      -0.017149683088064194,\n      0.027151113376021385,\n      0.03196785971522331,\n      -0.06716837733983994,\n      -0.035878416150808334,\n      0.01067489292472601,\n      0.06625119596719742,\n      0.04097176715731621,\n      -0.050365447998046875,\n      -0.08357492089271545,\n      -0.0029169260524213314,\n      -0.020485740154981613,\n      -0.0355997160077095,\n      -0.07658220827579498,\n      0.06609397381544113,\n      -0.10696357488632202,\n      -0.0715591236948967,\n      -0.1204143539071083,\n      -0.08070259541273117,\n      -0.09268750995397568,\n      0.04470162093639374,\n      -0.06495466083288193,\n      -0.14117343723773956,\n      -0.058478184044361115,\n      -0.05230773612856865,\n      0.07718631625175476,\n      -0.04550657048821449,\n      -0.008048909716308117,\n      0.012425931170582771,\n      0.034650884568691254,\n      -0.001440999680198729,\n      -0.05408042296767235,\n      -0.030074331909418106,\n      0.0826391950249672,\n      -0.06574054807424545,\n      -0.020789241418242455,\n      -0.01505943201482296,\n      0.0246138833463192,\n      0.009079571813344955,\n      -0.011372472159564495,\n      -0.09150809049606323,\n      -0.11051902174949646,\n      -0.01542199682444334,\n      -0.04474123194813728,\n      -0.08859237283468246,\n      -0.038031354546546936,\n      -0.10906211286783218,\n      -0.05911867320537567,\n      -0.12419234961271286,\n      0.03941365331411362,\n      -0.04128376021981239,\n      0.07750644534826279,\n      -0.014399847947061062,\n      -0.03236105293035507,\n      0.10294566303491592,\n      0.0845458060503006,\n      -0.04404809698462486,\n      0.0063047814182937145,\n      -0.02357233501970768,\n      0.05477827042341232,\n      0.07096879929304123,\n      -0.07487490028142929,\n      -0.06950244307518005,\n      0.03423080965876579,\n      -0.016858404502272606,\n      -0.08767304569482803,\n      -0.05457039922475815,\n      0.06221868097782135,\n      -0.09254754334688187,\n      0.006753545720130205,\n      -0.08251214772462845,\n      0.005471812095493078,\n      0.03852171078324318,\n      -0.07833690941333771,\n      -0.05925960838794708,\n      0.06551865488290787,\n      -0.021899379789829254,\n      0.0479632206261158,\n      -0.01439166534692049,\n      0.0035165532026439905,\n      -0.0833451971411705,\n      -0.06895629316568375,\n      0.03370348736643791,\n      -0.0543905645608902,\n      -0.10073040425777435,\n      -0.020926041528582573,\n      -0.11182185262441635,\n      -0.03516446053981781,\n      -0.11457773298025131,\n      -0.06419026106595993\n    ],\n    [\n      0.07867980748414993,\n      -0.026140611618757248,\n      0.03661023825407028,\n      0.07185503840446472,\n      -0.08401014655828476,\n      -0.02362380363047123,\n      -0.047182466834783554,\n      -0.09529899805784225,\n      0.06103136017918587,\n      -0.061399832367897034,\n      -0.026388019323349,\n      0.07616815716028214,\n      -0.062243130058050156,\n      -0.05777015909552574,\n      0.10933373868465424,\n      0.0034276386722922325,\n      -0.054165031760931015,\n      -0.05389992520213127,\n      0.028134742751717567,\n      0.0177729781717062,\n      0.0035290939267724752,\n      0.06358963251113892,\n      0.03585859388113022,\n      -0.032263852655887604,\n      -0.07142898440361023,\n      -0.06489111483097076,\n      0.04205864295363426,\n      0.021997056901454926,\n      -0.08190101385116577,\n      0.031015029177069664,\n      0.00982300378382206,\n      -0.04000309109687805,\n      -0.05010693520307541,\n      -0.08794105798006058,\n      -0.014535489492118359,\n      -0.01087202038615942,\n      -0.03776119276881218,\n      0.03467709571123123,\n      0.035710517317056656,\n      -0.01122848130762577,\n      0.05285929515957832,\n      -0.09223706275224686,\n      0.0545540377497673,\n      -0.08462971448898315,\n      -0.07980421185493469,\n      0.03258004039525986,\n      -0.0035003991797566414,\n      0.01987108588218689,\n      -0.06623344123363495,\n      -0.09105376899242401,\n      -0.00441896403208375,\n      0.038883090019226074,\n      -0.11775287985801697,\n      -0.0727732852101326,\n      -0.04106916859745979,\n      0.04322732612490654,\n      -0.0028048264794051647,\n      -0.016764380037784576,\n      0.04113640636205673,\n      0.0712372213602066,\n      0.08271802216768265,\n      -0.03513379395008087,\n      -0.05441248416900635,\n      0.0833258181810379,\n      0.07009465992450714,\n      0.004338535945862532,\n      0.04059268906712532,\n      0.023507578298449516,\n      -0.0684981495141983,\n      -0.002066597808152437,\n      0.11983602494001389,\n      0.043337758630514145,\n      0.011660595424473286,\n      0.07785642147064209,\n      0.0034646885469555855,\n      -0.06537073105573654,\n      0.03975759819149971,\n      0.009650807827711105,\n      -0.02090614102780819,\n      -0.0378308929502964,\n      0.015267737209796906,\n      0.02475873753428459,\n      -0.01871616020798683,\n      -0.034983422607183456,\n      -0.011888982728123665,\n      0.09869180619716644,\n      -0.021052338182926178,\n      -0.03743991628289223,\n      0.09015537053346634,\n      -0.051503121852874756,\n      -0.014023376628756523,\n      0.03354869782924652,\n      0.061240848153829575,\n      -0.06968723982572556,\n      0.019225457683205605,\n      -0.07313140481710434,\n      0.0899084061384201,\n      0.05218876898288727,\n      -0.03941852226853371,\n      0.01693861000239849,\n      -0.09952621906995773,\n      0.06487742811441422,\n      0.028735456988215446,\n      0.04942942038178444,\n      0.07088101655244827,\n      0.0715002715587616,\n      -0.06584276258945465,\n      0.006703519262373447,\n      0.07213512808084488,\n      0.010401126928627491,\n      -0.05325058847665787,\n      0.024104449898004532,\n      -0.04810405150055885,\n      0.015948757529258728,\n      0.02451840415596962,\n      -0.13818314671516418,\n      -0.006150449626147747,\n      -0.057755205780267715,\n      -0.03317372873425484,\n      0.04795164614915848,\n      0.13545262813568115,\n      -0.03597766533493996,\n      0.04437403008341789,\n      -0.05429390072822571,\n      0.006112892646342516\n    ],\n    [\n      -0.008518603630363941,\n      -0.14161764085292816,\n      -0.06386459618806839,\n      0.027218764647841454,\n      -0.0637982189655304,\n      0.04726777970790863,\n      0.013455684296786785,\n      0.05998913571238518,\n      -0.06281911581754684,\n      -0.014371802099049091,\n      0.006959160789847374,\n      0.05930439010262489,\n      0.15083825588226318,\n      -0.17413192987442017,\n      -0.12688912451267242,\n      0.08777667582035065,\n      0.02544165775179863,\n      -0.09102867543697357,\n      -0.007067193277180195,\n      -0.025592755526304245,\n      0.015316951088607311,\n      0.06266237795352936,\n      0.061570920050144196,\n      0.013481922447681427,\n      0.03343261405825615,\n      -0.030189864337444305,\n      -0.0421714223921299,\n      -0.08253316581249237,\n      0.13387495279312134,\n      -0.034967485815286636,\n      0.016376513987779617,\n      0.1054244339466095,\n      0.048241026699543,\n      0.06377580016851425,\n      0.09323404729366302,\n      0.013508468866348267,\n      0.11368321627378464,\n      0.11060544848442078,\n      -0.016741499304771423,\n      0.06155477464199066,\n      0.0008814865141175687,\n      0.023999175056815147,\n      -0.0403621643781662,\n      -0.017079386860132217,\n      -0.04206911474466324,\n      -0.06101259961724281,\n      -0.05174601823091507,\n      0.021975859999656677,\n      -0.045638520270586014,\n      -0.028437288478016853,\n      -0.0006146940286271274,\n      0.039187394082546234,\n      0.013616807758808136,\n      -0.13026949763298035,\n      -0.02118038758635521,\n      -0.015496354550123215,\n      -0.0608949288725853,\n      0.05880158767104149,\n      0.052262403070926666,\n      -0.053079359233379364,\n      0.04252098873257637,\n      0.026269566267728806,\n      -0.07492132484912872,\n      -0.056742098182439804,\n      0.0498993806540966,\n      0.011543077416718006,\n      -0.0440681092441082,\n      0.09031842648983002,\n      -0.0410851314663887,\n      0.07805270701646805,\n      0.08315781503915787,\n      -0.025058262050151825,\n      -0.11107118427753448,\n      -0.08567071706056595,\n      0.02716767229139805,\n      0.08241450786590576,\n      0.0007689623162150383,\n      0.014695649035274982,\n      -0.1505293846130371,\n      -0.08545663952827454,\n      -0.040904685854911804,\n      0.0046071275137364864,\n      -0.035924866795539856,\n      -0.08197662234306335,\n      0.005103489384055138,\n      -0.022342748939990997,\n      -0.09878305345773697,\n      -0.006578792352229357,\n      0.038190264254808426,\n      0.034913111478090286,\n      -0.05449909716844559,\n      -0.1040317490696907,\n      -0.03698728233575821,\n      -0.03420630469918251,\n      0.02777841128408909,\n      0.05113993585109711,\n      -0.049465112388134,\n      -0.02796001173555851,\n      -0.004608887247741222,\n      -0.047315195202827454,\n      0.0655796229839325,\n      0.07634685933589935,\n      0.11668431758880615,\n      0.03529500961303711,\n      -0.08489879965782166,\n      0.015158049762248993,\n      -0.019887056201696396,\n      -0.03157292678952217,\n      0.1392013281583786,\n      0.12422702461481094,\n      -0.031112661585211754,\n      -0.04341984912753105,\n      -0.03563348948955536,\n      0.07087893784046173,\n      -0.019378527998924255,\n      -0.08902950584888458,\n      0.00029852124862372875,\n      0.0236804336309433,\n      -0.09294109791517258,\n      -0.05259096994996071,\n      -0.11787188798189163,\n      -0.11443789303302765,\n      0.00040653347969055176,\n      -0.03377852961421013,\n      0.1042579934000969\n    ],\n    [\n      0.038509827107191086,\n      -0.03927667438983917,\n      0.0022766173351556063,\n      0.054186247289180756,\n      0.02419963851571083,\n      0.03410796448588371,\n      -0.03883273899555206,\n      -0.00556847732514143,\n      -0.08408794552087784,\n      0.09653647989034653,\n      -0.01201407890766859,\n      0.06721647083759308,\n      -5.218872684054077e-05,\n      -0.08438028395175934,\n      -0.09905245900154114,\n      -0.08070658892393112,\n      0.03755347803235054,\n      -0.0329553596675396,\n      -0.06568894535303116,\n      0.027343254536390305,\n      0.03354933112859726,\n      -0.07753712683916092,\n      0.011655865237116814,\n      0.05339967831969261,\n      -0.00747302221134305,\n      -0.026907918974757195,\n      0.04437150061130524,\n      -0.12097598612308502,\n      0.002193198772147298,\n      -0.026038408279418945,\n      -0.008190901018679142,\n      -0.059397339820861816,\n      -0.04687628149986267,\n      0.0378170944750309,\n      0.02431829832494259,\n      0.03627483546733856,\n      -0.039317067712545395,\n      0.10777883976697922,\n      0.03808913379907608,\n      0.09490834176540375,\n      0.08494782447814941,\n      -0.02354867197573185,\n      -0.030323950573801994,\n      0.00326897157356143,\n      0.04106540605425835,\n      0.01850149780511856,\n      0.04123943671584129,\n      0.07517468929290771,\n      0.06302912533283234,\n      0.018003122881054878,\n      0.05454230681061745,\n      0.0328444242477417,\n      0.09151721745729446,\n      -0.0648745521903038,\n      0.016867198050022125,\n      0.04386765509843826,\n      -0.03388645127415657,\n      -0.03697244077920914,\n      -0.04563869908452034,\n      0.05616219341754913,\n      0.012015397660434246,\n      -0.111976757645607,\n      -0.06435792148113251,\n      0.0666721910238266,\n      0.03209495544433594,\n      0.047224804759025574,\n      -0.08036114275455475,\n      -0.06514719128608704,\n      0.022787339985370636,\n      -0.09590539336204529,\n      0.07445959746837616,\n      0.05400485917925835,\n      0.005461382679641247,\n      0.001420195447281003,\n      -0.017991840839385986,\n      0.035223811864852905,\n      -0.09067782759666443,\n      -0.01296659279614687,\n      -0.037696871906518936,\n      0.006222043186426163,\n      -0.029466969892382622,\n      0.03293445333838463,\n      0.04577090963721275,\n      -0.048142995685338974,\n      -0.10916590690612793,\n      -0.09317535161972046,\n      -0.10376375913619995,\n      0.0756426751613617,\n      0.08414984494447708,\n      0.00620327377691865,\n      -0.028875015676021576,\n      0.005810731090605259,\n      0.078811414539814,\n      -0.05957043915987015,\n      -0.04891800135374069,\n      -0.04042642191052437,\n      0.08605651557445526,\n      0.04501290246844292,\n      -0.05026346817612648,\n      0.026766352355480194,\n      -0.027520256116986275,\n      0.02897246927022934,\n      -0.04097260534763336,\n      -0.005611712578684092,\n      -0.11272061616182327,\n      -0.051535844802856445,\n      0.007299894466996193,\n      0.1077045351266861,\n      0.0868665799498558,\n      0.11736048012971878,\n      0.09825772047042847,\n      0.02736106514930725,\n      0.0018017564434558153,\n      0.09147531539201736,\n      0.0023190828505903482,\n      -0.006599527783691883,\n      -0.01306164637207985,\n      -0.013429451733827591,\n      0.03530849516391754,\n      -0.07457984983921051,\n      -0.039771176874637604,\n      -0.1143854409456253,\n      0.06898705661296844,\n      -0.002225825795903802,\n      0.06654264032840729\n    ],\n    [\n      -0.11514975130558014,\n      0.03022717870771885,\n      0.1233503594994545,\n      -0.04708151891827583,\n      0.11660561710596085,\n      0.004256398417055607,\n      -0.02715228870511055,\n      -0.05211924389004707,\n      0.006854696664959192,\n      0.04744194447994232,\n      -0.06912378966808319,\n      0.07102524489164352,\n      -0.06354306638240814,\n      -0.0490741953253746,\n      0.0540284626185894,\n      -0.17246071994304657,\n      -0.07632725685834885,\n      -0.01385935116559267,\n      -0.05441143736243248,\n      0.015965647995471954,\n      -0.11000286042690277,\n      -0.11359360814094543,\n      -0.09463199228048325,\n      0.021832343190908432,\n      0.058736249804496765,\n      0.018480224534869194,\n      -6.864624447189271e-05,\n      0.09706126898527145,\n      -0.07022476196289062,\n      0.0665631890296936,\n      0.1115102618932724,\n      -0.06075901910662651,\n      0.06937705725431442,\n      0.003231312846764922,\n      -0.06678856909275055,\n      -0.04710979759693146,\n      0.013759911060333252,\n      -0.05657971650362015,\n      -0.08049898594617844,\n      -0.09243322908878326,\n      -3.3403048291802406e-06,\n      0.008013785816729069,\n      -0.015339416451752186,\n      -0.002684076316654682,\n      0.028673553839325905,\n      -0.04724058136343956,\n      0.04295706748962402,\n      -0.09516777843236923,\n      0.019188253208994865,\n      -0.002585069043561816,\n      0.043513473123311996,\n      -0.022482609376311302,\n      -0.09986979514360428,\n      -0.0865119993686676,\n      0.059077635407447815,\n      0.11211086064577103,\n      -0.056296322494745255,\n      0.04112971946597099,\n      0.0539388582110405,\n      -0.06175299733877182,\n      -0.06950284540653229,\n      -0.06306485086679459,\n      0.017655756324529648,\n      -0.02046441286802292,\n      -0.05848173797130585,\n      -0.009934117086231709,\n      -0.03743699565529823,\n      -0.05305609852075577,\n      0.04631638899445534,\n      0.08460744470357895,\n      0.036551058292388916,\n      0.055187057703733444,\n      0.09557139873504639,\n      0.056094080209732056,\n      0.04752923548221588,\n      -0.031540077179670334,\n      0.05830059200525284,\n      0.014639756642282009,\n      0.033080827444791794,\n      4.551687743514776e-05,\n      0.07939612120389938,\n      -0.07335635274648666,\n      -0.0286833755671978,\n      0.026395373046398163,\n      0.09467817842960358,\n      0.07149821519851685,\n      -0.0008056981605477631,\n      -0.028636783361434937,\n      -0.024691618978977203,\n      -0.08637585490942001,\n      0.07428443431854248,\n      -0.03121384046971798,\n      0.09575515985488892,\n      -0.0149172842502594,\n      -0.06720590591430664,\n      -0.08076724410057068,\n      0.06873977929353714,\n      -0.004956316202878952,\n      0.031835440546274185,\n      0.08976699411869049,\n      0.0411796011030674,\n      -0.028442855924367905,\n      0.055197492241859436,\n      0.12709639966487885,\n      -0.12096720933914185,\n      -0.014784933999180794,\n      0.07021886855363846,\n      -0.006623448338359594,\n      -0.010201473720371723,\n      -0.11212056130170822,\n      0.030544718727469444,\n      0.07584220170974731,\n      0.07933275401592255,\n      -0.07153709977865219,\n      0.05788876861333847,\n      -0.012309382669627666,\n      -0.007745105307549238,\n      0.028266413137316704,\n      -0.013044041581451893,\n      0.0752539113163948,\n      0.057319242507219315,\n      -0.024591444060206413,\n      -0.01295245811343193,\n      0.07285706698894501,\n      0.048896368592977524\n    ],\n    [\n      -0.022398721426725388,\n      0.05644271895289421,\n      -0.022838924080133438,\n      0.08988580852746964,\n      0.07572369277477264,\n      0.03872818872332573,\n      -0.036765843629837036,\n      -0.08924701809883118,\n      0.025435741990804672,\n      -0.04420175030827522,\n      -0.010875084437429905,\n      -0.07347462326288223,\n      -0.10305402427911758,\n      0.0531696118414402,\n      0.13710393011569977,\n      -0.04291712865233421,\n      0.015297197736799717,\n      -0.07113362103700638,\n      -0.07833302021026611,\n      0.0961705893278122,\n      -0.03448919206857681,\n      0.018618470057845116,\n      0.11310125142335892,\n      -0.01046839915215969,\n      -0.009811967611312866,\n      0.05509685352444649,\n      0.020353157073259354,\n      0.07389308512210846,\n      0.03171446919441223,\n      0.04561107978224754,\n      0.08182589709758759,\n      0.009303724393248558,\n      -0.002880157670006156,\n      -0.021428972482681274,\n      0.0635228231549263,\n      -0.030022500082850456,\n      0.06337394565343857,\n      -0.09488563239574432,\n      0.04516361281275749,\n      -0.10401784628629684,\n      -0.07284744828939438,\n      -0.02378753013908863,\n      0.0056275236420333385,\n      0.013391620479524136,\n      -0.014606865122914314,\n      -0.03037179633975029,\n      -0.08820494264364243,\n      0.03756677359342575,\n      -0.006237218622118235,\n      0.04363434016704559,\n      -0.02737193927168846,\n      0.010153383016586304,\n      0.051145680248737335,\n      0.08743587136268616,\n      0.03997364640235901,\n      0.11967223882675171,\n      -0.024030933156609535,\n      -0.04595518484711647,\n      0.006599895656108856,\n      0.04273339360952377,\n      -0.09560275077819824,\n      0.07376328855752945,\n      0.02649158053100109,\n      -0.06388393044471741,\n      -0.05992897227406502,\n      0.0790630504488945,\n      -0.028354186564683914,\n      -0.040866490453481674,\n      0.04432849958539009,\n      -0.0345018096268177,\n      0.07020770013332367,\n      -0.07686403393745422,\n      0.04203977435827255,\n      0.10816580057144165,\n      0.02882762998342514,\n      0.013444332405924797,\n      -0.03895684704184532,\n      -0.022508403286337852,\n      0.07299819588661194,\n      -0.006184814963489771,\n      0.03375186026096344,\n      -0.022586964070796967,\n      -0.008821449242532253,\n      0.054557230323553085,\n      0.050713229924440384,\n      0.010431905277073383,\n      0.08068373799324036,\n      -0.05190211907029152,\n      0.04111485928297043,\n      0.07453633844852448,\n      -0.08211388438940048,\n      -0.020956862717866898,\n      -0.025983018800616264,\n      -0.04230654239654541,\n      -0.06479833275079727,\n      0.05778338015079498,\n      0.07138317823410034,\n      0.15388445556163788,\n      0.0388512909412384,\n      0.010732126422226429,\n      -0.03228716924786568,\n      -0.036318276077508926,\n      -0.018661655485630035,\n      0.009391609579324722,\n      -0.01906028389930725,\n      -0.03484037518501282,\n      -0.05590592697262764,\n      -0.027234472334384918,\n      0.04281066730618477,\n      0.0874963328242302,\n      -0.028059037402272224,\n      0.08665946125984192,\n      -0.025287313386797905,\n      0.051016081124544144,\n      0.0067079863511025906,\n      -0.10735377669334412,\n      0.0627499595284462,\n      0.03745448589324951,\n      -0.03752274438738823,\n      0.07628500461578369,\n      0.011833278462290764,\n      -0.021961823105812073,\n      -0.03292086720466614,\n      -0.007929457351565361,\n      0.10100803524255753\n    ],\n    [\n      0.10373946279287338,\n      -0.0635923445224762,\n      -0.12243478000164032,\n      0.02966116927564144,\n      -0.0869077816605568,\n      0.06407225877046585,\n      -2.0485545974224806e-05,\n      0.09217532724142075,\n      -0.10895223170518875,\n      -0.08757797628641129,\n      0.10227587074041367,\n      0.0774054303765297,\n      0.08475932478904724,\n      -0.08649684488773346,\n      -0.1176556795835495,\n      0.012645140290260315,\n      -0.06733540445566177,\n      0.04434937238693237,\n      -0.03824698552489281,\n      -0.06113831698894501,\n      0.06420819461345673,\n      0.06184975057840347,\n      0.09011653810739517,\n      0.10591176152229309,\n      -0.10817217826843262,\n      -0.025811785832047462,\n      -0.07041474431753159,\n      -0.014326917938888073,\n      0.06164831295609474,\n      -0.05316166207194328,\n      -0.027386344969272614,\n      -0.046905335038900375,\n      -0.07557357102632523,\n      -0.006346689071506262,\n      0.012754829600453377,\n      0.0342521034181118,\n      0.06130829080939293,\n      -0.012022693641483784,\n      -0.018259083852171898,\n      0.08265669643878937,\n      -0.06401046365499496,\n      -0.05877701938152313,\n      0.10667545348405838,\n      -0.017377177253365517,\n      -0.03238626569509506,\n      0.0068602049723267555,\n      -0.028175951912999153,\n      0.03728942573070526,\n      -0.06911556422710419,\n      0.01240030862390995,\n      -0.02238117717206478,\n      -0.027533214539289474,\n      0.08329575508832932,\n      -0.09657925367355347,\n      -0.05372899770736694,\n      -0.046080972999334335,\n      -0.020273221656680107,\n      0.030558552592992783,\n      -0.03456294909119606,\n      0.055836960673332214,\n      -0.05301196500658989,\n      -0.06342089176177979,\n      -0.058259762823581696,\n      0.050088781863451004,\n      0.0753946304321289,\n      -0.09537336975336075,\n      -0.10358558595180511,\n      -0.024903004989027977,\n      -0.025144251063466072,\n      -0.05277921259403229,\n      -0.06932491064071655,\n      -0.08187276124954224,\n      -0.14843116700649261,\n      -0.09485992789268494,\n      -0.002815810963511467,\n      0.027004515752196312,\n      0.06370827555656433,\n      -0.09664299339056015,\n      -0.08548843115568161,\n      0.03336280584335327,\n      -0.14234063029289246,\n      0.031046396121382713,\n      -0.0219093170017004,\n      0.03125045448541641,\n      0.02270841971039772,\n      -0.13991424441337585,\n      0.015123145654797554,\n      0.07498936355113983,\n      0.06729051470756531,\n      -0.028202520683407784,\n      0.004652744624763727,\n      -0.002019475679844618,\n      -0.01499923411756754,\n      0.005876804701983929,\n      0.009995798580348492,\n      0.035082634538412094,\n      0.02300882153213024,\n      0.033100832253694534,\n      -0.004266238305717707,\n      0.038605447858572006,\n      0.05780383571982384,\n      0.09691199660301208,\n      0.01911252550780773,\n      -0.05559377372264862,\n      0.09301824867725372,\n      -0.01727091148495674,\n      0.02592817135155201,\n      0.11489881575107574,\n      -0.03373545780777931,\n      0.09818723797798157,\n      -0.04375416040420532,\n      0.030516183003783226,\n      -0.07693682610988617,\n      -0.010796098969876766,\n      0.03273475915193558,\n      -0.0866466537117958,\n      0.07270939648151398,\n      -0.10420667380094528,\n      0.015242689289152622,\n      -0.018269583582878113,\n      -0.04987869784235954,\n      -0.10455428808927536,\n      0.03831034526228905,\n      0.03356418386101723,\n      0.03336355835199356\n    ],\n    [\n      -0.05566401407122612,\n      0.025408891960978508,\n      -0.04020947962999344,\n      0.09004658460617065,\n      0.01259814016520977,\n      -0.003802185645326972,\n      -0.12915103137493134,\n      0.005566589999943972,\n      0.11238112300634384,\n      -0.066278375685215,\n      -0.08753994107246399,\n      -0.07931048423051834,\n      -0.13005167245864868,\n      0.01524347998201847,\n      0.025135429576039314,\n      -0.13835108280181885,\n      0.031635306775569916,\n      -0.08241448551416397,\n      0.035535525530576706,\n      -0.09791305661201477,\n      -0.009271185845136642,\n      0.060775257647037506,\n      0.09174851328134537,\n      -0.08750699460506439,\n      0.042022980749607086,\n      -0.08970432728528976,\n      -0.08672710508108139,\n      -0.03373324126005173,\n      -0.04736526310443878,\n      0.200872004032135,\n      0.02431345544755459,\n      0.04887541010975838,\n      0.04212993010878563,\n      0.05407176911830902,\n      0.05050986260175705,\n      -0.07424912601709366,\n      -0.038095828145742416,\n      -0.12072232365608215,\n      0.02079615369439125,\n      -0.03296920284628868,\n      0.0027083405293524265,\n      0.04331479221582413,\n      -0.07661233842372894,\n      0.040318556129932404,\n      -0.06214335188269615,\n      0.07868103682994843,\n      -0.040184274315834045,\n      -0.07452055811882019,\n      -0.05443989112973213,\n      0.07521359622478485,\n      0.11991333961486816,\n      0.08902578800916672,\n      -0.02715027704834938,\n      0.0020181077998131514,\n      0.014169352129101753,\n      0.019560903310775757,\n      0.05325554683804512,\n      0.01223824918270111,\n      -0.07344216108322144,\n      0.06215605512261391,\n      0.03809437155723572,\n      0.0978439450263977,\n      0.04557573422789574,\n      0.04244515672326088,\n      0.029464587569236755,\n      -0.047901201993227005,\n      -0.03393126651644707,\n      0.016518618911504745,\n      0.01668745093047619,\n      0.004051238298416138,\n      -0.0036790661979466677,\n      0.03809858113527298,\n      0.07128562778234482,\n      0.05282336473464966,\n      -0.04149390012025833,\n      0.05112311989068985,\n      -0.05544639751315117,\n      -0.052369192242622375,\n      -0.015933049842715263,\n      -0.06982877105474472,\n      -0.01972905918955803,\n      -0.11033465713262558,\n      0.019483119249343872,\n      0.09907566010951996,\n      -0.004283803515136242,\n      0.06562796235084534,\n      0.10588736832141876,\n      -0.056369420140981674,\n      -0.027769213542342186,\n      0.05075663700699806,\n      0.03435127064585686,\n      0.04147816076874733,\n      -0.07933618873357773,\n      -0.05644006282091141,\n      -0.08369111269712448,\n      -0.06411045044660568,\n      0.0655076876282692,\n      -0.038769472390413284,\n      -0.05020470544695854,\n      0.0866231694817543,\n      -0.035117171704769135,\n      -0.0840165913105011,\n      0.14521698653697968,\n      0.09556666016578674,\n      0.03483262285590172,\n      -0.009710141457617283,\n      0.047093700617551804,\n      -0.07896985113620758,\n      0.07550006359815598,\n      0.00723156938329339,\n      -0.033393699675798416,\n      0.05065116658806801,\n      -0.06846103817224503,\n      0.04915108531713486,\n      0.051883380860090256,\n      -0.08104648441076279,\n      0.052609529346227646,\n      -0.03682990372180939,\n      0.09454566240310669,\n      0.08114355802536011,\n      0.0355505645275116,\n      0.046066392213106155,\n      0.06751216948032379,\n      0.11591421067714691,\n      0.03936360776424408\n    ],\n    [\n      -0.08989867568016052,\n      -0.055920522660017014,\n      0.017088335007429123,\n      0.04612753540277481,\n      -0.017722109332680702,\n      0.041773997247219086,\n      -0.012110278941690922,\n      -0.008408996276557446,\n      0.13861756026744843,\n      -0.04562797397375107,\n      0.08380894362926483,\n      0.038023293018341064,\n      0.02299962192773819,\n      0.18363094329833984,\n      0.14237651228904724,\n      0.059169281274080276,\n      0.02325615845620632,\n      -0.11244173347949982,\n      -0.029253242537379265,\n      -0.019971098750829697,\n      -0.014010527171194553,\n      0.08172385394573212,\n      0.056988202035427094,\n      -0.018733704462647438,\n      0.09820084273815155,\n      0.05045655742287636,\n      0.01966073550283909,\n      0.05657825991511345,\n      0.08064485341310501,\n      0.14832070469856262,\n      0.062039781361818314,\n      0.051146138459444046,\n      0.11944916844367981,\n      0.008327459916472435,\n      -0.10826725512742996,\n      -0.02672949619591236,\n      0.03148394823074341,\n      0.038167666643857956,\n      -0.024559499695897102,\n      0.013104240410029888,\n      -0.036191415041685104,\n      0.0609566904604435,\n      0.046789806336164474,\n      -0.012304422445595264,\n      -0.16604529321193695,\n      -0.018709799274802208,\n      0.02994639426469803,\n      0.03040824644267559,\n      0.039455391466617584,\n      0.011401975527405739,\n      0.007687292993068695,\n      0.06725708395242691,\n      -0.023239891976118088,\n      0.0010913894511759281,\n      0.11542435735464096,\n      -0.038395997136831284,\n      -0.008151475340127945,\n      0.039709895849227905,\n      0.013411940075457096,\n      -0.0324043333530426,\n      0.08232730627059937,\n      0.004308530129492283,\n      -0.04630797728896141,\n      -0.011330973356962204,\n      -0.09847340732812881,\n      -0.010028001852333546,\n      -0.045094095170497894,\n      -0.09165982156991959,\n      -0.06282606720924377,\n      -0.12091917544603348,\n      0.008893168531358242,\n      0.12182064354419708,\n      -0.11712028831243515,\n      0.043278198689222336,\n      0.08875082433223724,\n      -0.027753811329603195,\n      -0.14452728629112244,\n      -0.07542451471090317,\n      -0.023959288373589516,\n      -0.06378795206546783,\n      0.10245190560817719,\n      -0.009216918610036373,\n      0.03967816382646561,\n      -0.019748250022530556,\n      -0.01190861500799656,\n      0.08830054849386215,\n      -0.10346448421478271,\n      -0.048009175807237625,\n      -0.014635720290243626,\n      0.05219145864248276,\n      0.005558257922530174,\n      0.036702509969472885,\n      -0.12436313927173615,\n      -0.04944959655404091,\n      -0.10862623900175095,\n      -0.116926409304142,\n      0.08558467775583267,\n      -0.06033886969089508,\n      0.05525127053260803,\n      0.04488224908709526,\n      0.02361677959561348,\n      0.08002321422100067,\n      -0.06986283510923386,\n      -0.04523041471838951,\n      0.0723005160689354,\n      0.10346885025501251,\n      0.09356539696455002,\n      0.087337426841259,\n      -0.06264961510896683,\n      -0.00043645768892019987,\n      0.09365957230329514,\n      -0.007710244506597519,\n      -0.04208746552467346,\n      -0.12617819011211395,\n      -0.03239160403609276,\n      -0.19305837154388428,\n      0.05781363695859909,\n      0.0398537740111351,\n      0.07675522565841675,\n      0.0770437940955162,\n      4.6345521695911884e-05,\n      0.01360782515257597,\n      0.04465112462639809,\n      -0.009534858167171478,\n      0.09090758115053177\n    ],\n    [\n      -0.09198497235774994,\n      -0.027390437200665474,\n      0.06370504200458527,\n      -0.032711997628211975,\n      0.026014618575572968,\n      -0.0034657057840377092,\n      -0.015534019097685814,\n      -0.10062023252248764,\n      0.05462052673101425,\n      -0.01528199389576912,\n      0.04924400523304939,\n      0.03543602302670479,\n      -0.033357709646224976,\n      0.09021621942520142,\n      0.1434003859758377,\n      -0.13471007347106934,\n      0.003743155859410763,\n      0.00664799427613616,\n      0.010635347105562687,\n      0.03394993767142296,\n      -0.014390519820153713,\n      0.06923989206552505,\n      0.0039053098298609257,\n      -0.06946812570095062,\n      0.08987154066562653,\n      -0.00015701178926974535,\n      0.04137023910880089,\n      -0.02060347981750965,\n      -0.017411431297659874,\n      0.1583968997001648,\n      0.03739140182733536,\n      -0.01184164173901081,\n      0.05337962880730629,\n      -0.011780894361436367,\n      -0.07385655492544174,\n      0.04572979360818863,\n      0.05353637412190437,\n      -0.0481717586517334,\n      -0.05169083550572395,\n      0.019281532615423203,\n      -0.01801728457212448,\n      -0.1024387925863266,\n      -0.050134334713220596,\n      -0.033685363829135895,\n      0.01624670997262001,\n      0.05625759810209274,\n      0.07605409622192383,\n      0.06431528180837631,\n      -0.002697757910937071,\n      0.022561652585864067,\n      -0.013140792958438396,\n      -0.057330865412950516,\n      -0.027230558916926384,\n      -0.006562214344739914,\n      0.06610116362571716,\n      -0.025336284190416336,\n      0.05286480486392975,\n      -0.015800772234797478,\n      -0.07295151799917221,\n      -0.07753747701644897,\n      0.040672458708286285,\n      -0.0013661346165463328,\n      -0.034235045313835144,\n      0.02250426635146141,\n      0.05697569251060486,\n      -0.015096720308065414,\n      0.09277471899986267,\n      0.06689150631427765,\n      0.07443855702877045,\n      0.011439820751547813,\n      0.09047815948724747,\n      0.027802303433418274,\n      0.1227998286485672,\n      0.0968390703201294,\n      -0.038084227591753006,\n      -0.08033166080713272,\n      0.005523594096302986,\n      0.021614253520965576,\n      0.04950513318181038,\n      0.06151905283331871,\n      0.06370098143815994,\n      0.062246594578027725,\n      -0.034723538905382156,\n      -0.05664864182472229,\n      0.08054854720830917,\n      0.019721802324056625,\n      0.06384122371673584,\n      0.03687148913741112,\n      -0.08707525581121445,\n      0.04048841819167137,\n      -0.00802216399461031,\n      -0.03717661648988724,\n      -0.061821021139621735,\n      0.0548066571354866,\n      -0.034959495067596436,\n      -0.02684224769473076,\n      0.08103237301111221,\n      0.1277313381433487,\n      -0.02290777489542961,\n      0.0008628003997728229,\n      0.052740417420864105,\n      0.019343987107276917,\n      -0.011626283638179302,\n      0.06742726266384125,\n      0.06779233366250992,\n      0.02786080911755562,\n      -0.018122093752026558,\n      -0.03659748286008835,\n      0.03785887733101845,\n      0.07236528396606445,\n      0.002022462897002697,\n      0.10206788778305054,\n      0.021067766472697258,\n      0.021042130887508392,\n      0.040145378559827805,\n      -0.039683688431978226,\n      0.05833734571933746,\n      0.09401842206716537,\n      0.07370173186063766,\n      0.027303123846650124,\n      0.0959388017654419,\n      0.11577741801738739,\n      -0.033029451966285706,\n      0.09292298555374146,\n      -0.01535059418529272\n    ],\n    [\n      -0.0034803960006684065,\n      0.03918684646487236,\n      0.041331905871629715,\n      0.052527423948049545,\n      -0.010699818842113018,\n      0.06668269634246826,\n      0.09174899756908417,\n      0.11228707432746887,\n      -0.140687957406044,\n      0.08822941780090332,\n      0.0921531692147255,\n      0.026432272046804428,\n      0.1390203982591629,\n      -0.1254291832447052,\n      -0.1793178915977478,\n      0.15950186550617218,\n      -0.07413934171199799,\n      -0.05904155969619751,\n      -0.03283878043293953,\n      0.07044000178575516,\n      0.0894002690911293,\n      0.07677167654037476,\n      0.06637491285800934,\n      0.10993947088718414,\n      0.001345593947917223,\n      0.06581297516822815,\n      0.09711360186338425,\n      0.035907600075006485,\n      0.018896130844950676,\n      -0.09909534454345703,\n      -0.028757421299815178,\n      -0.02107216790318489,\n      0.042728494852781296,\n      0.11015567183494568,\n      0.1036069393157959,\n      0.041986022144556046,\n      0.04226422309875488,\n      0.08777052164077759,\n      0.07358302175998688,\n      -0.029063716530799866,\n      0.07562597841024399,\n      0.07428020983934402,\n      0.00039781309897080064,\n      -0.02036464214324951,\n      0.07223989814519882,\n      0.05533933639526367,\n      0.025531591847538948,\n      0.06976418197154999,\n      0.09259973466396332,\n      -0.07140280306339264,\n      -0.017024647444486618,\n      -0.039431165903806686,\n      0.0657898560166359,\n      -0.031104784458875656,\n      -0.055416449904441833,\n      -0.04146983101963997,\n      -0.051376793533563614,\n      0.019063666462898254,\n      0.08929388225078583,\n      0.004709460306912661,\n      0.006694369949400425,\n      -0.0177972000092268,\n      0.021301759406924248,\n      0.01752694509923458,\n      0.005561213940382004,\n      0.04380619525909424,\n      0.02524624578654766,\n      -0.021378403529524803,\n      0.06315040588378906,\n      -0.08008268475532532,\n      -0.041048962622880936,\n      -0.036931030452251434,\n      -0.15187738835811615,\n      -0.01615886203944683,\n      -0.08367268741130829,\n      -0.04460349306464195,\n      0.011571403592824936,\n      -0.06872515380382538,\n      -0.0386253222823143,\n      -0.03461979329586029,\n      -0.08388067781925201,\n      -0.06355703622102737,\n      0.07398702949285507,\n      -0.010543674230575562,\n      0.039559997618198395,\n      0.009053179062902927,\n      -0.043614450842142105,\n      0.07130918651819229,\n      -0.03973546624183655,\n      0.0003398781700525433,\n      0.027544299140572548,\n      -0.03976475074887276,\n      -0.03441818803548813,\n      0.04213551804423332,\n      0.017590517178177834,\n      0.040886908769607544,\n      0.06267495453357697,\n      -0.04627406224608421,\n      0.0409451425075531,\n      -0.07039127498865128,\n      0.06986559182405472,\n      0.0007910120766609907,\n      0.10699921101331711,\n      -0.005133482161909342,\n      0.040170636028051376,\n      0.019720321521162987,\n      0.04208874702453613,\n      0.10926027595996857,\n      0.020135561004281044,\n      -0.007544699124991894,\n      -0.06464028358459473,\n      -0.061440609395504,\n      -0.0012066001072525978,\n      0.07502202689647675,\n      -0.013645408675074577,\n      -0.020511014387011528,\n      0.10549287497997284,\n      0.03870892897248268,\n      -0.009361968375742435,\n      -0.10764537006616592,\n      -0.03534248098731041,\n      -0.1342771351337433,\n      -0.012911180034279823,\n      0.05820201709866524,\n      0.010942071676254272\n    ],\n    [\n      -0.018113013356924057,\n      0.03620040789246559,\n      0.05979670584201813,\n      0.08188684284687042,\n      0.12283109873533249,\n      -0.04328088462352753,\n      -0.10991805046796799,\n      -0.07393749058246613,\n      0.17201606929302216,\n      -0.09735121577978134,\n      -0.0643329918384552,\n      -0.004314768593758345,\n      -0.13705475628376007,\n      0.04409085586667061,\n      0.06449215114116669,\n      -0.11976127326488495,\n      0.009557446464896202,\n      -0.07726628333330154,\n      -0.0556691475212574,\n      -0.032234616577625275,\n      -0.024416999891400337,\n      0.07089169323444366,\n      0.015633147209882736,\n      -0.06932897120714188,\n      0.09511741995811462,\n      0.03717861324548721,\n      0.009457297623157501,\n      0.048626963049173355,\n      0.056420762091875076,\n      0.08639594167470932,\n      0.09032431989908218,\n      -0.030292995274066925,\n      -0.05478380620479584,\n      -0.09087176620960236,\n      -0.024562662467360497,\n      0.06259071081876755,\n      -0.0676039382815361,\n      0.042265500873327255,\n      -0.09456551820039749,\n      0.04688660055398941,\n      -0.027959920465946198,\n      0.05171305686235428,\n      -0.09093999862670898,\n      -0.0215500108897686,\n      -0.010443296283483505,\n      0.08454306423664093,\n      0.059391237795352936,\n      0.00688105309382081,\n      0.008924937807023525,\n      -0.012579169124364853,\n      0.007946009747684002,\n      0.006759689189493656,\n      0.03315899521112442,\n      -0.05118974670767784,\n      0.09349451959133148,\n      0.10189449787139893,\n      0.05796089395880699,\n      0.07353000342845917,\n      -0.02456587366759777,\n      -0.006320641841739416,\n      0.07732576131820679,\n      0.07137865573167801,\n      -0.06719361990690231,\n      -0.07185922563076019,\n      0.10073629021644592,\n      0.09145359694957733,\n      0.005658748559653759,\n      -0.02074483036994934,\n      -0.01626909337937832,\n      0.028137292712926865,\n      0.05401904881000519,\n      0.017176590859889984,\n      0.03088558092713356,\n      0.06289628893136978,\n      0.053895022720098495,\n      0.04137715324759483,\n      0.04027196019887924,\n      -0.08957119286060333,\n      0.11350905895233154,\n      -0.02294689603149891,\n      -0.027658218517899513,\n      -0.10218868404626846,\n      -0.04793747514486313,\n      0.03397345542907715,\n      0.10572833567857742,\n      -0.016667896881699562,\n      0.029339613392949104,\n      -0.004090112634003162,\n      0.02621055208146572,\n      0.06199302151799202,\n      -0.057972948998212814,\n      -0.04493794962763786,\n      0.07313159853219986,\n      0.002612682990729809,\n      0.06670207530260086,\n      0.0807216688990593,\n      0.052557725459337234,\n      0.051961325109004974,\n      0.1105210930109024,\n      0.09440173953771591,\n      -0.012543588876724243,\n      0.004018052481114864,\n      0.08467625826597214,\n      0.10629680007696152,\n      0.021466046571731567,\n      -0.04949837550520897,\n      -0.053998563438653946,\n      -0.01532748807221651,\n      0.07945922762155533,\n      0.0045671043917536736,\n      -0.06691446155309677,\n      0.05572988837957382,\n      -0.07272190600633621,\n      -0.05740562826395035,\n      0.04490220546722412,\n      0.0150696886703372,\n      0.036945246160030365,\n      -0.009813148528337479,\n      0.09148567914962769,\n      0.10706371814012527,\n      0.0813022032380104,\n      0.13966907560825348,\n      0.07592926174402237,\n      -0.017588095739483833,\n      -0.032937873154878616\n    ],\n    [\n      -0.019458239898085594,\n      -0.027897901833057404,\n      -0.029330262914299965,\n      0.020884916186332703,\n      -0.0020542226266115904,\n      0.08238861709833145,\n      0.0024169792886823416,\n      0.05581746995449066,\n      -0.06126843020319939,\n      -0.08146057277917862,\n      -0.04771777614951134,\n      -0.0066259801387786865,\n      0.061460867524147034,\n      -0.001038875081576407,\n      -0.02872416377067566,\n      0.04080313816666603,\n      -0.06106734648346901,\n      -0.01609671488404274,\n      -0.134674072265625,\n      -0.07017366588115692,\n      -0.040430713444948196,\n      -0.014059055596590042,\n      -0.011330030858516693,\n      -0.0665409043431282,\n      0.02774852328002453,\n      -0.08443474769592285,\n      -0.027245815843343735,\n      -0.020911775529384613,\n      -0.048895563930273056,\n      -0.018908077850937843,\n      0.010019654408097267,\n      0.05382120981812477,\n      -0.09189155697822571,\n      0.057538073509931564,\n      -0.06333515048027039,\n      -0.023469554260373116,\n      -0.04794546961784363,\n      -0.03454887494444847,\n      -0.0705413967370987,\n      0.11092320084571838,\n      -0.009786220267415047,\n      -0.00042794138425961137,\n      0.07826873660087585,\n      0.026306437328457832,\n      -0.005606234073638916,\n      -0.0011513496283441782,\n      0.0975317507982254,\n      -0.10376829653978348,\n      0.0021489206701517105,\n      0.06093968078494072,\n      -0.04349223151803017,\n      -0.006681662052869797,\n      0.06822264939546585,\n      -0.12930016219615936,\n      -0.006792346481233835,\n      -0.010249214246869087,\n      -0.04403522610664368,\n      0.04220762103796005,\n      0.11457881331443787,\n      -0.04279395937919617,\n      0.06504733115434647,\n      0.10382279753684998,\n      0.016513075679540634,\n      0.06344055384397507,\n      0.018857339397072792,\n      0.04613073542714119,\n      -0.058767169713974,\n      0.10013286024332047,\n      -0.07671304792165756,\n      0.018997544422745705,\n      0.10429277271032333,\n      0.03282199054956436,\n      0.0671410784125328,\n      -0.07021685689687729,\n      0.10494076460599899,\n      0.06564795970916748,\n      0.06026683375239372,\n      0.016049427911639214,\n      -0.06390874832868576,\n      0.040184568613767624,\n      -0.00888351071625948,\n      0.038785047829151154,\n      -0.1092558279633522,\n      0.028222063556313515,\n      0.039497099816799164,\n      0.01735410839319229,\n      -0.05150007829070091,\n      -0.05906332656741142,\n      -0.035741206258535385,\n      0.0036173861008137465,\n      -0.03011605143547058,\n      -0.10153147578239441,\n      -0.04011617600917816,\n      -0.02944699116051197,\n      -0.07333803921937943,\n      -0.007085050921887159,\n      -0.0659857988357544,\n      -0.02004593051970005,\n      0.044037554413080215,\n      0.009302119724452496,\n      0.025744954124093056,\n      -0.016877206042408943,\n      0.019252503290772438,\n      -0.08398667722940445,\n      0.03129502385854721,\n      -0.07475020736455917,\n      -0.018526645377278328,\n      0.022739265114068985,\n      0.00656603928655386,\n      0.06616608798503876,\n      -0.04037575051188469,\n      -0.09362710267305374,\n      0.011254630982875824,\n      -0.029637662693858147,\n      -0.08674826472997665,\n      -0.07833623886108398,\n      -0.09538059681653976,\n      -0.012145120650529861,\n      -0.09683303534984589,\n      -0.004495061933994293,\n      0.07828693091869354,\n      0.004844192881137133,\n      0.015557891689240932,\n      0.04108908027410507,\n      -0.02286335825920105\n    ],\n    [\n      0.10916728526353836,\n      0.0027983244508504868,\n      0.041171617805957794,\n      -0.03327435627579689,\n      -0.08987447619438171,\n      0.10041525959968567,\n      0.08605856448411942,\n      0.02876647375524044,\n      -0.17731501162052155,\n      0.013253418728709221,\n      0.029323481023311615,\n      -0.056419774889945984,\n      0.07825975865125656,\n      -0.008904984220862389,\n      -0.09027367830276489,\n      -0.032252587378025055,\n      -0.010183240287005901,\n      0.024854807183146477,\n      0.06227507442235947,\n      0.07826510816812515,\n      0.02051660232245922,\n      -0.004315379075706005,\n      -0.03934815526008606,\n      0.07257598638534546,\n      0.0020193408709019423,\n      -0.005405060946941376,\n      0.12202140688896179,\n      0.01540396362543106,\n      0.031902506947517395,\n      -0.18825575709342957,\n      -0.10658483952283859,\n      -0.02848249115049839,\n      0.06657089293003082,\n      0.06490440666675568,\n      0.09338540583848953,\n      0.08867001533508301,\n      0.0800989642739296,\n      0.022800445556640625,\n      0.09634949266910553,\n      0.05859049782156944,\n      0.08051636815071106,\n      0.05117424577474594,\n      0.04021497443318367,\n      -0.1071968823671341,\n      0.0715685784816742,\n      0.03515627607703209,\n      -0.09283258765935898,\n      -0.05179772153496742,\n      -0.0009272009483538568,\n      -0.0011232354445382953,\n      -0.09931793063879013,\n      -0.06599574536085129,\n      0.0367276556789875,\n      0.0299307182431221,\n      -0.11215227842330933,\n      0.022403666749596596,\n      -0.06868024915456772,\n      -0.07161845266819,\n      0.021608324721455574,\n      0.018246522173285484,\n      -0.03838326781988144,\n      -0.006087364628911018,\n      -0.03677403926849365,\n      -0.0691101923584938,\n      0.005664747208356857,\n      0.040565501898527145,\n      -0.01672983355820179,\n      0.014959805645048618,\n      -0.00905641634017229,\n      -0.0684700608253479,\n      0.024842509999871254,\n      0.05091828852891922,\n      -0.07615740597248077,\n      -0.0469636544585228,\n      -0.06777562946081161,\n      0.06745243072509766,\n      0.009325572289526463,\n      0.0031042657792568207,\n      0.01105613261461258,\n      0.007008776534348726,\n      -0.030378134921193123,\n      -0.029173126444220543,\n      -0.039865151047706604,\n      0.04014508053660393,\n      -0.06014637649059296,\n      -0.08487940579652786,\n      0.007849738001823425,\n      0.0824190303683281,\n      -0.08585076034069061,\n      -0.0832425057888031,\n      -0.049232520163059235,\n      0.05660519003868103,\n      0.017949210479855537,\n      -0.07997648417949677,\n      -0.040789660066366196,\n      -0.08129039406776428,\n      0.02234799787402153,\n      -0.01825970970094204,\n      0.021093999966979027,\n      0.002683397615328431,\n      0.10115782916545868,\n      -0.030653607100248337,\n      0.01982707716524601,\n      -0.01864703558385372,\n      0.024900153279304504,\n      -0.047164011746644974,\n      -0.09964737296104431,\n      0.1137523353099823,\n      -0.055347006767988205,\n      -0.014308271929621696,\n      0.053767211735248566,\n      -0.007929982617497444,\n      -0.05902037024497986,\n      0.004281232133507729,\n      -0.02334127016365528,\n      -0.07014250010251999,\n      -0.056910764425992966,\n      -0.06663153320550919,\n      -0.04992415010929108,\n      -0.006447651423513889,\n      -0.018411096185445786,\n      -0.14602968096733093,\n      0.06519819796085358,\n      0.04933435469865799,\n      0.08150491863489151\n    ],\n    [\n      -0.07704812288284302,\n      -0.0035795068833976984,\n      -0.07834751904010773,\n      -0.006803426891565323,\n      -0.05020441114902496,\n      0.02982313372194767,\n      -0.028382131829857826,\n      0.027341987937688828,\n      -0.017497897148132324,\n      0.09936141222715378,\n      -0.01912648044526577,\n      0.027758173644542694,\n      0.005796593613922596,\n      -0.012607662007212639,\n      -0.09565678238868713,\n      0.10924439132213593,\n      -0.08396518975496292,\n      -0.0903441458940506,\n      0.011581890285015106,\n      0.0685652419924736,\n      0.03261663392186165,\n      -0.1207553967833519,\n      0.02236863411962986,\n      0.04087572917342186,\n      0.005179916974157095,\n      0.04833245277404785,\n      0.005602085962891579,\n      -0.05349276214838028,\n      -0.09376810491085052,\n      -0.1275717318058014,\n      0.024967070668935776,\n      -0.0019112664740532637,\n      0.08632232248783112,\n      -0.009329433552920818,\n      -0.06147504970431328,\n      0.09640828520059586,\n      -0.028240125626325607,\n      -0.0201298575848341,\n      0.024104570969939232,\n      0.008451295085251331,\n      0.08068990707397461,\n      -0.07301045954227448,\n      -0.06335382163524628,\n      -0.010551044717431068,\n      0.06402497738599777,\n      -0.0137679073959589,\n      -0.08520106226205826,\n      0.03783213347196579,\n      0.041178055107593536,\n      -0.0435316264629364,\n      -0.035243380814790726,\n      0.08192455768585205,\n      -0.01587740331888199,\n      0.043299686163663864,\n      0.09142644703388214,\n      -0.03539440408349037,\n      -0.06681597232818604,\n      0.05637071281671524,\n      0.08443290740251541,\n      -0.042247265577316284,\n      -0.02791830338537693,\n      -0.018916647881269455,\n      0.009993455372750759,\n      -0.05958176031708717,\n      0.02502904273569584,\n      -0.13658227026462555,\n      0.0019186516292393208,\n      0.037093162536621094,\n      0.04221070930361748,\n      0.05015568807721138,\n      -0.029672903940081596,\n      -0.09055431187152863,\n      -0.07046011835336685,\n      -0.0033694326411932707,\n      0.05944887176156044,\n      -0.0786326453089714,\n      -0.008737120777368546,\n      0.026567116379737854,\n      0.0003376716049388051,\n      0.0002335280878469348,\n      -0.11902007460594177,\n      0.015880484133958817,\n      0.06068575754761696,\n      0.042634569108486176,\n      -0.053266748785972595,\n      0.0550115592777729,\n      -0.06729184836149216,\n      -0.030578091740608215,\n      0.006065827794373035,\n      0.0241527259349823,\n      0.034238170832395554,\n      0.03910516947507858,\n      -0.010331491008400917,\n      0.0041689202189445496,\n      0.04070168361067772,\n      -0.05886964499950409,\n      0.03430677205324173,\n      -0.07088650017976761,\n      -0.1349201202392578,\n      -0.06369379907846451,\n      0.0353931188583374,\n      0.045197512954473495,\n      0.051938463002443314,\n      -0.002273824065923691,\n      0.12855741381645203,\n      0.1234917864203453,\n      0.0806892141699791,\n      0.04526214301586151,\n      0.0020277223084121943,\n      0.017209000885486603,\n      0.025732537731528282,\n      0.018330110237002373,\n      -0.05483952537178993,\n      -0.028016353026032448,\n      0.0751153826713562,\n      0.1250726580619812,\n      -0.0629456639289856,\n      0.013440156355500221,\n      -0.023727601394057274,\n      -0.004911031108349562,\n      -0.031169457361102104,\n      0.017024900764226913,\n      0.03685354068875313,\n      -0.042643219232559204,\n      0.0007292967056855559\n    ],\n    [\n      -0.046713460236787796,\n      -0.0828726515173912,\n      0.07060448080301285,\n      0.09816942363977432,\n      -0.08958936482667923,\n      -0.04554589465260506,\n      -0.02911400981247425,\n      0.024885764345526695,\n      0.10725171118974686,\n      -0.07020682841539383,\n      -0.02039908617734909,\n      0.07409293204545975,\n      0.07111576944589615,\n      -0.04447374492883682,\n      0.1256401687860489,\n      -0.008736415766179562,\n      -0.02411269024014473,\n      -0.0953778326511383,\n      -0.02182846888899803,\n      0.028455611318349838,\n      -0.049840185791254044,\n      -0.010588452219963074,\n      0.006511180195957422,\n      -0.0009261122904717922,\n      -0.1285673975944519,\n      0.04533692076802254,\n      0.03328763693571091,\n      0.0977988988161087,\n      -0.05041327327489853,\n      0.1642272174358368,\n      0.08603659272193909,\n      0.0018599778413772583,\n      0.04286918789148331,\n      -0.0366230234503746,\n      -0.05354036018252373,\n      -0.03664783388376236,\n      -0.023834506049752235,\n      -0.012601139955222607,\n      -0.025915037840604782,\n      0.02953510545194149,\n      -0.04475514963269234,\n      0.007382597308605909,\n      -0.13648530840873718,\n      0.010535179637372494,\n      0.01774517633020878,\n      -0.02947637252509594,\n      -0.04356866702437401,\n      0.02548995614051819,\n      0.02735361084342003,\n      -0.04017616808414459,\n      -0.06534159183502197,\n      0.033572129905223846,\n      -0.08743958175182343,\n      -0.06594464182853699,\n      -0.07295601814985275,\n      -0.008760168217122555,\n      -0.035823747515678406,\n      0.052687518298625946,\n      -0.11712900549173355,\n      -0.03607422485947609,\n      0.059511296451091766,\n      -0.048088692128658295,\n      -0.03542982041835785,\n      -0.03816010430455208,\n      -0.017308896407485008,\n      0.01764806918799877,\n      0.018707970157265663,\n      -0.08186400681734085,\n      -0.08073324710130692,\n      -0.07129551470279694,\n      0.007026756182312965,\n      -0.14036162197589874,\n      -0.07344571501016617,\n      -0.07846488058567047,\n      -0.012124814093112946,\n      -0.02363486774265766,\n      -0.004476225934922695,\n      0.08459886163473129,\n      0.09270939230918884,\n      -0.07712513208389282,\n      -0.06626667827367783,\n      0.040567029267549515,\n      -0.006238148082047701,\n      0.10370516777038574,\n      0.07152192294597626,\n      0.021374203264713287,\n      -0.0690055713057518,\n      0.0234826747328043,\n      -0.0847933292388916,\n      -0.04975670203566551,\n      -0.03923249989748001,\n      0.005172845907509327,\n      0.05109040066599846,\n      0.09899595379829407,\n      -0.04189801216125488,\n      0.023005547001957893,\n      -0.03338058292865753,\n      -0.04417862743139267,\n      -0.06285499781370163,\n      0.02634977363049984,\n      -0.03515012189745903,\n      0.06636732816696167,\n      0.017718974500894547,\n      0.09137976169586182,\n      -0.07298561185598373,\n      -0.035227928310632706,\n      -0.08332906663417816,\n      -0.04469633474946022,\n      -0.04688390716910362,\n      -0.004110804293304682,\n      0.01565229706466198,\n      -0.018040494993329048,\n      0.04840174317359924,\n      -0.09589244425296783,\n      0.03968812897801399,\n      -0.1630208045244217,\n      -0.03544966131448746,\n      0.022074300795793533,\n      0.04714381694793701,\n      0.09395476430654526,\n      -0.024863852187991142,\n      -0.06769146025180817,\n      -0.03354407846927643,\n      0.04033803567290306,\n      0.05097155272960663\n    ],\n    [\n      0.10523400455713272,\n      -0.10154499858617783,\n      -0.050783269107341766,\n      -0.060374561697244644,\n      -0.10351749509572983,\n      0.043747201561927795,\n      0.007280610967427492,\n      0.14554554224014282,\n      -0.11000363528728485,\n      0.09076912701129913,\n      0.011135957203805447,\n      0.05123492330312729,\n      0.11275438219308853,\n      -0.11759494245052338,\n      -0.04624409228563309,\n      0.0493643619120121,\n      0.07287729531526566,\n      -0.03219214081764221,\n      0.009359681978821754,\n      0.10376863926649094,\n      0.07040347158908844,\n      0.009864522144198418,\n      -0.08143135905265808,\n      0.07353254407644272,\n      -0.09562414884567261,\n      -0.032948147505521774,\n      -0.021219713613390923,\n      -0.08788230270147324,\n      -0.04591010883450508,\n      -0.05616861209273338,\n      -0.0016919607296586037,\n      0.045724064111709595,\n      0.006462473887950182,\n      0.11847805976867676,\n      0.08674038201570511,\n      -0.03783461079001427,\n      -0.020295964553952217,\n      0.09463249146938324,\n      -0.0022857447620481253,\n      -0.027446361258625984,\n      0.050631724298000336,\n      -0.0264555886387825,\n      0.027650149539113045,\n      0.034318044781684875,\n      -0.0562019869685173,\n      -0.022015556693077087,\n      0.01904768869280815,\n      0.02631516568362713,\n      -0.054607342928647995,\n      -0.05831952765583992,\n      -0.12971797585487366,\n      0.062306445091962814,\n      0.004185241647064686,\n      -0.09518486261367798,\n      0.04432903230190277,\n      0.00864739902317524,\n      -0.03503851592540741,\n      -0.06781208515167236,\n      -0.07153131067752838,\n      -0.02991911955177784,\n      -0.02832394652068615,\n      0.03661924600601196,\n      -0.01935734786093235,\n      -0.028179893270134926,\n      -0.07502668350934982,\n      -0.06886472553014755,\n      -0.1241203173995018,\n      -0.055264703929424286,\n      -0.07650591433048248,\n      -0.07593785226345062,\n      -0.004608701448887587,\n      0.043378014117479324,\n      -0.0509682223200798,\n      -0.06446988880634308,\n      -0.09230934828519821,\n      -0.06788620352745056,\n      0.040424663573503494,\n      -0.07834120839834213,\n      0.0009129967074841261,\n      -0.04857344925403595,\n      -0.1344287395477295,\n      -0.015870554372668266,\n      0.07163680344820023,\n      -0.03813247010111809,\n      -0.10368586331605911,\n      0.010388120077550411,\n      -0.013721371069550514,\n      -0.06816468387842178,\n      0.006548698991537094,\n      -0.01807573437690735,\n      -0.02862989529967308,\n      -0.04235412925481796,\n      -0.004370854701846838,\n      -0.11208033561706543,\n      0.024238595739006996,\n      -0.021127969026565552,\n      0.03041650541126728,\n      0.03315287083387375,\n      -0.13692274689674377,\n      -0.029916364699602127,\n      0.002981747267767787,\n      0.05923756584525108,\n      0.034211769700050354,\n      -0.0684150904417038,\n      -0.047479838132858276,\n      0.07233579456806183,\n      -0.004538635723292828,\n      0.007523334585130215,\n      0.03812732547521591,\n      0.08316423743963242,\n      0.024145858362317085,\n      -0.013380847871303558,\n      0.05148574709892273,\n      0.06527478247880936,\n      0.07505597174167633,\n      -0.03872448205947876,\n      -0.10725019127130508,\n      -0.0027100115548819304,\n      -0.13703632354736328,\n      -0.1296108514070511,\n      -0.01016857847571373,\n      -0.09199190139770508,\n      0.049368903040885925,\n      -0.00227055000141263,\n      0.07803767919540405\n    ],\n    [\n      -0.05567590892314911,\n      -0.026869704946875572,\n      -0.0007214383222162724,\n      0.09895125031471252,\n      0.07028026133775711,\n      -0.014627424068748951,\n      0.016874507069587708,\n      -0.07111026346683502,\n      0.1799641102552414,\n      -0.04186953976750374,\n      0.06143835932016373,\n      -0.0015423368895426393,\n      0.09143485873937607,\n      0.0978216752409935,\n      0.01856963336467743,\n      0.0014047474833205342,\n      -0.05221540108323097,\n      -0.11819970607757568,\n      -0.08585555851459503,\n      0.057163480669260025,\n      0.06776336580514908,\n      0.05823410674929619,\n      -0.10146826505661011,\n      0.04030190408229828,\n      -0.13945849239826202,\n      -0.08086620271205902,\n      -0.07694049179553986,\n      0.044340331107378006,\n      -0.007730093318969011,\n      0.20456473529338837,\n      0.0554085299372673,\n      -0.06366293132305145,\n      0.05524430423974991,\n      0.04612252116203308,\n      -0.03451203927397728,\n      -0.04731674864888191,\n      0.03939857706427574,\n      0.06242513284087181,\n      -0.03815963491797447,\n      0.03969692066311836,\n      -0.013145464472472668,\n      0.05496138334274292,\n      0.012769678607583046,\n      0.00477982684969902,\n      0.04271009936928749,\n      -0.031018944457173347,\n      -0.01567740924656391,\n      -0.12901653349399567,\n      -0.05029210075736046,\n      0.05575605854392052,\n      0.036887481808662415,\n      0.024047261103987694,\n      -0.05243570730090141,\n      0.032470498234033585,\n      -0.08689741790294647,\n      0.004166776314377785,\n      -0.11255764216184616,\n      -0.006063980981707573,\n      -0.05081897974014282,\n      -0.09284188598394394,\n      0.0023524067364633083,\n      -0.031219327822327614,\n      -0.09964165836572647,\n      -0.12352567911148071,\n      -0.047777969390153885,\n      -0.13359354436397552,\n      -0.0485694445669651,\n      -0.015500877983868122,\n      -0.0649447962641716,\n      -0.026792285963892937,\n      -0.017978420481085777,\n      0.008044836111366749,\n      0.015310662798583508,\n      0.043630875647068024,\n      -0.07160806655883789,\n      -0.03623451292514801,\n      -0.10093189030885696,\n      -0.01829654537141323,\n      -0.05212236940860748,\n      0.04244033619761467,\n      -0.04048475995659828,\n      0.0028939913026988506,\n      0.0004371317627374083,\n      0.016911255195736885,\n      0.07032489776611328,\n      0.06482847034931183,\n      -0.042615268379449844,\n      0.004535425454378128,\n      0.00722380168735981,\n      -0.08342580497264862,\n      -0.11361491680145264,\n      -0.02651403844356537,\n      -0.011991533450782299,\n      -0.01576162874698639,\n      0.03157370910048485,\n      0.03493442386388779,\n      0.06308814883232117,\n      -0.073886439204216,\n      -0.03865474462509155,\n      -0.047504693269729614,\n      -0.020797325298190117,\n      -0.04449206590652466,\n      0.03450315073132515,\n      -0.04065437242388725,\n      -0.04630759358406067,\n      -0.052411191165447235,\n      0.07638337463140488,\n      -0.07099312543869019,\n      0.008428610861301422,\n      0.08195146173238754,\n      0.00334475957788527,\n      0.033868882805109024,\n      -0.0019395834533497691,\n      -0.12789656221866608,\n      0.07104720920324326,\n      -0.14546088874340057,\n      -0.03249156102538109,\n      -0.0364832729101181,\n      0.028360385447740555,\n      0.12892527878284454,\n      -0.1270408034324646,\n      -0.10152643918991089,\n      -0.024876482784748077,\n      0.10033849626779556,\n      -0.020375141873955727\n    ],\n    [\n      0.08512986451387405,\n      -0.11782722175121307,\n      -0.08354011178016663,\n      -0.08310893923044205,\n      0.03923974186182022,\n      0.09855421632528305,\n      -0.054123181849718094,\n      0.08849731832742691,\n      0.014316564425826073,\n      0.05276646092534065,\n      0.05066707730293274,\n      -0.09094234555959702,\n      -0.03465322032570839,\n      -0.10239662975072861,\n      -0.03379511088132858,\n      -0.04051978513598442,\n      0.01308939978480339,\n      0.07230953872203827,\n      0.09786763787269592,\n      -0.03595493361353874,\n      0.027531974017620087,\n      -0.027834121137857437,\n      -0.008723727427423,\n      -0.09394770860671997,\n      0.05474364757537842,\n      0.10646557807922363,\n      0.018549075350165367,\n      -0.07950122654438019,\n      0.030439598485827446,\n      0.10590684413909912,\n      0.1278933733701706,\n      -0.08156134188175201,\n      0.12486042827367783,\n      0.11310568451881409,\n      -0.08634515106678009,\n      0.05471018701791763,\n      0.09638122469186783,\n      -0.0197460800409317,\n      -0.06951313465833664,\n      -0.09653373062610626,\n      -0.01928470842540264,\n      0.03877374157309532,\n      0.01711180992424488,\n      0.08941982686519623,\n      -0.044754624366760254,\n      0.015460838563740253,\n      0.025738704949617386,\n      -0.023191919550299644,\n      0.0967460349202156,\n      -0.016095349565148354,\n      -0.092520572245121,\n      0.12679652869701385,\n      0.15573884546756744,\n      0.1686146855354309,\n      -0.010725469328463078,\n      -0.05501101166009903,\n      -0.04285121336579323,\n      -0.008538639172911644,\n      0.0019792714156210423,\n      -0.0009606886887922883,\n      0.06263501197099686,\n      0.11124540120363235,\n      0.01754816062748432,\n      0.025774413719773293,\n      0.08753778785467148,\n      0.02310408093035221,\n      -0.004104190040379763,\n      0.05808468535542488,\n      -0.05644402280449867,\n      0.05925282835960388,\n      0.06648949533700943,\n      0.02117478847503662,\n      -0.06766422092914581,\n      0.03527950495481491,\n      -0.009941354393959045,\n      -0.013513240963220596,\n      0.00023858575150370598,\n      -0.02748546376824379,\n      -0.006296456325799227,\n      0.11248648166656494,\n      0.05495337024331093,\n      -0.022033603861927986,\n      0.0852835401892662,\n      -0.026345960795879364,\n      0.019532814621925354,\n      -0.07047287374734879,\n      -0.08283258974552155,\n      0.09918232262134552,\n      -0.005210519302636385,\n      0.06785045564174652,\n      -0.14974813163280487,\n      -0.05260908603668213,\n      0.009526398032903671,\n      0.09258037805557251,\n      -0.11858870834112167,\n      0.007210107054561377,\n      0.02545580454170704,\n      -0.012782377190887928,\n      0.01989566721022129,\n      -0.013092723675072193,\n      -0.10284333676099777,\n      -0.058145035058259964,\n      -0.06335175037384033,\n      -0.08615696430206299,\n      0.020281637087464333,\n      0.045723121613264084,\n      0.049000050872564316,\n      -0.0921032652258873,\n      -0.014977696351706982,\n      -0.06116039678454399,\n      0.06642649322748184,\n      -0.040070485323667526,\n      -0.04902861639857292,\n      -0.05090729892253876,\n      -0.03324180841445923,\n      0.011710062623023987,\n      0.011688782833516598,\n      -0.06574038416147232,\n      0.058611106127500534,\n      -0.09170668572187424,\n      0.09132711589336395,\n      -0.035145461559295654,\n      0.060027606785297394,\n      0.02536272257566452,\n      0.09029781818389893\n    ],\n    [\n      0.007979591377079487,\n      0.016694890335202217,\n      0.05860874056816101,\n      -0.06633272022008896,\n      0.12121210992336273,\n      0.011979888193309307,\n      0.018628325313329697,\n      -0.03699957951903343,\n      -0.10434091091156006,\n      0.056245606392621994,\n      -0.13027606904506683,\n      -0.0783868134021759,\n      0.06096412613987923,\n      -0.05222511291503906,\n      -0.027447056025266647,\n      0.1191890761256218,\n      -0.12793299555778503,\n      -0.05609063804149628,\n      0.0036642805207520723,\n      -0.043825387954711914,\n      -0.006790165789425373,\n      -0.06996175646781921,\n      0.061555054038763046,\n      0.06479156762361526,\n      0.0637681856751442,\n      0.037189558148384094,\n      0.10754702240228653,\n      0.1028689444065094,\n      -0.07650632411241531,\n      -0.024385198950767517,\n      -0.046169377863407135,\n      -0.019216688349843025,\n      0.04731755331158638,\n      -0.06568409502506256,\n      -0.09052871912717819,\n      0.007172617129981518,\n      0.06464260071516037,\n      -0.06421107053756714,\n      0.06381592899560928,\n      0.010595565661787987,\n      -0.09339579939842224,\n      -0.006485242396593094,\n      -0.10938077419996262,\n      0.029209725558757782,\n      0.06148233637213707,\n      -0.03796969726681709,\n      -0.10629485547542572,\n      -0.06023151054978371,\n      -0.04729472100734711,\n      0.010981567203998566,\n      0.07770434767007828,\n      -0.05093885958194733,\n      0.054282981902360916,\n      -0.033930640667676926,\n      0.03595642000436783,\n      0.06312385946512222,\n      0.0539201945066452,\n      0.02369026653468609,\n      -0.06926506012678146,\n      0.10445322096347809,\n      -0.06607861071825027,\n      0.10773243010044098,\n      -0.09777466952800751,\n      -0.08372148126363754,\n      -0.05828559398651123,\n      0.049071695655584335,\n      0.00040383823215961456,\n      -0.08027882874011993,\n      -0.04346458241343498,\n      -0.05093076452612877,\n      -0.028504151850938797,\n      -0.052222661674022675,\n      0.020939480513334274,\n      -0.07301025092601776,\n      -0.0513935387134552,\n      0.05437202751636505,\n      0.08639009296894073,\n      -0.01936158910393715,\n      0.006976291071623564,\n      0.06564915180206299,\n      0.0552009753882885,\n      -0.012260787189006805,\n      0.008317392319440842,\n      -0.025806427001953125,\n      0.033066872507333755,\n      0.1334831863641739,\n      0.02847328409552574,\n      -0.0596611388027668,\n      -0.05919044837355614,\n      -0.08317199349403381,\n      0.03306106477975845,\n      -0.009638920426368713,\n      -0.01678306795656681,\n      -0.07409656792879105,\n      0.014985350891947746,\n      0.04905366897583008,\n      0.049162477254867554,\n      0.026420725509524345,\n      -0.12638695538043976,\n      -0.03795222193002701,\n      -0.05770302191376686,\n      0.054847441613674164,\n      0.05295222997665405,\n      0.09663985669612885,\n      0.05953863263130188,\n      0.05234503746032715,\n      -0.03078192099928856,\n      -0.0816001370549202,\n      0.0008000439265742898,\n      0.12013859301805496,\n      -0.0308015625923872,\n      0.0626155212521553,\n      0.057194631546735764,\n      0.06091548129916191,\n      -0.03383198752999306,\n      -0.024382146075367928,\n      0.027137314900755882,\n      -0.04422784969210625,\n      -0.0006912354729138315,\n      0.0829055979847908,\n      -0.020583519712090492,\n      0.02928466536104679,\n      -0.0015264826361089945,\n      0.10188909620046616,\n      0.00875399261713028\n    ]\n  ],\n  "hidden_bias": [\n    -0.059366002678871155,\n    0.01755364052951336,\n    -0.01755266822874546,\n    0.0989345908164978,\n    -0.025840947404503822,\n    -0.04106292873620987,\n    -0.07462307810783386,\n    0.008240658789873123,\n    -0.09676116704940796,\n    0.010334385558962822,\n    -0.030790019780397415,\n    0.07493358105421066,\n    0.0530235692858696,\n    0.013715156354010105,\n    0.05339706316590309,\n    0.07512697577476501,\n    -0.08717875927686691,\n    0.14186373353004456,\n    0.029470710083842278,\n    0.02572190761566162,\n    0.046229906380176544,\n    -0.012891204096376896,\n    0.02110261097550392,\n    0.012393234297633171,\n    0.006228527054190636,\n    0.05633559823036194,\n    0.05465266853570938,\n    -0.03845210745930672,\n    -0.01429440826177597,\n    0.03769984841346741,\n    -0.024121178314089775,\n    -0.021350571885704994,\n    0.009118910878896713,\n    0.0797470435500145,\n    0.08283727616071701,\n    0.05997795611619949,\n    0.07750875502824783,\n    0.027563422918319702,\n    -0.019622009247541428,\n    0.015812570229172707,\n    -0.012416464276611805,\n    -0.038354065269231796,\n    -0.04274454712867737,\n    0.1077321320772171,\n    0.05492435395717621,\n    -0.04352631792426109,\n    -0.032259806990623474,\n    -0.04384304955601692,\n    0.08080541342496872,\n    0.038744986057281494,\n    0.020926041528582573,\n    0.07097527384757996,\n    -0.029574109241366386,\n    -0.051515281200408936,\n    0.04872919246554375,\n    0.03723809868097305,\n    0.07537657767534256,\n    0.012739671394228935,\n    -0.11799649894237518,\n    0.08359985798597336,\n    0.01995047740638256,\n    -0.04643488675355911,\n    -0.05857226252555847,\n    0.10229139775037766\n  ],\n  "output_weight": [\n    0.047145068645477295,\n    -0.05211591348052025,\n    0.06420345604419708,\n    -0.007236778736114502,\n    -0.057570137083530426,\n    0.08976568281650543,\n    -0.04531039670109749,\n    0.11937057971954346,\n    0.08534686267375946,\n    -0.059995707124471664,\n    0.11489525437355042,\n    -0.05398575961589813,\n    -0.04722172021865845,\n    -0.06468089669942856,\n    -0.08640875667333603,\n    0.03614569082856178,\n    0.05937705561518669,\n    0.04280667379498482,\n    -0.03038957715034485,\n    0.08814207464456558,\n    -0.050944581627845764,\n    0.049477092921733856,\n    -0.05430493503808975,\n    0.08161074668169022,\n    0.11567285656929016,\n    -0.05219365283846855,\n    0.05929102003574371,\n    0.061442092061042786,\n    0.05016840249300003,\n    -0.050816070288419724,\n    -0.03531648963689804,\n    0.06323553621768951,\n    -0.04621729627251625,\n    0.06200838461518288,\n    0.04300665855407715,\n    -0.11142922192811966,\n    -0.01603156514465809,\n    -0.04835321009159088,\n    -0.0407029464840889,\n    0.07323216646909714,\n    0.040685318410396576,\n    0.054149653762578964,\n    0.04798121750354767,\n    0.06324859708547592,\n    -0.050479426980018616,\n    0.01245635561645031,\n    -0.08381222933530807,\n    -0.006676835939288139,\n    0.03139606490731239,\n    0.03355368971824646,\n    -0.04599680006504059,\n    0.03215000778436661,\n    0.08446186035871506,\n    0.052086591720581055,\n    -0.058734696358442307,\n    0.04986153542995453,\n    -0.031090978533029556,\n    -0.04415573552250862,\n    -0.03757194057106972,\n    0.06765136122703552,\n    -0.05031956359744072,\n    0.06882426142692566,\n    0.08863687515258789,\n    -0.014135395176708698\n  ],\n  "output_bias": 0.058406323194503784,\n  "training": {\n    "method": "paired counterfactual policy distillation with residual scorer",\n    "objective": "ranking",\n    "pair_gap": 0.0,\n    "uncertainty_only_weights": true,\n    "base_calibration_scale": 0.05,\n    "seed": 20261022,\n    "regularization": 0.01,\n    "best_epoch": 2,\n    "validation_loss": 0.6875867247581482,\n    "baseline_validation_loss": 0.6922053098678589,\n    "training_games": 2400,\n    "validation_games": 599,\n    "source_dataset": {\n      "method": "paired one-step policy-improvement rollouts",\n      "input_features": [\n        "play",\n        "number",\n        "skip",\n        "reverse",\n        "draw_two",\n        "wild",\n        "wild_draw_four",\n        "remaining_colour_count",\n        "remaining_colour_fraction",\n        "remaining_match_count",\n        "attack_next_threat",\n        "reverse_escape_threat",\n        "wild_when_small_hand",\n        "finish",\n        "keep",\n        "choose_colour",\n        "next_can_finish",\n        "own_hand_size",\n        "successor_colour_absence",\n        "successor_symbol_absence",\n        "successor_colour_preference",\n        "successor_finish_risk",\n        "remaining_colour_components",\n        "largest_component_fraction",\n        "remaining_bridge_symbols",\n        "remaining_duplicates",\n        "remaining_colour_entropy",\n        "remaining_wilds",\n        "chosen_colour_exhausted",\n        "stack_penalty_play",\n        "successor_one_card_colour_absence",\n        "successor_one_card_symbol_absence",\n        "remaining_colour_slot_0_number_0",\n        "remaining_colour_slot_0_number_1",\n        "remaining_colour_slot_0_number_2",\n        "remaining_colour_slot_0_number_3",\n        "remaining_colour_slot_0_number_4",\n        "remaining_colour_slot_0_number_5",\n        "remaining_colour_slot_0_number_6",\n        "remaining_colour_slot_0_number_7",\n        "remaining_colour_slot_0_number_8",\n        "remaining_colour_slot_0_number_9",\n        "remaining_colour_slot_0_skip_None",\n        "remaining_colour_slot_0_reverse_None",\n        "remaining_colour_slot_0_draw_two_None",\n        "remaining_colour_slot_1_number_0",\n        "remaining_colour_slot_1_number_1",\n        "remaining_colour_slot_1_number_2",\n        "remaining_colour_slot_1_number_3",\n        "remaining_colour_slot_1_number_4",\n        "remaining_colour_slot_1_number_5",\n        "remaining_colour_slot_1_number_6",\n        "remaining_colour_slot_1_number_7",\n        "remaining_colour_slot_1_number_8",\n        "remaining_colour_slot_1_number_9",\n        "remaining_colour_slot_1_skip_None",\n        "remaining_colour_slot_1_reverse_None",\n        "remaining_colour_slot_1_draw_two_None",\n        "remaining_colour_slot_2_number_0",\n        "remaining_colour_slot_2_number_1",\n        "remaining_colour_slot_2_number_2",\n        "remaining_colour_slot_2_number_3",\n        "remaining_colour_slot_2_number_4",\n        "remaining_colour_slot_2_number_5",\n        "remaining_colour_slot_2_number_6",\n        "remaining_colour_slot_2_number_7",\n        "remaining_colour_slot_2_number_8",\n        "remaining_colour_slot_2_number_9",\n        "remaining_colour_slot_2_skip_None",\n        "remaining_colour_slot_2_reverse_None",\n        "remaining_colour_slot_2_draw_two_None",\n        "remaining_colour_slot_3_number_0",\n        "remaining_colour_slot_3_number_1",\n        "remaining_colour_slot_3_number_2",\n        "remaining_colour_slot_3_number_3",\n        "remaining_colour_slot_3_number_4",\n        "remaining_colour_slot_3_number_5",\n        "remaining_colour_slot_3_number_6",\n        "remaining_colour_slot_3_number_7",\n        "remaining_colour_slot_3_number_8",\n        "remaining_colour_slot_3_number_9",\n        "remaining_colour_slot_3_skip_None",\n        "remaining_colour_slot_3_reverse_None",\n        "remaining_colour_slot_3_draw_two_None",\n        "remaining_wild_count",\n        "remaining_wild_draw_four_count",\n        "action_number_0",\n        "action_number_1",\n        "action_number_2",\n        "action_number_3",\n        "action_number_4",\n        "action_number_5",\n        "action_number_6",\n        "action_number_7",\n        "action_number_8",\n        "action_number_9",\n        "opponent_slot_0_present",\n        "opponent_slot_0_hand_size",\n        "opponent_slot_0_one_card",\n        "opponent_slot_0_colour_absence",\n        "opponent_slot_0_symbol_absence",\n        "opponent_slot_0_colour_preference",\n        "opponent_slot_1_present",\n        "opponent_slot_1_hand_size",\n        "opponent_slot_1_one_card",\n        "opponent_slot_1_colour_absence",\n        "opponent_slot_1_symbol_absence",\n        "opponent_slot_1_colour_preference",\n        "opponent_slot_2_present",\n        "opponent_slot_2_hand_size",\n        "opponent_slot_2_one_card",\n        "opponent_slot_2_colour_absence",\n        "opponent_slot_2_symbol_absence",\n        "opponent_slot_2_colour_preference",\n        "selected_is_active_colour",\n        "pending_penalty_size",\n        "turn_progress",\n        "active_opponents",\n        "closest_opponent_risk",\n        "route_sequence_fraction",\n        "route_sequence_stranded",\n        "route_ending_wild",\n        "route_next_colour_absence",\n        "route_next_symbol_absence",\n        "route_next_colour_preference"\n      ],\n      "feature_version": 4,\n      "games": 3000,\n      "states": 11872,\n      "actions": 38902,\n      "trials": 8,\n      "future_shuffle": true,\n      "reward_conditioning": "present hidden simulator hands held fixed; future draw-pile permutations averaged",\n      "max_actions": 8,\n      "seed_start": 49000000,\n      "sdk_commit": "0cda3acc1bb5c05fa1803a8fbd7761770dc6d5fe",\n      "incumbent": {\n        "format": "triad-uno-route-linear-v1",\n        "features": [\n          "play",\n          "number",\n          "skip",\n          "reverse",\n          "draw_two",\n          "wild",\n          "wild_draw_four",\n          "remaining_colour_count",\n          "remaining_colour_fraction",\n          "remaining_match_count",\n          "attack_next_threat",\n          "reverse_escape_threat",\n          "wild_when_small_hand",\n          "finish",\n          "keep",\n          "choose_colour",\n          "next_can_finish",\n          "own_hand_size",\n          "sequence_fraction",\n          "sequence_stranded",\n          "ending_wild",\n          "next_colour_absence",\n          "next_symbol_absence",\n          "next_colour_preference"\n        ],\n        "weights": [\n          8.022466349142684,\n          0.44153069585311683,\n          -0.06834035646341297,\n          1.4643534535685507,\n          -2.338619866848944,\n          -3.4540620544700498,\n          -4.276279368110987,\n          1.6602353386949849,\n          1.0165100023007814,\n          0.901341302040876,\n          3.853185081203389,\n          2.4436903583872347,\n          1.7223969671761579,\n          11.872419584625783,\n          -4.598739912392293,\n          3.347870880522802,\n          0.32778952616834095,\n          -0.24548152213113314,\n          0.054801310646275025,\n          -1.1349214815798592,\n          -0.04225874297028254,\n          1.1191018681573717,\n          0.21891410472226475,\n          -0.08122782541248305\n        ],\n        "training": {\n          "method": "frozen-base residual CEM, own-hand bounded sequence features",\n          "seed": 20261019,\n          "iterations": 24,\n          "sdk_commit": "0cda3acc1bb5c05fa1803a8fbd7761770dc6d5fe",\n          "selection_seed_start": 21000000,\n          "test_seed_start": 22000000,\n          "base": {\n            "format": "triad-uno-linear-v1",\n            "features": [\n              "play",\n              "number",\n              "skip",\n              "reverse",\n              "draw_two",\n              "wild",\n              "wild_draw_four",\n              "remaining_colour_count",\n              "remaining_colour_fraction",\n              "remaining_match_count",\n              "attack_next_threat",\n              "reverse_escape_threat",\n              "wild_when_small_hand",\n              "finish",\n              "keep",\n              "choose_colour",\n              "next_can_finish",\n              "own_hand_size"\n            ],\n            "weights": [\n              8.022466349142684,\n              0.44153069585311683,\n              -0.06834035646341297,\n              1.4643534535685507,\n              -2.338619866848944,\n              -3.4540620544700498,\n              -4.276279368110987,\n              1.6602353386949849,\n              1.0165100023007814,\n              0.901341302040876,\n              3.853185081203389,\n              2.4436903583872347,\n              1.7223969671761579,\n              11.872419584625783,\n              -4.598739912392293,\n              3.347870880522802,\n              0.32778952616834095,\n              -0.24548152213113314\n            ],\n            "training": {\n              "method": "cross-entropy policy optimization",\n              "seed": 20261014,\n              "generations": 30,\n              "population": 32,\n              "games_per_candidate": 64,\n              "validation": 3.005859375,\n              "sdk_commit": "0cda3acc1bb5c05fa1803a8fbd7761770dc6d5fe",\n              "feature_count": 18\n            }\n          }\n        }\n      },\n      "seconds": 1638.4,\n      "reservoir_sample": true,\n      "own_hand_size_histogram": {\n        "1": 106,\n        "2": 1146,\n        "3": 1762,\n        "4": 1882,\n        "5": 1900,\n        "6": 1847,\n        "7": 1744,\n        "8": 566,\n        "9": 361,\n        "10": 225,\n        "11": 139,\n        "12": 74,\n        "13": 52,\n        "14": 27,\n        "15": 21,\n        "16": 9,\n        "17": 5,\n        "18": 4,\n        "19": 1,\n        "22": 1\n      },\n      "phase_histogram": {\n        "TURN": 9755,\n        "AFTER_DRAW": 2104,\n        "CHOOSE_INITIAL_COLOR": 13\n      }\n    },\n    "sdk_commit": "0cda3acc1bb5c05fa1803a8fbd7761770dc6d5fe",\n    "residual_scale": 1.0\n  }\n}\n', 'neural-mse.json': '{\n  "format": "triad-uno-residual-mlp-v3",\n  "features": [\n    "play",\n    "number",\n    "skip",\n    "reverse",\n    "draw_two",\n    "wild",\n    "wild_draw_four",\n    "remaining_colour_count",\n    "remaining_colour_fraction",\n    "remaining_match_count",\n    "attack_next_threat",\n    "reverse_escape_threat",\n    "wild_when_small_hand",\n    "finish",\n    "keep",\n    "choose_colour",\n    "next_can_finish",\n    "own_hand_size",\n    "successor_colour_absence",\n    "successor_symbol_absence",\n    "successor_colour_preference",\n    "successor_finish_risk",\n    "remaining_colour_components",\n    "largest_component_fraction",\n    "remaining_bridge_symbols",\n    "remaining_duplicates",\n    "remaining_colour_entropy",\n    "remaining_wilds",\n    "chosen_colour_exhausted",\n    "stack_penalty_play",\n    "successor_one_card_colour_absence",\n    "successor_one_card_symbol_absence",\n    "remaining_colour_slot_0_number_0",\n    "remaining_colour_slot_0_number_1",\n    "remaining_colour_slot_0_number_2",\n    "remaining_colour_slot_0_number_3",\n    "remaining_colour_slot_0_number_4",\n    "remaining_colour_slot_0_number_5",\n    "remaining_colour_slot_0_number_6",\n    "remaining_colour_slot_0_number_7",\n    "remaining_colour_slot_0_number_8",\n    "remaining_colour_slot_0_number_9",\n    "remaining_colour_slot_0_skip_None",\n    "remaining_colour_slot_0_reverse_None",\n    "remaining_colour_slot_0_draw_two_None",\n    "remaining_colour_slot_1_number_0",\n    "remaining_colour_slot_1_number_1",\n    "remaining_colour_slot_1_number_2",\n    "remaining_colour_slot_1_number_3",\n    "remaining_colour_slot_1_number_4",\n    "remaining_colour_slot_1_number_5",\n    "remaining_colour_slot_1_number_6",\n    "remaining_colour_slot_1_number_7",\n    "remaining_colour_slot_1_number_8",\n    "remaining_colour_slot_1_number_9",\n    "remaining_colour_slot_1_skip_None",\n    "remaining_colour_slot_1_reverse_None",\n    "remaining_colour_slot_1_draw_two_None",\n    "remaining_colour_slot_2_number_0",\n    "remaining_colour_slot_2_number_1",\n    "remaining_colour_slot_2_number_2",\n    "remaining_colour_slot_2_number_3",\n    "remaining_colour_slot_2_number_4",\n    "remaining_colour_slot_2_number_5",\n    "remaining_colour_slot_2_number_6",\n    "remaining_colour_slot_2_number_7",\n    "remaining_colour_slot_2_number_8",\n    "remaining_colour_slot_2_number_9",\n    "remaining_colour_slot_2_skip_None",\n    "remaining_colour_slot_2_reverse_None",\n    "remaining_colour_slot_2_draw_two_None",\n    "remaining_colour_slot_3_number_0",\n    "remaining_colour_slot_3_number_1",\n    "remaining_colour_slot_3_number_2",\n    "remaining_colour_slot_3_number_3",\n    "remaining_colour_slot_3_number_4",\n    "remaining_colour_slot_3_number_5",\n    "remaining_colour_slot_3_number_6",\n    "remaining_colour_slot_3_number_7",\n    "remaining_colour_slot_3_number_8",\n    "remaining_colour_slot_3_number_9",\n    "remaining_colour_slot_3_skip_None",\n    "remaining_colour_slot_3_reverse_None",\n    "remaining_colour_slot_3_draw_two_None",\n    "remaining_wild_count",\n    "remaining_wild_draw_four_count",\n    "action_number_0",\n    "action_number_1",\n    "action_number_2",\n    "action_number_3",\n    "action_number_4",\n    "action_number_5",\n    "action_number_6",\n    "action_number_7",\n    "action_number_8",\n    "action_number_9",\n    "opponent_slot_0_present",\n    "opponent_slot_0_hand_size",\n    "opponent_slot_0_one_card",\n    "opponent_slot_0_colour_absence",\n    "opponent_slot_0_symbol_absence",\n    "opponent_slot_0_colour_preference",\n    "opponent_slot_1_present",\n    "opponent_slot_1_hand_size",\n    "opponent_slot_1_one_card",\n    "opponent_slot_1_colour_absence",\n    "opponent_slot_1_symbol_absence",\n    "opponent_slot_1_colour_preference",\n    "opponent_slot_2_present",\n    "opponent_slot_2_hand_size",\n    "opponent_slot_2_one_card",\n    "opponent_slot_2_colour_absence",\n    "opponent_slot_2_symbol_absence",\n    "opponent_slot_2_colour_preference",\n    "selected_is_active_colour",\n    "pending_penalty_size",\n    "turn_progress",\n    "active_opponents",\n    "closest_opponent_risk",\n    "route_sequence_fraction",\n    "route_sequence_stranded",\n    "route_ending_wild",\n    "route_next_colour_absence",\n    "route_next_symbol_absence",\n    "route_next_colour_preference"\n  ],\n  "base_weights": [\n    0.40112331745713425,\n    0.022076534792655842,\n    -0.0034170178231706485,\n    0.07321767267842753,\n    -0.11693099334244722,\n    -0.1727031027235025,\n    -0.21381396840554934,\n    0.08301176693474925,\n    0.05082550011503907,\n    0.04506706510204381,\n    0.19265925406016948,\n    0.12218451791936175,\n    0.0861198483588079,\n    0.5936209792312892,\n    -0.22993699561961467,\n    0.16739354402614012,\n    0.01638947630841705,\n    -0.012274076106556658,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0027400655323137515,\n    -0.05674607407899296,\n    -0.002112937148514127,\n    0.055955093407868584,\n    0.010945705236113238,\n    -0.0040613912706241526\n  ],\n  "hidden_weight": [\n    [\n      0.04263515770435333,\n      0.05156116932630539,\n      -0.03423198685050011,\n      -0.07111416757106781,\n      -0.050904396921396255,\n      -0.0330461747944355,\n      -0.11784877628087997,\n      -0.08668777346611023,\n      0.05475462228059769,\n      0.05671612173318863,\n      -0.11146381497383118,\n      -0.11079604178667068,\n      -0.06450983136892319,\n      -0.06128299981355667,\n      0.17302057147026062,\n      -0.04246027022600174,\n      -0.09424950927495956,\n      -0.007960150018334389,\n      -0.033333420753479004,\n      0.03051326610147953,\n      0.06039796769618988,\n      -0.1326654851436615,\n      -0.10819371789693832,\n      0.04563678428530693,\n      0.02325543202459812,\n      -0.11094581335783005,\n      -0.08316620439291,\n      0.05650278925895691,\n      0.09220601618289948,\n      -0.04666605591773987,\n      0.043494902551174164,\n      -0.07344934344291687,\n      -0.02555551566183567,\n      0.01695602759718895,\n      0.0366407111287117,\n      -0.065343476831913,\n      0.04122272878885269,\n      -0.10092677921056747,\n      -0.08570262044668198,\n      -0.12516631186008453,\n      -0.12202900648117065,\n      -0.09446239471435547,\n      -0.01652873121201992,\n      0.022930998355150223,\n      -0.0429571159183979,\n      -0.08370955288410187,\n      0.057299140840768814,\n      -0.001235017436556518,\n      -0.09508315473794937,\n      -0.019352391362190247,\n      -0.02671467512845993,\n      -0.037908125668764114,\n      -0.005102860275655985,\n      0.04231798276305199,\n      0.0304265059530735,\n      -0.06751501560211182,\n      -0.15771980583667755,\n      0.05912699177861214,\n      -0.09111803770065308,\n      0.0422612689435482,\n      -0.07133985310792923,\n      0.05226055160164833,\n      -0.05258368328213692,\n      0.010109125636518002,\n      0.08298368752002716,\n      -0.09288813173770905,\n      0.05981706455349922,\n      0.022108163684606552,\n      -0.01656019687652588,\n      0.04611291363835335,\n      0.06637471914291382,\n      0.010786179453134537,\n      0.006195493973791599,\n      0.01376103051006794,\n      0.08180559426546097,\n      -0.07221081852912903,\n      -0.04499240592122078,\n      -0.04728732258081436,\n      -0.028024792671203613,\n      0.004362138453871012,\n      0.019853048026561737,\n      0.0458197146654129,\n      -0.004075372125953436,\n      -0.010538555681705475,\n      0.005582340527325869,\n      0.001792520866729319,\n      0.07984436303377151,\n      -0.005039297044277191,\n      -0.049705974757671356,\n      -0.06284650415182114,\n      0.025013534352183342,\n      0.03192894533276558,\n      0.03970399871468544,\n      -0.042975541204214096,\n      -0.06622868776321411,\n      -0.0021838454995304346,\n      0.0442386232316494,\n      0.06542207300662994,\n      0.0332755371928215,\n      0.07824750244617462,\n      -0.048252373933792114,\n      -0.03547922894358635,\n      -0.04678860306739807,\n      0.1279294341802597,\n      0.006380883045494556,\n      -0.021364161744713783,\n      0.06832435727119446,\n      -0.02980591170489788,\n      0.036393068730831146,\n      0.10754130780696869,\n      -0.0750068873167038,\n      -0.0635349377989769,\n      0.04128235951066017,\n      0.08598055690526962,\n      0.0021640565246343613,\n      -0.22108611464500427,\n      -5.105696618556976e-05,\n      -0.03257022053003311,\n      -0.004387338645756245,\n      0.037363775074481964,\n      0.06051197648048401,\n      0.07676053792238235,\n      0.044842351227998734,\n      0.09157897531986237,\n      0.07755625247955322\n    ],\n    [\n      0.009932365268468857,\n      -0.08361970633268356,\n      0.15133869647979736,\n      0.04301109537482262,\n      0.0392671674489975,\n      0.08345923572778702,\n      -0.024312999099493027,\n      0.022891271859407425,\n      -0.033497415482997894,\n      -0.04126674309372902,\n      0.09648498892784119,\n      -0.04397108778357506,\n      0.034265123307704926,\n      -0.04380505532026291,\n      -0.10725913941860199,\n      -0.02344120293855667,\n      0.09207576513290405,\n      -0.07635649293661118,\n      0.030237138271331787,\n      0.10691561549901962,\n      -0.06923635303974152,\n      -0.0008815762121230364,\n      0.09802350401878357,\n      -0.04597445949912071,\n      -0.08716540783643723,\n      0.11700493097305298,\n      0.001711034681648016,\n      -0.02285771258175373,\n      -0.049364760518074036,\n      -0.03251085430383682,\n      0.05581144243478775,\n      -0.036376386880874634,\n      -0.012444714084267616,\n      0.05430624261498451,\n      -0.07007946819067001,\n      0.016433415934443474,\n      -0.07041793316602707,\n      -0.15781313180923462,\n      0.09840312600135803,\n      -0.050646062940359116,\n      -0.1001555398106575,\n      0.018135439604520798,\n      0.05697889253497124,\n      -0.007464334834367037,\n      0.061844900250434875,\n      -0.04065949469804764,\n      0.13424715399742126,\n      -0.1507607251405716,\n      -0.14105528593063354,\n      -0.02004924975335598,\n      0.07982742786407471,\n      -0.08455903083086014,\n      -0.08158688992261887,\n      0.022449472919106483,\n      -0.03092595376074314,\n      -0.051085181534290314,\n      0.07774869352579117,\n      -0.024546829983592033,\n      -0.062347155064344406,\n      -0.012603553012013435,\n      0.10985074937343597,\n      0.012567335739731789,\n      -0.03585028275847435,\n      0.1672118753194809,\n      -0.06409406661987305,\n      0.14178359508514404,\n      -0.004546409472823143,\n      -0.11468523740768433,\n      -0.00040467671351507306,\n      0.009935468435287476,\n      -0.09397760778665543,\n      -0.03403094783425331,\n      0.07940127700567245,\n      -0.07309405505657196,\n      -0.012269599363207817,\n      -0.0712665244936943,\n      0.10228291898965836,\n      -0.096824049949646,\n      0.01277387049049139,\n      -0.03170618414878845,\n      0.13649649918079376,\n      0.05044370889663696,\n      -0.09598352760076523,\n      -0.07280800491571426,\n      -0.10095244646072388,\n      0.11211702227592468,\n      0.02222839742898941,\n      0.08514691144227982,\n      0.019129954278469086,\n      -0.03846809267997742,\n      -0.05436372011899948,\n      0.03919195383787155,\n      -0.012639731168746948,\n      -0.03998832404613495,\n      -0.11798684298992157,\n      -0.0462467223405838,\n      -0.042439136654138565,\n      0.05133401229977608,\n      -0.002357236808165908,\n      -0.012094425968825817,\n      0.016963446512818336,\n      -0.015250502154231071,\n      -0.06644883006811142,\n      0.017717961221933365,\n      0.0839470624923706,\n      -0.09482631087303162,\n      0.1761036515235901,\n      -0.030864417552947998,\n      -0.10547316819429398,\n      -0.06349500268697739,\n      0.025408750399947166,\n      -0.027837207540869713,\n      0.03402706980705261,\n      0.005206422880291939,\n      -0.08383707702159882,\n      0.01937245763838291,\n      0.139844611287117,\n      0.039660003036260605,\n      0.0802713930606842,\n      2.4454551748931408e-05,\n      0.06353655457496643,\n      0.04076950252056122,\n      -0.07030003517866135,\n      0.08532115072011948,\n      -0.06610813736915588\n    ],\n    [\n      -0.0645844042301178,\n      0.07573027163743973,\n      -0.01662426069378853,\n      0.03906211256980896,\n      -0.08268707245588303,\n      0.021672315895557404,\n      -0.194929301738739,\n      0.007564892992377281,\n      0.0395614430308342,\n      0.05597393587231636,\n      -0.08288979530334473,\n      -0.05269024521112442,\n      -0.1620568186044693,\n      -0.04858619347214699,\n      0.1634603887796402,\n      -0.05903556942939758,\n      -0.10618459433317184,\n      -0.12940849363803864,\n      0.042839113622903824,\n      -0.10919762402772903,\n      -0.03726841136813164,\n      -0.04561949521303177,\n      0.005917576607316732,\n      0.025468163192272186,\n      0.10360461473464966,\n      0.0009637210168875754,\n      -0.0038922205567359924,\n      0.08217157423496246,\n      0.08859993517398834,\n      0.0714321881532669,\n      0.08381760120391846,\n      -0.006510550156235695,\n      0.012654855847358704,\n      -0.052415020763874054,\n      -0.03650850057601929,\n      -0.044183891266584396,\n      -0.0585852675139904,\n      0.05733984336256981,\n      -0.09614519029855728,\n      0.06637277454137802,\n      0.05576883628964424,\n      -0.03464353829622269,\n      -0.12790435552597046,\n      -0.0014580492861568928,\n      0.024240896105766296,\n      0.026510179042816162,\n      -0.013692999258637428,\n      0.021709542721509933,\n      -0.033181775361299515,\n      -0.065473772585392,\n      0.013775011524558067,\n      0.057896677404642105,\n      -0.12205051630735397,\n      -0.045818179845809937,\n      0.08165685832500458,\n      0.051835183054208755,\n      -0.13697180151939392,\n      -0.08415184915065765,\n      0.003111282829195261,\n      0.058965642005205154,\n      0.07781064510345459,\n      -0.04123830422759056,\n      -0.11142409592866898,\n      -0.02093670144677162,\n      0.0036500797141343355,\n      0.031186014413833618,\n      0.032596003264188766,\n      0.07736494392156601,\n      -0.10278848558664322,\n      0.004825233947485685,\n      0.13246367871761322,\n      -0.1223367229104042,\n      0.10354730486869812,\n      0.0047033680602908134,\n      -0.029910007491707802,\n      -0.04272313043475151,\n      -0.05614279583096504,\n      -0.0070968815125525,\n      0.045937519520521164,\n      -0.049448445439338684,\n      -0.0399617999792099,\n      -0.035865768790245056,\n      0.06583668291568756,\n      0.032915420830249786,\n      -0.023906558752059937,\n      0.0581648088991642,\n      -0.05155413597822189,\n      0.037089958786964417,\n      0.06717723608016968,\n      0.04771121218800545,\n      0.07598928362131119,\n      -0.06560973078012466,\n      0.007812727242708206,\n      0.0025097904726862907,\n      -0.0676758885383606,\n      0.059846729040145874,\n      0.05781743675470352,\n      -0.04666652902960777,\n      -0.06212887167930603,\n      0.07950429618358612,\n      -0.07443252205848694,\n      -0.117244653403759,\n      0.02497818134725094,\n      0.11217424273490906,\n      -0.028021518141031265,\n      0.053794413805007935,\n      -0.0011503567220643163,\n      -0.10647711157798767,\n      0.016686275601387024,\n      0.014647798612713814,\n      0.06884153187274933,\n      0.01864519901573658,\n      0.06975090503692627,\n      0.032929740846157074,\n      -0.007738005835562944,\n      -0.2734231948852539,\n      -0.013186406344175339,\n      -0.005526951048523188,\n      0.04550483450293541,\n      0.05061553791165352,\n      0.1627579629421234,\n      0.044155240058898926,\n      0.010297378525137901,\n      0.005018899217247963,\n      -0.03401026129722595\n    ],\n    [\n      0.04214869439601898,\n      -0.06584550440311432,\n      -0.05062231421470642,\n      -0.06693464517593384,\n      0.009843830019235611,\n      0.09666416049003601,\n      0.040040839463472366,\n      0.14167919754981995,\n      -0.0010138910729438066,\n      0.07612711191177368,\n      0.10674113035202026,\n      -0.04768931865692139,\n      0.0379861518740654,\n      0.012881810776889324,\n      0.00038181175477802753,\n      -0.029956277459859848,\n      0.05954688414931297,\n      0.02355748787522316,\n      -0.04198111221194267,\n      0.09987030178308487,\n      -0.072983019053936,\n      0.0598815493285656,\n      -0.03239420801401138,\n      0.018009072169661522,\n      -0.08503181487321854,\n      -0.06477811932563782,\n      0.05971395969390869,\n      -0.09706977009773254,\n      0.0955904945731163,\n      -0.12049148231744766,\n      -0.1350204050540924,\n      0.04293758422136307,\n      -0.023249153047800064,\n      0.018200479447841644,\n      -0.011109398677945137,\n      0.05333840847015381,\n      -0.046889565885066986,\n      0.07856317609548569,\n      -0.05766315385699272,\n      0.10951168835163116,\n      0.09652604162693024,\n      -0.024456333369016647,\n      0.018655510619282722,\n      -0.08055194467306137,\n      0.06719715893268585,\n      0.01996753178536892,\n      -0.07668908685445786,\n      -0.03430086374282837,\n      -0.055352583527565,\n      0.0054427157156169415,\n      -0.04828726127743721,\n      -0.07921309769153595,\n      0.009983432479202747,\n      -0.08996685594320297,\n      -0.08012161403894424,\n      -0.024649903178215027,\n      0.016438473016023636,\n      -0.032649967819452286,\n      0.11575488746166229,\n      0.0413707010447979,\n      -0.013704406097531319,\n      -0.008494256064295769,\n      0.11213794350624084,\n      0.03922846168279648,\n      0.0630759671330452,\n      0.015325931832194328,\n      0.023187050595879555,\n      0.08729593455791473,\n      -0.03458140045404434,\n      0.004708608612418175,\n      -0.027294175699353218,\n      0.0012026852928102016,\n      -0.04930917173624039,\n      -0.037967268377542496,\n      -0.03797464445233345,\n      -0.0016367724165320396,\n      0.01625199057161808,\n      -0.04591949284076691,\n      -0.1104198768734932,\n      0.03233784809708595,\n      -0.08133567869663239,\n      0.06375090777873993,\n      -0.08347911387681961,\n      0.02560698613524437,\n      -0.010266410186886787,\n      -0.05980781093239784,\n      0.037482790648937225,\n      0.010021618567407131,\n      -0.026852479204535484,\n      0.05640983209013939,\n      -0.03595270216464996,\n      0.07825387269258499,\n      0.04735587164759636,\n      0.054628968238830566,\n      -0.048326488584280014,\n      0.0775139182806015,\n      0.05381820723414421,\n      0.03037153370678425,\n      0.03664705157279968,\n      0.07402463257312775,\n      0.01040127594023943,\n      0.08587782829999924,\n      -0.05802195146679878,\n      -0.041120193898677826,\n      -0.031129781156778336,\n      -0.0991566851735115,\n      0.05481495335698128,\n      0.0919310599565506,\n      0.04830864071846008,\n      -0.004873674362897873,\n      -0.04112977534532547,\n      0.004545845557004213,\n      0.06217309832572937,\n      0.021584393456578255,\n      0.03884310647845268,\n      -0.0651467889547348,\n      -0.05268770083785057,\n      -0.017381630837917328,\n      0.03612762317061424,\n      -0.05257362872362137,\n      0.008301449008286,\n      -0.060042381286621094,\n      0.04125189408659935,\n      0.0085104089230299,\n      0.0746915191411972\n    ],\n    [\n      -0.05148330703377724,\n      0.07232766598463058,\n      -0.06887469440698624,\n      0.03995388746261597,\n      0.03244081884622574,\n      -0.024617785587906837,\n      -0.08256039023399353,\n      0.08344753831624985,\n      -0.04112895950675011,\n      0.01713869348168373,\n      -0.09764108061790466,\n      0.0165230855345726,\n      0.021540053188800812,\n      0.0714133158326149,\n      -0.1683030128479004,\n      -0.0027267099358141422,\n      -0.017439333721995354,\n      0.062322210520505905,\n      -0.008457276038825512,\n      0.07495034486055374,\n      0.05491653457283974,\n      0.05818573385477066,\n      0.016135243698954582,\n      0.11534670740365982,\n      0.11418752372264862,\n      0.0012937848223373294,\n      0.022704917937517166,\n      0.021210797131061554,\n      -0.05379844456911087,\n      -0.09834050387144089,\n      0.03382156044244766,\n      -0.0973980724811554,\n      0.058825280517339706,\n      0.09244756400585175,\n      0.0352286733686924,\n      0.15782864391803741,\n      0.0066764457151293755,\n      -0.04457330331206322,\n      -0.012726487591862679,\n      0.043859031051397324,\n      -0.01571768708527088,\n      -0.10561176389455795,\n      -0.030917229130864143,\n      0.06739329546689987,\n      -0.041621096432209015,\n      0.11747770011425018,\n      -0.0011300460901111364,\n      0.17136770486831665,\n      0.07735467702150345,\n      0.12686952948570251,\n      0.009008710272610188,\n      0.020157113671302795,\n      -0.07455980777740479,\n      0.13697753846645355,\n      -0.019978811964392662,\n      -0.0296494010835886,\n      0.07592958956956863,\n      -0.05612603947520256,\n      0.03103020042181015,\n      -0.12168272584676743,\n      -0.06489687412977219,\n      -0.011716127395629883,\n      0.13736815750598907,\n      0.02880939282476902,\n      -0.030758215114474297,\n      0.02350468374788761,\n      0.038788966834545135,\n      0.02477801777422428,\n      -0.03618668392300606,\n      0.09452634304761887,\n      0.033646468073129654,\n      -0.0038918100763112307,\n      0.017219996079802513,\n      -0.017333984375,\n      -0.03896623104810715,\n      -0.11442707479000092,\n      -0.049567628651857376,\n      -0.027611851692199707,\n      0.09337744116783142,\n      0.09028834104537964,\n      0.0652429535984993,\n      0.043453969061374664,\n      0.011119836010038853,\n      -0.04964780807495117,\n      0.0017293865093961358,\n      0.02163860946893692,\n      0.05908920243382454,\n      0.005978199653327465,\n      -0.01474142074584961,\n      0.003384837880730629,\n      0.09445878118276596,\n      -0.001506928587332368,\n      0.06855014711618423,\n      0.012569032609462738,\n      0.036710742861032486,\n      0.0822514072060585,\n      -0.00870161410421133,\n      0.03331679478287697,\n      0.12706956267356873,\n      0.08131347596645355,\n      -0.00675049377605319,\n      0.056687384843826294,\n      -0.056537553668022156,\n      0.030975837260484695,\n      0.02213701233267784,\n      -0.07300133258104324,\n      -0.04684885963797569,\n      0.1637236624956131,\n      -0.08569119870662689,\n      0.012795508839190006,\n      -0.11440132558345795,\n      0.018956612795591354,\n      -0.04103627800941467,\n      -0.03819555416703224,\n      0.06401845812797546,\n      0.10066727548837662,\n      0.02075846679508686,\n      -0.034465089440345764,\n      0.06293829530477524,\n      0.09986840188503265,\n      -0.162654310464859,\n      -0.0652451440691948,\n      0.08614212274551392,\n      0.09294233471155167,\n      -0.06745796650648117\n    ],\n    [\n      -0.05528494343161583,\n      -0.06483688205480576,\n      0.007759074680507183,\n      -0.12858368456363678,\n      -0.05874452739953995,\n      0.08649242669343948,\n      0.11046294867992401,\n      0.056871674954891205,\n      0.03019481711089611,\n      0.06989365816116333,\n      -0.07703746855258942,\n      -0.12838946282863617,\n      -0.09632309526205063,\n      -0.25983577966690063,\n      0.03390737622976303,\n      0.016312971711158752,\n      0.12098311632871628,\n      0.08740916848182678,\n      0.09597212076187134,\n      -0.12934185564517975,\n      0.050403960049152374,\n      -0.045020777732133865,\n      0.07755470275878906,\n      0.014549137093126774,\n      0.113253653049469,\n      0.016849428415298462,\n      0.013309753499925137,\n      -0.19202803075313568,\n      -0.04242674261331558,\n      -0.004755191504955292,\n      0.0782465785741806,\n      -0.05501747131347656,\n      -0.04016153886914253,\n      0.011525343172252178,\n      -0.07058121263980865,\n      0.03079724870622158,\n      -0.04157313331961632,\n      0.026590270921587944,\n      -0.035736486315727234,\n      0.008859537541866302,\n      0.1227153092622757,\n      -0.001170353963971138,\n      0.1191641241312027,\n      0.0261347908526659,\n      0.024760238826274872,\n      0.09308203309774399,\n      -0.112623430788517,\n      0.002073705429211259,\n      0.013295881450176239,\n      0.08679182827472687,\n      -0.0029358018655329943,\n      0.04858504235744476,\n      0.11681519448757172,\n      0.10954252630472183,\n      0.07394915074110031,\n      0.10786421597003937,\n      0.12362933903932571,\n      0.040164969861507416,\n      0.12374056875705719,\n      0.07069473713636398,\n      -0.002851720666512847,\n      -0.010042907670140266,\n      0.08797246217727661,\n      0.07611062377691269,\n      0.044841062277555466,\n      0.18512314558029175,\n      0.09023614227771759,\n      0.15751931071281433,\n      0.07201021164655685,\n      -0.05068640038371086,\n      -0.014945301227271557,\n      0.08682510256767273,\n      -0.017965378239750862,\n      -0.03963283449411392,\n      -0.06057395413517952,\n      -0.07495417445898056,\n      0.035294026136398315,\n      0.024098552763462067,\n      -0.057971809059381485,\n      0.07416093349456787,\n      0.02629896253347397,\n      -0.04662264510989189,\n      0.033950112760066986,\n      0.08367183059453964,\n      -0.17007976770401,\n      -0.12372205406427383,\n      0.026711435988545418,\n      0.10715016722679138,\n      -0.11768238246440887,\n      0.13655045628547668,\n      -0.00867894571274519,\n      -0.0530087947845459,\n      -0.04346403852105141,\n      -0.031845152378082275,\n      -0.039499230682849884,\n      -0.06590654700994492,\n      -0.010579327121376991,\n      0.08179619908332825,\n      0.14768864214420319,\n      0.067537322640419,\n      -0.07077840715646744,\n      0.0002597586717456579,\n      0.04867694526910782,\n      -0.0047456384636461735,\n      0.0533716157078743,\n      0.008082212880253792,\n      3.977969754487276e-05,\n      -0.06740822643041611,\n      -0.12565669417381287,\n      0.07704852521419525,\n      -0.12413415312767029,\n      -0.09490817040205002,\n      -0.030618924647569656,\n      0.12632639706134796,\n      -0.019088778644800186,\n      -0.09628471732139587,\n      0.10537530481815338,\n      -0.030042380094528198,\n      0.032634053379297256,\n      -0.1336866319179535,\n      0.11435366421937943,\n      -0.07862319052219391,\n      0.02705245092511177,\n      -0.02914881333708763,\n      -0.03147830814123154\n    ],\n    [\n      0.07874619960784912,\n      -0.08039449155330658,\n      -0.0581076480448246,\n      -0.003145985072478652,\n      -0.13976100087165833,\n      -0.00047303110477514565,\n      0.06457952409982681,\n      -0.006304840557277203,\n      -0.13940715789794922,\n      0.013602404855191708,\n      0.0680989921092987,\n      0.035043295472860336,\n      0.10260314494371414,\n      -0.01917491853237152,\n      -0.058940477669239044,\n      -0.12803687155246735,\n      -0.11470705270767212,\n      0.014622651971876621,\n      0.08537230640649796,\n      0.10802061855792999,\n      0.015251406468451023,\n      0.02205648459494114,\n      0.07637505233287811,\n      0.0810759887099266,\n      -0.013600094243884087,\n      -0.02778519131243229,\n      0.007871599867939949,\n      0.009848145768046379,\n      0.07187842577695847,\n      -0.1872728317975998,\n      -0.04539009928703308,\n      -0.0655631348490715,\n      -0.018600933253765106,\n      -0.03612681105732918,\n      0.05369670316576958,\n      0.09745815396308899,\n      -0.025769243016839027,\n      -0.030890772119164467,\n      -0.056848615407943726,\n      0.03538553789258003,\n      0.049620311707258224,\n      -0.09253671020269394,\n      -0.03374679386615753,\n      0.07649681717157364,\n      0.10315736383199692,\n      -0.08107344806194305,\n      -0.06896359473466873,\n      0.013386488892138004,\n      0.06560604274272919,\n      -0.10568166524171829,\n      -0.12006867676973343,\n      0.0153055964037776,\n      0.05797751992940903,\n      0.0001332263636868447,\n      -0.011186475865542889,\n      -0.07465170323848724,\n      0.05616016313433647,\n      -0.019679781049489975,\n      0.048735059797763824,\n      -0.06879892945289612,\n      0.03475280851125717,\n      -0.008888429030776024,\n      0.06779570877552032,\n      -0.08721377700567245,\n      0.035369642078876495,\n      0.06472378969192505,\n      -0.09754332154989243,\n      -0.05833312124013901,\n      0.06861822307109833,\n      -0.044701945036649704,\n      0.04711068049073219,\n      0.0711851567029953,\n      -0.1134650781750679,\n      0.016969820484519005,\n      0.04059075936675072,\n      -0.06034140661358833,\n      -0.02951863780617714,\n      0.019321966916322708,\n      -0.11668457090854645,\n      0.06069611385464668,\n      -0.018810203298926353,\n      0.03821657970547676,\n      -0.11236114799976349,\n      0.01475378405302763,\n      -0.009621539153158665,\n      -0.07903945446014404,\n      -0.01575571671128273,\n      0.021840021014213562,\n      0.032192446291446686,\n      0.05397184193134308,\n      -0.04957534372806549,\n      -0.08717905730009079,\n      0.035374779254198074,\n      -0.07320640236139297,\n      0.09841131418943405,\n      -0.008182880468666553,\n      -0.047449175268411636,\n      -0.10584522038698196,\n      0.043601904064416885,\n      -0.04503515735268593,\n      -0.034158576279878616,\n      -0.018155869096517563,\n      0.002348701935261488,\n      0.002901257248595357,\n      -0.09307520091533661,\n      -0.02310197427868843,\n      -0.08899364620447159,\n      0.07153088599443436,\n      0.01944827288389206,\n      0.08605585992336273,\n      0.0733184814453125,\n      -0.019617650657892227,\n      -0.03917888551950455,\n      0.015219043008983135,\n      -0.007017839699983597,\n      -0.12653684616088867,\n      -0.08745007961988449,\n      0.019070373848080635,\n      -0.017442846670746803,\n      -0.02961229719221592,\n      -0.08090808987617493,\n      -0.14903630316257477,\n      0.022691089659929276,\n      0.056723497807979584,\n      0.04822862148284912\n    ],\n    [\n      0.11340031027793884,\n      0.05497550591826439,\n      0.16742931306362152,\n      0.10555577278137207,\n      0.05283980816602707,\n      -0.14834649860858917,\n      -0.035117968916893005,\n      -0.13178278505802155,\n      0.18744303286075592,\n      0.008559211157262325,\n      0.09607154875993729,\n      -0.033865541219711304,\n      0.09979865700006485,\n      0.14469172060489655,\n      -0.08486846834421158,\n      0.013501600362360477,\n      -0.07880740612745285,\n      -0.0814308300614357,\n      -0.06350965797901154,\n      0.055224012583494186,\n      0.06997694820165634,\n      -0.030900081619620323,\n      -0.1414581835269928,\n      0.0644599050283432,\n      -0.07432952523231506,\n      0.00298078591004014,\n      -0.10659316927194595,\n      0.0820108950138092,\n      -0.19617599248886108,\n      0.11259561032056808,\n      0.07637529075145721,\n      -0.02240881137549877,\n      -0.08513900637626648,\n      -0.03210483491420746,\n      0.010933584533631802,\n      0.000936277094297111,\n      -0.0030925592873245478,\n      0.050025638192892075,\n      0.08315488696098328,\n      0.027693985030055046,\n      -0.1645721197128296,\n      -0.09846532344818115,\n      -0.11446763575077057,\n      -0.06745576113462448,\n      0.0074419183656573296,\n      -0.0501834973692894,\n      -0.0640082061290741,\n      -0.07549318671226501,\n      -0.03015923500061035,\n      -0.0695294514298439,\n      -0.014669925905764103,\n      -0.15310047566890717,\n      -0.20453393459320068,\n      -0.09201107174158096,\n      -0.08246012032032013,\n      -0.09426809102296829,\n      -0.10234677046537399,\n      -0.04605131968855858,\n      -0.13514794409275055,\n      -0.05898844823241234,\n      -0.06530557572841644,\n      -0.14808320999145508,\n      -0.0787842646241188,\n      0.00634270953014493,\n      -0.02452094294130802,\n      -0.01025039330124855,\n      -0.03730759397149086,\n      -0.12651953101158142,\n      -0.02273496612906456,\n      -0.17031557857990265,\n      -0.1286683827638626,\n      -0.13870884478092194,\n      -0.02790144644677639,\n      0.041117776185274124,\n      -0.010144740343093872,\n      0.06871358305215836,\n      0.08094923198223114,\n      0.05604574456810951,\n      -0.02089185081422329,\n      -0.057292889803647995,\n      0.028505515307188034,\n      0.06806471943855286,\n      -0.13667245209217072,\n      -0.04290180653333664,\n      -0.027925509959459305,\n      0.0867595449090004,\n      0.0835508182644844,\n      0.05751104652881622,\n      -0.11735372245311737,\n      0.0139821982011199,\n      0.05773942172527313,\n      0.05723147466778755,\n      -0.015654031187295914,\n      -0.034738894551992416,\n      -0.042022254317998886,\n      -0.031365767121315,\n      0.016389954835176468,\n      -0.03536638244986534,\n      0.10462291538715363,\n      -0.06326314061880112,\n      -0.07120006531476974,\n      -0.03905370086431503,\n      0.005860143806785345,\n      0.06511709839105606,\n      -0.08329561352729797,\n      0.07757436484098434,\n      -0.062137421220541,\n      -0.002396711613982916,\n      0.13779599964618683,\n      -0.03452349826693535,\n      0.10738124698400497,\n      0.08512651175260544,\n      0.01247238740324974,\n      0.0011049024760723114,\n      -0.07008207589387894,\n      0.10300981998443604,\n      -0.19353674352169037,\n      -0.024598363786935806,\n      0.02374555729329586,\n      0.058738820254802704,\n      -0.14956195652484894,\n      0.10876962542533875,\n      -0.03866300731897354,\n      -0.04310418665409088,\n      0.0897989273071289\n    ],\n    [\n      0.007980850525200367,\n      0.06226126104593277,\n      0.03132523596286774,\n      0.07148157805204391,\n      0.1273348182439804,\n      -0.12642823159694672,\n      -0.05198159068822861,\n      -0.14667250216007233,\n      0.144379124045372,\n      0.04996538534760475,\n      -0.022120194509625435,\n      0.028489094227552414,\n      0.015012023039162159,\n      0.14644820988178253,\n      0.018007706850767136,\n      -0.12347046285867691,\n      -0.017155664041638374,\n      -0.1780516505241394,\n      -0.09728250652551651,\n      -0.08429712057113647,\n      0.05263811722397804,\n      0.04667409881949425,\n      -0.03960375860333443,\n      0.03204791247844696,\n      -0.06579792499542236,\n      -0.07655864953994751,\n      -0.09371846169233322,\n      0.12406505644321442,\n      -0.01514406781643629,\n      0.23250006139278412,\n      0.008684499189257622,\n      0.02160729095339775,\n      -0.03423267975449562,\n      -0.044565290212631226,\n      0.061043981462717056,\n      0.050351228564977646,\n      0.05201222375035286,\n      -0.09481556713581085,\n      0.024632669985294342,\n      -0.03614475578069687,\n      -0.1702834516763687,\n      0.010072499513626099,\n      -0.09340466558933258,\n      -0.10808094590902328,\n      0.0802745446562767,\n      -0.07039934396743774,\n      0.08389386534690857,\n      -0.10326443612575531,\n      -0.06250163912773132,\n      -0.07542381435632706,\n      -0.046696145087480545,\n      -0.025307564064860344,\n      -0.02060818485915661,\n      -0.05903852730989456,\n      -0.1154734417796135,\n      -0.07944740355014801,\n      -0.10748936235904694,\n      -0.04258674383163452,\n      -0.09765402227640152,\n      -0.13958536088466644,\n      -0.06967199593782425,\n      0.0020571269560605288,\n      -0.1845049262046814,\n      -0.08609875291585922,\n      0.004933775402605534,\n      -0.022658366709947586,\n      -0.03436518460512161,\n      -0.07323963195085526,\n      -0.1081063523888588,\n      -0.02988625317811966,\n      0.01595706306397915,\n      -0.14888067543506622,\n      0.02971459925174713,\n      0.08432913571596146,\n      0.016634831205010414,\n      0.08674284815788269,\n      0.07467099279165268,\n      -0.02874579094350338,\n      0.12492317706346512,\n      -0.09552260488271713,\n      0.05398203805088997,\n      -0.003663347801193595,\n      -0.02198740467429161,\n      0.005993480794131756,\n      0.09690400213003159,\n      0.08727472275495529,\n      -0.03397320210933685,\n      -0.02652445062994957,\n      0.01962684653699398,\n      -0.02543788030743599,\n      0.07098817080259323,\n      -0.02216303162276745,\n      0.015952829271554947,\n      0.001024188008159399,\n      0.021873511373996735,\n      -0.0024388290476053953,\n      0.040141116827726364,\n      -0.01915064826607704,\n      -0.07802120596170425,\n      -0.10633716732263565,\n      0.045139871537685394,\n      -0.020730730146169662,\n      0.05364488065242767,\n      0.04274466261267662,\n      -0.053346313536167145,\n      0.060801416635513306,\n      0.04374706372618675,\n      -0.040853213518857956,\n      0.10974834859371185,\n      -0.02954653836786747,\n      0.14850591123104095,\n      0.07817674428224564,\n      -0.00565265491604805,\n      -0.11590667068958282,\n      -0.026173898950219154,\n      0.027920881286263466,\n      -0.10036686062812805,\n      0.08016183227300644,\n      -0.1403540074825287,\n      0.11491992324590683,\n      -0.07326419651508331,\n      0.1401391178369522,\n      -0.06713130325078964,\n      0.07116682827472687,\n      -0.046119607985019684\n    ],\n    [\n      0.0436713807284832,\n      -0.0174289308488369,\n      -0.00333846895955503,\n      0.018069269135594368,\n      -0.018431974574923515,\n      0.04745163023471832,\n      0.03906562179327011,\n      0.12356697022914886,\n      -0.05246748402714729,\n      -0.0802408829331398,\n      -0.010905304923653603,\n      0.05130751058459282,\n      0.08740245550870895,\n      -0.19083599746227264,\n      -0.17539338767528534,\n      0.12948447465896606,\n      -0.026813389733433723,\n      0.11139759421348572,\n      0.1135895699262619,\n      0.07495100796222687,\n      0.09545540809631348,\n      -0.04387729987502098,\n      0.0068021961487829685,\n      0.026199623942375183,\n      -0.01370445266366005,\n      -0.024819033220410347,\n      0.1768902689218521,\n      -0.030359812080860138,\n      0.06853637844324112,\n      -0.2555534839630127,\n      0.05348778888583183,\n      -0.07747284322977066,\n      7.19941162969917e-05,\n      -0.03949369490146637,\n      0.029317786917090416,\n      0.010753032751381397,\n      -0.03783436864614487,\n      -0.025038644671440125,\n      0.019933871924877167,\n      0.0868385061621666,\n      0.009782959707081318,\n      0.07415857166051865,\n      0.01811571978032589,\n      -0.05730335786938667,\n      0.10788360983133316,\n      0.06020037829875946,\n      -0.020444821566343307,\n      -0.04954146966338158,\n      0.017981069162487984,\n      -0.006791621912270784,\n      0.0022492578718811274,\n      -0.06133529171347618,\n      -0.009092605672776699,\n      0.08657889813184738,\n      0.034119270741939545,\n      0.016811540350317955,\n      -0.026490243151783943,\n      0.051334403455257416,\n      0.08616703003644943,\n      -0.08049749583005905,\n      0.03716385364532471,\n      -0.08502507209777832,\n      -0.006632291246205568,\n      0.036128461360931396,\n      0.026107748970389366,\n      0.0409451462328434,\n      -0.10446814447641373,\n      0.0013115284964442253,\n      -0.03975795954465866,\n      0.01174011267721653,\n      -0.002172306180000305,\n      0.034687139093875885,\n      -0.062463875859975815,\n      0.012658455409109592,\n      0.01651867851614952,\n      -0.07561001926660538,\n      -0.004111603368073702,\n      0.08214125782251358,\n      -0.14211297035217285,\n      -0.02677369862794876,\n      0.008937464095652103,\n      0.034952856600284576,\n      -0.0023355779703706503,\n      0.025035806000232697,\n      -0.023113392293453217,\n      0.006144861690700054,\n      -0.09107917547225952,\n      -0.04793233424425125,\n      0.005135543644428253,\n      -0.05156736075878143,\n      0.044398073107004166,\n      -0.07068038731813431,\n      -0.01672559604048729,\n      -0.05199727416038513,\n      -0.05094003304839134,\n      -0.07124706357717514,\n      0.0215623639523983,\n      0.013143589720129967,\n      0.03673752769827843,\n      -0.039655979722738266,\n      -0.05586562678217888,\n      0.02105029672384262,\n      0.0220970306545496,\n      -0.10522031038999557,\n      0.057884346693754196,\n      -0.1055828183889389,\n      0.01761731691658497,\n      0.09157675504684448,\n      0.030138682574033737,\n      0.08639197051525116,\n      0.013520237058401108,\n      -0.05448729917407036,\n      -0.021276284009218216,\n      0.04744737595319748,\n      0.0720532238483429,\n      0.01111858431249857,\n      0.006476570852100849,\n      0.04510616138577461,\n      -0.06326176971197128,\n      -0.12552936375141144,\n      -0.15580902993679047,\n      -0.0747109204530716,\n      -0.04991404339671135,\n      -0.09025992453098297,\n      -0.02647811360657215\n    ],\n    [\n      0.0732126235961914,\n      0.11818024516105652,\n      -0.10520203411579132,\n      -0.08481886237859726,\n      0.040088288486003876,\n      -0.04362108185887337,\n      0.00111288670450449,\n      0.013827163726091385,\n      0.10175951570272446,\n      -0.007517819758504629,\n      0.13337752223014832,\n      -0.0319196954369545,\n      -0.06346675753593445,\n      0.1444518119096756,\n      0.026609210297465324,\n      -0.09414782375097275,\n      0.05719130113720894,\n      -0.06847437471151352,\n      0.024403689429163933,\n      0.10098287463188171,\n      0.0544755645096302,\n      0.007170473225414753,\n      -0.160747691988945,\n      -0.09122646600008011,\n      -0.09867000579833984,\n      -0.016219809651374817,\n      -0.1273050308227539,\n      0.0838017612695694,\n      0.01528869103640318,\n      0.19258522987365723,\n      -0.07078329473733902,\n      0.040292512625455856,\n      0.04926954209804535,\n      0.020539753139019012,\n      0.07345052808523178,\n      -0.018281878903508186,\n      -0.03613445907831192,\n      0.00333860469982028,\n      0.02701665833592415,\n      0.0582071952521801,\n      -0.008658790960907936,\n      -0.0004925559624098241,\n      -0.007179582957178354,\n      -0.09261135011911392,\n      0.012596587650477886,\n      -0.11569565534591675,\n      0.07345070689916611,\n      0.008875830098986626,\n      -0.09683501720428467,\n      -0.09950011223554611,\n      -0.050942473113536835,\n      0.027324087917804718,\n      0.09096689522266388,\n      -0.029920542612671852,\n      -0.11406487971544266,\n      -0.14650577306747437,\n      -0.16293761134147644,\n      -0.11273455619812012,\n      -0.06994200497865677,\n      0.05545530095696449,\n      -0.0979389026761055,\n      -0.020865315571427345,\n      -0.16297122836112976,\n      -0.011226157657802105,\n      0.002436957089230418,\n      -0.0648297518491745,\n      0.041553858667612076,\n      0.06951180845499039,\n      0.11651987582445145,\n      0.09251832216978073,\n      0.042744964361190796,\n      0.03744763880968094,\n      0.008098435588181019,\n      0.04423859342932701,\n      0.1230633482336998,\n      0.09376206248998642,\n      0.030285103246569633,\n      -0.08184973895549774,\n      -0.04648419842123985,\n      -0.0844985693693161,\n      0.12264696508646011,\n      -0.09159361571073532,\n      -0.09080332517623901,\n      0.03444255143404007,\n      -0.03895603492856026,\n      0.1312270164489746,\n      -0.0445258691906929,\n      -0.023189745843410492,\n      0.02673908695578575,\n      -0.019758321344852448,\n      0.01809746026992798,\n      -0.03537232056260109,\n      0.02273944579064846,\n      0.045018453150987625,\n      0.036186836659908295,\n      0.10449378192424774,\n      -0.037894733250141144,\n      0.001596994698047638,\n      -0.0107507836073637,\n      -0.030307861045002937,\n      -0.03444567322731018,\n      -0.13607510924339294,\n      0.10397034138441086,\n      0.07465826719999313,\n      -0.04365130886435509,\n      0.010530280880630016,\n      0.059545814990997314,\n      0.10562201589345932,\n      0.047936443239450455,\n      -0.03698224946856499,\n      0.1270456463098526,\n      0.07878325134515762,\n      0.010298414155840874,\n      -0.020687468349933624,\n      -0.11843035370111465,\n      0.11615671217441559,\n      -0.11445006728172302,\n      -0.031338952481746674,\n      -0.04990925267338753,\n      0.1383746862411499,\n      0.09664242714643478,\n      0.15072962641716003,\n      0.019257253035902977,\n      0.07980385422706604,\n      0.009821765124797821\n    ],\n    [\n      0.07911240309476852,\n      0.05117233097553253,\n      0.03093487024307251,\n      -0.0388810858130455,\n      0.06810948997735977,\n      0.03077125921845436,\n      -0.017987951636314392,\n      0.013918709009885788,\n      -0.028539186343550682,\n      -0.004668083041906357,\n      0.09911814332008362,\n      -0.02738173119723797,\n      -0.00659019174054265,\n      -0.15014566481113434,\n      -0.19258876144886017,\n      0.037354812026023865,\n      0.03687572106719017,\n      -0.050643812865018845,\n      -0.004491277504712343,\n      0.10015419870615005,\n      -0.029738038778305054,\n      -0.053300805389881134,\n      -0.014539893716573715,\n      0.07428397983312607,\n      -0.07773234695196152,\n      -0.06754869222640991,\n      0.0708502009510994,\n      -0.05242067202925682,\n      0.09788428246974945,\n      -0.15418830513954163,\n      -0.043950799852609634,\n      -0.06883668899536133,\n      -0.08556394279003143,\n      -0.0005704670329578221,\n      0.021781334653496742,\n      0.05125519260764122,\n      -0.06100412830710411,\n      0.03332469239830971,\n      0.02836008556187153,\n      0.08420711755752563,\n      -0.00946484412997961,\n      -0.02919754758477211,\n      0.048711806535720825,\n      -0.0719304159283638,\n      0.022871263325214386,\n      0.06357061862945557,\n      0.09434429556131363,\n      -0.02505442127585411,\n      0.06950053572654724,\n      0.07693156599998474,\n      0.05384029820561409,\n      0.005069389473646879,\n      0.035277049988508224,\n      0.024557217955589294,\n      0.04662927985191345,\n      -0.09229349344968796,\n      0.09209325909614563,\n      0.002127137267962098,\n      -0.055617354810237885,\n      -0.037937674671411514,\n      0.01746692880988121,\n      0.06903442740440369,\n      -0.043600570410490036,\n      0.04737836867570877,\n      0.08529523760080338,\n      0.06492607295513153,\n      0.007596577983349562,\n      0.06636284291744232,\n      0.05484867841005325,\n      -0.13788072764873505,\n      -0.0002324549714103341,\n      0.04730217158794403,\n      -0.037497684359550476,\n      -0.08250238746404648,\n      -0.0710594430565834,\n      0.048555146902799606,\n      -0.023855475708842278,\n      -0.05490342155098915,\n      -0.0336715430021286,\n      -0.005705189425498247,\n      -0.1104489415884018,\n      0.055448584258556366,\n      0.04024819657206535,\n      -0.06395478546619415,\n      -0.08024616539478302,\n      -0.039142340421676636,\n      0.009702730923891068,\n      0.07961642742156982,\n      -0.013824986293911934,\n      -0.05623703449964523,\n      -0.050358667969703674,\n      -0.06267181783914566,\n      0.09457184374332428,\n      0.07733255624771118,\n      0.045234210789203644,\n      0.032397784292697906,\n      0.023829948157072067,\n      0.08396090567111969,\n      0.048096511512994766,\n      -0.04792143031954765,\n      -0.07762162387371063,\n      0.03089534305036068,\n      -0.049618449062108994,\n      0.043474409729242325,\n      0.02605443261563778,\n      0.07824896275997162,\n      0.044148609042167664,\n      0.01554543524980545,\n      -0.038723573088645935,\n      0.001219101483002305,\n      0.004789057187736034,\n      0.058435868471860886,\n      -0.015484263189136982,\n      0.06861502677202225,\n      0.07067184150218964,\n      -0.10858392715454102,\n      0.035036277025938034,\n      0.0364140085875988,\n      0.05881889536976814,\n      -0.09107646346092224,\n      -0.12348729372024536,\n      -0.1719561368227005,\n      0.11975589394569397,\n      -0.006617595907300711,\n      0.07025892287492752\n    ],\n    [\n      0.07220946252346039,\n      0.021747319027781487,\n      -0.03278159722685814,\n      -0.08899607509374619,\n      -0.07877659052610397,\n      -0.04195123910903931,\n      0.12638193368911743,\n      0.12775520980358124,\n      -0.1385863870382309,\n      -0.05639699101448059,\n      -0.006570059806108475,\n      -0.04861121624708176,\n      0.09581854939460754,\n      -0.183163583278656,\n      -0.06846504658460617,\n      -0.07724014669656754,\n      -0.02998366393148899,\n      0.08758296817541122,\n      0.030922342091798782,\n      0.05232717841863632,\n      0.00831620953977108,\n      -0.009033327922224998,\n      0.008267671801149845,\n      -0.032451413571834564,\n      -0.07877948135137558,\n      -0.04439469054341316,\n      0.15527723729610443,\n      -0.037001464515924454,\n      0.0035876049660146236,\n      -0.2133200764656067,\n      0.031606290489435196,\n      -0.08091145008802414,\n      -0.03819402679800987,\n      0.01734522357583046,\n      -0.010997934266924858,\n      0.07168842107057571,\n      -0.006588699296116829,\n      0.10799679160118103,\n      -0.00162860460113734,\n      -0.024055492132902145,\n      0.09458868950605392,\n      0.06743816286325455,\n      0.07806695997714996,\n      0.04219765588641167,\n      0.023949841037392616,\n      0.009859231300652027,\n      0.04319760948419571,\n      0.018445946276187897,\n      -0.017107980325818062,\n      -0.026444952934980392,\n      -0.09328507632017136,\n      0.05609610676765442,\n      -0.0007529049180448055,\n      -0.003951147198677063,\n      -0.004144368693232536,\n      0.06455996632575989,\n      -0.035403575748205185,\n      -0.0487801730632782,\n      -0.03624853864312172,\n      0.0019881250336766243,\n      0.05482654273509979,\n      -0.061138708144426346,\n      -0.040036916732788086,\n      -0.050644565373659134,\n      0.04033506289124489,\n      0.014316613785922527,\n      -0.08170565217733383,\n      -0.030782047659158707,\n      -0.0317610502243042,\n      -0.034529753029346466,\n      -0.03789889067411423,\n      -0.04565613716840744,\n      -0.149491548538208,\n      -0.08994879573583603,\n      -0.06078355759382248,\n      -0.042393069714307785,\n      -0.09128779917955399,\n      0.0443524532020092,\n      -0.048471882939338684,\n      -0.021234260872006416,\n      -0.023222384974360466,\n      -0.005268710665404797,\n      0.0461021363735199,\n      -0.0821390300989151,\n      0.07666093856096268,\n      -0.1330007165670395,\n      0.0680377259850502,\n      0.028057407587766647,\n      -0.037348661571741104,\n      -0.044076427817344666,\n      -6.231025326997042e-05,\n      -0.08186198770999908,\n      -0.04958323761820793,\n      0.06356022506952286,\n      -0.08467386662960052,\n      0.09133122116327286,\n      0.09099925309419632,\n      -0.0720696821808815,\n      0.017605923116207123,\n      -0.04376509413123131,\n      0.03035760298371315,\n      0.043783243745565414,\n      0.11024625599384308,\n      0.0842522382736206,\n      -0.029714660719037056,\n      -0.07876298576593399,\n      0.07463996857404709,\n      -0.01465167012065649,\n      0.11993112415075302,\n      0.1633988469839096,\n      -0.04094104841351509,\n      -0.08894602954387665,\n      0.07126425206661224,\n      0.0395464189350605,\n      0.0715487077832222,\n      -0.012916198931634426,\n      -0.07569550722837448,\n      0.03600023314356804,\n      0.07803188264369965,\n      -0.11887438595294952,\n      -0.01622404344379902,\n      0.007090602070093155,\n      0.0037496802397072315,\n      -0.064547598361969,\n      0.069722980260849\n    ],\n    [\n      0.04210694506764412,\n      -0.1565835028886795,\n      0.0006683572428300977,\n      -0.02087257243692875,\n      -0.10476921498775482,\n      0.03615663945674896,\n      0.09961416572332382,\n      0.13843758404254913,\n      -0.11153202503919601,\n      -0.06410790234804153,\n      0.07050252705812454,\n      -0.07179524004459381,\n      0.03253253921866417,\n      -0.06482650339603424,\n      -0.03454369679093361,\n      0.07980643212795258,\n      -0.08940323442220688,\n      0.03295697271823883,\n      -0.005153990816324949,\n      0.030877796933054924,\n      -0.04805558919906616,\n      -0.08866649866104126,\n      -0.0673435851931572,\n      0.040128737688064575,\n      -0.04819187894463539,\n      -0.10175459831953049,\n      -0.052834417670965195,\n      -0.015575026161968708,\n      -0.001380066853016615,\n      -0.23323720693588257,\n      -0.06175819784402847,\n      0.047770194709300995,\n      -0.09786869585514069,\n      0.0017068565357476473,\n      0.027627043426036835,\n      -0.0759744942188263,\n      0.038835249841213226,\n      -0.0035762665793299675,\n      0.036343082785606384,\n      0.075205497443676,\n      0.038333773612976074,\n      0.019901374354958534,\n      0.07649727910757065,\n      -0.035560112446546555,\n      0.07902202755212784,\n      -0.07559722661972046,\n      0.016057416796684265,\n      -0.009324431419372559,\n      0.06707170605659485,\n      -0.10286843031644821,\n      -0.08748240023851395,\n      -0.011767376214265823,\n      -0.003207332454621792,\n      0.0011946819722652435,\n      -0.07564835250377655,\n      -0.06836049258708954,\n      -0.09734765440225601,\n      0.024542732164263725,\n      -0.011819712817668915,\n      0.015940217301249504,\n      -0.01206742599606514,\n      -0.10851848125457764,\n      -0.07765041291713715,\n      -0.1465793401002884,\n      0.063997782766819,\n      -0.03590751811861992,\n      -0.052414193749427795,\n      0.05982784554362297,\n      -0.14412806928157806,\n      -0.025839442387223244,\n      -0.11340513825416565,\n      0.03515848517417908,\n      -0.14790813624858856,\n      -0.04714719578623772,\n      -0.024556366726756096,\n      0.03420807048678398,\n      -0.09284073114395142,\n      0.054241206496953964,\n      -0.08583495765924454,\n      -0.04882180318236351,\n      -0.019487660378217697,\n      -0.0836758092045784,\n      0.033654604107141495,\n      -0.04405246302485466,\n      -0.05537017807364464,\n      -0.130379781126976,\n      -0.08026011288166046,\n      -0.10103536397218704,\n      0.00010074093006551266,\n      -0.10866931825876236,\n      0.0303962305188179,\n      0.08898824453353882,\n      -0.04270148277282715,\n      -0.062290310859680176,\n      -0.08977613598108292,\n      0.011670364998281002,\n      -0.055534571409225464,\n      -0.014372129924595356,\n      -0.08805236965417862,\n      -0.05398239940404892,\n      0.10700011998414993,\n      -0.03070702590048313,\n      0.02808319218456745,\n      0.10522889345884323,\n      -0.02371739223599434,\n      0.055300164967775345,\n      -0.11465740948915482,\n      -0.0017717841546982527,\n      0.16112668812274933,\n      0.059505023062229156,\n      0.04460936039686203,\n      0.030694127082824707,\n      0.03306569159030914,\n      0.01483918260782957,\n      0.025075281038880348,\n      -0.1877325028181076,\n      -0.21627400815486908,\n      0.010529257357120514,\n      -0.06323406845331192,\n      -0.14132796227931976,\n      -0.022486122325062752,\n      -0.08737127482891083,\n      0.07667549699544907,\n      -0.09444931149482727,\n      0.10428429394960403\n    ],\n    [\n      -0.10762067139148712,\n      -0.15922768414020538,\n      -0.11392398178577423,\n      -0.08197406679391861,\n      -0.02748590148985386,\n      0.0966423824429512,\n      0.13495856523513794,\n      -0.021482016891241074,\n      -0.04181978106498718,\n      -0.0052720834501087666,\n      -0.059360478073358536,\n      -0.07540766149759293,\n      0.06671968847513199,\n      -0.219081312417984,\n      -0.19580329954624176,\n      -0.005818542093038559,\n      -0.1459115892648697,\n      0.03264783322811127,\n      0.02630452625453472,\n      0.06978467106819153,\n      0.03567119687795639,\n      -0.012605534866452217,\n      -0.01900036260485649,\n      0.02815341390669346,\n      -0.08113601058721542,\n      -0.04281099513173103,\n      -0.05412603169679642,\n      -0.11643834412097931,\n      0.018633464351296425,\n      -0.15005366504192352,\n      -0.008961011655628681,\n      0.04401213675737381,\n      -0.09106822311878204,\n      -0.04105880856513977,\n      0.04264535382390022,\n      -0.03643300011754036,\n      0.10186702758073807,\n      0.04860152676701546,\n      -0.031533390283584595,\n      0.018833104521036148,\n      0.052039649337530136,\n      0.03916072100400925,\n      -0.014657886698842049,\n      0.04721662402153015,\n      -0.10379991680383682,\n      0.05533381178975105,\n      -0.03434847295284271,\n      -0.028481438755989075,\n      0.08221883326768875,\n      0.033492933958768845,\n      0.01630110666155815,\n      0.04783432185649872,\n      0.01323087140917778,\n      -0.13276425004005432,\n      -0.14251555502414703,\n      -0.04835066199302673,\n      -0.09458190947771072,\n      0.00958830863237381,\n      0.038072533905506134,\n      0.03084958717226982,\n      -0.04387720674276352,\n      -0.031344521790742874,\n      -0.11029056459665298,\n      -0.018613073974847794,\n      -0.06524443626403809,\n      0.05374087765812874,\n      0.04210519790649414,\n      -0.040458474308252335,\n      -0.057955361902713776,\n      0.06571163982152939,\n      -0.0013828104129061103,\n      -0.05554474890232086,\n      -0.11847154051065445,\n      -0.01464818324893713,\n      -0.0853366106748581,\n      0.0625956803560257,\n      0.06466683000326157,\n      -0.015261229127645493,\n      -0.14686913788318634,\n      0.04903962090611458,\n      -0.06550303101539612,\n      0.08024675399065018,\n      0.09193412959575653,\n      -0.10834819823503494,\n      -0.06389288604259491,\n      -0.1344148814678192,\n      0.04682121053338051,\n      -0.1531563699245453,\n      0.03687916323542595,\n      -0.010707863606512547,\n      -0.0912400335073471,\n      -0.11350998282432556,\n      0.044786639511585236,\n      -0.07332122325897217,\n      -0.010683935135602951,\n      -0.07133358716964722,\n      -0.06016651913523674,\n      0.029876383021473885,\n      -0.08981053531169891,\n      -0.03795229271054268,\n      -0.011416781693696976,\n      0.006259957328438759,\n      0.045083433389663696,\n      0.15495949983596802,\n      -0.09062402695417404,\n      0.05786530673503876,\n      -0.10347270220518112,\n      0.10656599700450897,\n      0.1352008879184723,\n      0.06208708509802818,\n      0.0878792330622673,\n      -0.0030028698965907097,\n      0.0371098630130291,\n      0.10786324739456177,\n      0.004473903216421604,\n      -0.047260135412216187,\n      -0.15963855385780334,\n      0.12938284873962402,\n      -0.12727853655815125,\n      -0.04626641795039177,\n      -0.17379605770111084,\n      0.06958272308111191,\n      0.02039007656276226,\n      0.01245127059519291,\n      0.11209814250469208\n    ],\n    [\n      -0.0662018433213234,\n      0.04137163236737251,\n      0.042191240936517715,\n      0.03825914114713669,\n      0.039716798812150955,\n      -0.09239303320646286,\n      -0.05801123380661011,\n      -0.01574963703751564,\n      0.15506161749362946,\n      0.008192646317183971,\n      0.03247401490807533,\n      -0.047091539949178696,\n      -0.15749675035476685,\n      0.035096876323223114,\n      0.03317879140377045,\n      0.0930451899766922,\n      -0.02368784323334694,\n      -0.14020204544067383,\n      -0.03257574141025543,\n      -0.04797985777258873,\n      0.026757221668958664,\n      -0.009934544563293457,\n      -0.11081629246473312,\n      -0.04764064401388168,\n      -0.14614863693714142,\n      -0.01004351582378149,\n      -0.14680363237857819,\n      0.09892145544290543,\n      -0.052439771592617035,\n      0.09039946645498276,\n      0.04950885474681854,\n      -0.03803452476859093,\n      -0.01040360052138567,\n      -0.030710803344845772,\n      -0.1039656400680542,\n      0.07739716023206711,\n      -0.025074955075979233,\n      -0.07184384018182755,\n      -0.07763437926769257,\n      -0.02559826895594597,\n      -0.0667162537574768,\n      -0.00034426632919348776,\n      -0.10512498766183853,\n      -0.005877080839127302,\n      -0.10133825987577438,\n      -0.018469808623194695,\n      -0.0436302088201046,\n      0.038942448794841766,\n      0.0038550738245248795,\n      -0.0007856441079638898,\n      -0.09094030410051346,\n      -0.024285079911351204,\n      -0.08522732555866241,\n      -0.05324901267886162,\n      -0.01558268629014492,\n      -0.053095027804374695,\n      -0.13462615013122559,\n      0.03632129728794098,\n      -0.10251393169164658,\n      -0.08407077938318253,\n      0.04606309160590172,\n      0.03776451572775841,\n      -0.08209933340549469,\n      -0.0030861308332532644,\n      -0.1002930998802185,\n      -0.10348492115736008,\n      -0.05743471533060074,\n      -0.11437804996967316,\n      -0.06773579120635986,\n      -0.002390175126492977,\n      -0.02625861205160618,\n      -0.17397621273994446,\n      -0.02282843552529812,\n      0.005907549988478422,\n      0.023956002667546272,\n      0.02266508713364601,\n      -0.09135867655277252,\n      -0.09448248147964478,\n      0.06650296598672867,\n      -0.1444685310125351,\n      0.060257043689489365,\n      -0.048050835728645325,\n      -0.11814405769109726,\n      -0.04890083149075508,\n      0.005875524133443832,\n      0.13093885779380798,\n      -0.03641941025853157,\n      0.06278073787689209,\n      0.0315842367708683,\n      0.06932634860277176,\n      0.07314272969961166,\n      0.04285860061645508,\n      -0.07424239814281464,\n      -0.004766156431287527,\n      -0.0024938685819506645,\n      -0.07638098299503326,\n      -0.014752325601875782,\n      -0.003607101971283555,\n      0.04916048422455788,\n      0.020390653982758522,\n      0.07502367347478867,\n      -0.10047242045402527,\n      0.09867265075445175,\n      0.1372435986995697,\n      -0.09532620757818222,\n      0.07220930606126785,\n      0.02412831410765648,\n      -0.024012327194213867,\n      0.11306844651699066,\n      0.026361210271716118,\n      0.09787318110466003,\n      0.07458522170782089,\n      -0.0015846539754420519,\n      -0.10119960457086563,\n      0.034887854009866714,\n      -0.05487342178821564,\n      -0.09885521978139877,\n      -0.009827688336372375,\n      0.04082348197698593,\n      0.09146623313426971,\n      -0.003582080826163292,\n      0.07308714091777802,\n      0.031005235388875008,\n      -0.024919427931308746,\n      0.038695916533470154\n    ],\n    [\n      0.06951949000358582,\n      0.11405415087938309,\n      0.005610412452369928,\n      0.00945291854441166,\n      0.010986074805259705,\n      -0.0238659605383873,\n      -0.048626046627759933,\n      0.008612008765339851,\n      0.16285677254199982,\n      -0.08696658909320831,\n      0.06369716674089432,\n      -0.11144303530454636,\n      -0.11910024285316467,\n      0.1919231414794922,\n      0.1899636685848236,\n      -0.043755099177360535,\n      -0.046281713992357254,\n      -0.04582476243376732,\n      -0.05018562451004982,\n      -0.039450254291296005,\n      -0.10018081218004227,\n      -0.023570196703076363,\n      -0.15190339088439941,\n      -0.032795049250125885,\n      0.0662769302725792,\n      0.036771804094314575,\n      -0.0951213389635086,\n      0.0885605663061142,\n      -0.03817995637655258,\n      0.16484233736991882,\n      0.0312725231051445,\n      0.0064955526031553745,\n      -0.06211552023887634,\n      0.01177082397043705,\n      0.05939841270446777,\n      -0.04221789166331291,\n      0.07695657759904861,\n      -0.05947001278400421,\n      -0.011032392270863056,\n      0.02930212952196598,\n      -0.060598861426115036,\n      0.014356818050146103,\n      -0.004481892101466656,\n      0.054659154266119,\n      -0.06394760310649872,\n      -0.12086781859397888,\n      -0.07013604044914246,\n      -0.05946044996380806,\n      -0.006288576405495405,\n      0.11323187500238419,\n      0.046406373381614685,\n      -0.007246389053761959,\n      -0.09561270475387573,\n      0.08520194888114929,\n      -0.003912020940333605,\n      -0.029868843033909798,\n      -0.02515411749482155,\n      -0.03949417546391487,\n      0.0006988701643422246,\n      -0.11723647266626358,\n      -0.07666415721178055,\n      -0.029346292838454247,\n      -0.06833448261022568,\n      -0.03728852793574333,\n      0.0037465556524693966,\n      0.06767722964286804,\n      0.03320229426026344,\n      0.07701107859611511,\n      0.0557045079767704,\n      -0.057220619171857834,\n      -0.009443136863410473,\n      -0.15330128371715546,\n      -0.001890762709081173,\n      0.01702377386391163,\n      0.02727196179330349,\n      -0.07244840264320374,\n      0.06755395233631134,\n      -0.0026081944815814495,\n      0.09845854341983795,\n      0.0579015389084816,\n      0.1001681536436081,\n      -0.03377656266093254,\n      -0.034014735370874405,\n      -0.01280591357499361,\n      -0.05075649917125702,\n      0.10794532299041748,\n      -0.004211763385683298,\n      -0.04231631010770798,\n      -0.06897784024477005,\n      0.05382983386516571,\n      0.042693864554166794,\n      0.03423900157213211,\n      0.009066433645784855,\n      0.027101097628474236,\n      0.07707242667675018,\n      0.031421322375535965,\n      0.033297982066869736,\n      0.005811532028019428,\n      -0.027083484455943108,\n      0.018356986343860626,\n      0.034241933375597,\n      0.022516801953315735,\n      0.11465968191623688,\n      0.04530569165945053,\n      0.005369878374040127,\n      0.017218200489878654,\n      -0.04958827793598175,\n      0.02149735763669014,\n      0.08501306921243668,\n      0.046550214290618896,\n      -0.04926485940814018,\n      0.054387468844652176,\n      -0.011723622679710388,\n      0.015812410041689873,\n      -0.09882974624633789,\n      0.048231638967990875,\n      -0.17263363301753998,\n      0.05149034783244133,\n      0.03526821359992027,\n      0.052857160568237305,\n      0.07427763938903809,\n      0.06614285707473755,\n      0.05462189391255379,\n      0.053455982357263565,\n      0.05240306630730629\n    ],\n    [\n      0.05264705419540405,\n      0.06460322439670563,\n      0.11357761174440384,\n      0.02632860094308853,\n      0.06811340898275375,\n      -0.07870887964963913,\n      0.0061389366164803505,\n      -0.032301392406225204,\n      0.08741573989391327,\n      -0.07859672605991364,\n      -0.01896943710744381,\n      -0.05697758495807648,\n      -0.06742464005947113,\n      0.10988781601190567,\n      0.1250462532043457,\n      -0.07602281123399734,\n      0.08252880722284317,\n      -0.010949991643428802,\n      0.0597219280898571,\n      -0.08981658518314362,\n      -0.022742953151464462,\n      0.03371090814471245,\n      0.004176806658506393,\n      -0.08422435820102692,\n      -0.03245944902300835,\n      -0.001667509670369327,\n      -0.11407832056283951,\n      0.04521219804883003,\n      -0.014608257450163364,\n      0.27734798192977905,\n      -0.012947204522788525,\n      0.06978141516447067,\n      0.07689918577671051,\n      -0.03272763267159462,\n      0.030838627368211746,\n      0.042434412986040115,\n      0.021838102489709854,\n      -0.09307581186294556,\n      -0.02133556455373764,\n      -0.0013098898343741894,\n      0.03839954361319542,\n      -0.11966506391763687,\n      0.003192418720573187,\n      -0.06603685021400452,\n      -0.055135346949100494,\n      0.04981306567788124,\n      -0.04217752441763878,\n      0.033177316188812256,\n      0.04751540347933769,\n      -0.02644418738782406,\n      0.024552086368203163,\n      0.007009574212133884,\n      -0.0669357031583786,\n      -0.0232088565826416,\n      -0.05207456648349762,\n      0.10419841855764389,\n      -0.04402215778827667,\n      -0.0240127295255661,\n      -0.08358784019947052,\n      -0.02901390939950943,\n      0.03489388898015022,\n      -0.058108966797590256,\n      0.04484406113624573,\n      -0.008687153458595276,\n      0.03172903135418892,\n      -0.046637143939733505,\n      0.08149664103984833,\n      0.0007652848726138473,\n      -0.062048740684986115,\n      -0.03884095698595047,\n      0.08978617191314697,\n      0.033853672444820404,\n      0.12663498520851135,\n      0.021971281617879868,\n      0.030638456344604492,\n      -0.07417932152748108,\n      -0.014665942639112473,\n      -0.045736998319625854,\n      -0.024313386529684067,\n      -0.019211748614907265,\n      0.035359252244234085,\n      -0.11414363235235214,\n      0.06787578761577606,\n      0.02721480280160904,\n      -0.03695012629032135,\n      0.08921542763710022,\n      0.08927790820598602,\n      0.04842764884233475,\n      0.0808807760477066,\n      -0.07510440051555634,\n      0.004345711320638657,\n      0.07290156930685043,\n      -0.027871018275618553,\n      0.04855657368898392,\n      0.05385219678282738,\n      -0.08756612241268158,\n      0.01172640360891819,\n      0.15893670916557312,\n      0.014362598769366741,\n      0.0439133495092392,\n      0.057003509253263474,\n      -0.056148745119571686,\n      0.01864849403500557,\n      0.08190891891717911,\n      0.019611839205026627,\n      -0.012616772204637527,\n      -0.016905780881643295,\n      0.03084360621869564,\n      0.07287552952766418,\n      0.007441436871886253,\n      0.011848369613289833,\n      -0.011733364313840866,\n      -0.07751379907131195,\n      0.05462329834699631,\n      0.041642121970653534,\n      -0.030530419200658798,\n      0.029364684596657753,\n      0.11273719370365143,\n      0.05289716646075249,\n      0.08671039342880249,\n      0.12076831609010696,\n      0.12156425416469574,\n      0.0002403648104518652,\n      0.08260844647884369,\n      -0.01148137729614973\n    ],\n    [\n      0.060752492398023605,\n      0.04735178500413895,\n      -0.0016404478810727596,\n      -0.0589975044131279,\n      0.04437759518623352,\n      0.03870893642306328,\n      0.08217479288578033,\n      0.10080593079328537,\n      -0.11241049319505692,\n      0.05641446262598038,\n      -0.0003322312841191888,\n      -0.08151832967996597,\n      0.06224481761455536,\n      -0.13062171638011932,\n      -0.04061548784375191,\n      0.05954485759139061,\n      0.009803330525755882,\n      -0.023791851475834846,\n      0.014220510609447956,\n      0.10619093477725983,\n      0.010648716241121292,\n      0.10857557505369186,\n      -0.012954623438417912,\n      0.08524798601865768,\n      -0.06632629036903381,\n      -0.02746024914085865,\n      0.0651303306221962,\n      -0.01979127712547779,\n      0.06257673352956772,\n      -0.11657857149839401,\n      -0.006345648784190416,\n      -0.07630050182342529,\n      -0.020107630640268326,\n      -0.05460246279835701,\n      -0.029624292626976967,\n      0.06499867141246796,\n      -0.06514740735292435,\n      0.044555433094501495,\n      0.07104634493589401,\n      0.008868900127708912,\n      0.04957519471645355,\n      -0.03332970291376114,\n      0.10238498449325562,\n      -0.008592860773205757,\n      0.05178585648536682,\n      -0.09389860928058624,\n      -0.06894521415233612,\n      -0.08468523621559143,\n      -0.03006107732653618,\n      0.051595356315374374,\n      -0.017536289989948273,\n      0.004051718860864639,\n      -0.06484649330377579,\n      0.06826641410589218,\n      -0.025051146745681763,\n      -0.03303665667772293,\n      0.060618530958890915,\n      -0.02242433838546276,\n      0.0822785347700119,\n      -0.04439542070031166,\n      -0.024838071316480637,\n      -0.05129622295498848,\n      0.06907586008310318,\n      -0.0063753495924174786,\n      -0.09037497639656067,\n      -0.09430942684412003,\n      0.052783478051424026,\n      -0.02268342487514019,\n      0.022312676534056664,\n      0.05754268169403076,\n      -0.06013725325465202,\n      -0.04783746227622032,\n      -0.014688391238451004,\n      0.042491454631090164,\n      -0.05413395166397095,\n      0.01040059607475996,\n      0.011488238349556923,\n      -0.047707974910736084,\n      -0.06415195763111115,\n      -0.032366517931222916,\n      -0.1404600590467453,\n      0.09884711354970932,\n      0.06396245956420898,\n      0.03220096230506897,\n      0.02292042039334774,\n      -0.00838442798703909,\n      -0.0729903057217598,\n      0.09391079843044281,\n      -0.07807769626379013,\n      -0.04859485849738121,\n      -0.011560476385056973,\n      0.08184054493904114,\n      -0.03911120072007179,\n      0.09868547320365906,\n      -0.048168305307626724,\n      -0.07814690470695496,\n      0.009362515062093735,\n      -0.05332421138882637,\n      -0.060742639005184174,\n      0.04955321177840233,\n      0.010506361722946167,\n      0.06060560792684555,\n      -0.03457522392272949,\n      0.09388650208711624,\n      0.024100979790091515,\n      0.017988091334700584,\n      0.0036546853370964527,\n      0.029044317081570625,\n      0.0760858952999115,\n      -0.005151105113327503,\n      0.06432728469371796,\n      0.028686728328466415,\n      0.05262625217437744,\n      0.02716323919594288,\n      0.02736673876643181,\n      0.02851943112909794,\n      0.05699368938803673,\n      0.012705042958259583,\n      -0.06425967067480087,\n      -0.1118977814912796,\n      -0.010283532552421093,\n      -0.06419377774000168,\n      0.028335437178611755,\n      -0.029198579490184784,\n      0.0641644299030304\n    ],\n    [\n      -0.04822608456015587,\n      -0.015370856039226055,\n      0.04504237323999405,\n      -0.014897978864610195,\n      -0.0562148354947567,\n      0.0569298230111599,\n      0.04744439199566841,\n      -0.11331022530794144,\n      0.171427384018898,\n      0.061115510761737823,\n      0.07763981819152832,\n      -0.10119915008544922,\n      0.029461849480867386,\n      0.010887166485190392,\n      0.17235113680362701,\n      -0.10260938853025436,\n      -0.0770454853773117,\n      -0.09459980577230453,\n      -6.305077113211155e-05,\n      -0.09425599873065948,\n      0.08969447761774063,\n      -0.0014073450583964586,\n      -0.0011477828957140446,\n      0.05154304951429367,\n      -0.029145672917366028,\n      -0.10956289619207382,\n      -0.0387975350022316,\n      0.07469858229160309,\n      -0.031869206577539444,\n      0.21962009370326996,\n      -0.014321133494377136,\n      0.08539876341819763,\n      0.10983117669820786,\n      0.03924545645713806,\n      0.05920153111219406,\n      0.053927503526210785,\n      -0.03788358345627785,\n      0.045437075197696686,\n      -0.06965400278568268,\n      0.015060825273394585,\n      0.09943420439958572,\n      0.011894294060766697,\n      -0.004364178515970707,\n      0.1216602772474289,\n      -0.22189044952392578,\n      0.09261903166770935,\n      -0.008842920884490013,\n      -0.12238039076328278,\n      -0.05061517283320427,\n      0.08939486742019653,\n      0.10183452069759369,\n      -0.06490503251552582,\n      -0.04594916105270386,\n      0.08345981687307358,\n      0.0261975284665823,\n      -0.06255880743265152,\n      0.047575633972883224,\n      0.11735450476408005,\n      -0.09174890071153641,\n      -0.009415281005203724,\n      -0.17568977177143097,\n      -0.03312230855226517,\n      0.0656442642211914,\n      0.055678337812423706,\n      -0.02945447526872158,\n      -0.0036500212736427784,\n      -0.15967939794063568,\n      0.0821235328912735,\n      0.03893573582172394,\n      -0.10323294252157211,\n      -0.03949879854917526,\n      -0.009664960205554962,\n      0.12151862680912018,\n      0.1457536220550537,\n      0.01764138974249363,\n      0.10000049322843552,\n      -0.03478439524769783,\n      0.09676084667444229,\n      0.13712601363658905,\n      -0.06610787659883499,\n      0.0480712428689003,\n      -0.06816846132278442,\n      -0.00690070353448391,\n      0.037700261920690536,\n      -0.007135260850191116,\n      0.05271841585636139,\n      0.08175302296876907,\n      -0.019814424216747284,\n      0.06214703619480133,\n      -0.10911151766777039,\n      0.060461852699518204,\n      0.0555160716176033,\n      -0.17953281104564667,\n      -0.058969005942344666,\n      -0.12048903852701187,\n      -0.12878364324569702,\n      0.055033937096595764,\n      -0.07779054343700409,\n      -0.035558462142944336,\n      0.05724576115608215,\n      -0.04809114336967468,\n      -0.09788576513528824,\n      -0.07469191402196884,\n      -0.10059421509504318,\n      -0.013989472761750221,\n      -0.00519889872521162,\n      -0.11520780622959137,\n      0.025439491495490074,\n      -0.07075408101081848,\n      -0.08325986564159393,\n      0.058397501707077026,\n      0.09815863519906998,\n      0.01243091281503439,\n      0.06548544764518738,\n      -0.032992418855428696,\n      -0.14909178018569946,\n      0.03740829601883888,\n      0.09383175522089005,\n      -0.04393076151609421,\n      0.04831865429878235,\n      0.07780096679925919,\n      -0.060159746557474136,\n      0.007437095511704683,\n      0.011675139889121056,\n      0.02166345715522766\n    ],\n    [\n      -0.01237466186285019,\n      0.04822153225541115,\n      0.013833840377628803,\n      -0.07048346847295761,\n      -0.0306247491389513,\n      0.05040581896901131,\n      0.11704114079475403,\n      -0.027261093258857727,\n      -0.05498447269201279,\n      -0.07322247326374054,\n      0.008080746047198772,\n      0.05791626498103142,\n      0.13414885103702545,\n      -0.06564763188362122,\n      -0.18605633080005646,\n      0.007039433810859919,\n      -0.004916987847536802,\n      -0.062402427196502686,\n      0.0042293136939406395,\n      -0.06674635410308838,\n      -0.06003022938966751,\n      0.010286270640790462,\n      -0.011929611675441265,\n      0.08803393691778183,\n      -0.14732231199741364,\n      -0.026651838794350624,\n      0.09189312160015106,\n      0.05195051059126854,\n      0.08650258928537369,\n      -0.08627352863550186,\n      -0.016547074541449547,\n      0.11922616511583328,\n      0.08561419695615768,\n      0.05637518689036369,\n      -0.03184797987341881,\n      -0.04474317654967308,\n      -0.03651255741715431,\n      -0.09807158261537552,\n      0.05198526009917259,\n      0.0020067880395799875,\n      -0.0629904642701149,\n      0.06020306423306465,\n      -0.03392283618450165,\n      -0.017982926219701767,\n      -0.07442067563533783,\n      -0.01176473032683134,\n      -0.06260872632265091,\n      0.0582764707505703,\n      0.06744282692670822,\n      0.040891896933317184,\n      -0.04385156184434891,\n      -0.06321816891431808,\n      0.11783456802368164,\n      0.05813991650938988,\n      -0.04196450114250183,\n      0.015297401696443558,\n      -0.01562198344618082,\n      0.04111543297767639,\n      0.01552726048976183,\n      0.06366407871246338,\n      0.04194992035627365,\n      -0.05144067481160164,\n      -0.007906247861683369,\n      -0.03584461659193039,\n      -0.01947661116719246,\n      0.0043735383078455925,\n      -0.01486301515251398,\n      0.011607591062784195,\n      0.0072015621699392796,\n      -0.11831635981798172,\n      0.012760083191096783,\n      -0.00018489290960133076,\n      -0.00476253405213356,\n      -0.17205436527729034,\n      0.0738055557012558,\n      0.024429576471447945,\n      0.021200083196163177,\n      -0.06981329619884491,\n      -0.15601426362991333,\n      0.020625557750463486,\n      -0.039700962603092194,\n      -0.056635286659002304,\n      0.0425601527094841,\n      0.06647678464651108,\n      -0.08448026329278946,\n      -0.07765661925077438,\n      -0.028715237975120544,\n      0.015846120193600655,\n      -0.10249310731887817,\n      0.08854206651449203,\n      -0.06670776009559631,\n      -0.046645186841487885,\n      -0.0630604475736618,\n      -0.0340934582054615,\n      0.05468708276748657,\n      -0.07658734172582626,\n      -0.044363390654325485,\n      -0.04831603914499283,\n      -0.031003717333078384,\n      0.046183839440345764,\n      -0.0679246187210083,\n      0.015975140035152435,\n      -0.04157442972064018,\n      -0.0027511022053658962,\n      0.12587325274944305,\n      0.008198867551982403,\n      -0.005178465042263269,\n      0.06327139586210251,\n      0.11155813187360764,\n      0.13950003683567047,\n      0.09015089273452759,\n      0.03955286741256714,\n      -0.0973123237490654,\n      0.052350401878356934,\n      0.016397731378674507,\n      -0.1604384183883667,\n      0.04195825755596161,\n      -0.03363976255059242,\n      0.07824788987636566,\n      -0.11807629466056824,\n      -0.0761435255408287,\n      -0.0011344803497195244,\n      0.04134874790906906,\n      0.028566835448145866,\n      -0.042642030864953995\n    ],\n    [\n      -0.03545313701033592,\n      0.04572116583585739,\n      0.056791651993989944,\n      -0.03559894859790802,\n      0.09745709598064423,\n      -0.037719957530498505,\n      -0.12674537301063538,\n      -0.11481983959674835,\n      0.15809164941310883,\n      0.05070393905043602,\n      0.06416498869657516,\n      -0.05934375151991844,\n      -0.09507135301828384,\n      0.006785318721085787,\n      0.08773454278707504,\n      -0.01976017840206623,\n      -0.04242311045527458,\n      -0.018305497244000435,\n      -0.04918373376131058,\n      -0.05356870964169502,\n      -0.014815730974078178,\n      -0.013852025382220745,\n      0.11059413850307465,\n      0.0207318477332592,\n      0.06549958139657974,\n      -0.030715245753526688,\n      -0.04156234487891197,\n      0.09726538509130478,\n      0.030970264226198196,\n      0.14921487867832184,\n      -0.020922748371958733,\n      -0.031188108026981354,\n      0.027824096381664276,\n      -0.06662032753229141,\n      -0.03994666039943695,\n      0.05697068199515343,\n      0.05123356357216835,\n      -0.09992268681526184,\n      0.042413610965013504,\n      -0.0416889488697052,\n      0.02441183477640152,\n      -0.04068851098418236,\n      -0.05837278813123703,\n      -0.05035034567117691,\n      0.05923299118876457,\n      0.09845739603042603,\n      0.02693404257297516,\n      0.05459314584732056,\n      -0.05696104094386101,\n      0.05129627510905266,\n      0.07411433011293411,\n      0.007058895193040371,\n      -0.02520841546356678,\n      0.10133343189954758,\n      0.05183526501059532,\n      -0.04462844878435135,\n      0.016677672043442726,\n      -0.015998903661966324,\n      -0.08896667510271072,\n      0.05320153757929802,\n      -0.02913934551179409,\n      -0.007168604992330074,\n      0.0012693974422290921,\n      0.0653088241815567,\n      -0.027170909568667412,\n      -0.0052109332755208015,\n      0.10031203925609589,\n      -0.05126236751675606,\n      -0.04867764934897423,\n      0.06161895766854286,\n      0.010366070084273815,\n      -0.03548947349190712,\n      0.02974236011505127,\n      0.03675806522369385,\n      0.06598727405071259,\n      -0.01734057255089283,\n      -0.06162272021174431,\n      -0.043317489326000214,\n      0.1332722306251526,\n      -0.07152681797742844,\n      0.06892827153205872,\n      -0.015579148195683956,\n      -0.050553109496831894,\n      -0.036940593272447586,\n      0.08690478652715683,\n      0.03973773494362831,\n      0.08104894310235977,\n      -0.08154912292957306,\n      -0.018593410030007362,\n      0.01150908786803484,\n      0.03760931268334389,\n      -0.0322696715593338,\n      0.012346681207418442,\n      -0.03126409277319908,\n      0.06825100630521774,\n      -0.07657001167535782,\n      0.0358358696103096,\n      0.0897952988743782,\n      0.11562537401914597,\n      -0.017083022743463516,\n      -0.04839930683374405,\n      0.03979003056883812,\n      0.08810260891914368,\n      0.09513749182224274,\n      0.010030291974544525,\n      0.0674838125705719,\n      0.08571068942546844,\n      0.04246703162789345,\n      0.008922948502004147,\n      0.07421921193599701,\n      0.10161296278238297,\n      0.10338668525218964,\n      -0.07730691879987717,\n      0.0018075412372127175,\n      -0.06902015954256058,\n      -0.07196599245071411,\n      -0.11496124416589737,\n      -0.014789501205086708,\n      0.1054394543170929,\n      0.0693964883685112,\n      0.07021328061819077,\n      0.051757171750068665,\n      0.02293253131210804,\n      0.06235023960471153,\n      0.05352845415472984\n    ],\n    [\n      0.02891566976904869,\n      0.042873796075582504,\n      0.025667864829301834,\n      0.001829861314035952,\n      -0.03923358395695686,\n      -0.03975126892328262,\n      0.12394978106021881,\n      0.11668252944946289,\n      -0.14179480075836182,\n      -0.014725171029567719,\n      0.004265205003321171,\n      0.005957949440926313,\n      0.07948270440101624,\n      -0.1918097734451294,\n      -0.19791533052921295,\n      0.07885163277387619,\n      -0.055231958627700806,\n      0.09406830370426178,\n      0.03827998414635658,\n      0.06705574691295624,\n      0.06861049681901932,\n      0.117556132376194,\n      0.053834155201911926,\n      0.08153820782899857,\n      0.024526186287403107,\n      -0.03763832524418831,\n      0.09788412600755692,\n      -0.008764552883803844,\n      0.13063785433769226,\n      -0.1626526266336441,\n      -0.0814630463719368,\n      -0.08094004541635513,\n      -0.033932242542505264,\n      -0.026158759370446205,\n      -0.04259152337908745,\n      -0.018989253789186478,\n      -0.07011116296052933,\n      -0.012608475983142853,\n      -0.000851108692586422,\n      0.0634111762046814,\n      -0.010685872286558151,\n      0.027822105213999748,\n      0.036358680576086044,\n      -0.015183325856924057,\n      0.015445113182067871,\n      -0.010688458569347858,\n      0.08553425967693329,\n      -0.06337719410657883,\n      -0.02800668217241764,\n      0.04199795797467232,\n      -0.05610496178269386,\n      0.015471434220671654,\n      0.07662422209978104,\n      0.01768340915441513,\n      -0.012063728645443916,\n      0.013520875945687294,\n      -0.047092992812395096,\n      0.03230429068207741,\n      -0.043371621519327164,\n      -0.05902184918522835,\n      0.10981404036283493,\n      -0.03931606188416481,\n      0.03311245143413544,\n      0.07178399711847305,\n      -0.0215610284358263,\n      0.07440801709890366,\n      -0.09611628204584122,\n      0.05587426573038101,\n      -0.026993032544851303,\n      -0.003683581482619047,\n      0.08674455434083939,\n      0.09041832387447357,\n      -0.1173531711101532,\n      -0.015102357603609562,\n      -0.0017434055916965008,\n      -0.07936937361955643,\n      -0.10205956548452377,\n      -0.027903784066438675,\n      -0.06527568399906158,\n      0.0020552861969918013,\n      -0.04708732292056084,\n      0.08085419237613678,\n      -0.05920922011137009,\n      0.06671779602766037,\n      0.040904007852077484,\n      0.011457213200628757,\n      0.03943636640906334,\n      0.0067669786512851715,\n      0.0679733008146286,\n      -0.02177559584379196,\n      0.03382667899131775,\n      0.02478799782693386,\n      -0.0031247141305357218,\n      0.09012018144130707,\n      -0.07567790150642395,\n      -0.03189370781183243,\n      -0.010368256829679012,\n      -0.032445844262838364,\n      -0.03883402794599533,\n      -0.01427649985998869,\n      0.077581025660038,\n      0.09720809012651443,\n      -0.034103475511074066,\n      0.048816509544849396,\n      0.10903201997280121,\n      0.0019525786628946662,\n      -0.007293703965842724,\n      -0.015352156944572926,\n      -0.07060573250055313,\n      0.07078680396080017,\n      -0.0400182344019413,\n      -0.05214608460664749,\n      0.0026643467135727406,\n      0.06695360690355301,\n      -0.03176663815975189,\n      -0.14115026593208313,\n      0.06678878515958786,\n      0.11133300513029099,\n      0.07498105615377426,\n      -0.015276503749191761,\n      -0.03193705156445503,\n      -0.0024427934549748898,\n      0.022230982780456543,\n      -0.0005811147857457399,\n      0.03229270875453949\n    ],\n    [\n      -0.13066470623016357,\n      -0.07107345014810562,\n      -0.06263340264558792,\n      -0.056763432919979095,\n      -0.01385753508657217,\n      -0.12051054835319519,\n      0.015117625705897808,\n      -0.009332967922091484,\n      -0.1228111982345581,\n      -0.0950445607304573,\n      -0.055150408297777176,\n      -0.05776672810316086,\n      -0.02612391859292984,\n      0.08132952451705933,\n      0.19841520488262177,\n      -0.03737073764204979,\n      -0.011912791058421135,\n      -0.0625075250864029,\n      -0.08464536815881729,\n      -0.02844511903822422,\n      0.029689893126487732,\n      -0.0880311131477356,\n      0.03131171315908432,\n      0.07027468830347061,\n      -0.03547365963459015,\n      -0.04864109680056572,\n      0.04539736360311508,\n      -0.08332414925098419,\n      0.05497211217880249,\n      0.13074404001235962,\n      0.05660340189933777,\n      0.04907648637890816,\n      -0.12305492162704468,\n      -0.06532501429319382,\n      -0.024555126205086708,\n      -0.12243881076574326,\n      -0.0210212841629982,\n      0.01971505768597126,\n      -0.02425786852836609,\n      -0.01937028020620346,\n      0.031922005116939545,\n      0.02805633656680584,\n      -0.030169840902090073,\n      -0.053918611258268356,\n      0.061436086893081665,\n      -0.00020074413623660803,\n      0.007779315579682589,\n      -0.03849274292588234,\n      0.11960628628730774,\n      0.13377033174037933,\n      0.10782639682292938,\n      -0.0038913872558623552,\n      -0.07697241753339767,\n      0.10296998172998428,\n      -0.07412592321634293,\n      0.13020120561122894,\n      0.15925277769565582,\n      0.09420248866081238,\n      0.040850598365068436,\n      -0.05991089344024658,\n      -0.007171453908085823,\n      0.08985400199890137,\n      0.1619207262992859,\n      0.12870286405086517,\n      -0.105612613260746,\n      0.01774030178785324,\n      0.07672794908285141,\n      -0.07811391353607178,\n      -0.09556198120117188,\n      -0.1293954998254776,\n      -0.04832027480006218,\n      0.0537596270442009,\n      0.0746859759092331,\n      0.10741740465164185,\n      0.049207523465156555,\n      0.05588856339454651,\n      -0.09746500849723816,\n      -0.07157423347234726,\n      -0.005867031868547201,\n      -0.0016397868748754263,\n      -0.0014972292119637132,\n      0.15611164271831512,\n      -0.051091086119413376,\n      -0.07640160620212555,\n      0.011884449049830437,\n      -0.019309639930725098,\n      -0.09820110350847244,\n      -0.1329328864812851,\n      -0.032082535326480865,\n      -0.04311418905854225,\n      -0.04722822830080986,\n      0.03421831130981445,\n      0.010144512169063091,\n      -0.08156511187553406,\n      -0.06784176081418991,\n      -0.08581165224313736,\n      0.005137494299560785,\n      0.05721285194158554,\n      -0.025262219831347466,\n      -0.04008367657661438,\n      -0.05259969085454941,\n      0.10595867782831192,\n      -0.026098325848579407,\n      0.0004283644084353,\n      0.07077191025018692,\n      -0.08227014541625977,\n      -0.004962807521224022,\n      0.09362374991178513,\n      -0.09273885935544968,\n      0.05691423639655113,\n      0.03149920701980591,\n      0.08089899271726608,\n      -0.015299905091524124,\n      0.04624975845217705,\n      0.11813205480575562,\n      -0.18946115672588348,\n      0.05812466889619827,\n      -0.023411523550748825,\n      -0.06682826578617096,\n      -0.0065334755927324295,\n      0.05802607536315918,\n      -0.09257496893405914,\n      -0.06472157686948776,\n      0.04880065470933914,\n      0.061725273728370667\n    ],\n    [\n      -0.06150130555033684,\n      0.12828302383422852,\n      0.0986008569598198,\n      0.14680233597755432,\n      -0.07752444595098495,\n      0.0033503551967442036,\n      -0.014368006028234959,\n      -0.15784546732902527,\n      0.19541162252426147,\n      0.021993914619088173,\n      -0.006503088865429163,\n      0.034943949431180954,\n      -0.009080873802304268,\n      0.20740486681461334,\n      0.18800263106822968,\n      -0.0283376332372427,\n      0.0037378163542598486,\n      -0.13304416835308075,\n      -0.03498447686433792,\n      -0.02584599331021309,\n      -0.060182124376297,\n      -0.0008099312544800341,\n      -0.04899859428405762,\n      0.01425939705222845,\n      -0.2315690815448761,\n      -0.07413787394762039,\n      -0.10814453661441803,\n      0.05034380778670311,\n      -0.017710065469145775,\n      0.0893779769539833,\n      0.031376175582408905,\n      0.09426914155483246,\n      -0.010929594747722149,\n      -0.07190948724746704,\n      0.10294676572084427,\n      -0.01581769809126854,\n      0.0815093144774437,\n      0.024421727284789085,\n      0.0019087621476501226,\n      -0.05659538879990578,\n      -0.05798855796456337,\n      -0.13429568707942963,\n      -0.013594178482890129,\n      -0.10172511637210846,\n      -0.13716228306293488,\n      -0.10478079319000244,\n      0.03794330358505249,\n      -0.07935067266225815,\n      -0.12334693968296051,\n      -0.04316350445151329,\n      -0.07183525711297989,\n      -0.07801719754934311,\n      -0.1176375225186348,\n      -0.14657822251319885,\n      -0.06332126259803772,\n      -0.0804516151547432,\n      0.02378976345062256,\n      0.01874767802655697,\n      -0.14389999210834503,\n      0.010424269363284111,\n      -0.045973874628543854,\n      -0.01579512469470501,\n      0.006487474776804447,\n      -0.13318896293640137,\n      -0.14120467007160187,\n      -0.14533773064613342,\n      -0.03416062146425247,\n      -0.18221043050289154,\n      -0.015062745660543442,\n      -0.0019471836276352406,\n      -2.9122202249709517e-06,\n      -0.08783158659934998,\n      0.09372681379318237,\n      -0.07615253329277039,\n      0.03831712529063225,\n      -0.04722771793603897,\n      0.10093383491039276,\n      -0.025491822510957718,\n      0.10128091275691986,\n      -0.059040527790784836,\n      0.0780249685049057,\n      -0.004703978076577187,\n      -0.05073317512869835,\n      -0.07802259922027588,\n      0.09714972227811813,\n      0.009721667505800724,\n      0.07940647006034851,\n      -0.05547494441270828,\n      -0.022832639515399933,\n      -0.04314633086323738,\n      0.02208099514245987,\n      0.009715449996292591,\n      -0.006057149730622768,\n      0.08205027878284454,\n      0.0950734093785286,\n      -0.08728805929422379,\n      0.014114984311163425,\n      0.009121019393205643,\n      -0.0720766931772232,\n      -0.04421732574701309,\n      0.0690721869468689,\n      0.07007765769958496,\n      0.07040771842002869,\n      0.00649189343675971,\n      -0.147858664393425,\n      0.02993283048272133,\n      0.05138647183775902,\n      -0.03372149541974068,\n      -0.005655831657350063,\n      0.001341015798971057,\n      0.10980445891618729,\n      0.05369619280099869,\n      0.019315142184495926,\n      -0.0495871901512146,\n      0.06365388631820679,\n      -0.22245997190475464,\n      -0.0756831243634224,\n      0.07938113808631897,\n      -0.10640238970518112,\n      0.0752868503332138,\n      -0.1251821368932724,\n      0.12142281979322433,\n      -0.0918535515666008,\n      0.09450719505548477,\n      0.06392816454172134\n    ],\n    [\n      -0.014603987336158752,\n      -0.12071631103754044,\n      -0.08503386378288269,\n      -0.04531889408826828,\n      0.006535728462040424,\n      0.12826651334762573,\n      0.05698465555906296,\n      0.07980808615684509,\n      0.059971630573272705,\n      0.11481238901615143,\n      0.05069774389266968,\n      0.03448702022433281,\n      -0.08597680181264877,\n      -0.09412763267755508,\n      -0.045942407101392746,\n      0.08447026461362839,\n      -0.04934876039624214,\n      -0.04679562523961067,\n      -0.08050582557916641,\n      0.19759275019168854,\n      -0.037287116050720215,\n      -0.052612073719501495,\n      0.041677333414554596,\n      -0.11421257257461548,\n      -0.15574297308921814,\n      -0.07035630941390991,\n      0.08263992518186569,\n      -0.09418918937444687,\n      -0.042356375604867935,\n      -0.16455742716789246,\n      -0.14068575203418732,\n      -0.02667345106601715,\n      0.02858642488718033,\n      0.08035609126091003,\n      -0.0033051460050046444,\n      -0.00017511658370494843,\n      -0.04959987848997116,\n      0.06489986926317215,\n      0.05022513121366501,\n      0.00887661799788475,\n      0.10852319002151489,\n      0.09159411489963531,\n      0.11588160693645477,\n      0.05302351713180542,\n      0.06503846496343613,\n      -0.015431334264576435,\n      0.024065354838967323,\n      -0.016953425481915474,\n      -0.05789003148674965,\n      -0.02502455934882164,\n      -0.023036988452076912,\n      -0.14952360093593597,\n      -0.09756922721862793,\n      -0.06773734837770462,\n      -0.061682507395744324,\n      -0.06701471656560898,\n      -0.09108522534370422,\n      0.055852118879556656,\n      0.008515204302966595,\n      0.020715340971946716,\n      0.09459837526082993,\n      0.07005549222230911,\n      -0.008867192082107067,\n      -0.06459258496761322,\n      -0.10435426980257034,\n      -0.05111192539334297,\n      -0.05396801233291626,\n      0.05535931885242462,\n      0.01327125821262598,\n      -0.031201321631669998,\n      0.06818190962076187,\n      -0.12059348821640015,\n      -0.019764460623264313,\n      -0.026836352422833443,\n      -0.01694360002875328,\n      0.011721758171916008,\n      0.05135483667254448,\n      -0.055850863456726074,\n      -0.00046621664660051465,\n      -0.003571990178897977,\n      -0.07759888470172882,\n      0.10419625788927078,\n      0.10140515863895416,\n      -0.05155472457408905,\n      0.03708261996507645,\n      -0.04904763773083687,\n      -0.0638057067990303,\n      -0.07939198613166809,\n      0.01295811589807272,\n      -0.008081282489001751,\n      -0.05031851679086685,\n      -0.0485970713198185,\n      -0.021640285849571228,\n      -0.06390289217233658,\n      0.03473743423819542,\n      -0.06988950818777084,\n      -0.07539498805999756,\n      -0.0702240988612175,\n      -0.052771177142858505,\n      0.06214328110218048,\n      0.015443489886820316,\n      -0.03218859061598778,\n      0.03690915182232857,\n      -0.04438331723213196,\n      0.08312072604894638,\n      -0.13650581240653992,\n      -0.07015998661518097,\n      -0.02210814505815506,\n      0.10458363592624664,\n      0.12510144710540771,\n      -0.008701992221176624,\n      -0.1356498748064041,\n      0.05047815293073654,\n      0.022409463301301003,\n      -0.025884943082928658,\n      -0.21132470667362213,\n      0.03414095938205719,\n      -0.06323409825563431,\n      -0.09261219203472137,\n      0.013271664269268513,\n      0.054118696600198746,\n      0.10832267254590988,\n      -0.04435785859823227,\n      0.026045124977827072,\n      -0.019509654492139816\n    ],\n    [\n      -0.007043620105832815,\n      -0.027410294860601425,\n      -0.0386771634221077,\n      0.01838650554418564,\n      0.08907952159643173,\n      0.029936257749795914,\n      -0.12206530570983887,\n      -0.0906636118888855,\n      0.05760413035750389,\n      0.0656508207321167,\n      0.05641178786754608,\n      -0.04220011085271835,\n      -0.027882056310772896,\n      0.1526484191417694,\n      0.1706586331129074,\n      0.014704879373311996,\n      -0.08742701262235641,\n      -0.03783294931054115,\n      -0.06524168699979782,\n      -0.009902370162308216,\n      -0.04405730962753296,\n      0.009071215987205505,\n      0.02319326251745224,\n      0.011171876452863216,\n      0.031297218054533005,\n      -0.07407780736684799,\n      -0.04179970175027847,\n      0.056719280779361725,\n      -0.02478058636188507,\n      0.262037992477417,\n      0.12888456881046295,\n      -0.05673475190997124,\n      0.06557533144950867,\n      0.029402727261185646,\n      0.061451856046915054,\n      -0.023931872099637985,\n      -0.03193023428320885,\n      -0.07553493976593018,\n      -0.12629809975624084,\n      -0.021649086847901344,\n      -0.0536901094019413,\n      -0.10398948937654495,\n      -0.046886660158634186,\n      -0.054379746317863464,\n      -0.10650164633989334,\n      -0.030134674161672592,\n      -0.043913424015045166,\n      -0.05967133864760399,\n      0.0892389565706253,\n      0.0007087026606313884,\n      0.11317980289459229,\n      0.04277408495545387,\n      0.011532176285982132,\n      0.053084295243024826,\n      0.041413601487874985,\n      0.020481616258621216,\n      0.04024302214384079,\n      -0.053098421543836594,\n      0.049351342022418976,\n      -0.05975884571671486,\n      0.00035760554601438344,\n      -0.03418876975774765,\n      -0.07141883671283722,\n      0.013167856261134148,\n      0.05974544584751129,\n      0.03504328429698944,\n      0.0975799635052681,\n      0.052154261618852615,\n      0.08145233988761902,\n      -0.03627603128552437,\n      0.09212002903223038,\n      0.05227673053741455,\n      0.11102953553199768,\n      0.004083107691258192,\n      -0.03559712693095207,\n      -0.017657743766903877,\n      -0.002930011134594679,\n      0.02939910814166069,\n      0.12955215573310852,\n      -0.05394637584686279,\n      0.09553902596235275,\n      -0.05102834105491638,\n      0.04898609593510628,\n      0.01835518144071102,\n      0.08233386278152466,\n      0.07603128999471664,\n      -0.005020344164222479,\n      0.05778123065829277,\n      0.002511661034077406,\n      -0.0683942586183548,\n      0.02383732609450817,\n      -0.01491939090192318,\n      -0.009360658004879951,\n      0.06340914964675903,\n      -0.008350252173841,\n      -0.06582345813512802,\n      0.06074902042746544,\n      0.14509397745132446,\n      -0.06887441128492355,\n      0.03339285030961037,\n      -0.023179281502962112,\n      0.06027721241116524,\n      0.015583097003400326,\n      0.127432182431221,\n      0.044811274856328964,\n      -0.047331664711236954,\n      -0.0440557524561882,\n      -0.0880245566368103,\n      0.03743778169155121,\n      -0.05453464388847351,\n      0.08338285982608795,\n      0.025862127542495728,\n      -0.06149064004421234,\n      -0.09552614390850067,\n      0.0517176128923893,\n      0.08574600517749786,\n      0.008494186215102673,\n      0.06018110364675522,\n      0.07026873528957367,\n      0.004946691915392876,\n      0.15351712703704834,\n      0.14077143371105194,\n      -0.07553257048130035,\n      0.10519742965698242,\n      0.03776298090815544\n    ],\n    [\n      -0.03861207515001297,\n      -0.036344483494758606,\n      0.1038304790854454,\n      -0.031745534390211105,\n      0.09314201027154922,\n      -0.02303786762058735,\n      -0.10373523086309433,\n      0.03897012770175934,\n      0.14735771715641022,\n      -0.09194962680339813,\n      -0.04865393415093422,\n      -0.11703655123710632,\n      0.023724732920527458,\n      -0.13846102356910706,\n      0.17765282094478607,\n      -0.07416629046201706,\n      0.02759324014186859,\n      -0.11749258637428284,\n      -0.0552598275244236,\n      -0.007532957009971142,\n      -0.03836709260940552,\n      -0.06878239661455154,\n      -0.049041468650102615,\n      0.10338139533996582,\n      0.09933497756719589,\n      0.004929358139634132,\n      -0.003260585945099592,\n      0.07488290965557098,\n      0.07206021994352341,\n      0.08822371065616608,\n      0.05853801220655441,\n      0.08616282790899277,\n      -0.11488746851682663,\n      -0.06186814606189728,\n      -0.15176594257354736,\n      0.04314781725406647,\n      0.01770910434424877,\n      -0.0335240438580513,\n      0.003325721714645624,\n      -0.03935634344816208,\n      0.13773931562900543,\n      0.11364149302244186,\n      -0.012515201233327389,\n      -0.08599795401096344,\n      0.05436161905527115,\n      0.06084800139069557,\n      -0.11502587795257568,\n      0.11062412708997726,\n      -0.06259417533874512,\n      0.09799225628376007,\n      0.05611617490649223,\n      -0.060495465993881226,\n      -0.000985410762950778,\n      0.03521271049976349,\n      0.045979179441928864,\n      0.16546587646007538,\n      0.017471710219979286,\n      0.03142083063721657,\n      0.024758318439126015,\n      0.09430229663848877,\n      0.07547737658023834,\n      0.1027870774269104,\n      -0.08301219344139099,\n      -0.0037789451889693737,\n      -0.03146205097436905,\n      0.02054123766720295,\n      -0.09621676802635193,\n      -0.0818185806274414,\n      -0.03053051233291626,\n      -0.006011859979480505,\n      -0.09694764018058777,\n      0.0029040160588920116,\n      0.04094433784484863,\n      0.040075644850730896,\n      0.10135704278945923,\n      -0.04871794581413269,\n      -0.021748168393969536,\n      0.13101156055927277,\n      -0.040764205157756805,\n      -0.08101490139961243,\n      0.06862087547779083,\n      0.013637657277286053,\n      0.053023770451545715,\n      0.03089374490082264,\n      -0.06418711692094803,\n      0.05045043304562569,\n      0.12023026496171951,\n      0.042416758835315704,\n      -0.010149559937417507,\n      0.048033226281404495,\n      -0.05913940444588661,\n      0.11016218364238739,\n      0.02135777287185192,\n      -0.068818598985672,\n      -0.11107991635799408,\n      0.05069832131266594,\n      -0.004629646893590689,\n      -0.013919823803007603,\n      0.11165895313024521,\n      0.06900282949209213,\n      0.05361064895987511,\n      0.0651291236281395,\n      -0.04350406304001808,\n      0.05892469733953476,\n      -0.062088482081890106,\n      -0.009915972128510475,\n      0.039548881351947784,\n      0.043590620160102844,\n      0.05770541727542877,\n      0.07695547491312027,\n      -0.0773707702755928,\n      0.08527728915214539,\n      -0.0520801804959774,\n      -0.05849283188581467,\n      -0.01706770434975624,\n      -0.21586136519908905,\n      0.008153802715241909,\n      0.005979632958769798,\n      0.038064438849687576,\n      -0.007764082867652178,\n      0.020842617377638817,\n      0.10277571529150009,\n      -0.022461142390966415,\n      0.11100033670663834,\n      0.040341202169656754\n    ],\n    [\n      0.007388305850327015,\n      -0.06046187877655029,\n      -0.0697033479809761,\n      0.06796237081289291,\n      -0.1390622854232788,\n      -0.03321469947695732,\n      0.0028506771195679903,\n      0.1320538967847824,\n      -0.10535580664873123,\n      0.0030655045993626118,\n      -0.02484298311173916,\n      -0.053542423993349075,\n      0.013085843063890934,\n      -0.028575032949447632,\n      -0.06921806186437607,\n      0.044704657047986984,\n      0.05561763048171997,\n      0.056555334478616714,\n      -0.01143938209861517,\n      -0.08593285083770752,\n      0.10366936773061752,\n      0.037220731377601624,\n      -0.02676066756248474,\n      0.10896303504705429,\n      -0.04059544950723648,\n      0.023884309455752373,\n      -0.08297024667263031,\n      -0.07881075888872147,\n      0.09166954457759857,\n      -0.21364498138427734,\n      -0.11622092127799988,\n      0.04280399531126022,\n      0.03429699316620827,\n      0.08566925674676895,\n      0.09856375306844711,\n      0.04344402626156807,\n      0.002808129880577326,\n      -0.029670368880033493,\n      0.027241300791502,\n      0.049525532871484756,\n      -0.014135044068098068,\n      0.054427310824394226,\n      0.030976327136158943,\n      -0.04249363765120506,\n      0.0913686752319336,\n      0.030158035457134247,\n      -0.059045277535915375,\n      -0.04798140004277229,\n      0.034104134887456894,\n      -0.001220947364345193,\n      -0.1553805023431778,\n      -0.0030192688573151827,\n      0.03327872231602669,\n      0.06705677509307861,\n      -0.12110547721385956,\n      -0.10014401376247406,\n      0.07631024718284607,\n      -0.011080292984843254,\n      -0.07114902883768082,\n      0.010799562558531761,\n      -0.010266521014273167,\n      0.06135798618197441,\n      -0.020184354856610298,\n      -0.06409145891666412,\n      0.04328928142786026,\n      0.048708170652389526,\n      -0.0021908264607191086,\n      0.055302977561950684,\n      -0.017919868230819702,\n      0.08341022580862045,\n      -0.04930087551474571,\n      0.07597973197698593,\n      -0.17940519750118256,\n      -0.042911507189273834,\n      0.026797212660312653,\n      0.026752689853310585,\n      0.008703511208295822,\n      -0.028430990874767303,\n      -0.05452689155936241,\n      0.011353832669556141,\n      0.022500772029161453,\n      -0.07815825939178467,\n      -0.054427944123744965,\n      -0.088577039539814,\n      0.02619391307234764,\n      -0.1411692202091217,\n      0.020613905042409897,\n      0.01712585985660553,\n      0.02532893419265747,\n      0.053956061601638794,\n      -0.06459815055131912,\n      0.009098418988287449,\n      0.028640050441026688,\n      0.023272352293133736,\n      0.0339646115899086,\n      0.08931533247232437,\n      -0.05555747449398041,\n      -0.08920525759458542,\n      -0.04845299944281578,\n      0.06388592720031738,\n      -0.10901235789060593,\n      0.03849932551383972,\n      0.0838223546743393,\n      -0.02675328031182289,\n      -0.08034128695726395,\n      -0.038179341703653336,\n      0.03678387030959129,\n      0.06901415437459946,\n      0.05932416394352913,\n      -0.00851872656494379,\n      -0.0035704730544239283,\n      0.07345942407846451,\n      0.004294934682548046,\n      -0.06799837201833725,\n      -0.008876921609044075,\n      -0.04837498068809509,\n      -0.06971390545368195,\n      0.07925470918416977,\n      0.07284814864397049,\n      -0.12415928393602371,\n      -0.0329606756567955,\n      -0.1081836149096489,\n      0.04785556346178055,\n      -0.16750261187553406,\n      -0.04359174519777298\n    ],\n    [\n      0.059304460883140564,\n      -0.05659075081348419,\n      0.05337407812476158,\n      0.05543702468276024,\n      -0.07253602147102356,\n      -0.03503834456205368,\n      -0.0044496855698525906,\n      0.1167212575674057,\n      -0.11390375345945358,\n      0.04888474568724632,\n      0.10489832609891891,\n      -0.05680575966835022,\n      0.0817394107580185,\n      -0.032883550971746445,\n      -0.17709165811538696,\n      0.10465658456087112,\n      0.03494120389223099,\n      -0.010922644287347794,\n      0.04682394489645958,\n      0.10735411942005157,\n      0.05409609526395798,\n      0.07924710214138031,\n      0.03324767202138901,\n      0.03573214262723923,\n      -0.027607019990682602,\n      0.05224952846765518,\n      0.18706674873828888,\n      0.004750209394842386,\n      0.011309121735394001,\n      -0.09649933874607086,\n      -0.03190511837601662,\n      -0.06173580884933472,\n      -0.01993630640208721,\n      0.07664485275745392,\n      -0.018607739359140396,\n      -0.04603401571512222,\n      0.09569823741912842,\n      0.06564043462276459,\n      -0.04136776179075241,\n      0.004362618550658226,\n      0.09499157965183258,\n      -0.04979173094034195,\n      0.01955385133624077,\n      -0.03267385810613632,\n      0.07583383470773697,\n      -0.053186796605587006,\n      -0.025373663753271103,\n      0.026611613109707832,\n      0.0192097220569849,\n      0.052400801330804825,\n      0.04926373064517975,\n      -0.03402525931596756,\n      0.00268586166203022,\n      0.07818074524402618,\n      0.025854790583252907,\n      0.015461518429219723,\n      -0.031672246754169464,\n      -0.009731495752930641,\n      0.08838994801044464,\n      -0.0216760765761137,\n      0.00526159442961216,\n      -0.02368319220840931,\n      0.0618014894425869,\n      0.09206706285476685,\n      0.021052811294794083,\n      -0.07930292189121246,\n      -0.01786934956908226,\n      -0.0046864463947713375,\n      0.05602908134460449,\n      -0.07549024373292923,\n      -0.0069352672435343266,\n      0.08422691375017166,\n      -0.05047943443059921,\n      0.00507445028051734,\n      -0.0061135306023061275,\n      0.00012547419464681298,\n      0.03197780251502991,\n      0.06944570690393448,\n      -0.048255689442157745,\n      -0.01229197159409523,\n      -0.09688181430101395,\n      0.056878309696912766,\n      0.06758151948451996,\n      -0.016576558351516724,\n      0.08218652755022049,\n      -0.0972680002450943,\n      0.04874630644917488,\n      -0.06820815801620483,\n      -0.027263080701231956,\n      0.0634709820151329,\n      -0.03744811564683914,\n      -0.0121842660009861,\n      0.004321611486375332,\n      -0.07363261282444,\n      0.03016827628016472,\n      -0.05033981800079346,\n      0.11207113415002823,\n      0.06413110345602036,\n      0.0603809654712677,\n      0.014163212850689888,\n      0.011999820359051228,\n      0.07487623393535614,\n      -0.010589255020022392,\n      0.08326634019613266,\n      0.08415451645851135,\n      -0.05265868082642555,\n      -0.09196951240301132,\n      -0.015235449187457561,\n      0.07967532426118851,\n      0.03641536459326744,\n      0.07487542927265167,\n      -0.046900246292352676,\n      -0.01902785338461399,\n      0.09738236665725708,\n      -0.0015689063584432006,\n      -0.09051185101270676,\n      0.1137424111366272,\n      0.12193537503480911,\n      0.06405769288539886,\n      -0.1094551756978035,\n      -0.10209856927394867,\n      -0.0657183900475502,\n      -0.0009505731286481023,\n      -0.020495016127824783,\n      0.05548463389277458\n    ],\n    [\n      0.08142338693141937,\n      -0.019523655995726585,\n      -0.09274175763130188,\n      0.05458562448620796,\n      -0.0006281565292738378,\n      -0.021096786484122276,\n      -0.002445314545184374,\n      0.12816594541072845,\n      -0.1035383939743042,\n      0.018121683970093727,\n      0.11874189227819443,\n      0.011646835133433342,\n      0.018927734345197678,\n      -0.12389451265335083,\n      -0.06824224442243576,\n      0.06746401637792587,\n      0.03782637044787407,\n      0.09623438119888306,\n      0.0707726702094078,\n      -0.0019529464188963175,\n      0.08106347918510437,\n      0.06673400849103928,\n      -0.07347693294286728,\n      0.10555517673492432,\n      -0.05819980055093765,\n      0.0336027666926384,\n      0.1264531910419464,\n      -0.12220235913991928,\n      0.10558659583330154,\n      -0.14105288684368134,\n      -0.04724074527621269,\n      0.001999819651246071,\n      -0.0798589214682579,\n      0.004680695477873087,\n      -0.006270238198339939,\n      0.11256778240203857,\n      0.06673740595579147,\n      -0.029823383316397667,\n      0.008664241060614586,\n      0.02320750802755356,\n      0.10647302865982056,\n      0.017653953284025192,\n      0.06625311076641083,\n      0.0558449849486351,\n      -0.011033067479729652,\n      0.05172356963157654,\n      -0.07196864485740662,\n      0.051089096814394,\n      0.04495394974946976,\n      -0.10220267623662949,\n      -0.011131355538964272,\n      0.06995875388383865,\n      0.023747364059090614,\n      -0.03164738789200783,\n      0.06669189035892487,\n      -0.08384974300861359,\n      0.0596846267580986,\n      0.08864382654428482,\n      0.03425757959485054,\n      -0.007230274844914675,\n      -0.05654895678162575,\n      -0.008161847479641438,\n      0.08640142530202866,\n      -0.0019497430184856057,\n      0.03787102922797203,\n      -0.03612449765205383,\n      -0.026077061891555786,\n      0.07140801101922989,\n      0.011587215587496758,\n      0.0603545606136322,\n      -0.026482246816158295,\n      0.016104506328701973,\n      -0.013706561177968979,\n      -0.10681682825088501,\n      0.04315079376101494,\n      -0.04760308191180229,\n      0.035447489470243454,\n      0.0012521120952442288,\n      -0.02262641303241253,\n      -0.022297875955700874,\n      -0.03655543178319931,\n      -0.018622981384396553,\n      0.0598609484732151,\n      0.02193104662001133,\n      0.0534011572599411,\n      0.032790206372737885,\n      0.06676266342401505,\n      -0.01651740074157715,\n      0.07372800260782242,\n      0.07891058176755905,\n      -0.07229941338300705,\n      0.005795310717076063,\n      0.0348222516477108,\n      0.05423209071159363,\n      -0.07647617906332016,\n      0.03567730262875557,\n      -0.04266738146543503,\n      -0.06267999112606049,\n      -0.07809922099113464,\n      -0.06935988366603851,\n      0.06356217712163925,\n      0.08353813737630844,\n      -0.08768387138843536,\n      -0.04632233828306198,\n      -0.03409869596362114,\n      -0.027860455214977264,\n      0.07804059237241745,\n      -0.012994373217225075,\n      0.014563064090907574,\n      0.03553315997123718,\n      0.07042065262794495,\n      -0.07131684571504593,\n      0.08474336564540863,\n      -0.07264696806669235,\n      -0.017741231247782707,\n      -0.1778072863817215,\n      0.05542276054620743,\n      0.029248099774122238,\n      0.07106199115514755,\n      -0.10289058834314346,\n      0.006953296717256308,\n      -0.06230633705854416,\n      0.03479496017098427,\n      -0.0017339261248707771,\n      -0.05164894834160805\n    ],\n    [\n      -0.017174510285258293,\n      -0.07261811196804047,\n      -0.02579774148762226,\n      0.0389661006629467,\n      0.1175714060664177,\n      0.05147887021303177,\n      -0.007952727377414703,\n      -0.16554099321365356,\n      0.07081826031208038,\n      0.053082164376974106,\n      0.048102300614118576,\n      0.04863874614238739,\n      -0.17475749552249908,\n      0.03309616073966026,\n      0.18876545131206512,\n      -0.01421151403337717,\n      0.002683078171685338,\n      -0.032603517174720764,\n      -0.08711391687393188,\n      -0.011745809577405453,\n      -0.02939179353415966,\n      -0.0077522918581962585,\n      0.022038346156477928,\n      0.03221948444843292,\n      -0.011849690228700638,\n      -0.10831192880868912,\n      -0.09629515558481216,\n      -0.053663723170757294,\n      -0.050204068422317505,\n      0.12000571936368942,\n      0.00652872072532773,\n      0.084805428981781,\n      -0.04013647139072418,\n      0.04268356412649155,\n      0.03861714154481888,\n      -0.004652954638004303,\n      0.060020070523023605,\n      -0.12767018377780914,\n      -0.04038914293050766,\n      -0.012202270328998566,\n      0.0008603350142948329,\n      -0.10582534223794937,\n      -0.11706208437681198,\n      -0.016589859500527382,\n      -0.005948265083134174,\n      -0.07572123408317566,\n      0.007775850594043732,\n      -0.009650439023971558,\n      0.08049339056015015,\n      0.07420478761196136,\n      -0.0222342349588871,\n      0.07603190839290619,\n      0.026002967730164528,\n      -0.024272723123431206,\n      0.07480660825967789,\n      -0.0633232519030571,\n      -0.0340387187898159,\n      0.04574904218316078,\n      -0.1039343923330307,\n      -0.09388602524995804,\n      -0.0210716649889946,\n      -0.008515545167028904,\n      -0.04851561784744263,\n      0.026324214413762093,\n      0.07767406851053238,\n      0.013481328263878822,\n      0.03099152445793152,\n      0.0454833023250103,\n      -0.008654958568513393,\n      -0.026921585202217102,\n      0.02486705407500267,\n      -0.01671094261109829,\n      0.04748683422803879,\n      0.13701879978179932,\n      0.04688360169529915,\n      -0.03795280680060387,\n      0.09659118205308914,\n      0.02654784731566906,\n      0.14134368300437927,\n      -0.08502902090549469,\n      0.03138367831707001,\n      -0.05244933441281319,\n      -0.03668808937072754,\n      -0.031606584787368774,\n      -0.03801162540912628,\n      0.03772240877151489,\n      -0.03116818703711033,\n      -0.07896103709936142,\n      0.038180626928806305,\n      -0.034348249435424805,\n      0.08552730828523636,\n      -0.05937177687883377,\n      0.04681545868515968,\n      0.055869389325380325,\n      0.0822015106678009,\n      -0.035551369190216064,\n      0.05002571642398834,\n      -0.044995203614234924,\n      0.01008147094398737,\n      -0.011940970085561275,\n      0.014604395255446434,\n      0.059959642589092255,\n      -0.03137177601456642,\n      -0.04496374353766441,\n      -0.10539188235998154,\n      0.046147920191287994,\n      0.09089531004428864,\n      -0.05441932752728462,\n      0.06506096571683884,\n      -0.08464330434799194,\n      0.0661163330078125,\n      0.03469832241535187,\n      0.061628490686416626,\n      -0.12924489378929138,\n      0.05825331062078476,\n      -0.04413271322846413,\n      -0.0620977059006691,\n      -0.04376331344246864,\n      0.07744558900594711,\n      0.10934465378522873,\n      0.17392338812351227,\n      0.0005523003055714071,\n      0.0051030651666224,\n      -0.07060334086418152,\n      -0.05820126459002495\n    ],\n    [\n      0.05841708183288574,\n      -0.05322045460343361,\n      -0.0841464176774025,\n      -0.07966389507055283,\n      -0.009596619755029678,\n      -0.0541486032307148,\n      0.11314000189304352,\n      0.014907633885741234,\n      -0.007224022410809994,\n      0.11486119776964188,\n      0.054845333099365234,\n      -0.03887084499001503,\n      0.13352756202220917,\n      -0.17630060017108917,\n      -0.12994974851608276,\n      -0.0272673349827528,\n      -0.06703335046768188,\n      0.11388465017080307,\n      0.09653828293085098,\n      0.07429727911949158,\n      0.05422600358724594,\n      0.0680130124092102,\n      0.055619802325963974,\n      -0.04253823682665825,\n      0.016511783003807068,\n      0.061226312071084976,\n      0.07431329041719437,\n      0.03244239464402199,\n      0.036058004945516586,\n      -0.19188059866428375,\n      -0.014687892980873585,\n      0.06427403539419174,\n      0.06020267680287361,\n      -0.045911844819784164,\n      0.08910705149173737,\n      0.10550247132778168,\n      -0.0022346528712660074,\n      0.048281945288181305,\n      0.11024229228496552,\n      -0.01508173905313015,\n      0.06795213371515274,\n      0.07403615862131119,\n      0.0245722196996212,\n      0.013750070706009865,\n      -0.025660578161478043,\n      -0.06704666465520859,\n      0.09702887386083603,\n      0.021814553067088127,\n      -0.001785883679986,\n      0.006187316495925188,\n      -0.09101739525794983,\n      0.009165314957499504,\n      0.010150427930057049,\n      -0.01985890045762062,\n      0.04724481701850891,\n      -0.06406835466623306,\n      0.024881398305296898,\n      0.01910281553864479,\n      -0.013923341408371925,\n      0.021392175927758217,\n      0.02630412019789219,\n      -0.007678450550884008,\n      0.08600278198719025,\n      0.07718589156866074,\n      0.01484794169664383,\n      -0.07928360253572464,\n      -0.017702415585517883,\n      0.04291711375117302,\n      -0.05597177520394325,\n      0.05882064253091812,\n      -0.02424810267984867,\n      -0.07536482065916061,\n      -0.1574518233537674,\n      -0.10490111261606216,\n      -0.07237518578767776,\n      -0.05295227840542793,\n      0.0026925986167043447,\n      0.08872181177139282,\n      -0.02840392477810383,\n      -0.03360121697187424,\n      0.039647072553634644,\n      -0.047242604196071625,\n      -0.07135595381259918,\n      -0.04258326441049576,\n      0.07301025837659836,\n      0.015565676614642143,\n      -0.04388121888041496,\n      -0.0009018126875162125,\n      -0.064618781208992,\n      0.03683130443096161,\n      -0.07320521026849747,\n      0.03480635583400726,\n      0.06498942524194717,\n      0.03287004679441452,\n      0.0813530758023262,\n      0.05868055671453476,\n      0.059881240129470825,\n      0.09170296043157578,\n      0.010385308414697647,\n      -0.05296289920806885,\n      0.07961296290159225,\n      -0.030376063659787178,\n      -0.01113503985106945,\n      0.009449789300560951,\n      0.051094383001327515,\n      0.02581794373691082,\n      0.06103668361902237,\n      0.03266970440745354,\n      0.08350498229265213,\n      0.051446519792079926,\n      -0.07176236808300018,\n      -0.06023673340678215,\n      0.07079079747200012,\n      -0.032740090042352676,\n      0.09842202812433243,\n      -0.11802073568105698,\n      0.03703346103429794,\n      -0.005117777269333601,\n      0.11787570267915726,\n      -0.059095464646816254,\n      -0.016833657398819923,\n      -4.862307105213404e-05,\n      0.004821947310119867,\n      -0.024362871423363686,\n      0.0857798308134079\n    ],\n    [\n      -0.047847896814346313,\n      -0.10410794615745544,\n      -0.034546300768852234,\n      0.05666043981909752,\n      0.06929642707109451,\n      0.11969850957393646,\n      -0.008525240235030651,\n      -0.05681030452251434,\n      0.09533634036779404,\n      0.0008839316433295608,\n      -0.0357990600168705,\n      0.013686668127775192,\n      0.0534142442047596,\n      0.10353917628526688,\n      0.1753372848033905,\n      0.08072547614574432,\n      0.11842704564332962,\n      -0.03947308287024498,\n      0.02715395763516426,\n      -0.06522481888532639,\n      0.09791846573352814,\n      -0.07040075957775116,\n      -0.017235297709703445,\n      -0.0001297151029575616,\n      0.036328453570604324,\n      -0.022417180240154266,\n      -0.08119463175535202,\n      -0.028521370142698288,\n      0.01053010206669569,\n      0.01716257631778717,\n      0.06539417803287506,\n      0.09697876870632172,\n      0.08037564903497696,\n      -0.13256199657917023,\n      -0.012222684919834137,\n      0.002029108814895153,\n      0.011790966615080833,\n      0.006774998269975185,\n      -0.04363089054822922,\n      0.02016456238925457,\n      -0.03051178902387619,\n      -0.05063200369477272,\n      0.04108308255672455,\n      -0.03802494704723358,\n      0.025335945188999176,\n      0.02994556352496147,\n      -0.035125646740198135,\n      0.11305864155292511,\n      0.13119104504585266,\n      0.05801386013627052,\n      0.0822732001543045,\n      0.07160430401563644,\n      -0.060375623404979706,\n      0.09317187964916229,\n      0.05708444491028786,\n      0.05708928778767586,\n      0.03190033137798309,\n      0.010944105684757233,\n      0.005934868007898331,\n      0.1071108877658844,\n      0.08686669915914536,\n      0.0005292967543937266,\n      0.0769476369023323,\n      0.03475193679332733,\n      0.04126428812742233,\n      -0.05187024548649788,\n      -0.07767526060342789,\n      0.10576853901147842,\n      0.040676310658454895,\n      0.02056577429175377,\n      0.025127755478024483,\n      -0.009838729165494442,\n      0.06438177078962326,\n      0.15177617967128754,\n      0.08304110169410706,\n      -0.08644142746925354,\n      0.10247557610273361,\n      0.07972098886966705,\n      -0.07865288853645325,\n      0.05567466840147972,\n      -0.006203130818903446,\n      0.002268770942464471,\n      0.0055863806046545506,\n      -0.04318665713071823,\n      0.013399318791925907,\n      0.058994948863983154,\n      0.0316762700676918,\n      0.05311425402760506,\n      -0.04466552287340164,\n      -0.04777340218424797,\n      0.08182356506586075,\n      0.041988346725702286,\n      -0.013437499292194843,\n      -0.02405627816915512,\n      -0.15261682868003845,\n      -0.006711347494274378,\n      -0.0015569353708997369,\n      0.06713628768920898,\n      0.04520397260785103,\n      0.07751747965812683,\n      0.037904780358076096,\n      -0.06960704177618027,\n      -0.031468965113162994,\n      0.08804164081811905,\n      -0.00895190890878439,\n      0.015985136851668358,\n      -0.0115635572001338,\n      0.012805485166609287,\n      -0.0760529413819313,\n      -0.06720343232154846,\n      -0.00845285877585411,\n      -0.028535859659314156,\n      -0.04905902221798897,\n      0.00019202218391001225,\n      -0.07026427984237671,\n      -0.095570869743824,\n      0.013895956799387932,\n      0.059706199914216995,\n      0.0264380294829607,\n      0.04691949114203453,\n      0.03465351089835167,\n      -0.005152381490916014,\n      0.05828449875116348,\n      0.05812690407037735,\n      0.024269144982099533\n    ],\n    [\n      -0.005107361823320389,\n      0.0685172900557518,\n      -0.01827196590602398,\n      0.01617792621254921,\n      -0.0217291209846735,\n      -0.061233896762132645,\n      -0.11306256800889969,\n      -0.1265772134065628,\n      0.040001802146434784,\n      -0.08705975860357285,\n      -0.08410818874835968,\n      0.03255649656057358,\n      -0.1300528347492218,\n      0.198642760515213,\n      0.07087044417858124,\n      -0.1006268635392189,\n      -0.037051890045404434,\n      -0.12109199166297913,\n      0.0453614704310894,\n      0.044849470257759094,\n      -0.05025681108236313,\n      -0.028599947690963745,\n      0.015753978863358498,\n      -0.009095095098018646,\n      -0.08782684057950974,\n      0.01080775074660778,\n      -0.1009906530380249,\n      0.00988788716495037,\n      0.017344634979963303,\n      0.22579185664653778,\n      -0.02884647436439991,\n      -0.04652037099003792,\n      0.10595404356718063,\n      0.016179200261831284,\n      -0.02264433167874813,\n      -0.07832032442092896,\n      -0.05219472944736481,\n      -0.0939122885465622,\n      -0.12743760645389557,\n      -0.038958221673965454,\n      0.0588851198554039,\n      -0.0818631500005722,\n      -0.10316963493824005,\n      -0.06401800364255905,\n      -0.07601509988307953,\n      0.0412149541079998,\n      0.08120401948690414,\n      -0.05544520542025566,\n      0.011387781240046024,\n      -0.01893800124526024,\n      0.04648488014936447,\n      -0.05143653228878975,\n      -0.02251179702579975,\n      -0.022064080461859703,\n      0.10223886370658875,\n      -0.03112695924937725,\n      -0.09553895145654678,\n      -0.01210714876651764,\n      0.007308559026569128,\n      -0.05173983424901962,\n      -0.04469456523656845,\n      -0.004327137488871813,\n      -0.08173143118619919,\n      0.07673683762550354,\n      0.04404217749834061,\n      -0.09242688864469528,\n      -0.0394480898976326,\n      -0.07414057850837708,\n      0.07665548473596573,\n      0.011037872172892094,\n      -0.0016319031128659844,\n      -0.008784063160419464,\n      0.04140924662351608,\n      0.053459394723176956,\n      0.008467861451208591,\n      0.01744273491203785,\n      -0.09202531725168228,\n      -0.041340362280607224,\n      0.12839652597904205,\n      -0.049119409173727036,\n      0.10106091946363449,\n      -0.030912667512893677,\n      -0.09728802740573883,\n      0.09435293823480606,\n      -0.06470952928066254,\n      0.0659940242767334,\n      0.06420562416315079,\n      0.04707222431898117,\n      -0.024235103279352188,\n      -0.07587216794490814,\n      -0.0024904103484004736,\n      0.027426550164818764,\n      -0.06082397699356079,\n      -0.0023309113457798958,\n      -0.03937395662069321,\n      -0.06180604174733162,\n      0.053244512528181076,\n      0.1288089156150818,\n      0.041595131158828735,\n      -0.01602226123213768,\n      -0.0851627066731453,\n      0.09149862080812454,\n      0.03896506130695343,\n      0.09330969303846359,\n      0.09111788123846054,\n      0.061440691351890564,\n      0.00026570024783723056,\n      0.010781032964587212,\n      0.09266436845064163,\n      -0.03406959027051926,\n      0.03202809393405914,\n      -0.02317616157233715,\n      -0.061625245958566666,\n      -0.050513166934251785,\n      -0.057240985333919525,\n      0.04314204305410385,\n      -0.08197309821844101,\n      0.031145047396421432,\n      -0.02315383031964302,\n      0.11517021805047989,\n      0.05071912333369255,\n      0.08156271278858185,\n      -0.015745270997285843,\n      0.021952209994196892,\n      -0.07570908963680267\n    ],\n    [\n      -0.013687822967767715,\n      0.07452581822872162,\n      0.117286816239357,\n      -0.03784388303756714,\n      -0.14422424137592316,\n      -0.0963008925318718,\n      0.13553613424301147,\n      0.07131203263998032,\n      -0.06676951050758362,\n      -0.01327117532491684,\n      -0.0027431014459580183,\n      0.10126164555549622,\n      0.1872570812702179,\n      -0.065677709877491,\n      -0.02936982735991478,\n      0.08437204360961914,\n      0.104399673640728,\n      0.119514100253582,\n      -0.08779849857091904,\n      0.08780532330274582,\n      -0.04486023262143135,\n      -0.013620052486658096,\n      0.049908947199583054,\n      0.10026585310697556,\n      -0.09120898693799973,\n      0.05995786935091019,\n      0.01663076877593994,\n      0.029090672731399536,\n      0.13541600108146667,\n      -0.18655197322368622,\n      0.14067772030830383,\n      0.030652562156319618,\n      0.05165758356451988,\n      0.10686953365802765,\n      -0.05812850594520569,\n      -0.15592460334300995,\n      0.04597870633006096,\n      -0.04193047806620598,\n      -0.07509021461009979,\n      0.0542704276740551,\n      -0.014512035064399242,\n      -0.001663304166868329,\n      0.09839372336864471,\n      -0.018958667293190956,\n      0.04154607653617859,\n      0.02987864799797535,\n      0.003165067872032523,\n      -0.013429266400635242,\n      0.010846598073840141,\n      -0.08577857911586761,\n      -0.13036781549453735,\n      -0.02300664596259594,\n      -0.01695595681667328,\n      0.008875477127730846,\n      -0.037560924887657166,\n      0.06928547471761703,\n      -0.18601390719413757,\n      -0.06511463969945908,\n      0.08753751218318939,\n      -0.01641247421503067,\n      0.08825802803039551,\n      -0.012616254389286041,\n      0.14165161550045013,\n      -0.04287315160036087,\n      -0.1020599752664566,\n      0.028416067361831665,\n      0.02076907269656658,\n      -0.002156957983970642,\n      -0.03431063890457153,\n      0.022069348022341728,\n      -0.1054704561829567,\n      -0.020056627690792084,\n      0.09328216314315796,\n      -0.18488912284374237,\n      -0.016573889181017876,\n      0.09264197200536728,\n      -0.032997071743011475,\n      0.07690837234258652,\n      0.058843906968832016,\n      0.09952575713396072,\n      -0.09119394421577454,\n      -0.09778441488742828,\n      0.04716477543115616,\n      -0.05531910061836243,\n      -0.026827650144696236,\n      -0.15823180973529816,\n      -0.03249547258019447,\n      -0.08231835067272186,\n      0.05526985973119736,\n      0.019207663834095,\n      0.07752428948879242,\n      0.0956399142742157,\n      0.044912125915288925,\n      0.10387744754552841,\n      -0.0653151422739029,\n      -0.13260522484779358,\n      -0.06594057381153107,\n      0.02496524155139923,\n      -0.062260162085294724,\n      0.02522704005241394,\n      -0.060694001615047455,\n      -0.13574747741222382,\n      0.054638128727674484,\n      -0.0574868768453598,\n      -0.07018931210041046,\n      -0.04012105241417885,\n      0.08671154081821442,\n      0.0045784758403897285,\n      -0.0643673911690712,\n      -0.0712207704782486,\n      0.038650933653116226,\n      0.06361178308725357,\n      -0.0028326755855232477,\n      -0.08391505479812622,\n      -0.08461447805166245,\n      -0.2514379620552063,\n      -0.05924651399254799,\n      0.0624600350856781,\n      0.020139524713158607,\n      -0.15562719106674194,\n      -0.134658083319664,\n      0.05985744297504425,\n      0.021327726542949677,\n      -0.081266850233078,\n      -0.054313790053129196\n    ],\n    [\n      -0.018560582771897316,\n      -0.07757974416017532,\n      -0.09754161536693573,\n      0.023436380550265312,\n      0.04045647382736206,\n      0.07191786170005798,\n      0.00579518498852849,\n      0.016801441088318825,\n      -0.08128499984741211,\n      -0.04973152279853821,\n      0.11699550598859787,\n      0.02253134734928608,\n      0.06767783313989639,\n      -0.023640302941203117,\n      -0.15018320083618164,\n      0.08605172485113144,\n      0.021232973784208298,\n      0.09529605507850647,\n      -0.06097685173153877,\n      -0.048579394817352295,\n      0.01929323561489582,\n      0.07579204440116882,\n      0.0547601655125618,\n      0.06928187608718872,\n      -0.027322258800268173,\n      -0.12104567885398865,\n      0.03692604973912239,\n      0.04569822549819946,\n      0.09251970052719116,\n      -0.08293972164392471,\n      0.04225005954504013,\n      -0.01358475349843502,\n      0.0719446912407875,\n      0.05888860672712326,\n      -0.025404928252100945,\n      0.07769209146499634,\n      0.05718695744872093,\n      0.09549509733915329,\n      0.053646355867385864,\n      0.03272131085395813,\n      -0.06765256077051163,\n      -0.020624326542019844,\n      0.11385059356689453,\n      0.07071666419506073,\n      0.09573407471179962,\n      0.049041759222745895,\n      -0.006621317472308874,\n      0.023651883006095886,\n      0.058199163526296616,\n      0.045980606228113174,\n      -0.08143696933984756,\n      -0.07017616927623749,\n      0.04504174366593361,\n      0.06038608029484749,\n      -0.01741766557097435,\n      -0.04842592775821686,\n      -0.03835819289088249,\n      -0.05624690651893616,\n      -0.034047145396471024,\n      0.039060112088918686,\n      -0.036763083189725876,\n      0.0622640885412693,\n      0.02383417822420597,\n      -0.00400477834045887,\n      -0.009426584467291832,\n      0.01297449879348278,\n      0.014921442605555058,\n      -0.015233478508889675,\n      -0.035430435091257095,\n      -0.0958566665649414,\n      0.02936403825879097,\n      0.06127613037824631,\n      -0.15529534220695496,\n      0.006548361387103796,\n      -0.11259890347719193,\n      -0.011621142737567425,\n      -0.10464243590831757,\n      -0.014469930902123451,\n      -0.07153617590665817,\n      0.0743512436747551,\n      0.022243935614824295,\n      0.07151453197002411,\n      0.06087096780538559,\n      -0.036595668643713,\n      -0.007508631329983473,\n      -0.04680867865681648,\n      0.04501275718212128,\n      -0.010179383680224419,\n      -0.034941621124744415,\n      -0.05518393963575363,\n      0.01272139884531498,\n      0.06970727443695068,\n      -0.08395775407552719,\n      -0.07815726846456528,\n      0.0788920596241951,\n      0.06504657864570618,\n      0.00587754650041461,\n      -0.05575501173734665,\n      0.008075601421296597,\n      0.0024453015066683292,\n      -0.060464490205049515,\n      -0.044570304453372955,\n      -0.08846930414438248,\n      -0.005290632601827383,\n      -0.0022637785878032446,\n      -0.07961823791265488,\n      0.059618089348077774,\n      0.10487290471792221,\n      0.11050716042518616,\n      0.08081524819135666,\n      -0.013829657807946205,\n      -0.08809985220432281,\n      0.03154972568154335,\n      0.07798590511083603,\n      0.005124774295836687,\n      -0.07896603643894196,\n      0.040920134633779526,\n      0.003404076676815748,\n      -0.060004185885190964,\n      -0.0020617989357560873,\n      0.021441185846924782,\n      -0.03735829517245293,\n      0.012479035183787346,\n      0.03928130120038986,\n      -0.05023251473903656\n    ],\n    [\n      0.06605260074138641,\n      -0.03702326864004135,\n      -0.11264515668153763,\n      -0.06509378552436829,\n      0.05416831374168396,\n      0.08623805642127991,\n      0.03345106169581413,\n      0.02886616438627243,\n      -0.11418618261814117,\n      0.11240828037261963,\n      0.09042894840240479,\n      0.07303430140018463,\n      0.14198288321495056,\n      -0.12439817190170288,\n      -0.16091586649417877,\n      0.03987317159771919,\n      -0.018807025626301765,\n      0.10898856073617935,\n      0.004377422388643026,\n      0.07559492439031601,\n      -0.01052517257630825,\n      0.06460600346326828,\n      0.08895141631364822,\n      0.03313251957297325,\n      -0.027016794309020042,\n      0.005185647401958704,\n      0.057608712464571,\n      -0.01944052055478096,\n      0.006593190133571625,\n      -0.23056530952453613,\n      0.0297564584761858,\n      -0.08851896971464157,\n      -0.0075696296989917755,\n      0.0007011947454884648,\n      0.061855219304561615,\n      -0.04745146259665489,\n      0.05343378707766533,\n      0.08730711787939072,\n      -0.002999016782268882,\n      0.09086181223392487,\n      0.053318459540605545,\n      0.0028472363483160734,\n      0.016311589628458023,\n      0.012267326936125755,\n      -0.029535071924328804,\n      0.012857339344918728,\n      0.08549783378839493,\n      0.03586013615131378,\n      0.061868805438280106,\n      -0.030004022642970085,\n      -0.09074623882770538,\n      -0.019866803660988808,\n      0.07098911702632904,\n      0.09269899874925613,\n      0.05702421814203262,\n      0.008232863619923592,\n      0.05898416414856911,\n      0.015376085415482521,\n      -0.016585731878876686,\n      0.07198070734739304,\n      -0.04594346135854721,\n      0.007030070759356022,\n      -0.005835060030221939,\n      -0.01631765626370907,\n      0.07569897919893265,\n      -0.0057058935053646564,\n      0.03821682184934616,\n      0.01924484595656395,\n      -0.07522491365671158,\n      0.02191515639424324,\n      0.000620119331870228,\n      0.01683334819972515,\n      -0.15624406933784485,\n      -0.06386465579271317,\n      0.035816241055727005,\n      0.005442994646728039,\n      0.06361132115125656,\n      0.07254824042320251,\n      -0.060140855610370636,\n      0.05773945525288582,\n      0.006662753876298666,\n      0.07055645436048508,\n      0.09370593726634979,\n      -0.023038547486066818,\n      -0.01496174931526184,\n      0.039748892188072205,\n      0.011143343523144722,\n      0.06916752457618713,\n      -0.06319739669561386,\n      0.008346031419932842,\n      0.04990794137120247,\n      0.025317294523119926,\n      0.027587538585066795,\n      0.08691620826721191,\n      -0.013775680214166641,\n      -0.03719594329595566,\n      0.010921076871454716,\n      0.07168667763471603,\n      0.005269194021821022,\n      -0.057110745459795,\n      0.08967172354459763,\n      0.006357895210385323,\n      0.06718839704990387,\n      0.02736879698932171,\n      0.08890016376972198,\n      -0.03657522797584534,\n      -0.02688629925251007,\n      0.12207973003387451,\n      0.05532030388712883,\n      0.06425916403532028,\n      -0.06936950981616974,\n      0.029375407844781876,\n      -0.0393289178609848,\n      0.09492047876119614,\n      0.01957898959517479,\n      -0.10671056807041168,\n      0.08628635108470917,\n      0.019976478070020676,\n      0.06356053054332733,\n      0.020391788333654404,\n      -0.16459189355373383,\n      -0.031544480472803116,\n      -0.03076459839940071,\n      0.015290371142327785,\n      -0.057035744190216064\n    ],\n    [\n      0.018054401502013206,\n      -0.06201259791851044,\n      -0.04567059874534607,\n      -0.07882440835237503,\n      -0.005356860347092152,\n      0.07291697710752487,\n      -0.03823421522974968,\n      -0.01044849306344986,\n      -0.04969818517565727,\n      0.07251584529876709,\n      -0.06770394742488861,\n      0.10784609615802765,\n      -0.046900227665901184,\n      -0.15194839239120483,\n      -0.0973687618970871,\n      -0.014537733048200607,\n      0.03999560698866844,\n      0.03153334930539131,\n      -0.14384445548057556,\n      0.07998205721378326,\n      0.01696142740547657,\n      0.05523478984832764,\n      -0.009789347648620605,\n      0.007141043897718191,\n      -0.019960928708314896,\n      -0.01100953109562397,\n      0.049624811857938766,\n      -0.06887494772672653,\n      -0.030748417600989342,\n      -0.2370203137397766,\n      0.03998022526502609,\n      -0.018274443224072456,\n      0.05574192851781845,\n      0.06916225701570511,\n      -0.06641826778650284,\n      -0.01518599595874548,\n      -0.061741650104522705,\n      0.12011626362800598,\n      0.008878122083842754,\n      -0.017958154901862144,\n      0.05319203436374664,\n      0.11412117630243301,\n      0.06543266028165817,\n      -0.007290767040103674,\n      0.001227248110808432,\n      0.018171869218349457,\n      0.006622934248298407,\n      -0.00560428062453866,\n      0.08322124183177948,\n      0.05613706633448601,\n      0.009674652479588985,\n      0.05413937568664551,\n      -0.10370071232318878,\n      -0.04075360670685768,\n      -0.10954184085130692,\n      -0.10558422654867172,\n      0.03688361868262291,\n      0.08756448328495026,\n      0.1137862578034401,\n      0.08030254393815994,\n      0.0566854365170002,\n      -0.021460793912410736,\n      0.10205510258674622,\n      0.008962742052972317,\n      -0.020597049966454506,\n      -0.06529069691896439,\n      -0.09180419147014618,\n      -0.024262582883238792,\n      0.03577865660190582,\n      0.014420421794056892,\n      0.07315552979707718,\n      -0.061920106410980225,\n      -0.07156504690647125,\n      0.0285896435379982,\n      -0.06018328666687012,\n      -0.038880717009305954,\n      0.06365034729242325,\n      -0.059413716197013855,\n      -0.11137183010578156,\n      0.03416822850704193,\n      0.08538854867219925,\n      0.12495369464159012,\n      -0.020394504070281982,\n      -0.06658604741096497,\n      0.023028723895549774,\n      -0.09913178533315659,\n      0.012225918471813202,\n      -0.06690016388893127,\n      0.01636248454451561,\n      0.009411757811903954,\n      -0.05723559856414795,\n      0.06148979812860489,\n      -0.0391145795583725,\n      -0.0013798812869936228,\n      -0.03365456312894821,\n      -0.008753308095037937,\n      -0.019828010350465775,\n      -0.055865947157144547,\n      -0.07923386991024017,\n      -0.09071522951126099,\n      0.005042409524321556,\n      0.09489922970533371,\n      0.11285287886857986,\n      0.06607546657323837,\n      0.014193505980074406,\n      -0.13342836499214172,\n      0.0259204413741827,\n      -0.02045198529958725,\n      -0.02076135203242302,\n      0.11736583709716797,\n      0.0746108740568161,\n      -0.10180997848510742,\n      0.12484470009803772,\n      -0.06790100038051605,\n      -0.0974203571677208,\n      -0.1585090458393097,\n      -0.07331804931163788,\n      0.11236497014760971,\n      0.03578046336770058,\n      -0.02730666846036911,\n      -0.04297502711415291,\n      -0.042107194662094116,\n      -0.16694684326648712,\n      -0.02501698210835457,\n      0.09279551357030869\n    ],\n    [\n      0.0026247415225952864,\n      0.03534970059990883,\n      -0.058353595435619354,\n      0.07078508287668228,\n      0.09295499324798584,\n      -0.07837516069412231,\n      -0.022075453773140907,\n      -0.059469230473041534,\n      -0.06617985665798187,\n      -0.003602013224735856,\n      -0.03967547416687012,\n      0.014765970408916473,\n      -0.07056257128715515,\n      0.18349270522594452,\n      0.023224620148539543,\n      -0.11301218718290329,\n      0.04855324327945709,\n      0.0077246068976819515,\n      -0.0670488178730011,\n      -0.02038787677884102,\n      -0.06941308081150055,\n      0.004395098425447941,\n      -0.04557499289512634,\n      -0.032735567539930344,\n      0.01808682456612587,\n      -0.08332253992557526,\n      0.09298966079950333,\n      0.0713294968008995,\n      -0.03309044986963272,\n      0.2076277732849121,\n      -0.037210699170827866,\n      0.007684135343879461,\n      0.019286299124360085,\n      0.13690316677093506,\n      0.04492403194308281,\n      0.0024516303092241287,\n      0.09751887619495392,\n      -0.09416814148426056,\n      -0.07088490575551987,\n      -0.02252117730677128,\n      -0.0032289817463606596,\n      -0.1174125224351883,\n      -0.026169076561927795,\n      0.08174988627433777,\n      -0.027644244953989983,\n      0.037109289318323135,\n      0.05295851081609726,\n      -0.022324904799461365,\n      0.09823944419622421,\n      -0.03994442895054817,\n      0.07687681168317795,\n      0.03313426673412323,\n      0.08856111764907837,\n      0.1075519323348999,\n      0.03799675405025482,\n      -0.08594044297933578,\n      -0.07902839034795761,\n      -0.01474352553486824,\n      0.007552556227892637,\n      -0.08166900277137756,\n      -0.08679058402776718,\n      0.014901731163263321,\n      0.018549518659710884,\n      0.16228850185871124,\n      -0.01044717151671648,\n      -0.0865030288696289,\n      -0.03306785970926285,\n      -0.014892859384417534,\n      0.03523676469922066,\n      0.10101306438446045,\n      -0.12689653038978577,\n      -0.16000010073184967,\n      0.10065010190010071,\n      0.04682017117738724,\n      0.0900609940290451,\n      -0.023763278499245644,\n      0.022003155201673508,\n      0.007689302321523428,\n      -0.09357818961143494,\n      -0.024330992251634598,\n      0.06572005152702332,\n      -0.01004147157073021,\n      -0.04989182576537132,\n      0.08939048647880554,\n      0.03298884257674217,\n      0.021175798028707504,\n      0.039256442338228226,\n      0.037884313613176346,\n      0.040338847786188126,\n      0.028406109660863876,\n      -0.09829125553369522,\n      -0.07527656108140945,\n      0.07049860060214996,\n      0.09773261100053787,\n      0.05060148984193802,\n      -0.028489738702774048,\n      0.023274065926671028,\n      -0.01677529327571392,\n      -0.11133158206939697,\n      -0.07794102281332016,\n      -0.03080390766263008,\n      -0.07864819467067719,\n      0.02595624327659607,\n      -0.007385646924376488,\n      0.028575677424669266,\n      0.013026606291532516,\n      0.03529123216867447,\n      -0.18805353343486786,\n      0.12901878356933594,\n      0.05765102431178093,\n      0.031180912628769875,\n      -0.00982521753758192,\n      -0.11320733278989792,\n      -0.09522691369056702,\n      0.06481624394655228,\n      0.10993544012308121,\n      0.001458154059946537,\n      -0.0218609981238842,\n      -0.008840315043926239,\n      -0.12328913807868958,\n      0.1808139532804489,\n      0.08696865290403366,\n      0.053572338074445724,\n      0.09709780663251877,\n      -0.15806666016578674\n    ],\n    [\n      -0.0506446473300457,\n      0.008125810883939266,\n      0.030635710805654526,\n      -0.08176034688949585,\n      0.05530053377151489,\n      0.024953380227088928,\n      -0.11128146201372147,\n      -0.06862921267747879,\n      0.1281156837940216,\n      -0.04005904495716095,\n      0.04979374259710312,\n      -0.13441279530525208,\n      -0.07046426087617874,\n      0.09713329374790192,\n      0.1419832408428192,\n      0.0700073316693306,\n      -0.0009261462837457657,\n      -0.04239599034190178,\n      0.059010617434978485,\n      -0.058068398386240005,\n      -0.01221531257033348,\n      0.02201814576983452,\n      -0.05396200343966484,\n      0.039298344403505325,\n      -0.07889560610055923,\n      -0.04976952075958252,\n      -0.02974425069987774,\n      0.05155656114220619,\n      0.06375063955783844,\n      0.10494145005941391,\n      -0.046760402619838715,\n      0.0393863171339035,\n      0.021111026406288147,\n      -0.04700859263539314,\n      -0.1055004745721817,\n      -0.003310379572212696,\n      -0.03709980100393295,\n      -0.10516849160194397,\n      -0.005993749015033245,\n      -0.05739574879407883,\n      0.005376241635531187,\n      -0.016917085275053978,\n      -0.10431549698114395,\n      0.06151590496301651,\n      0.020307982340455055,\n      -0.05593140795826912,\n      0.06338705867528915,\n      -0.01512826792895794,\n      -0.03813288360834122,\n      -0.07794956117868423,\n      0.0776805728673935,\n      0.018275119364261627,\n      0.04709266871213913,\n      0.04806259274482727,\n      -0.03062446042895317,\n      0.109935462474823,\n      -0.007902896031737328,\n      -0.0495988130569458,\n      0.06747246533632278,\n      0.025729980319738388,\n      0.01294059306383133,\n      -0.04400556534528732,\n      -0.07998084276914597,\n      0.0040514832362532616,\n      -0.05747777223587036,\n      0.001145732938311994,\n      -0.005707215517759323,\n      0.04414715617895126,\n      -0.04852219671010971,\n      -0.016233421862125397,\n      0.004286172334104776,\n      0.04499787464737892,\n      0.06894701719284058,\n      -0.020118949934840202,\n      0.025286979973316193,\n      0.11160778999328613,\n      -0.014247010461986065,\n      -0.027354726567864418,\n      0.08063136786222458,\n      -0.04324152693152428,\n      -0.03694560006260872,\n      0.006406186148524284,\n      0.020142510533332825,\n      -0.02801825851202011,\n      0.06038327515125275,\n      0.11837468296289444,\n      -0.037268996238708496,\n      0.029843645170331,\n      -0.012577465735375881,\n      0.024685174226760864,\n      0.021718893200159073,\n      -0.06256577372550964,\n      0.06197815015912056,\n      -0.07558868080377579,\n      0.07781665027141571,\n      0.04018473997712135,\n      -0.025109779089689255,\n      0.046390350908041,\n      0.05493249371647835,\n      0.10011845827102661,\n      0.06800904870033264,\n      0.02799299545586109,\n      0.03883006051182747,\n      0.030942730605602264,\n      0.06412367522716522,\n      -0.02513570338487625,\n      -0.001964984927326441,\n      -0.09565484523773193,\n      0.0849657878279686,\n      0.09045413881540298,\n      -0.09945522248744965,\n      0.013530995696783066,\n      -0.04109606891870499,\n      -0.12592655420303345,\n      0.00021282723173499107,\n      -0.18052318692207336,\n      -0.053981684148311615,\n      -0.03116830624639988,\n      0.09784240275621414,\n      0.0505053885281086,\n      0.017884988337755203,\n      0.12131158262491226,\n      -0.04177180305123329,\n      0.07341297715902328,\n      0.0031339111737906933\n    ],\n    [\n      -0.045783881098032,\n      0.06048193946480751,\n      0.1069350466132164,\n      -0.0635341927409172,\n      0.11150216311216354,\n      -0.025763077661395073,\n      -0.08088222146034241,\n      -0.08775746077299118,\n      0.09485995024442673,\n      0.0036179115995764732,\n      -0.04333471134305,\n      0.009101355448365211,\n      -0.037309471517801285,\n      -0.017334671691060066,\n      0.1497925966978073,\n      -0.06087367609143257,\n      0.05755520239472389,\n      -0.06131358817219734,\n      -0.01382676512002945,\n      -0.07953105866909027,\n      0.005201619118452072,\n      -0.005429064389318228,\n      0.030539078637957573,\n      -0.003941524773836136,\n      0.0635436400771141,\n      -0.02198597975075245,\n      0.0472930446267128,\n      0.00094892259221524,\n      0.018103009089827538,\n      0.10270584374666214,\n      0.06435119360685349,\n      -0.07237453758716583,\n      0.040377914905548096,\n      0.002808336168527603,\n      -0.08537251502275467,\n      -0.05864696204662323,\n      0.03600930795073509,\n      -0.061010681092739105,\n      -0.008121936582028866,\n      -0.07401346415281296,\n      0.015061899088323116,\n      -0.08000924438238144,\n      -0.06467249244451523,\n      0.004153602756559849,\n      0.04969988018274307,\n      0.09033717960119247,\n      0.003715402213856578,\n      -0.02001996338367462,\n      0.005493465345352888,\n      0.06265858560800552,\n      -0.021509822458028793,\n      0.04054221138358116,\n      0.04417959228157997,\n      0.009104137308895588,\n      0.09706823527812958,\n      0.1070261225104332,\n      -0.09082581102848053,\n      -0.07337827980518341,\n      0.019257351756095886,\n      -0.05371425673365593,\n      0.010163262486457825,\n      0.02806147374212742,\n      -0.08722753077745438,\n      0.017911141738295555,\n      0.06390321999788284,\n      -0.021779119968414307,\n      0.09826262295246124,\n      0.04765241965651512,\n      0.010964538902044296,\n      0.07944930344820023,\n      0.021260177716612816,\n      -0.04233188182115555,\n      0.023187147453427315,\n      0.07528480142354965,\n      0.12583154439926147,\n      0.01583578623831272,\n      0.02414695732295513,\n      -0.09276692569255829,\n      0.06298355013132095,\n      0.027511386200785637,\n      0.04486563429236412,\n      0.05940066650509834,\n      0.09358860552310944,\n      0.05143273249268532,\n      -0.03839961066842079,\n      0.008524714037775993,\n      -0.028682943433523178,\n      -0.05554312467575073,\n      0.03954099118709564,\n      0.050365205854177475,\n      0.0764993205666542,\n      -0.07763571292161942,\n      -0.040987882763147354,\n      -0.013987365178763866,\n      0.07555780559778214,\n      0.06442534923553467,\n      0.09475339949131012,\n      0.06227942183613777,\n      0.11940988153219223,\n      6.488274084404111e-05,\n      0.008466262370347977,\n      0.0746055394411087,\n      -0.049492258578538895,\n      0.03668366000056267,\n      -0.004858308471739292,\n      0.08243191987276077,\n      0.01581573486328125,\n      0.0352686382830143,\n      0.0937819853425026,\n      0.07666473090648651,\n      -0.00249467883259058,\n      0.012702999636530876,\n      0.07178386300802231,\n      -0.019883690401911736,\n      -0.02866273559629917,\n      -0.13042770326137543,\n      0.07523488253355026,\n      0.04415399953722954,\n      -0.04454859346151352,\n      0.040891315788030624,\n      0.17288076877593994,\n      0.010854561813175678,\n      0.07968653738498688,\n      -0.0035513488110154867,\n      0.03738730400800705\n    ],\n    [\n      -0.004723675549030304,\n      -0.08036410063505173,\n      0.052364252507686615,\n      -0.03153814747929573,\n      0.022748887538909912,\n      0.031774114817380905,\n      -0.02197747491300106,\n      0.0015741197858005762,\n      0.03601403161883354,\n      -0.06938895583152771,\n      0.015081986784934998,\n      0.07018114626407623,\n      -0.08813024312257767,\n      0.15028777718544006,\n      0.1291060596704483,\n      0.03615087643265724,\n      0.030619105324149132,\n      -0.0460062138736248,\n      -0.015567893162369728,\n      -0.030696356669068336,\n      0.013449383899569511,\n      0.08312731236219406,\n      -0.023635435849428177,\n      -0.004754056688398123,\n      -0.06122136861085892,\n      0.04507090523838997,\n      -0.03364594653248787,\n      0.05531832575798035,\n      0.06347768753767014,\n      0.1051550954580307,\n      0.04509075731039047,\n      -0.011856675148010254,\n      0.06647279113531113,\n      0.06430009007453918,\n      -0.14084303379058838,\n      -0.01582038216292858,\n      0.04999076947569847,\n      -0.11635138839483261,\n      -0.0793115571141243,\n      -0.04302888363599777,\n      -0.08481701463460922,\n      0.08368955552577972,\n      -0.033530134707689285,\n      0.021806538105010986,\n      -0.08035152405500412,\n      0.002725393045693636,\n      0.0727725401520729,\n      0.09833770990371704,\n      -0.05299970135092735,\n      0.017013918608427048,\n      0.04478180408477783,\n      0.10593435913324356,\n      -0.047020770609378815,\n      0.08976863324642181,\n      -0.03181454539299011,\n      0.09032907336950302,\n      0.05687625706195831,\n      -0.06623724102973938,\n      -0.07858473062515259,\n      -0.05186508595943451,\n      0.10591176897287369,\n      -0.025513112545013428,\n      -0.011531632393598557,\n      -0.08378593623638153,\n      -0.0642767995595932,\n      -0.006202922202646732,\n      0.05906121805310249,\n      0.03728024289011955,\n      0.004289540927857161,\n      0.08619546890258789,\n      0.10014981031417847,\n      -0.06395446509122849,\n      0.06215522065758705,\n      0.08201133459806442,\n      -0.041456304490566254,\n      -0.04296891763806343,\n      0.09651581943035126,\n      0.039620883762836456,\n      0.08420711010694504,\n      0.06559630483388901,\n      0.017168935388326645,\n      0.012401217594742775,\n      0.06657417118549347,\n      -0.047036346048116684,\n      -0.012053615413606167,\n      0.09645979106426239,\n      0.09561153501272202,\n      -0.018179215490818024,\n      -0.0516500324010849,\n      -0.08209365606307983,\n      -0.03688952699303627,\n      -0.009727833792567253,\n      -0.14293761551380157,\n      0.07190343737602234,\n      0.08514445275068283,\n      0.05557002127170563,\n      -0.019162749871611595,\n      -0.04196503758430481,\n      0.09493610262870789,\n      0.0530807264149189,\n      0.0909326896071434,\n      0.04206961765885353,\n      0.06727473437786102,\n      0.056669533252716064,\n      -0.0002751292195171118,\n      0.007061529438942671,\n      0.007822615094482899,\n      0.01172499917447567,\n      0.04936783015727997,\n      -0.09380173683166504,\n      -0.05249002203345299,\n      -0.00879999715834856,\n      -0.03692888841032982,\n      -0.06327523291110992,\n      0.09664179384708405,\n      -0.15373001992702484,\n      0.029175415635108948,\n      -0.04368261620402336,\n      -0.0577399842441082,\n      0.1209787130355835,\n      0.0897693783044815,\n      0.10924234986305237,\n      -0.0748409554362297,\n      0.06066925451159477,\n      -0.06017230078577995\n    ],\n    [\n      -0.03767332807183266,\n      0.018232181668281555,\n      0.07304468005895615,\n      0.06242057681083679,\n      0.030453002080321312,\n      -0.08967990428209305,\n      -0.1076517179608345,\n      -0.17407742142677307,\n      0.17750956118106842,\n      -0.07714097946882248,\n      -0.027134940028190613,\n      -0.011407161131501198,\n      -0.12378942966461182,\n      0.20363077521324158,\n      0.1325230896472931,\n      -0.06691551953554153,\n      0.07725005596876144,\n      -0.13127349317073822,\n      -0.04822223260998726,\n      -0.04479075223207474,\n      0.050965044647455215,\n      0.01338601391762495,\n      0.06477810442447662,\n      -0.09176486730575562,\n      -0.08913277834653854,\n      -0.08401525020599365,\n      -0.15910093486309052,\n      -0.008853280916810036,\n      -0.010218971408903599,\n      0.2604008913040161,\n      0.03050038404762745,\n      -0.036072421818971634,\n      0.0786481648683548,\n      0.03130250796675682,\n      -0.030177513137459755,\n      -0.05978816747665405,\n      -0.017594091594219208,\n      -0.027085259556770325,\n      -0.03396153822541237,\n      0.0054904744029045105,\n      -0.02689056470990181,\n      0.048517197370529175,\n      -0.07681705057621002,\n      0.07382693886756897,\n      0.04175744578242302,\n      0.005269601009786129,\n      0.015856385231018066,\n      0.0772809162735939,\n      -0.016445305198431015,\n      -0.03510864078998566,\n      -0.02182159572839737,\n      -0.00045446009607985616,\n      -0.024164937436580658,\n      0.05514949560165405,\n      0.06800686568021774,\n      -0.06767833232879639,\n      -0.07584727555513382,\n      -0.004619581624865532,\n      -0.041183020919561386,\n      -0.048341114073991776,\n      0.022231221199035645,\n      0.023583661764860153,\n      0.01587735116481781,\n      0.016177695244550705,\n      0.06677529215812683,\n      0.02464737743139267,\n      -0.010303249582648277,\n      -0.02505364827811718,\n      0.07989760488271713,\n      -0.005131063982844353,\n      0.0033897238317877054,\n      -0.054709672927856445,\n      0.011984339915215969,\n      0.022081658244132996,\n      -0.003929984290152788,\n      -0.017504986375570297,\n      -0.019038526341319084,\n      -0.11071435362100601,\n      0.13062527775764465,\n      -0.0954262912273407,\n      -0.009727554395794868,\n      -0.04863959550857544,\n      -0.09915865212678909,\n      0.05373912304639816,\n      0.04584581032395363,\n      0.13750572502613068,\n      0.02680143155157566,\n      0.013328381814062595,\n      -0.04738406091928482,\n      -0.05300597846508026,\n      0.04971827566623688,\n      -0.03199427202343941,\n      0.06377620995044708,\n      -0.05150672048330307,\n      -0.06855276226997375,\n      -0.029579252004623413,\n      0.06638199836015701,\n      -0.004858703352510929,\n      0.03357769548892975,\n      0.06380388140678406,\n      -0.007281584199517965,\n      0.009898772463202477,\n      0.10380959510803223,\n      0.11413987725973129,\n      -0.013783704489469528,\n      0.02374660223722458,\n      0.037575192749500275,\n      0.0573112927377224,\n      0.0110589899122715,\n      -0.021947244182229042,\n      -0.01730341464281082,\n      -0.03440934047102928,\n      0.007178173400461674,\n      -0.029728839173913002,\n      0.063576839864254,\n      -0.009969817474484444,\n      0.015640344470739365,\n      0.04728630185127258,\n      0.06738103926181793,\n      0.03706095740199089,\n      0.1148085743188858,\n      -0.0028542252257466316,\n      -0.01025602500885725,\n      0.10788068920373917,\n      -0.06985357403755188\n    ],\n    [\n      -0.02528819814324379,\n      0.017757821828126907,\n      0.06100120395421982,\n      -0.04222201555967331,\n      -0.08660691231489182,\n      0.07347869127988815,\n      0.07541188597679138,\n      0.07312304526567459,\n      0.029123324900865555,\n      0.055124152451753616,\n      0.00986920390278101,\n      0.0009205740643665195,\n      0.03130185604095459,\n      -0.04344666376709938,\n      0.049504559487104416,\n      0.127476304769516,\n      0.09359144419431686,\n      0.030763546004891396,\n      -0.09459912031888962,\n      -0.11156585067510605,\n      -0.057491742074489594,\n      -0.1073087602853775,\n      -0.09591430425643921,\n      0.012751156464219093,\n      -0.0006701715174131095,\n      -0.06638513505458832,\n      -0.11826488375663757,\n      -0.10993146151304245,\n      0.0447489358484745,\n      -0.024734079837799072,\n      0.08898872137069702,\n      -0.027187662199139595,\n      0.006223367527127266,\n      0.16226254403591156,\n      -0.03000970184803009,\n      -0.02776940166950226,\n      0.014636918902397156,\n      0.004933745600283146,\n      -0.04449857026338577,\n      -0.0005816279444843531,\n      0.019153844565153122,\n      0.0075081405229866505,\n      0.03604980930685997,\n      -0.04770442843437195,\n      -0.05463213473558426,\n      -0.0183276254683733,\n      0.0055761998519301414,\n      -0.1068495512008667,\n      -0.15181563794612885,\n      0.08484891802072525,\n      -0.10585194826126099,\n      -0.04026293754577637,\n      -0.11662616580724716,\n      -0.11903373897075653,\n      -0.14029750227928162,\n      0.08268237859010696,\n      -0.10240805149078369,\n      -0.15511126816272736,\n      -0.038499098271131516,\n      -0.11032092571258545,\n      0.10596245527267456,\n      0.05469352379441261,\n      -0.0722724199295044,\n      0.01651620678603649,\n      -0.014225351624190807,\n      -0.06108574941754341,\n      0.01935470476746559,\n      -0.0058889444917440414,\n      0.04139629751443863,\n      -0.06070549786090851,\n      -0.03369387611746788,\n      -0.04485679790377617,\n      0.12014932930469513,\n      0.00786858331412077,\n      -0.00461482722312212,\n      -0.14472341537475586,\n      -0.10304269194602966,\n      -0.047843851149082184,\n      -0.09543288499116898,\n      -0.13684114813804626,\n      -0.025925204157829285,\n      -0.15452122688293457,\n      -0.042413901537656784,\n      -0.14285242557525635,\n      0.024370914325118065,\n      0.0021952237002551556,\n      0.07134847342967987,\n      -0.013695851899683475,\n      -0.02907009795308113,\n      0.125506192445755,\n      0.08039231598377228,\n      -0.05632466450333595,\n      0.04362374544143677,\n      0.020889418199658394,\n      0.036155398935079575,\n      0.05189219489693642,\n      -0.06371606886386871,\n      -0.06181757152080536,\n      -0.03005295805633068,\n      -0.035120099782943726,\n      -0.12642252445220947,\n      -0.048390571027994156,\n      0.07727810740470886,\n      -0.0746079832315445,\n      0.022202104330062866,\n      -0.09344041347503662,\n      0.0106425229460001,\n      0.003401101566851139,\n      -0.1004578024148941,\n      -0.08811590820550919,\n      0.033500757068395615,\n      -0.03791021928191185,\n      0.019276488572359085,\n      -0.0210020262748003,\n      -0.008038575761020184,\n      -0.05607660487294197,\n      -0.04808548837900162,\n      0.03631150349974632,\n      -0.060569629073143005,\n      -0.08000639826059341,\n      -0.00645030103623867,\n      -0.0918378233909607,\n      -0.0706561878323555,\n      -0.14515744149684906,\n      -0.06969502568244934\n    ],\n    [\n      0.055382415652275085,\n      -0.0032224461901932955,\n      0.05462569370865822,\n      0.10234887152910233,\n      -0.07441328465938568,\n      -0.05035421624779701,\n      -0.09524410963058472,\n      -0.10929691046476364,\n      0.0997558981180191,\n      -0.023755159229040146,\n      -0.024364804849028587,\n      0.04717373102903366,\n      -0.10666738450527191,\n      -0.10435570776462555,\n      0.15810184180736542,\n      -0.009051820263266563,\n      -0.05787443369626999,\n      -0.12289056926965714,\n      0.01079384796321392,\n      0.0037238947115838528,\n      -0.04031728580594063,\n      0.05445193499326706,\n      -0.01796029508113861,\n      -0.03232867270708084,\n      -0.13942012190818787,\n      -0.07996700704097748,\n      -0.031507257372140884,\n      0.053563836961984634,\n      -0.08576630800962448,\n      0.04441490024328232,\n      0.02199183963239193,\n      -0.058792874217033386,\n      -0.025634977966547012,\n      -0.06636358797550201,\n      -0.030121460556983948,\n      -0.00786585547029972,\n      -0.028445003554224968,\n      0.04488357529044151,\n      0.0030816274229437113,\n      -0.020311782136559486,\n      0.06483443081378937,\n      -0.11314374208450317,\n      0.0006688673747703433,\n      -0.08050563186407089,\n      -0.09075632691383362,\n      0.03042006865143776,\n      -0.009237216785550117,\n      -0.0109871169552207,\n      -0.05786934867501259,\n      -0.11066321283578873,\n      -0.03403320163488388,\n      0.05425756052136421,\n      -0.13896147906780243,\n      -0.07856417447328568,\n      -6.629549898207188e-05,\n      0.07201504707336426,\n      0.013378375209867954,\n      -0.005830259993672371,\n      -0.010929090902209282,\n      0.04914621636271477,\n      0.060639411211013794,\n      -0.04935802146792412,\n      -0.08448820561170578,\n      0.0933474451303482,\n      0.03774402290582657,\n      0.0043326485902071,\n      0.03819737583398819,\n      0.0003373335348442197,\n      -0.05708421766757965,\n      -0.04824380204081535,\n      0.120905302464962,\n      0.0016223826678469777,\n      0.03623500466346741,\n      0.05299072712659836,\n      -0.060047246515750885,\n      -0.11092282831668854,\n      0.03512628749012947,\n      0.019982190802693367,\n      0.003583843819797039,\n      -0.08301790803670883,\n      0.026952620595693588,\n      0.006145119667053223,\n      -0.07169781625270844,\n      -0.025865444913506508,\n      0.009030323475599289,\n      0.12970057129859924,\n      -0.03120129182934761,\n      -0.03578579053282738,\n      0.06322341412305832,\n      -0.050834160298109055,\n      0.03386446088552475,\n      0.02584664896130562,\n      0.07463397085666656,\n      -0.04845618084073067,\n      0.04147975891828537,\n      -0.10102668404579163,\n      0.08596403151750565,\n      0.06885310262441635,\n      -0.011228415183722973,\n      0.02268894948065281,\n      -0.10997175425291061,\n      0.04247870296239853,\n      0.07578027993440628,\n      0.08535996079444885,\n      0.07052811235189438,\n      0.09792141616344452,\n      -0.038506343960762024,\n      0.015334269031882286,\n      0.11092191934585571,\n      -0.006121296901255846,\n      -0.002801609691232443,\n      0.056817058473825455,\n      -0.05793563276529312,\n      0.02261086367070675,\n      0.033982664346694946,\n      -0.24799785017967224,\n      -0.0428302176296711,\n      -0.0351349338889122,\n      -0.017088962718844414,\n      0.08435169607400894,\n      0.11951474100351334,\n      0.014055805280804634,\n      0.027033040300011635,\n      -0.038323815912008286,\n      -0.03773351386189461\n    ],\n    [\n      -0.04398509860038757,\n      -0.1688680648803711,\n      -0.09698394685983658,\n      -0.016839267686009407,\n      -0.08570355176925659,\n      0.04735996946692467,\n      0.028708478435873985,\n      0.05733579397201538,\n      -0.07891660183668137,\n      -0.014300025999546051,\n      -0.01200793869793415,\n      0.06493695080280304,\n      0.16804218292236328,\n      -0.2225356549024582,\n      -0.13360263407230377,\n      0.029440926387906075,\n      -0.004935570526868105,\n      -0.12092488259077072,\n      -0.012368408031761646,\n      -0.024699807167053223,\n      0.03150041401386261,\n      0.0728893056511879,\n      0.07236827909946442,\n      -0.004945802967995405,\n      0.00030156681896187365,\n      -0.03394032642245293,\n      -0.03272416815161705,\n      -0.07988009601831436,\n      0.12212596833705902,\n      -0.11591214686632156,\n      0.024458516389131546,\n      0.11749447137117386,\n      0.023455897346138954,\n      0.03466359153389931,\n      0.11053873598575592,\n      0.02058786153793335,\n      0.13520413637161255,\n      0.13261455297470093,\n      -0.001754408236593008,\n      0.06731496006250381,\n      -0.003882386488839984,\n      0.005538241937756538,\n      -0.04111458361148834,\n      -0.026214193552732468,\n      -0.06869235634803772,\n      -0.06843803077936172,\n      -0.06607818603515625,\n      0.034841056913137436,\n      -0.021234415471553802,\n      -0.04062769189476967,\n      -0.005470735486596823,\n      0.003490305505692959,\n      0.04688198119401932,\n      -0.16393610835075378,\n      -0.0372181236743927,\n      -0.03491703420877457,\n      -0.09348643571138382,\n      0.05375087261199951,\n      0.04185929149389267,\n      -0.058757975697517395,\n      0.039772145450115204,\n      0.004842387977987528,\n      -0.10382666438817978,\n      -0.06918127834796906,\n      0.04387883469462395,\n      -0.018570829182863235,\n      -0.054621126502752304,\n      0.09851773083209991,\n      -0.042428236454725266,\n      0.08665527403354645,\n      0.08161722123622894,\n      -0.035717256367206573,\n      -0.1417328119277954,\n      -0.07865223288536072,\n      0.029212873429059982,\n      0.06958118081092834,\n      -0.01306547224521637,\n      0.008962382562458515,\n      -0.18097127974033356,\n      -0.09250497072935104,\n      -0.05606701970100403,\n      0.006539376452565193,\n      -0.03087994083762169,\n      -0.08307985961437225,\n      0.03467366471886635,\n      -0.04980292543768883,\n      -0.08131662756204605,\n      -0.00048762676306068897,\n      0.053673144429922104,\n      0.004875323735177517,\n      -0.101323701441288,\n      -0.12875433266162872,\n      -0.06480642408132553,\n      -0.028974538668990135,\n      0.03644224628806114,\n      0.0456361323595047,\n      -0.0556233674287796,\n      -0.04241348057985306,\n      0.017089614644646645,\n      -0.05446910858154297,\n      0.06286721676588058,\n      0.08490434288978577,\n      0.13003353774547577,\n      0.06923262029886246,\n      -0.13845603168010712,\n      0.005765980575233698,\n      -0.03157297521829605,\n      -0.025921985507011414,\n      0.17226563394069672,\n      0.16226397454738617,\n      -0.02706320397555828,\n      -0.025333529338240623,\n      -0.06114589050412178,\n      0.08073882758617401,\n      -0.0071244752034544945,\n      -0.11049161851406097,\n      -0.05574996769428253,\n      0.03767561540007591,\n      -0.09282748401165009,\n      -0.05734726041555405,\n      -0.14029601216316223,\n      -0.14199896156787872,\n      -0.004894731100648642,\n      -0.015208764001727104,\n      0.12044084072113037\n    ],\n    [\n      0.021910035982728004,\n      -0.04381491616368294,\n      -0.02463226392865181,\n      0.051574572920799255,\n      0.004181262105703354,\n      0.03926172852516174,\n      -0.02397524006664753,\n      -0.03663632273674011,\n      -0.11359461396932602,\n      0.07116465270519257,\n      0.0011501380940899253,\n      0.04983970522880554,\n      0.0474223718047142,\n      -0.10784512758255005,\n      -0.10994844138622284,\n      -0.13100501894950867,\n      0.05302036181092262,\n      -0.06058266758918762,\n      -0.07735447585582733,\n      0.036901336163282394,\n      0.05405990034341812,\n      -0.07358360290527344,\n      -0.011102433316409588,\n      0.031123096123337746,\n      -0.03345784917473793,\n      -0.03454523906111717,\n      0.005683430470526218,\n      -0.11631779372692108,\n      0.007815354503691196,\n      -0.03673160821199417,\n      -0.006099517457187176,\n      -0.07158612459897995,\n      -0.07627177238464355,\n      0.03324679285287857,\n      0.013196506537497044,\n      0.033862415701150894,\n      -0.04897419363260269,\n      0.10966914147138596,\n      0.04769148677587509,\n      0.049378901720047,\n      0.07160177826881409,\n      -0.03035200573503971,\n      -0.04935991019010544,\n      0.011325270868837833,\n      -0.0093672014772892,\n      0.027360720559954643,\n      4.086468834429979e-05,\n      0.09164407104253769,\n      0.0679006427526474,\n      0.03402595594525337,\n      0.046603646129369736,\n      0.04236876219511032,\n      0.10515429079532623,\n      -0.1011638268828392,\n      0.014969466254115105,\n      0.041810452938079834,\n      -0.047287654131650925,\n      -0.05912301689386368,\n      -0.06071947515010834,\n      0.05108076333999634,\n      -0.01254679262638092,\n      -0.12510539591312408,\n      -0.0953553169965744,\n      0.03322925791144371,\n      0.0280875563621521,\n      0.03153551369905472,\n      -0.09869571030139923,\n      -0.05624108016490936,\n      -0.0076880743727087975,\n      -0.07929927855730057,\n      0.09527501463890076,\n      0.04114175960421562,\n      -0.03209880739450455,\n      0.0029467393178492785,\n      0.014790525659918785,\n      0.010495714843273163,\n      -0.08949928730726242,\n      0.004048654809594154,\n      -0.0277516171336174,\n      0.00025991909205913544,\n      -0.05038847774267197,\n      0.029035121202468872,\n      0.026230741292238235,\n      -0.055751871317625046,\n      -0.10109948366880417,\n      -0.08992083370685577,\n      -0.09297701716423035,\n      0.08692244440317154,\n      0.10593518614768982,\n      -0.013708408921957016,\n      -0.024000396952033043,\n      -0.00639713741838932,\n      0.06959326565265656,\n      -0.06476162374019623,\n      -0.02905462309718132,\n      -0.08035010099411011,\n      0.0543828010559082,\n      -0.004946950823068619,\n      -0.027542369440197945,\n      0.012181971222162247,\n      -0.01852830871939659,\n      0.06457725912332535,\n      -0.049097079783678055,\n      -0.0034400811418890953,\n      -0.1350180208683014,\n      -0.06863120198249817,\n      0.00779009610414505,\n      0.12219049781560898,\n      0.12092128396034241,\n      0.12245302647352219,\n      0.13212695717811584,\n      -0.0015321265673264861,\n      -0.009616483934223652,\n      0.1011347770690918,\n      0.009530100971460342,\n      0.0063512553460896015,\n      -0.0766291618347168,\n      -0.017180178314447403,\n      0.04000268131494522,\n      -0.09595998376607895,\n      -0.06013255938887596,\n      -0.11364810168743134,\n      0.05732057988643646,\n      0.006027940195053816,\n      0.08705297857522964\n    ],\n    [\n      -0.10045837610960007,\n      0.032375022768974304,\n      0.12280665338039398,\n      -0.023601585999131203,\n      0.09575127810239792,\n      0.02042923867702484,\n      -0.03907037526369095,\n      -0.04514377564191818,\n      0.005290952976793051,\n      0.052290178835392,\n      -0.09928745776414871,\n      0.060184311121702194,\n      -0.06755665689706802,\n      -0.05524454265832901,\n      0.06333368271589279,\n      -0.12440948933362961,\n      -0.0710885301232338,\n      -0.03539077937602997,\n      -0.05930202826857567,\n      0.013102336786687374,\n      -0.11795339733362198,\n      -0.11737290024757385,\n      -0.09654789417982101,\n      0.016719050705432892,\n      -0.022078610956668854,\n      0.015141153708100319,\n      -0.026734722778201103,\n      0.10189522802829742,\n      -0.06833028048276901,\n      0.07142749428749084,\n      0.11720122396945953,\n      -0.053801558911800385,\n      0.08413398265838623,\n      -0.013203027658164501,\n      -0.0666678175330162,\n      -0.036222487688064575,\n      0.02804701216518879,\n      -0.06942667812108994,\n      -0.07361859083175659,\n      -0.09927625954151154,\n      0.002879613544791937,\n      0.018707849085330963,\n      -0.030259547755122185,\n      -0.016825277358293533,\n      0.037820588797330856,\n      -0.06473521888256073,\n      0.03676801547408104,\n      -0.11747952550649643,\n      0.006248348392546177,\n      -0.022089513018727303,\n      0.038973934948444366,\n      -0.013573544099926949,\n      -0.08428307622671127,\n      -0.12984997034072876,\n      0.037530213594436646,\n      0.10675211250782013,\n      -0.08298635482788086,\n      0.035259947180747986,\n      0.024345722049474716,\n      -0.0785703957080841,\n      -0.04711880907416344,\n      -0.05214792862534523,\n      -0.00884317047894001,\n      -0.006291382014751434,\n      -0.042101021856069565,\n      -0.04835394397377968,\n      -0.04650413617491722,\n      -0.0789550170302391,\n      0.025023065507411957,\n      0.05860568955540657,\n      0.01979907788336277,\n      0.04413880407810211,\n      0.09120341390371323,\n      0.03891189396381378,\n      0.04330503195524216,\n      -0.02708391286432743,\n      0.0668458566069603,\n      0.011849251575767994,\n      0.019531387835741043,\n      -0.03224186599254608,\n      0.0749785378575325,\n      -0.03996475040912628,\n      -0.09067364782094955,\n      0.01926838979125023,\n      0.0944015234708786,\n      0.07665159553289413,\n      -0.0008458453812636435,\n      -0.05147547647356987,\n      -0.02467467449605465,\n      -0.08866992592811584,\n      0.0749199390411377,\n      -0.036348648369312286,\n      0.09087158739566803,\n      -0.005912795662879944,\n      -0.05331743136048317,\n      -0.07496587932109833,\n      0.06149541959166527,\n      0.012183389626443386,\n      0.06159023195505142,\n      0.08820497989654541,\n      0.041177861392498016,\n      -0.03393607959151268,\n      0.05015306547284126,\n      0.1369137018918991,\n      -0.16119295358657837,\n      -0.025714794173836708,\n      0.07055880129337311,\n      -0.019511563703417778,\n      -0.0008100211853161454,\n      -0.12687644362449646,\n      0.04025362804532051,\n      0.11038900166749954,\n      0.07907432317733765,\n      -0.06079966574907303,\n      0.05620374158024788,\n      -0.06154659017920494,\n      -0.010214132256805897,\n      0.025162523612380028,\n      -0.010547762736678123,\n      0.06988106667995453,\n      0.06902483850717545,\n      -0.01105483714491129,\n      -0.017843401059508324,\n      0.05793463811278343,\n      0.04094476252794266\n    ],\n    [\n      -0.0016606264980509877,\n      0.05407439172267914,\n      -0.024306969717144966,\n      0.09808551520109177,\n      0.07282886654138565,\n      0.0543300025165081,\n      -0.03779064863920212,\n      -0.08192134648561478,\n      0.034333836287260056,\n      -0.04466397315263748,\n      -0.0164190586656332,\n      -0.07401721179485321,\n      -0.11103596538305283,\n      0.08686406910419464,\n      0.14935506880283356,\n      0.0011529717594385147,\n      0.041984282433986664,\n      -0.07071627676486969,\n      -0.07711493968963623,\n      0.09278798848390579,\n      -0.04498996585607529,\n      0.019549300894141197,\n      0.11972051858901978,\n      0.019208407029509544,\n      0.0022839459124952555,\n      0.03586803376674652,\n      0.012502113357186317,\n      0.05903720110654831,\n      0.04151793196797371,\n      0.08168748766183853,\n      0.06955906748771667,\n      0.012099512852728367,\n      0.007075640372931957,\n      -0.01775357872247696,\n      0.059382643550634384,\n      -0.03800944611430168,\n      0.06289287656545639,\n      -0.09039254486560822,\n      0.038525115698575974,\n      -0.09175125509500504,\n      -0.06468117237091064,\n      -0.018321089446544647,\n      0.000632291950751096,\n      0.01898868754506111,\n      -0.007695823907852173,\n      -0.026946697384119034,\n      -0.08659464120864868,\n      0.03352499380707741,\n      -0.016910329461097717,\n      0.03999578207731247,\n      -0.017412392422556877,\n      0.014595352113246918,\n      0.054572973400354385,\n      0.08774589747190475,\n      0.03754754737019539,\n      0.1208888366818428,\n      -0.031084023416042328,\n      -0.04298722743988037,\n      0.009117023088037968,\n      0.046221859753131866,\n      -0.07879165560007095,\n      0.07686908543109894,\n      0.028719399124383926,\n      -0.07157491892576218,\n      -0.07047402113676071,\n      0.08843294531106949,\n      -0.03612527251243591,\n      -0.040495846420526505,\n      0.04900480806827545,\n      -0.02894214168190956,\n      0.06764353811740875,\n      -0.07931851595640182,\n      0.050825536251068115,\n      0.11036716401576996,\n      0.03468059375882149,\n      0.001014638226479292,\n      -0.031225571408867836,\n      -0.012938756495714188,\n      0.06297162175178528,\n      -0.011945314705371857,\n      0.03396616131067276,\n      -0.027721036225557327,\n      -0.014727029018104076,\n      0.0644206777215004,\n      0.03808525577187538,\n      0.0039059545379132032,\n      0.07161912322044373,\n      -0.0580882728099823,\n      0.040937747806310654,\n      0.07841161638498306,\n      -0.08198121935129166,\n      -0.02092013508081436,\n      -0.027958806604146957,\n      -0.04805954918265343,\n      -0.0646156296133995,\n      0.05626172199845314,\n      0.0841197595000267,\n      0.17926263809204102,\n      0.05348123610019684,\n      0.016104135662317276,\n      -0.027386752888560295,\n      -0.041436366736888885,\n      0.002627770882099867,\n      0.03992527723312378,\n      -0.007442593574523926,\n      -0.031988028436899185,\n      -0.05513983219861984,\n      -0.029606124386191368,\n      0.059046175330877304,\n      0.0828075259923935,\n      -0.021322138607501984,\n      0.08447686582803726,\n      -0.033310823142528534,\n      0.057568345218896866,\n      0.00776534853503108,\n      -0.09146890044212341,\n      0.07694913446903229,\n      0.05451057478785515,\n      -0.022131016477942467,\n      0.06987619400024414,\n      0.03462251275777817,\n      -0.009619886986911297,\n      -0.03170306980609894,\n      -0.018869496881961823,\n      0.09050631523132324\n    ],\n    [\n      0.0859169289469719,\n      -0.05700624734163284,\n      -0.12250979244709015,\n      0.025039447471499443,\n      -0.08949819952249527,\n      0.049310103058815,\n      0.00025757859111763537,\n      0.09176525473594666,\n      -0.11304262280464172,\n      -0.07250604033470154,\n      0.10589206218719482,\n      0.07711653411388397,\n      0.09236501157283783,\n      -0.1107717826962471,\n      -0.13658809661865234,\n      0.006818454246968031,\n      -0.06590599566698074,\n      0.06676164269447327,\n      -0.031201692298054695,\n      -0.05416698753833771,\n      0.06775074452161789,\n      0.07263525575399399,\n      0.10794082283973694,\n      0.11315321922302246,\n      -0.09537297487258911,\n      0.0330657958984375,\n      0.003725870046764612,\n      0.0047671799547970295,\n      0.051195044070482254,\n      -0.09822095185518265,\n      -0.013277123682200909,\n      -0.045403607189655304,\n      -0.0667416974902153,\n      -0.012362809851765633,\n      0.01904202811419964,\n      0.04402013495564461,\n      0.06108696386218071,\n      -0.014184370636940002,\n      -0.014318414963781834,\n      0.07441731542348862,\n      -0.061876457184553146,\n      -0.05848587676882744,\n      0.11310340464115143,\n      -0.014328578487038612,\n      -0.03414778783917427,\n      0.015398772433400154,\n      -0.006362156942486763,\n      0.05684974044561386,\n      -0.07374350726604462,\n      0.018902411684393883,\n      -0.021925173699855804,\n      -0.014359751716256142,\n      0.0826226994395256,\n      -0.07855188101530075,\n      -0.03312863036990166,\n      -0.0537479929625988,\n      -0.01803271286189556,\n      0.03099196031689644,\n      -0.03646231070160866,\n      0.07645092159509659,\n      -0.057929627597332,\n      -0.051815908402204514,\n      -0.050451938062906265,\n      0.05983857065439224,\n      0.07729800790548325,\n      -0.07994549721479416,\n      -0.08472241461277008,\n      -0.024548888206481934,\n      -0.00766551261767745,\n      -0.05747803673148155,\n      -0.056438617408275604,\n      -0.048495110124349594,\n      -0.15769504010677338,\n      -0.09036707878112793,\n      -0.0078766830265522,\n      0.042875368148088455,\n      0.0621379017829895,\n      -0.08144032210111618,\n      -0.07313387095928192,\n      0.04482392966747284,\n      -0.13097865879535675,\n      0.04500436782836914,\n      0.007149686571210623,\n      0.023547431454062462,\n      0.03687937930226326,\n      -0.12682877480983734,\n      0.0271222535520792,\n      0.08533640205860138,\n      0.06978698819875717,\n      -0.021561715751886368,\n      0.004244512412697077,\n      0.0023020741064101458,\n      -0.009557422250509262,\n      0.030391549691557884,\n      -0.014421327970921993,\n      0.03027588129043579,\n      0.06777755916118622,\n      0.027522718533873558,\n      -0.012157960794866085,\n      0.047069936990737915,\n      0.05654669925570488,\n      0.10099904239177704,\n      0.0666915625333786,\n      -0.04296951740980148,\n      0.09865985810756683,\n      -0.009053586982190609,\n      0.02098751626908779,\n      0.11211275309324265,\n      0.005455584730952978,\n      0.14964894950389862,\n      -0.03910897299647331,\n      0.021938366815447807,\n      -0.06309652328491211,\n      -0.008692203089594841,\n      0.03728369623422623,\n      -0.01716955378651619,\n      0.1178387999534607,\n      -0.06013073772192001,\n      0.04994822293519974,\n      -0.003582906676456332,\n      -0.07044565677642822,\n      -0.12236703187227249,\n      0.04535507410764694,\n      0.05985081568360329,\n      0.036906369030475616\n    ],\n    [\n      -0.05006511136889458,\n      0.02346372976899147,\n      -0.03678513690829277,\n      0.09441512078046799,\n      0.01053700502961874,\n      0.0061079091392457485,\n      -0.14815586805343628,\n      0.01706085167825222,\n      0.1193394660949707,\n      -0.06676524877548218,\n      -0.10529378801584244,\n      -0.08361006528139114,\n      -0.14813779294490814,\n      0.04179784655570984,\n      0.037457019090652466,\n      -0.10088708996772766,\n      0.03273041173815727,\n      -0.09248337149620056,\n      0.029708094894886017,\n      -0.09270326048135757,\n      -0.015438685193657875,\n      0.05840310454368591,\n      0.06838670372962952,\n      -0.08607228100299835,\n      0.045306649059057236,\n      -0.10894801467657089,\n      -0.1226973906159401,\n      -0.042732980102300644,\n      -0.05180499330163002,\n      0.2522467374801636,\n      0.016165532171726227,\n      0.053255993872880936,\n      0.05538766086101532,\n      0.06639385968446732,\n      0.05019998922944069,\n      -0.08222616463899612,\n      -0.04193877428770065,\n      -0.1190047487616539,\n      0.019223874434828758,\n      -0.029612472280859947,\n      0.01950228586792946,\n      0.05669158697128296,\n      -0.09649544209241867,\n      0.036011703312397,\n      -0.04959928244352341,\n      0.07291671633720398,\n      -0.04771151766180992,\n      -0.09081695228815079,\n      -0.06127437576651573,\n      0.07507216930389404,\n      0.12364783138036728,\n      0.0808236300945282,\n      -0.030532564967870712,\n      -0.0015430018538609147,\n      0.009891850873827934,\n      0.014937277883291245,\n      0.04632749408483505,\n      0.009800332598388195,\n      -0.04810253530740738,\n      0.04657631367444992,\n      0.035627901554107666,\n      0.10258481651544571,\n      0.04888362064957619,\n      0.03498048335313797,\n      0.02294887788593769,\n      -0.05879567191004753,\n      -0.047521334141492844,\n      -0.001478077843785286,\n      0.02063782326877117,\n      -0.004252278711646795,\n      -0.00862417183816433,\n      0.030372409150004387,\n      0.09341119974851608,\n      0.05018392577767372,\n      -0.06250791251659393,\n      0.05921924486756325,\n      -0.06178637221455574,\n      -0.02114998735487461,\n      -0.022410940378904343,\n      -0.07789413630962372,\n      -0.00796268880367279,\n      -0.12115132063627243,\n      0.009220578707754612,\n      0.1047612652182579,\n      -0.023579133674502373,\n      0.07477322220802307,\n      0.09602667391300201,\n      -0.06179911643266678,\n      -0.03144235908985138,\n      0.05600108951330185,\n      0.03819701820611954,\n      0.043184224516153336,\n      -0.08783084154129028,\n      -0.0629255622625351,\n      -0.08143209666013718,\n      -0.06140369176864624,\n      0.03727060556411743,\n      -0.05963384732604027,\n      -0.03765963762998581,\n      0.07828561961650848,\n      -0.025056123733520508,\n      -0.10272926837205887,\n      0.13936980068683624,\n      0.10500585287809372,\n      -0.016307152807712555,\n      -0.012291382066905499,\n      0.045590758323669434,\n      -0.07461199909448624,\n      0.08642292767763138,\n      -0.004870069678872824,\n      0.0028069966938346624,\n      0.054529402405023575,\n      -0.07613104581832886,\n      0.0555192232131958,\n      0.04627600312232971,\n      -0.10509055107831955,\n      0.006199425086379051,\n      -0.04570522531867027,\n      0.0869997888803482,\n      0.07833077013492584,\n      0.0515529066324234,\n      0.05754677578806877,\n      0.06168447434902191,\n      0.11006282269954681,\n      0.03319580480456352\n    ],\n    [\n      -0.08751273155212402,\n      -0.0707702711224556,\n      -0.05472322553396225,\n      0.03153436630964279,\n      -0.019018614664673805,\n      0.07378184795379639,\n      -0.01681530475616455,\n      -0.024750620126724243,\n      0.12432880699634552,\n      -0.0541536845266819,\n      0.06690368056297302,\n      0.029636910185217857,\n      0.03662390261888504,\n      0.16920429468154907,\n      0.1558293104171753,\n      0.027221349999308586,\n      0.020289452746510506,\n      -0.12593460083007812,\n      -0.03453981131315231,\n      -0.06488059461116791,\n      -0.02412542514503002,\n      0.07449100911617279,\n      0.07845627516508102,\n      -0.000672436086460948,\n      0.1216418668627739,\n      0.0726172998547554,\n      0.043344542384147644,\n      0.06543022394180298,\n      0.1070449948310852,\n      0.10147576779127121,\n      0.05097559466958046,\n      0.00901105161756277,\n      0.09657991677522659,\n      0.014424380846321583,\n      -0.10607770830392838,\n      -0.0866352990269661,\n      0.02488388493657112,\n      0.06998920440673828,\n      -0.03559765964746475,\n      -0.004136089701205492,\n      -0.0384787917137146,\n      0.0846552699804306,\n      0.060425661504268646,\n      -0.030042186379432678,\n      -0.16680213809013367,\n      -0.033475618809461594,\n      0.04276525601744652,\n      0.041655123233795166,\n      0.06329353898763657,\n      -0.018866324797272682,\n      0.03652370348572731,\n      0.07992905378341675,\n      0.014734577387571335,\n      -0.019869817420840263,\n      0.09609631448984146,\n      -0.02930026315152645,\n      0.0016151166055351496,\n      0.08307262510061264,\n      0.04124634340405464,\n      -0.0323442704975605,\n      0.08871794492006302,\n      0.034256212413311005,\n      0.002116505056619644,\n      -0.0007521780207753181,\n      -0.11928578466176987,\n      -0.011999125592410564,\n      0.012018975801765919,\n      -0.08034588396549225,\n      -0.031318023800849915,\n      -0.08585033565759659,\n      0.012935586273670197,\n      0.16210897266864777,\n      -0.1529078483581543,\n      0.04160566255450249,\n      0.09818961471319199,\n      -0.025171341374516487,\n      -0.15439678728580475,\n      -0.08108561486005783,\n      -0.011255072429776192,\n      -0.043577924370765686,\n      0.08684667944908142,\n      0.01453933585435152,\n      0.06852628290653229,\n      0.0037926044315099716,\n      0.0013822154141962528,\n      0.0821429044008255,\n      -0.10943014174699783,\n      -0.05874095484614372,\n      -0.02601763792335987,\n      0.05514022707939148,\n      -0.002728099934756756,\n      0.04219132661819458,\n      -0.10834188759326935,\n      -0.08824777603149414,\n      -0.1280079334974289,\n      -0.11684384196996689,\n      0.09262299537658691,\n      -0.0368478037416935,\n      0.04478253424167633,\n      0.03647349774837494,\n      -0.006424070335924625,\n      0.07740487903356552,\n      -0.08866769820451736,\n      -0.06321487575769424,\n      0.08609195053577423,\n      0.06644123047590256,\n      0.08843814581632614,\n      0.09978955239057541,\n      -0.07778121531009674,\n      0.013021739199757576,\n      0.04519272968173027,\n      -0.05675029754638672,\n      0.0074347518384456635,\n      -0.10429883748292923,\n      -0.028417669236660004,\n      -0.22021956741809845,\n      0.08463293313980103,\n      0.031746041029691696,\n      0.06704487651586533,\n      0.08329883962869644,\n      0.01928713358938694,\n      -0.04936395585536957,\n      0.039364043623209,\n      -0.06338515132665634,\n      0.08079197257757187\n    ],\n    [\n      -0.09240776300430298,\n      -0.03401502966880798,\n      0.05891916900873184,\n      -0.018258996307849884,\n      0.026051711291074753,\n      -0.008050601929426193,\n      -0.010637091472744942,\n      -0.09282481670379639,\n      0.06251722574234009,\n      -0.02160477824509144,\n      0.05021156370639801,\n      0.045717887580394745,\n      -0.053798697888851166,\n      0.10956329107284546,\n      0.15776567161083221,\n      -0.12769314646720886,\n      0.03394298627972603,\n      -0.0209369994699955,\n      -0.006380394101142883,\n      0.036397673189640045,\n      -0.012826619669795036,\n      0.0690409317612648,\n      -0.01769961230456829,\n      -0.08091668784618378,\n      0.051171135157346725,\n      -0.03674371913075447,\n      -0.002991698682308197,\n      -0.03902900218963623,\n      -0.01418611966073513,\n      0.2025076299905777,\n      0.01636076718568802,\n      -0.003663979470729828,\n      0.054901305586099625,\n      0.0013721303548663855,\n      -0.09024669975042343,\n      0.05251099541783333,\n      0.048727307468652725,\n      -0.0592212937772274,\n      -0.03402775153517723,\n      0.004192447289824486,\n      -0.007358372211456299,\n      -0.10451206564903259,\n      -0.04866228625178337,\n      -0.026271330192685127,\n      0.026592381298542023,\n      0.04818422719836235,\n      0.0656328946352005,\n      0.04961257427930832,\n      -0.018875470384955406,\n      0.024070288985967636,\n      0.008627593517303467,\n      -0.06982675194740295,\n      -0.031726036220788956,\n      0.009160008281469345,\n      0.05825785547494888,\n      -0.02831275574862957,\n      0.04095570743083954,\n      -0.016996996477246284,\n      -0.07038067281246185,\n      -0.09203127026557922,\n      0.04834683611989021,\n      -0.001377331092953682,\n      -0.0449436716735363,\n      0.0145398685708642,\n      0.05487306788563728,\n      -0.020359091460704803,\n      0.08567235618829727,\n      0.0644286572933197,\n      0.07656636834144592,\n      0.022375784814357758,\n      0.07676682621240616,\n      0.009368255734443665,\n      0.12748485803604126,\n      0.09234053641557693,\n      -0.04855715483427048,\n      -0.0831696167588234,\n      0.0006689568399451673,\n      0.020739812403917313,\n      0.029811512678861618,\n      0.0467342883348465,\n      0.07204122841358185,\n      0.0407821387052536,\n      -0.03828684613108635,\n      -0.05621189996600151,\n      0.06899191439151764,\n      0.00650585163384676,\n      0.048039622604846954,\n      0.022716792300343513,\n      -0.07762067019939423,\n      0.04519527405500412,\n      0.00738883949816227,\n      -0.051487166434526443,\n      -0.09923253953456879,\n      0.05302336812019348,\n      -0.023171182721853256,\n      -0.005302142817527056,\n      0.05146742984652519,\n      0.11134618520736694,\n      -0.017367646098136902,\n      -0.022984536364674568,\n      0.06332577019929886,\n      0.008785443380475044,\n      -0.0481647253036499,\n      0.052724383771419525,\n      0.05740645155310631,\n      0.018613947555422783,\n      -0.017481066286563873,\n      -0.015161268413066864,\n      0.0012090880190953612,\n      0.0453459694981575,\n      -0.010004546493291855,\n      0.0985819548368454,\n      0.024137627333402634,\n      0.019135478883981705,\n      0.04412013292312622,\n      -0.03006097301840782,\n      0.029840800911188126,\n      0.06171601265668869,\n      0.061850666999816895,\n      0.012348049320280552,\n      0.11255084723234177,\n      0.12716098129749298,\n      -0.05004487559199333,\n      0.07820656895637512,\n      -0.013786692172288895\n    ],\n    [\n      -0.024221567437052727,\n      0.04132644087076187,\n      0.04767407104372978,\n      0.046922147274017334,\n      -0.009635599330067635,\n      0.05205679312348366,\n      0.09051729738712311,\n      0.0994957759976387,\n      -0.150839701294899,\n      0.0889587476849556,\n      0.09813471138477325,\n      0.03431626781821251,\n      0.1486644297838211,\n      -0.15469706058502197,\n      -0.19669225811958313,\n      0.19408923387527466,\n      -0.08577562123537064,\n      -0.05055906996130943,\n      -0.023865720257163048,\n      0.06797505915164948,\n      0.09294836223125458,\n      0.07805030047893524,\n      0.07729315012693405,\n      0.08196064084768295,\n      -5.91493007959798e-05,\n      0.09001720696687698,\n      0.13647934794425964,\n      0.05432191118597984,\n      0.0113303167745471,\n      -0.14907944202423096,\n      -0.01156554464250803,\n      -0.02748432382941246,\n      0.03978782519698143,\n      0.1012551337480545,\n      0.10802138596773148,\n      0.047480177134275436,\n      0.04302896186709404,\n      0.08714260905981064,\n      0.07228273898363113,\n      -0.03896024823188782,\n      0.06236421689391136,\n      0.06285277009010315,\n      0.0027559297159314156,\n      -0.021965956315398216,\n      0.06662104278802872,\n      0.0588703490793705,\n      0.033999186009168625,\n      0.08245302736759186,\n      0.0990782380104065,\n      -0.07874064147472382,\n      -0.01987670175731182,\n      -0.033213574439287186,\n      0.07104036211967468,\n      -0.020967181771993637,\n      -0.05018513649702072,\n      -0.03997104987502098,\n      -0.03920462355017662,\n      0.02094244211912155,\n      0.08537808805704117,\n      0.011111960746347904,\n      0.004475362598896027,\n      -0.012998167425394058,\n      0.018290312960743904,\n      0.0336645171046257,\n      0.007141406182199717,\n      0.043380893766880035,\n      0.0329459086060524,\n      -0.013981174677610397,\n      0.061092667281627655,\n      -0.0822896808385849,\n      -0.03828464075922966,\n      -0.037865977734327316,\n      -0.16686062514781952,\n      -0.013580244965851307,\n      -0.0818636491894722,\n      -0.040520038455724716,\n      0.005923197604715824,\n      -0.07132934033870697,\n      -0.025241496041417122,\n      -0.02811347506940365,\n      -0.07939301431179047,\n      -0.06210402026772499,\n      0.08370806276798248,\n      -0.018159179016947746,\n      0.0562562569975853,\n      0.01703721098601818,\n      -0.035404570400714874,\n      0.08425654470920563,\n      -0.04173693433403969,\n      -0.0037760213017463684,\n      0.025561071932315826,\n      -0.04136493802070618,\n      -0.032217081636190414,\n      0.0557747557759285,\n      0.01914026588201523,\n      0.04356009513139725,\n      0.060577139258384705,\n      -0.052741263061761856,\n      0.04119841009378433,\n      -0.06374696642160416,\n      0.06060614064335823,\n      0.005861896090209484,\n      0.08147116005420685,\n      -0.02958432212471962,\n      0.057538628578186035,\n      0.015210985206067562,\n      0.04115716367959976,\n      0.11166670173406601,\n      -0.007892785593867302,\n      0.019866982474923134,\n      -0.04907141625881195,\n      -0.06608014553785324,\n      0.0027266605757176876,\n      0.07058613747358322,\n      -0.013008643873035908,\n      -0.02976858988404274,\n      0.0941924899816513,\n      0.021473318338394165,\n      -0.009691726416349411,\n      -0.09999740868806839,\n      -0.057504162192344666,\n      -0.15180735290050507,\n      -0.003938228357583284,\n      0.0613551139831543,\n      0.01449076272547245\n    ],\n    [\n      -0.005442483816295862,\n      0.03177092596888542,\n      0.05555319786071777,\n      0.08754166215658188,\n      0.11939620226621628,\n      -0.028986813500523567,\n      -0.11221098154783249,\n      -0.07221096009016037,\n      0.1802358776330948,\n      -0.10058392584323883,\n      -0.07536176592111588,\n      -0.009647953324019909,\n      -0.1488332897424698,\n      0.0666738823056221,\n      0.07911483943462372,\n      -0.07189920544624329,\n      0.022520633414387703,\n      -0.10098712891340256,\n      -0.05868898332118988,\n      -0.03586232289671898,\n      -0.031213779002428055,\n      0.06973819434642792,\n      0.011716204695403576,\n      -0.04866746440529823,\n      0.08147163689136505,\n      0.003412768943235278,\n      -0.03509018197655678,\n      0.03112601675093174,\n      0.06181594729423523,\n      0.13530923426151276,\n      0.08194062858819962,\n      -0.02385462448000908,\n      -0.04324915260076523,\n      -0.0810515433549881,\n      -0.02961616963148117,\n      0.060751378536224365,\n      -0.06573022156953812,\n      0.04138867184519768,\n      -0.10141962766647339,\n      0.04966786503791809,\n      -0.012304521165788174,\n      0.062453750520944595,\n      -0.11164411157369614,\n      -0.02143409475684166,\n      -0.00777203356847167,\n      0.07810219377279282,\n      0.05082928016781807,\n      -0.001245636260136962,\n      0.0014755998272448778,\n      -0.016361147165298462,\n      0.01335784699767828,\n      0.002280791522935033,\n      0.032287321984767914,\n      -0.05609391629695892,\n      0.08713016659021378,\n      0.10212875157594681,\n      0.048199716955423355,\n      0.07136772572994232,\n      -0.022205032408237457,\n      -0.015186244621872902,\n      0.08098818361759186,\n      0.06463813036680222,\n      -0.07215073704719543,\n      -0.08335568010807037,\n      0.0987149178981781,\n      0.09764569997787476,\n      0.00019720126874744892,\n      -0.027691300958395004,\n      -0.009529409930109978,\n      0.0356203056871891,\n      0.05268670618534088,\n      0.014660745859146118,\n      0.044537439942359924,\n      0.06290820986032486,\n      0.04759687930345535,\n      0.03535400331020355,\n      0.04484165087342262,\n      -0.08540670573711395,\n      0.09851507842540741,\n      -0.03374297544360161,\n      -0.03654477000236511,\n      -0.11350978910923004,\n      -0.05323892459273338,\n      0.04703342914581299,\n      0.08866393566131592,\n      -0.023305293172597885,\n      0.015516327694058418,\n      -0.01396671961992979,\n      0.027206530794501305,\n      0.06369643658399582,\n      -0.056002527475357056,\n      -0.04226553812623024,\n      0.06607599556446075,\n      -0.007015559822320938,\n      0.06613708287477493,\n      0.07917726039886475,\n      0.032825127243995667,\n      0.04968724399805069,\n      0.11792872846126556,\n      0.0884685292840004,\n      -0.0025980547070503235,\n      -0.003979676403105259,\n      0.08891545981168747,\n      0.11567473411560059,\n      0.020365556702017784,\n      -0.04575974866747856,\n      -0.05079750716686249,\n      -0.013298784382641315,\n      0.06683795154094696,\n      -0.030347539111971855,\n      -0.06343575567007065,\n      0.05005257576704025,\n      -0.07733210921287537,\n      -0.05007856339216232,\n      0.04454660043120384,\n      0.04114411771297455,\n      0.018333327025175095,\n      -0.019727155566215515,\n      0.09413645416498184,\n      0.09690559655427933,\n      0.10693199932575226,\n      0.15124404430389404,\n      0.07290849834680557,\n      -0.029717111960053444,\n      -0.03973454236984253\n    ],\n    [\n      -0.018606562167406082,\n      -0.01475998293608427,\n      -0.0716819167137146,\n      -0.022779293358325958,\n      -0.009814533405005932,\n      0.09772497415542603,\n      -0.02898109331727028,\n      0.038291387259960175,\n      -0.028799770399928093,\n      -0.09873241931200027,\n      -0.08570236712694168,\n      0.02287798561155796,\n      0.07301218807697296,\n      -0.08155293017625809,\n      -0.03155010566115379,\n      0.0836912989616394,\n      -0.05515081807971001,\n      -0.032767318189144135,\n      -0.11857905238866806,\n      -0.07954972982406616,\n      -0.09224167466163635,\n      -0.02365647815167904,\n      0.019618917256593704,\n      -0.08328362554311752,\n      -0.031342048197984695,\n      -0.10740186274051666,\n      -0.025861423462629318,\n      -0.02090887911617756,\n      -0.0678410530090332,\n      -0.09723946452140808,\n      0.07230162620544434,\n      0.023855367675423622,\n      -0.031787317246198654,\n      0.056716542690992355,\n      -0.07365065813064575,\n      0.019865943118929863,\n      -0.06204932555556297,\n      -0.019272711127996445,\n      -0.07116365432739258,\n      0.08496659994125366,\n      -0.0624784454703331,\n      -0.04174842685461044,\n      0.12015897035598755,\n      0.025247016921639442,\n      -0.042771611362695694,\n      -0.03810075670480728,\n      0.09996987879276276,\n      -0.1673937886953354,\n      -0.016664981842041016,\n      0.058996181935071945,\n      -0.03975550830364227,\n      -0.013825977221131325,\n      0.05102913826704025,\n      -0.19495202600955963,\n      -0.0005028487648814917,\n      0.010526415891945362,\n      -0.03747649863362312,\n      0.05922689288854599,\n      0.09205383062362671,\n      -0.02306387573480606,\n      0.0865614265203476,\n      0.08244433999061584,\n      -0.010669199749827385,\n      0.041812360286712646,\n      0.017533499747514725,\n      0.028106030076742172,\n      -0.059180498123168945,\n      0.09546849131584167,\n      -0.09212054312229156,\n      0.016641799360513687,\n      0.12228719145059586,\n      0.03986547514796257,\n      0.06288160383701324,\n      -0.038651395589113235,\n      0.10726707428693771,\n      0.07920493930578232,\n      0.08455917239189148,\n      0.022468969225883484,\n      -0.10686706006526947,\n      0.004051047842949629,\n      -0.02965157851576805,\n      0.11269398033618927,\n      -0.16264450550079346,\n      -0.015725083649158478,\n      0.011624669656157494,\n      0.03300399333238602,\n      -0.04442332312464714,\n      -0.02986413985490799,\n      -0.05367687717080116,\n      0.0373050719499588,\n      -0.04895011708140373,\n      -0.08770471811294556,\n      -0.07060901075601578,\n      -0.015473584644496441,\n      -0.07483231276273727,\n      0.03355051577091217,\n      -0.06125614419579506,\n      -0.014940190128982067,\n      0.060262929648160934,\n      0.04917961731553078,\n      0.029367895796895027,\n      -0.06120498478412628,\n      0.025920389220118523,\n      -0.0565263032913208,\n      0.05085502192378044,\n      -0.15449237823486328,\n      -0.012529943138360977,\n      0.030679354444146156,\n      0.01008949987590313,\n      0.05424295738339424,\n      -0.10559392720460892,\n      -0.10535354912281036,\n      -0.039543330669403076,\n      -0.027232075110077858,\n      -0.033048976212739944,\n      -0.14760829508304596,\n      -0.11585833132266998,\n      -0.006889089476317167,\n      -0.09928546100854874,\n      -0.027498990297317505,\n      0.10114698857069016,\n      -0.008757546544075012,\n      0.03165176883339882,\n      0.05011201277375221,\n      -0.07467440515756607\n    ],\n    [\n      0.109488345682621,\n      0.004652429837733507,\n      0.05017305165529251,\n      -0.02159031666815281,\n      -0.09896906465291977,\n      0.08969144523143768,\n      0.10156287252902985,\n      0.01055147685110569,\n      -0.19065843522548676,\n      0.007856719195842743,\n      0.041028186678886414,\n      -0.05686652660369873,\n      0.10218614339828491,\n      -0.02526840940117836,\n      -0.09827914088964462,\n      -0.05806557089090347,\n      -0.014753321185708046,\n      0.03306969255208969,\n      0.06322348117828369,\n      0.08156272023916245,\n      0.02534802071750164,\n      0.0014791602734476328,\n      -0.026690611615777016,\n      0.06361286342144012,\n      0.0033824576530605555,\n      0.01697121374309063,\n      0.16096337139606476,\n      0.02818598411977291,\n      0.0340917631983757,\n      -0.22068357467651367,\n      -0.08323712646961212,\n      -0.03825776278972626,\n      0.05882797762751579,\n      0.057897601276636124,\n      0.09937063604593277,\n      0.08795354515314102,\n      0.07538478821516037,\n      0.014889099635183811,\n      0.0933002158999443,\n      0.056971706449985504,\n      0.06243416294455528,\n      0.04024943336844444,\n      0.03748783841729164,\n      -0.11965255439281464,\n      0.07036761939525604,\n      0.03720467537641525,\n      -0.09327705204486847,\n      -0.03703876957297325,\n      0.016039015725255013,\n      -0.001883961958810687,\n      -0.09363622963428497,\n      -0.06384190917015076,\n      0.04368572309613228,\n      0.022266974672675133,\n      -0.10974106192588806,\n      0.02023422159254551,\n      -0.07191815227270126,\n      -0.06706110388040543,\n      0.023372875526547432,\n      0.008307202719151974,\n      -0.03973723575472832,\n      0.007605386432260275,\n      -0.03279628977179527,\n      -0.046260133385658264,\n      0.009182288311421871,\n      0.04319271817803383,\n      -0.010333742015063763,\n      0.024594441056251526,\n      -0.012913578189909458,\n      -0.0726325511932373,\n      0.027442730963230133,\n      0.03829655423760414,\n      -0.09552167356014252,\n      -0.04480081424117088,\n      -0.07037930935621262,\n      0.07342574000358582,\n      0.0017644392792135477,\n      -0.002670116489753127,\n      0.02405996434390545,\n      0.016086740419268608,\n      -0.02170865423977375,\n      -0.01628323271870613,\n      -0.02652987651526928,\n      0.026879876852035522,\n      -0.04071367532014847,\n      -0.09003397077322006,\n      0.020514409989118576,\n      0.09560350328683853,\n      -0.09289972484111786,\n      -0.07946879416704178,\n      -0.048014529049396515,\n      0.05466417223215103,\n      0.022711152210831642,\n      -0.08592712134122849,\n      -0.033556245267391205,\n      -0.08623220026493073,\n      0.022907156497240067,\n      -0.01621588133275509,\n      0.024871405214071274,\n      0.0005903574055992067,\n      0.09639771282672882,\n      -0.025182001292705536,\n      0.002910053124651313,\n      -0.019097715616226196,\n      -0.009844686836004257,\n      -0.057667240500450134,\n      -0.080962173640728,\n      0.11966154724359512,\n      -0.04974619671702385,\n      -0.002151504857465625,\n      0.09598755836486816,\n      0.003173242323100567,\n      -0.03282012790441513,\n      -0.003873614128679037,\n      -0.029582936316728592,\n      -0.10572589933872223,\n      -0.05555609613656998,\n      -0.07126230001449585,\n      -0.043232258409261703,\n      -0.009423226118087769,\n      -0.03131081536412239,\n      -0.15974871814250946,\n      0.06614658981561661,\n      0.06262488663196564,\n      0.08633600175380707\n    ],\n    [\n      -0.011377651244401932,\n      0.013707616366446018,\n      -0.05454430729150772,\n      -0.018075862899422646,\n      -0.08421008288860321,\n      0.02610602043569088,\n      0.0536712110042572,\n      0.0667223110795021,\n      -0.05528474599123001,\n      0.09211722016334534,\n      0.06950537860393524,\n      0.06127951666712761,\n      0.12129270285367966,\n      -0.09992900490760803,\n      -0.17791925370693207,\n      0.20021378993988037,\n      -0.03428451716899872,\n      -0.02639113925397396,\n      0.04394412413239479,\n      0.10995980352163315,\n      0.04099016636610031,\n      -0.050542201846838,\n      0.07412578910589218,\n      0.11941540241241455,\n      -0.053161442279815674,\n      0.011879642494022846,\n      0.1215241327881813,\n      -0.08035973459482193,\n      -0.055205002427101135,\n      -0.22630412876605988,\n      0.03627245873212814,\n      -0.009055770933628082,\n      0.07642704248428345,\n      0.014527050778269768,\n      -0.06200559437274933,\n      0.10519209504127502,\n      -0.00868287868797779,\n      -0.005058840848505497,\n      0.06239492446184158,\n      0.031465526670217514,\n      0.07145695388317108,\n      -0.04266759380698204,\n      -0.020237373188138008,\n      -0.017177866771817207,\n      0.07646264880895615,\n      -0.008675704710185528,\n      -0.035163696855306625,\n      -0.024576939642429352,\n      0.05757641792297363,\n      -0.08191272616386414,\n      -0.053503766655921936,\n      0.06537981331348419,\n      -0.008781693875789642,\n      0.014829516410827637,\n      0.04106781259179115,\n      -0.07384347915649414,\n      -0.05699016898870468,\n      0.03458383306860924,\n      0.1008789911866188,\n      -0.016492638736963272,\n      0.006040142383426428,\n      -0.0352633111178875,\n      0.05540304258465767,\n      -0.031431134790182114,\n      0.024973398074507713,\n      -0.07172713428735733,\n      0.013238522224128246,\n      0.016627950593829155,\n      -0.0005426605930551887,\n      0.040194857865571976,\n      -0.0638551190495491,\n      -0.06662316620349884,\n      -0.10993789881467819,\n      -0.012315120548009872,\n      0.02611958049237728,\n      -0.007696960587054491,\n      -0.043812140822410583,\n      0.043861594051122665,\n      -0.042250242084264755,\n      -0.0044852495193481445,\n      -0.1178363561630249,\n      0.030784981325268745,\n      0.10027182102203369,\n      0.006475627422332764,\n      -0.036853112280368805,\n      -0.022881977260112762,\n      -0.06766217201948166,\n      -0.03855014592409134,\n      0.005308839026838541,\n      0.03741835802793503,\n      0.00656304182484746,\n      0.07577620446681976,\n      0.0022415840066969395,\n      0.028740428388118744,\n      0.032885123044252396,\n      -0.032022424042224884,\n      0.10070410370826721,\n      -0.048917267471551895,\n      -0.09789275377988815,\n      -0.03353509679436684,\n      0.06608176976442337,\n      0.04043776914477348,\n      0.08573794364929199,\n      0.013631477020680904,\n      0.1297198235988617,\n      0.06902094930410385,\n      0.06395670026540756,\n      0.04564953222870827,\n      0.03240414708852768,\n      0.09039885550737381,\n      0.07372694462537766,\n      -0.008955655619502068,\n      -0.04832063987851143,\n      -0.06195317581295967,\n      0.055211037397384644,\n      0.020580563694238663,\n      -0.015809783712029457,\n      0.07109788805246353,\n      0.036748532205820084,\n      -0.03907553106546402,\n      -0.04256925731897354,\n      -0.04834846407175064,\n      0.0692155510187149,\n      -0.0791420117020607,\n      0.009103083051741123\n    ],\n    [\n      -0.03600648045539856,\n      -0.07262129336595535,\n      0.06398013234138489,\n      0.12280068546533585,\n      -0.12498269975185394,\n      -0.0405556745827198,\n      -0.028495382517576218,\n      -0.035409290343523026,\n      0.12271245568990707,\n      -0.10739840567111969,\n      -0.010411020368337631,\n      0.08242074400186539,\n      0.0991995632648468,\n      0.011106085032224655,\n      0.1448003202676773,\n      0.014240157790482044,\n      -0.059467531740665436,\n      -0.1154688149690628,\n      -0.02757258154451847,\n      0.023984448984265327,\n      -0.055990613996982574,\n      -0.02529573068022728,\n      -0.018065789714455605,\n      0.0190596766769886,\n      -0.16161826252937317,\n      0.017017951235175133,\n      0.009858454577624798,\n      0.10755108296871185,\n      -0.04692881181836128,\n      0.20466774702072144,\n      0.07905726879835129,\n      0.02004365250468254,\n      0.05779758468270302,\n      -0.06337779015302658,\n      -0.10455959290266037,\n      -0.058192379772663116,\n      0.008854609914124012,\n      -0.05331464856863022,\n      -0.03484184667468071,\n      -0.014512517489492893,\n      -0.07967275381088257,\n      0.058342497795820236,\n      -0.15388844907283783,\n      -0.055905532091856,\n      -0.028249038383364677,\n      -0.04526891931891441,\n      -0.05206318572163582,\n      -0.01929725706577301,\n      0.040102552622556686,\n      -0.021908460184931755,\n      -0.06410852819681168,\n      0.020794903859496117,\n      -0.09922831505537033,\n      -0.07837209105491638,\n      -0.04317334666848183,\n      -0.018153615295886993,\n      -0.042770788073539734,\n      0.039422471076250076,\n      -0.1025564894080162,\n      -0.07597421854734421,\n      0.0804634690284729,\n      -0.05151259899139404,\n      -0.06022221967577934,\n      -0.05065862834453583,\n      -0.05572156980633736,\n      0.009212663397192955,\n      -0.017878277227282524,\n      -0.11543086171150208,\n      -0.10043978691101074,\n      -0.061526086181402206,\n      0.07351239770650864,\n      -0.15578460693359375,\n      -0.05518217012286186,\n      -0.06643226742744446,\n      -0.05615269020199776,\n      0.05426548793911934,\n      -0.04709159582853317,\n      0.06656286120414734,\n      0.08526735752820969,\n      -0.05809348076581955,\n      -0.03967796638607979,\n      0.08189477771520615,\n      -0.057266343384981155,\n      0.04318271577358246,\n      0.04367859661579132,\n      0.06639306992292404,\n      -0.08689077198505402,\n      -0.016788754612207413,\n      -0.08838311582803726,\n      -0.018744179978966713,\n      -0.047431111335754395,\n      0.027639279142022133,\n      0.08027976006269455,\n      0.10670747607946396,\n      -0.043805450201034546,\n      0.004995177965611219,\n      -0.03166887164115906,\n      -0.022985074669122696,\n      -0.06912190467119217,\n      0.029811926186084747,\n      -0.02664634771645069,\n      0.079837366938591,\n      0.02636784128844738,\n      0.09521109610795975,\n      -0.08661250025033951,\n      -0.034890417009592056,\n      -0.09425507485866547,\n      -0.016557704657316208,\n      -0.040397945791482925,\n      -0.008818970061838627,\n      0.08157911151647568,\n      -0.001953905913978815,\n      0.09107568114995956,\n      -0.09830094873905182,\n      0.035114411264657974,\n      -0.20274220407009125,\n      -0.04965128004550934,\n      0.027947625145316124,\n      0.02936054952442646,\n      0.12080573290586472,\n      -0.06621501594781876,\n      -0.0959291085600853,\n      -0.039288103580474854,\n      0.017526770010590553,\n      0.04482033848762512\n    ],\n    [\n      0.09830684214830399,\n      -0.10768554359674454,\n      -0.07626236975193024,\n      -0.05619726702570915,\n      -0.08243055641651154,\n      0.03985178470611572,\n      0.019244760274887085,\n      0.141124427318573,\n      -0.12044864892959595,\n      0.08842920511960983,\n      0.029106920585036278,\n      0.03301899507641792,\n      0.1311226785182953,\n      -0.11993753910064697,\n      -0.04821699112653732,\n      0.045001544058322906,\n      0.0462636724114418,\n      -0.04773934185504913,\n      -0.015467171557247639,\n      0.09284444153308868,\n      0.0907730758190155,\n      -0.012133289128541946,\n      -0.10750611126422882,\n      0.07046086341142654,\n      -0.13129587471485138,\n      -0.03880058228969574,\n      -0.0477973073720932,\n      -0.0857577845454216,\n      -0.0546211414039135,\n      -0.07497437298297882,\n      0.005218079313635826,\n      0.049916379153728485,\n      -0.0036097944248467684,\n      0.11777504533529282,\n      0.0988961011171341,\n      -0.037658751010894775,\n      -0.007768211420625448,\n      0.09297127276659012,\n      -0.008601351641118526,\n      -0.029435811564326286,\n      0.06303875148296356,\n      -0.04483426734805107,\n      0.019447896629571915,\n      0.059993207454681396,\n      -0.07979694753885269,\n      -0.04938006028532982,\n      -0.03456743061542511,\n      0.04884328693151474,\n      -0.028362609446048737,\n      -0.04567357897758484,\n      -0.1198388934135437,\n      0.033120714128017426,\n      0.005881207529455423,\n      -0.12318795174360275,\n      0.03312849625945091,\n      0.0026866246480494738,\n      -0.04882070794701576,\n      -0.06663797050714493,\n      -0.07403916120529175,\n      -0.05135532096028328,\n      -0.014486332423985004,\n      0.01578199490904808,\n      -0.0622345432639122,\n      -0.01616135612130165,\n      -0.03720813989639282,\n      -0.05679822340607643,\n      -0.11570484191179276,\n      -0.04746367782354355,\n      -0.12324181944131851,\n      -0.10802517831325531,\n      -0.006913985591381788,\n      0.03658344969153404,\n      -0.08887706696987152,\n      -0.044585712254047394,\n      -0.0973065048456192,\n      -0.09404698014259338,\n      0.020735247060656548,\n      -0.11051038652658463,\n      -0.01646198332309723,\n      -0.02343127503991127,\n      -0.13534721732139587,\n      -0.006888449192047119,\n      0.05567797273397446,\n      -0.05268055573105812,\n      -0.08856295794248581,\n      -0.00022114207968115807,\n      -0.022441398352384567,\n      -0.07716088742017746,\n      -0.0004647963505703956,\n      -0.0185664352029562,\n      -0.01411595568060875,\n      -0.038363244384527206,\n      -0.008115628734230995,\n      -0.11358065158128738,\n      0.007405912037938833,\n      -0.003481280291453004,\n      0.008242224343121052,\n      0.02143683284521103,\n      -0.15215057134628296,\n      -0.05548764392733574,\n      -0.007695394568145275,\n      0.06977327167987823,\n      0.02644483745098114,\n      -0.06183498352766037,\n      -0.04397786781191826,\n      0.0788029357790947,\n      -0.02133575640618801,\n      0.026165936142206192,\n      0.037092991173267365,\n      0.06914570182561874,\n      0.09524132311344147,\n      0.021841412410140038,\n      0.054979413747787476,\n      0.08590911328792572,\n      0.07718392461538315,\n      -0.06377799808979034,\n      -0.15554209053516388,\n      -0.014962729066610336,\n      -0.16204720735549927,\n      -0.1505080610513687,\n      -0.024798568338155746,\n      -0.11298659443855286,\n      0.024541731923818588,\n      -0.010371178388595581,\n      0.0984073057770729\n    ],\n    [\n      -0.03975580260157585,\n      -0.008561247028410435,\n      0.03533746302127838,\n      0.14172552525997162,\n      0.062439706176519394,\n      0.0065179564990103245,\n      -0.013656697236001492,\n      -0.10853375494480133,\n      0.1862076073884964,\n      -0.055043917149305344,\n      0.06023549288511276,\n      -0.010843660682439804,\n      0.11212129145860672,\n      0.13057979941368103,\n      0.018675547093153,\n      0.026497436687350273,\n      -0.045429352670907974,\n      -0.12277526408433914,\n      -0.0801643505692482,\n      0.05650002881884575,\n      0.059968747198581696,\n      0.052609823644161224,\n      -0.1284126341342926,\n      0.045710500329732895,\n      -0.154446080327034,\n      -0.0977729856967926,\n      -0.0924360603094101,\n      0.07492709159851074,\n      0.0026032384485006332,\n      0.2590620219707489,\n      0.06961846351623535,\n      -0.07194139808416367,\n      0.08606201410293579,\n      0.04188266396522522,\n      -0.03755522519350052,\n      -0.0767713338136673,\n      0.025187170132994652,\n      0.030040061101317406,\n      -0.05468309670686722,\n      0.018138671293854713,\n      -0.05610944330692291,\n      0.06373599171638489,\n      -0.007484606932848692,\n      -0.05933971703052521,\n      0.027221905067563057,\n      -0.057710740715265274,\n      -0.0227652695029974,\n      -0.17688509821891785,\n      -0.025335654616355896,\n      0.04799323156476021,\n      0.06618557870388031,\n      0.0474494993686676,\n      -0.08261002600193024,\n      0.012075502425432205,\n      -0.11699265241622925,\n      -0.011164968833327293,\n      -0.12863856554031372,\n      -0.028767161071300507,\n      -0.06693911552429199,\n      -0.10838573426008224,\n      0.028844786807894707,\n      -0.06208617612719536,\n      -0.12897993624210358,\n      -0.14625155925750732,\n      -0.007512785028666258,\n      -0.15414851903915405,\n      -0.03304729983210564,\n      -0.03543352335691452,\n      -0.05037187039852142,\n      -0.03863596171140671,\n      0.04129675030708313,\n      -0.013654479756951332,\n      0.040800679475069046,\n      0.09951243549585342,\n      -0.04803387075662613,\n      -0.046647824347019196,\n      -0.09086165577173233,\n      -0.052585311233997345,\n      -0.024246210232377052,\n      0.06104318052530289,\n      -0.06950341910123825,\n      0.011570615693926811,\n      -0.012140273116528988,\n      0.005650973413139582,\n      0.05839156359434128,\n      0.12096177041530609,\n      -0.08898816257715225,\n      -0.016130656003952026,\n      0.008917475119233131,\n      -0.07294164597988129,\n      -0.11654794216156006,\n      0.0006086945068091154,\n      -0.0032763450872153044,\n      -0.0012113897828385234,\n      0.05012750253081322,\n      0.04897872358560562,\n      0.061354853212833405,\n      -0.06668602675199509,\n      -0.03169593587517738,\n      -0.05184536799788475,\n      -0.0018590142717584968,\n      -0.05270271375775337,\n      0.045167725533246994,\n      -0.032718200236558914,\n      -0.05136583372950554,\n      -0.04389738291501999,\n      0.08657042682170868,\n      -0.06067280471324921,\n      0.03297969698905945,\n      0.06737712770700455,\n      0.0733848512172699,\n      0.023936351761221886,\n      0.03901486471295357,\n      -0.15714819729328156,\n      0.06210295855998993,\n      -0.1811000555753708,\n      -0.05359386280179024,\n      -0.02605845220386982,\n      0.025790568441152573,\n      0.15048162639141083,\n      -0.17012187838554382,\n      -0.07436154037714005,\n      -0.019185712561011314,\n      0.07227609306573868,\n      -0.028169050812721252\n    ],\n    [\n      0.07109589874744415,\n      -0.16969633102416992,\n      -0.16780155897140503,\n      -0.12145794928073883,\n      0.008830723352730274,\n      0.11106884479522705,\n      0.014045185409486294,\n      0.14561821520328522,\n      -0.0032853707671165466,\n      0.09084050357341766,\n      0.0034160625655204058,\n      -0.1436244696378708,\n      -0.05506463348865509,\n      -0.14870108664035797,\n      0.023579031229019165,\n      -0.04051951691508293,\n      0.04019380360841751,\n      0.1022534891963005,\n      0.09301165491342545,\n      -0.04364527389407158,\n      -0.008406138978898525,\n      -0.0001578353112563491,\n      -0.014951620250940323,\n      -0.07811087369918823,\n      0.12005338072776794,\n      0.14297975599765778,\n      0.03310757875442505,\n      -0.08224429190158844,\n      0.034801188856363297,\n      0.04543266445398331,\n      0.13470359146595,\n      -0.10522439330816269,\n      0.147295743227005,\n      0.11910324543714523,\n      -0.12349000573158264,\n      0.0764249712228775,\n      0.11864879727363586,\n      0.034610725939273834,\n      -0.07096297293901443,\n      -0.0874444842338562,\n      0.0275998804718256,\n      0.0695219486951828,\n      0.03688616678118706,\n      0.13495317101478577,\n      0.008848016150295734,\n      0.012742413207888603,\n      0.06183028221130371,\n      0.005267830099910498,\n      0.10874375700950623,\n      0.008932745084166527,\n      -0.06058850884437561,\n      0.1688472330570221,\n      0.19406525790691376,\n      0.2177387923002243,\n      0.0015599194448441267,\n      -0.05464029312133789,\n      -0.05339214950799942,\n      0.0028494899161159992,\n      0.05960208550095558,\n      0.03204435855150223,\n      0.0640111118555069,\n      0.10165831446647644,\n      0.08081476390361786,\n      0.03580482304096222,\n      0.06747719645500183,\n      0.060187049210071564,\n      -0.0037059755995869637,\n      0.10089822858572006,\n      -0.09477867931127548,\n      0.05940939113497734,\n      0.05380439758300781,\n      0.0736917182803154,\n      -0.023553788661956787,\n      0.026858791708946228,\n      0.012800141237676144,\n      -0.04518068581819534,\n      -0.013661105185747147,\n      -0.0191109087318182,\n      -0.028581883758306503,\n      0.13565050065517426,\n      0.03761850297451019,\n      -0.026975002139806747,\n      0.1310114860534668,\n      -0.03135323151946068,\n      0.012976786121726036,\n      -0.06577901542186737,\n      -0.08681842684745789,\n      0.10613152384757996,\n      -0.02988854981958866,\n      0.06474226713180542,\n      -0.2076800912618637,\n      -0.07364057749509811,\n      -0.010940881446003914,\n      0.07999227941036224,\n      -0.11839189380407333,\n      -0.012872402556240559,\n      0.02068808116018772,\n      -0.031833015382289886,\n      0.04943454638123512,\n      -0.03167076036334038,\n      -0.08303041756153107,\n      -0.10334520041942596,\n      -0.08388365060091019,\n      -0.12947866320610046,\n      0.04307956248521805,\n      0.06296081840991974,\n      0.031597647815942764,\n      -0.14097854495048523,\n      -0.05100687965750694,\n      -0.0740637257695198,\n      0.06736815720796585,\n      -0.08575987815856934,\n      -0.028365839272737503,\n      -0.033837370574474335,\n      -0.00274243694730103,\n      -0.0386001281440258,\n      0.05603116378188133,\n      -0.08544936776161194,\n      0.08586958795785904,\n      -0.08282443881034851,\n      0.08531232923269272,\n      -0.045945122838020325,\n      0.05517188832163811,\n      0.019267074763774872,\n      0.05435928702354431\n    ],\n    [\n      0.048177871853113174,\n      0.06544478982686996,\n      0.03058682195842266,\n      -0.0919572189450264,\n      0.07181625068187714,\n      -0.014200394973158836,\n      0.08760573714971542,\n      0.044622112065553665,\n      -0.17854060232639313,\n      0.10384581983089447,\n      -0.08087368309497833,\n      -0.05634131655097008,\n      0.1313227266073227,\n      -0.12207915633916855,\n      -0.14793120324611664,\n      0.18388892710208893,\n      -0.08040927350521088,\n      0.026690764352679253,\n      0.03837065026164055,\n      -0.010465445928275585,\n      -0.0032844562083482742,\n      -0.05205993354320526,\n      0.06203015521168709,\n      0.09980381280183792,\n      0.03747744485735893,\n      0.07341359555721283,\n      0.14816269278526306,\n      0.10110469162464142,\n      -0.051548998802900314,\n      -0.15483920276165009,\n      -0.07730374485254288,\n      -0.021900493651628494,\n      0.012280680239200592,\n      -0.05755915120244026,\n      -0.07021362334489822,\n      0.052533045411109924,\n      0.05244153365492821,\n      -0.028977416455745697,\n      0.08921685814857483,\n      0.04997451230883598,\n      -0.03308722749352455,\n      0.02995588816702366,\n      -0.031214486807584763,\n      0.04820423200726509,\n      0.08311056345701218,\n      -0.013957168906927109,\n      -0.05331539735198021,\n      -0.07273249328136444,\n      -0.03885602205991745,\n      0.013733858242630959,\n      0.014129143208265305,\n      -0.040794048458337784,\n      0.0489640049636364,\n      -0.055347952991724014,\n      -0.014023671858012676,\n      0.01172875240445137,\n      0.07044723629951477,\n      0.05906829982995987,\n      -0.06219961494207382,\n      0.0849749818444252,\n      -0.03583869710564613,\n      0.0642164945602417,\n      -0.06215958669781685,\n      -0.03834511339664459,\n      -0.057029906660318375,\n      0.01368633471429348,\n      -0.06863302737474442,\n      -0.07051841914653778,\n      -0.06748451292514801,\n      -0.08985026180744171,\n      -0.0481082983314991,\n      -0.051697537302970886,\n      -0.06215956062078476,\n      -0.11553419381380081,\n      -0.07205385714769363,\n      0.06509734690189362,\n      0.045563891530036926,\n      -0.004557901993393898,\n      -0.04936417564749718,\n      0.08373074233531952,\n      0.02577734924852848,\n      -0.015585188753902912,\n      0.0022121223155409098,\n      -0.054336704313755035,\n      0.06762263178825378,\n      0.0614423044025898,\n      0.016916347667574883,\n      -0.04372672364115715,\n      -0.06232316792011261,\n      -0.06122177094221115,\n      0.046097226440906525,\n      0.01347972359508276,\n      -0.014327993616461754,\n      -0.019914232194423676,\n      0.04271848872303963,\n      0.056704748421907425,\n      0.04910196363925934,\n      -0.019866526126861572,\n      -0.11326329410076141,\n      -0.025497661903500557,\n      -0.03798598423600197,\n      0.07930208742618561,\n      0.05618339404463768,\n      0.0063173468224704266,\n      0.09508146345615387,\n      0.04381081089377403,\n      -0.06499193608760834,\n      0.0017094104550778866,\n      -0.02254173532128334,\n      0.14309588074684143,\n      0.0026640237774699926,\n      0.06409749388694763,\n      0.054033346474170685,\n      0.0901520624756813,\n      0.0169213954359293,\n      -0.10814869403839111,\n      0.04695489630103111,\n      -0.05379706248641014,\n      0.005350610241293907,\n      0.03835168480873108,\n      -0.08678561449050903,\n      -0.03224557265639305,\n      0.03317989408969879,\n      0.057647012174129486,\n      0.012259582988917828\n    ]\n  ],\n  "hidden_bias": [\n    -0.07422218471765518,\n    0.04769578576087952,\n    -0.03948693349957466,\n    0.09125815331935883,\n    -0.038184572011232376,\n    -0.029220130294561386,\n    -0.08633722364902496,\n    -0.003050123108550906,\n    -0.10350099951028824,\n    -0.00765001866966486,\n    -0.038660649210214615,\n    0.0657229945063591,\n    0.02945590950548649,\n    -0.006790105253458023,\n    0.0420505665242672,\n    0.05386434122920036,\n    -0.12374017387628555,\n    0.1124410629272461,\n    0.06022528186440468,\n    0.022755147889256477,\n    0.04203921556472778,\n    -0.041379984468221664,\n    0.03657274320721626,\n    0.026750195771455765,\n    -0.0017855650512501597,\n    0.054029930382966995,\n    0.03358159214258194,\n    -0.027184147387742996,\n    0.003157140454277396,\n    0.03239120915532112,\n    -0.03368687257170677,\n    -0.050139106810092926,\n    0.006645390763878822,\n    0.0878707692027092,\n    0.05747509375214577,\n    0.05746740475296974,\n    0.07171259075403214,\n    0.03489675745368004,\n    -0.02010141871869564,\n    0.012798401527106762,\n    -0.026644540950655937,\n    -0.0678088590502739,\n    -0.04606543481349945,\n    0.08630137890577316,\n    0.0660821944475174,\n    -0.047469738870859146,\n    -0.038418181240558624,\n    -0.07551571726799011,\n    0.07356099039316177,\n    0.05148177966475487,\n    0.06569477915763855,\n    0.04273805022239685,\n    -0.02253495343029499,\n    -0.08107928931713104,\n    0.04663136601448059,\n    0.017505712807178497,\n    0.08010519295930862,\n    0.013298952020704746,\n    -0.05159810557961464,\n    0.08531086146831512,\n    -0.0022237617522478104,\n    -0.048167333006858826,\n    -0.06333929300308228,\n    0.1022304967045784\n  ],\n  "output_weight": [\n    0.06309229880571365,\n    -0.11826402693986893,\n    0.07883037626743317,\n    0.013170122168958187,\n    -0.0864688977599144,\n    0.153001606464386,\n    -0.05896884202957153,\n    0.19503475725650787,\n    0.13501784205436707,\n    -0.05688941106200218,\n    0.16442428529262543,\n    -0.05110658332705498,\n    -0.044882941991090775,\n    -0.09894964843988419,\n    -0.1414792686700821,\n    0.05160979554057121,\n    0.07055862993001938,\n    0.029682492837309837,\n    -0.018195977434515953,\n    0.12195465713739395,\n    -0.05141095072031021,\n    0.034687381237745285,\n    -0.05013133957982063,\n    0.12108516693115234,\n    0.1490292102098465,\n    -0.07584413886070251,\n    0.0564471036195755,\n    0.10464183986186981,\n    -0.04147222265601158,\n    -0.04513047635555267,\n    -0.028229793533682823,\n    0.06687689572572708,\n    -0.04912390187382698,\n    0.06467515975236893,\n    0.04627872630953789,\n    -0.15687768161296844,\n    -0.012363845482468605,\n    -0.04626663029193878,\n    -0.06031527742743492,\n    0.12260821461677551,\n    0.04521757736802101,\n    0.04714829474687576,\n    0.053050488233566284,\n    0.06540530920028687,\n    -0.07258530706167221,\n    0.06504590809345245,\n    -0.13400259613990784,\n    -0.03432918339967728,\n    0.03363534435629845,\n    0.03039093315601349,\n    -0.0420764684677124,\n    0.03765863552689552,\n    0.09020864963531494,\n    0.04840902239084244,\n    -0.05870692431926727,\n    0.04783988371491432,\n    -0.06574137508869171,\n    -0.04319070279598236,\n    -0.0549791045486927,\n    0.1021905392408371,\n    -0.06607483327388763,\n    0.10928201675415039,\n    0.1497369110584259,\n    -0.040101733058691025\n  ],\n  "output_bias": 0.06941737979650497,\n  "training": {\n    "method": "paired counterfactual policy distillation with residual scorer",\n    "objective": "mse",\n    "pair_gap": 0.0,\n    "uncertainty_only_weights": true,\n    "base_calibration_scale": 0.05,\n    "seed": 20261022,\n    "regularization": 0.01,\n    "best_epoch": 3,\n    "validation_loss": 0.4327501952648163,\n    "baseline_validation_loss": 0.47088056802749634,\n    "training_games": 2400,\n    "validation_games": 599,\n    "source_dataset": {\n      "method": "paired one-step policy-improvement rollouts",\n      "input_features": [\n        "play",\n        "number",\n        "skip",\n        "reverse",\n        "draw_two",\n        "wild",\n        "wild_draw_four",\n        "remaining_colour_count",\n        "remaining_colour_fraction",\n        "remaining_match_count",\n        "attack_next_threat",\n        "reverse_escape_threat",\n        "wild_when_small_hand",\n        "finish",\n        "keep",\n        "choose_colour",\n        "next_can_finish",\n        "own_hand_size",\n        "successor_colour_absence",\n        "successor_symbol_absence",\n        "successor_colour_preference",\n        "successor_finish_risk",\n        "remaining_colour_components",\n        "largest_component_fraction",\n        "remaining_bridge_symbols",\n        "remaining_duplicates",\n        "remaining_colour_entropy",\n        "remaining_wilds",\n        "chosen_colour_exhausted",\n        "stack_penalty_play",\n        "successor_one_card_colour_absence",\n        "successor_one_card_symbol_absence",\n        "remaining_colour_slot_0_number_0",\n        "remaining_colour_slot_0_number_1",\n        "remaining_colour_slot_0_number_2",\n        "remaining_colour_slot_0_number_3",\n        "remaining_colour_slot_0_number_4",\n        "remaining_colour_slot_0_number_5",\n        "remaining_colour_slot_0_number_6",\n        "remaining_colour_slot_0_number_7",\n        "remaining_colour_slot_0_number_8",\n        "remaining_colour_slot_0_number_9",\n        "remaining_colour_slot_0_skip_None",\n        "remaining_colour_slot_0_reverse_None",\n        "remaining_colour_slot_0_draw_two_None",\n        "remaining_colour_slot_1_number_0",\n        "remaining_colour_slot_1_number_1",\n        "remaining_colour_slot_1_number_2",\n        "remaining_colour_slot_1_number_3",\n        "remaining_colour_slot_1_number_4",\n        "remaining_colour_slot_1_number_5",\n        "remaining_colour_slot_1_number_6",\n        "remaining_colour_slot_1_number_7",\n        "remaining_colour_slot_1_number_8",\n        "remaining_colour_slot_1_number_9",\n        "remaining_colour_slot_1_skip_None",\n        "remaining_colour_slot_1_reverse_None",\n        "remaining_colour_slot_1_draw_two_None",\n        "remaining_colour_slot_2_number_0",\n        "remaining_colour_slot_2_number_1",\n        "remaining_colour_slot_2_number_2",\n        "remaining_colour_slot_2_number_3",\n        "remaining_colour_slot_2_number_4",\n        "remaining_colour_slot_2_number_5",\n        "remaining_colour_slot_2_number_6",\n        "remaining_colour_slot_2_number_7",\n        "remaining_colour_slot_2_number_8",\n        "remaining_colour_slot_2_number_9",\n        "remaining_colour_slot_2_skip_None",\n        "remaining_colour_slot_2_reverse_None",\n        "remaining_colour_slot_2_draw_two_None",\n        "remaining_colour_slot_3_number_0",\n        "remaining_colour_slot_3_number_1",\n        "remaining_colour_slot_3_number_2",\n        "remaining_colour_slot_3_number_3",\n        "remaining_colour_slot_3_number_4",\n        "remaining_colour_slot_3_number_5",\n        "remaining_colour_slot_3_number_6",\n        "remaining_colour_slot_3_number_7",\n        "remaining_colour_slot_3_number_8",\n        "remaining_colour_slot_3_number_9",\n        "remaining_colour_slot_3_skip_None",\n        "remaining_colour_slot_3_reverse_None",\n        "remaining_colour_slot_3_draw_two_None",\n        "remaining_wild_count",\n        "remaining_wild_draw_four_count",\n        "action_number_0",\n        "action_number_1",\n        "action_number_2",\n        "action_number_3",\n        "action_number_4",\n        "action_number_5",\n        "action_number_6",\n        "action_number_7",\n        "action_number_8",\n        "action_number_9",\n        "opponent_slot_0_present",\n        "opponent_slot_0_hand_size",\n        "opponent_slot_0_one_card",\n        "opponent_slot_0_colour_absence",\n        "opponent_slot_0_symbol_absence",\n        "opponent_slot_0_colour_preference",\n        "opponent_slot_1_present",\n        "opponent_slot_1_hand_size",\n        "opponent_slot_1_one_card",\n        "opponent_slot_1_colour_absence",\n        "opponent_slot_1_symbol_absence",\n        "opponent_slot_1_colour_preference",\n        "opponent_slot_2_present",\n        "opponent_slot_2_hand_size",\n        "opponent_slot_2_one_card",\n        "opponent_slot_2_colour_absence",\n        "opponent_slot_2_symbol_absence",\n        "opponent_slot_2_colour_preference",\n        "selected_is_active_colour",\n        "pending_penalty_size",\n        "turn_progress",\n        "active_opponents",\n        "closest_opponent_risk",\n        "route_sequence_fraction",\n        "route_sequence_stranded",\n        "route_ending_wild",\n        "route_next_colour_absence",\n        "route_next_symbol_absence",\n        "route_next_colour_preference"\n      ],\n      "feature_version": 4,\n      "games": 3000,\n      "states": 11872,\n      "actions": 38902,\n      "trials": 8,\n      "future_shuffle": true,\n      "reward_conditioning": "present hidden simulator hands held fixed; future draw-pile permutations averaged",\n      "max_actions": 8,\n      "seed_start": 49000000,\n      "sdk_commit": "0cda3acc1bb5c05fa1803a8fbd7761770dc6d5fe",\n      "incumbent": {\n        "format": "triad-uno-route-linear-v1",\n        "features": [\n          "play",\n          "number",\n          "skip",\n          "reverse",\n          "draw_two",\n          "wild",\n          "wild_draw_four",\n          "remaining_colour_count",\n          "remaining_colour_fraction",\n          "remaining_match_count",\n          "attack_next_threat",\n          "reverse_escape_threat",\n          "wild_when_small_hand",\n          "finish",\n          "keep",\n          "choose_colour",\n          "next_can_finish",\n          "own_hand_size",\n          "sequence_fraction",\n          "sequence_stranded",\n          "ending_wild",\n          "next_colour_absence",\n          "next_symbol_absence",\n          "next_colour_preference"\n        ],\n        "weights": [\n          8.022466349142684,\n          0.44153069585311683,\n          -0.06834035646341297,\n          1.4643534535685507,\n          -2.338619866848944,\n          -3.4540620544700498,\n          -4.276279368110987,\n          1.6602353386949849,\n          1.0165100023007814,\n          0.901341302040876,\n          3.853185081203389,\n          2.4436903583872347,\n          1.7223969671761579,\n          11.872419584625783,\n          -4.598739912392293,\n          3.347870880522802,\n          0.32778952616834095,\n          -0.24548152213113314,\n          0.054801310646275025,\n          -1.1349214815798592,\n          -0.04225874297028254,\n          1.1191018681573717,\n          0.21891410472226475,\n          -0.08122782541248305\n        ],\n        "training": {\n          "method": "frozen-base residual CEM, own-hand bounded sequence features",\n          "seed": 20261019,\n          "iterations": 24,\n          "sdk_commit": "0cda3acc1bb5c05fa1803a8fbd7761770dc6d5fe",\n          "selection_seed_start": 21000000,\n          "test_seed_start": 22000000,\n          "base": {\n            "format": "triad-uno-linear-v1",\n            "features": [\n              "play",\n              "number",\n              "skip",\n              "reverse",\n              "draw_two",\n              "wild",\n              "wild_draw_four",\n              "remaining_colour_count",\n              "remaining_colour_fraction",\n              "remaining_match_count",\n              "attack_next_threat",\n              "reverse_escape_threat",\n              "wild_when_small_hand",\n              "finish",\n              "keep",\n              "choose_colour",\n              "next_can_finish",\n              "own_hand_size"\n            ],\n            "weights": [\n              8.022466349142684,\n              0.44153069585311683,\n              -0.06834035646341297,\n              1.4643534535685507,\n              -2.338619866848944,\n              -3.4540620544700498,\n              -4.276279368110987,\n              1.6602353386949849,\n              1.0165100023007814,\n              0.901341302040876,\n              3.853185081203389,\n              2.4436903583872347,\n              1.7223969671761579,\n              11.872419584625783,\n              -4.598739912392293,\n              3.347870880522802,\n              0.32778952616834095,\n              -0.24548152213113314\n            ],\n            "training": {\n              "method": "cross-entropy policy optimization",\n              "seed": 20261014,\n              "generations": 30,\n              "population": 32,\n              "games_per_candidate": 64,\n              "validation": 3.005859375,\n              "sdk_commit": "0cda3acc1bb5c05fa1803a8fbd7761770dc6d5fe",\n              "feature_count": 18\n            }\n          }\n        }\n      },\n      "seconds": 1638.4,\n      "reservoir_sample": true,\n      "own_hand_size_histogram": {\n        "1": 106,\n        "2": 1146,\n        "3": 1762,\n        "4": 1882,\n        "5": 1900,\n        "6": 1847,\n        "7": 1744,\n        "8": 566,\n        "9": 361,\n        "10": 225,\n        "11": 139,\n        "12": 74,\n        "13": 52,\n        "14": 27,\n        "15": 21,\n        "16": 9,\n        "17": 5,\n        "18": 4,\n        "19": 1,\n        "22": 1\n      },\n      "phase_histogram": {\n        "TURN": 9755,\n        "AFTER_DRAW": 2104,\n        "CHOOSE_INITIAL_COLOR": 13\n      }\n    },\n    "sdk_commit": "0cda3acc1bb5c05fa1803a8fbd7761770dc6d5fe",\n    "residual_scale": 1.0\n  }\n}\n'}
for name,source in models.items():(work/name).write_text(source)
subprocess.run(["git","clone","--quiet","https://github.com/Sir-Nike/unoSDK.git",str(work/"unoSDK")],check=True)
subprocess.run(["git","checkout","--quiet","0cda3acc1bb5c05fa1803a8fbd7761770dc6d5fe"],cwd=work/"unoSDK",check=True)
os.environ["UNO_SDK"]=str(work/"unoSDK")
os.environ["OPENBLAS_NUM_THREADS"]="1"
os.environ["OMP_NUM_THREADS"]="1"
command=["python",str(work/"train_uno_joint_route.py"),"--incumbent",str(work/"incumbent.json"),"--v1",str(work/"v1.json"),"--neural-rival",str(work/"neural-route.json"),"--neural-rival",str(work/"neural-ranking.json"),"--neural-rival",str(work/"neural-mse.json"),"--workers","4"]
subprocess.run(command+["--iterations","2","--population","4","--train-blocks","2","--selection-blocks","4","--test-blocks","4","--train-seed-start","87000000","--selection-seed-start","88000000","--test-seed-start","89000000","--output",str(work/"smoke")],check=True,timeout=600)
subprocess.run(command+["--iterations","24","--population","24","--train-blocks","32","--selection-blocks","256","--test-blocks","512","--train-seed-start","80000000","--selection-seed-start","81000000","--test-seed-start","82000000","--output","/kaggle/working/output"],check=True,timeout=10800)
